Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0); you may not use this file except in compliance with the Apache 2.0 license. You may obtain a copy of the Apache 2.0 license at: https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0 International License (CC-BY). You may obtain a copy of the CC-BY license at: https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and materials distributed here under the Apache 2.0 or CC-BY licenses are distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the licenses for the specific language governing permissions and limitations under those licenses.

This is not an official Google product.

# Improving the matrix multiplication exponent $\omega < 2.371177$

Despite its prominence, the asymptotic computational complexity of matrix multiplication, i.e., the number of arithmetic operations needed to multiply large matrices is unknown, and determining it is a major open question in theoretical computer science. Strassen showed that two $n\times n$ matrices can be multiplied in sub-cubic time, specifically, $O(n^\omega)$ operations for
$\omega < 2.81$, spurring a line of work attempting to further reduce the complexity exponent $\omega$. The current best bound is $\omega < 2.371339$, obtained by [Alman et al. 2025](https://doi.org/10.1137/1.9781611978322.63), using a technique called combination loss analysis, which relies on solving a complex non-convex optimization problem. Here, building on their work and using AlphaEvolve, we improve the bound to $\omega < 2.371177$.

The combination loss analysis has a parameter, the maximum recursion level $\ell^*$, that introduces a trade-off between the complexity of the optimization (which
grows doubly exponentially in $\ell^*$) and the best possible bound on ω it can achieve.
The SOTA bound was found with $\ell^* = 3$. In contrast, our optimization framework can handle up to $\ell^*=4$. Our AlphaEvolve pipeline applied on level $\ell^*=3$ also tightens the SOTA bound, offering a more modest improvement $\omega < 2.371277$.

For completeness, we present the best solutions AlphaEvolve has found for the optimization problems for levels $\ell^*=2,3,4$, together with their resulting $\omega$s. All the calculations use exact rational arithmetic, hence the resulting upper bounds are rigorous and do not suffer from floating-point errors.

For computing the omega upper bounds from maximum-level $\ell^*=4$, you need to
download the `clean_level4.npz.xz` file from the repository to your machine, and then upload it on this Colab. This computation can take 30 minutes to complete and requires 20 GB of memory (see the last cell).

Please refer to the paper's supplementary information for details of the optimization problem.

In [ ]:
#@title Imports and constants (run this cell first)
from __future__ import annotations

import dataclasses
import functools
import lzma
import math
import pathlib
from typing import Callable, Iterable, Sequence, TypeVar
import time
import decimal
from google.colab import files

!pip install gmpy2
import gmpy2
import mpmath
import numpy as np

Fraction = gmpy2.mpq

RATIONAL_DENOM = gmpy2.mpz(10**20)
ONE = Fraction(1)
ZERO = Fraction(0)
DECIMAL_PRECISION = 60
NUM_REGIONS = 6

T = TypeVar("T")

In [ ]:
#@title Verification function using exact rational arithmetic

"""Verifier for the omega optimization problem defined in the appendix.

We use rational arithmetic throughout to avoid floating-point errors and
obtain rigorous bounds.

Every quantity is rounded in the direction that can only increase the
final computed bound, so a passing run certifies the bound.

Although the appendix's formulation specifies base 2 logarithms, the code uses
natural logarithms via `mpmath.log`. This uniformly scales `E_total`, `M_total`,
and the target by ln(2). Hence, the ln(2) scalar factors cancel algebraically,
preserving the exact same upper bound for Omega.
"""

@functools.lru_cache(maxsize=None)
def floor_rational(x: Fraction) -> Fraction:
  """Round *down* to nearest multiple of 1/RATIONAL_DENOM.

  Args:
    x: The fraction.

  Returns:
    The floored fractional value.
  """
  num = gmpy2.mul(x.numerator, RATIONAL_DENOM)
  q = gmpy2.f_div(num, x.denominator)
  return Fraction(q, RATIONAL_DENOM)


@functools.lru_cache(maxsize=None)
def ceil_rational(x: Fraction) -> Fraction:
  """Round *up* to nearest multiple of 1/RATIONAL_DENOM.

  Args:
    x: The fraction.

  Returns:
    The ceiled fractional value.
  """
  num = gmpy2.mul(x.numerator, RATIONAL_DENOM)
  q = gmpy2.c_div(num, x.denominator)
  return Fraction(q, RATIONAL_DENOM)


def mpf_to_fraction(x: mpmath.mpf) -> Fraction:
  """Convert mpmath.mpf to exact Fraction."""
  man, exp = x.man_exp
  if man == 0:
    return ZERO
  frac = Fraction(man, 1)
  if exp > 0:
    frac = frac * Fraction(2**exp)
  elif exp < 0:
    frac = Fraction(man, 2 ** (-exp))
  # mpmath's man_exp returns the absolute (positive) mantissa, so we must
  # manually apply the sign if the value is negative.
  if x < 0:
    frac = -frac
  return frac


def fractional_logarithm(
    x: Fraction,
    round_fn: Callable[[Fraction, Fraction], Fraction],
    func_name: str,
) -> Fraction:
  """Rigorous lower or upper bound on ln(x), rounded using round_fn."""
  if x <= ZERO:
    raise ValueError(f"{func_name}: x must be positive, got {x}")

  required_binary_precision = (
      int(math.ceil((DECIMAL_PRECISION + 1) * math.log2(10)))
      + x.numerator.bit_length()
      + x.denominator.bit_length()
  )

  with mpmath.workprec(required_binary_precision):
    # We compute log(numerator) and log(denominator) separately to avoid
    # precision loss that could occur if we computed numerator / denominator
    # before taking the logarithm.
    v1 = mpmath.log(mpmath.mpf(x.numerator))
    v2 = mpmath.log(mpmath.mpf(x.denominator))

    # Calculate the Unit in the Last Place (ULP) of the mpmath approximation.
    # When x is close to 1, log(x) is close to 0, and the subtraction
    # v1 - v2 can suffer from catastrophic cancellation. Therefore, we base
    # our error bound (ULP) on the magnitude of the largest operand,
    # guaranteeing safety against cancellation errors.
    # The binary exponent of the most significant bit is exp + bc.
    # Subtracting the precision gives the exponent of the least significant
    # bit of the precision grid, which is the maximum absolute error (1 ULP).
    max_val = max(abs(v1), abs(v2))
    man, exp = max_val.man_exp
    bc = man.bit_length() if man else 0
    ulp_exp = exp + bc - required_binary_precision

    # Represent ULP as a Fraction to maintain exact rational arithmetic.
    if ulp_exp >= 0:
      ulp = Fraction(2**ulp_exp)
    else:
      ulp = Fraction(1, 2 ** (-ulp_exp))

    val = v1 - v2
    # Convert the mpmath float approximation to a rational Fraction.
    frac = mpf_to_fraction(val)

    # round_fn applies the ULP guard (subtracting for lower bound, adding for
    # upper bound) and then rounds to the target grid (1/RATIONAL_DENOM).
    # Since we compute log(num) - log(den), it accumulates up to 2 ULPs of
    # error.
    return round_fn(frac, 2 * ulp)


@functools.lru_cache(maxsize=None)
def log_lower_bound(x: Fraction) -> Fraction:
  """Rigorous lower bound on ln(x), rounded down to 1/RATIONAL_DENOM.

  Computes ln(x) with DECIMAL_PRECISION digits then floors.
  Since the mpmath approximation error is much smaller than the rounding
  grid (10^-60 vs 10^-20), flooring gives a strict lower bound.

  Args:
    x: The input Fraction.

  Returns:
    The lower bound on ln(x).
  """
  return fractional_logarithm(
      x, lambda x, delta: floor_rational(x - delta), "log_lower_bound"
  )


@functools.lru_cache(maxsize=None)
def log_upper_bound(x: Fraction) -> Fraction:
  """Rigorous upper bound on ln(x), rounded up to 1/RATIONAL_DENOM.

  Computes ln(x) with DECIMAL_PRECISION digits then ceils.
  Since the mpmath approximation error is much smaller than the rounding
  grid (10^-60 vs 10^-20), ceiling gives a strict upper bound.

  Args:
    x: The input Fraction.

  Returns:
    The upper bound on ln(x).
  """
  return fractional_logarithm(
      x, lambda x, delta: ceil_rational(x + delta), "log_upper_bound"
  )


def xlogx_lower(p: Fraction) -> Fraction:
  """Returns a lower bound on p * ln(p). Requires p > 0.

  Since p > 0, multiplying both sides of `log_lower_bound(p) <= ln(p)`
  by p preserves the inequality direction:
  `p * log_lower_bound(p) <= p * ln(p)`.

  Args:
    p: The input Fraction.

  Returns:
    The lower bound.
  """
  if p <= ZERO:
    raise ValueError(f"xlogx_lower: p must be positive, got {p}")
  return p * log_lower_bound(p)


def xlogx_upper(p: Fraction) -> Fraction:
  """Upper bound on p * ln(p). Requires p > 0.

  Args:
    p: The input Fraction.

  Returns:
    The upper bound.
  """
  if p <= ZERO:
    raise ValueError(f"xlogx_upper: p must be positive, got {p}")
  return p * log_upper_bound(p)


ENTROPY_LOWER_BOUND_CACHE = {}


def entropy_lower_bound(dist: Sequence[Fraction] | np.ndarray) -> Fraction:
  """Computes an exact lower bound on Shannon entropy.

  Args:
    dist: The distribution.

  Returns:
    The lower bound.
  """
  key = tuple(dist)
  if key in ENTROPY_LOWER_BOUND_CACHE:
    return ENTROPY_LOWER_BOUND_CACHE[key]

  total = ZERO
  for p in key:
    # Negative entries are not explicitly guarded against here, but xlogx_upper
    # will raise a ValueError if it encounters a negative value.
    if p > ZERO:
      total += xlogx_upper(p)
  val = floor_rational(-total)
  ENTROPY_LOWER_BOUND_CACHE[key] = val
  return val


ENTROPY_UPPER_BOUND_CACHE = {}


def entropy_upper_bound(dist: Sequence[Fraction] | np.ndarray) -> Fraction:
  """Computes an exact upper bound on Shannon entropy.

  Args:
    dist: The distribution.

  Returns:
    The upper bound.
  """
  key = tuple(dist)
  if key in ENTROPY_UPPER_BOUND_CACHE:
    return ENTROPY_UPPER_BOUND_CACHE[key]

  total = ZERO
  for p in key:
    # Negative entries are not explicitly guarded against here, but xlogx_lower
    # will raise a ValueError if it encounters a negative value.
    if p > ZERO:
      total += xlogx_lower(p)
  val = ceil_rational(-total)
  ENTROPY_UPPER_BOUND_CACHE[key] = val
  return val


def normalized_entropy_upper_bound(
    weights: Sequence[Fraction] | np.ndarray, sum_weight: Fraction
) -> Fraction:
  """Upper bound on sum(-w_i * ln(w_i / p)), where p = sum(w_i)."""
  if sum_weight != sum(weights):
    raise ValueError(f"sum_weight {sum_weight} != sum(weights) {sum(weights)}")
  if sum_weight <= ZERO:
    raise ValueError(f"sum_weight must be positive, got {sum_weight}")
  total = ZERO
  for w in weights:
    if w > ZERO:
      total -= xlogx_lower(w)
  total += xlogx_upper(sum_weight)
  return ceil_rational(total)


# The six regions' permutations pi_r over {X, Y, Z} in lexicographic order.
# Dimensions X, Y, Z are coordinate indices 0, 1, 2 throughout this file, and
# regions are 0-based (the appendix's region r has index r - 1 here).
_DIMS_TABLE = (
    (0, 1, 2),
    (0, 2, 1),
    (1, 0, 2),
    (1, 2, 0),
    (2, 0, 1),
    (2, 1, 0),
)


def dims(region: int) -> tuple[int, int, int]:
  """Return (pi_r(X), pi_r(Y), pi_r(Z)) as coordinate indices."""
  return _DIMS_TABLE[region]


def prepare_shapes(level: int) -> np.ndarray:
  """Return all shapes at a level: (i, j, k) >= 0 with i + j + k = 2**level."""
  sum_shape = 2**level
  shapes = []
  for i in range(sum_shape + 1):
    for j in range(sum_shape - i + 1):
      dim2 = sum_shape - i - j
      shapes.append((i, j, dim2))
  return np.array(shapes, dtype=int)


def prepare_splits(shape: Sequence[int] | np.ndarray) -> np.ndarray:
  """Return all split shapes (u, s - u) for a given shape s."""
  shape = np.asarray(shape, dtype=int)
  sum_shape = int(np.sum(shape))
  sum_half = sum_shape // 2
  splits = []
  for i in range(min(sum_half, shape[0]) + 1):
    for j in range(min(sum_half - i, shape[1]) + 1):
      dim2 = sum_half - i - j
      # The <= here allows dim2 == shape[2], which correctly takes the entire
      # budget in the third dimension for the left split.
      if dim2 <= shape[2]:
        splits.append((i, j, dim2, shape[0] - i, shape[1] - j, shape[2] - dim2))
  return np.array(splits, dtype=int)


def encode_csd(arr: Sequence[int] | np.ndarray) -> int:
  """Index of a complete-split vector L in {0,1,2}^power in base-3 encoding."""
  arr = list(arr)
  val = 0
  for a in arr:
    val = val * 3 + int(a)
  return val


DECODE_CSD_CACHE = {}


def decode_csd(idx: int, power: int) -> np.ndarray:
  """Decodes a CSD index to array."""
  if idx < 0 or idx >= 3**power:
    raise IndexError(f"idx {idx} out of bounds for power {power}")
  table = DECODE_CSD_CACHE.get(power, None)
  if table is None:
    table = []
    for val in range(3**power):
      arr = [0] * power
      for j in range(power - 1, -1, -1):
        arr[j] = val % 3
        val //= 3
      arr_np = np.array(arr, dtype=int)
      arr_np.flags.writeable = False
      table.append(arr_np)
    DECODE_CSD_CACHE[power] = table
  return table[idx]


def concat_csd(
    lhs: np.ndarray, rhs: np.ndarray
) -> tuple[list[int], np.ndarray]:
  """Elementwise product of nonzero entries of two distributions.

  Args:
    lhs: Left distribution.
    rhs: Right distribution.

  Returns:
    Valid concatenations (indices, values).
  """
  lhs_nz = np.nonzero(lhs != ZERO)[0]
  rhs_nz = np.nonzero(rhs != ZERO)[0]
  n_rhs = len(rhs)
  indices = []
  values = []
  for li in lhs_nz:
    for ri in rhs_nz:
      indices.append(li * n_rhs + ri)
      values.append(lhs[li] * rhs[ri])
  return indices, np.array(values, dtype=object)


def marginal_dist(
    alph: Sequence[Fraction] | np.ndarray, shapes: np.ndarray, sum_shape: int
) -> tuple[list[Fraction], list[Fraction], list[Fraction]]:
  """Compute the marginal distributions of `alph` in all dimensions.

  Args:
    alph: The distribution.
    shapes: A matrix of shape (num_shapes, 3).
    sum_shape: The sum of coordinates (equal to 2**level).

  Returns:
    The marginal distributions in each dimension.
  """
  marginal_x = [ZERO] * (sum_shape + 1)
  marginal_y = [ZERO] * (sum_shape + 1)
  marginal_z = [ZERO] * (sum_shape + 1)

  for i, (cx, cy, cz) in enumerate(shapes):
    marginal_x[cx] += alph[i]
    marginal_y[cy] += alph[i]
    marginal_z[cz] += alph[i]

  return marginal_x, marginal_y, marginal_z


def rot3(vec: list[T], n: int = 1) -> list[T]:
  """Rotate a 3-element list/array to the left by `n` positions."""
  assert len(vec) == 3, "rot3 requires a 3-element list"
  n = n % 3
  if n == 0:
    return vec
  if n == 1:
    return [vec[1], vec[2], vec[0]]
  return [vec[2], vec[0], vec[1]]


def compute_max_derivative(
    num_items: int,
    shapes: np.ndarray,
    lam_margin: list[list[Fraction]],
    lam_margin_low: np.ndarray | tuple[int, int, int],
    lam_sum: Fraction,
    dist_max: list[Fraction],
) -> Fraction:
  """Computes the certificate epsilon for the distribution.

  The certificate epsilon is defined as the maximum absolute difference between
  the linear approximation (sum of marginal lambdas plus lambda_0) and the
  natural logarithm of the maximum distribution value, taken over all shapes.

  Specifically, for each shape s, it computes:
    | l_0(s_0) + l_1(s_1) + l_2(s_2) + lambda_0 - ln(y(s)) |
  where lambda_0 = lam_sum - 1, and y(s) is the corresponding value in dist_max.
  Since ln(y(s)) cannot be computed as a fraction, the function uses fractional
  lower and upper bounds on ln(y(s)) to find the worst-case difference.

  Args:
    num_items: The number of shapes to verify.
    shapes: A numpy array of shape (num_items, 3) containing the 3D
      shapes.
    lam_margin: A list of 3 lists containing the lambda margins for each
      dimension, one list per spatial dimension (X, Y, Z).
    lam_margin_low: The lower bounds (offsets) for each dimension, used to index
      into lam_margin.
    lam_sum: The sum of lambdas (used to compute lambda_0 as lam_sum - 1).
    dist_max: A list of length num_items containing the maximum distribution
      values for each shape.

  Returns:
    The computed maximum derivative (epsilon), rounded up to the nearest
    multiple of 1/RATIONAL_DENOM.
  """
  max_derivative = ZERO
  for i in range(num_items):
    s = shapes[i]
    l0 = lam_margin[0][s[0] - lam_margin_low[0]]
    l1 = lam_margin[1][s[1] - lam_margin_low[1]]
    l2 = lam_margin[2][s[2] - lam_margin_low[2]]
    linear_part = l0 + l1 + l2 + (lam_sum - ONE)  # lam_sum - 1 == lambda_0.
    dm = dist_max[i]
    log_lo = log_lower_bound(dm)
    log_hi = log_upper_bound(dm)
    worst = max(abs(linear_part - log_lo), abs(linear_part - log_hi))
    max_derivative = max(max_derivative, worst)
  return ceil_rational(max_derivative)


def decode_string(value: bytes | np.bytes_ | str) -> str:
  """Decode bytes or numpy bytes to string."""
  if isinstance(value, bytes):
    return value.decode("ascii")
  if isinstance(value, np.bytes_):
    return bytes(value).decode("ascii")
  return str(value)


def load_fraction_vector(
    path: str | pathlib.Path,
) -> list[Fraction]:
  """Loads a rational vector from an .npz or .npz.xz file.

  For .npz files, it expects two arrays: `numerators` and `denominators` as strings.
  For .npz.xz files, it expects three binary arrays: `numerators_binary`,
  `denominators_dict_binary`, and `denominators_indices`.

  Args:
    path: The path to the `.npz` or `.npz.xz` file.

  Returns:
    A list of exact Fraction (gmpy2.mpq) objects.
  """
  path_str = str(path)
  if path_str.endswith('.xz'):
    with lzma.open(path, 'rb') as f:
      with np.load(f, allow_pickle=False) as data:
        num_bin = data["numerators_binary"]
        den_dict_bin = data["denominators_dict_binary"]
        den_idx = data["denominators_indices"]

        # Unpack denominators
        dens = []
        for row in den_dict_bin:
          dens.append(int.from_bytes(row.tobytes(), byteorder='big', signed=True))

        # Unpack numerators and combine
        values = []
        for n_row, d_i in zip(num_bin, den_idx):
          n_val = int.from_bytes(n_row.tobytes(), byteorder='big', signed=True)
          values.append(Fraction(n_val, dens[d_i]))
        return values

  with np.load(path, allow_pickle=False) as data:
    nums = data["numerators"]
    dens = data["denominators"]
    values = [
        Fraction(int(decode_string(n)), int(decode_string(d)))
        for n, d in zip(nums, dens)
    ]
  return values


def load_values(
    obj: str | pathlib.Path | Iterable[Fraction],
) -> list[Fraction]:
  """Loads optimization parameter values.

  Args:
    obj: The source of the values. Can be a string path, a pathlib.Path, or an
      iterable of values (which can be Fraction objects, ints, floats, or
      strings representing fractions).

  Returns:
    A list of Fraction (gmpy2.mpq) objects.
  """
  if isinstance(obj, (str, pathlib.Path)):
    return load_fraction_vector(obj)
  loaded = []
  for v in obj:
    if isinstance(v, Fraction):
      loaded.append(v)
    else:
      loaded.append(Fraction(v))
  return loaded


class InputReader:
  """Sequential reader for the optimizable parameter vector."""

  def __init__(self, values: list[Fraction]):
    self.values = values
    self.position = 0

  def read(self, size: int) -> list[Fraction]:
    if size < 0:
      raise ValueError(f"Cannot read negative number of values: {size}")
    end = self.position + size
    if end > len(self.values):
      raise ValueError(
          f"Input vector ended early: need {size} values at position "
          f"{self.position}, but length is {len(self.values)}"
      )
    vals = self.values[self.position : end]
    self.position = end
    return vals

  def read_one(self) -> Fraction:
    return self.read(1)[0]

  def require_exhausted(self) -> None:
    if self.position != len(self.values):
      raise ValueError(
          f"Input vector has {len(self.values) - self.position} unused trailing"
          f" values after position {self.position}"
      )


@dataclasses.dataclass
class EvalContext:
  """Evaluation context containing global configuration and tree structures."""

  q: int
  max_level: int
  input_reader: InputReader
  terms: list[list["TermInfoBase"]]
  term_index: list[dict[tuple[int, tuple[int, ...], int], int]]


def _term_identifier(
    parent_id: int,
    shape: Sequence[int] | np.ndarray,
    region: int,
) -> tuple[int, tuple[int, ...], int]:
  """Node identity (parent term_id, shape, region): the appendix's T[u, r].

  Args:
    parent_id: The ID of the parent term.
    shape: The shape of the node.
    region: The region index.

  Returns:
    A tuple containing (parent_id, shape as a tuple, region).
  """
  return (int(parent_id), tuple(int(s) for s in shape), int(region))


def create_context(
    q: int,
    max_level: int,
    values: list[Fraction],
) -> EvalContext:
  """Create the global evaluation context.

  Initializes the empty node structures and the reader for the optimizable
  parameters.

  Args:
    q: The dimension factor.
    max_level: The maximum verification level.
    values: The optimizable parameters sequence.

  Returns:
    The initialized EvalContext.
  """
  terms: list[list["TermInfoBase"]] = [[] for _ in range(max_level + 1)]
  term_index: list[dict[tuple[int, tuple[int, ...], int], int]] = [
      {} for _ in range(max_level + 1)
  ]
  return EvalContext(
      q=q,
      max_level=max_level,
      input_reader=InputReader(values),
      terms=terms,
      term_index=term_index,
  )


def require_distribution(values: Sequence[Fraction]) -> None:
  """Verify that `values` is a vector summing to 1 and its entries are positive.

  Zero entries are not accepted, because the logarithm of zero is undefined,
  and all distributions modeled here must have positive mass.

  Args:
    values: The distribution vector.

  Raises:
    ValueError: If sum is not 1 or entries are not positive.
  """
  if sum(values) != ONE:
    raise ValueError(f"distribution must sum exactly to 1, got {sum(values)}")
  for idx, value in enumerate(values):
    if value <= ZERO:
      raise ValueError(
          f"distribution entry {idx} must be positive, got {value}"
      )


def require_same_marginals(
    reference: Sequence[Fraction],
    candidate: Sequence[Fraction],
    shapes: np.ndarray,
    sum_shape: int,
) -> None:
  """Verify the marginal distributions of reference and candidate are the same.

  Args:
    reference: The reference distribution.
    candidate: The candidate distribution.
    shapes: Matrix of size (num_shapes, 3).
    sum_shape: Equal to 2**level.

  Raises:
    ValueError: If length mismatch or marginal distributions do not match.
  """
  if len(reference) != shapes.shape[0] or len(candidate) != shapes.shape[0]:
    raise ValueError("distribution length does not match the number of shapes")
  reference_margs = marginal_dist(reference, shapes, sum_shape)
  candidate_margs = marginal_dist(candidate, shapes, sum_shape)
  if reference_margs != candidate_margs:
    raise ValueError("marginal distributions mismatch")
class TermInfoBase:
  """Base class for all term info types.

  Each term info class corresponds to a non-root node in the tree.
  """

  def __init__(self, context: EvalContext):
    self.ctx: EvalContext = context
    self.level: int = 0
    self.power: int = 0
    self.sum_shape: int = 0
    self.sum_half: int = 0
    self.term_id: int = 0
    self.shape: np.ndarray = np.array([])
    self.mass: Fraction = ZERO
    self.complete_split: list[np.ndarray] = []
    self.mat_size_contribution: list[Fraction] = []
    self.num_retain_contribution: list[list[Fraction]] = []
    self.left_ptr: list[list["TermInfoBase"]] = []
    self.right_ptr: list[list["TermInfoBase"]] = []

  def _build_common(
      self,
      level: int,
      term_id: int,
      shape: Sequence[int],
  ) -> None:
    self.level = level
    self.power = 1 << (level - 1)
    self.sum_shape = 1 << level
    self.sum_half = self.sum_shape // 2
    self.term_id = term_id
    self.shape = np.array(shape, dtype=int)

  def evaluate_init(self):
    """Initialize evaluation arrays and check inputs."""
    pass

  def evaluate_pre(self):
    """Propagate masses downward."""
    pass

  def evaluate_post(self):
    """Aggregate sizes and entropies upward."""
    pass

  def build(
      self,
      level: int,
      term_id: int,
      shape: Sequence[int],
  ) -> None:
    """Build the configuration for this node."""
    pass


class TermInfoLv2(TermInfoBase):
  """A level-2 node.

  Shapes are reduced to a standard form (022, 112, 013, 031, or 004) by cyclic
  shifts; derived vectors are transported back to actual coordinates by
  ``rot3``.
  """

  def __init__(self, context: EvalContext):
    super().__init__(context)
    self.shape_type: str = ""
    self.rotate_num: int = 0
    self.split_0: Fraction | None = None

  def build(
      self,
      level: int,
      term_id: int,
      shape: Sequence[int],
  ) -> None:
    """Build the configuration for this TermInfoLv2."""
    if level != 2:
      raise ValueError("TermInfoLv2 only supports level 2")
    self._build_common(level, term_id, shape)

    if max(shape) == 2 and min(shape) == 0:
      self.shape_type = "022"
      standard = [0, 2, 2]
    elif max(shape) == 2:
      self.shape_type = "112"
      standard = [1, 1, 2]
    elif max(shape) == 3:
      if (
          np.array_equal(shape, [0, 1, 3])
          or np.array_equal(shape, [1, 3, 0])
          or np.array_equal(shape, [3, 0, 1])
      ):
        self.shape_type = "013"
        standard = [0, 1, 3]
      else:
        self.shape_type = "031"
        standard = [0, 3, 1]
    else:
      self.shape_type = "004"
      standard = [0, 0, 4]
    self.rotate_num = 0
    tmp_shape = list(shape)
    while tmp_shape != standard:
      tmp_shape = [tmp_shape[2], tmp_shape[0], tmp_shape[1]]
      self.rotate_num += 1
    assert self.rotate_num < 3, "Shape could not be reduced to standard form."

    if self.shape_type in ("112", "022"):
      # Optimizable parameter:
      # split_0 := mu_T for shape 112;
      # split_0 := beta_{T,W_1}(0,2) = \beta_{T,W_1}(2,0) for 022
      self.split_0 = self.ctx.input_reader.read_one()
    else:
      self.split_0 = None

  def evaluate_init(self):
    """Initialize evaluate phase."""
    if self.shape_type in ("112", "022"):
      if self.split_0 <= ZERO or self.split_0 >= Fraction(1, 2):
        raise ValueError(f"level-2 split value out of (0, 1/2): {self.split_0}")
    self.mass = ZERO

  def evaluate_post(self):
    """Aggregate evaluations up the tree."""
    if self.mass == ZERO:
      self.num_retain_contribution = [
          [ZERO] * 3 for _ in range(NUM_REGIONS)
      ]
      self.mat_size_contribution = [ZERO] * 3
      self.complete_split = [
          np.array([], dtype=object) for _ in range(3)
      ]
      return
    q = Fraction(self.ctx.q)
    complete_split: list[list[Fraction]] = [[], [], []]  # beta_{T,X/Y/Z}
    s0 = self.split_0 if self.split_0 is not None else ZERO

    # Calculate the derived quantities; masses will be multiplied at the end.
    if self.shape_type == "022":
      base_num_retain = [ZERO] * 3
      dist_022 = [s0, s0, ONE - 2 * s0]
      inner_h = entropy_lower_bound(dist_022)
      log_q = log_lower_bound(q)
      inner_prod_size = floor_rational(inner_h + 2 * log_q * (ONE - 2 * s0))
      # M_{T,W_0} (W_0 = X in standard coordinates); the factor 2 counts
      # the entries equal to 1 in support point (1,1).
      self.mat_size_contribution = [inner_prod_size, ZERO, ZERO]

      # Construct the complete split distributions.
      cs0 = [ZERO] * 9
      cs0[encode_csd([0, 0])] = ONE
      cs1 = [ZERO] * 9
      cs1[encode_csd([0, 2])] = s0
      cs1[encode_csd([2, 0])] = s0
      cs1[encode_csd([1, 1])] = ONE - 2 * s0
      complete_split[0] = cs0
      complete_split[1] = cs1
      complete_split[2] = list(cs1)

    elif self.shape_type == "112":
      log_2 = log_lower_bound(Fraction(2))
      dist_112 = [s0, s0, ONE - 2 * s0]
      h_112 = entropy_lower_bound(dist_112)
      # (E_{T,X}, E_{T,Y}, E_{T,Z}) / m_T = (1, 1, H(mu, mu, 1-2mu)),
      # with natural logs.
      base_num_retain = [log_2, log_2, h_112]

      log_q = log_lower_bound(q)
      ms0 = floor_rational(2 * s0 * log_q)
      ms1 = floor_rational((ONE - 2 * s0) * log_q)
      # (M_{T,X}, M_{T,Y}, M_{T,Z}) / m_T =
      # ((1-2mu) log q, (1-2mu) log q, 2mu log q).
      self.mat_size_contribution = [ms1, ms1, ms0]

      # Construct complete split distributions.
      half = Fraction(1, 2)
      cs0 = [ZERO] * 9
      cs0[encode_csd([0, 1])] = half
      cs0[encode_csd([1, 0])] = half
      complete_split[0] = cs0
      complete_split[1] = list(cs0)
      cs2 = [ZERO] * 9
      cs2[encode_csd([0, 2])] = s0
      cs2[encode_csd([2, 0])] = s0
      cs2[encode_csd([1, 1])] = ONE - 2 * s0
      complete_split[2] = cs2

    elif self.shape_type in ("013", "031"):
      # Always set the symmetric distribution. For 013,
      # beta_{T,W_1}(0,1) = beta_{T,W_1}(1,0) = 1/2.
      # Similarly for 031.
      base_num_retain = [ZERO] * 3
      log_2 = log_lower_bound(Fraction(2))
      log_q = log_lower_bound(q)
      inner_prod_size = floor_rational(log_2 + log_q)
      # M_{T,W_0} (W_0 = X in standard coordinates): H(1/2, 1/2) + log q
      # (each support point has one entry equal to 1).
      self.mat_size_contribution = [inner_prod_size, ZERO, ZERO]

      # Complete split distributions.
      half = Fraction(1, 2)
      cs0 = [ZERO] * 9
      cs0[encode_csd([0, 0])] = ONE
      cs1 = [ZERO] * 9
      cs2 = [ZERO] * 9
      if self.shape_type == "013":
        cs1[encode_csd([0, 1])] = half
        cs1[encode_csd([1, 0])] = half
        cs2[encode_csd([1, 2])] = half
        cs2[encode_csd([2, 1])] = half
      else:
        cs1[encode_csd([1, 2])] = half
        cs1[encode_csd([2, 1])] = half
        cs2[encode_csd([0, 1])] = half
        cs2[encode_csd([1, 0])] = half
      complete_split[0] = cs0
      complete_split[1] = cs1
      complete_split[2] = cs2

    else:  # 004
      base_num_retain = [ZERO] * 3
      self.mat_size_contribution = [ZERO] * 3
      cs0 = [ZERO] * 9
      cs0[encode_csd([0, 0])] = ONE
      cs1 = list(cs0)
      cs2 = [ZERO] * 9
      cs2[encode_csd([2, 2])] = ONE
      complete_split[0] = cs0
      complete_split[1] = cs1
      complete_split[2] = cs2

    # To get actual derived quantities: scale by the mass m_T,
    # and rotate `rotate_num` times to the left.

    # Contribution to retained exponents.
    base_contrib = [
        floor_rational(v * self.mass)
        for v in rot3(base_num_retain, self.rotate_num)
    ]
    # Note: Level-2 nodes have no sub-region structure, so the contribution
    # is identical across all 6 regions.
    self.num_retain_contribution = [
        list(base_contrib) for _ in range(NUM_REGIONS)
    ]

    # Local matrix sizes.
    self.mat_size_contribution = [
        floor_rational(v * self.mass)
        for v in rot3(self.mat_size_contribution, self.rotate_num)
    ]
    rotated = rot3(complete_split, self.rotate_num)
    self.complete_split = [np.array(x, dtype=object) for x in rotated]


class TermInfoZero(TermInfoBase):
  """A zero-shape node at level >= 3."""

  def __init__(self, context: EvalContext):
    super().__init__(context)
    self.zero_dim: int = 0
    self.nonzero_dim_1: int = 0
    self.nonzero_dim_2: int = 0
    self.base_mat_size: list[Fraction] = []

  def build(
      self,
      level: int,
      term_id: int,
      shape: Sequence[int],
  ) -> None:
    """Build configuration and splits for this TermInfoZero."""
    self._build_common(level, term_id, shape)
    reader = self.ctx.input_reader

    # zero_dim / nonzero_dim_1 / nonzero_dim_2 are the appendix's
    # W_0 / W_1 / W_2.
    self.zero_dim = next(i for i, v in enumerate(shape) if v == 0)
    self.nonzero_dim_1 = next(i for i, v in enumerate(shape) if v != 0)
    self.nonzero_dim_2 = 3 - self.zero_dim - self.nonzero_dim_1

    self.base_mat_size = [ZERO] * 3
    self.base_mat_size[self.zero_dim] = ONE

    size = 3**self.power
    complete_split_lists = [[ZERO] * size for _ in range(3)]
    # beta_{W_0} is trivial (point mass on 0 vector).
    complete_split_lists[self.zero_dim][0] = ONE
    # Build beta_{W_1} and beta_{W_2}.
    for idx in range(size):
      arr = decode_csd(idx, self.power)
      if np.sum(arr) != self.shape[self.nonzero_dim_1]:
        continue
      value = reader.read_one()
      if value <= ZERO:
        raise ValueError("active zero-term split entry is not positive")
      complete_split_lists[self.nonzero_dim_1][idx] = value
      # beta_{W2} is not a free parameter: it is derived from beta_{W1} by
      # flipping (L -> 2 - L), as in the appendix.
      arr_opposite = np.ones(self.power, dtype=int) * 2 - arr
      opp_idx = encode_csd(arr_opposite)
      complete_split_lists[self.nonzero_dim_2][opp_idx] = value

    self.complete_split = [
        np.array(x, dtype=object) for x in complete_split_lists
    ]
    # Zero-shape nodes contribute to only `mat_size` but not `num_retain`.
    self.num_retain_contribution = [
        [ZERO] * 3 for _ in range(NUM_REGIONS)
    ]

  def evaluate_init(self):
    """Initialize evaluate phase."""
    self.mass = ZERO
    for t in range(3):
      vals = self.complete_split[t]
      if sum(vals) != ONE:
        raise ValueError(
            f"zero-term split distribution {t} sum mismatch: {sum(vals)}"
        )

  def evaluate_post(self):
    """Aggregate evaluations up the tree."""
    # M_{T,W_0} = m_T * (H(beta_{T,W_1})
    #   + sum_L beta_{T,W_1}(L) * |{p : L_p = 1}| * log q).
    q = Fraction(self.ctx.q)
    cs = self.complete_split[self.nonzero_dim_1]
    inner_prod_size = entropy_lower_bound(cs)
    log_q = log_lower_bound(q)
    nonzero_indices = np.nonzero(cs != ZERO)[0]
    for i in nonzero_indices:
      arr = decode_csd(i, self.power)
      num_ones = int(np.sum(arr == 1))
      inner_prod_size += floor_rational(cs[i] * Fraction(num_ones) * log_q)
    self.mat_size_contribution = [
        floor_rational(self.mass * inner_prod_size * self.base_mat_size[t])
        for t in range(3)
    ]


class TermInfo(TermInfoBase):
  """A positive-shape node at level >= 3."""

  def __init__(self, context: EvalContext):
    super().__init__(context)
    self.splits: np.ndarray = np.array([])
    self.num_split: int = 0
    self.lam_margin_low: np.ndarray = np.array([])
    self.lam_margin_high: np.ndarray = np.array([])
    self.split_dist: list[list[Fraction]] = []
    self.split_dist_max: list[list[Fraction]] = []
    self.region_prop: list[Fraction] = []
    self.lam_margin: list[list[list[Fraction]]] = []
    self.lam_sum: list[Fraction] = []
    self.marginal_dists: list[
        tuple[list[Fraction], list[Fraction], list[Fraction]]
    ] = []

  def _find_or_create_child(
      self, shape: Sequence[int] | np.ndarray, region: int
  ) -> "TermInfoBase":
    """Return the child node T[u, r] at level - 1, building it if needed."""
    ctx = self.ctx
    level = self.level - 1
    shape_tuple = tuple(int(s) for s in shape)
    identifier = _term_identifier(self.term_id, shape_tuple, region)
    idx = ctx.term_index[level].get(identifier, -1)
    if idx != -1:
      return ctx.terms[level][idx]
    term = self.create_instance(level, shape_tuple)(ctx)
    idx = len(ctx.terms[level])
    ctx.terms[level].append(term)
    ctx.term_index[level][identifier] = idx
    term.build(level, idx, shape_tuple)
    return term

  def build(
      self,
      level: int,
      term_id: int,
      shape: Sequence[int],
  ) -> None:
    """Build the configuration for this TermInfo."""
    self._build_common(level, term_id, shape)
    self.splits = prepare_splits(self.shape)
    self.num_split = self.splits.shape[0]

    # Compute the support of lambda_W in the H^max certificate.
    self.lam_margin_low = self.splits[:, :3].min(axis=0)
    self.lam_margin_high = self.splits[:, :3].max(axis=0)

    self._read_parameters()

    # Pointers to child nodes.
    self.left_ptr = [[] for _ in range(NUM_REGIONS)]
    self.right_ptr = [[] for _ in range(NUM_REGIONS)]
    for r in range(NUM_REGIONS):
      for i in range(self.num_split):
        # Children are labeled by (parent term_id, u, r), the appendix's
        # T[u, r]; shape u as the left half of split (u, s - u) and as the
        # right half of split (s - u, u) is the same node.
        self.left_ptr[r].append(
            self._find_or_create_child(self.splits[i, :3], r)
        )
        self.right_ptr[r].append(
            self._find_or_create_child(self.splits[i, 3:], r)
        )

  def _read_parameters(self):
    """Load parameter blocks from the configuration."""
    # split_dist[r] is alpha_T^{(r)} (over splits); region_prop is A_T.
    # split_dist_max[r], lam_margin[r], lam_sum[r] are the certificates
    # (y, lambda_W, lambda_0) bounding H^max_{Split(s_T)}.
    reader = self.ctx.input_reader
    self.split_dist = [[] for _ in range(NUM_REGIONS)]
    self.split_dist_max = [[] for _ in range(NUM_REGIONS)]
    for r in range(NUM_REGIONS):
      self.split_dist[r] = reader.read(self.num_split)
      self.split_dist_max[r] = reader.read(self.num_split)
    self.region_prop = reader.read(NUM_REGIONS)
    self.lam_margin = [
        [[] for _ in range(3)] for _ in range(NUM_REGIONS)
    ]
    self.lam_sum = [ZERO for _ in range(NUM_REGIONS)]
    for r in range(NUM_REGIONS):
      for t in range(3):
        size = int(self.lam_margin_high[t] - self.lam_margin_low[t] + 1)
        self.lam_margin[r][t] = reader.read(size)
      self.lam_sum[r] = reader.read_one()

  def evaluate_init(self):
    """Initialize evaluate phase."""
    require_distribution(self.region_prop)
    for r in range(NUM_REGIONS):
      require_distribution(self.split_dist[r])
      require_distribution(self.split_dist_max[r])
      require_same_marginals(
          self.split_dist[r],
          self.split_dist_max[r],
          self.splits[:, :3],
          self.sum_half,
      )
    self.mass = ZERO

  def evaluate_pre(self):
    """Propagate masses downward."""
    # Propagate mass to child nodes.
    # m_{T[u,r]} = m_T * A_T^{(r)} * (alpha(u) + alpha(s_T - u)):
    # child u receives alpha(u) from split (u, s_T - u) and alpha(s_T - u) from
    # split (s_T - u, u).
    for i in range(self.num_split):
      for r in range(NUM_REGIONS):
        frac = self.mass * self.split_dist[r][i] * self.region_prop[r]
        self.left_ptr[r][i].mass += frac
        self.right_ptr[r][i].mass += frac

  def evaluate_post(self):
    """Aggregate evaluations up the tree."""
    self.marginal_dists = []
    for r in range(NUM_REGIONS):
      self.marginal_dists.append(
          marginal_dist(self.split_dist[r], self.splits[:, :3], self.sum_half)
      )

    # E_{T,X}^{(r)} = H((alpha^{(r)})_{pi_r(X)}) - P_{Split(s_T)}(alpha),
    # mass-weighted; the penalty upper-bounds H^max by H(y) + 2*eps.
    e_x = []
    for r in range(NUM_REGIONS):
      dim_x = dims(r)[0]
      h_x = entropy_lower_bound(self.marginal_dists[r][dim_x])
      penalty = (
          entropy_upper_bound(self.split_dist_max[r])
          + 2 * self._get_max_derivative(r)
          - entropy_lower_bound(self.split_dist[r])
      )
      # Ceil penalty separately for a tighter bound on the floor.
      # Note: TermInfo floors the product instead of flooring before
      # multiplying by region_prop. This differs from GlobalStage, but both
      # are valid lower bounds.
      # Also note: The ceil_rational(penalty) is only applied to e_x because
      # the split only introduces communication overhead along the X dimension.
      e_x.append(
          floor_rational(
              self.mass * self.region_prop[r] * (h_x - ceil_rational(penalty))
          )
      )

    # Positive-shape nodes have no local matrix size.
    self.mat_size_contribution = [ZERO] * 3

    size = 3**self.power
    self.complete_split = [
        np.full(size, ZERO, dtype=object) for _ in range(3)
    ]
    # num_retain_contribution[r] is m_T A_T^{(r)} (E_{T,X}, E_{T,Y}, E_{T,Z})
    # in the sub-region-r coordinate order (pi_r(X), pi_r(Y), pi_r(Z)).
    self.num_retain_contribution = []

    for r in range(NUM_REGIONS):
      # rg_complete_split[t] is beta_{T,W}^{(r)} = sum_u alpha^{(r)}(u) *
      # (beta_{T[u,r],W} x beta_{T[s_T-u,r],W}).
      rg_complete_split = []
      for t in range(3):
        g = np.full(size, ZERO, dtype=object)
        for i in range(self.num_split):
          indices, values = concat_csd(
              self.left_ptr[r][i].complete_split[t],
              self.right_ptr[r][i].complete_split[t],
          )
          if len(indices) > 0:
            g[indices] += self.split_dist[r][i] * values

        rg_complete_split.append(g)

        # beta_{T,W} = sum_r A_T^{(r)} * beta_{T,W}^{(r)}.
        mask = g != ZERO
        if np.any(mask):
          self.complete_split[t][mask] += self.region_prop[r] * g[mask]

      # E_{T,Y/Z}^{(r)} = H(beta_{T,W}^{(r)}) - eta_{T,W}, mass-weighted.
      _, dim_y, dim_z = dims(r)
      e_y = (
          (entropy_lower_bound(rg_complete_split[dim_y]) - self._get_eta_y(r))
          * self.mass
          * self.region_prop[r]
      )
      e_z = (
          (entropy_lower_bound(rg_complete_split[dim_z]) - self._get_eta_z(r))
          * self.mass
          * self.region_prop[r]
      )
      self.num_retain_contribution.append(
          [e_x[r], floor_rational(e_y), floor_rational(e_z)]
      )

  def _get_eta(
      self,
      region: int,
      bucket_dim: int,
      single_entropy_cond: Callable[["TermInfoBase"], bool],
  ) -> Fraction:
    """Compute an upper bound on eta for evaluating E_Y or E_Z."""
    res = ZERO
    size = 3 ** (
        self.power // 2
    )  # children's complete-split distribution length
    sum_beta = [
        np.full(size, ZERO, dtype=object) for _ in range(self.sum_half + 1)
    ]
    sum_weight = [ZERO for _ in range(self.sum_half + 1)]
    used = [False] * (self.sum_half + 1)

    for i in range(self.num_split):
      for ptr in (self.left_ptr[region][i], self.right_ptr[region][i]):
        if ptr.shape[bucket_dim] == 0:
          continue
        if single_entropy_cond(ptr):
          res += self.split_dist[region][i] * entropy_upper_bound(
              ptr.complete_split[bucket_dim]
          )
        else:
          idx = ptr.shape[bucket_dim]
          beta = ptr.complete_split[bucket_dim]
          mask = beta != ZERO
          if np.any(mask):
            sum_beta[idx][mask] += self.split_dist[region][i] * beta[mask]
          sum_weight[idx] += self.split_dist[region][i]
          used[idx] = True

    for i in range(1, self.sum_half + 1):
      if used[i]:
        res += normalized_entropy_upper_bound(sum_beta[i], sum_weight[i])
    return res

  def _get_eta_y(self, region: int) -> Fraction:
    """Upper bound on the appendix's eta_{T,Y}^{(r)}."""
    _, dim_y, dim_z = dims(region)
    return self._get_eta(region, dim_y, lambda ptr: ptr.shape[dim_z] == 0)

  def _get_eta_z(self, region: int) -> Fraction:
    """Upper bound on the appendix's eta_{T,Z}^{(r)}."""
    _, _, dim_z = dims(region)
    return self._get_eta(region, dim_z, lambda ptr: np.min(ptr.shape) == 0)

  def _get_max_derivative(self, r: int) -> Fraction:
    """The certificate's eps over shapes u.

    Args:
      r: The region index.

    Returns:
      The maximum derivative bound,
      max_u |sum_W lambda_W(u_W) + lambda_0 - ln y(u)|.
    """
    return compute_max_derivative(
        self.num_split,
        self.splits[:, :3],
        self.lam_margin[r],
        self.lam_margin_low,
        self.lam_sum[r],
        self.split_dist_max[r],
    )

  @staticmethod
  def create_instance(level: int, shape: Sequence[int]) -> type[TermInfoBase]:
    if level == 2:
      return TermInfoLv2
    if min(shape) == 0:
      return TermInfoZero
    return TermInfo


class GlobalStage:
  """The root node G."""

  def __init__(self, context: EvalContext):
    self.ctx: EvalContext = context
    self.level: int = 0
    self.power: int = 0
    self.sum_shape: int = 0
    self.shapes: np.ndarray = np.array([])
    self.num_shape: int = 0
    self.term_ptr: list[list["TermInfoBase"]] = []
    self.region_prop: list[Fraction] = []
    self.dist: list[list[Fraction]] = []
    self.dist_max: list[list[Fraction]] = []
    self.lam_margin: list[list[list[Fraction]]] = []
    self.lam_sum: list[Fraction] = []
    self.marginal_dists: list[
        tuple[list[Fraction], list[Fraction], list[Fraction]]
    ] = []
    self.num_retain: list[list[Fraction]] = []

  def build(self) -> None:
    """Build root term and children for GlobalStage."""
    ctx = self.ctx
    self.level = ctx.max_level
    self.power = 1 << (self.level - 1)
    self.sum_shape = 1 << self.level
    self.shapes = prepare_shapes(self.level)
    self.num_shape = self.shapes.shape[0]
    self._read_parameters()

    self.term_ptr = [[] for _ in range(NUM_REGIONS)]
    for r in range(NUM_REGIONS):
      ptrs = []
      for i in range(self.num_shape):
        cur_id = len(ctx.terms[self.level])
        term_cls = TermInfo.create_instance(self.level, self.shapes[i])
        term = term_cls(ctx)
        ctx.terms[self.level].append(term)
        # Root children G[s, r] are labeled by (0, s, r); the root has no
        # term_id, so 0 stands in for it.
        # Note: GlobalStage creates new term instances for all children
        # (not deduplicated across regions like TermInfo)
        # but uniqueness is preserved because regions are distinct.
        ctx.term_index[self.level][
            _term_identifier(0, self.shapes[i], r)
        ] = cur_id
        term.build(self.level, cur_id, self.shapes[i])
        ptrs.append(term)
      self.term_ptr[r] = ptrs

  def _read_parameters(self):
    """Load parameter blocks from the configuration."""
    # dist[r] is alpha_G^{(r)}; region_prop is A_G.
    # dist_max[r], lam_margin[r], lam_sum[r] are the certificate
    # (y, lambda_W, lambda_0) for bounding H^max_{Shapes}.
    reader = self.ctx.input_reader
    self.region_prop = reader.read(NUM_REGIONS)
    self.dist = [[] for _ in range(NUM_REGIONS)]
    self.dist_max = [[] for _ in range(NUM_REGIONS)]
    self.lam_margin = [
        [[] for _ in range(3)] for _ in range(NUM_REGIONS)
    ]
    self.lam_sum = [ZERO for _ in range(NUM_REGIONS)]
    for r in range(NUM_REGIONS):
      self.dist[r] = reader.read(self.num_shape)
      self.dist_max[r] = reader.read(self.num_shape)
    for r in range(NUM_REGIONS):
      for t in range(3):
        self.lam_margin[r][t] = reader.read(self.sum_shape + 1)
      self.lam_sum[r] = reader.read_one()

  def evaluate_init(self):
    """Initialize evaluate phase."""
    require_distribution(self.region_prop)
    for r in range(NUM_REGIONS):
      require_distribution(self.dist[r])
      require_distribution(self.dist_max[r])
      require_same_marginals(
          self.dist[r],
          self.dist_max[r],
          self.shapes,
          self.sum_shape,
      )

  def evaluate_pre(self):
    """Propagate masses downward."""
    # Propagate mass to children: m_{G[s,r]} = A_G^{(r)} * alpha_G^{(r)}(s).
    for r in range(NUM_REGIONS):
      for i in range(self.num_shape):
        self.term_ptr[r][i].mass = self.dist[r][i] * self.region_prop[r]

  def evaluate_post(self):
    """Aggregate evaluations up the tree."""
    self.marginal_dists = []
    for r in range(NUM_REGIONS):
      self.marginal_dists.append(
          marginal_dist(self.dist[r], self.shapes, self.sum_shape)
      )

    # E_{G,X}^{(r)} = H((alpha_G^{(r)})_{pi_r(X)}) - P_{Shapes_{l*}}(alpha_G),
    # weighted by region_prop; we upper bound H^max by H(y) + 2*eps.
    e_x = []
    for r in range(NUM_REGIONS):
      dim_x = dims(r)[0]
      h_x = entropy_lower_bound(self.marginal_dists[r][dim_x])
      penalty = (
          entropy_upper_bound(self.dist_max[r])
          + 2 * self._get_max_derivative(r)
          - entropy_lower_bound(self.dist[r])
      )
      # The root has implicit mass 1, so we omit it here.
      # Note: floor_rational is applied before multiplying by region_prop.
      # This differs from TermInfo which floors the product, but both are valid
      # lower bounds.
      e_x.append(
          floor_rational(h_x - ceil_rational(penalty)) * self.region_prop[r]
      )

    size = 3**self.power
    # num_retain[r] is A_G^{(r)} (E_{G,X}, E_{G,Y}, E_{G,Z}) in the
    # region-r coordinate order (pi_r(X), pi_r(Y), pi_r(Z)).
    self.num_retain = []

    for r in range(NUM_REGIONS):
      # rg_complete_split[t] is
      # bar-beta_{G,W,*,*,*}^{(r)} = sum_s alpha_G^{(r)}(s) * beta_{G[s,r],W}.
      rg_complete_split = []
      for t in range(3):
        csd = np.full(size, ZERO, dtype=object)
        for i in range(self.num_shape):
          ptr = self.term_ptr[r][i]
          arr = ptr.complete_split[t]
          mask = arr != ZERO
          if np.any(mask):
            csd[mask] += self.dist[r][i] * arr[mask]
        rg_complete_split.append(csd)

      # E_{G,Y/Z}^{(r)} = H(bar-beta_{G,W,*,*,*}^{(r)}) - eta_{G,W},
      # A_G-weighted.
      _, dim_y, dim_z = dims(r)
      e_y = (
          entropy_lower_bound(rg_complete_split[dim_y]) - self._get_eta_y(r)
      ) * self.region_prop[r]
      e_z = (
          entropy_lower_bound(rg_complete_split[dim_z]) - self._get_eta_z(r)
      ) * self.region_prop[r]
      self.num_retain.append([e_x[r], floor_rational(e_y), floor_rational(e_z)])

  def _get_eta(
      self,
      region: int,
      bucket_dim: int,
      single_entropy_cond: Callable[["TermInfoBase"], bool],
  ) -> Fraction:
    """Compute an upper bound on eta for evaluating E_Y or E_Z."""
    result = ZERO
    size = 3**self.power
    sum_beta = [
        np.full(size, ZERO, dtype=object) for _ in range(self.sum_shape + 1)
    ]
    sum_weight = [ZERO for _ in range(self.sum_shape + 1)]
    used = [False] * (self.sum_shape + 1)

    for i in range(self.num_shape):
      ptr = self.term_ptr[region][i]
      if ptr.shape[bucket_dim] == 0:
        continue
      if single_entropy_cond(ptr):
        result += self.dist[region][i] * entropy_upper_bound(
            ptr.complete_split[bucket_dim]
        )
      else:
        idx = ptr.shape[bucket_dim]
        beta = ptr.complete_split[bucket_dim]
        mask = beta != ZERO
        if np.any(mask):
          sum_beta[idx][mask] += self.dist[region][i] * beta[mask]
        sum_weight[idx] += self.dist[region][i]
        used[idx] = True

    for i in range(1, self.sum_shape + 1):
      if used[i]:
        result += normalized_entropy_upper_bound(sum_beta[i], sum_weight[i])
    return result

  def _get_eta_y(self, region: int) -> Fraction:
    """Upper bound on the appendix's eta_{G,Y}^{(r)}."""
    _, dim_y, dim_z = dims(region)
    return self._get_eta(region, dim_y, lambda ptr: ptr.shape[dim_z] == 0)

  def _get_eta_z(self, region: int) -> Fraction:
    """Upper bound on the appendix's eta_{G,Z}^{(r)}."""
    _, _, dim_z = dims(region)
    return self._get_eta(region, dim_z, lambda ptr: np.min(ptr.shape) == 0)

  def _get_max_derivative(self, r: int) -> Fraction:
    """The certificate's epsilon over shapes s.

    Args:
      r: The region index.

    Returns:
      The maximum derivative bound,
      max_s |sum_W lambda_W(s_W) + lambda_0 - ln y(s)|.
    """
    return compute_max_derivative(
        self.num_shape,
        self.shapes,
        self.lam_margin[r],
        (0, 0, 0),
        self.lam_sum[r],
        self.dist_max[r],
    )


class Workspace:
  """Builds whole tree from input vector and computes omega upper bound."""

  def __init__(self, context: EvalContext):
    self.ctx = context

  def build(self, max_level: int) -> None:
    self.ctx.max_level = max_level
    self.globstage = GlobalStage(self.ctx)
    self.globstage.build()

  def evaluate(self):
    """Evaluate all stages of optimization tree, compute omega upper bound."""
    ctx = self.ctx
    max_level = ctx.max_level
    self.num_retain_comp = [ZERO] * (max_level + 1)

    self.globstage.evaluate_init()
    for level in range(2, max_level + 1):
      for term in ctx.terms[level]:
        term.evaluate_init()
    # Propagate masses down the tree
    self.globstage.evaluate_pre()
    for level in range(max_level, 1, -1):
      for term in ctx.terms[level]:
        term.evaluate_pre()
    # Compute other derived quantities up the tree
    for level in range(2, max_level + 1):
      for term in ctx.terms[level]:
        term.evaluate_post()
      if level > 2:
        # Free memory: this level has consumed its children's vectors.
        for term in ctx.terms[level - 1]:
          del term.complete_split
          del term.left_ptr
          del term.right_ptr

    self.globstage.evaluate_post()
    for term in ctx.terms[max_level]:
      del term.complete_split
      del term.left_ptr
      del term.right_ptr

    # num_retain_comp[l] is E_l = sum_r min_W sum_T m_T A_T^{(r)} E_{T,W}^{(r)}.
    for level in range(3, max_level + 1):
      e_level = ZERO
      for r in range(NUM_REGIONS):
        total = [ZERO] * 3
        for term in ctx.terms[level]:
          for w in range(3):
            total[w] += term.num_retain_contribution[r][w]
        e_level += min(total)
      self.num_retain_comp[level] = e_level

    # E_2 (level 2 has no sub-region structure).
    total = [ZERO] * 3
    for term in ctx.terms[2]:
      for t in range(3):
        total[t] += term.num_retain_contribution[0][t]
    self.num_retain_comp[2] = min(total)

    # num_retain_glob is E_G = sum_r min_W A_G^{(r)} E_{G,W}^{(r)}.
    self.num_retain_glob = sum(
        min(self.globstage.num_retain[r]) for r in range(NUM_REGIONS)
    )

    # single_mat_size is M_total: sum of local matrix sizes over the
    # leaves, minimized over coordinates.
    mat_size = [ZERO] * 3
    for level in range(2, max_level + 1):
      for term in ctx.terms[level]:
        for t in range(3):
          mat_size[t] += term.mat_size_contribution[t]
    self.single_mat_size = floor_rational(
        min(mat_size[0], mat_size[1], mat_size[2])
    )

    # num_mat is E_total = E_G + E_2 + sum_l E_l.
    self.num_mat = self.num_retain_glob + sum(
        self.num_retain_comp[2 : max_level + 1]
    )

    # The final constraint: E_total + M_total * Omega >= 2^{l*-1} log(q+2);
    # self.omega is the smallest Omega satisfying this constraint.
    q_plus_2 = Fraction(ctx.q + 2)
    power = Fraction(2 ** (max_level - 1))
    target = ceil_rational(log_upper_bound(q_plus_2) * power)

    if self.single_mat_size > ZERO:
      if target < self.num_mat:
        raise ValueError(
            "Failed to bound: E_total exceeds target (yields negative"
            " numerator)."
        )
      self.omega = ceil_rational((target - self.num_mat) / self.single_mat_size)
    else:
      raise ValueError("Failed to bound: single_mat_size is not positive.")


def clear_caches():
  """Clear all module-level LRU caches.

  Call this function between verification runs to release memory
  and ensure timing accuracy for independent runs.
  """
  DECODE_CSD_CACHE.clear()
  ENTROPY_LOWER_BOUND_CACHE.clear()
  ENTROPY_UPPER_BOUND_CACHE.clear()
  floor_rational.cache_clear()
  ceil_rational.cache_clear()
  log_lower_bound.cache_clear()
  log_upper_bound.cache_clear()


def verify_omega(
    q: int,
    max_level: int,
    x_vector: Iterable[Fraction],
) -> Fraction:
  """Compute a valid omega upper bound from the given parameters.

  Args:
    q: The dimension factor.
    max_level: The maximum verification level (l* in the appendix).
    x_vector: An iterable of rational distribution fractions.

  Raises:
    ValueError: If the verified level
      omega was not computed accurately.
    Exception: If any math bounding error surfaces.

  Returns:
    The computed exact omega bound as a Fraction.
  """
  values = list(x_vector)

  clear_caches()

  ctx = create_context(q=q, max_level=max_level, values=values)
  workspace = Workspace(ctx)
  workspace.build(max_level=max_level)
  ctx.input_reader.require_exhausted()
  workspace.evaluate()

  result = workspace.omega

  # Free memory.
  ctx.terms.clear()

  return result


def verify_from_dict(level):
  print(f'--- Running Verification on Level {level} ---')
  start_time = time.time()
  lvl_data = LEVEL_DATA[level]
  nums = lvl_data['nums']
  dens = lvl_data['dens']
  x_vector = [gmpy2.mpq(int(n), int(d)) for n, d in zip(nums, dens)]

  omega_val = verify_omega(q=5, max_level=level, x_vector=x_vector)
  duration = time.time() - start_time

  a = int(omega_val.numerator)
  b = int(omega_val.denominator)
  d_a = decimal.Decimal(a)
  d_b = decimal.Decimal(b)

  orig_prec = decimal.getcontext().prec
  try:
    decimal.getcontext().prec = DECIMAL_PRECISION
    val = d_a / d_b
    c_dec = val.quantize(decimal.Decimal('1e-6'), rounding=decimal.ROUND_CEILING)
  finally:
    decimal.getcontext().prec = orig_prec

  print(f'Max-level {level} omega upper bound: {a}/{b}, which is <= {c_dec}')
  print(f'Max-level {level} verification time: {duration:.2f} seconds\n')


In [ ]:
#@title Data
LEVEL_DATA = {
  2: {
    'nums': ['1627604166666667', '8138020833333333', '8138020833333333', '8138020833333333', '8138020833333333', '8138020833333333', '4942070172352783', '6376870749335611', '494914063325749', '7084417085747227', '3115603651275699', '6351383887164303', '966952583782509', '2489499609887472117', '209830489112761', '4952619838502661', '4851354471617231', '2354711800439369', '889405935061953', '6770858083027839', '3172296047348107', '4942070172352783', '1628353539269558669', '10140355938647018963', '56663132024249013', '3115603651275699', '326015929534883863', '19806902952621820979', '9953883460304030417', '53728809874595619', '5067367735380866813', '19875663837479857619', '9648613571355692083', '1825607940938238099', '1729235083323768429', '3172296047348107', '1246241460510349', '1771104271436821', '618642579157187', '3188435374667819', '4942070172352709', '6714575651608381', '2489499609887477577', '4834762918912519', '793922985895539', '4709423600878769', '2425677235808617', '2476309919251323', '1354171616605561', '1423049496099111', '6344592094696017', '1246241460510349', '56663132024249301', '5070177969323519611', '814176769634781493', '4942070172352709', '53728809874596019', '4976941730152025247', '9903451476310855019', '815039823837212107', '4824306785677875563', '9937831918739938939', '506736773538084969', '864617541661879397', '912803970469110683', '6344592094696017', '4942070172352563', '15878459717911', '4952619838502649', '444702967530981', '1268918418939273', '3188435374667883', '2417381459456263', '4851354471617191', '6770858083027939', '618642579157181', '497899921977501073', '2354711800439363', '1416883417149461', '3357287825804141', '6231207302551499', '4942070172352563', '1630079647674455713', '1013473547076168963', '365121588187652877', '1268918418939273', '1628353539269586783', '3961380590524346157', '19875663837479725569', '69169403332951447', '10140355938646964737', '9953883460304153699', '9648613571355655137', '11332626404849801', '53728809874595691', '6231207302551499', '1557801825637913', '6714575651608361', '2354711800439359', '6770858083027673', '3172296047347987', '1416883417149463', '2489499609887485503', '2425677235808617', '711524748049551', '4949140633257511', '2417381459456263', '1238154959625661', '1594217687333921', '10162214219463', '4942070172352887', '1557801825637913', '1679025308581111', '9648613571355655907', '1729235083323725597', '3172296047347987', '110670179734863', '9953883460304086553', '19875663837479859939', '1825607940938209251', '10140355938647052003', '19806902952621747939', '5067367735380851997', '1628353539269582877', '1630079647674432227', '4942070172352887', '6344592094696047', '1778811870123903', '19346171244151', '6351383887164283', '98841403447053', '3385429041513891', '4851354471617231', '4834762918912529', '6376870749335549', '941884720175751', '311187451235935843', '2474570316628741', '6714575651608333', '7084417085747163', '6231207302551727', '6344592094696047', '1825607940938227261', '1013473547076173831', '1630079647674407037', '98841403447053', '1729235083323761603', '19875663837479845309', '19806902952621752829', '325670707853909991', '1929722714271144857', '9953883460304096387', '10140355938646990269', '26864404937297653', '28331566012124331', '6231207302551727', '158614802367411', '3385429041513743', '235471180043939', '1342915130321687', '623120730255193', '889405935061953', '4851354471617237', '124474980494375209', '3542208542873601', '4952619838502537', '4834762918912517', '4949140633257547', '3175691943582093', '1594217687333889', '494207017235257', '158614802367411', '1729235083323727287', '9648613571355756809', '5372880987459541', '623120730255193', '1825607940938188873', '19875663837479846473', '398155338412167487', '28331566012124843', '5067367735380768343', '19806902952621685001', '10140355938647099977', '1630079647674381577', '13026828314156419', '494207017235257', '-1368259833982957', '-6744988676963737', '803539689727347', '516899716131597', '-822225417924819', '-1364392277972239', '-596182236774643', '6462641883021679', '5147764584909677', '-129709010179636', '-5346669109426691', '5864166973012463', '3492228406528309', '1235179097376523', '-1186785609194967', '-2591574412197547', '-1368259833982949', '-1686247169240569', '401769844863677', '2584498580657973', '-4111127089624167', '-5346669109426671', '2932083486506419', '6984456813056583', '617589548688281', '-4747142436779917', '-2728784555944497', '-5961822367748769', '3231320941510827', '5147764584909521', '-207534416287411', '-518314882439509', '-1091513822377791', '-1192364473549513', '1292528376604337', '102955291698191', '-8301376651496619', '-2736519667965921', '-3372494338481901', '64283175178187', '1033799432263217', '-4111127089624049', '-5346669109426669', '5864166973013369', '436528550816039', '4940716389506387', '-1186785609194997', '-323946801524693', '-5346669109426707', '2932083486506253', '1396891362611313', '4940716389506159', '-4747142436779851', '-547303933593177', '-6744988676961607', '6428317517818821', '5168997161315909', '-4111127089624173', '-341098069493061', '-1490455591937081', '3231320941510829', '1029552916981909', '-8301376651496507', '-518314882439509', '-5457569111888991', '-1490455591937197', '6462641883021643', '1286941146227353', '-415068832574821', '-534666910942667', '2932083486506611', '6984456813056613', '4940716389506189', '-148348201149373', '-5473039335931817', '-674498867696257', '1607079379454709', '646124645164497', '-8222254179248289', '-2591574412197549', '-167083409669582', '2932083486507237', '6984456813056629', '4940716389506031', '-2373571218389997', '-1364392277972251', '-5961822367748429', '1615660470755411', '5147764584909423', '-4150688325748187', '-2736519667965897', '-1686247169241023', '6428317517818739', '516899716131557', '-328890167169919', '-518314882439509', '8917946794778219', '555289142935733', '3627884442800247', '4840192699043977', '2048807599168423', '8204436028377343', '8917946794778203', '1813942221400131', '1110578285870781', '410221801418867', '4097615198336847', '605024087380497', '4840192699043981', '1388222857339743', '327809215866947', '1783589358955643', '453485555350033', '2051109007094337', '8204436028377337', '362788444280027', '4097615198336837', '891794679477823', '2776445714674333', '4840192699043989', '605024087380497', '8195230396673689', '694111428669549', '8204436028377359', '7255768885600497', '891794679477819', '8204436028377379', '4097615198336847', '7255768885600469', '4840192699043953', '2776445714689249', '8917946794778147'],
    'dens': ['9765625000000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '12500000000000000000', '390625000000000000', '4882812500000000', '390625000000000000', '6250000000000000000', '390625000000000000', '4882812500000000', '12500000000000000000', '12207031250000000', '48828125000000000', '24414062500000000', '24414062500000000', '48828125000000000', '390625000000000000', '6250000000000000000', '12500000000000000000', '100000000000000000000', '100000000000000000000', '3125000000000000000', '6250000000000000000', '20000000000000000000', '100000000000000000000', '50000000000000000000', '3125000000000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '6250000000000000000', '2500000000000000000', '97656250000000000', '6103515625000000', '195312500000000000', '12500000000000000000', '390625000000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '12207031250000000', '24414062500000000', '78125000000000000', '78125000000000000', '12500000000000000000', '2500000000000000000', '3125000000000000000', '50000000000000000000', '50000000000000000000', '12500000000000000000', '3125000000000000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '5000000000000000000', '50000000000000000000', '50000000000000000000', '12500000000000000000', '12500000000000000000', '976562500000000', '48828125000000000', '24414062500000000', '2500000000000000000', '195312500000000000', '12207031250000000', '24414062500000000', '390625000000000000', '6103515625000000', '2500000000000000000', '24414062500000000', '78125000000000000', '195312500000000000', '12500000000000000000', '12500000000000000000', '100000000000000000000', '10000000000000000000', '20000000000000000000', '2500000000000000000', '100000000000000000000', '20000000000000000000', '100000000000000000000', '4000000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '625000000000000000', '3125000000000000000', '12500000000000000000', '3125000000000000000', '390625000000000000', '24414062500000000', '390625000000000000', '6250000000000000000', '78125000000000000', '12500000000000000000', '12207031250000000', '39062500000000000', '48828125000000000', '12207031250000000', '12207031250000000', '97656250000000000', '625000000000000', '12500000000000000000', '3125000000000000000', '97656250000000000', '100000000000000000000', '100000000000000000000', '6250000000000000000', '6103515625000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '190734863281250', '390625000000000000', '250000000000000000', '195312500000000000', '24414062500000000', '24414062500000000', '390625000000000000', '9765625000000000', '1562500000000000000', '24414062500000000', '390625000000000000', '390625000000000000', '12500000000000000000', '12500000000000000000', '100000000000000000000', '10000000000000000000', '100000000000000000000', '250000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '20000000000000000000', '20000000000000000000', '50000000000000000000', '100000000000000000000', '1562500000000000000', '1562500000000000000', '12500000000000000000', '312500000000000000', '195312500000000000', '2441406250000000', '78125000000000000', '1250000000000000000', '48828125000000000', '24414062500000000', '625000000000000000', '195312500000000000', '48828125000000000', '24414062500000000', '48828125000000000', '195312500000000000', '97656250000000000', '1250000000000000000', '312500000000000000', '100000000000000000000', '100000000000000000000', '312500000000000000', '1250000000000000000', '100000000000000000000', '100000000000000000000', '2000000000000000000', '1562500000000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '800000000000000000', '1250000000000000000', '762939453125000', '195312500000000000', '762939453125000', '1220703125000000', '610351562500000', '762939453125000', '19531250000000000', '6103515625000000', '12207031250000000', '95367431640625', '3051757812500000', '97656250000000000', '3051757812500000', '3051757812500000', '762939453125000', '1525878906250000', '762939453125000', '48828125000000000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '48828125000000000', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '195312500000000000', '3051757812500000', '12207031250000000', '152587890625000', '305175781250000', '610351562500000', '39062500000000000', '1220703125000000', '244140625000000', '6103515625000000', '1525878906250000', '97656250000000000', '61035156250000', '2441406250000000', '3051757812500000', '3051757812500000', '97656250000000000', '381469726562500', '12207031250000000', '762939453125000', '190734863281250', '3051757812500000', '48828125000000000', '1220703125000000', '12207031250000000', '3051757812500000', '305175781250000', '195312500000000000', '6103515625000000', '12207031250000000', '3051757812500000', '190734863281250', '48828125000000000', '3051757812500000', '2441406250000000', '6103515625000000', '305175781250000', '3051757812500000', '48828125000000000', '6103515625000000', '3051757812500000', '305175781250000', '305175781250000', '48828125000000000', '6103515625000000', '12207031250000000', '95367431640625', '3051757812500000', '19531250000000000', '1525878906250000', '1525878906250000', '6103515625000000', '1525878906250000', '95367431640625', '48828125000000000', '6103515625000000', '12207031250000000', '1525878906250000', '762939453125000', '195312500000000000', '1525878906250000', '12207031250000000', '3051757812500000', '1525878906250000', '48828125000000000', '6103515625000000', '1220703125000000', '244140625000000', '305175781250000', '195312500000000000', '10000000000000000000', '195312500000000000', '97656250000000000', '97656250000000000', '195312500000000000', '195312500000000000', '97656250000000000', '20000000000000000000', '9765625000000000', '195312500000000000', '12207031250000000', '97656250000000000', '25000000000000000000', '15625000000000000', '39062500000000000', '24414062500000000', '48828125000000000', '195312500000000000', '19531250000000000', '195312500000000000', '19531250000000000', '50000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '12500000000000000000', '195312500000000000', '390625000000000000', '19531250000000000', '195312500000000000', '195312500000000000', '390625000000000000', '97656250000000000', '50000000000000000000', '195312500000000000'],
  },
  3: {
    'nums': ['254933073952891', '2023899318557749', '1672147357583243', '3903251199740827', '8111775364780771', '2073913701774463', '791052751551', '321061859727583', '6315565561315067', '1497004278300689', '4911242568223103', '7458134547768563', '3126501622719831', '1708487643325041', '3473168460023', '398593611207387', '4101154255582403', '7536498786958291', '568764283636547', '4577052408323789', '1531702589422189', '8325995413821357', '838279826407031', '6248395381900219', '3753860097207587', '7006468605867119', '3129313602549491077', '140809412158161', '7577610854135401', '6251945321298441', '7403878464352699', '56623297821199', '1527112224559223', '1525411014873741', '141361850558633', '736024164011229', '4909427944558589', '57060423672621', '7055512193720601', '90633357984457', '4774466582802807', '1883439847291207', '3837344063508523', '7622836019779031', '3678539390812031', '1601308613706347', '2089069850464603', '1255702695623741', '874654245162519', '108025462154869', '3621242355293', '791052751551', '639380626149557', '5551978419520976133', '156645589141552939759', '428544651292434083983', '158841069465877680751', '803921468955778579', '27339955189301977', '3473168460023', '3202464356187141', '536885176189369141', '983603625921326198003', '4687021260644177270131', '4711728082747446919411', '996631690625648332019', '7572780091338747221', '26820801548923671', '221528261798096733', '979459927454872713443', '25054745970196601537', '28202438899905110099', '75503049650088610427', '98646854882832028961', '2732147716276039443', '93715539310572755833', '518372285637165231499', '39467126844506837159', '197185717643712565993', '776369146304049538103', '231082637056488235579', '1284733352652616328669', '1566750354820710283009', '113456671555455011033', '778075507995414934567', '25124495827497676339', '478240010441724605309', '142640770085541491749', '496242228551811418037', '232587824723006013451', '17051401791491737919', '22905983252517912527', '8299104215969741497', '28112585457120361', '27530868699726711', '3621242355293', '554116718293', '85328796750417', '1248765688872139', '3724939003674827', '245489812501321', '7489830288144041', '6317573180613371', '401219834003213', '3161889044231', '836719248796927', '8320885022388893', '7657470524018553', '457780014534179', '142225760874933', '3768596929453477', '1640205344962859', '159294294638433', '3122271107879303', '3784631320056397', '439907280593887', '6258651973082557537', '1401221147661917', '7505391377733363', '1561082075654487', '7355679827909743', '4520827581748591', '3050858861594877', '3054252549915679', '905845389340031', '370064603789149', '59927097876873', '453112087835027', '70559394965301', '2283049111716459', '490923709271827', '1839729952908857', '381249714460573', '7677296779698143', '7523125068132227', '1256784084706531', '522616314397277', '6403372722697433', '1730309379685669', '875727656048053', '1450417296703', '554116718293', '13630112702485263', '8426951762403095149', '238030931990594480999', '642758751681984975431', '234977114569153020647', '333089743152030517', '3195418991043381', '3161889044231', '13410002758332289', '11339713542252970201', '99587718774876681029', '336629345607661816607', '13395266333361863543', '491938188889115735257', '225530533318796261', '3200225573750983', '4100487166317473693', '49253978102496092389', '113233876291402477361', '394825670677279361351', '112734889440847181933', '13991056971125930663', '8301605266872729241', '115682888264314878631', '1164008947447667004793', '19717241834741927611', '98664006557799237731', '466488332245955703697', '234153004309293790571', '62969373593139134843', '1167108844168995034633', '11348664912728576207', '2350858247216524192373', '642210213164179711979', '4661884099167794383', '248003055705921750473', '99829645819835725361', '47793619121973988303', '4159335438586016573', '11449818559914122221', '1217367682320875359', '2760644954242969', '2810178552607871', '1450417296703', '12630385428519', '63670250539341', '6240325157311631', '3699282278418087', '2454245246064377', '3769619693104341', '6404151357807867', '109512389622617', '906970472221', '802010127341499', '1637188510102193', '7503972817228919', '4529384698139747', '228198966619383', '7677566263897723', '8364025990932073', '1731224557101739', '6315362113343877', '1884034957013003', '218939150313177', '3054217659883133', '3527380128212939', '9531415999179', '314204575028023', '59913586053457', '182043744422693', '3129685885235877849', '610192442864773', '566278997932769', '1840608401008109', '4909842365146381', '2288549306231731', '6874299646802', '1130378118814509', '4795350071047997', '3731868384428561', '1531490666576501', '946604879148059', '1470435773733243', '78022254361681', '831894560964781', '6242935043487211', '426448998246801', '1672463009876301', '6918699333783', '12630385428519', '2558597427748567', '414788464511314379', '4181993148223237009', '11469727713546994201', '1708308204033235633', '426269236422707591', '14060738879071017', '906970472221', '12773225006589349', '11260669604102086603', '424947059721141007', '759164818128455383', '39167715088303291691', '41607514795552126991', '11449243661908010593', '13806643449437871', '8320311585319365529', '54638544752876568049', '901903416734126479069', '789318760256146322441', '453831916849303444469', '165402227496676829429', '8319620943097575187', '235009892410064848207', '2343923780598818498777', '131623383225364633507', '87636411694211769761', '466760117683278641011', '46643215467941385197', '642998422043921726011', '2356240736854056457241', '43141136678432612729', '2328186235737385508459', '629624376920861689741', '238195308515818677007', '498039013487076084281', '70391559187406271869', '1851708476871415313', '401273377260754741', '944523152452067467', '2733033878293583393', '6812901845767501', '6700134165686519', '6918699333783', '13880648012949', '335050159301393', '3124361903997179', '7356969373889401', '1194112421950523', '3677629761864603', '3139940457946091', '172961457285561', '7250875257603', '853809675716591', '1664564014727', '1894099611681061', '4523687267993193', '453083444166281', '190594687240273', '8359646608106543', '218843812952883', '2442416361941', '7657892815742407', '7039940598718979', '1220236022015057', '3527155671809281', '7675383491487487', '1281861422854681', '7462457670027889', '4576867973550147', '12517922511408282131', '6108791126075593', '4563757424598419', '3770390224448921', '4911378329517987', '4550772022126501', '7007255539614173', '4530636171803809', '4910302215310783', '467855158023377', '3768378058228487', '1876815936944737', '7403264112673551', '6315219516926449', '8201960588354261', '1249365684180377', '1604851717937891', '12749450425769', '12648916409101', '13880648012949', '26803945305702963', '8194486207541592689', '25677538087941305339', '209369875061923140017', '3102042746336859097', '1660627315591605629', '27549830072954491', '7250875257603', '27321977061339389', '22709301758436012259', '98637553086499690313', '465815506064627327369', '466768558674056312777', '33087727415437153667', '22912450556934347963', '28136011150704293', '2410998091913983021', '996488000181530508623', '56623681270588438877', '49293295323577449733', '3544855406577024349', '332846526336219457949', '2437220542117868741', '476560856676218483249', '4711866991583976913999', '263230299800749244983', '98673356153318706593', '4699598279727878025431', '53161750439672792993', '1285942821328346114993', '937506753499189390019', '56380839674342490407', '933156456589920612139', '428349555865662179939', '469961125135276293809', '983552830110785707343', '979365946037626302551', '156225147795582990307', '16654653939242003899', '11271756066180923057', '16615488846774965347', '3195475840972873', '3201590201345159', '12648916409101', '14499993848767', '875455371092723', '400174468967249', '1507874486654343', '4908893039425743', '7401174355265729', '1248942810980047', '99569372617907', '12644491151999', '432474580790791', '334412020278821', '1919035196416439', '2282094188586401', '8846707177741', '7505993251503967', '2050445170047541', '1604998512337601', '6281956770001027', '1524817135333553', '7054384039849101', '610830440720171', '3503063490284171', '60294263360513', '6317562394706463', '7358631105262633', '18121924198921', '6101921937447009', '12518496292681227279', '4550628903124851', '7489334947236537', '4787703239233927', '282654411532927', '703956946316471', '2288759693080567', '4910926305696777', '7353566311377261', '302989139705653', '7657865036087713', '7464536301927657', '780573613554307', '4160525177623579', '3122279538504737', '209206942116731', '1706619807235839', '13854687489401', '14499993848767', '224931440481861', '17050247801028322873', '478344685114473635803', '1284692741268372015043', '468529300748218446403', '5536991741058068531', '512099001538331', '12644491151999', '5515302997069027', '22898570781341140603', '998524077808748596261', '671440370920371209779', '133275883174455417167', '326427885298627673711', '1878747963984525821', '12782392755334629', '4155068134868378323', '49620003267020110271', '64826954185187783749', '789326005894461745471', '42946981319084404543', '327863032324404677213', '5552611591301816537', '1848919028136601723', '129658581464944572339', '262908841257181076389', '789733584588478531667', '4687583413518633592151', '2686249344377783927', '14976284919992880547', '388099670226504191831', '181200168604175498783', '4712546391715071098327', '183725568011142333431', '5507358959753651459', '35209283368355161331', '142329241278637531121', '68076115336649758751', '4097922751425836171', '22682932164290143817', '16863189969784156043', '26801266559541573', '27283138947173419', '13854687489401', '7256253415457', '173097846540223', '78561058047979', '7379802846208629', '599252054578057', '7361551105093629', '6244017769482883', '83652573499203', '13846267803257', '876041282012971', '4181757651815551', '1525023519457797', '905980049157597', '2260242577946253', '7571867042097301', '4160317222024591', '34132509390971', '6404583614779193', '3838835921641031', '282193818272027', '190679205878871', '703830502105153', '765660212356079', '624516955366407', '7507083365581563', '4566452476937843', '610786203631203', '12517810855347676793', '45787630343943', '7441660132341087', '4908593860596683', '283111898676209', '7007065694570667', '4551649176206123', '613929935980509', '7398561721609341', '750475651415299', '7537055220146049', '7489793506139039', '3120558058678723', '32792827955151', '98704489174739', '1591950365837647', '100270275376831', '12640627320623', '7256253415457', '27638454189532007', '2081540374539918793', '58328478994452378917', '38459525412004643', '3617571721833122411', '2051567438493636589', '26831104806313713', '13846267803257', '5618104456263749', '22903343317917243803', '124002778958589973643', '583464256034492995507', '290975940820217862959', '61560319542277643911', '22675327517461548941', '27243726226208047', '17034753501087682973', '998473097456131082329', '453864204558408882269', '87630615946164256657', '905882168082275068381', '995839934398765706599', '16848933884812569227', '159228915664925910541', '4701438199340031223321', '789241475692122536417', '157936626587063368273', '4713030407888116037671', '52885393501045583777', '142628387116450579527', '4665794683790723054833', '36079897867422086693', '4689454354421753190823', '428527426781541297323', '155997032236859639381', '108828525536145360937', '109319206884365497071', '156637200387842019427', '663633641890636741', '563819550042770299', '16649608142050256011', '12791253453722677', '12778944721212867', '12640627320623', '-5674133000535621', '-5168363499158949', '2373776130781329', '1599487875903647', '3118867604273873', '339892384705461', '5089798380812129', '-1222547416095159', '-5424925518577411', '-2830383534661443', '-2573467880080249', '6943472750666391', '322743208949471', '3133302805600071', '5454035004030183', '2541224347702287', '-5006819719015927', '-5475264849852009', '-707303862302069', '-317403296113699', '6251881499614063', '2061744479652723', '3169879948023773', '1375189658220889', '1293449337001651', '-1055422249794907', '-2807542480362837', '-115177626214881', '-5675645631772593', '-5169841562602007', '144691755823199', '7995294323936117', '6238489207587993', '5438405764983027', '509278187759201', '-4883331595915939', '-1356060132346701', '-1131318252864283', '-5082048344683969', '773631736631073', '1030607007976583', '6339268518382703', '1375266808268263', '517528600988423', '-2638205695809297', '-5614189442478973', '-5661151033072199', '-1286359850547549', '283508392518189', '4037358278127553', '3135041962992711', '340962707710061', '2541304432146297', '-1252731765294197', '-342490348692399', '-5759000365816039', '-2830516789036161', '-1029275265822291', '3536655059858853', '4037704849391637', '3135424941047643', '5457448297541241', '5083233353165527', '-1252846994328547', '-1096216727453177', '-709509145593851', '-517173761522423', '4571281401909813', '7993962562542047', '3118738339603571', '5439743965079143', '5094108081495513', '-609970884871309', '-135588491577653', '-2828277069230897', '-5081763912438589', '1241844743631617', '8247017365939429', '634065918893653', '5502165239824253', '1293172710204503', '-2638268283531723', '-280793521816287', '-719885285763339', '-707285776947481', '-5078465539100439', '6252472305737203', '8247950480992193', '6340675988527907', '5501285918600027', '258713548572493', '-2639128085417541', '-2808008655188809', '-1418648042694519', '-5169495141518467', '468687181886951', '7995915944441961', '779587131472001', '2719148649918823', '159102843639461', '-2443411088970999', '-1355934738214983', '-5660343752273899', '-2573416044552289', '435656703039319', '4034815002947589', '6267316603504123', '5454683823565003', '2541130066743179', '-626127620444611', '-5476283940309279', '-2879434570882411', '-5660759909024291', '-5146465992619573', '70285063012823', '2018312255523217', '1253914467789583', '5456475382967249', '2541142733787467', '-626284454608693', '-5479223856753487', '-1131512566660687', '-203235979000047', '776644746590369', '4123283934951261', '6340023232886739', '1100323328203117', '5176183844126097', '-329762182508189', '-2807747570198487', '-5675133269524941', '-1292605472844889', '2319257705267133', '3997705495460063', '6237534424654837', '5439310082168343', '509212356430193', '-305258968797397', '-1084826437181849', '-1439741995612911', '-1413908785386129', '-5082685981639223', '123508755185491', '8244157655889411', '6339491344630141', '2750425242614493', '646775859501813', '-211104023641109', '-5613892109895833', '-5661835314356977', '-205783941777567', '7122516119362729', '1615278655609479', '3135456656848333', '2728026336261233', '2541728384919081', '-501155016152521', '-5481085162428281', '-1135215618127321', '-2585073022981299', '4608838766604121', '7994426306410727', '623785043720261', '2719281933187617', '509343022646971', '-4881698941729887', '-542361911594543', '-5759017127122673', '1', '1', '1', '1', '1', '6165207186135019', '38539521085985789', '6165207186135019', '7004266940508687', '7004266940508687', '6165207186135019', '7004266940508687', '7004266940508687', '192697503398701', '6165207186135019', '2733625904467719', '2733625904467719', '538840905030727', '10445942806669413', '538840905030727', '538840905030727', '538840905030727', '538840905030727', '1044594280666941', '538840905030727', '1044594280666941', '1044594280666941', '2733627161685363', '538840905030727', '538840905030727', '2733627161685363', '3138400923826419', '1556703048928013', '1556703048928013', '2359775980158093', '3169261467564773', '2359775980158093', '1556703048928013', '1556703048928013', '3169261467564773', '1890870931991820033', '3169261467564773', '6226809312303623', '6226809312303623', '2359775980158093', '3169261467564773', '2359775980158093', '6226809312303623', '6226809312303623', '6276813957968241', '6929260547643703', '7713527605215523', '7713527605215523', '6929260547643703', '8647809212096263', '1351220189390041', '7713527605215523', '1351220189390041', '7713527605215523', '7713527605215523', '7713527605215523', '7713527605215523', '1351220189390041', '7713527605215523', '6929260547643703', '6929260547643703', '191970881971317', '1336022898595363', '6853615600345127', '856701950043141', '191970881971317', '856701950043141', '856701950043141', '191970881971317', '1336022898595363', '191970881971317', '381469909739547', '381469909739547', '381469543385453', '381469543385453', '1', '6103516985049741', '6103516985049741', '6103514264950259', '6103514264950259', '348784985493551', '6088782969394239', '389682296305759321', '2027262378551011', '348784985493551', '6088782969394239', '389682296305759321', '2027262378551011', '3095592314300989', '3048580531171477', '3121749444151957451', '426062451768639', '3095592314300989', '3048580531171477', '3121749444151957451', '426062451768639', '7127749247341691', '623457344619750219', '6088446265477247', '33204158918227', '7127749247341691', '623457344619750219', '6088446265477247', '33204158918227', '2624352721247009', '1561818700945468753', '6100851390185261', '565803989434143', '2624352721247009', '1561818700945468753', '6100851390185261', '565803989434143', '3160232844900649', '6100318861827607', '12493458989487349207', '6767515799910359', '3160232844900649', '6100318861827607', '12493458989487349207', '6767515799910359', '1055182893957231', '49998927101060862611', '6103384655891219', '109061498431411', '1055182893957231', '49998927101060862611', '6103384655891219', '109061498431411', '1694927608915151', '8858918658400067', '333758364653693', '3761815634431309', '8027598294072851', '99925572443657674441', '-1252539959921983', '-1175966911263331', '-5010158380975613', '-4703869103765633', '-153469618449707', '1330386683820153', '-3913313145223249', '-151781679449082', '-5686426326199581', '-5490514939093241', '-5686423412784559', '-171578682890883', '-880036033517679', '3027551068638209', '-8898314572022453', '-5588470632646407', '-4992448706686613', '-4682422280430507', '-4992450894875889', '-4682420092241223', '-7643094880017859', '662735535856241', '-7798109187240553', '-1209358873389639', '-642221036027339', '-626942849090219', '-642221181610001', '-3134713517537779', '-4974043855714869', '6813373829935303', '-200489587480573', '-793227428198473', '-1251261728822119', '-3058633343998173', '-10010091501051', '-1223453628790011', '-2428210645594513', '6654208874391453', '-4891181416229159', '-1237357533210693', '-4502569621262631', '-4468972176778843', '-9005139242525259', '-8937944353557681', '-6954350738349369', '4719864649959951', '-3485574730295633', '-897154179804147', '3492897283138733', '4898312370164547', '295062610417361', '8311996883561123', '6700050407228721', '4520817434413663', '2660032260821', '31878788925334309581', '4424059308067031', '69125915319863', '1945726863118549', '53054930515251', '2660032260821', '31878788925334309581', '4424059308067031', '69125915319863', '1945726863118549', '53054930515251', '212782613915619', '968106989384499', '8914199064858929', '445699839661941', '7930870292777792579', '100828109809073', '212782613915619', '968106989384499', '8914199064858929', '445699839661941', '7930870292777792579', '100828109809073', '1414398904097051', '3897129967284179', '4411058515883851', '1102761627638367', '1995334827333456771', '6939887864953417', '1414398904097051', '3897129967284179', '4411058515883851', '1102761627638367', '1995334827333456771', '6939887864953417', '964712919508543', '3894614831140541', '8835132633666783', '2208782976517743', '31904693637400641041', '59698474601019', '964712919508543', '3894614831140541', '8835132633666783', '2208782976517743', '31904693637400641041', '59698474601019', '7681130463264137', '968030810889801', '69674037920537', '2229525012013323', '634418387782135961', '3596210918820291', '7681130463264137', '968030810889801', '69674037920537', '2229525012013323', '634418387782135961', '3596210918820291', '1592203904405427', '3906255161439001', '4388441941775631', '8776643049367071', '8000181933971668619', '6032890129449129', '1592203904405427', '3906255161439001', '4388441941775631', '8776643049367071', '8000181933971668619', '6032890129449129', '1249648336968630311', '623480047149239', '2054291392810993', '4553816796940397', '665936712636761', '10279824769589', '-556996671544773', '-4457647564775861', '-283532075708231', '-364394232299077', '-7094997658571541', '-4803446465134641', '3689034615765881', '461077008457683', '-2401304726288107', '-222840523428351', '-5690654315236473', '-719527246436679', '-974848777022141', '-1253831697789619', '-200213551175287', '-5905635345410901', '4466592393881039', '8900349920693847', '-5872850080499537', '-5723436143364949', '-3252695130881203', '-6528409275300813', '-5479776157400573', '-8112850808942077', '-44206485419417', '-851672884972069', '518128214525289', '646940514465779', '-3400936532734187', '-6516899768531607', '-398327893165467', '-7978685487690631', '-12913658100319', '-37591609927019', '-81013568833487', '-531885910259897', '1628875808530593', '3256235450213897', '-8504110408141807', '-3986310837749993', '-5926246681084549', '-1501624291299097', '-5020416218146603', '-6925751900536023', '-1295222413250081', '-3096579076508337', '4691543945387397', '4671457956692467', '-6153029710633423', '-1193274384628093', '-5558227893382077', '-351523629496099', '-2417429167117483', '-552250272712499', '-62089024366293', '-1148051572056321', '8668354597982079', '4317607069781309', '-356698624988791', '-223652119306393', '1721775385739689', '37203419254219', '4630918549509047', '8709162601660853', '2839570093652181', '3124514104331421', '804723711030043', '6389165336626043', '2303924219488509', '2263213279129661', '129019840499751', '4598007129306433', '416968783711319', '35432168329197', '6239567310425047', '1664060225556279', '1146843101589487', '1261800707628397', '27471517582757', '2944140567887993', '42682352506498433739', '5003945607262891', '1250993313975431', '1042049403461957', '5888284773754781', '7040498868974901', '27471517582757', '2944140567887993', '42682352506498433739', '5003945607262891', '1250993313975431', '1042049403461957', '5888284773754781', '7040498868974901', '3411109008093859', '581097886486711', '5212617568324841', '1042874942419421', '5210872793680581', '21351339025509396217', '5810517932950037', '6319155393944257', '3411109008093859', '581097886486711', '5212617568324841', '1042874942419421', '5210872793680581', '21351339025509396217', '5810517932950037', '6319155393944257', '2360374163219219', '5901488471741749', '651254736478291', '123716764872257', '247420087413659', '5335087741926319033', '5901452455753803', '2341032122500219', '2360374163219219', '5901488471741749', '651254736478291', '123716764872257', '247420087413659', '5335087741926319033', '5901452455753803', '2341032122500219', '5748148224653929', '1122942231865543', '42975018864401313429', '4960035750838609', '4960033204253359', '5245973982470863', '1122942231865543', '2874008083009409', '5748148224653929', '1122942231865543', '42975018864401313429', '4960035750838609', '4960033204253359', '5245973982470863', '1122942231865543', '2874008083009409', '4085980561454683', '5771512721985843', '5217788393806299', '5193909067169099', '519324695509427', '267151846096958029', '1442856261676401', '4039490835057757', '4085980561454683', '5771512721985843', '5217788393806299', '5193909067169099', '519324695509427', '267151846096958029', '1442856261676401', '4039490835057757', '3152961721445657', '5847381306435627', '21375585062718397043', '4875895061069237', '4877233837299079', '2609301974906221', '2923778692306841', '1299625319711711', '3152961721445657', '5847381306435627', '21375585062718397043', '4875895061069237', '4877233837299079', '2609301974906221', '2923778692306841', '1299625319711711', '4647024112015103', '971436339901327', '7945452325769781', '24994301507830356539', '251080644626327', '1602950291960587', '-3344082394795219', '-1337456915772851', '-8072426855231913', '691479609280339', '553536283117549', '-8069822439577329', '-3015143125008767', '7640752895250491', '4896712403353517', '2445603463761', '-6031104425325557', '-1337544936845469', '-6266166998265433', '-3163758458513379', '-76207582107763', '1444454569445887', '221275820429517', '-7800354151646263', '-2745157742926021', '6418940166340531', '4596682064531527', '629659371120817', '-5433671026889707', '-6296841957646093', '-650368630275473', '-6510248847569677', '-8011898204369237', '1214305220244201', '6045235008126499', '-4015613611646021', '-5739380827722697', '436773040949731', '2364142825234097', '6975270586153199', '-2866649210331911', '-32534837875811', '-1708976744885731', '-273437031225201', '-8026915366445753', '190565603471571', '4764064882440837', '-4013484318017363', '-3119614397922159', '3994114832560849', '633297116396563', '3195276825179', '-6239211757388723', '-854489547510809', '-6162286006581577', '-6171492673965241', '-3811332155964153', '2947555140434683', '1464546220806027', '-191235953991047', '-5257148504558043', '183757356973247', '4486855317079089', '1465472531450697', '-2624341434947211', '-6166889340273407', '-6331876981373081', '-197119260461089', '-7979847037993441', '322551290675221', '1309495547698261', '-988684020766677', '-5413843481074883', '1557534281339699', '570362810087069', '3139063482507993', '-1087209670249469', '-3159923329031981', '4135771940239599', '8974735726550953', '2642212504896413', '6726248408289423', '57765186363933', '3593276698840853', '331159597626953', '4482637689176947', '753786779764067', '4116498690719073', '1783167226381583', '7870861847681847', '637070011535679', '451120860128503', '1791474501486167', '5300346153052713', '3609273679457491', '896694569450257', '2555047761514003', '527749270694655841', '1257782389299249', '7971261002907913', '8019484593466723', '628899853038949', '80527823986701', '2555388276321051', '2555047761514003', '527749270694655841', '1257782389299249', '7971261002907913', '8019484593466723', '628899853038949', '80527823986701', '2555388276321051', '4899461532005789', '10551405234841487343', '794633056131801', '3810365895096433', '7620724514234543', '794633056131801', '1030410667464989', '4899461532005789', '4899461532005789', '10551405234841487343', '794633056131801', '3810365895096433', '7620724514234543', '794633056131801', '1030410667464989', '4899461532005789', '5114259086614943', '1320583312261745831', '25004965761507', '7579722287329357', '7752251265827643', '781443924525063', '1031689923874969', '2558352813797347', '5114259086614943', '1320583312261745831', '25004965761507', '7579722287329357', '7752251265827643', '781443924525063', '1031689923874969', '2558352813797347', '4780009618698523', '5283062660445205661', '1582601107135463', '7371128981250041', '1474121167984361', '1582600925236531', '2579620439670511', '2390000989471691', '4780009618698523', '5283062660445205661', '1582601107135463', '7371128981250041', '1474121167984361', '1582600925236531', '2579620439670511', '2390000989471691', '606430314133343', '10551476982201629553', '318425533542697', '7691879367184657', '3845932407634367', '796063924806217', '5152088370215641', '606430314133343', '606430314133343', '10551476982201629553', '318425533542697', '7691879367184657', '3845932407634367', '796063924806217', '5152088370215641', '606430314133343', '4873170603501817', '2656630869920401617', '243019304539853', '6269962781138483', '6269956232776457', '3037741670546053', '1037746433562657', '4873170603501817', '4873170603501817', '2656630869920401617', '243019304539853', '6269962781138483', '6269956232776457', '3037741670546053', '1037746433562657', '4873170603501817', '1692691029513359', '8707815256480177', '3593548283130271', '1325866740018423', '6963502038918439', '99959026701587008933', '-1142324735740149', '-1426668366242133', '-4540330626868051', '358147944085211', '7979346693451067', '1121672563857253', '-2272400947449169', '-7725468795790027', '2006843939129843', '8047274842376659', '-3855745498801627', '-1427287142958659', '-2870416998857593', '-229633389985439', '-4547679853220847', '2248441630316023', '1590600224129017', '4496881756790447', '-1136919787073777', '-7895854778602049', '862008763507221', '8620084627388907', '-1973964258591097', '-5740834373675579', '-1149422963962523', '-5728672764563979', '-4578678961633281', '4548772193054013', '4021538764194719', '286590442907133', '-4595361518547619', '-1547531845367427', '7857595077179799', '396586846225269', '-7685461357960753', '-5737893792188297', '-1441768428735209', '-5767132017118839', '-2279981344389419', '4500726537195847', '7971974083330697', '9001221268715493', '-4559910078789827', '-3985705351227613', '8817676849918903', '22043609103017', '-1992893788096469', '-5767102866029837', '-1433927850383441', '-5735712689018043', '-4529096483410043', '8908777274150607', '1585256807515039', '1113596341196599', '-4529095058480931', '-1585767996326579', '8772523439942853', '548282393125351', '-1982211135351507', '-2867856022637951', '-5882860719002259', '-2941430664654907', '-1187892191269329', '2419648060563623', '4142058644818881', '4839294169559183', '-4751567964491191', '-7845027486816207', '7848672195006987', '1569733950755337', '-7845030048691783', '-1176572204831207', '7498683348637417', '7907313861545007', '718812056399829', '4631653148256961', '4485792248003069', '8125978377650313', '1500417165688613', '1972159491758167', '5117589462157387', '7507616018079771', '1973448094227209', '4646167585833317', '926170118380641', '936365466821991', '2032799847029707', '286948855308639', '4065680266514271', '4054090625227149', '4357455067683827', '394966392519093', '8664885019368827', '8664885019368827', '1542837470777707', '4357455067683827', '4357455067683827', '394966392519093', '8664885019368827', '8664885019368827', '1542837470777707', '4357455067683827', '8687361890679863', '247258675223410347', '8737115617553493', '8737115617553493', '7726832145540017', '8687361890679863', '8687361890679863', '247258675223410347', '8737115617553493', '8737115617553493', '7726832145540017', '8687361890679863', '108988726138221', '493702351319766473', '4268265615230623', '8536534868439801', '192852462794391', '108988726138221', '108988726138221', '493702351319766473', '4268265615230623', '8536534868439801', '192852462794391', '108988726138221', '8578154762346087', '3891493256736577', '431892044672769', '431892044672769', '62263909570082227', '8578154762346087', '8578154762346087', '3891493256736577', '431892044672769', '431892044672769', '62263909570082227', '8578154762346087', '2163697936231329', '61940170000235603', '2193826040126261', '2193826040126261', '7742521250029451', '2163697936231329', '2163697936231329', '61940170000235603', '2193826040126261', '2193826040126261', '7742521250029451', '2163697936231329', '8669643592255687', '3872439805309409', '8149063402087151', '8149063402087151', '247836194105928689', '8669643592255687', '8669643592255687', '3872439805309409', '8149063402087151', '8149063402087151', '247836194105928689', '8669643592255687', '186164703259057', '99958805235022717099', '1585711984446913', '451397036244583', '1219306844398971', '8108321910250183', '-3323183251491239', '-6646366502982463', '-3063936584946761', '8932563088295807', '8932563088295811', '-3063936584946761', '-422188732906399', '3476903152637513', '-6755019726502403', '-664636650298247', '-414634506307763', '-829269127562807', '-6108848330494067', '8900620150736899', '4450310420210301', '-1527212111360337', '-1691606179900079', '8465403455515689', '-6766424259811211', '-1658538140178333', '-3334002143247827', '-41675023166249', '-6154082403940111', '1121770369322537', '4487081910106653', '-3077041490865281', '-6767444311870911', '1591026555876343', '-3383721000354563', '-1667000999136937', '-6649998183086111', '-6649996813671061', '-6108962900801531', '4446463697383763', '1778585273541251', '-6108962729624651', '-6857926978498329', '1663438579788087', '-1371585532641173', '-3324998749189293', '-1656904302485783', '-6627617209943111', '-3047822570981831', '8877481678955759', '1109685209869471', '-6095645141963661', '-6785604798086471', '78993794071681', '-3392802399043247', '-3313808604971561', '-6734937709340259', '-6734936791904889', '-1244930017418529', '2270457944558209', '4540915545078167', '-3112324986206613', '-6851414327384853', '7454547591752259', '-1370282957220511', '-3367468625311287', '693512249622153', '3612259366853947', '7030333509024247', '2612419816121213', '32594579156161', '4110073350837961', '691438852109837', '1449326821762399', '5101172840518239', '3467492894936863', '7336562944558771', '3760462743035851', '5145446576212043', '7914238635890667', '8830693849222847', '5056310256796829', '789201460580523', '2261951859438557', '4923124361685467', '472156505325813', '472156505325813', '1230781090421367', '4923124361685467', '472156505325813', '472156505325813', '1230781090421367', '4884583104259027', '487573008296389', '487573008296389', '1221145776064757', '4884583104259027', '487573008296389', '487573008296389', '1221145776064757', '19712293578777461', '4701768557424657', '2350884642510209', '2464036697347183', '19712293578777461', '4701768557424657', '2350884642510209', '2464036697347183', '492804058282133', '4701898713523073', '4701898713523073', '9856082620834267', '492804058282133', '4701898713523073', '4701898713523073', '9856082620834267', '9769274183836193', '4875514132327611', '4875514132327611', '2442318545959049', '9769274183836193', '4875514132327611', '4875514132327611', '2442318545959049', '4923901140482859', '1179614484517141', '1179614484517141', '4923901140482859', '4923901140482859', '1179614484517141', '1179614484517141', '4923901140482859', '2503100881566799', '3412097089342843', '6179845623149607', '4510098244741746333', '274059015161277', '8010797245512387', '-4499862350130383', '-8999724700260763', '-7310897310157307', '1012193227018423', '-7310897310157307', '-224993117506519', '-899972470026077', '-224993117506519', '-1760603674514959', '-1760603674514959', '-3557698729765109', '122866071659663', '-3557698729765109', '-1760603674514959', '-8803018372574791', '-1100377296571849', '-4512764558836337', '-451276329948277', '-7336381918215281', '5080000268473897', '-7336382547892063', '-2256382279418167', '-451276329948277', '-35255968196559', '-4512680035262267', '-4512678232967033', '-7336214852776069', '1015975057143749', '-7336214852776069', '-451267823296703', '-4512680035262267', '-564084891764331', '-8803288780027779', '-8803288780027783', '-55591154100458', '30717784561573', '-55591154100458', '-4401644390013889', '-8803288780027777', '-8803288780027777', '-9003766280364181', '-9003766280364173', '-7314891442174699', '5063949872083653', '-3657445721087349', '-9003766280364167', '-4501883140182093', '-562735392522761', '7403749396748287', '3760798953087231', '1848341583140883', '7393245918216577', '1880260681099351', '3703496045649651', '1', '1', '1', '1', '246645025879281', '38535281343969631', '246645025879281', '350236971371783', '350236971371783', '246645025879281', '350236971371783', '350236971371783', '4816907771259723', '246645025879281', '236264130702729', '4414506201841121', '7792064088916633', '997384669042587633', '2207253828516277', '233150430309043', '236264130702729', '4414506201841121', '7792064088916633', '997384669042587633', '2207253828516277', '233150430309043', '1181259515351213', '4413502799151551', '487005541048299', '6383293332743149653', '4413516259673267', '2880917809858939', '1181259515351213', '4413502799151551', '487005541048299', '6383293332743149653', '4413516259673267', '2880917809858939', '266207477687321', '1769368634643373', '7970350021072478137', '972943117806699', '4423421222810561', '265476562289923', '266207477687321', '1769368634643373', '7970350021072478137', '972943117806699', '4423421222810561', '265476562289923', '65622435413497', '4460922403296941', '7743555117536183', '7929480161570300311', '2230489941681357', '2466559116621697', '65622435413497', '4460922403296941', '7743555117536183', '7929480161570300311', '2230489941681357', '2466559116621697', '3564673211102973', '87745073963999', '976605235593331', '8000565411762282683', '109685171427737', '6702743198087499', '3564673211102973', '87745073963999', '976605235593331', '8000565411762282683', '109685171427737', '6702743198087499', '6159551001014503', '1784073709982121', '7743851375476611', '15859602822174684291', '8920509703478889', '2885339188680011', '6159551001014503', '1784073709982121', '7743851375476611', '15859602822174684291', '8920509703478889', '2885339188680011', '4664352806189627', '7051998924605361', '24988316254243823143', '98634886524819', '481927212425399', '449942786455731', '-245092844431967', '-1337718830956387', '-96245467907902', '-7473137359999793', '-7489331358817863', '-396869318473693', '6065053697432203', '1515251121330413', '-1982322248246103', '-7481234359408831', '-2780129986484133', '-212100489262103', '-5620713192228869', '-414742155620909', '-3333051783941801', '-871505968872613', '530552341024159', '2648981360609471', '-6956932894934167', '-1662747257227269', '-1417675388126849', '-1821323140283511', '-7095089600377149', '-8911378371347157', '-222868361255781', '-2401532855093157', '7377447765983587', '3688304373131287', '-4802226648634067', '-8913056410789201', '-5208607995238159', '-3836875130171929', '-1072098550182911', '-6205793646428621', '-98152260387657', '-814511211304249', '309103754872648', '123166424561149', '-1619527986946001', '-780471144452417', '-596630754791739', '-5304979161129447', '-615456149959871', '-684225877921251', '-693641625615653', '-1129838591937087', '8523854928549673', '8486109803730501', '-2805762206764231', '-2755735007073809', '-5128076372319039', '-736128313710649', '-1321818415564791', '-1522007680790153', '-1541910023923549', '-638160166016191', '604962460567957', '481977856027613', '-3170897144685791', '-3063917704713703', '4575927674610189', '2318143510505219', '616076248832189', '6399536979629641', '286727225469151', '3562491325837133', '856532682523457', '926385804904493', '33157324856147', '6405108688495979', '6189491785459611', '2811578061201893', '7837407735530871', '629580180416907', '718680652600527', '4347606986272793', '3101175700278269', '5642296788049709', '3352304445240703', '7083430139426587', '287292328521387', '1788006056143729', '55990751264976735603', '1788006056143729', '287292328521387', '7083430139426587', '6704416805215119', '3352304445240703', '21250290418279753', '21546924639104027', '5364018168431189', '55990751264976735603', '1072803633686237', '21546924639104023', '21250290418279769', '6704416805215119', '1144268575777079', '6515906218405431', '7153582807730993', '3734970364177817', '28182987372484409613', '3734970364177817', '357679176766339', '6515906218405431', '1143993617329143', '1144268575777079', '3909543636366431', '42921497083078027', '140061391615319', '28182987372484409613', '5602455427920691', '42921500975268611', '19547719128600431', '1143993617329143', '3381596990048347', '3576768005264927', '7181140993799813', '1416745366998769', '5599356310259740243', '1416745366998769', '7181140993799813', '3576768005264927', '1690759750549763', '3381596990048347', '14307072021059681', '57449127950398531', '14167453669987717', '5599356310259740243', '14167453669987663', '57449127950398477', '2861414404211947', '1690759750549763', '5821331899107457', '3734694902950631', '894721431219887', '6498926200286679', '56366569791973231737', '6498926200286679', '894721431219887', '1493878397737717', '582030999084263', '5821331899107457', '1867347564680521', '14315542673107781', '6498925747465857', '56366569791973231737', '6498926653107501', '14315543125928603', '7469391535867763', '582030999084263', '2646446199606329', '6781666816024953', '7151941022630019', '7330087022134043', '56244629017477027709', '7330087022134043', '7151941022630019', '6781666816024953', '1058346231277981', '2646446199606329', '20345000448074849', '42911646135780119', '21990261066402139', '56244629017477027709', '21990261066402119', '42911646135780109', '20345000448074869', '1058346231277981', '2718586291577407', '7347507293706361', '7151208503095953', '6766380189516157', '11249030974123530041', '3383190822353849', '7151208503095953', '7347507293706361', '5436400240244069', '2718586291577407', '3673753478229563', '28604834349631047', '52862347865339', '11249030974123530041', '6766381307460463', '28604833675136577', '1836876907738399', '5436400240244069', '8011954928376701', '2029713324426141', '99864953812644918221', '1096343749559067', '1293287096634083', '267844116175241', '-6136799328204651', '-4056246542986767', '-1227369858018563', '-1223438498608473', '-2067336941817961', '-6117242454930529', '-6499093983733361', '8365073315920289', '131005052295571', '66920186832257', '-3249537624012649', '-7150885944895429', '-1270782308909963', '-1825222702878191', '-635433035125323', '-1518978729244077', '-4762430809097183', '-3038167138578173', '-331967110159703', '2154602262712023', '533606134822323', '1723597989705773', '-6639185044543723', '-3633366149670311', '-6113396640093627', '-4126808057435837', '-3056718300480829', '-613334826576551', '-8094003109496639', '-6133388226633557', '-3245255176822243', '2086600091248487', '5235256212240179', '8346360404125927', '-649049536831897', '-1786279658721653', '-6065780814949523', '-4761034017674337', '-6066086736946597', '-6348936017628863', '-3628412820831359', '-6349242132688753', '-1655477666161469', '8575616560195063', '5327878775078779', '8575309594595001', '-3310897894656873', '-7256192280366649', '-6321744666746979', '-395718052556401', '-1580531826526349', '-96305194394739', '-1147507356874691', '-6163915080621717', '-6721988868060027', '8833000708605981', '335738374561791', '8832618069247553', '-1344369075660123', '-1827807779454299', '-3073621070479569', '-916978520162571', '-6147489853342053', '-6314874749742039', '-1957181226759937', '-315756115910283', '-267820475688973', '2193592711680177', '334813306961911', '1754824514808127', '-836927368504187', '-911698643419227', '6693680880064133', '411397625841989', '2561962508776419', '5124900450298309', '4120751825925179', '4139952124042781', '4761282168191659', '8408753549100571', '1718692416036003', '8977596377256653', '2254332375772969', '5822150500995697', '1201541941147', '2056801640972347', '2560058903109741', '5121985574484339', '2059783630285579', '8629790095108407', '2378830407865651', '3095545887400561', '4485665999106403', '1718979653020613', '131398405545923', '5823467747487269', '2402749192006413', '1033854626336761', '7537135382754061', '7476439557996903', '1465271572202631', '719096777111039', '481867457223131', '534462339750129', '3736654681772823', '471226035922713', '1530088575137081', '719017351984767', '4618552223403371', '7860802420378979', '8794694185856347', '1300435703253757', '7672496989217309', '61379981734504869', '1300435703253757', '8794694185856347', '393040193778529', '4618552223403371', '3706778717335003', '125713153939337999', '17649073252102493', '2541186713445983', '19643979687611903', '491099538569100483', '508237305223543', '705962922590969', '25142635444480723', '18533893774003283', '8791276222161621', '7851881697325609', '1158435501434063', '529392238387861', '982450696819429', '7675394613710263', '529392238387861', '1158435501434063', '7851881697325609', '549454900309307', '35332626707824909', '5021853867285277', '37237456127022867', '1025280112940037', '982618216900561851', '982618032374092089', '5126402302745759', '37237457865068441', '5021853867285277', '35332633700928499', '4684945351111663', '7884545344900207', '8652294976684053', '5086281374863771', '7661828269832911', '490357055835437533', '5086281374863771', '8652299342258867', '7884545344900207', '4684945351111663', '300383007142893013', '2017897094262546499', '277419962645956101', '40143746327163587', '7846258671700967269', '7846259398038747109', '40143727606843763', '11096803344961213', '2017897094262546499', '300383025863212837', '1119288006725401', '7990803495396243', '209470412882609', '933182659735639', '763351611885007', '488545171304799613', '4665915481465619', '209470412882609', '1997700691950109', '1119288006725401', '287653582571388967', '408905966195526617', '53847600665532691', '36211453539739249', '1563567276108079123', '7817838593726480119', '36211449013801017', '269237981339425831', '2044529644713106493', '287653604559626591', '8641999373342817', '3943225039350919', '293209599031887', '5107194224674013', '490280587284477663', '7660634176319967', '5107194224674013', '293209599031887', '3943225039350919', '8641999373342817', '27723098337979417', '504561054178711917', '150466816420736991', '20085275182284039', '3922588199992232151', '3922588199992235117', '4017055036457041', '150466816420738157', '504561054178711917', '138615491689895919', '4195465486329203', '499048097897531', '8972448046300679', '4680170125538011', '488670530775776881', '1908869260842879', '4680170125538011', '8972448046300679', '499048097897531', '4195465486329203', '67391893269118763', '1021521613518441779', '36022014929127813', '9095894763224507', '977473284295478859', '1954946568590959611', '2273973690806457', '72044029858256947', '1021521613518441779', '33695946634558721', '546402634444461', '705535467227213', '2042377673270933', '99439924958533445011', '1482954539871483', '6007323099957153', '-1238221734899491', '-11092516244157', '-198115462542593', '-5399993578492649', '5413799146401009', '67672488217647', '-1349998422695631', '-5325350934397231', '658064240180939', '1052902651210173', '-1331337705430193', '-336636760460549', '-2476365812600471', '-3422956580418291', '-1238182812994031', '-2657479853224011', '656164771371251', '20997271137963', '-1062991856484637', '-539508043211747', '135239002096983', '2704780421791691', '-1079016208550947', '-5380601011140827', '-989565550522837', '-7120033291877517', '-2473913496054867', '-5433634310381003', '684304967246491', '547444040072897', '-5433635212073757', '-132807211025053', '5231746858585691', '653968369541637', '-5312287919561611', '-5392829453104309', '-4960598111138399', '-125619061883057', '-620074788058449', '-1379220811518821', '278534585144287', '87042074190373', '-2758441717397377', '-270796469984907', '5368784492639387', '2684391905032907', '-135398230174743', '-546307445533523', '-4946673414276499', '-1781484557606143', '-494667341427651', '-2654779984643411', '2613537692447791', '5227075384895513', '-2654779984643387', '-5434741118296231', '219097507316577', '5477437682914479', '-1086948223659259', '-2696022276839023', '-496053799773763', '-1997863486200859', '-1240134499434413', '-1352820643870137', '536256128167317', '1072512256334603', '-5411282575480407', '-5513139150708397', '556627443212899', '5566274432129123', '-5513139150708549', '-2730001934643931', '787819086745739', '1105717437375459', '1647119230235527', '5578729114565199', '4546750749254749', '2048042741505093', '2818284255843413', '1095726484972409', '2065627946059741', '3985125270413371', '8075431886624919', '182482446823819', '622224987456733', '5547260501133717', '7555929856435877', '5238579222706681', '50897062048427', '8232987980655587', '5433458441303631', '5043564865048081', '7116602277317', '2607829871873203', '1146713821311303', '82330180631959', '5295075836826479', '1106432989768839', '7570799102190047', '988600158386233', '811016407343259', '2278780203805579', '692266748017961', '2500057649586163', '3542522900158581', '5386887292076987', '4054276844648251', '1247433709084631', '3529061317270003', '86410362983557', '6510375455188487', '1986410801520069', '6977846389785620939', '1986410801520069', '260414901792219', '86410362983557', '3529061317270003', '169394943228960151', '41476974232107353', '6510375455188487', '47673859236481649', '6977846389785620939', '47673859236481663', '260414901792219', '41476974232107367', '169394943228960137', '706183426246153', '3757398753415047', '7357946792601619', '304483664123507', '6940741197889869101', '304483664123507', '7357946792601619', '7514798962021483', '7061833534865837', '56494673110253521', '15029596003098907', '7357946792601619', '15224184195614069', '6940741197889869101', '15224182216736631', '7357946792601619', '15029596934604247', '11298933853673083', '7054850550204409', '1441696623820707', '591420088543629', '7790163902058953', '3481087749350553919', '7790163902058953', '591420088543629', '1441696623820707', '7054850550204409', '84658206602452943', '2162544935731057', '591420088543629', '2921311463272103', '3481087749350553919', '11685245853088447', '591420088543629', '270318116966383', '84658206602452873', '7054736356478083', '7781397871340193', '2961164626932329', '721767996900691', '3481085160795548091', '721767996900691', '2961163171740833', '7781397871340193', '7054737811669579', '169313674601159287', '46688385182355863', '2961164626932329', '8661215553671233', '3481085160795548091', '8661216371945351', '2961163171740833', '46688389273726453', '169313705434384601', '1765452962090551', '3794966441283593', '7230310662796561', '1507526805042537', '3470396730481359427', '1507526805042537', '7230306297221859', '3794966441283593', '1765452962090551', '7061811848362203', '758993288256719', '7230310662796561', '7537634025212689', '3470396730481359427', '7537634025212681', '7230306297221859', '3794966441283591', '1412362369672441', '7060883802059831', '396150782013983', '3231024245941287', '3462397874396151', '3488922793526136409', '3462397874396151', '3231024245941287', '396150782013983', '7060883802059831', '56487070416478673', '3169206256111859', '3231024245941287', '13849591497584579', '3488922793526136409', '13849591497584629', '3231024245941287', '3169206256111869', '56487070416478623', '1287436354849577', '6705452414139219', '49696672912999043661', '7818319544515937', '624926289613823', '2017284631983523', '-2837934026521647', '-902062305600253', '-2837934416307641', '-2590308930501043', '2779832843354433', '8923148178577051', '2779832453568433', '-323788598041412', '-5392573807020833', '-6776665496180907', '-2696287293296417', '-6239657383829447', '-2739938890147433', '-2670293792896811', '-1095975580827951', '-1014738439409789', '1345550140877703', '8765117101619797', '5382200061381809', '-5073692155204863', '-85214164086776', '-5549959475987659', '-5453706210332443', '-768431411305189', '-176523674969553', '-5176112105491451', '-5648757599025677', '-2640654883346687', '5816026081087159', '4506720686737111', '5816026081087149', '-2640654883346687', '-5490873166535701', '-25153076427388', '-2745436583267857', '-3147885806106061', '-5492701074397543', '-6417004364549011', '-549270137861271', '-1319998541520039', '2906658255312851', '2252958290352243', '2906657835775339', '-2639996923102649', '-1129140050709639', '-1298251621342407', '-5645701669256903', '-6294826772073753', '-1093474194982461', '-5531865470367967', '-2733686013917649', '-2545442911280739', '1084112855047779', '4392062033303241', '5420563222315891', '-2545442713857677', '-2740719184430787', '-5419326318773689', '-2740719710892297', '-6158854685169797', '-5400301101222067', '-3378211569827187', '-5400301101222071', '-8300567702897', '2786980217083617', '4466302414698207', '557396043416723', '-5187854814310627', '-5674274366067871', '-456463702088789', '-5674274366067881', '-6244853993678863', '7267252016648447', '114032921302031', '1030340645725539', '4470743256822163', '6067259807426603', '3290502881280773', '160723916601987', '2573580557341043', '1650044733585187', '3243553722081489', '8247701470339259', '1681490855199621', '3937531255322669', '1562164822751423', '10299041263963', '3283888461577687', '299016353329', '1582839938477243', '7806391247768343', '3939585866818661', '329310569875277', '4742419920280869', '1405696789889971', '1582845007700347', '2569942466190253', '5147505839460097', '1029630553883313', '6335114284108947', '1028361129598437', '6749963888084949', '911032316697919', '363578519172813', '732768402256181', '8920898290484357', '4115099140926869', '205783351992283', '2169532849116873', '406160433122049', '247713294234794307', '1935260111209331', '4061602512231091', '2169532849116873', '2169532849116873', '406160433122049', '247713294234794307', '1935260111209331', '4061602512231091', '2169532849116873', '8659647624091803', '1749350203252009', '247697256685635491', '7740539271426109', '1749350203252009', '2164912087921877', '8659647624091803', '1749350203252009', '247697256685635491', '7740539271426109', '1749350203252009', '2164912087921877', '8571537481791019', '2187232970745523', '249105933440357881', '3892280210005593', '4374467760480513', '8571537481791019', '8571537481791019', '2187232970745523', '249105933440357881', '3892280210005593', '4374467760480513', '8571537481791019', '8713406771182871', '4308400485463257', '771568962164207', '123451057229337687', '4308400485463257', '8713406771182871', '8713406771182871', '4308400485463257', '771568962164207', '123451057229337687', '4308400485463257', '8713406771182871', '1736331582323991', '8736704895520491', '247350120966320469', '1545938256039503', '8736704895520491', '1736331582323991', '1736331582323991', '8736704895520491', '247350120966320469', '1545938256039503', '8736704895520491', '1736331582323991', '348398285466671', '8701589057160059', '246914891284751493', '7716090352648489', '8701589057160059', '348398285466671', '348398285466671', '8701589057160059', '246914891284751493', '7716090352648489', '8701589057160059', '348398285466671', '5161957344234877', '3219839051000921', '2124221691914131', '38413509999739', '99928267959243094721', '1962564280266379', '-3424260838281551', '6971120944822273', '-1712130965832759', '-3115787699408281', '4546675510847513', '2273337618750757', '-194736714128893', '-6739598458492231', '-6739599551876213', '-6739599005184223', '-6785447530754781', '4898558131054189', '-6785447188870851', '-3051033932807933', '555497006724627', '555496996040753', '-762758483201983', '-6632367247275409', '-1658091897289837', '-331618370910869', '-6849530615457397', '1744576909520749', '-1369905717017557', '-3042984026150413', '2214051903776651', '2214052030674751', '-6085968559893229', '-1657864498543723', '-6631456978990111', '-2652582994633', '-676187830670361', '6877829714746313', '-6761877846250479', '-6137106695624643', '1118375895928833', '447350392905517', '-1534276702684481', '-51987590052586', '-6654412447637257', '-3327205993592067', '-846327613333939', '8732158489155317', '-1354124181334299', '-6107921469645949', '8898752474152679', '8898752474152669', '-190872545926436', '-829272616284807', '-829272616284807', '-3317090465139227', '-3377116570403647', '57010915311297', '-84427914260091', '-1223957282650347', '2229866792852781', '8919467171411121', '-3059893206625869', '-3320105655092347', '-6640211310184699', '-6640211310184699', '1028673219533511', '8964627093166271', '1171508337100397', '325707389028299', '8596308969891951', '8625424158755343', '6976297905359923', '7717264272772079', '7323192776119489', '27113473990371', '7138373761306777', '7231604920902259', '2614719487679729', '4113173976254077', '66958260654301', '3471083503649599', '1142642951443121', '7228978767606721', '6129218858754719', '36104604051774113', '6129218858754719', '1462175912971213', '1462175912971213', '6129218858754719', '1462175912971213', '1462175912971213', '2256537753235883', '6129218858754719', '137721469345379', '137721469345379', '1724819098123199', '20880823933892209', '1724819098123199', '1724819098123199', '1724819098123199', '1724819098123199', '5220205983473053', '1724819098123199', '5220205983473053', '5220205983473053', '137721469345379', '1724819098123199', '1724819098123199', '137721469345379', '1315235160116251', '627068507278659', '5886431010692329', '2134075360745990597', '5210141285717379', '2943207683691557', '501666410975577', '6542190529413653', '1315235160116251', '627068507278659', '5886431010692329', '2134075360745990597', '5210141285717379', '2943207683691557', '501666410975577', '6542190529413653', '2890800765298339', '2506051576078969', '5611632974999379', '5244726798845131', '21482406928534235319', '5611632974999379', '2506051576078969', '1445364639506771', '2890800765298339', '2506051576078969', '5611632974999379', '5244726798845131', '21482406928534235319', '5611632974999379', '2506051576078969', '1445364639506771', '2537595860257573', '5042108307946987', '2939409492816689', '5334913262066125969', '5209855132211223', '587872148780991', '252141540523731', '2486104458441209', '2537595860257573', '5042108307946987', '2939409492816689', '5334913262066125969', '5209855132211223', '587872148780991', '252141540523731', '2486104458441209', '1183944445584861', '2648514023770419', '5784729296071537', '10677738361084531701', '2606867716173333', '5784712925167099', '5297159378574217', '2358380669872273', '1183944445584861', '2648514023770419', '5784729296071537', '10677738361084531701', '2606867716173333', '5784712925167099', '5297159378574217', '2358380669872273', '6001667884155593', '2471757952811339', '729074375156373', '21375092943976036543', '652310293555547', '5832411283320539', '309057123805217', '724930717279599', '6001667884155593', '2471757952811339', '729074375156373', '21375092943976036543', '652310293555547', '5832411283320539', '309057123805217', '724930717279599', '1007295932162143', '5299143911104037', '2878950700948619', '1335609599093115851', '4173752057489', '2878878123272071', '1060050698926041', '966778125462069', '1007295932162143', '5299143911104037', '2878950700948619', '1335609599093115851', '4173752057489', '2878878123272071', '1060050698926041', '966778125462069', '2033835127457747', '2499430520426309317', '4549934223116497', '1936710908397381', '787914874750489', '2031433333664979', '-100977906688417', '2709841637952743', '2701601549818161', '-4045213363983147', '-3364705471715637', '-3366763362861459', '-3051856361907983', '7805585230400073', '4937124715593663', '3898682812139337', '-1524971030683983', '-6731468834577099', '-7990902695599551', '463983704378377', '4639763730596673', '-7990961067428177', '-6830971773833333', '-3415495897799637', '-6243079529031483', '8001154695413799', '1267520567539741', '8001116345351123', '-6243061393676861', '-1366196356943261', '-7907044409125947', '92183969919737', '2917110079981959', '-1988796162005259', '-6456183369663819', '-6472547128776383', '-1139474888241133', '861832667192581', '294265228497331', '686201032486859', '-5682219754963993', '-808045656152513', '-61965877226797', '15623741216207', '2493376352200971', '-3877595024856989', '-3250282698297921', '-6503771833070713', '-5801142807885629', '7114967493355417', '4792343613075421', '7108567208785327', '-5798168788141617', '-40638553842708', '-7879662528752983', '250178649399903', '3072282163102771', '-7960062739529141', '-6356273094300817', '-3191847266840259', '-5530728829934867', '1298237681208129', '1154800883950267', '3218241872005809', '-5505384650374601', '-636998381399067', '-774377909858239', '4766392446959737', '2317807549598547', '-1960087161867129', '-6600486190108539', '-414571229337697', '-5993250416186631', '7532899589295443', '4897602787177837', '3733852821603433', '-1192582028798941', '-6616812929755847', '8272592100936447', '77870412827243', '8980107895421169', '8226225837498267', '192051309495709', '8921992210818943', '8267369256906083', '7185564866801489', '8970609438124763', '7441366213769903', '143733081409747', '7221754781084053', '1099349593523017', '3587078133379277', '361145702393201', '2659559274236777', '7165427322014719', '3614845644640577', '20564171322411', '109365417206671', '584732781328993', '49036879518162459', '984768640675991', '984768640675991', '3831006212356443', '584732781328993', '4374614505479413', '20564171322411', '20297080540685361', '140254824806264519', '149958682801947847', '3923217460099612443', '1008135993338044103', '1008135993338044103', '3923217452234723271', '149958690666837931', '140254762821956939', '20297072675795277', '4730004707180977', '8466166799975433', '8960903501073569', '954423843940541', '3990091173788067', '498761351248777', '488665054663681697', '8960903501073569', '423308449138127', '4730004707180977', '36721623545299519', '272035751711362153', '57573465229300501', '7819758538993613533', '81672330768282613', '2041808082942565613', '7819759288731055449', '287867321467055869', '272035816881102941', '7344325644949231', '131399411794591', '8890787243679159', '2306624260881437', '7673374649305757', '7852857257989743', '7852857985585561', '491096024121700781', '2306624260881437', '4445392530445853', '2627989509184501', '41130246066216051', '285422757505786157', '59233094220175401', '7858453233252470649', '2009413879009659933', '2009414065274189341', '1571690790331053113', '296165497756199417', '285422714311910073', '8226047956715307', '5290773274055539', '4689566037648839', '109443254019423', '196438807184637849', '1962800004002219', '7851200743604579', '7673390905649921', '8755458138766731', '4689566037648839', '10581551641281', '10311375392633077', '3012129110314377', '70313853727908721', '392931649788114197', '31387914181233301', '31387917091616113', '1964658243001857777', '70313842204326343', '75303233696573919', '2062274909417701', '4740305351008171', '1135876035243683', '8369807618888617', '7636823219604571', '7975294248849329', '7975294248849329', '488756918885356507', '1046225770462121', '2271751706689453', '4740305351008171', '36851862092056671', '58370969147678309', '268904424520444441', '7821177581103483961', '2040604735233222247', '2040604735233222247', '7821181282881712809', '268904401466706233', '291854822684653241', '36851838579662111', '5172261503505949', '949828698577671', '8628329161329363', '490484404356089881', '1574538962832103', '1574538962832103', '7663818818063903', '4314163853068963', '949828698577671', '5172261503505949', '40742334303899457', '60916188253800571', '276742290886687751', '7848386232751739627', '2014774112035867007', '2014774112035867007', '7848386227421584807', '276742249650715163', '304580946599156251', '40742328973746061', '49681330521794624027', '289834206430009', '48029626527431', '1240432649696457', '685102579198623', '453528526100629', '-2706704516813909', '1088475072628051', '1360593933552547', '-338338113342679', '-988784345326081', '-3524175591944421', '-4943920972254513', '-2655664841986541', '5238215475190571', '2619107929248263', '-663916160163249', '-5384443298435509', '-343534508007631', '5540127391386533', '5540127303208377', '-5496551836091861', '-495707620343667', '-7934404977677641', '-198283056327469', '-2705119476354917', '16773442881383', '1341875262168949', '-5410239049565907', '-5452976616916547', '-1076007479511681', '84099145572999', '5382345968371343', '-538003772269781', '-2474745665045787', '-1370362359908557', '-1237372787608709', '-106473219594589', '1053918458768837', '1317398096893167', '-2661830419356809', '-168054046179207', '-2648464676331227', '2609063768102457', '1304531882605197', '-66211617314797', '-989760002833681', '-3415447286698553', '-4948800573223783', '-5401535000461919', '2713669286165779', '1356834646249199', '-1350383695683921', '-167991600141669', '-107755744126281', '332802481847521', '5324841148819651', '-1346947044322677', '-2478575449318421', '-3948672405075889', '-1239287727230953', '-5512758518483779', '1114956366241149', '1393695095414863', '-2756378771181823', '-5450734954414809', '-2643058306734979', '1037985796229249', '5189928852040533', '-2643058374788937', '-4944513197802567', '-7004657945899919', '-4944513451868929', '-1086474533495547', '1096488137568041', '5482440562879553', '-108647448086733', '-672788055806811', '2619503137105097', '2485919925649017', '9060978784381', '4113130766237877', '7573534018291853', '2775329975864183', '208159294125559', '135701790372459', '5728288864062293', '8226242745627039', '1426415623957881', '1009844210695141', '5566260386089101', '985279923263567', '7549207238499973', '8184827617617623', '8247125179146779', '5523955466176637', '7884198693544317', '5626526764375487', '1613920280374693', '1302845647132421', '103211403539929', '1101013294115461', '1075429427533071', '1374345412437153', '8108427688254371', '6170652429669691', '443497359551117', '31316797472967', '4934105325420081', '1056188875962019', '506853054092433', '4767575201710089', '1894008872878297', '345277659303741', '8486082161280683', '4731938160102887', '7985702165006127', '976837655674274249', '8975280465401711', '8975278282614331', '1907886046238817', '24955321539381', '1182985085722567', '1060760088261137', '34078389524462537', '2298938640381541', '510816813580350393', '976971719774976179', '36035182740946649', '18017588615579627', '976971716553614109', '102163372029296229', '2298938121094169', '17039193462529157', '8747033488836823', '642313095208279', '246555925031887', '489967840309651327', '4686396026018913', '4686396026018913', '3827873752419153', '1577957774684921', '2569253654125729', '8747033488836823', '280491370783561603', '40521738952546589', '2019199830790077009', '1568014351991184111', '300515644805993699', '60103132133341951', '7840071744095208611', '403839928905111517', '40521743464512341', '280491386644277659', '141951665132887', '4729299883441251', '7986494492651671', '488393308119201437', '523575157412963', '523575157412963', '3815572719681263', '7986494492651671', '4729299883441251', '141951665132887', '58360402035998241', '36749399079691379', '2043457590130976671', '7815377929895086573', '53831096116655137', '269155480583300637', '7815377929895065253', '2043457590130976671', '36749399079666427', '291802010180016157', '4750913480089567', '40242083786141', '7884869856903077', '61260144139453269', '2159232749752093', '2159232749752093', '7657518017431663', '7884869856903077', '40242083786141', '4750914571483171', '6092880723339461', '20311160177883811', '8071764454197977', '3920942008022082297', '138483682704754333', '138483679081207033', '3920942011645631837', '8071764454197977', '20311163801431111', '152322049384534553', '8899796413070601', '1056674377531777', '7862570088246433', '7667545319538521', '4620879937004261', '4620879937004261', '490723040148852047', '1965642340162667', '1056674377531777', '8899796413070601', '285893807699411223', '41166652620119089', '2011717615560869347', '1570533347757345703', '59367327689884939', '59367329509511143', '7852668964862784743', '2011717429296353507', '41166643521988069', '285893816797542243', '469109200399293', '2662578041560679', '1963458349521127', '245477408630921721', '1096598178268943', '137074726808881', '7671169019716303', '1963458349521127', '2662578041560679', '469109200399293', '150738004619117231', '20677563841141961', '1004667611814531393', '3928261603883975327', '5639505092390727', '28197517208304387', '3928261598586090607', '1004667611814531393', '20677558543257641', '150738009917001551', '1672703859541917', '76030036444887', '19840537993510869347', '4544520354309709', '2444391877941201', '93498070821979', '-343359225517303', '2768421633478473', '2768421691951191', '-5493747923714641', '-135213866511887', '536645667195829', '1341614202256731', '-1352138571644739', '-4953036009433677', '-796185673498463', '-4953036311816907', '-1362663051640457', '-1083127058933781', '1089435308527971', '13617940864789', '-5415635123329577', '-2655207659029641', '5236736923051409', '5236736738673589', '-5310415486523973', '-4939085897588871', '-7120130673527223', '-4939085715661749', '-5384093973720759', '-5390335859111421', '5329375918754723', '5329375918755373', '-5390335859112129', '-2756802998526433', '5575916194636843', '1115183238927239', '-2756802998526081', '-2476586384077557', '-7969968501014423', '-4953172768155107', '-1362823949867127', '-661039730860961', '2596924372779693', '5193848571638889', '-5288317486297199', '-679206130449431', '5484511686235389', '5484511506673123', '-1358412306861139', '-4939952887602721', '-7085337763866411', '-1234988310271371', '-21531146698343', '-5378716279571691', '5381915212711559', '5381915888555739', '-5378716616354259', '-2660701152121577', '105345749054159', '1316821599581203', '-532140196656553', '-309011697237547', '-6901284524539307', '-77252921394007', '-2687758672646153', '-1323835550410811', '5217518175055877', '5217517920807487', '-5295342467528119', '-5397380080490959', '2710796573366583', '2710796450357541', '-5397379564472849', '-2473014495865867', '-6834198069995481', '-4946029491998207', '-2686583310619843', '5426723021340049', '7076268355500757', '631058869958329', '379817212227941', '1028293259225321', '5435117664205029', '2628902256452367', '7552401276570427', '696903591549091', '4659555374767757', '822630286368731', '4976290913721859', '5429931906400429', '7072724554042103', '2527007133329947', '8233010542550969', '6011841892208753', '1348537057178931', '4971627984017287', '7542787956868851', '2790958935104817', '4116498690719073', '22725715352629', '2631181103114769', '2804169784382529', '8238654752673691', '5553419308672367', '700939314306847', '4095159318189289', '1578795881365451', '3979022325135833', '4126629596742961', '689092447828087', '4099942913584839', '2384388559829067', '5598218696674019', '493766048580559', '1256892409551939', '5313119957217826641', '6060411669334477', '6060411669334477', '5188593708220533', '1256892409551939', '987531878882393', '493766048580559', '1256892409551939', '5313119957217826641', '6060411669334477', '6060411669334477', '5188593708220533', '1256892409551939', '987531878882393', '982927317255597', '395905370256149', '10552570402260729919', '3173781720210147', '3173781720210147', '2576310405768611', '7917964068745437', '245731938453273', '982927317255597', '395905370256149', '10552570402260729919', '3173781720210147', '3173781720210147', '2576310405768611', '7917964068745437', '245731938453273', '1217591345039573', '7335001691074939', '10567602800479909971', '1575555256409131', '6302199197764631', '322497257522383', '3661352297823129', '243526872827419', '1217591345039573', '7335001691074939', '10567602800479909971', '1575555256409131', '6302199197764631', '322497257522383', '3661352297823129', '243526872827419', '5203728813505149', '7413021505452091', '10564423244937042067', '3115478942659011', '6230957157722319', '1031681957513383', '3706323396832601', '5203733542877217', '5203728813505149', '7413021505452091', '10564423244937042067', '3115478942659011', '6230957157722319', '1031681957513383', '3706323396832601', '5203733542877217', '99273409288467', '7829291709682937', '5152669640815451', '6335262637211161', '6335262637211161', '2638167414891034741', '1565838987890071', '4963672647210551', '99273409288467', '7829291709682937', '5152669640815451', '6335262637211161', '6335262637211161', '2638167414891034741', '1565838987890071', '4963672647210551', '5202298886979259', '7778891425602911', '1031034792088209', '1563909386768657', '3127859518903639', '10557839484354628829', '1956360024061489', '5201648416309713', '5202298886979259', '7778891425602911', '1031034792088209', '1563909386768657', '3127859518903639', '10557839484354628829', '1956360024061489', '5201648416309713', '24994279790223966017', '7955399743049511', '1495362924070759', '128113005712301', '4742037774284251', '237024099631293', '-311846535545793', '38312634068529', '7662528032236941', '-1949040955958631', '-2381003140688419', '4863683248808793', '8303809886847391', '1215920964518591', '-2381003157008033', '-5880531902813237', '-367533224886277', '-1176106350099367', '-7869945154655999', '538228485013361', '8611591614374273', '-7869987105737833', '-1128398017721361', '4418561909918639', '3959367113620687', '1767412624295957', '-902715702917317', '-2858526526006991', '-5717067364732849', '-1143412041674683', '-7900068535974099', '170632414988681', '266440496082859', '-1975990195004291', '-573078488192561', '71172789627354', '8012532519826977', '22761622769641', '-2291692083442593', '-1153822625675481', '-5770487031592483', '-360612504999059', '-7659233135416697', '1521257173556789', '7606120790075341', '-7659350683849559', '-927272718809607', '2339409991745623', '1621771874763219', '2339369078675563', '-2318163083385483', '-359857527667947', '-5757761712114287', '-5757741077400719', '-3918471159282463', '8453949222549377', '8453911364503523', '-1959242653412387', '-2267119890015973', '8930763808192397', '3974047620188131', '1786144672911957', '-4534230683113023', '-5723468514073211', '-114469585427561', '-5723473892725631', '-3831250353178017', '7716142825783291', '1933946684237453', '-3824330223703659', '-183287991146623', '2283596067218753', '8041502951209379', '4576909599644581', '-2293310910614829', '-5726532379526737', '-1430411595888337', '-5724089381540043', '3861858679889819', '4621784268055367', '8226231473537499', '613372511464307', '4658540998347607', '827017477471273', '7910056647851767', '7551414121949281', '7127167121191183', '7899566254323403', '60473536857913', '5023243186040087', '310862528532387', '4658104969898151', '8262426887655203', '7920284990348489', '3771272439680853', '3729723597965851', '4976457393734347', '4976457393734347', '8594521899250907', '10510658838376841', '8594521899250907', '8594521899250907', '8594521899250907', '8594521899250907', '2627664709594211', '8594521899250907', '2627664709594211', '2627664709594211', '4976457393734347', '8594521899250907', '8594521899250907', '4976457393734347', '3714729898337759', '1273215680150681', '1273215680150681', '2708672171151909', '641466875478531', '2708672171151909', '1273215680150681', '1273215680150681', '641466875478531', '1863074856522641339', '641466875478531', '1273215680150681', '1273215680150681', '2708672171151909', '641466875478531', '2708672171151909', '1273215680150681', '1273215680150681', '7429470708772191', '2284456650751383', '5089235553247033', '65622898282379', '5242626057277519379', '5119722541433493', '328108961684437', '1018028427502371', '4537020962186687', '2284456650751383', '5089235553247033', '65622898282379', '5242626057277519379', '5119722541433493', '328108961684437', '1018028427502371', '4537020962186687', '4567919403225829', '4760178931887233', '6636381624143719', '5243650509149087463', '512075135894733', '829546975422247', '2380110930017291', '1141600955336437', '4567919403225829', '4760178931887233', '6636381624143719', '5243650509149087463', '512075135894733', '829546975422247', '2380110930017291', '1141600955336437', '74327928003899', '5082388937741393', '3301309540956793', '1022941311427509', '209498991527933659', '1320534730318619', '5081938555986519', '4772760212414557', '74327928003899', '5082388937741393', '3301309540956793', '1022941311427509', '209498991527933659', '1320534730318619', '5081938555986519', '4772760212414557', '1200806769697399', '4857581392797371', '6675138896136827', '654417046522985237', '5112633175960823', '6675138896136827', '2428791242095509', '1200806769697399', '1200806769697399', '4857581392797371', '6675138896136827', '654417046522985237', '5112633175960823', '6675138896136827', '2428791242095509', '1200806769697399', '8024342912075691', '4843925455155499', '6387068378831267', '10546874924796158919', '5149841271873123', '6387068378831267', '4843925455155499', '802433927409691', '8024342912075691', '4843925455155499', '6387068378831267', '10546874924796158919', '5149841271873123', '6387068378831267', '4843925455155499', '802433927409691', '2426977464817801', '1205303007419637', '1670612765071553', '523551467782517139', '5112805847372153', '3341222619760123', '964249536374019', '303296399457821', '2426977464817801', '1205303007419637', '1670612765071553', '523551467782517139', '5112805847372153', '3341222619760123', '964249536374019', '303296399457821', '741625892340533', '2612094523730057', '175806720456729', '8585643515171541', '24989712873036447773', '3535215607324631', '-1756918613794209', '339843647570933', '385765812809861', '8519078231818327', '-439749305743629', '-1122672177224519', '-5607580258624391', '-3909729637101247', '8791949332604397', '8815212362400993', '-7803272294295831', '-5610470572373491', '-8655186044758379', '511838539301573', '760222810370353', '8190508150978829', '-1731135696782353', '-5613792683419877', '-701689722076419', '-4025936503659729', '2437696860706349', '1218986534183187', '-8051107381784491', '-5613655230015613', '-4347859311019079', '2081424181137861', '7646751857814113', '1662961991100487', '-8690805857786531', '-5578276944575233', '-5581014959418977', '-3907692203482439', '4481873229398843', '4476361601129169', '-7823024912293337', '-5579645951997103', '-1073540133909819', '4045212176742353', '7554729349025371', '2022606398025879', '-8588321137633129', '-278835026415831', '-2788350109330947', '-798898351970969', '482456642515001', '1206141761114867', '-3994491981036781', '-1115340074697851', '-8970332677053373', '8724733656697901', '1574012630663941', '2181183771224553', '-358813333860891', '-5708235863381961', '-5708235506331873', '-7928716477117143', '8881920312640211', '8881921740840607', '-3964357702983443', '-2854117842428457', '-1710711308082441', '1001239160415959', '7531468163313523', '801073785568283', '-4276963734478287', '-34818481422983', '-278537454746399', '-8012611407852143', '61034149234249', '4883151278566719', '-8012036323406431', '-1392713265325657', '740725243814129', '478098305476729', '8238682973609013', '4248485603338801', '1912602929574033', '824492395051583', '4467358718582247', '3825566472305069', '4128269493217641', '8125228271921001', '4688754667038103', '1344817949923863', '8108183103343351', '2322780601903509', '5442912363211789', '8129666741889533', '2342547378217793', '1878692232988999', '7543650906655307', '8091654926462229', '1877111364654941', '3492201311051799', '864953629142737511', '3492201311051799', '750844836900277', '2022914277312371', '7543647268676549', '7543650906655307', '24274964175120217', '11262668490062881', '41906416034754823', '864953629142737511', '41906415430488353', '281566806284273', '12137485966007461', '7543647268676549', '7532933508710307', '7535154858565651', '4031627023764671', '6985039349499491', '6919530765831905757', '6985038621903731', '4031627023764671', '7535154858565651', '7532933508710307', '7532933508710307', '11302732554918313', '24189761608448353', '83820471659854219', '6919530765831905757', '16764092799396889', '24189762676727699', '141284150259733', '7532933508710307', '2113096117427613', '257957907803009', '7197459340707111', '6987895031070889', '1733968743412589827', '6987895031070889', '7197459340707111', '257957907803009', '2113096117427613', '2113096117427613', '4952791829817769', '2699047252765169', '83854740372850687', '1733968743412589827', '83854740372850649', '10796189011060657', '24763959149088883', '2113096117427613', '485772343966693', '3953091225403627', '7827370730075879', '3491686355466939', '345045155211700091', '872921497917261', '7827370730075879', '3953091225403627', '485772343966693', '485772343966693', '47437095782521841', '46964223302776957', '33520188796946951', '345045155211700091', '167600928677792429', '46964225458133591', '47437093627165207', '485772343966693', '8607771307554193', '7224564494480729', '2057992025764909', '3493225754788391', '6935892883303966703', '3493225754788391', '2057992025764909', '7224564494480729', '8607771307554193', '8607771307554193', '43347386966884333', '49391808618357857', '167674836229842809', '6935892883303966703', '167674836229842727', '1975672344734311', '8669477393376883', '8607771307554193', '236936401837117', '3924256222306737', '7882332551946363', '6985096353035703', '1725151262266497749', '6985096353035703', '7882332551946363', '3924256222306737', '592341322915949', '236936401837117', '23545537333840399', '2955874706979889', '83821156236428459', '1725151262266497749', '83821156236428413', '11823498827919533', '4709107466768089', '592341322915949', '5772149140106111', '49802126829331756711', '2389384756372387', '796729401343169', '6501825859526331', '5028627582371', '-126501152545543', '5444170069813107', '2170674208824153', '5444172620237067', '-2530023279822861', '-1105649631851863', '-2340276957574333', '-5528247647179881', '-1075211502827691', '-5898079428152297', '-5376056914049833', '-6113317142613183', '-5061509367343257', '5447422042154733', '4342193417634857', '5447422034058549', '-1265377341667143', '-538374645533519', '-5843856003926999', '-5383746732977637', '-2760779840202713', '-2370676217649761', '-220862376002571', '-6114228594647293', '-308869678646014', '643576116405857', '2143342656708089', '5148608931246847', '-2470957429168111', '-2774175716800947', '-3978627533310217', '-2774175716800953', '-5269524624975543', '-6209242002321079', '-5269524624975549', '-75570998440821', '-4817676041994413', '1233349264527603', '167897175161027', '4933397059307089', '-963535208418827', '-534582455973097', '-4772989217825883', '-1336456209661633', '-5325442386299413', '-308503052187951', '-5325442107782761', '-5942448351416989', '-4924195227971573', '2555367600074021', '8552585997699163', '2555367600074017', '-4924195227971573', '-2634106874782217', '-765245964474653', '-5268213749564439', '-5533794334633673', '-7994646070486641', '-5533794334633677', '-6033459714039089', '-4842077808906751', '4987060618126321', '4211247576854929', '2493529840488997', '-2421038728738063', '-5341835182045349', '-1233839157768431', '-1335459029798421', '-83602863360754', '-2432686023365251', '-5350584192236589', '-2979377614751867', '244501933066619', '3825711705074637', '7399154996661099', '1611849571482729', '319651433517601', '4113166461437477', '1516693632910283', '81957623629233', '22046108271427', '7991959367864587', '4030628014927157', '1645256439611343', '902409539114307', '5202514676250101', '4138048266981619', '3735173248303081', '921710379597301', '3449032337269409', '6629885352657457', '8313526319960051', '8324279808074193', '5205800312832103', '5207942967472509', '2015629100123897', '907821508556543', '4140035600125959', '1426267942121491', '4605943513303747', '7473246837734367', '3450527171536431', '6491547422819031', '4135848428046679', '8293259428403733', '5204717081716337', '5211460289320321', '1998566858924757', '59010327293263', '6620395392190931', '4965033518271319', '5238988705206863431', '5116192267672963', '1241196533927139', '1655106124005461', '4730411893970217', '59010327293263', '6620395392190931', '4965033518271319', '5238988705206863431', '5116192267672963', '1241196533927139', '1655106124005461', '4730411893970217', '4723324529674309', '817358651967037', '5250598650642759', '2558731222207641', '5240284150784579643', '5250656494508489', '6538862667374137', '3688387438689', '4723324529674309', '817358651967037', '5250598650642759', '2558731222207641', '5240284150784579643', '5250656494508489', '6538862667374137', '3688387438689', '4003996392769349', '3187611357492027', '2428501571650307', '2637271697751826171', '1030184256934307', '2428501571650307', '3187611357492027', '4003996392769349', '4003996392769349', '3187611357492027', '2428501571650307', '2637271697751826171', '1030184256934307', '2428501571650307', '3187611357492027', '4003996392769349', '5061613559083903', '1334208443646077', '4841779967857329', '2618054003036847137', '5113386724681343', '193671242370037', '1334208443646077', '5061613559083903', '5061613559083903', '1334208443646077', '4841779967857329', '2618054003036847137', '5113386724681343', '193671242370037', '1334208443646077', '5061613559083903', '4868002167817747', '1645953364836979', '1307463344686719', '2617524283457874541', '5112352116128661', '1307463344686719', '1645953364836979', '4868002167817747', '4868002167817747', '1645953364836979', '1307463344686719', '2617524283457874541', '5112352116128661', '1307463344686719', '1645953364836979', '4868002167817747', '5039010265610579', '3330449532281867', '2438249785055589', '5236317943053658281', '2556795870631669', '243824923935877', '3330449532281867', '1259751384059537', '5039010265610579', '3330449532281867', '2438249785055589', '5236317943053658281', '2556795870631669', '243824923935877', '3330449532281867', '1259751384059537', '13626595954269', '205933529795903', '99994652166220277469', '1092837243083403', '55131354499229', '324929298277527', '-2166801247445329', '8249861022550809', '7621007520137829', '4121606163092349', '-8664195780505391', '-7900031883741401', '2316092677242943', '1157206265177347', '-7904723149786239', '-698286174569467', '-5587960429374277', '-1396781228241251', '-1097040449532141', '8493674373944937', '1930389950917423', '4247596078487263', '-1097126397846571', '-7693055055019811', '4215536769130239', '8432609620590901', '-7691982419953311', '-5584748834286049', '-1396091962743559', '-5584558342630141', '-561287330418833', '8741484788576411', '3940277921060431', '1092685598572051', '-8980597286701333', '-7915653053461877', '1102878358234133', '8823026865873183', '-7915653053461873', '-2854948168496803', '-713737042124199', '-1427474084248399', '-4240233796506013', '1960612274886719', '3747529402611963', '313698013688319', '-8480467659583159', '-7981028988517173', '4872754936311493', '4872755557642073', '-1596205886464941', '-2772325871132597', '-554465143159991', '-1386162896733137', '-4352342881034013', '4181421477462891', '239522085285542', '8362842954925777', '-2176171440517007', '-3857432607713491', '1721297518312931', '1721297518312943', '-7714865215426989', '-347702707345989', '-1112648663507163', '-5563243317535817', '-8503176218351521', '7893471246135739', '7512580409864311', '394673648447537', '-2125794384543543', '-994471858002663', '4817472053439753', '4817472914847293', '-31823088823607', '-2773518754120823', '-5547037077537889', '-5547037292889767', '3591413731837749', '8609736345993077', '7784768642710089', '1189038237929467', '4254455701845899', '7756950276159997', '4116502451035791', '5264853735256943', '233680519175797', '8366309827671373', '3435173516547713', '4716971776437067', '6953798058871191', '8506494115508523', '387644368944859', '4173441940891041', '6868902359888851', '1179110620387477', '3014472144717171', '6248827234708879', '6248827234708879', '5711814589584367', '2950238577024353', '5711814589584367', '6248827234708879', '6248827234708879', '2950238577024353', '239076802962230493', '2950238577024353', '6248827234708879', '6248827234708879', '5711814589584367', '2950238577024353', '5711814589584367', '6248827234708879', '6248827234708879', '120578995050611', '8575434602414301', '8453369209844801', '8453369209844801', '8575434602414301', '42457586004154113', '165849945328727', '8453369209844801', '2653598592799621', '8453369209844801', '8453369209844801', '8453369209844801', '8453369209844801', '2653598592799621', '8453369209844801', '8575434602414301', '8575434602414301', '5802752178209331', '2162247583460569', '61609600801977497', '1925300025061797', '2162247583460569', '5802750723017833', '5802752178209331', '2162247583460569', '61609600801977497', '1925300025061797', '2162247583460569', '5802750723017833', '1468277277908203', '212786518463203', '31071064883989657', '1553553244199483', '212786518463203', '1468277277908203', '1468277277908203', '212786518463203', '31071064883989657', '1553553244199483', '212786518463203', '1468277277908203', '584099853163953', '86468647178201', '3850533661589161', '61608544406192859', '86468647178201', '584099853163953', '584099853163953', '86468647178201', '3850533661589161', '61608544406192859', '86468647178201', '584099853163953', '2949247050163029', '2160503274065731', '61613579910632579', '61613568269101', '2160503274065731', '2949247050163029', '2949247050163029', '2160503274065731', '61613579910632579', '61613568269101', '2160503274065731', '2949247050163029', '2765843022888429', '8613771671288421', '61818244713785299', '7727279861627363', '8613771671288421', '2765843022888429', '2765843022888429', '8613771671288421', '61818244713785299', '7727279861627363', '8613771671288421', '2765843022888429', '2953546619235417', '2154087435956353', '1542851797357853', '61714083535847167', '2154087435956353', '2953546619235417', '2953546619235417', '2154087435956353', '1542851797357853', '61714083535847167', '2154087435956353', '2953546619235417', '227771525587911', '1795607136074719', '5056641704963181', '19985632970019282737', '7572090296216439', '268832331784627', '-5617590003986343', '8123334662583047', '8123334968706243', '-1123518062021907', '-1244738260605129', '-6223690690779271', '-1299455117292139', '4377343636989561', '-6497274361967923', '-3111845498451229', '-5572462950867399', '402109828222739', '4021098282227389', '-5572462950867399', '-6205458674560029', '-6205458674560029', '-3292681045930697', '1519613669205463', '-32926810459307', '-1551364668640007', '-560600488259527', '8104907444585897', '810490779058147', '-5606004940261197', '-3107102089940921', '-6214204641209281', '-6491880877887181', '8885643264312489', '-162297016180587', '-3107102205272781', '-2794216410897739', '4038420418091683', '2019210036062199', '-111768654129461', '-3100025268697301', '-6200049614815169', '-6485060378428931', '285010532968901', '-6485060839718653', '-1240010015220977', '-5692957491007501', '8239685479915319', '1029960641886677', '-5692957433537183', '-1258491892792377', '-6292459004199363', '-657347960483261', '8992655542163809', '-1314695966942777', '-6292459234080623', '-5581387637792017', '806388674576279', '2015971859142651', '-5581387752926653', '-1239555227713081', '-1239555411928497', '-3251139223017709', '1218006466648821', '-812784748187111', '-6197776599103943', '138964147099779', '1474861125408337', '32557772508971', '707890438726023', '1479128274889811', '169251280664591', '700246634943429', '1476224444801023', '8296471075327727', '2055032451381591', '5544044480270189', '36572627121319', '8988553157086903', '5275978716119763', '220753317543457', '1700497487997541', '1095598930165459', '259464644010087', '1478288814607121', '31082778204451331', '2125775298530639', '2125775298530639', '7770694551112831', '1478288814607121', '1478288814607121', '31082778204451331', '2125775298530639', '2125775298530639', '7770694551112831', '1478288814607121', '5806152026162177', '123259892087298271', '432184574372629', '432184574372629', '3851871627728071', '5806152026162177', '5806152026162177', '123259892087298271', '432184574372629', '432184574372629', '3851871627728071', '5806152026162177', '5514681905932303', '123630500613550617', '8615581576589629', '861558303178117', '3863453144173459', '5514681905932303', '5514681905932303', '123630500613550617', '8615581576589629', '861558303178117', '3863453144173459', '5514681905932303', '5908219430875669', '24692730374488623', '8611842301514881', '8611842301514881', '1543295648405539', '5908219430875669', '5908219430875669', '24692730374488623', '8611842301514881', '8611842301514881', '1543295648405539', '5908219430875669', '5875602549381817', '123262551454755143', '345560765456599', '345560765456599', '1540781747665293', '5875602549381817', '5875602549381817', '123262551454755143', '345560765456599', '345560765456599', '1540781747665293', '5875602549381817', '117894043577101', '61629543664623899', '269945575789417', '269945575789417', '1925923057620557', '117894043577101', '117894043577101', '61629543664623899', '269945575789417', '269945575789417', '1925923057620557', '117894043577101', '130277789105603', '1104141715379277', '2113779651023063', '3363197690154979', '1731533744436513', '49963884819688674151', '-1111872854177919', '8020920456853613', '8020920456853619', '-5559364270889593', '-3292166218479821', '6219460273736057', '-205760388654989', '-6195616169851133', '-3097808084925573', '-6195616169851141', '-5615650914310877', '8119897022212121', '811989702221213', '-5615650914310877', '-6500524529090367', '1110859665084447', '-650052452909037', '-388925600801203', '-1244561922563853', '-3111404806409629', '-35615829237689', '8248608353782279', '1031076087176859', '-5698532678030241', '-6575396857837757', '1782294659497777', '-6575397545102911', '-50375312036451', '-6296913317291239', '-196778551903869', '-5580260306845659', '4030892295692909', '4030892295692911', '-5580260306845657', '-6505374374117143', '985287454419721', '-6505374374117149', '-247898881784439', '-6197472044611001', '-6197472044610989', '-5594535486386109', '202156342462807', '161725067052887', '-1398633857185363', '-1622840555396489', '1142910231107113', '-3245681226082287', '-1241127001935421', '-6205634548519867', '-3102817389549243', '-2794431797092957', '161545033823841', '403862567265719', '-1117772707307927', '-6488483810141709', '287532700489603', '-1622121010181707', '-3100475727764889', '-1550237748589889', '-6200951224944667', '2810743943493261', '8422519877055641', '464424342279409', '1675728390900149', '831091922884111', '7389680419722449', '8922170435677879', '2398948190941451', '530755184168497', '8508870657448377', '252522219562323', '5483909883314563', '1564228977731037', '8345358105833037', '1844827140422983', '4119591720106033', '19021266423153', '5547380078446139', '1781992261629179', '7131062196993991', '7131062196993991', '1781992261629179', '43035103574466539', '1344846986702079', '7131062196993991', '1344846986702079', '7131062196993991', '7131062196993991', '7131062196993991', '7131062196993991', '1344846986702079', '7131062196993991', '1781992261629179', '1781992261629179', '587430531334543', '1506138684997583', '129192648473549', '8074540529596813', '587430531334543', '8074540529596813', '8074540529596813', '2937151469874959', '1506138684997583', '2937151469874959', '717244130784209', '186761101813221', '4669027181532639', '717244130784209', '717244130784209', '186761101813221', '4669027181532639', '717244130784209', '2868756061617841', '2334568888045011', '4669137412292137', '2868756061617841', '2868756061617841', '2334568888045011', '4669137412292137', '2868756061617841', '2879171006497927', '72873905310189', '4663930303649977', '2879171006497927', '2879171006497927', '72873905310189', '4663930303649977', '2879171006497927', '118422851355499', '2311614991528131', '4623229983056263', '118422851355499', '118422851355499', '2311614991528131', '4623229983056263', '118422851355499', '5759723782681769', '1865433871731823', '2331792339664779', '5759723782681769', '5759723782681769', '1865433871731823', '2331792339664779', '5759723782681769', '5919441305628597', '18494621740068237', '4623655435017059', '591944021423493', '5919441305628597', '18494621740068237', '4623655435017059', '591944021423493', '6220831029856813', '6322091260335401', '261021161506229', '3095490881382360507', '557308671556537', '3111036881713359', '-6106112497505879', '8299925138857269', '-1526528124376469', '-3912299380585183', '-1956150165861893', '-7824600663447571', '-1956149690292593', '-782459971230897', '-1221319373091869', '8300667026660169', '-381662304091209', '-3912526228700443', '-1956263589908297', '-7825054359633191', '-7825052457400891', '-7825053408517039', '-6083746109548063', '8265658666787991', '-6083746109548063', '-7803668056794367', '-7803666152438171', '-312146646097527', '-7803668056794371', '-7803667104616271', '-5907320886203859', '7994760667511981', '-5907320886203857', '-381988110489573', '-3819881104895741', '-7639762209791473', '-7639762209791471', '-477485138111967', '-6082232556827719', '8263339206715257', '-6082232556827717', '-1560450362776069', '-195056295347009', '-7802251813880357', '-7802251813880353', '-1950562953470089', '-5909145290212139', '3998784069553813', '-1477286510108429', '-7641447883519601', '-7641444882633313', '-3820723941759811', '-477590305164581', '-7641446383076459', '8346295254994283', '1043088341513507', '1037323604609519', '4113124190838673', '1658802599654199', '8255683104262183', '5301861271073251', '1897911811484023', '32319365190359489', '8079841297589877', '5301861271073251', '8079841297589877', '8079841297589877', '5301861271073251', '1897911811484023', '5301861271073251', '1525879284777677', '1525879284777677', '1525878527722323', '1525878527722323', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '334200656153179', '210984442606081', '334200656153179', '68561488224228737', '535636626751787', '8355008573713691', '535636626751787', '535636626751787', '210984442606081', '8355008573713691', '1416379710338313', '1416379710338313', '3645134652322739', '69460025135607', '3645134652322739', '3645134652322739', '3645134652322739', '3645134652322739', '42395034872807', '3645134652322739', '42395034872807', '42395034872807', '1416379710338313', '3645134652322739', '3645134652322739', '1416379710338313', '1606915160815727', '6236831065011157', '6236831065011157', '1209336434608877', '3128524025340043', '1209336434608877', '6236831065011157', '6236831065011157', '3128524025340043', '947014925171532651', '3128524025340043', '6236831065011157', '6236831065011157', '1209336434608877', '3128524025340043', '1209336434608877', '6236831065011157', '6236831065011157', '3213827247488079', '273479336137449', '172217053406957', '172217053406957', '273479336137449', '10446976325862897', '652936020366431', '172217053406957', '652936020366431', '172217053406957', '172217053406957', '172217053406957', '172217053406957', '652936020366431', '172217053406957', '273479336137449', '273479336137449', '6164658059631399', '77077740619145203', '7004381243138557', '7004381243138557', '6164658059631399', '7004381243138557', '7004381243138557', '3082330460822171', '4817358788696577', '3082330460822171', '1', '1', '1', '1', '1', '6103517338262831', '6103517338262831', '6103513911737169', '6103513911737169', '1221098664269377', '3905933641344079', '1220604171970549', '4884395384673313', '1221098664269377', '3905933641344079', '1220604171970549', '4884395384673313', '2365409654319079', '153775330327087', '2460405467132333', '2365409836218021', '2365409654319079', '153775330327087', '2460405467132333', '2365409836218021', '4884586887677609', '4882368812131133', '4882368812131133', '4884587615273327', '4884586887677609', '4882368812131133', '4882368812131133', '4884587615273327', '4726047324909629', '9844007587545187', '307625237110787', '4726047324909629', '4726047324909629', '9844007587545187', '307625237110787', '4726047324909629', '4709387729712193', '9852337385143903', '307885543285747', '4709387729712193', '4709387729712193', '9852337385143903', '307885543285747', '4709387729712193', '147195105294749', '615744347830251', '615744347830251', '147195105294749', '147195105294749', '615744347830251', '615744347830251', '147195105294749', '1194638474735173', '8312066309348841', '2634078906960073', '2582238436793161', '28788131980443', '563970540850791573', '-8792180306470347', '-8792179701073687', '-4396091062803631', '-8792177881936757', '-7104560967916087', '613314533754973', '-7104561574049483', '-8792180003772009', '-4493853457878907', '-224692596479857', '-8987705110814543', '-1797541132827507', '-7299013663839453', '5052087473422979', '-7299013976744523', '-8987705387476039', '-8791941025067387', '-4395969300314767', '-8791941025067387', '-4395969300314761', '-888040093438683', '1226584024338021', '-710432135361893', '-4395969906424227', '-281059370604057', '-8993899859329821', '-8993899859329829', '-8993899859329809', '-3652569640413639', '2528332157997413', '-3652569640413641', '-8993899859329819', '-2253895541888983', '-4507791083777967', '-2253895541888983', '-2253895541888981', '-7326563498021391', '5072667956132411', '-7326563498021393', '-1126947770944491', '-2253616594596147', '-140851037162259', '-563404148649037', '-4507233189192281', '-7325461616305879', '5071845021709739', '-1465092323261177', '-140851037162259', '1753754667750957', '6846016333427479', '3499830135655587', '6836531686741801', '6833355134629483', '683164164544719', '8815722772733889', '247369269565862559', '2169498135770341', '2169498135770341', '7730288946337497', '8815722772733889', '8815722772733889', '247369269565862559', '2169498135770341', '2169498135770341', '7730288946337497', '8815722772733889', '1097943488825449', '7716638564628101', '8706298338931661', '217657440283397', '246932480634229177', '4391776138089139', '1097943488825449', '7716638564628101', '8706298338931661', '217657440283397', '246932480634229177', '4391776138089139', '1767983224902013', '309662374880867', '2163673926578339', '2163673926578339', '99091969275103351', '2209978121632799', '1767983224902013', '309662374880867', '2163673926578339', '2163673926578339', '99091969275103351', '2209978121632799', '8217388163401623', '247818936536954953', '2167146578482449', '2167146578482449', '7744341039184061', '8217388163401623', '8217388163401623', '247818936536954953', '2167146578482449', '2167146578482449', '7744341039184061', '8217388163401623', '218080048946377', '30867267704841227', '34014955415611', '34014955415611', '3858408463105153', '218080048946377', '218080048946377', '30867267704841227', '34014955415611', '34014955415611', '3858408463105153', '218080048946377', '8878456219428591', '498245069623154679', '8566452260575853', '8566452260575853', '1946269803215449', '4439229928703597', '8878456219428591', '498245069623154679', '8566452260575853', '8566452260575853', '1946269803215449', '4439229928703597', '31982982480548159', '940623755620379', '8913604448927059', '273576498767709', '403428588746603', '1780958798273861', '-6621122327060263', '-3310561393321877', '-3381974261983689', '4570427271023757', '-6763948294175621', '-1522976557131263', '277289095741507', '554578213025991', '-6091906285972991', '-3310561278426003', '-1656653748272127', '-6626613198926899', '-1686792188446751', '7715424289972861', '-1686791611822513', '-762892527166311', '555810875326473', '8892973751313117', '-1220628141788341', '-6626614096007699', '-827124738541779', '-3308499227078121', '-677806413223997', '2577073127357169', '-6778066371170149', '-6083107642265977', '8857716057453833', '8857715210899757', '-6083107082533433', '-3308499090622617', '-6722833938885639', '-6722834397635269', '-6843559047107753', '7726384906232733', '-3421779408866463', '-6210210822695711', '4529502223286893', '4529502395318009', '-6210210880039417', '-1680708542065113', '-6636604849367093', '-6636604849367089', '-6753206478953463', '746250429352769', '-6753206478953453', '-6115177403417773', '8912052382152001', '557003273884501', '-6115177403417777', '-414787803085443', '-3305077616247903', '-6610154232121221', '-3419007330482261', '729148570500571', '-6838012660215353', '-757496908557273', '4407436710427879', '1762974784208611', '-1514993942161369', '-1652538683077127', '1647411895881631', '66120199488567', '5211929793567723', '5307880082050767', '1789018942379729', '1383150082691999', '8707333687565889', '6968046130632301', '2589753779223549', '8992679682709129', '2115692876365049', '5085569768728467', '264237748169157', '287105210708049', '3454471714441581', '1056872940986967', '7277572710275193', '277515392189047', '4137992603196989', '6368244322789409', '1318573352792003491', '4888541494469269', '4888541494469269', '5150677159343763', '6368244322789409', '827596556130763', '4137992603196989', '6368244322789409', '1318573352792003491', '4888541494469269', '4888541494469269', '5150677159343763', '6368244322789409', '827596556130763', '4038800791189711', '3143553054892733', '643862375817139', '1302279578662213', '5209056105205197', '263726215399233127', '6287098833827143', '4036494312401541', '4038800791189711', '3143553054892733', '643862375817139', '1302279578662213', '5209056105205197', '263726215399233127', '6287098833827143', '4036494312401541', '8943044670954013', '6375800151043501', '5274082349592615671', '4854356304225869', '4854514556304033', '1287616578787897', '6375821251320591', '8954428634241141', '8943044670954013', '6375800151043501', '5274082349592615671', '4854356304225869', '4854514556304033', '1287616578787897', '6375821251320591', '8954428634241141', '3235132167917637', '6087535522451617', '2656061452371029067', '4857421439543961', '2428709992176253', '5187620024162167', '380470924678493', '3235132167917637', '3235132167917637', '6087535522451617', '2656061452371029067', '4857421439543961', '2428709992176253', '5187620024162167', '380470924678493', '3235132167917637', '5389897267130989', '6239473354763211', '1056010484936501463', '2592163182665191', '5184583934226349', '161134150894621', '6239506096572021', '2109110226053', '5389897267130989', '6239473354763211', '1056010484936501463', '2592163182665191', '5184583934226349', '161134150894621', '6239506096572021', '2109110226053', '4715119259394593', '6310262023967237', '2578272350059821', '614673678488541', '2456975405152669', '5280423962445530041', '6309840018421377', '2291598161991027', '4715119259394593', '6310262023967237', '2578272350059821', '614673678488541', '2456975405152669', '5280423962445530041', '6309840018421377', '2291598161991027', '599157266261231', '306938536894853', '141105427295591', '6249733835168125813', '233830080680557', '19239620900193', '-2844402818487193', '-284440375321059', '-492625680909501', '6853775192962', '4386412384602061', '-3941008251446213', '-558037483625119', '8626679839195353', '7855051348942387', '8626672361408173', '-2232149058197287', '-5688806571697783', '-570000740901403', '-5700474411258733', '-3827057421691499', '7819089084837563', '781720383269023', '-3827720103135859', '-1137550109905577', '4508508729501939', '7991670631303743', '2253788738858377', '-1137444220725607', '-285012045506819', '-1410239365222427', '-2819953304125971', '-788428562394973', '4483711729789479', '560728080507971', '-3940673316059271', '-1752260726615969', '165009658216717', '484861095885619', '8254657132112607', '-8763190946397997', '-5640432034570823', '-5862818068962463', '-1465704673410271', '-490683426168209', '7952465630979929', '7952463132265377', '-3925467067361767', '-4717139511065531', '595664365282091', '1648084994196953', '1191328600601311', '-4717139496243231', '-5862818381301773', '-5523573071416877', '-220884902752753', '-7559359874570149', '8135075093283759', '814091672145449', '-236104178729691', '-8529276391945537', '7696127382181083', '1922198806063953', '7701885144896919', '-8531869784197621', '-690355977514731', '-5607346206597133', '-5630732008267961', '-971176962209497', '8779802256220121', '8685694099224339', '-979379575756279', '-546190074300519', '13060087887839', '1939730005376759', '8069545633675529', '-1739388616205001', '-2809519553716273', '8293755489650321', '36007996701859', '2272346448611641', '8005411715791739', '1502335698843733', '1595893018328769', '8466135644510589', '920106599873697', '7055426994159611', '2039533842535223', '36561993352159', '7378129202858837', '39981542618167', '3746329715035003', '1606479254677599', '1015364597466991', '7507849002319807', '8076430484424029', '5198231753663853', '5214688877981823', '5792492302756083', '2389067171217081', '4742470145686731', '5792425727745579', '10679716349718620419', '519772461943083', '5198231753663853', '5214688877981823', '5792492302756083', '2389067171217081', '4742470145686731', '5792425727745579', '10679716349718620419', '519772461943083', '5016378430061219', '1302705004078063', '5880918567082161', '6729609850541533', '1341656949573143', '235236364333517', '2667940407145378411', '2508151561952597', '5016378430061219', '1302705004078063', '5880918567082161', '6729609850541533', '1341656949573143', '235236364333517', '2667940407145378411', '2508151561952597', '5215377728283437', '1304654791857981', '5760722398275981', '3740929277958901', '7475955571448291', '5760716213711949', '855018802810627627', '5215336619122519', '5215377728283437', '1304654791857981', '5760722398275981', '3740929277958901', '7475955571448291', '5760716213711949', '855018802810627627', '5215336619122519', '975623593080291', '2609668723541463', '2919891899204273', '7673855474522087', '7045842127525141', '5839197720041887', '5344750929948999809', '4873693455720943', '975623593080291', '2609668723541463', '2919891899204273', '7673855474522087', '7045842127525141', '5839197720041887', '5344750929948999809', '4873693455720943', '4967215161791347', '5210830958668133', '5892569636927177', '7314267023785741', '7254177799924579', '2946270812245109', '2667949362392917337', '993401195601773', '4967215161791347', '5210830958668133', '5892569636927177', '7314267023785741', '7254177799924579', '2946270812245109', '2667949362392917337', '993401195601773', '4970642077520041', '1343050030904020747', '5609946635400639', '1332299498014093', '1332272940768987', '5609946635400639', '5246289183218831', '1242660792228419', '4970642077520041', '1343050030904020747', '5609946635400639', '1332299498014093', '1332272940768987', '5609946635400639', '5246289183218831', '1242660792228419', '1859680475584829', '2344095205159667', '400885203847849', '396181096915619', '2012946711747547', '49988561500724942297', '-6063015320804367', '-1517266081706299', '-2550005461058851', '1384444044389223', '1098895261372661', '1381432288595061', '-5094466368264619', '-7585370481761141', '1531115015246343', '6100187391935987', '-1520574205819753', '-6066039823814783', '-335777419481039', '-3359035553163333', '-6079747710427943', '7749668066935319', '492517045865917', '7744631171929693', '-303870121596717', '-8070569888414117', '1358756130327903', '2712464270511279', '-8078037873116867', '-3358404873986861', '-1243727916848917', '-3109636701190867', '-66734504876231', '6061010620713351', '283579321572002', '6059747835079407', '-5338176615658321', '-1910675129998251', '5699944134582229', '1139480410853729', '-7644549013846031', '-6218956493313159', '-96865210094683', '-3133994553112693', '-1330333048833939', '188011515727369', '8977351149870789', '5879579166312513', '-2629085467533997', '-974736172426517', '3397068985886139', '1303798591214877', '-7997755904834627', '-1558420319035637', '-333179017209903', '-1334028128830679', '-5993047367942999', '377740748340079', '974487152889671', '301668577346531', '-5986950993709433', '-2018940099735729', '1137509139613399', '1415317176020627', '-8095164189945911', '-6666860494175727', '-6883327070930451', '-1376668493404929', '-6315485808615329', '8158119763960937', '5110237363694687', '815808897177257', '-6315471123501507', '-8041516777038947', '23164269406357', '2316396148447247', '-4020782152728809', '-6883334768977547', '1798500292430569', '900717543788007', '271907992516891', '1795787129089011', '2185533806890541', '8249029346560307', '7211200102222041', '7184145906372947', '4844955115752299', '3606061057048287', '7191781924939149', '267423411140591', '4487634074869967', '105738714643781', '4128136581864559', '1783044764924479', '4095494728223641', '8208700604363353', '4462316598025009', '774106840550941', '755033644536711', '1840584333752123', '7926900240843092967', '1115570600255731', '4462316598025009', '774106840550941', '755033644536711', '1840584333752123', '7926900240843092967', '1115570600255731', '8852377030968119', '31870035611370300613', '9260702641973', '296115929425051', '3890385206466103', '8852377030968119', '8852377030968119', '31870035611370300613', '9260702641973', '296115929425051', '3890385206466103', '8852377030968119', '8916870175798629', '7742246849531309', '6568637064163951', '6414265981190921', '991017282494824657', '8916758853646659', '8916870175798629', '7742246849531309', '6568637064163951', '6414265981190921', '991017282494824657', '8916758853646659', '2193330198543771', '7811143771018439', '4779197228232799', '1162537346438483', '3999371920930316117', '1754627342488621', '2193330198543771', '7811143771018439', '4779197228232799', '1162537346438483', '3999371920930316117', '1754627342488621', '4416904743182493', '15953769372901465669', '82160672595183', '25632054664849', '7789925724016323', '4416904743182493', '4416904743182493', '15953769372901465669', '82160672595183', '25632054664849', '7789925724016323', '4416904743182493', '4417153985565311', '7788828581699557', '1075791482156159', '2140096228674669', '15951528385900927637', '2208576083287997', '4417153985565311', '7788828581699557', '1075791482156159', '2140096228674669', '15951528385900927637', '2208576083287997', '126163098812923', '24977807375118495921', '445748023717771', '55978609320057', '5155766529920351', '630992697715119', '-18589835691256', '-298976420570873', '-1544675789241637', '4683824951989207', '4676120812428953', '-3081655782803229', '-5066081557006743', '-54205060771051', '-2563816165469503', '-5964137916309691', '-8833339427908991', '-8834273017668493', '-594595126148453', '182623844098953', '3652360183259123', '-594565951468468', '-7035968566213567', '-1796904066815639', '-3518917872866319', '-8833806222788741', '-1385893673305723', '-5572606741175501', '-5703735488799081', '8621180306921207', '8606634456298181', '-5689217251410211', '-4799224117537791', '-5838742598355677', '-4857271667271681', '-1389522679299799', '-1053518409600487', '-1325252215262209', '-2686702260992019', '4048162978825933', '2019891738030061', '-1339173040041209', '-4663377115516379', '-2350079838316269', '-4730183874376763', '-5284300454525637', '-2021542356938429', '-8088223169732097', '-540076208343961', '132383270821407', '6618650248096101', '-345607704201381', '-654377313912361', '-770684851340793', '-654788005299903', '-2021799074685727', '-3436275881872043', '-6879085932679251', '-7229051612245817', '1101856067705793', '5507646083593191', '-1806446058495543', '-1432826681475371', '-910382227751047', '-718046925143029', '-6875818848211667', '3122214588349593', '703045570750801', '8358120894590889', '231506055162589', '22424528570393', '8643087091105771', '6258139385726253', '1127839260080343', '253019298240963', '3155282044320217', '227706601698039', '6352945253099077', '2317278791689097', '24557921347', '8939893155293501', '4595617487931639', '4510743438767207', '4760963487477811', '3116821419905126599', '8188149070862747', '816975617696963', '6087540380560627', '3116821419905126599', '8188149070862747', '816975617696963', '6087540380560627', '1557068474397576343', '5440196543238617', '338928416350553', '1520574682028883', '1557068474397576343', '5440196543238617', '338928416350553', '1520574682028883', '1557557512256963537', '618799474959667', '4934579686395543', '1521052258063441', '1557557512256963537', '618799474959667', '4934579686395543', '1521052258063441', '49998898440183184347', '220400002452139', '22022392427391', '1525845289312231', '49998898440183184347', '220400002452139', '22022392427391', '1525845289312231', '778780937064473391', '989095208799287', '986155139622019', '3042113035408099', '778780937064473391', '989095208799287', '986155139622019', '3042113035408099', '312015460244813713', '4850652612397363', '4840142491328803', '6094051957906517', '312015460244813713', '4850652612397363', '4840142491328803', '6094051957906517', '138705812529611', '1129997168944751', '1782335608833177', '99978260154362805109', '2519984639842257', '4478304720424613', '-2390848730040167', '-2393136701526281', '-3820671665341487', '524820681153297', '-1909763930987353', '-478169818958543', '-239313633677373', '-956797086313289', '-70838390498703', '-4540152805626281', '-7269888081240159', '4999811678427021', '-3633320087192759', '-1133414247979249', '-181606112225051', '-2268452449385817', '-288486282709551', '-4622291008351357', '-924195902050483', '635302714028523', '-3695155986952297', '-2307890261676409', '-4622291008351359', '-4619035765852087', '-8947623582578031', '-1789849920816167', '-6945624011284339', '1177077902566033', '-6945217505908643', '-2236905895644513', '-8949249604080823', '-4474218296664717', '-2308387983102771', '-4622832505293053', '-3697305353961679', '2541597227638417', '-3695791219189803', '-4616775966205549', '-924566501058609', '-4619804235749297', '-2618899483868761', '-1048442400359811', '-8326963977293979', '1141171595278891', '-520297341266451', '-65472487096719', '-2621106000899527', '-2620002742384143', '7486236442575887', '1494650412159377', '6423015375992459', '1084362832116427', '1769841222544681', '615193006694679', '1', '1', '1', '1', '6331903512171391', '36008338705624561', '6331903512171391', '7310244848147909', '7310244848147909', '6331906524597271', '7310244848147909', '7310244848147909', '9002084676406141', '6331906524597271', '1763382736181129', '1082164897109407', '247699436329543091', '7740607385298221', '1082164897109407', '1763382736181129', '1763382736181129', '1082164897109407', '247699436329543091', '7740607385298221', '1082164897109407', '1763382736181129', '8213268975966721', '270625239918589', '7748693992159463', '247958254315231119', '8660009132586361', '8213268975966721', '8213268975966721', '270625239918589', '7748693992159463', '247958254315231119', '8660009132586361', '8213268975966721', '1776155561563931', '4323494469092179', '1936193475016233', '247832788085140457', '8646990393375771', '1776155561563931', '1776155561563931', '4323494469092179', '1936193475016233', '247832788085140457', '8646990393375771', '1776155561563931', '8724985799613457', '1741568001335457', '246937090276419471', '3858391307973343', '1741568001335457', '8724985799613457', '8724985799613457', '1741568001335457', '246937090276419471', '3858391307973343', '1741568001335457', '8724985799613457', '8785745846540383', '428461576760747', '7785137953429291', '498248875585603879', '8569230807619169', '4392877288844813', '8785745846540383', '428461576760747', '7785137953429291', '498248875585603879', '8569230807619169', '4392877288844813', '1072825063473477', '2177892988288213', '7717142060908601', '493897185030410389', '4355786340374317', '8582494278803779', '1072825063473477', '2177892988288213', '7717142060908601', '493897185030410389', '4355786340374317', '8582494278803779', '49978882439741669561', '2407149238179953', '76764468440177', '3558866102993909', '932250776364091', '523705201472467', '-1355771418195561', '5058252922694241', '-6778857090977809', '-6620786687143593', '-6620786687143611', '-380501172352529', '4432822085854561', '8865644171709123', '-6088018757640461', '-3310393343571803', '-6850320210076931', '8116277723671343', '-6850319984829369', '-1680875843660731', '-3361751570699317', '-6209548959557087', '4528673345142291', '1132168207584629', '-194048401404191', '-6723503258020781', '-1355779028407979', '539593588253747', '-84736183582913', '-1652567951246179', '-1322054406205923', '-6073870083357069', '1768520898716749', '8842603807156371', '-3036935013008287', '-103285498718862', '-84412329880493', '7467342232296689', '-270119437202063', '-1659077119369497', '-6636309398253747', '-764353794632013', '4455753007492669', '4455753352783571', '-6114830472153069', '-1659077234466467', '-1369373436486131', '7087998348271111', '-3423431451813783', '-6625366708680861', '-6625363480610401', '-6078387961675749', '8844094010300651', '8844095053903899', '-189949659929667', '-3312682547322817', '-6767306216955859', '1714733604843931', '-3383683386818279', '-6660150826396701', '-6660180463590211', '-3071795033147551', '2239272517679551', '1791414820241059', '-383973427635211', '-6660165644993459', '4113173976254077', '2609154485650733', '2686542136329', '8967276428642087', '2531125847440247', '5472296962313939', '913344241274753', '5050825181949243', '7653492045091091', '6497672781212443', '6911273492255199', '7153127681773829', '4315839006777613', '3463620302362447', '3616836146061383', '5181539087196361', '6867584386794903', '899467501020361', '461671901797049', '1506547049859181', '7059195532467561', '3806332642741963', '6939693871770218699', '3806332642741963', '7059195532467561', '1506547049859181', '3693371940195309', '461671901797049', '15065470498591817', '56473564259740481', '3045066114193569', '6939693871770218699', '15225330570967859', '11294712851948099', '15065470498591803', '3693371940195309', '6625560619071573', '3468385397280353', '440607536247833', '7946910927207287', '111660899896121103', '7946910927207287', '440607536247833', '3468385397280353', '828194713586059', '6625560619071573', '6936770794560711', '28198882319861307', '3973455463603641', '111660899896121103', '1986727731801823', '28198882319861317', '6936770794560701', '828194713586059', '1847163462096427', '7601803037710551', '7059023847594693', '7544525241219579', '1734915279120018709', '7544525241219579', '7059023847594693', '7601803037710551', '1847163462096427', '1847163462096427', '45610818226263347', '169416572342272591', '45267151447317433', '1734915279120018709', '9053430289463503', '169416572342272673', '9122163645252653', '1847163462096427', '3284269863743769', '3967732596330099', '7053987267904259', '693114072360617', '13957728847284852411', '693114072360617', '7053987267904259', '3967732596330099', '6568536089508627', '3284269863743769', '7935465192660199', '5643189814323407', '6931140723606169', '13957728847284852411', '6931140723606171', '28215949071617037', '7935465192660197', '6568536089508627', '1222765179750797', '3613953010393369', '880988535129681', '7788780511617449', '13926442518307460777', '7788780511617449', '880988535129681', '3613953010393369', '3056911857983307', '1222765179750797', '10841859031180117', '21143724843112339', '23366341534852327', '13926442518307460777', '23366341534852367', '21143724843112349', '10841859031180097', '3056911857983307', '6127215013769727', '1559087867521617', '110111571083467', '7224145684638819', '13926460166241109007', '1444828700370297', '110111571083467', '1559087867521617', '3063604960299641', '6127215013769727', '46772639025365107', '33826274036897743', '43344871108116317', '13926460166241109007', '43344864010825507', '169131376183921909', '46772633025931913', '3063604960299641', '1090808735798869', '3475798042893917', '1942634750465697613', '1461347878693', '2143280269391171', '3672279091848223', '-5475565022456091', '-83575034686111', '-342222910524497', '-136352264273607', '-5520597832005657', '-2727046058440069', '-5070355926216793', '2689264634331459', '4379363988403683', '268926386136353', '-5070355346490843', '-384010379557057', '-2832748559840307', '-2246432889533411', '-5665497885699781', '-2694456193622547', '-6705543638551', '-2694456576632131', '-2581820560044613', '2763756616038721', '2224986332925277', '221100498642331', '-1032728166566407', '-3113552862536777', '-2728507502012731', '-5495531863386713', '-2728507502012723', '-5472402565768283', '-671553921180521', '-2736201282884133', '-1267531626794899', '5378225157614397', '8758324384155189', '5378225157614407', '-5070126507179599', '-3071978243474397', '-2696579883078361', '-3361067509159379', '-2696580365996461', '-2834232035012539', '-2259850293685983', '-2834232517930639', '-206874630556353', '1385578809965387', '4455860223975011', '110846285480507', '-646483175215031', '-194794597698896', '-5632023156922747', '-5130338953241177', '-5632023779524669', '-2739987274543071', '-198335244247941', '-684996896461007', '-656099066140457', '718461159958081', '8974164843800623', '5747688657062739', '-1049758459129587', '-3136657918689431', '-5477668723009963', '-3176473411161499', '-2738835095813473', '-5632514868815519', '-2557088394931089', '-2816258276968303', '-5246672337525447', '229740125352943', '8971397789167641', '5743502448092431', '-5246671857591793', '-6271787810108833', '3079636068304337', '8226274683294259', '5162638867885789', '1032585673312009', '412069254287527', '5729744611979117', '2194002383277313', '2057059534527327', '288818534599593', '4556430060296981', '1317930286005219', '643813701788633', '384804738288563', '1646130628821347', '5160804241481847', '5165187608247919', '4118509048265927', '1145395783018153', '2173804622512507', '291013434880141', '227628966088509', '451453258321559', '8237050074191543', '6441195529446647', '8130815394908663', '589785171372569', '783713619781151', '7772617507365527', '255956183952703', '1945650409776211', '24268991980767', '294403379440981', '3884887338327621', '7839968861696863', '48733795964089', '7782592731208763', '4640488204507879', '49125100630247', '8754478663560373', '5207658552543589', '982109445592485889', '239772813865353', '2603827821080289', '8754478663560373', '7860017556031027', '4640488204507879', '297970041985851029', '402237066793420453', '281122096032214873', '40682471523001939', '7857854343538170049', '7857854361637233877', '40682466339001687', '281122114131278709', '2011185706496128057', '297970023886787193', '4410034619656049', '314462923118083', '920811136585331', '5188548315157809', '491029012313303311', '959041039674421', '5188548315157809', '36832454194563', '491348362846743', '4410034619656049', '28313230747379219', '201167261240776411', '147774831493717523', '20309147352728709', '3928677148146491051', '3928677144414330943', '20309143620566673', '147774862686154527', '1005836399336146119', '141566157469058131', '4703449082199573', '1971965198778891', '8613300712250427', '5080636191527057', '490257717606961573', '3830138418804387', '5080636191527057', '4306649264731539', '7887862250307131', '4703449082199573', '301582167046697509', '403746188342768229', '55237408135515061', '40083663746291619', '7844684899596946809', '7844684907497309413', '8016734329330963', '276186978728743333', '2018731314242882297', '301582159146334313', '8990666489865323', '7988529207944831', '8355804994426783', '1166069608311029', '122148007431671873', '7634249009287981', '1166069608311029', '8355804994426783', '7988529207944831', '8990666489865323', '18047026048821107', '1022006195429750069', '67109210866240719', '2266446147531597', '1954630889817576423', '97731525932704249', '9065785955661777', '16777303057944027', '1022006195429750069', '72188102829749039', '8672468933286757', '7888345559691773', '4669798547863783', '5077502451537199', '490260058806066053', '7660313418844781', '5077502451537199', '4669798547863783', '7888345559691773', '8672468933286757', '278075185238238701', '2018860283908032171', '299423286436343069', '8012768047847023', '1568943424054023561', '7844717120270118221', '8012768047847327', '299423286436344589', '2018860283908032171', '278075185238237181', '4204163520193161', '7989308254388839', '8930024873951069', '466573465543987', '488582561974678191', '3817051265427173', '1166434209556821', '8930024873951069', '7989308254388839', '1051041152896717', '270146462955170359', '2044182912263319363', '57368160004814599', '7249175916142181', '7818400995652489843', '7818400989257658359', '7249178129635897', '286840793629242263', '2044182912263319363', '270146539199198339', '3250571093341839', '1312986264819657', '563914850397701', '42953481237180436557', '115800400579041', '4590775532598429', '-247647556860289', '-3463121014929257', '-619118844350333', '-2703288149572327', '169603497023913', '339206971107691', '-5406576494969869', '-5317764530346633', '131242199633127', '1049937520844971', '-1329441084324687', '-1077168214574077', '-990602803679311', '-347083921500207', '-309563341731677', '-5330835681219983', '2637401546779433', '527480308343673', '-213233442088191', '-5396330763526341', '675724086807311', '2702896345067223', '-1079266114739423', '-5386868644926959', '-2473533253179973', '-1785980187664483', '-4947066513147651', '-1088249104847767', '5488929377245141', '5488928813764813', '-1360311281984017', '-5307022375286349', '5220483683905301', '326280194642383', '-5307022580475319', '-5393561556669923', '-4960703392427563', '-2005956054188873', '-2480351695121559', '-5520823968650123', '1394863898082539', '2789727506963023', '-2760411869680273', '-1352376213038021', '16740053873466', '5356817820097671', '-338094067658369', '-5462192404882563', '-4947172665316047', '-3573846665245107', '-989434533063213', '-5318167875760647', '5242432253049587', '5242432253049537', '-2659083937880309', '-5431032131050519', '1093632152725867', '2734080381814683', '-5431032131050561', '-5393903498471701', '-310043854167331', '-4014321481970807', '-248035063502159', '-5419842561395867', '1075438600903013', '5377193200335977', '-5419842194636249', '-5511338278538119', '2780092684069857', '556018556895291', '-2755669421807407', '-2731245826803297', '983792630809537', '2756625144555833', '823921166287861', '2801931875318519', '175087399714261', '8118602740515823', '5578082365538917', '5526617941227599', '8233572728862711', '7861586604376507', '2041868139492041', '1915940844589879', '247253330765011', '2776288413594951', '7562357388431421', '5279789025181713', '94094111708013', '8158154008230611', '674973725748251', '502021969227051', '7092692899638811', '5438894089289933', '1520829347445649', '4079077936020913', '5252827806530823', '5552956986097579', '3780465548855721', '1242514465124929', '8208709978630379', '16724090355193', '2663144802483071', '2517339103249589', '7105827924923581', '2713195637239651', '4104360613882161', '739416335933967', '3578928920805111', '6501965193387067', '2962438856877293', '1493926378976627', '56362670135920835483', '7469624618925589', '5904593799306379', '406373188384569', '3578928920805111', '85894294465908661', '4876473803393801', '2962438856877293', '1400555957378963', '56362670135920835483', '5602218555840691', '5904593799306379', '19505913409045309', '85894293732736667', '1436999644834309', '7070522679990899', '49920097515167', '7156146871295529', '13997415299984255941', '7156146871295529', '6237186207639881', '7070524135182329', '7184997496575831', '172439954464524877', '42423138995537597', '49920097515167', '42936884143365377', '13997415299984255941', '42936878312180971', '6237186207639881', '42423141895501771', '172439942833412147', '7155198350015887', '6764859969390499', '1125856111823179', '3667098180587673', '56243656527922856147', '7334182536856257', '558769482450911', '6764874521305331', '7155198350015887', '7155198245746587', '6764860386467699', '1125856111823179', '3667098389126273', '56243656527922856147', '7334182119779057', '558769482450911', '6764874104228131', '7155198454285187', '143098948941731', '3669161338620521', '2735661625769303', '6763304480859417', '56242334229265851163', '6763300842880649', '5462376734153783', '7338328498007071', '143098948941731', '42929684082410453', '1100748461597041', '2735661625769303', '4057982928559189', '56242334229265851163', '20289901328424253', '5462376734153783', '22014984293803519', '42929685282628147', '3591314905351193', '1786635265925317', '1254040884623067', '1770413155869603', '5599773693166085809', '6915677811201', '1254015709811651', '1786635265925317', '3591314905351193', '21547889188210063', '2679953020836523', '1254040884623067', '331952481969119', '5599773693166085809', '5311240315105273', '1254015709811651', '669988194234857', '21547889676004253', '1789995870145479', '1490969929481441', '733830700384851', '81297888961853', '56364437277874494429', '6503801285523949', '1448371674227041', '7454881661618641', '1789995870145479', '5727986875029523', '3727424597293627', '733830700384851', '6503830664128289', '56364437277874494429', '65038017383439', '1448371674227041', '116482533038103', '28639933469507713', '593404215031311', '2289378007669869', '1648725407587073', '45658488306387', '99919096226337498607', '448748904274447', '-3169247977225839', '-7232728643524679', '-6344475544591401', '-6607442049283533', '8544678597461447', '5319009654284177', '8538693313848907', '-1651299749452323', '-1211245616659097', '-4745435910583189', '-606220762133717', '-7245576829962127', '-308505904927999', '-4111379625646453', '-3085454077866043', '-262790990893581', '1704055495695713', '1056026627025589', '8519487004081887', '-821184812525919', '-6145628598399993', '-8418676899306623', '-6146418243888639', '-7198358033557643', '-6293176953016151', '-7793844705269369', '-3153052764058001', '-416699065901657', '1741873596395309', '26705678282317', '347857081208057', '-6662335269443753', '-3064390084826351', '-1138627282578477', '-6141708922639839', '-7273871828724751', '-307205144543417', '-4579861368640131', '-768369507111371', '-836307099597173', '4380369524000187', '5354940516427829', '1094735213271227', '-6689386316212831', '-394382321143061', '-7831608075352549', '-6312970474952257', '-1458098963207937', '-1229441720189999', '-8382106143096577', '-1229448725875563', '-262580157586741', '8508889526237527', '1319265989858643', '2127213763319579', '-6564490856170493', '-6165765391671861', '-65869210117151', '-6165800558269129', '-3597494691525489', '-11917841637777', '-4555721590616507', '-5981988877494271', '-6405294111008243', '1011795206575807', '5205834658645259', '8071266641559649', '-399790015256331', '-311829287010333', '-430632786418671', '-3129826802904841', '-1421877049169103', '107468806696267', '8972926865689971', '8389840308639177', '4793059574559197', '7498713447370717', '730424198292769', '2558544781276523', '2559265146824919', '8229796068313131', '1760617320497409', '8232976699730061', '451461399607013', '3765694031622859', '1493518705286913', '994826884705153', '5287954335081983', '3703653508575733', '7207057527809391', '7471749652403749', '7538251494845283', '446500153004973', '594833556090961', '773083080556681', '3602767591345321', '159600709681251', '5109397166437343', '8227123892983281', '19252296388501', '8228833839281493', '4990453203141297', '8973976003932183', '68693290487609', '972785357887081', '4657351550569217', '8474908604693777', '1461662214611413', '2208126985488599', '172478564773343', '15953791773328619413', '1557987041306649', '1715402572422581', '4416256153764569', '2208126985488599', '172478564773343', '15953791773328619413', '1557987041306649', '1715402572422581', '4416256153764569', '2207617194935649', '1234564204560523', '797849131695876473', '1947873856679387', '1231798067174541', '353218809397369', '2207617194935649', '1234564204560523', '797849131695876473', '1947873856679387', '1231798067174541', '353218809397369', '4381507471970273', '3243975524440111', '4000232751714675233', '7812862916126547', '1279074789348429', '219078738728919', '4381507471970273', '3243975524440111', '4000232751714675233', '7812862916126547', '1279074789348429', '219078738728919', '139337807429381', '6344365910195533', '792792524095738583', '7742035185183441', '6184583696159309', '1114716920400991', '139337807429381', '6344365910195533', '792792524095738583', '7742035185183441', '6184583696159309', '1114716920400991', '353973654873483', '11853232637139', '7782287335885177', '1593812893424088907', '296103987965321', '2212335161060341', '353973654873483', '11853232637139', '7782287335885177', '1593812893424088907', '296103987965321', '2212335161060341', '278732903129943', '53735816870287', '396397509656257537', '7742067556084967', '326485697883009', '557472581995813', '278732903129943', '53735816870287', '396397509656257537', '7742067556084967', '326485697883009', '557472581995813', '6540563281106069', '7312994197285037', '53059359937383', '59159684049617', '799469780282063189', '494373610631159', '-5839856245249191', '-4765043636576003', '-585317343181763', '-7417897795393153', '5657621704052079', '226238258594349', '-1482913649471271', '-7034446124411749', '-7041105370943073', '-1759443936919353', '-6345566990718407', '-5763360378397461', '-6351042953890829', '-4146690919729231', '6345753345988591', '3172534362432819', '-8292012764847759', '-1557555164931617', '-779051430914989', '-778914506690399', '-2264865331693659', '-8137072304087011', '-1141200323531107', '-1275456060933673', '3835680509245349', '7662552017822189', '-1018609754660481', '-630882475821651', '-2532303094474707', '-157994781180041', '-4814242630743707', '-1477566611399043', '-487650282857933', '-2867378197466213', '8669663361989159', '4327031443396141', '-2859592373569407', '-2784293410815933', '-5599725249090213', '-5584156035361041', '-7037074044800211', '-7183260777260891', '-3519471686198797', '-2378323294262751', '3652360125448811', '1826121715416849', '-4756412943485539', '-1104169533561549', '-8834291537335863', '-8833823902914131', '-1223046318528509', '-312023524088363', '-620150343284701', '-5878840974651427', '8895688472747313', '8878401505205781', '-366348776533343', '-5689493959414197', '-715501460177409', '-178336025638023', '2300894473707551', '4639410367382031', '7920915885365607', '463635605046013', '140948836648113', '1970691423021', '1269316872610357', '7936021222296033', '359735506135903', '392447281352431', '1337781259533697', '1421337804349227', '2315809582258621', '534701745462311', '9847682295519', '6279411453565613', '309830349318453', '1135955827601563', '1541511001591413', '9633493229116107', '1541511001591413', '3502453442522047', '3502453442522047', '1541511001591413', '3502453442522047', '3502453442522047', '1204186653639513', '1541511001591413', '4949817586556313', '4949817586556313', '8675092560752271', '1050391748158643', '8675092560752271', '8675092560752271', '8675092560752271', '8675092560752271', '5251958740793213', '8675092560752271', '5251958740793213', '5251958740793213', '4949817586556313', '8675092560752271', '8675092560752271', '4949817586556313', '8306366812929733', '2461532958232659', '3178692244028751', '2575470085096059', '421965137951423863', '3178692244028751', '1230766751964731', '2076587701455879', '8306366812929733', '2461532958232659', '3178692244028751', '2575470085096059', '421965137951423863', '3178692244028751', '1230766751964731', '2076587701455879', '1624718866068427', '4933122416711453', '6075685376589447', '10622405766884278749', '1037344313172293', '6075686104185243', '4933121689115657', '6498868915911543', '1624718866068427', '4933122416711453', '6075685376589447', '10622405766884278749', '1037344313172293', '6075686104185243', '4933121689115657', '6498868915911543', '8303319075331291', '4965169795227659', '1586823942149189', '5150887155543727', '10549019129727699113', '1586824124048127', '4965169795227659', '2075825767056191', '8303319075331291', '4965169795227659', '1586823942149189', '5150887155543727', '10549019129727699113', '1586824124048127', '4965169795227659', '2075825767056191', '2276146844907773', '5281236208415493', '1249691374551283', '10553583175962194171', '1030588723470387', '3123900654497839', '165204943982737', '8702771177639121', '2276146844907773', '5281236208415493', '1249691374551283', '10553583175962194171', '1030588723470387', '3123900654497839', '165204943982737', '8702771177639121', '30681434589261', '4972770379700029', '196688167457199', '2578758383958277', '10562624143015788061', '1573518618280371', '1243083546509173', '315513756025371', '30681434589261', '4972770379700029', '196688167457199', '2578758383958277', '10562624143015788061', '1573518618280371', '1243083546509173', '315513756025371', '991784400106837', '5279813826203939', '3111991043406469', '10560890095059400831', '5156641688579081', '6223822015748337', '660138936652603', '7836380830915353', '991784400106837', '5279813826203939', '3111991043406469', '10560890095059400831', '5156641688579081', '6223822015748337', '660138936652603', '7836380830915353', '2389318285049641', '9999387835427383273', '67200058230749', '339984787194793', '75887704559851', '12524672076671', '-7854020912620129', '4335653607451271', '8671304594340737', '-392701248678037', '-5686195277213199', '-5686197656656849', '-4467663254216249', '8640945258164761', '3931991388658937', '270029349656143', '-8935323847610733', '-227447858677401', '-487230419696247', '1937082836943259', '1549665967100763', '-7795689480489879', '-5858605260871649', '-1171721127787793', '-59092243363099', '4791817957066273', '2064773427365711', '4791816470003099', '-4727378717403497', '-1464651362476327', '-3911504262913313', '341946996161539', '8548673147849373', '-1955753282275261', '-5685841454833811', '-177682599597559', '-1119060469664691', '4340374887855249', '1575749805180743', '8680744033737827', '-4476240520963059', '-113716846419557', '-3754893220846499', '7763528124191447', '3807242873194721', '-3807185044203031', '-2810649394222463', '-5658354274008907', '-2243292301523261', '4357969590807927', '1979613208741179', '428429390147719', '-8906241906241149', '-2819913265613459', '-3908040366429207', '4173579135573419', '8361114413075679', '-487889686072427', '-5724359676710147', '-5720887862251459', '-4531608754618383', '1111119612695237', '3981083863505599', '8902775588206217', '-4534753554848631', '-2861311884740403', '-303063498513169', '1502945050870949', '3736970519449179', '-7605417097299317', '-2856173185862593', '-572249161509007', '-4594002663548949', '4588957444322867', '8061326521973983', '4568773121968487', '-573100341964337', '-5717418993407631', '522943529377949', '8931365665059831', '2314149415284531', '1645246670443613', '3528020647928369', '2295662245753209', '261079244581003', '8950208028092469', '578503268043489', '2679579240891397', '7974036469020969', '1879998294471071', '5202793969629577', '7972358981435543', '3766830988207463', '5194738364218599', '7922673244395307', '940918410998669', '2576758970698741', '4373304314739053', '187810456814949', '245061294262084579', '3941818950745999', '3941818950745999', '3829082359047213', '187810456814949', '4373304314739053', '2576758970698741', '10166244363119911', '70113660555499901', '75264974245654653', '392126228972978739', '252206017275770907', '252206017275770907', '1960630959351848109', '75264973494196663', '70113659804041911', '10166245114577901', '2371931269138621', '4237863301075661', '8995701499867351', '9770520820834377', '7981536245111627', '7981536972707447', '7633219391276857', '8995701499867351', '4237863301075661', '2371931269138621', '921763425354591', '1361518072804383', '36117851410973691', '1954374254989883979', '1021096457728271103', '1021096550860536063', '977187127494941983', '72235702821947387', '13615180728043831', '1843526850709181', '5258199435928713', '8888714817073411', '2314916420056511', '98162867814381311', '3929840628367661', '3929840628367661', '958621755999817', '2314916420056511', '8888714817073411', '5258199435928713', '20583323726641323', '28533782218049621', '148604124900690233', '3926964186592326919', '251397431711261803', '251397431711261803', '3926964186592323961', '148604124900691183', '28533782218049811', '20583323726640373', '5277161624359013', '2343406865125061', '8782634188976819', '98189139531801467', '7854788831662461', '1963696844117749', '479439112395391', '8782634188976819', '117170370541093', '5277161624359013', '41258204936592399', '300915166794287513', '282003382105537913', '1571218048026868517', '2009866858081337551', '2009866485552322511', '7856089505544365849', '282003371637483433', '60183045235084661', '41258215404646879', '4744469543557897', '2271692674635353', '8392102980171231', '488536360864971623', '7980739235957261', '7980739235957261', '1908345159628797', '8392102980171231', '4543386076866499', '4744469543557897', '7376486450321669', '58369997290036003', '269620619462334223', '1563531018659000143', '2041995922628904027', '2041995922628904027', '7817655097936407343', '269620614820934139', '291850028374910683', '36882436893008429', '1032480408365597', '949504947196231', '8656530241306227', '490178982954674947', '7880833086895667', '7880833086895667', '7659046608666797', '8656530241306227', '4747523644587461', '1032480408365597', '10182466109514453', '76102733749839997', '69394579904591333', '490214567902513807', '1008461957357148419', '1008461957357148419', '1960858269792841549', '8674322715225657', '594552485193397', '1018246429230053', '147092335281011', '7634219454676693', '5706208134049077', '1895794470710003', '65446343060345256421', '5435993611822653', '-270708145527261', '5446228887471321', '2723114299638347', '-2707081399018311', '-4939954744653239', '-88428444222859', '-4939954743533947', '-66324777624131', '130746685576609', '1307466928094569', '-1061196464599747', '-134552424130197', '-549442358344209', '5539528109153981', '1384882073679327', '-5494423583442089', '-619208223153649', '-7930449391003499', '-2476832985396271', '-337745125742819', '2679262483019973', '5358525151603287', '-5403922011885111', '-5449318964948581', '-5381363952299773', '5386869160263593', '2693434580131813', '-5381363952299821', '-2472642726902543', '-3444586324247561', '-2472642726902547', '-2659585135036667', '1315620448952651', '5262481795810557', '-5319170270073293', '-1343964686084007', '-5298998227527519', '2611592922950931', '1305796251143447', '-2649498875471523', '-1236618738733063', '-6853369727569949', '-4946474935587089', '-539804178601647', '1084254690363271', '5421273181202281', '-5398042083725651', '-5374810553232787', '-1077692004107649', '332984793613937', '5327756576903873', '-2694229883575611', '-3963039496943', '-1981454625664669', '-4953799616641091', '-2754673142298539', '5569529611795533', '5569529487252311', '-2754673207626471', '-34057269689538', '-5288319919961973', '5195439486023943', '2597719830009251', '-2644160140344709', '-4940323534116819', '-7054036325379323', '-4940323180471207', '-5429121135820217', '342315085642503', '54770415499311', '-5429120951915589', '-2690600313815107', '1052357593458241', '4969726860078691', '148809165717993', '1645246670443613', '1892639483407777', '5576110714288577', '2683420083852791', '2714001874194171', '2991639880880287', '4113127008865673', '710399319513261', '1010798100246587', '5591438181797413', '3924471991298043', '7114483021717057', '8185865329805463', '4121652463948439', '694476406114771', '9841627517061', '5586891535224441', '8168148021298679', '5811371199204019', '515469217854547', '5529419479373303', '1354310196353973', '2701526395439613', '8208702479215939', '188572841890669', '7094073317637413', '1009852197858289', '2480343894831411', '2635508578233701', '8208679981011923', '627125953745421', '7564371618889727', '5577904491557469', '424043856672571', '1178129764656387', '7984896940234687', '61068659413692067', '8932998582602971', '8932998582602971', '7633582426711507', '1596979533566087', '1178129764656387', '424043856672571', '272477827235621029', '7322078700765759', '2043043857734892137', '7817878163917784457', '286945713608470661', '286945713608494953', '7817878163917758757', '16344352351995229', '36610393503804503', '272477827235645321', '546965187973577', '1023960925486513', '7886192627304933', '980336274765909399', '233261393596367', '2332614299761571', '7658877146608667', '7886192627304933', '1279950611161289', '136741262887341', '70150461559459861', '16161106809637', '100915482396330103', '1960811464364707187', '14956512689935889', '74782572425258661', '1960811467030660757', '100915482396330103', '10100690056401923', '70150441431206981', '2271620005940059', '4713506865873179', '249464100471163', '488579655234476637', '1043089662696647', '1043089662696647', '1526811131569457', '249464100471163', '4713506865873179', '1135810548666809', '291823082518362057', '36652333168950927', '2042554198982617599', '7818330186147891081', '268086675408376137', '268086656046606521', '1563665743078730869', '2042554198982617599', '36652352530720543', '291823202854967993', '2375943033895149', '51339753410761', '7882480691282599', '122558301903610197', '4300728311853311', '4300728311853311', '3829946934487819', '197061962712383', '51339753410761', '2375943033895149', '304694980512149191', '40497530555038681', '2017340784794761923', '7844305594004649331', '275820884132182023', '275820884132208627', '7844305594004623431', '2017340226001218499', '40497530555012077', '60938996102435159', '4452237802063913', '1313804470283203', '7858780147385037', '30683234827592589', '2300089554772983', '2300089554772983', '7670808706898149', '7858780147385037', '1313804470283203', '222611926482981', '5719064417261159', '20515870759047489', '20108377187942571', '785591811220536001', '147710732270954671', '5908429187051981', '3927959058697336047', '20108377187942571', '4103174670740527', '142976631119936453', '187773411641061', '211760846154393', '15338151597871', '196419892223809047', '8737820186890979', '1747564473935639', '7672652039992541', '15338151597871', '264700930363737', '187773411641061', '150793410763221111', '20601403165066989', '100462642760882271', '3928972514044310921', '5615192177625119', '140379827482981101', '3928972525926553303', '100462642760882271', '20601404860968979', '150793398880978781', '97132183365641000997', '2504803794435641', '8394596612072411', '2307085670667997', '2444569058351737', '25976395464201', '-1373861111739683', '553612544938677', '5536125634873431', '-2747722223479711', '-1354125468800019', '2689120152934353', '5378240491354101', '-5416501875199381', '-2478273502259021', '-7971460072431897', '-1239136797501021', '-1090952670357611', '-5415123850813561', '340194065925269', '5443105369433939', '-1083024852561897', '-5317555494613899', '5247967833042761', '5247968139372777', '-1329388848274449', '-617854910957017', '-7108862789241809', '-4942838666696493', '-2693571450751963', '-5390250252273551', '5326407693544213', '5326406749752051', '-5390249261366681', '-5519701404269437', '2792655517778779', '5585311239934011', '-5519702025468451', '-2478352278103241', '-7958197857244319', '-4956705295622189', '-681761536499013', '-5288352654128491', '519094212743977', '648867695486541', '-5288352654129171', '-5440268437524893', '5494773694231853', '2747386565341567', '-680033554690527', '-2471585829073979', '-7081473379483283', '-617896386824983', '-1077152692518323', '-26892824960579', '336142516727389', '1075656029171487', '-5378564737237953', '-5328859728743147', '5278870128867829', '2639435001526983', '-2664429929911837', '-4948140273282829', '-1722836502103127', '-4948140520877467', '-2689424761302963', '-661938852831943', '5214688117680787', '2607344156130013', '-5295510789719973', '-5404701065262329', '2716534431672369', '5433069039464157', '-540470128354723', '-989718435066913', '-6843862518718109', '-2474295902317953', '-5376333397405113', '530736349324263', '7107377353379599', '5044528922443841', '184089888273621', '4113133584269183', '339600578839831', '5246225064303019', '7563789137896341', '111169648193519', '45541291016233', '411313828099213', '24880399718859', '2698044503793281', '7084331173798429', '2512864007812903', '1631623719178993', '305395636827579', '5454791718416259', '1235736428448091', '7558874435654299', '347555793624481', '8158193148351581', '62599625235879', '5284180330032259', '558862461600419', '329843201733531', '1383586040851543', '6825211478461291', '8189506731275869', '3949911685209043', '991464831694207', '8262645733877799', '550227453915309', '8127255937888213', '7855295172650151', '5619497080501891', '1253752098365501', '66850903349353', '21486117109423245331', '560408338691541', '560408338691541', '5245632604726251', '534796458377473', '5015009848653537', '1253752098365501', '66850903349353', '21486117109423245331', '560408338691541', '560408338691541', '5245632604726251', '534796458377473', '5015009848653537', '62919472823877', '7764203135368957', '2134420182617176349', '1468578402264463', '2937138250837819', '2605489567106277', '1536092100972019', '1006768390408521', '62919472823877', '7764203135368957', '2134420182617176349', '1468578402264463', '2937138250837819', '2605489567106277', '1536092100972019', '1006768390408521', '1265424566099417', '240880750087027', '21373087498064804409', '1161486589536311', '580698765909973', '2608959079533693', '1368105693812451', '158287244226767', '1265424566099417', '240880750087027', '21373087498064804409', '1161486589536311', '580698765909973', '2608959079533693', '1368105693812451', '158287244226767', '1074323769404779', '473478168100913', '667806780611731527', '5732794508216943', '5732624614589903', '5217196817779163', '460644696371569', '5372945981823563', '1074323769404779', '473478168100913', '667806780611731527', '5732794508216943', '5732624614589903', '5217196817779163', '460644696371569', '5372945981823563', '5079213359457511', '42334801254787', '42689005676941247337', '2931620552603269', '5863231282663497', '5211057716713259', '6751975340779447', '317455473389197', '5079213359457511', '42334801254787', '42689005676941247337', '2931620552603269', '5863231282663497', '5211057716713259', '6751975340779447', '317455473389197', '133524562037257', '1385770324295059', '10676196857416866079', '2883844689460779', '5767473646780423', '5212933268497643', '5418598758574933', '5342648675767513', '133524562037257', '1385770324295059', '10676196857416866079', '2883844689460779', '5767473646780423', '5212933268497643', '5418598758574933', '5342648675767513', '49997808283857362793', '1567533454182377', '15966188570119', '288700503928271', '2039257449539647', '46496372100443', '-4004835346087309', '22588837151279', '70589028316571', '-8009717748277091', '-3160179188925821', '2042822497851441', '1278733618566083', '8171256875584887', '-3160171672808007', '-6880253209250287', '-1720067230143089', '-6880261064911319', '-2001000123773873', '5565617955093809', '2765537981189899', '-4014753013027439', '-1487492109456469', '932233236939653', '4852462438978769', '7440642402793571', '-5941954336176087', '-6625353245944657', '-165849494895933', '-6629666520890989', '-3864628984878567', '74722351680197', '5671580423215243', '-497122002130993', '-5603108669266697', '1663403952734793', '1163515997463261', '1300224248532721', '-5532356550098363', '-3173614303779863', '-160590981422139', '-3192716966111321', '-3752608411781399', '2662427809983271', '654572701302087', '-3784815764995107', '-2688908152246853', '6151935592518691', '285106097139886', '3054015271235959', '-535749504498143', '-775779475970253', '-3892658273872', '-6217244522978609', '-8021015261447229', '5255413727334993', '655656866310697', '-8028528987890271', '-1521420841311337', '7762355248489977', '2466501004581987', '1551457811572303', '-6083323991727093', '-1342184036838869', '-209795528146501', '-3356094271220593', '-742600252357977', '371507725792189', '5870417853152217', '-7479024536988913', '-624227921671101', '5303995083606637', '4340130508298089', '2633716557800843', '-1244253027064833', '-2983256967119571', '-5984877759872737', '-5975695847055939', '8019719505386679', '8927264433386019', '8226227716177601', '1132756474600779', '8989730204894201', '8455334479346087', '1475033234361063', '7264035629063969', '4926244315107743', '1471772536505343', '29022246432771', '8630401559198877', '138690808955659', '1798642057275361', '839991566431211', '3715525799602753', '7250711048040159', '8246633636306039', '5507231955872607', '5507231955872607', '1078072872381291', '20881108766269261', '1078072872381291', '1078072872381291', '1078072872381291', '1078072872381291', '5220277191567317', '1078072872381291', '5220277191567317', '5220277191567317', '5507231955872607', '1078072872381291', '1078072872381291', '5507231955872607', '5787712450442661', '3105366703938471', '3105366703938471', '1348698078772443', '19771240266783', '1348698078772443', '3105366703938471', '3105366703938471', '253071739710919', '945978318764956947', '253071739710919', '3105366703938471', '3105366703938471', '1348698078772443', '19771240266783', '1348698078772443', '3105366703938471', '3105366703938471', '5787712450442661', '4535221305864673', '5108202668569589', '6559194307921257', '5242425419885099021', '2559778037053271', '6559194307921257', '5108202668569589', '453522530764181', '4535221305864673', '5108202668569589', '6559194307921257', '5242425419885099021', '2559778037053271', '6559194307921257', '5108202668569589', '453522530764181', '619990561901499', '477363086773467', '6618510397651211', '5245151243855022873', '5122171809244157', '3309169342521737', '477501839291667', '4911440057985807', '619990561901499', '477363086773467', '6618510397651211', '5245151243855022873', '5122171809244157', '3309169342521737', '477501839291667', '4911440057985807', '1175281362721033', '5105341647514143', '6595590217259017', '2618846160084268901', '1278733476603647', '824448868106849', '2552666458182421', '2350729708672441', '1175281362721033', '5105341647514143', '6595590217259017', '2618846160084268901', '1278733476603647', '824448868106849', '2552666458182421', '2350729708672441', '961286430293371', '4877856959112921', '3332731925328013', '5235922771181554537', '5113205831231987', '3332731925328013', '4877856959112921', '60080429178179', '961286430293371', '4877856959112921', '3332731925328013', '5235922771181554537', '5113205831231987', '3332731925328013', '4877856959112921', '60080429178179', '8019639821566867', '4864288731239931', '3188762651619449', '5274009725158024397', '5150400122224633', '3188762651619449', '2432143638024201', '8019639821566867', '8019639821566867', '4864288731239931', '3188762651619449', '5274009725158024397', '5150400122224633', '3188762651619449', '2432143638024201', '8019639821566867', '4846082044886819', '605226382225169', '266909713591063', '2556691449702093', '2618052789552996701', '6672743567372331', '605226382225169', '4846082044886819', '4846082044886819', '605226382225169', '266909713591063', '2556691449702093', '2618052789552996701', '6672743567372331', '605226382225169', '4846082044886819', '3674096175391117', '1229473268991889', '3428287778067403', '4317495989212917', '49979413883146218003', '3585229814817507', '-4402129392271881', '2135406502075747', '7725132400337309', '2135405807159309', '-8804257481575447', '-5612571209088633', '-1122514380801011', '-3899960183406747', '4374700400199017', '8749398020732347', '-7799922451562791', '-5612571556546843', '-4251876734636351', '1568067443596041', '7528005863286043', '984098887287621', '-4259170103443161', '-111397765874917', '-5561723173859523', '-4010782756838623', '2440340571838233', '4897121491192553', '-1999736817840657', '-1391451433450671', '-2182139034653587', '4199453605961349', '1917445096718483', '8398674567179039', '-2182112697579623', '-1117560369672491', '-5587860346204499', '-1951517096985111', '177467898445427', '221829023272579', '-121972387030002', '-5587831097283473', '-8595595260714007', '8104029692757139', '945362327973921', '8104028262128573', '-8595594590106879', '-557627655947953', '-5576276917136651', '-1594326743558453', '4790715389596867', '1197678668570659', '-7971634790763661', '-697034592288511', '-8980141802265289', '874386160847431', '3939869512607039', '8743859959259297', '-2245035428478621', '-5708613892880433', '-2854307152592081', '-791154045418419', '8811705067204333', '8811703417989407', '-791153986517883', '-2854307049516147', '-342617007069109', '2008470839182619', '942767155216613', '8033882682227547', '-267669530570586', '-1392942896765357', '-2785885609983353', '-7995347921149637', '1211788593825307', '4847151635554533', '-7995347891306377', '-2785885701757033', '4929834820792337', '1924818939538927', '257468367674969', '3811630247530427', '7664895222087459', '515417311204727', '2183404109627927', '961756763202943', '8257657039570027', '4112927872834573', '58910924000311', '6530106619663611', '8208706228922341', '2335175871207643', '1053859337191147', '8230076112202441', '2355352459225543', '6995743164086139', '1507267545935083', '1009979740462143', '3760157308136301', '349225764778549', '6919572305821855337', '349225764778549', '3760157308136301', '1009979740462143', '1507267545935083', '1507267545935083', '6059878442772853', '11280471924408913', '4190709177342589', '6919572305821855337', '4190709177342587', '11280471924408893', '6059878442772863', '1507267545935083', '941668785259589', '7523211235347907', '504684011225507', '3492563266051169', '34597589860835779', '3492563266051169', '504684011225507', '7523211235347907', '941668785259589', '941668785259589', '45139267412087341', '48449665077648773', '167643036770456213', '34597589860835779', '167643036770456011', '48449665077648571', '45139267412087543', '941668785259589', '2081857350542107', '8243695500463341', '3604539177263127', '279526367381153', '1733945096802357253', '279526309173493', '3604539177263127', '8243695500463341', '2081857350542107', '2081857350542107', '131899133639361', '2883631201011813', '55905272772237157', '1733945096802357253', '55905262538692043', '14418157413045951', '16487390296933239', '2081857350542107', '1208267255029143', '1578814403956973', '7839387377697789', '6983747133738589', '172519310836889309', '6983747133738589', '7839387377697789', '61672449023253', '1208267255029143', '1208267255029143', '47364434025437449', '1881452894378339', '167609929302997877', '172519310836889309', '33521986623290879', '47036326172914993', '9472887788626009', '1208267255029143', '4277613679497537', '7213608447124307', '4121126021784537', '87334173791883', '6935887008832322803', '87334173791883', '8242249860781799', '7213608447124307', '1069403965571203', '4277613679497537', '10820413342307101', '24726754787465941', '83840805496966399', '6935887008832322803', '83840808183448961', '12363375462793339', '541020599953291', '1069403965571203', '299550598050343', '195906772109063', '7894208605606133', '6984788144380331', '215646921131989749', '6984788144380331', '7894208605606133', '195906772109063', '299550598050343', '299550598050343', '4701762530617507', '2960328227102303', '83817457732563997', '215646921131989749', '83817457732563947', '11841312908409187', '4701762530617517', '299550598050343', '87537206293696365317', '6072144895502719', '40174797067211', '1823520337936827', '1242516207366733', '1081350884962833', '-5061041939961507', '5446540899305143', '4341857033693367', '5446540899305127', '-2530520969980751', '-5525526861307083', '-4707087454776371', '-55255268613071', '-134488073925939', '-2937559344466309', '-5379522957037571', '-191059774786067', '-2530712866695649', '1361760748055489', '4342211755890447', '5447042992221953', '-2530712866695647', '-2690324879990699', '-1467173339161863', '-5380649759981411', '-5524637560179923', '-4716790955059237', '-5524637560179939', '-191069888423823', '-4956629120623077', '518168173448699', '4294754665968027', '2590840883272993', '-4956629123294657', '-69396714982487', '-8057406796766229', '-693967220321387', '-263944756096807', '-6211444480717389', '-5278894568650357', '-151383135134573', '-4822705860146409', '4944558705438169', '8400490894681513', '1236139797275463', '-2411352950225857', '-5344847931214819', '-2403702969765361', '-5344847942391937', '-2665352497360879', '-4920548876007167', '-5330705144765891', '-5945773679244721', '-4930152444556829', '5123582029525591', '267483759342948', '160111887700057', '-4930152049753969', '-2634041440109901', '-769464256532179', '-131702088752073', '-2769645411624003', '-199302486287017', '-2769645844342003', '-6037547471683547', '-1207725059684393', '2481081022155947', '8410084649372677', '4962162044311887', '-4830900238737567', '-2666920693267441', '-4940102997846529', '-1066768277306979', '-5348828231496817', '-120505205953777', '-5348828231496827', '-1487838565316427', '227295060194941', '7184876382563', '102371397679679', '8062733799536561', '499582426949713', '1645256439611343', '4773061473915373', '316042783060039', '57055926011503', '199855456432797', '8061469391873103', '4113133584269183', '8875891593229059', '4365175103708889', '8262458959599963', '3737972055331839', '2303360657081427', '275968409971089', '6607307719488219', '130059331283369', '8323641405926967', '2605199107508467', '1041504294604429', '251172943916489', '1131881038634001', '1036056942404287', '4985577894879537', '2304860115272649', '3734050810570529', '3449190994954859', '3278338334036849', '2076091750953239', '8313723686178747', '1041581050717489', '5209610383360927', '4010718406824089', '1143072353280169', '6640347650831579', '1186012510031083', '1048730722872406693', '5120755482775423', '1186012510031083', '6640347650831579', '914456136394247', '1143072353280169', '6640347650831579', '1186012510031083', '1048730722872406693', '5120755482775423', '1186012510031083', '6640347650831579', '914456136394247', '4547323505292399', '164209955329409', '2535941464090853', '5242397499163839677', '1279882201944297', '2535941464090853', '6568397485580617', '1136830330626293', '4547323505292399', '164209955329409', '2535941464090853', '5242397499163839677', '1279882201944297', '2535941464090853', '6568397485580617', '1136830330626293', '4005744699603133', '1596131760316587', '966601235746299', '2637059659036951763', '1030101138273', '1208251180884987', '6384527768862121', '4005744699603133', '4005744699603133', '1596131760316587', '966601235746299', '2637059659036951763', '1030101138273', '1208251180884987', '6384527768862121', '4005744699603133', '1214507879878249', '668432807969591', '2404217114609497', '654459759868050247', '639120859246143', '2404217114609497', '668432807969591', '1214507879878249', '1214507879878249', '668432807969591', '2404217114609497', '654459759868050247', '639120859246143', '2404217114609497', '668432807969591', '1214507879878249', '4677723935884007', '6608964063786083', '1267210021478327', '2618585996575135769', '1278606443640203', '1267210021478327', '6608964063786083', '4677723935884007', '4677723935884007', '6608964063786083', '1267210021478327', '2618585996575135769', '1278606443640203', '1267210021478327', '6608964063786083', '4677723935884007', '4809974007713921', '6673492031663433', '4846513869275953', '2556588866761479', '654486889589334051', '4846513869275953', '6673492759259243', '4809969278341159', '4809974007713921', '6673492031663433', '4846513869275953', '2556588866761479', '654486889589334051', '4846513869275953', '6673492759259243', '4809969278341159', '130632509740047', '753457099631751', '99958850337744686021', '218740741720379', '1754981450780873', '8630178095344141', '-4323350144604411', '816970902123329', '7595343064336909', '8169715036799007', '-8646703109005247', '-4031731575686697', '1225127912110721', '306282166014111', '-100793232996239', '-5613205071314811', '-5613203567423377', '-1403301079842273', '-8785213333422963', '8498176359862273', '7709975124084983', '2124544326239473', '-2196303496929423', '-7825372875441177', '8856033135971501', '885603543326667', '-7825371828569627', '-1402841017003131', '-2805681746844373', '-701420472606329', '-1121219732217089', '2179763194708881', '393512546449417', '4359525239262723', '-358790302983411', '-49594631198727', '2225402881875841', '8901604387153843', '-3967570182427399', '-57097386586741', '-1427434679828181', '-570973868899341', '-1709378302656239', '3996757080411337', '7526502579229803', '7993514160822693', '-8546891513281197', '-501320475839161', '2450604246505319', '4901208493010667', '-1604225522685317', '-348157710432579', '-4456418693537', '-2785261683460627', '-4361970962035017', '8390807757670493', '766129981764067', '8390807757670477', '-4361970962035017', '-7835901564470481', '8978885651225271', '1795777130245071', '-1567180312894097', '-5591180151664169', '-698897518958019', '-2795590075832079', '-4290850973995051', '1614384304687503', '7551182261299519', '403596305176317', '-268178238833808', '-7994974053741987', '2418710673792361', '4837424274955107', '-3997485801395439', '-5576261503084841', '-5576261342178123', '-2788130711315739', '3217343875553381', '4122490216767711', '7671750299252013', '6475805503508931', '257451905356679', '3838629149536839', '8158140961563533', '5392298268063113', '290950714374549', '8180270652834279', '3473878353466571', '939445764252821', '2795564178369279', '825691982470923', '3836823292612093', '8175434104134607', '6780480196545127', '939556501750143', '3718876398473681', '636733122415301', '636733122415301', '5406437996781249', '3205511022863351', '5406437996781249', '636733122415301', '636733122415301', '3205511022863351', '1863174927689363923', '3205511022863351', '636733122415301', '636733122415301', '5406437996781249', '3205511022863351', '5406437996781249', '636733122415301', '636733122415301', '7437759908469511', '8760104725751489', '7495140829008187', '7495140829008187', '8760104725751489', '42890532625779481', '1340329144555609', '7495140829008187', '5361315522792073', '7495140829008187', '7495140829008187', '7495140829008187', '7495140829008187', '5361315522792073', '7495140829008187', '8760104725751489', '8760104725751489', '5752244032919407', '540583369001979', '123241718392819169', '308104237774387', '540583369001979', '5752244032919407', '5752244032919407', '540583369001979', '123241718392819169', '308104237774387', '540583369001979', '5752244032919407', '1460138210125241', '8510602175561631', '248614805802151307', '776921268131723', '1702120580631473', '5840554295692433', '1460138210125241', '8510602175561631', '248614805802151307', '776921268131723', '1702120580631473', '5840554295692433', '584385483321869', '8643906686744591', '61619659544716967', '385122872154481', '8643906686744591', '584385483321869', '584385483321869', '8643906686744591', '61619659544716967', '385122872154481', '8643906686744591', '584385483321869', '589814654451839', '8639311682763413', '61624465177625161', '1540611629440629', '1079914051294901', '589814654451839', '589814654451839', '8639311682763413', '61624465177625161', '1540611629440629', '1079914051294901', '589814654451839', '2766988487409659', '861013998192819', '61832195828582413', '7729024478572801', '861013998192819', '2766988487409659', '2766988487409659', '861013998192819', '61832195828582413', '7729024478572801', '861013998192819', '2766988487409659', '1476818670371983', '8612900472058513', '30863914720696981', '3857989340087123', '8612900472058513', '1476818670371983', '1476818670371983', '8612900472058513', '30863914720696981', '3857989340087123', '8612900472058513', '1476818670371983', '1759378100254627', '1887514617977553', '1216126125911503', '19985587537696259373', '1893133839691169', '1355475020945141', '-5632566303385571', '8146983269713801', '1629396515572187', '-140814154701919', '-1559074899146289', '-3118149337057339', '-3253130822655603', '431940384926253', '-6506262106546447', '-1247259827069983', '-5581792183502521', '8056849868194701', '805684973798851', '-5581791879360669', '-3106734194668487', '-6213468649749353', '-1647967283151809', '6054422324534999', '-6591870697045973', '-6213468519543163', '-2802323090597153', '8102552806131483', '324102112245259', '-175145193162322', '-6213479112514251', '-6213479112514257', '-1623380921744037', '2240356595695171', '-6493523686976159', '-3106739556257127', '-5588053982941031', '1009506208738263', '8076049841250467', '-5588053982941029', '-62001165919519', '-310005812463159', '-648709582268729', '2295836587392941', '-3243548082688011', '-387507276287971', '-5691604557975379', '8237290016927627', '2059322504231907', '-284580227898769', '-6291838198046253', '-6291838198046253', '-6575699990982163', '4541788686974481', '-6575699990982159', '-6291838198046253', '-348795092849333', '80625882066389', '1612517641327781', '-348795092849333', '-12395419058159', '-3098854764539749', '-1300953863594877', '4912956622318159', '-650476931797439', '-3098854764539749', '6388250807590789', '229611259957521', '8342434922692579', '3585403699239241', '1845508319017833', '2113284026376583', '1335611613957979', '3691944932091451', '4150954293198731', '1028603055380491', '5547116307951677', '154869818092131', '563655356742169', '131866485503443', '5628139647749497', '21288660673583', '5476837546480133', '1375799706120677', '5926256640085571', '1553824861071583', '8505421394087281', '4252711424639411', '24861200105451793', '5926256640085571', '5926256640085571', '1553824861071583', '8505421394087281', '4252711424639411', '24861200105451793', '5926256640085571', '2906422236849399', '61618718015291513', '4323040108285473', '4323040108285473', '96279246898893', '2906422236849399', '2906422236849399', '61618718015291513', '4323040108285473', '4323040108285473', '96279246898893', '2906422236849399', '5513297475265109', '123601020224745003', '4309676939851403', '4309676939851403', '3862531882023281', '5513297475265109', '5513297475265109', '123601020224745003', '4309676939851403', '4309676939851403', '3862531882023281', '5513297475265109', '5907960425392247', '123436511809209783', '8615250917680967', '8615250917680967', '1542956397615123', '5907962971977563', '5907960425392247', '123436511809209783', '8615250917680967', '8615250917680967', '1542956397615123', '5907962971977563', '5874460412860217', '123238860888534551', '4321025556564709', '4321025556564709', '7702428805533409', '5874460412860217', '5874460412860217', '123238860888534551', '4321025556564709', '4321025556564709', '7702428805533409', '5874460412860217', '1473712814052763', '30809102881565621', '1080135305803917', '1080135396753397', '481392232524463', '1473713177850683', '1473712814052763', '30809102881565621', '1080135305803917', '1080135396753397', '481392232524463', '1473713177850683', '57848498379417', '456075915833647', '4299173281751809', '6791662905618733', '379895481120383', '12490971204513191521', '-5556006388206383', '4007896679946837', '4007897025470791', '-5556006445367161', '-3290140488832037', '3102743640911337', '-6580281445187419', '-6192438375751937', '-1548109534217931', '-774054782038979', '-5614109194502461', '2029408953888393', '405881790777679', '-280705459725123', '-6497562502973117', '8844715394253923', '-20304882821791', '-1555291286725669', '-6221165146902691', '-1244233029380537', '-5699631987655629', '4125306828050513', '8250613656101029', '-5699631987655631', '-1643225980874321', '8819305122458347', '-6572903923497281', '-6297300638420457', '-6297300638420463', '-314865031921023', '-5580941637489013', '8063104304929971', '8063103778754491', '-5580941111313521', '-6502925458286841', '2442952773737769', '-6502927562988803', '-6197556887745097', '-774694742512013', '-6197557413920601', '-2797702583377353', '4043921197508241', '8087842395016479', '-2797702583377351', '-3244676658249969', '906935806445101', '-6489353316499951', '-6205935876985849', '-3102967938492927', '-6205935876985853', '-2794660804578151', '8078175980783111', '2019543962687997', '-1397330326953977', '-3243196513619281', '4567356781108741', '-1621598643804667', '-6200933872378179', '-6200934132440461', '-155023350060233', '6367785236245839', '8456887168623783', '7433233635485917', '4845491524294383', '4168398495736113', '7381040885091463', '8898809519719519', '1169622081274933', '5311083104591761', '8503412344992189', '3006203354703569', '2742928856131313', '1004005918708447', '8331830179765703', '7384216967500799', '8237067003588837', '159702261635233', '5546057819033529', '8717588966066699', '502501499898351', '502501499898351', '8717588966066699', '42628715759296509', '1065717893982413', '502501499898351', '1065717893982413', '502501499898351', '502501499898351', '502501499898351', '502501499898351', '1065717893982413', '502501499898351', '8717588966066699', '8717588966066699', '216971919709847', '945220692712063', '32348519418420437', '8087129854605113', '216971919709847', '8087129854605113', '8087129854605113', '216971919709847', '945220692712063', '216971919709847', '2869056248516031', '4668987682640919', '93379746376861', '2869056248516031', '2869056248516031', '4668987682640919', '93379746376861', '2869056248516031', '5737550472071197', '18676512573625611', '4669128143406403', '286877469033879', '5737550472071197', '18676512573625611', '4669128143406403', '286877469033879', '5758416891886243', '3731129303521701', '4663911265604227', '719802066011043', '5758416891886243', '3731129303521701', '4663911265604227', '719802066011043', '1480286162865029', '4623229462134971', '4623229462134971', '1480286162865029', '1480286162865029', '4623229462134971', '4623229462134971', '1480286162865029', '5759856074430527', '9327103212784737', '291471975399523', '5759856074430527', '5759856074430527', '9327103212784737', '291471975399523', '5759856074430527', '1183989862148801', '9247056594627997', '4623528297313999', '1183989862148801', '1183989862148801', '9247056594627997', '4623528297313999', '1183989862148801', '174654093429317', '926774090389811', '1483280544465919', '302370989028717809', '2465700172541453', '2262214753475149', '-1526484335728411', '8299656859319349', '-6105937342913641', '-3912217350934591', '-7824436604162551', '-156488732083251', '-782443470186919', '-7824435653015869', '-610655494937437', '4150301709236211', '-6106555723379311', '-978127007071549', '-7825012960552639', '-1956254014143101', '-978126620069079', '-1956253627140629', '-1216732830552349', '8265533270212381', '-6083664409828489', '-1950897536676809', '-780359102280417', '-1560718410214229', '-975448639805033', '-7803590584755703', '-1181463730440021', '1598951446073109', '-738414831525013', '-305590405922773', '-3819880074034663', '-3819880074034663', '-763976014806933', '-477485009254333', '-6082087662856513', '4131558579140567', '-6082087662856513', '-7802116334863777', '-243816135464493', '-975264541857973', '-243816135464493', '-7802116334863779', '-1477150081209773', '7996729130421109', '-5908600324839089', '-3820471519257069', '-7640943038514139', '-3820471519257067', '-7640943038514147', '-7640943038514139', '4139914614172789', '8307383637876213', '1034580093208999', '8226272804604671', '8263523060994981', '411494596061819', '5404071645511679', '7585787649245413', '6461107142997881', '1009547991093419', '5404071645511679', '1009547991093419', '1009547991093419', '5404071645511679', '379289186219363', '5404071645511679', '3051759303581263', '3051759303581263', '3051756321418737', '3051756321418737', '6103514132213831', '6103514132213831', '6103517117786169', '6103517117786169', '1', '1', '3051756274040229', '3051756274040229', '3051759350959771', '3051759350959771', '6164702778574277', '38526345432680131', '6164702778574277', '7005948763528431', '7005948763528431', '6164702778574277', '7005948763528431', '7005948763528431', '4815783946536337', '6164702778574277', '1376703006973047', '1376703006973047', '4312641210300461', '8352415687542229', '4312641210300461', '4312641210300461', '4312641210300461', '4312641210300461', '2610129902356947', '4312641210300461', '2610129902356947', '2610129902356947', '5506809380061367', '4312641210300461', '4312641210300461', '5506809380061367', '3711910670711881', '1273091736691087', '1273091736691087', '5402248270650137', '641719169187721', '5402248270650137', '1273091736691087', '1273091736691087', '641719169187721', '931532199335343987', '641719169187721', '1273091736691087', '1273091736691087', '5402248270650137', '641719169187721', '5402248270650137', '1273091736691087', '1273091736691087', '1855951785712649', '106473464653541', '4270817131939533', '4270817131939533', '106473464653541', '10606959643747411', '2651739910936853', '4270817131939533', '2651739910936853', '4270817131939533', '4270817131939533', '4270817131939533', '4270817131939533', '2651739910936853', '4270817131939533', '106473464653541', '106473464653541', '334555474126031', '7589877093187733', '1009692153660013', '4038768614640051', '334555474126031', '4038768614640051', '4038768614640051', '334555474126031', '7589877093187733', '334555474126031', '6103518428347899', '6103518428347899', '6103512821652101', '6103512821652101', '1', '3051759287313251', '762939821828313', '6103512675373497', '6103512675373497', '1750071582291671', '6087720151047011', '1558457848784041083', '4585649383340759', '1750071582291671', '6087720151047011', '1558457848784041083', '4585649383340759', '1137398027256991', '1219737514893449', '12197392611233551', '1332328671233761', '1137398027256991', '1219737514893449', '12197392611233551', '1332328671233761', '8630041545689283', '3115022870951382273', '608402467925243', '2831612931421071', '8630041545689283', '3115022870951382273', '608402467925243', '2831612931421071', '775785642693313', '24999196406740823761', '1525829131214159', '84332180370187', '775785642693313', '24999196406740823761', '1525829131214159', '84332180370187', '7562199592507053', '3047735545753393', '62417728285147593', '8902574189784783', '7562199592507053', '3047735545753393', '62417728285147593', '8902574189784783', '922246720595281', '6247573428685187809', '6101143016067467', '5101012120475561', '922246720595281', '6247573428685187809', '6101143016067467', '5101012120475561', '1811219146409401', '750508009792427', '2992517378957791', '99795033094566607113', '7972246032606167', '476209804077209', '-5076678985282003', '-1131774884355677', '-5076676067349161', '-905420491071109', '-150594134930349', '658269531040361', '-3902247505740339', '-150059039417261', '-5990927091300413', '-5669094372665677', '-23963690889091', '-5669098741693333', '-4565786428530633', '1259404915694303', '-2323121757966201', '-5830010731983041', '-4895472807098927', '-8684620519197229', '-4895474996878221', '-8684616139638629', '-181334659358119', '5082231791970279', '-1505993748592913', '-4618891533348767', '-8698289717194013', '-4264233097325487', '-8698291172432549', '-4264232369706211', '-6672890760953591', '1135193598548917', '-6715347005398987', '-1722675591184497', '-1387752629780217', '-2609509087320707', '-5551005419233567', '-5219023274528711', '-8460733768760051', '2925611703233761', '-862672739105613', '-5385014346881139', '-262004688730217', '-793093559179483', '-818764834251553', '-6344747017678857', '-5043717029759977', '1728765046685319', '-2547529789952059', '-6447432845845639', '4506076311291243', '1098572508729639', '4880560380257279', '4754156499381311', '7148744597567469', '2737549133284093', '281304128573957', '7783439182481813', '8847045977929349', '8847045977929349', '7970242467919456293', '281607149481319', '281304128573957', '7783439182481813', '8847045977929349', '8847045977929349', '7970242467919456293', '281607149481319', '911806696689729', '3969066531851144947', '8891980411672321', '277877298247763', '3876010248393601', '577765959571289', '911806696689729', '3969066531851144947', '8891980411672321', '277877298247763', '3876010248393601', '577765959571289', '674462016292933', '15961007330659889441', '110306279231853', '1764900904267093', '1948364788870573', '2707446326290891', '674462016292933', '15961007330659889441', '110306279231853', '1764900904267093', '1948364788870573', '2707446326290891', '8118703664015321', '15972000922074311933', '1761688370054689', '1101059142111171', '311952269894401', '4101604406810517', '8118703664015321', '15972000922074311933', '1761688370054689', '1101059142111171', '311952269894401', '4101604406810517', '5322677035601573', '991930201522028269', '356187588620939', '222618698079533', '3874708068408761', '5401515670174707', '5322677035601573', '991930201522028269', '356187588620939', '222618698079533', '3874708068408761', '5401515670174707', '5477261657267859', '500375411645582109', '8766602085328593', '4383314139388177', '1954587268064871', '5512755597973917', '5477261657267859', '500375411645582109', '8766602085328593', '4383314139388177', '1954587268064871', '5512755597973917', '99946815456316151327', '95555079238427', '915762363419217', '2239682377671871', '381296834217131', '1078033990847393', '-8871912822288431', '-2217649589986269', '-3532552981596959', '-7229855219951733', '-282499104362549', '-1194526583165767', '7338563075339867', '7338891548286581', '-597304359616168', '-554453474444797', '-5312603159095283', '-2647911807858797', '-2341998521186357', '-509590846831497', '-4650452615361413', '-539389991404941', '8139877365860511', '8148291770815853', '-5402291347991991', '-1060842677481287', '-1677248900654471', '-3352330154760951', '-5632983256609251', '-1725087692017929', '-5624313040508121', '-7028350876593771', '1338046418350477', '1338317445396839', '-7030518556430439', '-6706827956069891', '-2951922306422639', '-589120395807623', '-2553425847464997', '-1606621894565113', '-2540787490713199', '-1520146190125703', '4607781964402321', '230547319941513', '-380431609091259', '-5897524285460751', '-356681419323193', '-1137790659740519', '-1236014319405283', '-3087261497551173', '-30676049884231', '-1170844150124101', '4431979101196891', '1774589618743883', '-5863196399404513', '-5697928003936839', '-5683414032294691', '-177360258032651', '-2485753111814033', '-178962027538047', '-198229593816283', '-5823807413607461', '8809847659280783', '4406898539500761', '-5827750527868569', '-5679471144669757', '4276835829748829', '31808848796661', '2316510839496759', '7488167682437259', '1374903515266089', '382322992857323', '2315406709091963', '653609095433789', '4620736875424853', '4741363135828389', '597740237693259', '4503663957713911', '3289405373931773', '1362693986717651', '6101990283852187', '6376156879344679', '1099897836422201', '6119393885574461', '180594079064363', '1469904582409437', '2667510082870602923', '5018055047352063', '5018247496440493', '40702925359706', '5879644523086457', '90636708535283', '180594079064363', '1469904582409437', '2667510082870602923', '5018055047352063', '5018247496440493', '40702925359706', '5879644523086457', '90636708535283', '241186192443539', '5793569268641033', '5213013700523859', '5276075516333883', '2637181741756697', '10676363817381418597', '5793344441551559', '5782581481289307', '241186192443539', '5793569268641033', '5213013700523859', '5276075516333883', '2637181741756697', '10676363817381418597', '5793344441551559', '5782581481289307', '5530028430029097', '5888000073593107', '21340686733234661317', '998509892088321', '624161723859531', '1302526601327817', '1472025302349991', '2817804559609161', '5530028430029097', '5888000073593107', '21340686733234661317', '998509892088321', '624161723859531', '1302526601327817', '1472025302349991', '2817804559609161', '27369683718871', '5599525599210433', '1074414306698013689', '5015778037362223', '1003156116789503', '1049232721384779', '5599525599210433', '171065411777209', '27369683718871', '5599525599210433', '1074414306698013689', '5015778037362223', '1003156116789503', '1049232721384779', '5599525599210433', '171065411777209', '4287525820324259', '5749566470908537', '10688213309891637719', '5237686151292389', '327836302485471', '5218611502049853', '5750593108461223', '2412834171722659', '4287525820324259', '5749566470908537', '10688213309891637719', '5237686151292389', '327836302485471', '5218611502049853', '5750593108461223', '2412834171722659', '5364918470146833', '2911711192756343', '10691136576813318101', '152652519725317', '244312916687351', '2610118937951313', '5823608286224687', '2729294572965517', '5364918470146833', '2911711192756343', '10691136576813318101', '152652519725317', '244312916687351', '2610118937951313', '5823608286224687', '2729294572965517', '447505292203367', '1014524225386667', '2053775064740859', '99977151968773544093', '16449861610283', '798771985683211', '-3118988180041729', '-3117480494771163', '-391838130102583', '1594386573056757', '6389636755387603', '-7827970509477787', '-213074394170907', '6063386156271291', '2252533417535141', '6069396951761567', '-2664818547606133', '-6236468674812891', '-317302804421549', '-255180091737353', '-7648908636907821', '1351976892238531', '1318466796138877', '-7747093433334393', '-1118613021610571', '6649240908032341', '4665865678666999', '822815183311493', '-695266344963927', '-6362779190932401', '-6406650890762523', '-1278305801198591', '-1591122877930041', '6107048161772249', '308381984434039', '-791123746971853', '-5570150746772089', '6609468359267999', '742794281023', '3319816644517171', '-2792066947943057', '-3199544974188869', '-3433010886132927', '-6865998566386427', '-799965039106131', '181375257503821', '2267237130556583', '-499973992016759', '-314900786876461', '811935773938901', '2551678546119827', '1014925518893483', '-3149018456003861', '-3433005084663069', '-3072045208693323', '-756090355731123', '-3849799365832143', '2834396169128443', '6051397794547443', '-1855815516193107', '-5108278395188871', '5546126476222669', '4435300895663437', '179251739709941', '-5195771353018669', '-762050828952227', '-5999653208564913', '-23942370434581', '-3891524433809281', '22040146433261', '7109293427105877', '-1935683005257237', '-1227008576830933', '317997703412231', '8563197063060677', '2557970328815823', '-2460425828282991', '-2996311454302539', '8281687670323799', '447684252374677', '6079827198528149', '2496799987153531', '1444651702450493', '7192032959820999', '828535846057007', '2242477073518287', '1079174691495701', '8242004282249161', '17432177140189', '7885618074899061', '7273892521476627', '225264927974837', '358608145384979', '313393966045963', '7188161218092059', '897553101858897', '5087988630697413', '2618600363248583671', '1650988774636311', '584667426736211', '2338697537482029', '1650988774636311', '5114453834469889', '1271997430522757', '5087988630697413', '2618600363248583671', '1650988774636311', '584667426736211', '2338697537482029', '1650988774636311', '5114453834469889', '1271997430522757', '4871551998195589', '1309000433148847451', '3333241122508871', '2399438050040021', '2399438050040021', '3333241122508871', '2556641470993843', '4871551998195589', '4871551998195589', '1309000433148847451', '3333241122508871', '2399438050040021', '2399438050040021', '3333241122508871', '2556641470993843', '4871551998195589', '5090507080084553', '1310438599614886503', '6569013820401643', '2257826266235881', '2257826266235881', '3284506546402907', '5118900779745649', '5090507080084553', '5090507080084553', '1310438599614886503', '6569013820401643', '2257826266235881', '2257826266235881', '3284506546402907', '5118900779745649', '5090507080084553', '4750468962646081', '5242982740575933329', '3321929674140583', '2294129020145533', '917650807702897', '3321929674140583', '1024020066518737', '4750468962646081', '4750468962646081', '5242982740575933329', '3321929674140583', '2294129020145533', '917650807702897', '3321929674140583', '1024020066518737', '4750468962646081', '2412033204833593', '1309010090589532053', '6677862647800347', '2411546807038603', '2411546807038603', '6677862647800347', '63916508329567', '2412033204833593', '2412033204833593', '1309010090589532053', '6677862647800347', '2411546807038603', '2411546807038603', '6677862647800347', '63916508329567', '2412033204833593', '4849462191177459', '659268870002160247', '63802261947939', '999324232855153', '999324232855153', '63802261947939', '1287634511722969', '4849462191177459', '4849462191177459', '659268870002160247', '63802261947939', '999324232855153', '999324232855153', '63802261947939', '1287634511722969', '4849462191177459', '28074178663359', '2196031013424751', '379210224648579', '64766914065891', '857218251700833', '99958824208723347359', '-5590996862692533', '-2795493597571281', '-8731405557713283', '1681571674877949', '1533579573491977', '4203948522294823', '-2182855798352249', '-3911000153470133', '4461979195073539', '8923997060346849', '-7821972613518307', '-2795496014458773', '-557738774354713', '-2788693871773567', '-4298123009727687', '8104844168090707', '7562682211318273', '8104844168090669', '-4298123009727687', '-3988301572406597', '4798430802532137', '149950962579127', '-3988301572406583', '-1394346935886783', '-350947469151841', '-5615159702698109', '-1761069824990949', '4273845311853969', '1544338649252247', '4273845595353107', '-8805349179473809', '-7816370663531503', '8804844453947207', '8804843668872463', '-7816370576300981', '-5615159604563783', '-561060137451839', '-1402650515350659', '-8641872133612897', '8162974899015329', '1518330899746783', '2040743037869579', '-8641870845704943', '-4029822828085267', '2449045311978509', '489808925018847', '-2014911929205809', '-5610601717960513', '-2787483216710319', '-34843540208879', '-1713514444899999', '2009629856247359', '3770459151542323', '8038519424989393', '-8567572224499993', '-1602098150129403', '2435524317226389', '974209726890537', '-8010490750646993', '-2787483216710319', '-713855345007819', '-1142168552012511', '-8980483504704971', '8743414463138303', '7878417017778247', '4371707231569127', '-8980483504704967', '-3961631132511027', '276552438119939', '8849678019837897', '-7923262265022039', '-2855421380031277', '3824300680654617', '4135597250935627', '8146960510663209', '937090450068041', '1367651343208717', '31702343929401', '3828455862379473', '8253157426735149', '508263027963707', '3826663633637551', '8259523926306267', '5157352721821361', '4685178573439741', '216035554227733', '1015160021111543', '2320281289686527', '5684380077827049', '4049542372859601', '1081383741425567', '24641270237711679', '5795178720410581', '5795178720410581', '7700396949284903', '1081383741425567', '1081383741425567', '24641270237711679', '5795178720410581', '5795178720410581', '7700396949284903', '1081383741425567', '1081129041297189', '1539716244318209', '5885871981513671', '5885871981513671', '123177311186989417', '1081129041297189', '1081129041297189', '1539716244318209', '5885871981513671', '5885871981513671', '123177311186989417', '1081129041297189', '8652768236610743', '7700271161275647', '2885998064405267', '2885998064405267', '12320436186347493', '216319242295057', '8652768236610743', '7700271161275647', '2885998064405267', '2885998064405267', '12320436186347493', '216319242295057', '4255189754932167', '49722733266923923', '5845262053247777', '91332202528973', '121393391764951', '4255189754932167', '4255189754932167', '49722733266923923', '5845262053247777', '91332202528973', '121393391764951', '4255189754932167', '2149357831198959', '247147203734645411', '5918895796216983', '2959447170512691', '7723350116707671', '2149357831198959', '2149357831198959', '247147203734645411', '5918895796216983', '2959447170512691', '7723350116707671', '2149357831198959', '2149797403581427', '61873371550863541', '5544474692265559', '5544473600871989', '966771430482243', '4299595170960711', '2149797403581427', '61873371550863541', '5544474692265559', '5544473600871989', '966771430482243', '4299595170960711', '1014470042074791', '99958703554488162779', '1220910133657441', '584875212144283', '5956512046084927', '6189375775991453', '-6225588361623969', '-389099272601497', '-5620186781072757', '8127579381333529', '8127579381333533', '-2810093390536377', '-3248568628596977', '8689564658240277', '-324856862859699', '-6225588361623959', '-155079672869847', '-775398306664681', '-2796666213797189', '323402865141201', '25265847757571', '-2796666184954913', '-1296440360805883', '4464243765485773', '-6482202034767641', '-1240637336811133', '-6231380739876767', '-3115689566407517', '-1125465336725441', '65111707669529', '2034740777196061', '-5627326568283429', '-3250066296061503', '4300051657926831', '-130002674756193', '-62313799363459', '-1553076315368741', '-6212305717320243', '-5580377297444539', '8054601736228949', '1006825188538289', '-5580377069521893', '-6591182090347809', '3031016454364043', '-659118300203841', '-6212305489397603', '-6194775771320251', '-6194776371552861', '-696826660952313', '805183838946057', '8051838089344267', '-2787306493751097', '-6514013232247663', '1276951044174919', '-1628503608178229', '-6194776071436559', '-6289033427820843', '-6289033564108117', '-227456826946793', '1028540549142723', '1645664864999629', '-2843210216691189', '-6581716383407589', '585367080323623', '-1645429422211249', '-6289033495964479', '7353229056929811', '8326113574383073', '898628791722521', '5543287592273451', '146547298175007', '514276628250539', '7381359738343567', '4154013269043307', '8113065814271137', '57810376012717', '2096197435201551', '971054994041667', '1340663796324303', '8769543860487611', '8813883656291561', '2588111086166119', '4025895018215283', '4550004247892159', '2332088332158433', '5757355115136767', '5757355115136767', '9328354056229501', '2332088332158433', '5757355115136767', '5757355115136767', '9328354056229501', '18499130276064191', '5914933497228343', '2957466202917371', '4624782205218181', '18499130276064191', '5914933497228343', '2957466202917371', '4624782205218181', '2334658382324999', '2868397720700003', '2868397720700003', '4669316764649999', '2334658382324999', '2868397720700003', '2868397720700003', '4669316764649999', '4669252233269591', '1434263391730409', '1434263391730409', '4669252233269591', '4669252233269591', '1434263391730409', '1434263391730409', '4669252233269591', '9249213778204433', '1183126988718227', '1183126988718227', '578075861137777', '9249213778204433', '1183126988718227', '1183126988718227', '578075861137777', '4664984070853549', '2877063108292901', '2877063108292901', '93299681417071', '4664984070853549', '2877063108292901', '2877063108292901', '93299681417071', '3488986400080677', '7095712343697213', '1499151079701213', '2782690656238021', '1545127168008602737', '8526014967019343', '-3902339576919783', '-3902338624792051', '-243393092126331', '1653463101092127', '-243393092126331', '-7804677249584099', '-7804679153839569', '-3902339100855917', '-955738493547661', '-3822956436015239', '-5913981527364757', '800500709883329', '-236559231062859', '-152918197377147', '-477869434472151', '-7645910410205881', '-1565158188814503', '-1956447736018131', '-6107384224983847', '8301872977931433', '-6107384224983847', '-1956447736018131', '-3912895472036261', '-3912895472036261', '-7825525294584223', '-7825525294584233', '-3053550317419543', '8301438622386057', '-1221420126967817', '-7825525294584233', '-7825525294584231', '-7825525294584229', '-7645214956324923', '-7645214956324917', '-14783069863027', '400192420612957', '-5913227945210801', '-191130373908123', '-7645214956324923', '-3822607478162461', '-1561597555069291', '-3903993887673229', '-1522091430403263', '8272737555552873', '-121767314432261', '-156159755506929', '-1561597555069293', '-7807987775346457', '2073165764106993', '8265959289733609', '1667247136365531', '2076482780993207', '1648406879256549', '8325298308344017', '1', '1', '1', '1', '1540944598829469', '4816327196544551', '1540944598829469', '7005472531536809', '7005472531536809', '1540943925696563', '7005472531536809', '7005472531536809', '4816318109250323', '1540943925696563', '289063679577651', '35260431812053', '3898327930053559', '7983783051329738629', '1101889949318073', '2874206775185021', '289063679577651', '35260431812053', '3898327930053559', '7983783051329738629', '1101889949318073', '2874206775185021', '992354539396851', '8834725666809497', '31905491822693002117', '7789425259333199', '8834725666809497', '124203660889777', '992354539396851', '8834725666809497', '31905491822693002117', '7789425259333199', '8834725666809497', '124203660889777', '2251009032237', '4424142762370853', '3187842971789414779', '7782819755345249', '4424142762370853', '56335748130981', '2251009032237', '4424142762370853', '3187842971789414779', '7782819755345249', '4424142762370853', '56335748130981', '5370933498879', '1115183630968659', '15855819883750110297', '7742098097420753', '8921462499387049', '1721562973277691', '5370933498879', '1115183630968659', '15855819883750110297', '7742098097420753', '8921462499387049', '1721562973277691', '6940694264087391', '8779146234630523', '7998132430372970167', '1562128255387501', '2194773643832107', '701091307995441', '6940694264087391', '8779146234630523', '7998132430372970167', '1562128255387501', '2194773643832107', '701091307995441', '153246282972913', '4459456085420781', '3964302866285950631', '7742723738434409', '8918829952516841', '501731105662837', '153246282972913', '4459456085420781', '3964302866285950631', '7742723738434409', '8918829952516841', '501731105662837', '384582468198013', '8181840832878823', '49989132939898407389', '594550532834393', '181971160956921', '566121000699211', '-2627418541982767', '-1780502738736627', '-2634375886457707', '-6148566534417653', '-6155525061199901', '-3191527167072799', '4844173740554369', '4842432684886219', '-6379574990199381', '-6152045797808781', '-100845867074262', '-598101189299981', '-6451001122886321', '-7948604588477083', '-1986759493415471', '-2119445707124771', '6491023335083613', '6491415273846419', '-8478566420965693', '-3973910640534743', '-1766133861656467', '-1446280111954449', '-7061911197998649', '-8871729524412099', '-8870417400098523', '-4778051353704491', '458656147336522', '7338826388462761', '-2389189692391447', '-2217768365563829', '-1257532619341729', '-3325731366164727', '-5026066464266629', '-5860548148827623', '-1464628618972067', '-1209633431003171', '4583538162711721', '2292023433874077', '-3024591765811497', '-5859531312357949', '-2427293967638709', '-1060071912157421', '-4830018914282253', '-1377748195273467', '-1099740791732513', '-2809964361591123', '424524974058809', '4248328769014597', '-5626073971359551', '-2752424184939109', '-199483155367127', '-6366588895341893', '-4931260795122693', '-1153790228511209', '-1435258940145071', '-2959740934639021', '4484218240377839', '8982415966826327', '-5933441080297043', '-1150998690313633', '6951195902047923', '4589644782924043', '5855112683197421', '283907498861761', '2300351942818417', '274931121137541', '1083942186865807', '463215593377967', '7195058006447', '5735455106800749', '3125613884642273', '5684195851412313', '4574151912397179', '790289100681139', '5750145064522273', '31963848022969', '3128751319469473', '355745943373411', '6446119180605237', '7088797949543679', '1795298651248317', '7146595244760537', '55995891843809528609', '7146595244760537', '1795298651248317', '7088797949543679', '3223151994961621', '6446119180605237', '7088797949543679', '1795298651248317', '7146595244760537', '55995891843809528609', '7146595244760537', '1795298651248317', '7088797949543679', '3223151994961621', '588086924101981', '6519572053225859', '7154139575106117', '7462948664792677', '28183390434664763029', '7462948664792677', '7154139575106117', '6519572053225859', '1470336635954509', '588086924101981', '7823486463871027', '171699349802546827', '44777691988756081', '28183390434664763029', '44777691988756043', '171699349802546789', '39117432319355173', '1470336635954509', '1592676181229643', '7158188157114251', '3590790180291977', '1769433746071261', '55993919942537343053', '1769433746071261', '3590790180291977', '7158188157114251', '6370898992984711', '1592676181229643', '10737282235671389', '86178964327007423', '21233204952855107', '55993919942537343053', '21233204952855157', '86178964327007473', '2684320558917841', '6370898992984711', '582305215134163', '934100222432033', '715842509608223', '649217289693493', '56367310978764127501', '202880357554479', '715842509608223', '934100222432033', '5823451965234357', '582305215134163', '1120920319001039', '21475275027833693', '9738258824576401', '56367310978764127501', '4869128841720493', '21475275548659687', '5604601074179201', '5823451965234357', '269175817920309', '6784936743626691', '1788155724555271', '22883200605227', '56245947210935911533', '7322622010885343', '1788155724555271', '6784936743626691', '5384599384703343', '269175817920309', '13569874421990297', '57220982251031757', '2929049490521673', '56245947210935911533', '14645244956507601', '57220984120505587', '13569872552516467', '5384599384703343', '6214780320668959', '3674392806493231', '3575790483712799', '6758773114580999', '28124278893480463703', '6758774569772529', '3575790847510681', '734878706817799', '1241734576364759', '6214780320668959', '44092712558523709', '34327588867521883', '40552639806881057', '28124278893480463703', '40552646299240111', '1373103676488941', '44092723528463003', '1241734576364759', '12426103657219533717', '1493922082548067', '4544696277926529', '886118232707853', '832326714243657', '313162036179923', '-6152684030575229', '-8191335088255801', '-384539627189263', '-1227232660098719', '-1664700185781783', '-3068056652473289', '-65374904815021', '4225199890130013', '526149936290071', '1056306221975881', '-6537509229832231', '-7176575918833701', '-3170308301364677', '-1453132478213329', '-6340475073269159', '-3032834890597473', '-73947163776109', '-189547757866711', '-6612043073614651', '1711424599123527', '2660582761080853', '8557264525077839', '-6612096147162223', '-7248753636882591', '-122759105988063', '-1674119870267789', '-6137902122953847', '-385059451800391', '-4093315958057119', '-385056128272309', '-327454622338427', '423813053880419', '5267934186741109', '1059539281757211', '-3274556193968513', '-3592126815047937', '-1516082127192011', '-148978555600803', '-1516052198997591', '-6350528040406157', '-905628839155999', '-6350408475653113', '-6621312663707847', '8573385237191981', '1065579806984471', '2143376379289729', '-1655339402098763', '-7256097097185633', '-6313125141355731', '-3948807282404127', '-6312774390786439', '-6157966267200939', '-2284721190102959', '-6157615716040357', '-3352447635479607', '4397068509978713', '335168122024452', '8794488611386633', '-6705026882604097', '-3650075793336061', '-6087298108024423', '-8948696017461609', '-1522253060849283', '-1564393269634803', '-7586494087926591', '-6259287755266109', '-3282415428055463', '8482704473738717', '2642362120814263', '8480988819856879', '-6564187797863243', '-1801685544473371', '6152738630668099', '4111642164443071', '5119287394433517', '2558744652479791', '1648590208500817', '1130083981594547', '2382886676350101', '4178493122697911', '3437508252682231', '8973970175081841', '236453219637069', '5827690808783757', '631804395466539', '8224383050631439', '128027451591657', '1280045022974323', '8242156735625769', '8820522321688559', '2377458369626469', '1271937249508333', '8971681642841907', '6874359803914937', '417710966225129', '5827232515679979', '3513715130545351', '418410110378939', '941952479444877', '1494691490581443', '3022749600691171', '3597699811605913', '3199189930879011', '7532004016032013', '298595697682181', '7530027662383033', '85356918116357', '45005917831137', '72115091663641', '1963985428652621', '4444971669068123', '5255995021535979', '490990517479531901', '7671726835617687', '5255995021535979', '4444971669068123', '1963985428652621', '72115091663641', '148160411917716311', '25127295881989079', '142707797600757977', '20555275895564373', '3928392844026833249', '3928392844026835287', '164442207164527', '142707797600759479', '25127295881989079', '148160411917714809', '8785332381911897', '1963654885357281', '292503816062781', '5299115168218567', '19638467392610109', '7671276325238327', '5299115168218567', '292503816062781', '7854620996620633', '8785332381911897', '7056363711897831', '1004829345175561587', '75161954975755087', '5158626136377', '1964127717324694051', '982063858662347423', '10317252272753983', '7516195497575507', '1004829531440074739', '70563637118978327', '936304121562113', '394010811713781', '2186509446480361', '5142871648656243', '490293498317299733', '1915208977801953', '5142871648656243', '2186509446480361', '788021550667981', '936304121562113', '60034116013128461', '403356418961758617', '280426470315251333', '40589712023483799', '7845249234242560853', '7845249234242565633', '40589712023484819', '280426470315252353', '80671276341770629', '60034116013128257', '111861744207497', '7986620745247193', '2117565947205509', '4718076446120439', '97696138692194613', '1526502167065541', '4718076446120439', '2117565401508739', '7986620745247193', '111861744207497', '287491184310984659', '2043449795353341907', '54434710592478363', '36619492429171173', '7816816207095655703', '7816816214515362259', '36619499848876849', '272173490532910931', '2043449795353341907', '287491176891278983', '1079251697933781', '7885103574275081', '2370384209143647', '1284091475536159', '24506370091601769', '3829120326812777', '1284091475536159', '4740767690691477', '7885103574275081', '4317007519330941', '138430158751852869', '3153147870702281', '151990525733825403', '20259522192249643', '784261030200983007', '3921305156072612549', '20259527259946549', '30398101503691233', '3153147870702281', '138430176967222107', '1677329661792871', '3990117941919103', '4534651321639129', '472642263954589', '244303669246459621', '7634489663951863', '472642263954589', '4534651321639129', '3990117941919103', '1677329661792871', '33675015918186801', '510478251200986883', '1137676039232561', '4597999957216509', '244335774917041851', '195468619933633569', '459799995721697', '36405633255442413', '510478251200986883', '1683750795909317', '257180435832553', '879995801240079', '46483198570083220881', '1645981164675123', '2762566191251101', '1362311181354787', '-247411166948069', '-858841622555179', '-4948223338961389', '-2690319386702247', '5383633396569981', '53836333965699', '-5380638773404429', '-1064684766151431', '2634601755637673', '1053840702255083', '-2661711915378617', '-268882207511949', '-1236683925625259', '-6847789237245447', '-4946735325366279', '-5300174585411761', '261281350906471', '5225626640994639', '-165630455794118', '-337281479312437', '1354571296325961', '5418284808169061', '-337281479312437', '-167960073164422', '-4942944879302343', '-176506716467377', '-4942945067203751', '-5414189659108191', '5444167459844191', '340260477984097', '-270709482955409', '-1062073966795361', '5236527809581409', '81820749960669', '-5310369833976831', '-673026470552687', '-2477778199979999', '-1994966934693793', '-2477778394668639', '-5497336541923761', '5540374444909277', '2770187126558637', '-687167016926451', '-169180960674171', '5373283465440997', '5373283267855719', '-2706895476698503', '-1363574582080523', '-4941469976559443', '-3547364966375449', '-4941469958645907', '-2646112002223823', '649944629605303', '649944631217829', '-5292223627244493', '-169838133163662', '5484750317728013', '2742375161370669', '-679352580924641', '-538489058839961', '-4955704349593433', '-495766806855909', '-4955704349593453', '-5392141030273473', '5332810904097547', '5332810904097437', '-2696070515136689', '-2755554865292933', '5570748304722407', '5570748304722501', '-5511109730585973', '-5451471156449357', '78776210491813', '5526370221911803', '8232905254211553', '1392477200987029', '7381586920628741', '1640265995145641', '1410428250818041', '551005497180219', '8248507589262363', '7892446819822543', '8064455105653953', '1778638115205607', '2488885028509091', '693462223733697', '1510923455085431', '5234333789701429', '3038890906771', '8242017457125913', '271876939138161', '5044342352753857', '3555524475558123', '5211655535956769', '2863682892592319', '8242000518002361', '5293899761220433', '2772492135774427', '7560224113708283', '123070565089073', '8099042175125241', '19530064118979', '1110873332473429', '1249267960910483', '7070041830086709', '5378446682380037', '8099082894819361', '3137205259382619', '34944128127101', '3601690984726521', '8352171251651983', '8254597751698847', '13869495400447974441', '8254597751698847', '1044022406900649', '3601690984726521', '34944128127101', '167731815010084889', '43220291816718163', '8352171251651983', '49527586510192993', '13869495400447974441', '49527586510193171', '1044022406900649', '43220291816718341', '167731815010084711', '6988170973716551', '3908322854624751', '4707172437174093', '7908121959263901', '3449935857657337587', '7908124142051197', '4707172437174093', '3908322854624751', '6988170973716551', '33543220103080087', '46899877109293801', '4707172437174093', '9489746921876039', '3449935857657337587', '47448741998510393', '4707172437174093', '46899871401700223', '167716106222994013', '1397079345175597', '3757440633392307', '7535128074366409', '2022633303673629', '1383689664423548171', '2022633303673629', '7535128074366409', '3757440633392307', '6985398181069531', '33529904711594731', '45089285463805669', '7535128074366409', '48543197151265081', '1383689664423548171', '48543201425069111', '7535128074366409', '45089289737609699', '167649554208766729', '6985279602881737', '4043865997017597', '3767966044594817', '3759076643209971', '13836895193584590329', '3759076643209971', '7535929178806533', '4043865997017597', '6985279602881737', '6705868418766469', '48526391964211127', '3767966044594817', '9021783943703923', '13836895193584590329', '45108919718519689', '7535929178806533', '48526391964211201', '167646710469161651', '1747032646321997', '1578156896561709', '2365033967361991', '1958315371562593', '6899939122970146113', '1958315371562593', '4730065388138673', '1578156896561709', '1747032646321997', '83857567023455911', '1183617672421279', '2365033967361991', '23499784458751061', '6899939122970146113', '23499784458751171', '4730065388138673', '2367235344842569', '83857567023455801', '1747917883129309', '8237246248760201', '2077114373769567', '7209665752499559', '346737549084742381', '7209665752499559', '2077114373769567', '8237246248760201', '1747917883129309', '167800116780413707', '49423477492561163', '2077114373769567', '43257994514997311', '346737549084742381', '43257994514997397', '2077114373769567', '49423477492561249', '167800116780413621', '5103468062957531', '47121314048901', '4694770969783799', '15376568052139848141', '5174020114529937', '3223221618792101', '-5552243597090361', '-4007925059058647', '-1110448385203599', '-4953968855425963', '5176577044534093', '268314543781292', '2588289357803217', '-990793896415619', '-2637551939100599', '-6225042810171857', '-164846943972777', '-3026616696968257', '-670070100350509', '-4822410047416943', '-5360560819897033', '-4848711810474137', '5000524497735981', '8430960434986529', '5000525223425099', '-606088983868529', '-2668444676092311', '-1252945205382681', '-1334222394247009', '-745420258409709', '-2763633946371043', '-2346302365394969', '-5527267331859411', '-316343233253404', '544843395110647', '8683689779342297', '1362108483366323', '-5061491730584367', '-134427542355377', '-1473482463866529', '-2688551124608683', '-6113843203649489', '-672231148184191', '-73589678937341', '-5377849858954423', '-5061401561884117', '5448365906067631', '4341746930755621', '2724182616293363', '-2530700654664391', '-5526424646763049', '-4698570624671073', '-110528506404879', '-764218288948423', '-533919697806111', '-4968108793716643', '-333699869807859', '-2421963273549699', '2494893996786323', '4212853456700391', '4989787054708003', '-121098154875629', '-5354082632606591', '-60612794466909', '-5354083571471247', '-5960211046708009', '-132006096665807', '-6214239420258627', '-1320060966658069', '-2479512607000057', '2592932773598159', '1718642302927909', '2592932773598161', '-2479512607000059', '-2775664234350847', '-1011390650925809', '-2775664234350851', '-1514255948541151', '7478181269629449', '1151067735965997', '4127594609831309', '4446190563442297', '3303481729190439', '6917993935151981', '5202762889580781', '1300240004911417', '1654627218479413', '6448168872748063', '1656727835955797', '8146546509658189', '4031754907744717', '1998374327300679', '184822601389663', '6065790471469997', '5876468997001', '8242043806939809', '3996572803006237', '8063947236249189', '35401775008123', '4221441982528301', '121913468063511', '8242043806939809', '2600073429165103', '5203008929255889', '8285570320743993', '1622655699125303', '8287104569880493', '63743113414077', '2301329773660519', '7477417372975329', '5887210046894369', '890021578638723', '1650845584447227', '1729673895375661', '4308336352712111', '5510398016476481', '123623919348367951', '7726494959273001', '5510398016476481', '4308336352712111', '4308336352712111', '5510398016476481', '123623919348367951', '7726494959273001', '5510398016476481', '4308336352712111', '8604022820299271', '5907775091275611', '24705287142546721', '7720401504450153', '5907775091275611', '8604022820299271', '8604022820299271', '5907775091275611', '24705287142546721', '7720401504450153', '5907775091275611', '8604022820299271', '1701633885645557', '1170910984363241', '124319864202886571', '776999151268041', '1170910984363241', '2127042538955887', '1701633885645557', '1170910984363241', '124319864202886571', '776999151268041', '1170910984363241', '2127042538955887', '2162457153189741', '2895072774888939', '61609397706328161', '385058735664551', '2895072774888939', '8649829340354721', '2162457153189741', '2895072774888939', '61609397706328161', '385058735664551', '2895072774888939', '8649829340354721', '864448545545251', '1471481070175919', '30803413554007021', '1540170677700351', '1471481070175919', '864448545545251', '864448545545251', '1471481070175919', '30803413554007021', '1540170677700351', '1471481070175919', '864448545545251', '4323001214120563', '5838599952322243', '24645035683024507', '7701573650945159', '1167719408387827', '8646003155836903', '4323001214120563', '5838599952322243', '24645035683024507', '7701573650945159', '1167719408387827', '8646003155836903', '4276239890607341', '8420672174338169', '2017352694336973', '4519847312813383', '99946871890510986001', '871148200659257', '-1643854501713433', '221900637823473', '-3287709003426863', '-5700052638711477', '1650216834527153', '1650216834527151', '-5700052638711479', '-1574510552393599', '-6298042209574389', '-787255276196799', '-3255588611086733', '1003673585117707', '-3255588726129749', '-1115799228828631', '1611845546479147', '805922738726667', '-2789498043310823', '-1549382392988309', '-3098764555890581', '-3873455838667', '-411945673968661', '304902242187423', '-6591130435527767', '-55772690256783', '8049536734959853', '4024768280487223', '-5577269025678299', '-6210002632793491', '-388125186297769', '-3105001403389449', '-406185098687729', '174162461588241', '-1299792247346319', '-5621449227205149', '2032370564790079', '4064741044012143', '-5621449227205153', '-6226832390499977', '-97294261449563', '-6226832561636003', '-6485512566984647', '35296284046703', '-6485512566984647', '-5592510912523153', '4041725338870397', '8083450677740791', '-1118502182504631', '-6203142294611033', '-6203142294611023', '-6203142294611027', '-6492943690250503', '8900921258477501', '-3246472890888333', '-5606553528570259', '1013213920644951', '8105710585460421', '-5606552920082837', '-3107395083493491', '-124295834527707', '-3107395473343083', '5300083195973641', '8922645476369911', '1240490439956893', '1085485727655461', '863836353422437', '5746668139583309', '7397535336742687', '1715816839731561', '8427884386232443', '1845867098115071', '6176159972781039', '8317692835900777', '2772384323154991', '8223374475128957', '19428379644527', '7381395528783807', '2800671817458429', '8321994316473451', '2657239772624579', '14836808865909', '2657239772624579', '16153304535037419', '8076652267518707', '2657239772624579', '8076652267518707', '8076652267518707', '474778119016717', '2657239772624579', '5471235246808981', '5471235246808981', '8606340748458127', '1044733425059161', '8606340748458127', '8606340748458127', '8606340748458127', '8606340748458127', '2611833562647903', '8606340748458127', '2611833562647903', '2611833562647903', '5471235246808981', '8606340748458127', '8606340748458127', '5471235246808981', '1673440360101097', '4984869341891543', '1178680498686979', '1302561166882769', '21341169608787618281', '5893408314200779', '4984823867158077', '3353488017075781', '1673440360101097', '4984869341891543', '1178680498686979', '1302561166882769', '21341169608787618281', '5893408314200779', '4984823867158077', '3353488017075781', '688403335677869', '1241399368148379', '1124699516579603', '42964838213952335691', '1048946245457821', '1124699516579603', '1241399368148379', '1101477933375657', '688403335677869', '1241399368148379', '1124699516579603', '42964838213952335691', '1048946245457821', '1124699516579603', '1241399368148379', '1101477933375657', '961431808949381', '5014141472109479', '5885045531100877', '325625503495553', '2134024813137749589', '588510446635383', '2506849728856163', '4871274779194773', '961431808949381', '5014141472109479', '5885045531100877', '325625503495553', '2134024813137749589', '588510446635383', '2506849728856163', '4871274779194773', '8814337180505583', '1061504861768933', '2894474656617369', '21352365218838161159', '651619654285903', '2894430818973553', '5308217707592497', '8601191644111363', '8814337180505583', '1061504861768933', '2894474656617369', '21352365218838161159', '651619654285903', '2894430818973553', '5308217707592497', '8601191644111363', '6525022474493477', '4928594134942387', '2911552890687497', '2137484157770787759', '1043682449716007', '1455721602815661', '39441972038747', '400381332383247', '6525022474493477', '4928594134942387', '2911552890687497', '2137484157770787759', '1043682449716007', '1455721602815661', '39441972038747', '400381332383247', '1248010134779787', '1317339554421231', '1437704270845611', '5217489732772437', '1068558959102043081', '5751154324031323', '658345724235439', '518277868419481', '1248010134779787', '1317339554421231', '1437704270845611', '5217489732772437', '1068558959102043081', '5751154324031323', '658345724235439', '518277868419481', '1983290307734411', '49997459801427918369', '196923424931003', '466917699044541', '142732298384869', '187745145652271', '-810319843058551', '2763053652628933', '553238326039339', '-506159839631807', '-3359675020304929', '-3358891591335871', '-6074109080752959', '1934747188862957', '615014428489329', '483882410867967', '-60755664853169', '-3359283305820401', '-8041303770960583', '589694698272009', '943530162439987', '-4020616925721347', '-6861879414657611', '-6861856108151649', '-3140728463507873', '4043605078791383', '5088385091134149', '8087256770594707', '-6281478776865087', '-1372373552280927', '-7979163538549369', '5880665256576147', '236920128622597', '-317921632013433', '-3246713985410051', '-6482859250029817', '-714498340113027', '6936957876924917', '295246486014963', '6958049964666643', '-5725780820945563', '-6488143610424963', '-7743087524496263', '4914578929572769', '2418283243743881', '-780065425996663', '-408952460261351', '-1312543213203227', '-5890579797589703', '7317452265607161', '302348470323432', '7278564893435531', '-146812221872383', '-6552977715098877', '-7657654009814011', '1806565630723251', '1791507532735489', '-1925105982814373', '-5872505465060363', '-14718748281381', '-4755211959271331', '4749688124714051', '8378817611672833', '2359933058688587', '-4741586601491081', '-2940001194403191', '-7554915440270631', '5881143790451967', '6002658199109033', '-7467336012137503', '-1510197762589629', '-6010509904659357', '-5033255527351867', '1075836350964301', '8753539172438539', '271974179475343', '-2530502816851981', '-301282523875447', '523241478683047', '90563251229641', '4488983811146823', '1639062671613253', '7802259228970301', '278549503409047', '2095816727796927', '388274654222609', '448209998988207', '1507510196051559', '3642908778861141', '904804139127457', '988124988718507', '7364966937286169', '7221335446139807', '3968016417190357', '7309782278357579', '1805195154782239', '5099383701608719', '4333741220068943', '4686346499859763', '98092426916414021', '7879932777526827', '3939965661167643', '1532694170568969', '4686345044668221', '4333741220068943', '5099383701608719', '40273675935260121', '277880831762021983', '300447569668634463', '7847915556712630467', '2016741392508308503', '2016741019979274007', '7847915546990730911', '300447486258274931', '277880841483921139', '8054733242672193', '234557017712141', '8385832537465841', '4487993080245643', '488756994060507029', '3990996812527677', '3990996812527677', '3818414016097709', '44879945354371', '8385832537465841', '2345569085727817', '36422378390770929', '269453385642078543', '288338301578892783', '7821218626328176647', '2042283635112552717', '2042283635112552717', '7821218649411279663', '288338371628038583', '53890672511794219', '36422384011580889', '1046418845429211', '8807068110979677', '115622426212737', '3838696408041291', '7848969344643813', '7848969344643813', '24567663996383131', '115622426212737', '8807068110979677', '5232091680561049', '20458106450998367', '28276672046652137', '148466976009889213', '3931295388897815677', '1004197807353848291', '1004197807353848291', '3931296509878886813', '148466972615837053', '5655334273568341', '20458099658710503', '1308049010830981', '580703766091981', '1096590296047783', '122792689504539571', '314169060697087', '314169060697087', '7674540911246293', '1096590296047783', '580703766091981', '1308049010830981', '10220272627654791', '596593612150133', '70425898406051169', '1964927149273935131', '251213189392595837', '251213189392595837', '392985318547608813', '70425896148360107', '74574199261075563', '10220274885345853', '4694553657920547', '9000048012708629', '4191085007070013', '488836768990527073', '797817389665459', '3989086220731487', '7638073787881183', '4191085007070013', '9000048012708629', '4694553657920547', '36522635856042323', '289035329813998181', '53852646771960577', '7822422094509460217', '2041378725509400489', '2041378352980346793', '1564484270439530689', '269263231113507881', '289035327067703177', '36522638602337327', '5108492909418923', '2351749454684657', '4323253716026419', '490457158972030653', '7879112161401481', '7879112161401481', '383169655446899', '2161626312316373', '4703497454177751', '5108492909418923', '8066988480828091', '12062277242833881', '55444247739380349', '7847847562035005517', '2016519703641916137', '2016519703641916137', '7847847544423702449', '277221186459010813', '301556855549891133', '8066984958567263', '27828365311805619673', '1281563788518321', '319870666142679', '339791787709653', '7041082362542583', '8446141069475201', '-27148645317557', '5468624375878793', '1367155939447103', '-1357432328150607', '-2474046222238413', '-7083869027343917', '-247404627283287', '-1327646916979851', '1307585211058171', '1307585058425759', '-663823397541093', '-5390834309280271', '-5513112426440409', '2783322206648803', '111332893460237', '-2756556158157967', '-1240346247454099', '-7971127367882843', '-2480692244200787', '-2704875184213019', '5359920777232013', '5359921018932589', '-5409750729257937', '-5459580199601657', '-5394792216392957', '337925246024569', '2703402403727021', '-168587272607387', '-2477480534726403', '-3422555994614863', '-4954961067705121', '-665024543090859', '657201506254371', '525761118072171', '-53201958385573', '-672847570988227', '-664272005826999', '5245682440584151', '5245681574858977', '-1062835141772873', '-2476407094559081', '-214927778640081', '-4952814191839203', '-5401507181628163', '1355086130096631', '1084069076678131', '-2700753759009641', '-5382669747758839', '-216198075928399', '2676038526454167', '5352076391503651', '-5404951783206319', '-4961430875970139', '-1588468058704877', '-310089406610001', '-5513074744633701', '556832268563311', '1392080650855919', '-5513074859105057', '-5457826959210347', '-212180373410313', '2609665318242613', '5219330676941133', '-2652255012574691', '-4947534643107561', '-7074465726255327', '-123688363712467', '-1358225473392803', '684514301723093', '5476114467937713', '-679112656373013', '-673711087961759', '2621976802862809', '1240480921394419', '19970331256317', '4097671654488783', '7576585887289593', '5540104025716707', '1300244923742381', '5413801502243579', '2861942350828919', '4097672590456499', '7139864259021373', '5038616663773307', '5551306275491307', '7913079905337519', '7269997836938871', '2038446122742129', '8254292005402167', '685414313937313', '393396735760293', '1123674646008479', '8060113346147681', '1978041793759127', '4128306256654477', '1375246525955317', '1341894616705769', '5446711762758713', '8099058833158253', '610373128107569', '1779300986839281', '5005174595235139', '4915312466598239', '2646897553619021', '1265479388693', '86865901588743', '758720717843309', '2769653898904931', '4198419753947663', '592292565673267', '798378705558273', '12211016616903233', '7266121270849', '908264867817837', '190797134639113', '798378705558273', '592292565673267', '4198419753947663', '269758605712360087', '36846982743379433', '408557948025846723', '1563222077111649999', '58340918458733931', '58340901688377743', '1563222075255555627', '408557948025846723', '36846973462908213', '269758614992831307', '4326623313971677', '2571677311165683', '986145063645067', '48995016385500231', '4742148510207169', '1185537491349661', '3827734563723601', '986145063645067', '2571677311165683', '4326624405365283', '277483695580792647', '40567033492045609', '2019045300169082859', '1567956479709092071', '60815461627972827', '304077374650936611', '7839780189992540167', '2019045300169082859', '40567060113227533', '277483738808801507', '8988056961071819', '473642699716487', '3992952948065803', '244201568690986917', '8476773260544369', '8476773260544369', '381564951079667', '3992952948065803', '473642699716487', '8988056961071819', '288711064475066713', '7359634851310091', '2043298667688911637', '7815543439832371837', '272349986058188313', '272349986058210301', '1563108687966469733', '2043298667688911637', '36798174256528467', '288711064475088701', '4700722640706563', '5160534785902377', '3941488832440921', '490124337603395897', '8743609946101723', '4371805700646623', '3829096387526531', '3941488832440921', '5160534785902377', '117518038732823', '150710698437097191', '20354565209122333', '1008733566650650829', '784256455160225979', '140185333072114743', '28037071478930931', '3921282274761654919', '1008733566650650829', '20354564169646789', '150710664551976159', '8803592934551743', '5268323364954367', '3931060927237603', '490718614195771813', '584440749526387', '584440749526387', '1533495669361787', '7862124037262489', '5268323364954367', '8803592934551743', '282693986679693227', '8233514829119497', '2011724181971602271', '7852476839906412487', '60042535306309519', '300212676531573623', '7852476839906386891', '2011724740765146719', '41167574145571457', '56538797335943851', '4636959872458883', '5272303729565477', '1964333634321793', '306800688096747', '8875066324104111', '8875066324104111', '490881240653172807', '1964333634321793', '5272303729565477', '4636959872458883', '9303952409593729', '41217384566893', '1005258295862023739', '1963764667412507373', '4452549494390037', '71240794186159461', '490941305982524733', '1005258295862023739', '10304343865804381', '74431621552668701', '1377119917046347', '5679021937039857', '2982454123305357641', '8795434763349911', '1720553588716117', '1772804607156689', '-344327320017417', '5568997115947619', '5568996866767139', '-2754618690907629', '-5389892483763999', '5330307070664623', '5330306828729741', '-2694945988334867', '-154775500819473', '-1589315964658373', '-1238204129334623', '-5449477510736553', '-5431755223800987', '5480271336343639', '1370067631163409', '-135793856162243', '-2646053024596377', '2600487065029847', '2600487518898089', '-5292107074527257', '-61732447550071', '-7114284540954131', '-4938595707959211', '-215329541591683', '-2703106753997757', '2681071478383761', '5362142956768093', '-1081242701599227', '-5495230931183949', '2770088901571861', '1108035560628629', '-5495230931183327', '-4952699592233373', '-7961351471847571', '-4952699592233373', '-1090056811844769', '-5303813770662303', '1045266562534457', '5226333115771819', '-212152560021611', '-5414282203012581', '5447269520716429', '1361817456358221', '-1082856455408521', '-123490331667997', '-1413381901339679', '-308725791181493', '-5381294806980531', '-215835554370443', '216651575593479', '5416289954958899', '-5395888859261723', '-1326035908121853', '16352496675903', '5232799501409653', '-2652071816243379', '-2472055370397913', '-3451016182147017', '-1236027826479289', '-2687744023062433', '-166120436802239', '5256468405244999', '1314117317784459', '-5315854315214381', '-5382711105922503', '2695091426277039', '2695090994375493', '-5382710769424643', '-1236459079523867', '-214701568822011', '-4945836316005723', '-671904931836827', '5453577472017563', '3538301658419661', '5038846141104457', '6121370384539511', '2048830679428701', '1354993572256171', '5269464708465671', '7554967044754189', '5573163792338593', '23118924133879', '819533956510793', '4963827478112507', '1358457579352679', '7094797040271097', '2526744644900413', '4121014374946717', '5948956899898417', '1070806613633533', '4972321107224709', '1889999282462369', '1393869295571659', '8242026867764487', '21266052302369', '5260399051813959', '1401865654297377', '2059991075353029', '554712685856151', '465102474919099', '8156085428876429', '1965507817023487', '492829705765767', '329591798080479', '690471073235577', '8201723365723263', '1242812553421131', '557027940775863', '607881483079693', '2003326939971307', '263695384014943281', '1275731405132879', '3189327785236481', '257515023452093', '2003326939971307', '607881483079693', '607881483079693', '2003326939971307', '263695384014943281', '1275731405132879', '3189327785236481', '257515023452093', '2003326939971307', '607881483079693', '193942333532031', '2518604681376627', '1309133289346969309', '6666216925185509', '6666216925185509', '2556900955755799', '2518604681376627', '193942333532031', '193942333532031', '2518604681376627', '1309133289346969309', '6666216925185509', '6666216925185509', '2556900955755799', '2518604681376627', '193942333532031', '123881010232013', '699557006834867', '523936046569466613', '6614657284990269', '3306901543748029', '1279079802544953', '2662044645061899', '2481210890018393', '123881010232013', '699557006834867', '523936046569466613', '6614657284990269', '3306901543748029', '1279079802544953', '2662044645061899', '2481210890018393', '5258653697172641', '1189249278486231', '5117184694791823', '3269406354283383', '1307762978270847', '5239998617583070521', '4757693786945099', '1314658694920313', '5258653697172641', '1189249278486231', '5117184694791823', '3269406354283383', '1307762978270847', '5239998617583070521', '4757693786945099', '1314658694920313', '152893718159201', '4969676841945423', '523659315142576783', '6655265648937281', '6655265648937281', '5113860499439227', '993935877706143', '152893718159201', '152893718159201', '4969676841945423', '523659315142576783', '6655265648937281', '6655265648937281', '5113860499439227', '993935877706143', '152893718159201', '5255081199282349', '1233339413123183', '1047060655820877197', '657500604941273', '657500604941273', '5112600858500379', '4933338735003971', '5255081199282349', '5255081199282349', '1233339413123183', '1047060655820877197', '657500604941273', '657500604941273', '5112600858500379', '4933338735003971', '5255081199282349', '19998874285314246621', '265807779634539', '7613813435323', '209497553109763', '2974102370017241', '196690253598411', '-316523422094241', '8816058084131699', '137750882298829', '-63304682981509', '-4490547004899973', '218664764062981', '3939910707528819', '109332396629467', '-4490547061046039', '-5709070941489419', '-5709071345741057', '-1427267785903809', '-7976112415575623', '4856679851233687', '4856679851233671', '-997014051946953', '-8492994079554291', '3934208879814479', '469000424333457', '1967104439907239', '-849299407955429', '-2773886244979397', '-1386943122489697', '-554777248995879', '-195024411597141', '2381822992219757', '4680217761286919', '-7915909918432599', '-8419147186557031', '7688173390982439', '1865956374757837', '7523474963420373', '-4172533827901269', '-1095353116619831', '-5518188926355877', '-5497477254727517', '-1537115992219879', '8422588294707597', '2105769129074917', '-1921310315143981', '-4382393527226921', '8469465175249273', '7715236236996849', '8469945850533487', '-1095625556056307', '-5579762669321893', '-5579642349601619', '-1115940501892351', '-1589480394665957', '95673048071783', '149489162834673', '-7947400762649197', '-426841778596597', '7968188614544851', '3767001112645901', '7968190228785669', '-8536836328607317', '-2777787583097463', '-2777787381317363', '-222222998576593', '-1538369245429413', '8548384340278871', '8548372260469919', '-7691855287003777', '-868823352072419', '1664928019845579', '3829694083697199', '8324628019418961', '-6950582286651', '-1388688668001429', '-2777378845978967', '-222190247279273', '5453055056511579', '2333420767594667', '4097666974653271', '6781964396667769', '2355842636924261', '8332821033550123', '8620208969386309', '7753928874735531', '8923840588583197', '8497994195185309', '3871474275629989', '7533044379023297', '841731071332909', '117778557522059', '8293520709037499', '533019756528137', '3874068800610977', '88955531658579', '8678351092305819', '8678351092305819', '3828386817620411', '42830150409303129', '3828386817620411', '3828386817620411', '3828386817620411', '3828386817620411', '1338442200290723', '3828386817620411', '1338442200290723', '1338442200290723', '8678351092305819', '3828386817620411', '3828386817620411', '8678351092305819', '641641983492911', '194851368147001', '194851368147001', '2418611845180583', '6260707235759693', '2418611845180583', '194851368147001', '194851368147001', '6260707235759693', '947010908483736439', '6260707235759693', '194851368147001', '194851368147001', '2418611845180583', '6260707235759693', '2418611845180583', '194851368147001', '194851368147001', '64164014174169', '14691712442103', '2563329824826709', '12213275300583', '10563648302588625497', '2578986776941139', '1563241758406449', '5128476456365227', '7393010698700391', '14691712442103', '2563329824826709', '12213275300583', '10563648302588625497', '2578986776941139', '1563241758406449', '5128476456365227', '7393010698700391', '58374419405459', '1199482867022653', '6324969139610099', '161226417287533', '2113260573296972667', '6325291464546977', '479538706559793', '7651623655197081', '58374419405459', '1199482867022653', '6324969139610099', '161226417287533', '2113260573296972667', '6325291464546977', '479538706559793', '7651623655197081', '303934968613073', '5123660595301459', '393437631010529', '263824537457857377', '5152814266074639', '393435311799039', '1280987544603643', '7577847283700911', '303934968613073', '5123660595301459', '393437631010529', '263824537457857377', '5152814266074639', '393435311799039', '1280987544603643', '7577847283700911', '393561805777563', '30550062164861', '637561943545661', '2109427511895827203', '5149969511464421', '637561943545661', '30550062164861', '7871220836040917', '393561805777563', '30550062164861', '637561943545661', '2109427511895827203', '5149969511464421', '637561943545661', '30550062164861', '7871220836040917', '1298304996755511', '1217792214730939', '6091900287683099', '5311113597199114477', '129665859306619', '6091900287683099', '1217792214730939', '1298304996755511', '1298304996755511', '1217792214730939', '6091900287683099', '5311113597199114477', '129665859306619', '6091900287683099', '1217792214730939', '1298304996755511', '1583722369881251', '4852565357334063', '1595703251907301', '10547517263411627963', '515015491377521', '319140577621889', '4852574452280459', '791802613485839', '1583722369881251', '4852565357334063', '1595703251907301', '10547517263411627963', '515015491377521', '319140577621889', '4852574452280459', '791802613485839', '1767900302646511', '52379760866999', '84865529694897', '8444973961022501', '9995910050635988151', '3615033970408659', '-4609539417509177', '577467622653169', '2018832152921619', '4647925173524351', '-462237391789781', '-1152350004646891', '-5747579940270849', '-1546732585786417', '487945711835371', '7864085510108271', '-3846735629720881', '-719333122719081', '-454722943798493', '2235861282053969', '3975185898717747', '8865847759665589', '-36236881040277', '-358826937753501', '-5760737229210487', '-7922742786622991', '441818811161511', '1094745282552083', '-7977810067555231', '-179718253644789', '-287166193777463', '460220404850431', '502322835840241', '71978270200994', '-2298332016021987', '-5742181941098267', '-2869983250289809', '-966566401740207', '79488249446161', '3978864037444311', '-965781935329051', '-5741074220838941', '-4512801124568337', '276544238543287', '988348183718137', '4424710874254587', '-1128200639450131', '-178741041059673', '-2859855892564267', '-493833315392733', '8726469756810737', '436323793596739', '-3950664229970359', '-5719712549519033', '-4716169656141701', '2383210999769359', '1647642247661319', '4766421999538723', '-589521207017713', '-5860045386721499', '-2930022693360747', '-7843110180775183', '3966129588107341', '1983064794053691', '-7843110180775173', '-2930022693360747', '-9000009365033141', '8789814872245479', '1972145915205439', '351602282266783', '-450005951700233', '-1429151844229807', '-714568266881917', '-198137409268719', '1767042837247579', '176709183073871', '-7925325476208661', '-5716576755987279', '3949543487724839', '7444091465001351', '7768649614948173', '1244231799499091', '7437518013049929', '7773965571285021', '3815212258713529', '3725194833238701', '973406389062777', '8117421081347303', '1148817526181981', '8947123102680027', '8099086596619449', '4558699927242957', '7316038658452307', '8121098307108021', '2296112528975137', '1029027875704667', '1544244647033117', '779814483849461', '7210865210905513', '7048593899015383', '1740947496188916579', '1409718634283929', '7210863028118299', '779814483849461', '1544244647033117', '1544244647033117', '5848609132288819', '21632593619045093', '1691662495490263', '1740947496188916579', '42291560035353593', '21632591098026343', '731076015681637', '1544244647033117', '6150148797788643', '7242197294789717', '776510364276709', '3524643094642383', '6963680803736331417', '3524643094642383', '776510364276709', '7242197294789717', '6150145887405547', '6150148797788643', '2715823985546141', '23295310928301293', '16918286854283443', '6963680803736331417', '84591434271417169', '23295310928301247', '10863295942184587', '6150145887405547', '6924891549581827', '7947389092638341', '1730259914458357', '3525812086156229', '558358757679577477', '352581244995409', '1730259914458357', '7947389092638341', '27050317825163', '6924891549581827', '11921083385869697', '20763119479675913', '676955924591401', '558358757679577477', '84619498292722531', '4152623693464931', '5960541946022663', '27050317825163', '7399428345675141', '3808147029637017', '1883158049361739', '3529091894161907', '6939747911172245371', '3529091894161907', '1883158049361739', '3808147029637017', '7399428345675141', '7399428345675141', '11424441088911037', '706184268510653', '21174551364971449', '6939747911172245371', '4234910272994287', '564947414808521', '2284888217782213', '7399428345675141', '1760998665142809', '6952984076635673', '3960164935923331', '7050243252676051', '1744861818640887943', '7050243252676051', '3960164935923331', '6952984076635673', '1760998665142809', '1760998665142809', '1738246019158917', '7920329871846667', '28200973010704209', '1744861818640887943', '28200973010704199', '7920329871846657', '3476492038317839', '1760998665142809', '7393266366369531', '3023454896051', '151747341722809', '7058811697711821', '1387961347086887179', '7058811697711821', '151747341722809', '7558635057340233', '7393256180028951', '7393266366369531', '45351820548271009', '45524205409336691', '33882296727515539', '1387961347086887179', '169411477852589713', '45524199624348709', '45351813236535389', '7393256180028951', '2762933378398299', '4244353359224673', '6533395587153039', '7416840229155311', '258699643845081', '19785571054595495661', '-654803448607197', '5722935305867057', '8963829312365053', '2861467275360103', '-5238427525928669', '-5631384655743051', '-317780655621511', '-2815692349799247', '-2736044514440747', '-1589714215635647', '-684011091663807', '-6266945988913801', '-327674128917663', '5732691535093409', '17937393627131', '5732692360326531', '-1310696593036257', '-1370644194842833', '-314871896650009', '-2741287977069109', '-2812206307400023', '-5162751249570543', '-5624411789566929', '-6269756108379801', '-2559912235514871', '678357858885187', '8853553802876259', '2713432709894457', '-639978178949873', '-2824438956279669', '-8800085880567569', '-2824437533130841', '-5367551420013591', '-6650652653654649', '-67094364130809', '-6198881856945987', '-1013742148148183', '1075118055646499', '4378172869838041', '5375590278232481', '-5068710740740913', '-5475378805061193', '-2670112559975529', '-2737689402530599', '-5452906951780939', '-5519999946193143', '-1363226737945237', '-3071453472527541', '-1020661252439427', '5391914135928549', '8833949342583713', '5391914135928549', '-510330626219713', '-5368217768538753', '-3276575206950017', '-1342054442134691', '-2816618166163719', '-1773200761436217', '-5633236332327449', '-6187361570276263', '-2534740469165343', '268837300222367', '8757620540298313', '5376747652990957', '-5069481776488117', '-546063697492163', '-5463378382179461', '-2730317275386893', '-1367088931060541', '-5401629221295809', '-85443017057409', '-1228711612124029', '2338670012884011', '1446693789859983', '1504496741945107', '313494178338631', '3884831071310297', '7772855123753577', '2568015102387581', '719848072688821', '138031916309457', '1942862305647729', '979477699434839', '485799441507003', '2273638578369169', '1642259572922617', '4147373848606417', '7213526838096437', '1139821322939163', '642836809802089', '3087654704941589', '8232869531674567', '1648411396375391', '1292037926696801', '5168009580892247', '5599328642799067', '1152821762105111', '1039325062721649', '3843629962672249', '227957957345171', '1803782339750031', '321541746641881', '1538183283162741', '8228446722788021', '2061606631195837', '5165232757922829', '32307476703547', '1122800830437463', '7622299146494423', '394154273945541', '101101630100781', '659651320425242553', '515345936580891', '1010634094947863', '6306700486180479', '7772435624023129', '7622299146494423', '394154273945541', '101101630100781', '659651320425242553', '515345936580891', '1010634094947863', '6306700486180479', '7772435624023129', '516453324754073', '3112222761275951', '1070053886467463', '5277946459120374617', '515415509470833', '1069516265972823', '6224770757849107', '4236189018487967', '516453324754073', '3112222761275951', '1070053886467463', '5277946459120374617', '515415509470833', '1069516265972823', '6224770757849107', '4236189018487967', '6477433447956241', '379178357389927', '1233899057107577', '5312264929668772047', '259387936018983', '1233899057107577', '379178357389927', '6477426899594577', '6477433447956241', '379178357389927', '1233899057107577', '5312264929668772047', '259387936018983', '1233899057107577', '379178357389927', '6477426899594577', '1043403712013071', '794327489096077', '308569387096147', '10548894979503311283', '2575413813355301', '1234277912182473', '794327489096077', '1669443610914449', '1043403712013071', '794327489096077', '308569387096147', '10548894979503311283', '2575413813355301', '1234277912182473', '794327489096077', '1669443610914449', '8499721244988609', '6272332163779351', '2672325020464787', '10543755533663246063', '1029646455072969', '534204015472623', '3136325061576163', '4350703584041', '8499721244988609', '6272332163779351', '2672325020464787', '10543755533663246063', '1029646455072969', '534204015472623', '3136325061576163', '4350703584041', '8217354523873743', '6346739114987639', '4963851199670107', '5274665478539696247', '5151040506386423', '4963851199670107', '6346739114987639', '1643470177178919', '8217354523873743', '6346739114987639', '4963851199670107', '5274665478539696247', '5151040506386423', '4963851199670107', '6346739114987639', '1643470177178919', '58651124057849', '150992705375643', '4999803833556277213', '1342650266605589', '6845054848691', '217548255738499', '-4576017081226351', '4557200408433667', '500026110172729', '452550121168739', '-456156309227587', '-7750300175665597', '2047566003908049', '8126258667606897', '-7795614043504663', '-1145086105916889', '-1148270603755211', '-716673971772531', '-910410501801101', '450213976302229', '8010995559457473', '8922968724417313', '-4533530518714313', '-7503494528572209', '233052161149557', '460969286469937', '-3780758386618689', '-2834070675253457', '-227543020649059', '-2839179216683347', '-4731629455124839', '2399022390501741', '4133628768234117', '4798046373352531', '-2365815100769227', '-121734765545921', '3859614405819103', '3859615998168191', '-7791022606415389', '-5861216597767609', '-5861215801593089', '-5861216199680347', '-4465572364430057', '8632523110259477', '7863139018779973', '8632529128605951', '-4465573437670487', '-3921174916306433', '2159458197226939', '4318919403627157', '-7842347117837161', '-17759032118744', '-1136577754682289', '-142072238142619', '-452068579533939', '8900433662467353', '3979377165603759', '1103006454840979', '-4503303929763203', '-7521745087656117', '7601985647955593', '7524781263420813', '-1894038556896433', '-5648504138920413', '-5667703448477341', '-5658103793698877', '-8972439398632239', '545294011041113', '1578019374690821', '1744941114262463', '-897244005237653', '-1956711999153627', '2134417158725133', '8537670029555081', '-7826846950623657', '-1423107578723489', '-1138485993246063', '-1138486028112427', '5839880860625967', '4081156592120743', '1518262439586221', '7109869604052127', '8101804154579023', '3785322365583239', '8242004282249161', '1757447876945119', '2294962182808659', '8461950247112903', '4627801625067447', '4628492616333511', '3439062355294701', '8144927488038491', '7577905888966353', '4187725154634533', '8425117053563771', '72322023291267', '5785665582807969', '6209611689333141', '6209611689333141', '5385093255506861', '6338888557270781', '5385093255506861', '6209611689333141', '6209611689333141', '6338888557270781', '945684563165626483', '6338888557270781', '6209611689333141', '6209611689333141', '5385093255506861', '6338888557270781', '5385093255506861', '6209611689333141', '6209611689333141', '5785671163939589', '6877278568620159', '7805015096936337', '7805015096936337', '6877278568620159', '43206297809376791', '5400787226172099', '7805015096936337', '5400787226172099', '7805015096936337', '7805015096936337', '7805015096936337', '7805015096936337', '5400787226172099', '7805015096936337', '6877278568620159', '6877278568620159', '2213468795341601', '8710041863840487', '246837393861161627', '7713668558161301', '8710041863840487', '885387008819591', '2213468795341601', '8710041863840487', '246837393861161627', '7713668558161301', '8710041863840487', '885387008819591', '4477668497948043', '8576616839960271', '124460731031343781', '777879568945899', '8576616839960271', '4477668497948043', '4477668497948043', '8576616839960271', '124460731031343781', '777879568945899', '8576616839960271', '4477668497948043', '4473004785284373', '8707871844761789', '24682604569852703', '964164241009871', '8707871844761789', '4473004785284373', '4473004785284373', '8707871844761789', '24682604569852703', '964164241009871', '8707871844761789', '4473004785284373', '273316704262901', '8681625897407049', '7729560312962899', '7729559585367151', '8681625897407049', '273316704262901', '273316704262901', '8681625897407049', '7729560312962899', '7729559585367151', '8681625897407049', '273316704262901', '2074147246155451', '8674085313192193', '61922835124153509', '7740354390519187', '8674085313192193', '2074147246155451', '2074147246155451', '8674085313192193', '61922835124153509', '7740354390519187', '8674085313192193', '2074147246155451', '546577346832159', '8668196799927773', '15472577026656147', '3868144256664037', '8668196799927773', '546577346832159', '546577346832159', '8668196799927773', '15472577026656147', '3868144256664037', '8668196799927773', '546577346832159', '666892400382561', '45311867739551', '581784830592973', '49964828063907375041', '2156318004492393', '1962777869702539', '-3045094482739027', '1109103590487273', '2218207356526233', '-3045094833842403', '-6615099818529341', '-6615098414115847', '-168404404729389', '3874421390062709', '-6736173380348553', '-6615099116322591', '-755884588069537', '8795286067749809', '4397643033874901', '-1209415340911259', '-3298867341362781', '-6597734682725563', '-6821622857435469', '3582210795358533', '-6821622857435477', '-3298867341362781', '-189754295524369', '1768844912654177', '55276403520443', '-1518034364194951', '-1320020140115489', '-6600100700577459', '-6727002646161271', '2030431129341169', '-1345400529232257', '-6600100700577451', '-6106074636006283', '889583981159493', '4447919733437379', '-6106074578552921', '-3316309690231077', '-663261892083527', '-6769525475171893', '8762012123513701', '-3384762852492671', '-829077393831089', '-6194809996014083', '2258782927785291', '2258782927785291', '-3097404998007041', '-6708976561773981', '-1677244140443499', '-853705593368991', '7722763851387851', '-273185789878077', '-1677244140443497', '-3051938017844611', '277856696875661', '1778282860004229', '-305193801784461', '-3316337771357291', '-3316337771357287', '-3389701347298361', '4695268860228701', '-847425336824591', '-265307021708583', '6791859234406827', '5617917069533', '7382597553588457', '506717669479453', '3548915211957089', '936109597759693', '3485449952924897', '7023120476699837', '7410327934813919', '4114228998889763', '2616794224352691', '7980975147477', '903441193708187', '1028523145493343', '5856793306543583', '4248047615363091', '5235421330641733', '28344720802747', '8811256546840189', '973096103796199', '428458653608603', '8569171616980563', '249112625854890899', '8811256546840189', '8811256546840189', '973096103796199', '428458653608603', '8569171616980563', '249112625854890899', '8811256546840189', '4334272388695677', '123489884525265603', '2176733545613151', '34011467334547', '3859058891414551', '4334272388695677', '4334272388695677', '123489884525265603', '2176733545613151', '34011467334547', '3859058891414551', '4334272388695677', '162878251408753', '123874554389613767', '8675245933472103', '8675245933472103', '387107982467543', '162878251408753', '162878251408753', '123874554389613767', '8675245933472103', '8675245933472103', '387107982467543', '162878251408753', '2187102228177693', '495358676629590471', '4330358078191939', '4330358078191939', '154799586446747', '1749680763908033', '2187102228177693', '495358676629590471', '4330358078191939', '4330358078191939', '154799586446747', '1749680763908033', '4383089722063209', '1543645975858421', '8703659326832401', '4351829299618341', '493966758840820379', '8766175806147827', '4383089722063209', '1543645975858421', '8703659326832401', '4351829299618341', '493966758840820379', '8766175806147827', '349617824921753', '247431153181191727', '8676476864253247', '8676476136657547', '3866111768456121', '349617824921753', '349617824921753', '247431153181191727', '8676476864253247', '8676476136657547', '3866111768456121', '349617824921753', '1510203353647057', '678464763475519', '4508516070645553', '268980184659501', '847284600808637', '49956916187505808719', '-3036714998111903', '8836275924621267', '8836275921403043', '-6073430053269707', '-1368780521942741', '7127484709296963', '-3421950845271391', '-3310583839642717', '-3310584413319999', '-6621168252962717', '-6125748196849623', '4464323727633397', '4464323897646991', '-6125748196849623', '-1352074284664981', '7339150496919831', '-3380186051689629', '-6645697876864879', '-332284859840527', '-664569753683771', '-1556679937229763', '567843370930219', '908549393488351', '-6226719748919049', '-54783938172641', '7174473322940373', '-6847992271580131', '-6735891125909173', '-1683972781477299', '-6735891125909187', '-3050966809308393', '4443908636467011', '1777563596721247', '-6101934329288997', '-339223734227709', '4875923966030571', '-6784471841865339', '-829012668742939', '-3316049964299547', '-1326420127854261', '-1221210875871541', '4448689265291573', '8897379212325611', '-610605494348927', '-843892395465381', '7787325606217537', '-6751136567041623', '-207170656535959', '-1657365199104923', '-6629460902785189', '-6106275406652761', '355831525798349', '55598674839677', '-1526568851663191', '-1693486531597879', '8986908702534203', '-1354789157034137', '-829190667086699', '-3316762838957209', '-1326705101460801', '6597230144361967', '1477633975681467', '3499160738222653', '771951928464011', '7239704955767381', '434544999416531', '561421289841911', '5116257000830569', '2571935393461143', '1702786863810219', '754759521291287', '5229730657942927', '2950285859430449', '7240517766073463', '3476213732442417', '8242038160544337', '31893196395027', '5235680988151789', '4981441259326729', '8577099159042549', '8577099159042549', '4981441259326729', '10512213697703841', '65701335610649', '8577099159042549', '65701335610649', '8577099159042549', '8577099159042549', '8577099159042549', '8577099159042549', '65701335610649', '8577099159042549', '4981441259326729', '4981441259326729', '8218412145439903', '1689036705816071', '68577059670582889', '8572132458822861', '8218412145439903', '8572132458822861', '8572132458822861', '8218412145439903', '1689036705816071', '8218412145439903', '73553560223999', '307916166338501', '307916166338501', '73553560223999', '73553560223999', '307916166338501', '307916166338501', '73553560223999', '4706871725642001', '4926797739064237', '2463398687633171', '4706872817035683', '4706871725642001', '4926797739064237', '2463398687633171', '4706872817035683', '945415757124677', '9843491857188307', '2460872964297077', '945415757124677', '945415757124677', '9843491857188307', '2460872964297077', '945415757124677', '4885564267333703', '9764249116333147', '195284982326663', '4885564267333703', '4885564267333703', '9764249116333147', '195284982326663', '4885564267333703', '2364214267318469', '19685633601565173', '1230352100097823', '4728429262232721', '2364214267318469', '19685633601565173', '1230352100097823', '4728429262232721', '4884757217864183', '9764652641067909', '2441163160266977', '4884757217864183', '4884757217864183', '9764652641067909', '2441163160266977', '4884757217864183', '8298458700934477', '3918151113759775787', '1480039800705131', '368711037151641', '760852512764629', '1591513835792877', '-458068001638079', '1014910664349857', '-7329088026209263', '-281816838057497', '-112726735222999', '-2254534704459979', '-901813881783991', '-1127267352229989', '-7329804306678263', '5075087884926437', '-7329803363187131', '-4509430505833871', '-2254716647095583', '-4509431407208903', '-4509432392816139', '-112735797500313', '-3651907441721451', '5055674793306621', '-7303814883442899', '-8992560360545111', '-8992560360545111', '-8992560360545109', '-8992560360545119', '-8992560360545113', '-355155012275149', '490542044303137', '-7103100245502981', '-8790719209886451', '-8790719209886433', '-1758143841977289', '-1758143841977289', '-8790719209886443', '-57047515712352', '315898737962087', '-1825520346263629', '-1123850788830561', '-449540440757533', '-8990806310644503', '-8990808815150653', '-8990807562897577', '-1776027164755327', '490617679647201', '-888013582377663', '-4395863725098587', '-8791727450197179', '-2197931862549293', '-4395863725098593', '-4395863725098589', '6761759629738091', '3380865808045141', '270754276427703', '137357020906853', '6768654387682933', '6859643735291139', '1271114359286733', '36003745452656827', '7309339831140793', '7309339831140793', '1271114359286733', '7309339831140793', '7309339831140793', '1271114359286733', '1800187272632841', '1271114359286733', '305175635093303', '305175635093303', '6103518548133941', '6103518548133939', '3051757085867097', '3051757085867097', '6103517078265807', '1220703415653161', '1', '1', '76293961642667', '76293961642667', '76293928982333', '76293928982333', '1533587303939437', '1444404341256549', '1533587303939437', '1461974132997519', '1461974132997519', '3067177689492161', '1461974132997519', '1461974132997519', '902752713285343', '3067177689492161', '997456649075011', '997456649075011', '8568007367973547', '10512619923331421', '8568007367973547', '8568007367973547', '8568007367973547', '8568007367973547', '5256309961665713', '8568007367973547', '5256309961665713', '5256309961665713', '997456649075011', '8568007367973547', '8568007367973547', '997456649075011', '1198672142298483', '6250912820963163', '6250912820963163', '2877294603839593', '1186946415075351', '2877294603839593', '6250912820963163', '6250912820963163', '1186946415075351', '955027613919450621', '1186946415075351', '6250912820963163', '6250912820963163', '2877294603839593', '1186946415075351', '2877294603839593', '6250912820963163', '6250912820963163', '1198672142298483', '8790828507156451', '7169430935749257', '7169430935749257', '8790828507156451', '43045702405336267', '672589100083379', '7169430935749257', '672589100083379', '7169430935749257', '7169430935749257', '7169430935749257', '7169430935749257', '672589100083379', '7169430935749257', '8790828507156451', '8790828507156451', '5312214518087871', '7611028497265791', '6455602875189289', '8069503593986613', '5312214518087871', '8069503593986613', '8069503593986613', '5312214518087871', '7611028497265791', '5312214518087871', '1', '1', '1', '1', '1', '1', '1', '1', '1', '4877453029301943', '9768304735349029', '2442076183837257', '4877453029301943', '4877453029301943', '9768304735349029', '2442076183837257', '4877453029301943', '2361973964706141', '307658051534117', '4922528460747987', '2361973964706141', '2361973964706141', '307658051534117', '4922528460747987', '2361973964706141', '1219411060504633', '2442052100348751', '305256558018327', '1219411060504633', '1219411060504633', '2442052100348751', '305256558018327', '1219411060504633', '58987707234337', '24618808311061', '61547016230179', '58987707234337', '58987707234337', '24618808311061', '61547016230179', '58987707234337', '1175615181161849', '4927900443838151', '4927900443838151', '1175615181161849', '1175615181161849', '4927900443838151', '4927900443838151', '1175615181161849', '470327549170737', '1231924142543557', '4927696933972087', '470327549170737', '470327549170737', '1231924142543557', '4927696933972087', '470327549170737', '8000447010374131', '1991613252476267', '498598212523467', '964220967661423', '1947882747741951', '3988506515882349341', '-275026920368049', '-4400430725888783', '-4400430725888787', '-8800861451777549', '-3556620759820171', '2456513078347977', '-3556620759820173', '-4400430725888781', '-4498313188807529', '-4498314090964223', '-8996628181928443', '-1799325275523011', '-3653917824344379', '2529339414372911', '-182695891217219', '-2249156819942937', '-2200155987598453', '-1760124062675513', '-2200155078344391', '-550038996899613', '-1778250572961857', '614105844859501', '-7113002291847429', '-1760124426377137', '-9003038497256251', '-9003040301117823', '-112538003763973', '-9003038497256227', '-1462834632081637', '506341331061143', '-7314173160408189', '-9003039399187031', '-564038999534219', '-4512311996273747', '-2256155998136877', '-225615599813687', '-3667745063468067', '5079334128799263', '-7335490126936139', '-4512311996273747', '-1127945071341891', '-4511779384157107', '-451177938415711', '-2255890142683777', '-7334439594351987', '634818709621353', '-7334439594351991', '-1127944958690583', '7474441380677491', '1475747076936397', '7474755021241709', '1844744363491733', '7373487912277779', '1474690477714501', '8719871630106499', '49474450187091353', '1736160238378203', '1736160092859047', '7730382841733023', '8719871630106499', '8719871630106499', '49474450187091353', '1736160238378203', '1736160092859047', '7730382841733023', '8719871630106499', '1733213612915867', '246962560974190873', '1741617635178571', '8708087448297079', '964697503805433', '1733213612915867', '1733213612915867', '246962560974190873', '1741617635178571', '8708087448297079', '964697503805433', '1733213612915867', '8866878756716437', '496341582509783487', '1725259842184321', '1725259842184321', '38776678857619', '2216724964248539', '8866878756716437', '496341582509783487', '1725259842184321', '1725259842184321', '38776678857619', '2216724964248539', '4092222509264107', '124027435704043669', '1730973826033107', '1730973826033107', '7751714731502729', '4092222509264107', '4092222509264107', '124027435704043669', '1730973826033107', '1730973826033107', '7751714731502729', '4092222509264107', '1712297402762029', '7717699291461883', '348445789309443', '2177780180518909', '98786765134907063', '2139859526029819', '1712297402762029', '7717699291461883', '348445789309443', '2177780180518909', '98786765134907063', '2139859526029819', '1711475524924637', '99658884380156489', '8574935965913839', '8574943241871609', '7785843793837733', '4279031509922561', '1711475524924637', '99658884380156489', '8574935965913839', '8574943241871609', '7785843793837733', '4279031509922561', '9997561282469596027', '887466820038993', '979287376133051', '700184236256107', '1178494782582527', '421400583706781', '-3318483339715073', '-51851304847508', '-1354648822887919', '4360867006674091', '-211663867918397', '-3055530926796811', '8903640367472173', '556477512309171', '-6111061853593623', '-103702607030556', '-3323059672228471', '-6646119684439129', '-6759830987660537', '3638761703085027', '-844978830959793', '-6126439000427173', '8929818328625877', '279056817457337', '-3063219500213587', '-6646119514448033', '-1653101460295953', '-6612400947639083', '-6795615393084039', '5862590716152383', '-6795603315498371', '-3036433828429041', '552470927493539', '4419769215958547', '-6072870676254499', '-6612403394411443', '-6728562398310373', '-3364281199155181', '-3428846945546867', '8264415538135361', '-171442347277343', '-6214639890196007', '4532499290618417', '4532499290618421', '-6214639890196011', '-3364281199155181', '-6663600564489353', '-6664185532356379', '-211554475267851', '851452314772007', '-846363245870937', '-614804963066213', '4482006051445693', '8963706384390829', '-768469517148801', '-3331946524211431', '-104118279300087', '-1665843748747771', '-343506275275311', '6606663542145519', '-171743395901863', '-765530676553221', '1114600200131187', '1114613021822201', '-382771480808559', '-3331736217549161', '1648406126404559', '4678401605509', '5236156870596383', '641798118865303', '7248186434095807', '1734881519635481', '554199758864287', '5087682463835557', '5027968543250351', '4747063546779837', '8541337453345467', '5022585886309333', '1543006583135361', '1464168754275627', '86275750666551', '7306429588510191', '3685222655552077', '6909953748307501', '1584176538618617', '6336959794363401', '5276179076414373933', '2480138273240353', '1240073502194857', '80508103582983', '6336962704746523', '7922136340622153', '1584176538618617', '6336959794363401', '5276179076414373933', '2480138273240353', '1240073502194857', '80508103582983', '6336962704746523', '7922136340622153', '809978763369207', '783358837239679', '1030802172459909', '5191029841109319', '5189876601894061', '2111098048381433943', '195835116361941', '8017540213570471', '809978763369207', '783358837239679', '1030802172459909', '5191029841109319', '5189876601894061', '2111098048381433943', '195835116361941', '8017540213570471', '265956352941901', '1587552893337791', '5152270355120957', '98169555449001', '1226367109152989', '10552040422136352387', '158745484981647', '8298800911855197', '265956352941901', '1587552893337791', '5152270355120957', '98169555449001', '1226367109152989', '10552040422136352387', '158745484981647', '8298800911855197', '1260312604897739', '6053152364958577', '5314009059231780681', '4938646972106459', '4938646972106459', '5189461971906037', '6053151637362793', '1260312604897739', '1260312604897739', '6053152364958577', '5314009059231780681', '4938646972106459', '4938646972106459', '5189461971906037', '6053151637362793', '1260312604897739', '8705531843092907', '778400490104867', '1031828493669783', '5175688359566903', '2587476380130379', '660373216180864869', '6227110788582569', '1730727360723137', '8705531843092907', '778400490104867', '1031828493669783', '5175688359566903', '2587476380130379', '660373216180864869', '6227110788582569', '1730727360723137', '744208667411149', '3149034104359911', '10568768573839299853', '973375430775899', '4866962646381807', '5160528256014031', '3149039197530261', '7448193385361777', '744208667411149', '3149034104359911', '10568768573839299853', '973375430775899', '4866962646381807', '5160528256014031', '3149039197530261', '7448193385361777', '1195611701795897', '1964677642214483', '3862154900204199', '1562143291637502583', '2044087265850331', '15313668769943', '-1429048751828317', '-714508088115157', '-7836368542247091', '4239980452812219', '8480482115192449', '-1567200385038331', '-4522000376113189', '2219137117629583', '991712877458273', '4438532036874903', '-90442352596241', '-5716129856117261', '-5699243456579089', '-5707614934581063', '-7645606350086159', '7832857710332071', '1559839204959149', '-3834632737427973', '-2276940666552347', '4506715497413987', '7998571043830629', '4490071742942631', '-4546315596689811', '-2851714597790037', '-1417367030243693', '-5690234881267341', '-7828158375286593', '8751562685037063', '1083506792519401', '-1577289776650717', '-2231554000907197', '4305365168031661', '785046764084001', '2132049135942031', '-1111104357286267', '-2839925750560527', '-5879309602328949', '-5879309815323973', '-7791284809040687', '7647900637518083', '1911974946384481', '-1947821178594059', '-297499750992281', '2428179196609227', '8304315866796609', '4856358700877953', '-2379998022729569', '-2939654854413229', '-1131524455548621', '-1415636136924807', '-7618390743463103', '1967700009423577', '3925500389072299', '-3816113142969481', '-4479583809628991', '8646597259649033', '1982744014343527', '8627035656147301', '-4475150039893187', '-1132016682544233', '-720083526390703', '-5759996413780599', '-7898441615831603', '8547281517377473', '534374012290687', '-987067429697717', '-2283956202273351', '4515668107459927', '7996196549639079', '1129251425738799', '-2284260239551587', '-576033231245311', '8278593008612083', '581969960375227', '2486024252645701', '1536574906094609', '7540955336018463', '987398794942477', '8288773444963199', '1160777343195323', '7776428386442771', '412100308218659', '184784995076117', '7534322875849983', '2046703773273339', '753900118263103', '7879698071979683', '5250304551680121', '753256306081999', '7889988092594577', '2553633262996461', '5237814906119313417', '1318847581078689', '4688072434437257', '2345448298625079', '6594247364137891', '51150510726721', '2553674735952879', '2553633262996461', '5237814906119313417', '1318847581078689', '4688072434437257', '2345448298625079', '6594247364137891', '51150510726721', '2553674735952879', '2443781637216889', '52367141888126199', '665679744821263', '2406855971444763', '19254856502707', '665679744821263', '5113978700012323', '2443781637216889', '2443781637216889', '52367141888126199', '665679744821263', '2406855971444763', '19254856502707', '665679744821263', '5113978700012323', '2443781637216889', '5107977480080063', '2559987766915363', '1638703858288883', '4708012281100717', '931917647605289', '163866165773597', '2621449825062461091', '1021321119662503', '5107977480080063', '2559987766915363', '1638703858288883', '4708012281100717', '931917647605289', '163866165773597', '2621449825062461091', '1021321119662503', '4771149013192021', '5120929385614909', '3315983370529483', '14379858795141', '2299491927455707', '6631957282314397', '524383429857278433', '4771075889820537', '4771149013192021', '5120929385614909', '3315983370529483', '14379858795141', '2299491927455707', '6631957282314397', '524383429857278433', '4771075889820537', '4840470325493599', '327297662062988943', '6668100363514211', '301597208712549', '18849835492133', '6668099635918421', '2557012984867101', '4840470325493599', '4840470325493599', '327297662062988943', '6668100363514211', '301597208712549', '18849835492133', '6668099635918421', '2557012984867101', '4840470325493599', '4865876805010833', '1318712470552259047', '3185342286786121', '399443461337013', '399443461337013', '3185342286786121', '2575610294047381', '4865876805010833', '4865876805010833', '1318712470552259047', '3185342286786121', '399443461337013', '399443461337013', '3185342286786121', '2575610294047381', '4865876805010833', '3393275426116177', '4426176241634459', '3562986054591373', '629891187382941', '1398337987819723', '4997941804446251281', '-5589682820992419', '-2794592293748401', '-545864214513583', '8410126449575711', '1918389095793081', '8412107951266107', '-8734721255410061', '-121966332390992', '886186401194769', '8863869100739463', '-7804455413637519', '-5589433704244611', '-5575581923191643', '-696947695759429', '-8597210673599103', '4051677768176251', '7567162146865109', '2025839241204329', '-4298605671595999', '-3980625127495713', '4771335592250953', '4771336306483291', '-7961249183642773', '-2787790872316769', '-5588450330259743', '-5596991640181451', '-1749819891503377', '8414404668624829', '961427923953089', '8380449709846017', '-2183439628486533', '-388826178265077', '1760031128487169', '1753156447166133', '-1950100718676569', '-1118544197044119', '-5608845574618661', '-2804654192584297', '-2161581526461117', '1021093320817293', '75988323976781', '8166906664020447', '-4322748289683477', '-1004942131745349', '2431337605782623', '486174352003679', '-4020413135812737', '-5609076979893627', '-5574759395999003', '-696844896717261', '-4286894253752551', '321954325989387', '7548388933001463', '4024430185379811', '-53586183378053', '-3997781695178763', '2420803327971823', '484160710046543', '-3997781028792051', '-1114951856973709', '-1142284706577457', '-5711423532887299', '-898909296741739', '8758969728140881', '394363876893853', '2189742432035207', '-8989092967417387', '-3954336545414401', '4394499115883037', '1757799646353179', '-7908673090828771', '-571142353288729', '7694904157163743', '1034186638171693', '8584164682764129', '2357160885270579', '6759584228310689', '515009176634861', '961665936186111', '330209453793473', '583462724615491', '7695157668282553', '8261936390385391', '7344069602468063', '1178665236454947', '5233045083947', '257675114189797', '934205495795079', '5465254788449539', '4111151085837417', '8638002464404369', '123213349929663309', '5990260710203491', '5990260710203491', '1925208592650989', '8638002464404369', '8638002464404369', '123213349929663309', '5990260710203491', '5990260710203491', '1925208592650989', '8638002464404369', '1729220415330667', '1925012691074741', '36785962234451', '36785962234451', '7700050036703191', '1729220415330667', '1729220415330667', '1925012691074741', '36785962234451', '36785962234451', '7700050036703191', '1729220415330667', '1080184444492037', '1925105537447039', '237914855914489', '1189573988534151', '61603383019071139', '1080184444492037', '1080184444492037', '1925105537447039', '237914855914489', '1189573988534151', '61603383019071139', '1080184444492037', '8502301795098127', '62134801655213457', '604897629910787', '604897629910787', '3883425103450843', '8502300339906653', '8502301795098127', '62134801655213457', '604897629910787', '604897629910787', '3883425103450843', '8502300339906653', '1725094601627049', '123362625622712643', '5892180624410781', '5892180624410781', '7710164101419541', '1725094601627049', '1725094601627049', '123362625622712643', '5892180624410781', '5892180624410781', '7710164101419541', '1725094601627049', '107708713465069', '123578615685034559', '5600613939451019', '1120123369966827', '7723663480314659', '107708713465069', '107708713465069', '123578615685034559', '5600613939451019', '1120123369966827', '7723663480314659', '107708713465069', '1543024207406359', '49976189721466153439', '1304608922443007', '19520977687833', '2568131425172687', '764537462267527', '-6177772044310893', '-1544443011077721', '-5561383411445663', '2008470200183969', '8033880800735883', '-2780691705722831', '-6471641745081217', '4701915212325367', '-6471641745081233', '-6177772044310889', '-1550798745359893', '-1240639088565589', '-1398211746174733', '4042048123991943', '4042048297012587', '-2796423521186239', '-6484362770251523', '4498679084330857', '-6484362539557353', '-6203195212133757', '-6188030730322991', '-30940154331327', '-557411558106221', '1610843145798343', '503388442769559', '-5574115224737217', '-404789179608319', '4617548609422411', '-6476628299033093', '-1237606159658839', '-1540719795522003', '-385179992406873', '-5520476689268383', '7959513787492763', '397975671964089', '-5520476689268383', '-3280438865378783', '6367965635961593', '-6560877034335611', '-6162879530298989', '-6201481186696609', '-6201481186696587', '-5587322555520427', '161478090353851', '2018476129423141', '-5587322555520429', '-6498280952275519', '296799765578927', '-6498280952275533', '-3100740593348299', '-6274610885801643', '-62746096171147', '-5672347937199867', '2051847766460593', '2051847925046461', '-5672348571543327', '-327914752189167', '2269468189106039', '-1639573126602379', '-6274610251458173', '932075305381293', '847308026978829', '2315139476668029', '554440303761331', '35641720308999', '4114897094427359', '3733158015165517', '8508263366465051', '3770170284741223', '1513356541702551', '541399919759461', '3643345546057869', '692019581435189', '2623325660996897', '8495442313074919', '1068363242824801', '4320254238508341', '180176331819871', '1865660070628973', '5757461793710271', '5757461793710271', '291509386035777', '1865660070628973', '5757461793710271', '5757461793710271', '291509386035777', '184983385140819', '2957861992959049', '2957861992959049', '1156146157130119', '184983385140819', '2957861992959049', '2957861992959049', '1156146157130119', '186768776875361', '57371848124639', '57371848124639', '186768776875361', '186768776875361', '57371848124639', '57371848124639', '186768776875361', '1867671733816859', '5737345161831409', '5737345161831409', '1167294833635537', '1867671733816859', '5737345161831409', '5737345161831409', '1167294833635537', '2312306025592241', '2957807511428911', '2957807511428911', '4624611687386607', '2312306025592241', '2957807511428911', '2957807511428911', '4624611687386607', '145778356101407', '1438608411653913', '1438608411653913', '93298140628943', '145778356101407', '1438608411653913', '1438608411653913', '93298140628943', '6314521432632777', '5252590304408867', '2383095900282787', '4833223744215733', '855197215812970017', '295558521890613', '-975571112122021', '-7804568896976183', '-3042355212380613', '8267136401034363', '-243388416990449', '-7804568896976181', '-3902284448488087', '-312182755879047', '-95564084383691', '-7645126750695279', '-23652529646389', '400185072392343', '-5913132411597249', '-3822563375347637', '-7645126750695283', '-1911281687673819', '-7825390229104601', '-3912695114552297', '-3053478222727633', '2075304444089557', '-1221391289091053', '-7825390229104597', '-7825390229104609', '-7825390229104601', '-7825225217913091', '-489076576119567', '-6106780285418921', '8300947961881303', '-3053390142709461', '-489076576119567', '-7825225217913089', '-195630630447827', '-7645233729824517', '-7645235650371437', '-2956624659169671', '2000970322907423', '-295662465916967', '-1529047130074287', '-7645233729824523', '-3822617345048989', '-3903835806514861', '-60997449351461', '-6088028755131959', '4136110613879869', '-6088028755131961', '-7807673516987011', '-3903835806514861', '-7807672565008363', '2068103166701861', '8242045689072453', '8260966796910477', '8260681975917391', '1648403867850561', '8261176174620053', '1', '1', '1', '1', '597848936829173', '6670221676843941', '597848936829173', '8572995943163443', '8572995943163443', '597848936829173', '8572995943163443', '8572995943163443', '6670221676843941', '597848936829173', '8727690988162229', '1733632256994817', '495141145709879733', '7736580401716871', '1733632256994817', '4363847676868427', '8727690988162229', '1733632256994817', '495141145709879733', '7736580401716871', '1733632256994817', '4363847676868427', '8364596877896387', '4327751008769213', '3874291525607897', '247954680921968573', '4327751008769213', '8364596877896387', '8364596877896387', '4327751008769213', '3874291525607897', '247954680921968573', '4327751008769213', '8364596877896387', '1744788304034089', '868480777911753', '247306116415567347', '966039426298833', '868480777911753', '1744788304034089', '1744788304034089', '868480777911753', '247306116415567347', '966039426298833', '868480777911753', '1744788304034089', '4431222856849517', '34858238738487', '246760820915909811', '7711275653622183', '34858238738487', '69237896928669', '4431222856849517', '34858238738487', '246760820915909811', '7711275653622183', '34858238738487', '69237896928669', '2139252762957579', '8593152189837851', '124427983516007031', '7776721321110477', '4296556086034741', '1070053844004547', '2139252762957579', '8593152189837851', '124427983516007031', '7776721321110477', '4296556086034741', '1070053844004547', '2210105758087289', '8714643554127039', '61692622904448179', '481973570966267', '174292900186371', '2210105758087289', '2210105758087289', '8714643554127039', '61692622904448179', '481973570966267', '174292900186371', '2210105758087289', '1880288292498611', '302087544105981', '4996755745995979357', '114817069596729', '15408954990941', '221882812781379', '-169537799743077', '933941772269329', '-3390754773675027', '-1658895744303179', '-6635581756026231', '-61072970692293', '8896803260445483', '8896803871038751', '-1526824419955639', '-3317791183309739', '-6834729026291089', '2203224678395271', '-683472925554033', '-1339405565553041', '-6697027369266789', '-6177587083168343', '2251665120425823', '1801332027565891', '-6177587025856031', '-1674256899629', '-1692461575636513', '8545730882966469', '-3384923036347799', '-6636318912723989', '-414769960776559', '-3055486503658327', '8903786328420837', '8903786673196537', '-1527743266194817', '-663631914257447', '-52592902237912', '7562195078184529', '-16829721700859', '-6613731486828801', '-6613730083774251', '-3044665984273083', '2217949723801301', '8871799596732491', '-761166583759181', '-661373078530153', '-685747085359677', '154472468122123', '-274221206358961', '-6663898456745523', '-3331460692759977', '-306354864382947', '557670275498329', '8923234643425767', '-3063793599303473', '-6663409921132741', '-6734411670530503', '1873254236467903', '-6734410760728373', '-6617332255774431', '-6617333395925943', '-1218718847819479', '4439261059979109', '2219630531438959', '-6093594296684727', '-661733282585019', '8432473042082963', '2618188775156297', '84177664806051', '5064036552131147', '1265491255868027', '3747090159695047', '4106973084141623', '1308008919118957', '668909055343', '4754130762281117', '702911514412043', '1862333238922763', '5072760254049407', '3507672508236021', '3773844580186031', '6164686786648553', '7043848210481093', '3727679414661159', '7231981170938833', '7543427816432783', '1411963491554667', '1517609379993649', '6941312716749782063', '1517609379993649', '1411963491554667', '7543427816432783', '7231981170938833', '7231981170938833', '15086855632865559', '56478539662186687', '15176093799936497', '6941312716749782063', '15176093799936483', '56478539662186673', '15086855632865573', '7231981170938833', '3216582321162599', '6940686705732537', '7054899597250559', '1584974117073113', '3489333454152415769', '1584974117073113', '7054899597250559', '6940686705732537', '3216582321162599', '3216582321162599', '41644120234395251', '169317590334013387', '47549223512193361', '3489333454152415769', '47549223512193419', '33863518066802689', '41644120234395193', '3216582321162599', '7263087183328879', '304498040793257', '7059742357610583', '234986055489551', '6941252211896429329', '234986055489551', '7059742357610583', '304498040793257', '7263087183328879', '7263087183328879', '304498040793257', '7059742357610583', '234986055489551', '6941252211896429329', '234986055489551', '7059742357610583', '304498040793257', '7263087183328879', '639421766643467', '7947940907204179', '882407023454053', '431249683564971', '3489365366535156457', '6899996392231033', '882407023454053', '7947940907204179', '639421766643467', '639421766643467', '15895881125563173', '56474050189904577', '13799990562924257', '3489365366535156457', '13799992095616881', '56474048812214207', '15895882503253543', '639421766643467', '5951799177723871', '7236406238505163', '7052982257375923', '7762468888115461', '6963065268838363681', '7762468888115461', '7052982257375923', '7236406238505163', '5951799177723871', '5951799177723871', '43418437431030997', '169271574177022133', '46574813328692747', '6963065268838363681', '9314962665738557', '169271574177022171', '43418437431030959', '5951799177723871', '2978259801204067', '1560828134601243', '3526080851804859', '7197652483300223', '3481553582071096093', '7197652483300223', '3526080851804859', '1560828134601243', '2978259801204067', '2978259801204067', '3121656269202487', '56417293628877739', '14395304966600441', '3481553582071096093', '14395304966600451', '56417293628877749', '624331253840497', '2978259801204067', '903089339198413', '6349477593196651', '8514883959489039', '4197304010342439', '6085508378984794517', '1593031111668373', '-5480563405187681', '-1084775224499469', '-685070425648459', '-5468564836289871', '-689983084209981', '-2734282418144933', '-5090606583658919', '677595523281239', '2195778556971477', '2710382093124961', '-5090606583658919', '-6158547920499849', '-1134921975254363', '-4579315200315827', '-5674609876271799', '-216192950460111', '-6737604118468209', '-5404823761502757', '-5192390741389389', '1396980499768367', '1786923338034559', '5587921999073477', '-324524421336837', '-6247024276311291', '-5460197785187021', '-86941631713327', '-2730098892593507', '-2742589127965419', '-5364420663702497', '-685647281991353', '-635800522169147', '1082299457650227', '8778388582267923', '270574864412557', '-5086404177353177', '-6155730838893641', '-269925796731327', '-213345567601001', '-5398515922869129', '-1137235797211281', '-4525754347376847', '-2843089412701539', '-1299476097885911', '2797857183560317', '4472002586156401', '2797856924944259', '-519790434844097', '-3125949099575923', '-5638772722343159', '-326378045781767', '-2819386361171577', '-5495999830061931', '-795528983844773', '-34349998937887', '-5274975968114523', '725301734413477', '562872573952723', '1160482775061567', '-210999038724581', '-786441101738337', '-5484880891428601', '-1289833602487489', '-2742440445714293', '-5649471385168813', '-5132444062515781', '-7061839231461', '-5274122289094159', '5800568205380673', '2251223514503171', '362535512836293', '-5274122289094163', '-3145513446491639', '3127076865138781', '8215184416098939', '1287865163049659', '1285949867371427', '8247341754655267', '263160627856497', '8676101160690193', '1642835759033537', '1452797507498337', '911418597032547', '1858811446830187', '3279922849596223', '6327403981483023', '8221524788322447', '643775357602499', '514631585668441', '8245483148705567', '8208347123137', '540175852113761', '85062640787799', '1140023370374161', '1815296076230081', '515123267887733', '6558482498730673', '3223814327743031', '22519170524219', '7871943914797313', '7803523442076647', '61303535867551', '1578539246132771', '1554238516006887', '14355516142233', '1560891631084759', '983919656824633', '77802403964621', '493295094425789', '926373221928133', '1571022101158073', '8885827723388187', '1055883590070329', '7671055704765663', '98189540960676899', '1055883590070329', '8885827723388187', '1571022101158073', '926373221928133', '37177254730424529', '1004964839050613831', '71331275771059967', '1289273024263109', '1964034914403964199', '982017735460068199', '10314181916748819', '35665636746851957', '1004964839050613831', '74354511738205111', '880104759541899', '7862402441987167', '2337426440855829', '5259654735082993', '490721763032927063', '766752754738949', '5259654735082993', '2337426804653731', '314496155887151', '880104759541899', '282575721988546953', '402366564747153783', '300132788320298513', '41135038945524671', '7852490412417585409', '7852490407461977033', '41135033989911543', '300132829930816841', '2011833196264820563', '282575726944160081', '938974314494271', '985921869703681', '546517488290473', '515033308373041', '490010789626407729', '382820929395631', '1287582543336801', '546517488290473', '985921869703681', '938974314494271', '301062474669678431', '2018577304181848513', '56081525983158163', '40611970638331569', '7840763309933592303', '7840763328054034591', '40611965475708993', '280407648036233887', '2018577304181848513', '301062456549235359', '1795223494929667', '7988672535009023', '4241925461968893', '4728155514484189', '244164232435496881', '7630132263609277', '4728153331696919', '4241925461968893', '7988672535009023', '1795223494929667', '57663201023894393', '408803985984598437', '272563461713919313', '36744998185148267', '7814335670083810353', '7814335683866624893', '36744994505665191', '272563475496734397', '408803985984598437', '288315991336656881', '2163028964293911', '7889356088947197', '189673076007157', '10289432669743', '122486676917796711', '3827708653681149', '10289432669743', '189673076007157', '7889356088947197', '2163028964293911', '138731761530879867', '504769835784585853', '152036368621795057', '20280957523416437', '3919871569185564209', '3919871569185566139', '20280957523416543', '152036368621795163', '504769835784585853', '138731761530879761', '8399951861147129', '1996732668621687', '2267826298434873', '73924195028631', '122090032516433651', '476914189517319', '73924195028631', '2267826298434873', '1996732668621687', '8399951861147129', '13493285812777663', '102179349798488183', '72837255724626567', '1839096557790697', '1953707334437649047', '1953707334437649907', '9195482788954137', '72837255724627219', '102179349798488183', '67466429063887663', '1832665833527179', '4956602045428583', '214609303177807', '4811374295694863', '2828133065502957', '55357843466888539523', '-618302778843677', '-6854162338162551', '-2473211116365103', '-215208819505149', '5385633501931289', '1077126873133383', '-1345055206011339', '-531688239599583', '5258957509567859', '5258956643851437', '-664610257323603', '-5374807377874977', '-2472202098146077', '-6906925503755851', '-988880716251351', '-1326195376615363', '2616738326657919', '2616738196908317', '-2652390874931993', '-2698379696999261', '1083486411762589', '2708715901582911', '-5396759269750087', '-6720108415949', '-4938902812060221', '-709930583045129', '-493890280529529', '-1353812287823007', '5447889069947869', '5447889080382773', '-5415249346954263', '-1061359997155603', '209239614321211', '2615495177180147', '-5306799793498263', '-2691304711540483', '-2476089940831329', '-3990986344157139', '-2476089938723473', '-5494766058917899', '5538479095364927', '692309887856921', '-54947662127667', '-2704516898422723', '5367014271012857', '83859597933408', '-135225841127613', '-1362763293143613', '-617326046480363', '-3557723196642613', '-2469304185921457', '-2646088350686321', '1300257407830513', '2600514815661019', '-661522087671581', '-5431912821848003', '2740250936136373', '5480501872272743', '-1357978205462003', '-5383323771423241', '-77380473658394', '-7969908463479147', '-619043789267153', '-5392596516350361', '2667361719798011', '2667361719797973', '-5392596516350299', '-2754737476163321', '696060038943579', '5568480311548693', '-5509474952326719', '-5450469593104671', '7888843023622731', '2756949321332609', '8225799387903979', '5558323870001599', '8942646737013143', '2050406019969879', '2796141127937947', '2776066668568903', '2055508881255643', '1569150450343131', '8182238560157529', '165038674717773', '2487472658591087', '1115796482589317', '58944295443589', '525941883498069', '432138317541609', '20604991884393', '1359465916703067', '5050325327761227', '3535140325957681', '5410959669398967', '6038888006638501', '8242017457125913', '2630211276067043', '1394796264716019', '7536993124571047', '4973968550173397', '32118411869597', '24255344686659', '2724447990760901', '1261106078313257', '1764508600152431', '1358221569784209', '8222309682967861', '1221638740595837', '1747102707435813', '7218425670824959', '8581058967932377', '8237488259809827', '13869991477912665889', '8237488259809827', '4290310477648643', '7218427126016437', '1747102707435813', '167721857868610981', '43310556070176821', '8581058967932377', '49424931604086029', '13869991477912665889', '9884985502726379', '4290310477648643', '8662112142174311', '33544372391813023', '1746777912754299', '7829233179564773', '4845262513042781', '3947713102431661', '6900151913175686441', '3947713102431661', '2422628891835133', '7829233179564773', '1746777912754299', '33538135924882553', '46975399077388577', '4845262513042781', '47372557229179871', '6900151913175686441', '47372557229179993', '2422628891835133', '46975399077388699', '167690679624412643', '13970662888677', '376369006236919', '3768687190284327', '4039152373116361', '3459222771015301593', '4039152373116361', '3768687190284327', '940922333693349', '3492665358371353', '27941325929801617', '7527379972290763', '3768687190284327', '1615660918757021', '3459222771015301593', '8078304898680339', '3768687190284327', '7527378821994409', '27941322714523207', '174629686293211', '2025064433284337', '7536126573379497', '7505934261136403', '1383691064574923819', '7505934261136403', '7536126573379497', '2025064433284337', '174629686293211', '6985187451728443', '1012532216642167', '7536126573379497', '7505934261136391', '1383691064574923819', '1501186852227283', '7536126573379497', '101253221664217', '6985187451728437', '1397591372901349', '246987127759551', '472916388650271', '3910452053920311', '1724993212406269013', '7820902652649131', '472916388650271', '246987127759551', '1397591372901349', '3493978472330911', '1975896941921331', '472916388650271', '3910451893610157', '1724993212406269013', '7820902973269439', '472916388650271', '395179420446297', '1746989196087917', '436982456374219', '8250426478662223', '4131545141338339', '3598514680053879', '3467351139868581853', '3598514680053879', '4131545141338339', '8250426478662223', '436982456374219', '33560252649540037', '49502558871973249', '4131545141338339', '43182176160646459', '3467351139868581853', '43182176160646637', '4131545141338339', '49502558871973427', '167801263247700007', '5452080814854641', '3425240635542783', '1263953768625429', '3895173214746193777', '5587071289328857', '8813960230275351', '-2768141681049299', '-7987800350750909', '-1384093055259119', '-4927447228480011', '2558833640910797', '213913355753943', '639697221112057', '-4927413806023089', '-5267609590116853', '-1535822438952839', '-131692464947731', '-6035565313489467', '-5344883448302919', '-2398307039683619', '-2672442575227653', '-2410123336923591', '2468666788258773', '8399191634033083', '4937331874365143', '-1205061508885009', '-1065550965849611', '-2466821515903039', '-2663878265700227', '-1188892211860003', '-690485730685979', '-4717716636382693', '-5523886279737737', '-632654300357329', '5448071032982969', '1736653154027607', '136201788286043', '-2530617222198431', '-2690089376940741', '-91677752790953', '-2690089242892253', '-6113600642160619', '-2687320941638949', '-1478228938560983', '-2687320941638949', '-5061346116803389', '5448143221951313', '8683484892703873', '5448143221951309', '-5061346116803391', '-55296751221671', '-4672649843130247', '-5529675122167109', '-6113756352558387', '-166751481070073', '-2497105051766193', '-5336047383508783', '-2422054339224699', '499026223982449', '8425848983643359', '499026175536243', '-4844108638077563', '-5357443897964681', '-4823038633834631', '-2678721873605437', '-5960323651817107', '-5279240346885437', '-6251325109337031', '-659905043360679', '-2482180060866237', '1299378144528923', '8599271923261651', '5197512578115687', '-1241090030433119', '-2778573360885647', '-8056148220500167', '-1389286680442827', '-3030327992776281', '7461697672524161', '71944032691077', '1656671100570807', '940535831339039', '5043185619022049', '3459014468146657', '2601862470420621', '5197164379494747', '8313581354320873', '6745188251384613', '1667050029502409', '1639741955539311', '2015668018478241', '3995271195460501', '410235184423583', '638893432224117', '179624924078401', '2060497306317953', '7992562964581871', '1007670669400679', '12647047700563', '1687755253134091', '4688404781689', '824202122138011', '208045137068877', '5198986026453279', '8275924585290747', '3242385797235207', '8288336327590149', '2042641155596123', '2301515643374579', '467256999983903', '5314919108604699', '8835652178561037', '8249551135406413', '3459008899774011', '8621219004680871', '687858856754137', '3862228243887101', '38622282438871', '687858856754137', '2155305296867047', '8621219004680871', '687858856754137', '3862228243887101', '38622282438871', '687858856754137', '2155305296867047', '34440906207619', '5903144615400089', '1543488795444119', '123479126918594391', '5903144615400089', '34440906207619', '34440906207619', '5903144615400089', '1543488795444119', '123479126918594391', '5903144615400089', '34440906207619', '265970633909107', '1170596752106347', '24859505167800549', '7768595364937671', '1170596752106347', '265970633909107', '265970633909107', '1170596752106347', '24859505167800549', '7768595364937671', '1170596752106347', '265970633909107', '8653638403109029', '5775161750834669', '7699737879934051', '123195817720476051', '5775161750834669', '8653638403109029', '8653638403109029', '5775161750834669', '7699737879934051', '123195817720476051', '5775161750834669', '8653638403109029', '8647354827453653', '2943154317083481', '61595253531643647', '240606459107983', '2943154317083481', '8647354827453653', '8647354827453653', '2943154317083481', '61595253531643647', '240606459107983', '2943154317083481', '8647354827453653', '4324800397365721', '5827617404568479', '123201890760630193', '7700117444943641', '5827617404568479', '4324800397365721', '4324800397365721', '5827617404568479', '123201890760630193', '7700117444943641', '5827617404568479', '4324800397365721', '5311069877504911', '2970058755079009', '258084782646929', '5959389702513173', '19985612241533232929', '360727783367049', '-657353296122083', '546975685866073', '-3286765965498623', '-2851596880906047', '8256365479592083', '8256364964480277', '-5703193761812091', '-393752755504013', '-630004511828781', '-787505575397001', '-813433301782571', '2470128538925023', '-6507465953912111', '-5581438650887447', '8063527013240313', '8063527703762991', '-2790719382987281', '-1549674914093067', '-6198700577069161', '-6198700116720713', '-3294642900978499', '6062083136801539', '-1317857160391397', '-5578247225024433', '322051665883817', '402564582354771', '-1394561806256109', '-3105202802953449', '-6210405605906897', '-3105202802953449', '-259956776439241', '343454599065169', '-3249459590138747', '-5626545211827223', '2034448153311973', '2034448239825791', '-175829539671972', '-249223806376243', '-3115297810406561', '-6230595390109599', '-405209262117237', '8968963958559369', '-6483348193875801', '-5592908399499573', '404214138195687', '8084282763913729', '-5592908399499571', '-1240613614034163', '-3101534035085411', '-6203068070170819', '-324615424558191', '4396456944637901', '-811538590231997', '-1402600758834303', '1622408232457099', '4056020408123623', '-1402600744416043', '-3108765139081093', '-49740238534223', '-6217530047470029', '5302186692777263', '2222769328526387', '4803433899524393', '680183992786919', '8556655718991343', '153557297388381', '7383523719534923', '6379155262647149', '8463307195943581', '1844537252072353', '2963731665222547', '8302408932636649', '2770709009978853', '2053471534140343', '45396255265021', '7378893835433937', '1281995941941749', '8305523477659003', '2930988674235589', '14689847351443', '2930988674235589', '16160113487502289', '8080056743751143', '2930988674235589', '8080056743751143', '8080056743751143', '14689847351443', '2930988674235589', '3423752125920013', '3423752125920013', '7855993088563479', '5398531549094781', '7855993088563479', '7855993088563479', '7855993088563479', '7855993088563479', '2699265774547391', '7855993088563479', '2699265774547391', '2699265774547391', '3423752125920013', '7855993088563479', '7855993088563479', '3423752125920013', '1594423001317731', '2424527070386987', '6372181340274493', '527520498697717', '1287891842523723', '6372181340274493', '2424527070386987', '7972160845123441', '1594423001317731', '2424527070386987', '6372181340274493', '527520498697717', '1287891842523723', '6372181340274493', '2424527070386987', '7972160845123441', '396102204557027', '1217524633180007', '6086663103227377', '1062379030311540383', '1296849402235767', '6086663103227377', '1217524633180007', '3168818727849933', '396102204557027', '1217524633180007', '6086663103227377', '1062379030311540383', '1296849402235767', '6086663103227377', '1217524633180007', '3168818727849933', '1984689580821719', '4892807352562863', '254470892455521', '10550307828838332603', '5151517494549967', '254470892455521', '4892807352562863', '7938819441332841', '1984689580821719', '4892807352562863', '254470892455521', '10550307828838332603', '5151517494549967', '254470892455521', '4892807352562863', '7938819441332841', '170880565220659', '5123147650577663', '3139470579968161', '527817794373316299', '161073592514081', '6278479864222549', '128169959057969', '4142399129893303', '170880565220659', '5123147650577663', '3139470579968161', '527817794373316299', '161073592514081', '6278479864222549', '128169959057969', '4142399129893303', '7675542476042637', '4804734386547109', '1264415944441859', '5159308908069271', '10566395773947511123', '6322324194390981', '1200700291140693', '244198810308631', '7675542476042637', '4804734386547109', '1264415944441859', '5159308908069271', '10566395773947511123', '6322324194390981', '1200700291140693', '244198810308631', '7979578268967393', '2564940971085457', '781120040927579', '2112839090450674407', '5158163591646211', '1562108023214053', '160439959839173', '383999607289179', '7979578268967393', '2564940971085457', '781120040927579', '2112839090450674407', '5158163591646211', '1562108023214053', '160439959839173', '383999607289179', '8033530529493873', '6248575059525986741', '234044045997681', '1899138845439181', '1495760617799389', '2008855751572461', '-792082736934851', '4415343432620733', '17661409956807', '-7920813784476981', '-5713148860602377', '-2856572166155933', '-8986828463330809', '2187829465989397', '1576971072095979', '546958498569979', '-56167730961722', '-5713146596457123', '-7848861956751227', '246675639445429', '3946810773617783', '-7848861143014847', '-1175091286863919', '-5875456163074147', '-2369890883269751', '240853229009401', '1033325803076611', '2408532561339479', '-2369891010416061', '-734432037337109', '-3944711183858477', '271784330849321', '2174280709878293', '-7889404178465761', '-5715138626296787', '-285756628160653', '-281635627095291', '4405042381674397', '7904507462867563', '8810109015683733', '-2253087858832827', '-2857567797377463', '-7653355327825461', '1619387198317267', '7995058406153543', '-772469334840209', '-5664858835104197', '-284509562084419', '-4507297832492197', '2199978568343567', '3965597758246257', '8699207515844801', '-8968905337209177', '-567752503839629', '-7957598069603157', '877389818723263', '8833061392590557', '-791606946389501', '-5743321347030417', '-1432151572877987', '-4509831985617643', '8776446751957257', '198177118655327', '2208746608500269', '-4523121445369281', '-358497738704449', '-1912330719803723', '7940022354405507', '7813933308677053', '-3869031515755001', '-5708767218316487', '-1435032444159439', '-2291838215664271', '4559743320903477', '4012483980933531', '4497374462850151', '-4555311244281957', '-5724448497477123', '8133012911954429', '937058534726751', '4625503729857883', '4043777971846319', '1552602614243617', '1146941110169857', '4063034680162361', '4695578867486209', '2312705941841471', '3888430582688977', '7900180734613421', '3743412269841931', '6086833366661849', '493219372288717', '299643444874781', '18568073621181', '7880678893580921', '7490024227642403', '1020830639531201', '8643616750363271', '2349595422691417', '7661860842171983', '7882961922071977', '7882961922071977', '245179593515633737', '2349596150287203', '8643618933150629', '5104151014868647', '10063159604611769', '69294080793606409', '15066440063365117', '1961581516077515881', '504364419376512291', '504364419376512291', '1961581894915770137', '75332217290678729', '13858818389338653', '10063161548249061', '4693347980216729', '838132235107597', '1800418643131223', '3819035464655231', '7978122259078437', '7978122259078437', '97767326521624993', '1800418643131223', '8381319440692993', '4693345797429497', '36524272834330859', '269224826241834013', '289089493908398653', '7822407144302945673', '2041376786475862093', '2041376786475862093', '7822408632737402413', '57817899118005653', '269224734791208361', '36524253690403391', '5234581943321847', '8785975222662393', '2319130610839009', '98210773193951379', '7857932695979811', '7857932695979811', '7672716655777449', '2319130610839009', '2196494169463461', '5234581943321847', '20423519965617739', '141090411370267937', '5957566676054649', '3928945732947133999', '251325144646760391', '251325144646760391', '3928945735565723537', '29787832856555083', '141090432034740343', '20423522584208549', '1300504234564563', '4626853285687243', '351741460186817', '61396333216246151', '785574498902537', '785574498902537', '7674541652030769', '351741460186817', '4626853285687243', '1300504234564563', '20379049311752291', '148488323583272493', '141125602516007517', '157191773771241339', '125638292519245887', '125638292519245887', '785958868856206889', '141125602516006611', '148488323583271587', '20379049311753197', '46890204953537', '560727968153183', '2096080872076951', '122184184221270219', '7983025340928901', '7983025340928901', '1909127878457347', '2096080872076951', '560727968153183', '46890204953537', '36391572702849271', '11528532438176401', '269418942885830057', '7820908381421287981', '2042533896017811509', '2042533896017811509', '7820908381421273641', '269418942885843693', '288213310954423661', '7278314540567127', '1021500436165637', '4689232447115621', '8667315652101169', '490541002676052603', '3938495080326069', '3938495080326069', '1916175791703331', '8667315652101169', '4689232447115621', '1021500436165637', '4033532793022087', '150317783065902177', '138939395191821009', '3924590366166623867', '100799239580527099', '100799239580527099', '3924590366166624193', '138939395191821747', '30063556613180583', '20167663965109697', '2384772312374233', '5021093041514159', '127288317022733', '1131263962902867', '6584178700312677', '72682573824208405973', '-5434140763454541', '5477462277040713', '219098516621273', '-679267561661811', '-989389510297679', '-710193852932131', '-2473473726636031', '-663332001357161', '5222493950657381', '2611246705191297', '-5306656330125521', '-2695409330231403', '-5513281795705559', '1392193433392201', '1392193644461613', '-1378320681820253', '-620175588556503', '-496385922421247', '-4961404181263473', '-675581973488263', '535152069909659', '5351520382007487', '-5404655119924941', '-1364447591819747', '-5399638185121531', '677058927832809', '1354117824885161', '-5399637926903857', '-4951527440653001', '-6900434961840917', '-4951527690964643', '-5317031537383797', '2625629260250239', '2625629196655331', '-2658515835222831', '-2691402375461937', '-665235598493669', '2629438089466247', '164339880591639', '-5321884787949311', '-154804766054693', '-6898254131457117', '-4953752513750183', '-2699742468928487', '135351911968693', '5414076478747759', '-168733904308032', '-2692446698483121', '-135268465652517', '1340334931177683', '2680669862355539', '-5410738626101069', '-4961426291085717', '-1994844945620699', '-198457051643429', '-5513633664583967', '5567129801676823', '5567129801676467', '-2756816832291793', '-5460137527490889', '-5308857269238107', '261384621715221', '1045538486860889', '-1061771453847629', '-4948642206295647', '-1765519591504887', '-1237160551573913', '-2714482478811683', '5467907811074833', '1366976952768699', '-5428964957623333', '-2695011052085933', '5302369248219903', '15378221529601', '1460057407338799', '8088158536777689', '1896988709449723', '5540173114902017', '2746308916376829', '5365761948015801', '6182243895657217', '505473170671179', '3555693565189911', '625368819846437', '56450628012683', '314562749672011', '1559109083715399', '8050507146312387', '516189953205341', '276052913697969', '1573811172260149', '5565846512803579', '1636103089712001', '2696243652280169', '8248614951904607', '1104295249447519', '1355759713784997', '1614152672817', '8222300293852603', '113254119288747', '3571318228631237', '252383296635103', '621191113469129', '5239019654908811', '1644458180949519', '4466551828323', '7574236989314549', '2772816181573737', '32729875476031', '946819733073987', '3988933573282317', '195467752734992649', '4541030746635617', '1135257868557863', '152709181824213', '3988933573282317', '1183524120645607', '32729875476031', '269163887848099699', '36832041374505733', '2041293250777866409', '7819750838936612003', '58333343146626647', '291666743887718971', '7819750857348159347', '2041293250777866409', '36832042323753381', '53832773887310399', '134859541217423', '1287916314656671', '788381928667379', '980324179789558271', '4741963008072481', '948392819893219', '7658782654605927', '3941909279539023', '1030332542408317', '8631007727532101', '276847374111000147', '40558288371295629', '2017602709933206099', '7843248459529316607', '304140666214356627', '304140723578680519', '7843248472014187091', '403520504733739127', '40558280483482241', '276847268493877671', '559848150002477', '2362303169342111', '7984181875344579', '122136867281674439', '1058205181247561', '1058205181247561', '7633554205104657', '499011321734301', '2362303169342111', '8957566034465059', '57554941941773013', '7332878760375387', '2042818096156038087', '7817891976983915529', '272032983306972457', '272032997356127049', '7817891962934765609', '2042817909891522759', '36664379752722343', '287774584059633321', '4685596291310811', '1030155102158657', '1969187937126767', '7662269631194251', '874544572381963', '874544572381963', '490385302962557291', '1969187937126767', '1030155102158657', '4685596291310811', '12016875113429973', '40662488894488859', '2015904730924259689', '7846707820538155041', '280397978354085581', '280397981357470177', '7846708562592774077', '2015904730924259689', '40662485891104263', '300421880839133921', '8779501670743717', '1067303474918559', '3920617452680027', '491355880590715149', '4686835829840591', '2343417187324521', '307097425369197', '3920617452680027', '1067303474918559', '8779501670743717', '56427494263181963', '41498721144631489', '80246508517855773', '7862887517250890511', '60230182192381739', '60230165555397363', '7862887507303554071', '80246508517855773', '41498711197294233', '282137481263247071', '4620266848286301', '5252371477218311', '7854970198689713', '98205682266650401', '8877644588461403', '8877644588461403', '7672318927082063', '7854970198689713', '5252371477218311', '4620266848286301', '296612588869855601', '41103461238214151', '2009956860285021481', '7857370091911589837', '285000137410297233', '285000137410322653', '7857370091911564849', '2009956860285021481', '41103461238188731', '296612588869881021', '4691018766899237', '471184946845969', '352681359768961', '48167851562089180819', '177116203694879', '2235486542902707', '-2755903388572919', '1393214027574923', '5572856243319041', '-172243962523033', '-5389291775368311', '2663913096290773', '1331956577807391', '-2694645938805387', '-2478054002090767', '-7914392363610933', '-2478053876257083', '-5450757401073531', '-169826289496654', '1371137171805717', '1371137234804051', '-5434441778126967', '-105819439156453', '2598804743216261', '16242530402216', '-5290971878548531', '-4941680562213269', '-1416493425470377', '-4941679504468477', '-2692167114400181', '-1352913815614713', '5369651201653711', '671206422897903', '-5411656033408447', '-343593323403157', '2770662925386951', '22165304173081', '-5497492776020399', '-2478020302888047', '-995239725891011', '-4956039675230573', '-2726830001724417', '-5308189853927773', '2616562217304259', '5233124723755191', '-165880936465726', '-1353360434400893', '170113383286121', '2721813988549273', '-676680203146599', '-494357078018807', '-7034951392092311', '-4943570779098709', '-1345813810412289', '-5393864105805557', '2707734348578777', '2707734227828411', '-269693217637073', '-1058870272975029', '2608221239962097', '1304110561535859', '-5294350880298501', '-1237571679882063', '-6751566799533519', '-1237571798702433', '-2686129941069917', '-5321822329335473', '5265711091475253', '1316427772868963', '-332613895583507', '-5382761676143847', '5387589785091321', '1077517957018143', '-5382761676143207', '-989759435217547', '-6866182257732903', '-2474398588043867', '-5377933567196037', '5548847657422027', '884817729920501', '4997679333833339', '1252990046348769', '202194536223959', '671891309867313', '20696925248337', '7568667011296423', '5535937814925713', '2793861695181623', '202228827354013', '4931537230967093', '5433631136417269', '1780811562374627', '252411353275549', '8241981696793', '5676997550703813', '5179515658725953', '4974783398571111', '7561612149649383', '1386909499486781', '1648402362149539', '5277820325889', '5234728223172367', '2794521752127469', '2066259577700709', '2717314704129933', '8443672935435953', '251655766324467', '498313239295989', '3941648367472609', '8239243648679081', '5516359726510701', '8200818595706827', '77826242506111', '2779992310922641', '4845010821446529', '1603670891424489', '5273294931471628979', '1277586934602771', '3193966608911203', '5149702081515263', '1603670891424489', '4845010821446529', '4845010821446529', '1603670891424489', '5273294931471628979', '1277586934602771', '3193966608911203', '5149702081515263', '1603670891424489', '4845010821446529', '1204479550653827', '1239294643286967', '1308964257694302307', '6679784420322399', '6679784420322399', '79892837994037', '619647912815027', '1204479550653827', '1204479550653827', '1239294643286967', '1308964257694302307', '6679784420322399', '6679784420322399', '79892837994037', '619647912815027', '1204479550653827', '2386415796945439', '2313712060523951', '5242896314697995553', '6638637866423961', '6638637866423961', '2560007966161131', '4627430669410049', '2386415796945439', '2386415796945439', '2313712060523951', '5242896314697995553', '6638637866423961', '6638637866423961', '2560007966161131', '4627430669410049', '2386415796945439', '2550315161625689', '4634829705557867', '2620699046749601413', '6568335024751029', '6568335024751029', '2559276412841407', '4634829705557867', '5100631778442887', '2550315161625689', '4634829705557867', '2620699046749601413', '6568335024751029', '6568335024751029', '2559276412841407', '4634829705557867', '5100631778442887', '2433347089174553', '490353089957291', '1308977059113196041', '666760952297243', '666760952297243', '5113191637160923', '490353089957291', '2433347089174553', '2433347089174553', '490353089957291', '1308977059113196041', '666760952297243', '666760952297243', '5113191637160923', '490353089957291', '2433347089174553', '1274891075900121', '601919038768893', '5237684629804890473', '3298112916994523', '103065812651093', '511492275831513', '4811182095157327', '5099682174110647', '1274891075900121', '601919038768893', '5237684629804890473', '3298112916994523', '103065812651093', '511492275831513', '4811182095157327', '5099682174110647', '99977168681735930069', '772895243589859', '735985464482187', '2186178314737269', '4764964768400093', '239285880958689', '-7928634342077923', '8880087140934723', '4440042763138269', '-7928634162671471', '-2243021491699271', '136399487605269', '7870775481845807', '8729568372879219', '-2243021519731531', '-5708612467141031', '-2854306435402777', '-5708612668973293', '-4003358309430127', '2448839242047857', '244883999340617', '-1601342872957063', '-8511410695817947', '989107053855593', '750851795955627', '1978214859069503', '-4255706052307393', '-2778938124887467', '-2778937749208307', '-2778937937047887', '-804007550082007', '1217576968929949', '1217577526168859', '-8040072157386581', '-8634237238215039', '4073530263985741', '1897506463401979', '4073532492941401', '-8634239327860971', '-1120983978248689', '-5604918776765603', '-5604919334004523', '-487607389768211', '8812427623686723', '8812426050889801', '-7801717674578209', '-1751600103896729', '8443349377133233', '961964426653773', '4221673902168173', '-8758000435226667', '-2799305524756567', '-5598611442712333', '-1399652811528183', '-3986726186317507', '1204743757944997', '4818975031779997', '-3986726186317511', '-4276058872689793', '8005826608212987', '1884339332476913', '8005826608212977', '-4276058872689791', '-695495607093129', '-2781982428372509', '-173873901773282', '-7801556799377843', '2229690185849521', '8915877368731689', '-1950889195597567', '-8681015393600703', '8291733326165837', '1910590893492523', '8288884217151717', '-4339864796533309', '-1393217193897621', '-5573585278145199', '-34832668917924', '5645027338151901', '928032619534293', '323501942007169', '286242742703713', '4683244055063217', '4066562205849057', '8278920003704819', '7654644137249', '6122242221085419', '8265177961134989', '7659307825980387', '80984587328797', '1788449537134601', '468308455984607', '32493932931011', '8286410245950769', '7650515947624093', '354544483470247', '2123113873588771', '2123113873588771', '8591849101953743', '21204546040028303', '8591849101953743', '8591852861207967', '8591852861207967', '8591849101953743', '5301136510007077', '8591849101953743', '1325283724724531', '1325283724724531', '2123113873588771', '8591852861207967', '8591852861207967', '2123113873588771', '6296474618291541', '1556667236825151', '1556667236825151', '4699356457471613', '3168231968309457', '4699356457471613', '1556667236825151', '1556667236825151', '3168231968309457', '945531144582994529', '3168231968309457', '1556667236825151', '1556667236825151', '4699356457471613', '3168231968309457', '4699356457471613', '1556667236825151', '1556667236825151', '6296493187745141', '52170660593739', '5205023076659903', '1441046555027013', '21374248695030019801', '5218291148693439', '5764062165035159', '1041192335031723', '8214232800561479', '52170660593739', '5205023076659903', '1441046555027013', '21374248695030019801', '5218291148693439', '5764062165035159', '1041192335031723', '8214232800561479', '984869345939937', '245495070404071', '5701634747163', '2137741025485460011', '20876256421117', '1459588572899761', '2455407452253663', '958211912245449', '984869345939937', '245495070404071', '5701634747163', '2137741025485460011', '20876256421117', '1459588572899761', '2455407452253663', '958211912245449', '913216429033601', '5203804501253679', '1449010575805803', '5339452211660235019', '5214128356714563', '36220510011959', '1041907882120391', '832337981838689', '913216429033601', '5203804501253679', '1449010575805803', '5339452211660235019', '5214128356714563', '36220510011959', '1041907882120391', '832337981838689', '6872858602614499', '5004948416851897', '2945623451108897', '42679236496936409629', '208394651250043', '5891235624484761', '31281496039449', '2674643991733', '6872858602614499', '5004948416851897', '2945623451108897', '42679236496936409629', '208394651250043', '5891235624484761', '31281496039449', '2674643991733', '5743509293756191', '1241626000525071', '1123713238535677', '5245290306310169', '10742355292381242327', '1123713238535677', '2483252364848003', '5743333943187333', '5743509293756191', '1241626000525071', '1123713238535677', '5245290306310169', '10742355292381242327', '1123713238535677', '2483252364848003', '5743333943187333', '1061634636529321', '38769395357539', '5899552481226861', '1333750722473940811', '5209956119908157', '5899517192831607', '992549562885247', '1052116956151499', '1061634636529321', '38769395357539', '5899552481226861', '1333750722473940811', '5209956119908157', '5899517192831607', '992549562885247', '1052116956151499', '2001656264485407', '12747961968827', '377800050250451', '4583390416105329', '49988563431867491959', '4091979892528953', '-2615502822971947', '5820571048988577', '1119886219703147', '2923169274523687', '-2621447454124637', '-192553816191663', '-6148790985353159', '-7638154916829947', '2915230891037323', '1470583172228729', '-1900139201100953', '-3077628275871593', '-5691580104062421', '3426041592871397', '944069829251491', '6895776852221601', '-1142372970135177', '-3251603733799309', '-1296263145672079', '-8037287318307643', '6007348897021957', '761882137831807', '-3986418688785231', '-6492261597979503', '-5150306149142969', '5655238503429651', '4444074112768767', '1451019870128597', '-1304730053603419', '-6158003226218003', '-1216657558597891', '-309057119874349', '2919804654158829', '1227863183311589', '-7509594328567629', '-382540344350233', '-3027082038376251', '7695658880396621', '1227521658578723', '962702707616049', '-3028470000016321', '-3352443902117247', '-1675475356080377', '-4042140376942451', '1099978378227859', '68898052874909', '-4037721252477051', '-6703394614277997', '-1560224517867663', '799433748176597', '506666850824209', '3997192912136047', '-3120460302025111', '-6835950872995537', '-854490811260193', '-8025105539827493', '4756475851468313', '951314337967401', '-4012516854287953', '-1367187736307707', '-5839607148563817', '360706362787123', '4788628983622449', '3614223536494579', '-5846263269560983', '-1644195637142279', '-102650138448573', '-8055589126661923', '5872873761819609', '1475401078358869', '-125537840647338', '-657319570463889', '8053101991540981', '3606889252293377', '447551711052189', '4742001904153907', '7222767336140399', '7167749761050483', '6482749454429477', '1440314488107201', '7182293375942669', '4132945672656953', '8976154289225459', '471216519852991', '8222296538209371', '4458056179308293', '7655631653982727', '2067990788356919', '8970263715510801', '6084477566813433', '5565685044171131', '3668828273237937', '6762212679369857', '1788300206976469', '11250218562239516897', '1788300206976469', '6762212679369857', '3668828273237937', '173900793429617', '5565685044171131', '2751621204928451', '4057327607621917', '42919204967435263', '11250218562239516897', '42919204967435249', '20286638038109557', '5503242409856909', '173900793429617', '527144432585131', '3390367249424099', '7317641266221483', '3576913718372477', '1406266322683587807', '3576913718372477', '7317644904200497', '3390369068413607', '2628979350720061', '527144432585131', '13561469121067291', '14635282409072071', '57230619370588737', '1406266322683587807', '57230619617330527', '14635289931771889', '13561476150283533', '2628979350720061', '249095533086671', '7470372958382569', '3249918285513897', '7155958444500409', '11274252392974104303', '7155958444500409', '3249921923492851', '149407648342557', '3100865922195983', '249095533086671', '2988149274326049', '12999672687190481', '11449533420227633', '11274252392974104303', '57247668010868379', '12999688148836511', '14940764379390593', '3100865922195983', '6743580569357367', '7155511117408261', '221365816954957', '7180914397360233', '55992529205525655853', '7180914397360233', '221365816954957', '7155511117408261', '1685842027845543', '6743580569357367', '10733266676112359', '21251118427675937', '86170972768322861', '55992529205525655853', '86170972768322731', '21251118427675807', '1341658334514053', '1685842027845543', '1546693079114969', '6520625959270047', '1863586648369813', '223588426447197', '5637108699839383363', '223588426447197', '3727176934718059', '6520631780035541', '3082699232785407', '1546693079114969', '1304125165989839', '931793340350013', '7154829678640517', '5637108699839383363', '7154829613980091', '3727176870057633', '6520631909356393', '3082699232785407', '1683618619792533', '443583951858543', '7138974805120051', '7181457667566901', '174979032011458203', '7181457667566901', '7138976260311613', '443583951858543', '1683565687199483', '1683618619792533', '8516811488388067', '42833850767200099', '172354985958085417', '174979032011458203', '172354982085125831', '8566771125077977', '42584061314899921', '1683565687199483', '2182382018540839', '231655827154223', '680002541917717', '74195653374384323197', '154991630916073', '6297634550956439', '-3336052393203383', '435968000151813', '5345111657746091', '4359814687016711', '-6672205800530697', '-6304428694098299', '-7796873289985439', '-6304159323101143', '-383641996714663', '-912612726329501', '-1534500644109361', '-3639451584923951', '-3362916947940489', '2209590382680389', '2687722321787489', '8842831165343947', '-1681877305314083', '-6170032820063289', '-573700513851173', '-6165566322715827', '-6325067384795751', '-7938931598501259', '-1264120182752551', '-7315200599091909', '-3278097738915817', '8429411498962397', '1322982188232467', '8436617213586993', '-6558895793831917', '-790554283284273', '-7139805140086731', '-6317235811089459', '-3020648133967789', '-4702454369414217', '-6034097909742647', '-3606655340596333', '-12986814297451', '2088310108877103', '5236731539211663', '334131815084551', '-811678468978439', '-6134906285717743', '-8096480011652319', '-6134851344112369', '-3057193531605227', '-8260633791710601', '-191072878800159', '-1786734704092899', '-3281348185142073', '8445483648206703', '1323720181360693', '4225761807085781', '-3282479963998257', '-6048795889691607', '-2342537469877377', '-604276166940427', '-1580266892104639', '-7191976670331503', '-1578758323242959', '-1443409502897327', '-6494667147142721', '8355445743865111', '2618824382539831', '8355501105930291', '-6494687751996771', '-1529825105910979', '-8229326035856659', '-6119245753835829', '-122623996588333', '-4067065669139487', '-1532786255614379', '-1786984710805489', '919160373022317', '262015459850639', '330783417328511', '7530922419572963', '7467555708355341', '3587874844849243', '284321608959997', '2756724165317501', '7377517977190299', '3733103763192761', '1505779219954901', '7177953359481667', '4803386606303263', '5860689397480947', '1678215032714869', '3432990073273389', '448899450165903', '1452510537676139', '3073832268450009', '8230831712303947', '8241994871635237', '5128110036796279', '5128911069019321', '3978334667690651', '4753536535063171', '4200486835876693', '7304360033408997', '224452203353003', '3432537340014843', '5810859367663861', '384276849471613', '8229801706690837', '8243003747918747', '2563199455864071', '1025634043961337', '8043628000936801', '3984614245028461', '232898489712117', '153999716218613', '5219745547890523', '21380433901919808813', '4930624451910941', '5822113724425679', '7593442733467513', '3984614245028461', '232898489712117', '153999716218613', '5219745547890523', '21380433901919808813', '4930624451910941', '5822113724425679', '7593442733467513', '2017755247832247', '89834882678987', '2647725108155721', '1043672944212851', '42749016648424872959', '1059220792218081', '5749348090348479', '7879350443613733', '2017755247832247', '89834882678987', '2647725108155721', '1043672944212851', '42749016648424872959', '1059220792218081', '5749348090348479', '7879350443613733', '286671869515593', '2801441710948209', '626805540712189', '42973216281431070087', '5245753940604379', '626805540712189', '2801441710948209', '5733256946555077', '286671869515593', '2801441710948209', '626805540712189', '42973216281431070087', '5245753940604379', '626805540712189', '2801441710948209', '5733256946555077', '679742222368397', '2943417520031043', '5017315543858213', '325627501067161', '10670167170375131881', '5017401400157759', '735852924816243', '3386148807267181', '679742222368397', '2943417520031043', '5017315543858213', '325627501067161', '10670167170375131881', '5017401400157759', '735852924816243', '3386148807267181', '775259596867', '5790311565931411', '2628984861831473', '40730242615837', '21354510060916893611', '5258985447424889', '1447544331125747', '1558653287516817', '775259596867', '5790311565931411', '2628984861831473', '40730242615837', '21354510060916893611', '5258985447424889', '1447544331125747', '1558653287516817', '1375440956139757', '5876338679583277', '631685813458747', '5210212077911243', '21341039101936488663', '1263396274222031', '2938162609531359', '6849086117048753', '1375440956139757', '5876338679583277', '631685813458747', '5210212077911243', '21341039101936488663', '1263396274222031', '2938162609531359', '6849086117048753', '777496885553351', '512867634571221', '49988610557643148583', '4043250214054943', '58655220938833', '4604476453079353', '-1045533739962323', '5808924812841889', '4468205408899149', '588636180482737', '-5263403432442111', '-7894294738878381', '3240430731118221', '414791842974641', '-972658534291043', '-248700612277529', '-154466624220271', '-193690316808579', '-5959251876235483', '7454969525329123', '611702495872641', '936642221747829', '-2988506827913481', '-7815325006146553', '2332502411521529', '1185392469574473', '-7758793630202213', '-6620795772761741', '-6601679188251797', '-6611237480506767', '-390766746008713', '8018204096700029', '253851759035529', '4009126834665801', '-6252291173310463', '-1996925135832639', '920665596988951', '460342713020799', '-7987626184383189', '-170920784190517', '-1367361316260977', '-6836818974462777', '-6067446294295907', '1931227900016107', '4918583171973341', '7730787432885531', '-6070182614228899', '-4038912972937367', '5447017370541337', '5458799142869051', '-2017278023933439', '-6711713186518343', '-6708770726737569', '-6710241956627953', '-1057531816254799', '238560571376487', '180303040524347', '749201911549669', '-5301332056922443', '-7638264425632041', '5700666716261477', '5760227551841491', '-7594990064935227', '-6191441613480449', '-617658931006109', '-1236803092354153', '-6063182525845499', '7714903715021167', '2459466618934627', '7721405964524521', '-3033105016956109', '-8047177611030467', '5351856414723117', '1072978968392949', '-2009384452787417', '-6704392115808083', '-6701135492200097', '-837845475500511', '5832647018138363', '717043685865877', '450553335183481', '1312228013520631', '1789978247764777', '903275178532617', '20604991884393', '7673616652163739', '892016392402607', '4144955405136859', '4649748867126741', '4490553617667013', '206291902474279', '3591389160848599', '72165680837059', '1656686608058293', '5034094891133467', '8979098935704049', '7426602911386061', '6365967838090531', '6365967838090531', '5415834598840869', '6418301754563083', '5415834598840869', '6365967838090531', '6365967838090531', '6418301754563083', '1862868246069573289', '6418301754563083', '6365967838090531', '6365967838090531', '5415834598840869', '6418301754563083', '5415834598840869', '6365967838090531', '6365967838090531', '3713308528025947', '1367382838242411', '4310302096166417', '4310302096166417', '1367382838242411', '10445764306837191', '1044576430683719', '4310302096166417', '1044576430683719', '4310302096166417', '4310302096166417', '4310302096166417', '4310302096166417', '1044576430683719', '4310302096166417', '1367382838242411', '1367382838242411', '223122158692367', '8897812508540829', '7749415339519427', '1983864296756041671', '889789399926873', '440744348087559', '223122158692367', '8897812508540829', '7749415339519427', '1983864296756041671', '889789399926873', '440744348087559', '8907764411506061', '876361313707647', '500257274241696739', '3908237763342197', '8763546925861181', '8998956172449347', '8907764411506061', '876361313707647', '500257274241696739', '3908237763342197', '8763546925861181', '8998956172449347', '3613765605045413', '8903414939854291', '1983555085124112369', '1937059510151329', '4451688916234749', '3639237641918073', '3613765605045413', '8903414939854291', '1983555085124112369', '1937059510151329', '4451688916234749', '3639237641918073', '1916789598771', '4422903556490697', '7970870104114295693', '3892026418024559', '4422903556490697', '76636494509179', '1916789598771', '4422903556490697', '7970870104114295693', '3892026418024559', '4422903556490697', '76636494509179', '1318750954455803', '4417033018060307', '15952598752921167741', '1947338714956197', '883406676371643', '164469521259697', '1318750954455803', '4417033018060307', '15952598752921167741', '1947338714956197', '883406676371643', '164469521259697', '2938343663492341', '1765252104328933', '974058191870859', '15958978356308936621', '8826264159623499', '1464305398408623', '2938343663492341', '1765252104328933', '974058191870859', '15958978356308936621', '8826264159623499', '1464305398408623', '7995502789489', '1168286909823853', '1736385149632827', '24984189193385236787', '3549446889691943', '7585969681040707', '-5424279288161543', '4093797873873213', '1639037458930181', '-5431850773721003', '-5337047138538437', '-532190702419707', '-4697183442466377', '-5179418348337419', '-4666916133184777', '-666184635170969', '-5411502185907197', '8158844678012819', '4076304994592913', '-540528384688649', '-1328975009484419', '-5328334758839773', '-294573669763231', '-4772077518031421', '-2369018350528951', '-2661058699194361', '-5602626868919619', '211618007201773', '4230211666574147', '-55983388821497', '-5472500382934957', '-5481075378903873', '-190804203429027', '-27924442453501', '-478724855343811', '-2738393940459707', '-2370782714904441', '7281065725920831', '11649928678941', '-592713141028272', '-4404409664228437', '-352330421391509', '-3511492677861823', '-446528342391959', '-1755466942096323', '-8808539931622299', '-764367438205531', '2917860746666943', '1459103838652561', '-1911265505206139', '-7234586832424177', '-7231811387318749', '-1195812393622129', '-502764934206337', '-1493377895150151', '-3616599554935731', '-1391708853496881', '5297346657514859', '2649179955750973', '-6960569501327643', '-831104899734777', '-6644789601302733', '-5592024655564509', '-8470718339397747', '-1116784911753403', '-265872575983619', '6865004731466273', '6166314820455927', '1401372656694227', '8011431170080983', '6211343678144873', '2831734817939029', '4668952718191459', '6145674711965137', '2793965379668007', '8559763707036147', '1157955517686521', '1854842429387', '2287558843860471', '2297916934967269', '798732468936557', '709245069545773', '4617624880266977', '3521339330549551', '1217334092967129', '1000899814120826929', '35023886917157', '2188976743316741', '1563897373933871', '614147752465403', '1217334092967129', '1000899814120826929', '35023886917157', '2188976743316741', '1563897373933871', '614147752465403', '8742382036703259', '1587470671045394073', '8894344054257313', '1111792552034799', '968914931647061', '1093336721165213', '8742382036703259', '1587470671045394073', '8894344054257313', '1111792552034799', '968914931647061', '1093336721165213', '1159511843649123', '243727787605579', '8806429482282471', '8806451310156249', '7986487790479536557', '4607052157629639', '1159511843649123', '243727787605579', '8806429482282471', '8806451310156249', '7986487790479536557', '4607052157629639', '1537006760846387', '7792751446678779', '4413529691967959', '4413529691967959', '31919112905828528853', '3067918451979461', '1537006760846387', '7792751446678779', '4413529691967959', '4413529691967959', '31919112905828528853', '3067918451979461', '2875042823978427', '7749425313967493', '8903990280887647', '8904042667782723', '1983862007336953653', '2838746891143131', '2875042823978427', '7749425313967493', '8903990280887647', '8904042667782723', '1983862007336953653', '2838746891143131', '306286688117529', '31888428332147084651', '884339097003811', '552711981102117', '3892630411639049', '76536406372319', '306286688117529', '31888428332147084651', '884339097003811', '552711981102117', '3892630411639049', '76536406372319', '614982155268901', '1740934464218777', '1543048962740171', '7626389178205823', '2076139847558491', '49963884826050538831', '-5334783938490191', '8038203847146731', '4016357496803293', '-2664655492551187', '-4673996452312181', '-2289344682426979', '-1173968304834489', '-5251798906670841', '-5262743106192791', '-5257271006431817', '-136001426738899', '1026253915102863', '8209729438645621', '-2719878016573799', '-4689669626002367', '-5196753305583009', '-93817458789581', '-266978221157119', '-1068033293746959', '-5339865445938589', '-597493777332927', '4524412722996847', '2263231027980723', '-1195806185707583', '-5042097115411837', '-1544549038868593', '-5025729537355133', '-5810280533083761', '-2901047579275903', '-725773480727223', '-1878041074412711', '5732241504808483', '2866423642568117', '-3756687857918831', '-295317329916439', '-486791474052971', '-5901500925581503', '-7122113722805703', '-3559845585684717', '-7120902447087571', '-2900241616442391', '8779131460340191', '439345014723651', '-5808239062019827', '-613846336504481', '-151229563242301', '-4879759345533143', '-2829583814585367', '-564365804082127', '-5651412834996003', '-948403591967003', '3640867225144201', '455117192218974', '-1185539635800659', '-1756122891598633', '-7142230575789873', '-1755841766710947', '-1761953642199397', '-550575356883517', '-880948696056663', '4928662709546093', '2822882459469933', '1239945807257267', '7390227451553', '697444169483371', '6154935455156773', '81854944909779', '2194564798835473', '2258130226842653', '28543201120731', '2357019751125533', '4633954566579931', '1678837903829079', '110810871391009', '306638973438301', '8556571839702349', '18452972479047', '115819043506849', '2753996277680229', '8624417733448431', '8624417733448431', '2753996277680229', '20880962181794711', '1305060136362169', '8624417733448431', '1305059854612553', '8624417733448431', '8624417733448431', '8624417733448431', '8624417733448431', '1305059854612553', '8624417733448431', '2753996277680229', '2753996277680229', '6165199290610449', '38539737252283783', '7004241272260433', '7004241272260433', '6165199290610449', '7004241272260433', '7004241272260433', '6165199290610449', '963492577523471', '6165199290610449', '215402214938231', '12499722836992109263', '6103380291500053', '1140292988437617', '215402214938231', '12499722836992109263', '6103380291500053', '1140292988437617', '1659936912417043', '609473296856999', '6241021460976346509', '2418073661474629', '1659936912417043', '609473296856999', '6241021460976346509', '2418073661474629', '1454939007007423', '12492174310898143387', '762461083372163', '4194302048650453', '1454939007007423', '12492174310898143387', '762461083372163', '4194302048650453', '7210594465125989', '3116727485751512079', '6087354005034043', '2334167301482979', '7210594465125989', '3116727485751512079', '6087354005034043', '2334167301482979', '5557794260931503', '6091485008151289', '779713433804531813', '6748146347481277', '5557794260931503', '6091485008151289', '779713433804531813', '6748146347481277', '1582409485644821', '6089390183678081', '389721064887656141', '2033517228248529', '1582409485644821', '6089390183678081', '389721064887656141', '2033517228248529', '24988459770524436707', '2447122638992399', '1297428048831747', '1158709718570287', '1350731033287013', '270352279936001', '-6956010391717617', '4705691229140949', '-3463485045163411', '-222127886118587', '-9001276650306663', '-4442557722371747', '-281289895322083', '-8943196047525073', '-8506387447696149', '1443592597608129', '-8350698902836587', '-644082583503111', '-5464030471310517', '-5152653381591397', '-1366009439436003', '-663543196208463', '-599224882679299', '6499176641510713', '-2360666967169427', '-588698099823021', '-3088422209409959', '-5886983910437769', '-1544210376653091', '-1206382541705013', '-7767432862155433', '524539484538869', '-1500915329229179', '-564795313402951', '-5044077127824281', '-4518364695805207', '-1008814987848537', '-2390609908761973', '-8117885609875601', '5501189810446993', '-1584094183723857', '-302410548601567', '-654173129697091', '-4838555655064097', '-2616699080068853', '-5035976907600901', '-7939255599543487', '670335444286427', '-3842055044742229', '-4642853435027819', '-515314372586241', '-4642852705744349', '-128828611378647', '-2448999290222557', '1301384069158863', '3133469264924961', '4250846262501813', '1379787891179627', '4273231062394763', '1362387759953007', '3083061902608897', '77071043310487527', '1751177700059919', '1751177700059919', '3083061902608897', '1751177700059919', '1751177700059919', '6166118235049399', '2408467782549239', '6166118235049399', '1', '1', '1', '1', '6103512672495831', '6103512672495831', '6103518577504169', '6103518577504169', '1', '1', '1', '1', '1', '1', '1381297805474243', '947468515490721', '1381297805474243', '2017958184167139', '1614366547333711', '1381297805474243', '1614366547333711', '1614366547333711', '947468515490721', '1381297805474243', '1063915103267717', '1063915103267717', '2142922292335533', '2650903138049593', '2142922292335533', '2142922292335533', '2142922292335533', '2142922292335533', '1325451569024797', '2142922292335533', '1325451569024797', '1325451569024797', '1063915103267717', '2142922292335533', '2142922292335533', '1063915103267717', '929919365503107', '6367460664892891', '6367460664892891', '270511671123617', '1602737821639517', '270511671123617', '6367460664892891', '6367460664892891', '1602737821639517', '1863144502030100263', '1602737821639517', '6367460664892891', '6367460664892891', '270511671123617', '1602737821639517', '270511671123617', '6367460664892891', '6367460664892891', '7439361909979121', '5507202306207007', '8625046225824009', '8625046225824009', '5507202306207007', '20881000725271407', '5220250181317851', '8625046225824009', '5220250181317851', '8625046225824009', '8625046225824009', '8625046225824009', '8625046225824009', '5220250181317851', '8625046225824009', '5507199439973737', '5507199439973737', '616610023740519', '19267503111927157', '7004774207180387', '7004774207180387', '616610023740519', '7004774207180387', '7004774207180387', '616610023740519', '1204218944495447', '616610023740519', '1', '1', '1', '1', '1', '1', '1', '1', '1', '5918859096709639', '9247601701645181', '462380085082259', '5918859096709639', '5918859096709639', '9247601701645181', '462380085082259', '5918859096709639', '5761370117384759', '18652693655907869', '2331586525089537', '1440342256497769', '5761370117384759', '18652693655907869', '2331586525089537', '1440342256497769', '1479766345275523', '4623749279724477', '4623749279724477', '1479766345275523', '1479766345275523', '4623749279724477', '4623749279724477', '1479766345275523', '5756362159844513', '9328850170077743', '583053135629859', '5756362159844513', '5756362159844513', '9328850170077743', '583053135629859', '5756362159844513', '2869693483246867', '291791816579719', '4668668701477629', '2869693483246867', '2869693483246867', '291791816579719', '4668668701477629', '2869693483246867', '5740191858464917', '9336935320767541', '4668467660383771', '5740191858464917', '5740191858464917', '9336935320767541', '4668467660383771', '5740191858464917', '615230251687830901', '5834320054396387', '4806551097329301', '1442614380951219', '3081461324684767', '6652526167692137', '-955252769739873', '-7642022157918979', '-1910505539479747', '-3821011078959481', '-1181953890286879', '1999631955977327', '-7387211814293', '-29851649054371', '-3900282076852899', '-3900284570795503', '-7800566058371159', '-1560113447385127', '-760053819733963', '1032572127639281', '-6080429787066743', '-7800566647648397', '-764181795371219', '-764181795371219', '-3820908976856093', '-7641817953712181', '-236381928918061', '99977343363087', '-2954774111475763', '-955227244214023', '-1951423930368207', '-3902847860736409', '-1951423930368207', '-1951423930368203', '-6085915278264483', '4134491347896281', '-6085915278264487', '-3902847860736409', '-7823123599607007', '-61118167984611', '-7823125502030213', '-7823123599606993', '-6104537490750299', '82975127060913', '-6104537490750301', '-3911562275409301', '-977787088625669', '-244446772156417', '-1955574177251337', '-1564459341801067', '-3051826768899911', '8296158721096977', '-3051826768899913', '-7822296709005341', '8232642038994617', '129154227884773', '4116379302370523', '16145034172959', '1652754424983529', '4130621097835211', '2946161144975511', '7700628425141369', '4322267389499777', '4322267389499777', '12321006644379421', '2946161144975511', '2946161144975511', '7700628425141369', '4322267389499777', '4322267389499777', '12321006644379421', '2946161144975511', '1487060575444957', '6016830389267', '2159802526105931', '2159802526105931', '30806177413813107', '1487060575444957', '1487060575444957', '6016830389267', '2159802526105931', '2159802526105931', '30806177413813107', '1487060575444957', '590411964411759', '61689903227829549', '8622579215285263', '8622579215285263', '3855618951739347', '590411964411759', '590411964411759', '61689903227829549', '8622579215285263', '8622579215285263', '3855618951739347', '590411964411759', '1401667558586193', '30911380727892687', '8607955246407109', '8607955246407109', '7727845181973171', '1401667558586193', '1401667558586193', '30911380727892687', '8607955246407109', '8607955246407109', '7727845181973171', '1401667558586193', '1482775998396691', '30806099923706973', '1080039817318417', '1080039817318417', '1925381245231687', '1482775998396691', '1482775998396691', '30806099923706973', '1080039817318417', '1080039817318417', '1925381245231687', '1482775998396691', '1504162120427321', '6214985986082379', '4250278502343611', '4250278502343611', '3884366241301487', '1504162120427321', '1504162120427321', '6214985986082379', '4250278502343611', '4250278502343611', '3884366241301487', '1504162120427321', '12494773044464895389', '1093539419776099', '843716482379153', '3223264790712083', '343676177887739', '58641279122987', '-6201576798908431', '-775197042194339', '-3242163370308183', '2262002301785753', '-3242163485646607', '-174708318744846', '4040272115446393', '8080543884877509', '-5590666142165861', '-620157656823157', '-6187925593262533', '-1546981167666177', '-3239108076207607', '4644660017616653', '-809777076714263', '-5573606838879327', '1006656417803287', '2013312662619487', '-1114721344710921', '-6187925131963617', '-6198543898373559', '-3099271949186783', '-6498254772520339', '4795373986348471', '-324912738626017', '-5583354704600687', '806743746001459', '8067437460014601', '-5583354704600689', '-154963597459339', '-784124312797447', '-627299450237957', '-1640887307550751', '4648875645174881', '-6563549230203001', '-5668948721323379', '820140019145697', '4100700095728487', '-283447436066169', '-6272994502379571', '-48375714916901', '-154802287734083', '-6480462085578151', '4613929219437259', '-1620115521394537', '-278936305397291', '8061406461209977', '4030703230604993', '-2789363053972911', '-6192091509363321', '-1542648171787649', '-617059268715059', '-6567840236533871', '794495098766543', '-1313568047306771', '-5529304493420427', '7973312643265557', '797331264326557', '-5529304493420431', '-6170592687150589', '8235280195190347', '160729793870383', '86630620024467', '117925534501627', '839651633532201', '370931847167461', '8540275147324509', '5095836015461483', '2761396696932419', '2255267545715091', '68855699783981', '5290413399017391', '7874107194469427', '8444679386467597', '7430102036196127', '276344308727571', '8576353933164769', '7495448921062573', '308228460455421', '3335269879130213', '163613854129501867', '1215593151807177', '1215593151807177', '2556466470773467', '3335269879130213', '308228460455421', '308228460455421', '3335269879130213', '163613854129501867', '1215593151807177', '1215593151807177', '2556466470773467', '3335269879130213', '308228460455421', '4801286409643169', '3304944914119129', '5236301432377845173', '1272119936807779', '318030075151419', '5113575617556489', '3304944914119129', '2400651208375317', '4801286409643169', '3304944914119129', '5236301432377845173', '1272119936807779', '318030075151419', '5113575617556489', '3304944914119129', '2400651208375317', '1242540932273059', '417620550265977', '5235573288457921', '48178019021333', '48178019021333', '5112864539509689', '417620550265977', '1242540932273059', '1242540932273059', '417620550265977', '5235573288457921', '48178019021333', '48178019021333', '5112864539509689', '417620550265977', '1242540932273059', '400142537969737', '159542855989607', '1318522023677034851', '4845345538917987', '4845345538917987', '5150476654988417', '159542855989607', '400142537969737', '400142537969737', '159542855989607', '1318522023677034851', '4845345538917987', '4845345538917987', '5150476654988417', '159542855989607', '400142537969737', '233474012724911', '821273935269789', '327595058953442351', '2544141651293241', '2544141651293241', '1023734559229507', '821273935269789', '1167370609321401', '233474012724911', '821273935269789', '327595058953442351', '2544141651293241', '2544141651293241', '1023734559229507', '821273935269789', '1167370609321401', '4839473702211781', '3318722524833133', '5120204521841673', '1191611956478219', '2382239839687659', '262157842906018791', '663720712585193', '4769079175987951', '4839473702211781', '3318722524833133', '5120204521841673', '1191611956478219', '2382239839687659', '262157842906018791', '663720712585193', '4769079175987951', '2360676780193899', '251068984653001', '7739615607421239', '99977231584187087687', '546748232589399', '695274138566937', '-1390083307211717', '-2780166614423431', '-7975688561069201', '301919416527793', '4830710664444669', '-1993922140267301', '-8538993575468341', '7977664692035769', '941102697008991', '1994416173008949', '-4269496787734171', '-2780166614423431', '-1114848081295171', '-5574237386711597', '-977007519530131', '4483632185706597', '8967276450470187', '-7816052842417247', '-8679702614081957', '8296671613523353', '477302032034461', '1659336738516073', '-8679707621589071', '-5574238896593723', '-5556108569955247', '-5556108569955247', '-4003442521819177', '4901552947366241', '612694118420777', '-1000860630454797', '-4252997757554687', '49389365883461', '7504733189586611', '493893658834611', '-4252997757554683', '-2778054284977623', '-5710223703757821', '-571022370375781', '-7926256110039969', '8864129625128663', '8864129625128623', '-1585251222007997', '-4488382453585723', '4367660503607931', '492227850436071', '545957562950993', '-2244191226792861', '-1142044740751563', '-1118837586285453', '-349636722705023', '-1561553182719339', '4427154859898177', '2213577798095979', '-7807764809156001', '-4369440257464901', '4199657364544117', '1920979060022551', '524957262604741', '-4369440602602619', '-699273468419227', '-1393926286841447', '-5587852318151111', '-8015724939812627', '15355932903037', '1222361800551673', '-1609901078258707', '-8569179503253963', '7998596999630627', '1509521202346677', '3975144454313121', '-1709485343660417', '-5581778732758447', '4069285547661283', '187366995086557', '55501715830339', '1394278191168807', '7656121027288563', '8275310510761441', '4073429671455607', '234203089833753', '7161129387867669', '1621037872635669', '2320506579789303', '5575247664241679', '7002306272046137', '1914605246664717', '64498319537357', '5717294098939703', '3823305798809807', '4136535934701559', '2642601180288309', '21355096426644426301', '11573153232123', '2820815401493549', '5678860788310761', '723326306158233', '1303409828144577', '211418368075659', '2642601180288309', '21355096426644426301', '11573153232123', '2820815401493549', '5678860788310761', '723326306158233', '1303409828144577', '211418368075659', '2521360696969197', '21342709692364262899', '5871396214960523', '280662480199941', '1136323264166613', '5871460243387623', '5210609022965763', '5043203426131053', '2521360696969197', '21342709692364262899', '5871396214960523', '280662480199941', '1136323264166613', '5871460243387623', '5210609022965763', '5043203426131053', '5291328615898313', '2608974017506897', '5752440621769323', '955865057414391', '473316842984973', '5752398421214149', '21372755384552689067', '5291010292745057', '5291328615898313', '2608974017506897', '5752440621769323', '955865057414391', '473316842984973', '5752398421214149', '21372755384552689067', '5291010292745057', '4924768242144387', '10689427597688333253', '1165406708698109', '6921445969586717', '7503699378721369', '5827571964348659', '5219311372575597', '4928843142165829', '4924768242144387', '10689427597688333253', '1165406708698109', '6921445969586717', '7503699378721369', '5827571964348659', '5219311372575597', '4928843142165829', '1253251312159311', '2134365871339555977', '1470026189560773', '253819596786587', '2031350763161723', '1470026553358651', '5210854178075089', '1253254040643397', '1253251312159311', '2134365871339555977', '1470026189560773', '253819596786587', '2031350763161723', '1470026553358651', '5210854178075089', '1253254040643397', '5020504937310581', '42978137388043365529', '699611649771953', '674333336131857', '1078951891502739', '699611649771953', '1311588665406597', '5020504937310581', '5020504937310581', '42978137388043365529', '699611649771953', '674333336131857', '1078951891502739', '699611649771953', '1311588665406597', '5020504937310581', '488360663469283', '4564942230023747', '2075874289548061', '1590782714005499', '162540390492853', '24994280749089469791', '-6392112157748179', '-79835718477293', '-351170093983067', '1341267244816407', '4691774630377279', '671681841484861', '-1405896048840943', '-1541922524267659', '1047820856051837', '5260157069836881', '-7694172352117339', '-638948481796581', '-1279289308448029', '-6386766520749371', '-5571381197477149', '826313779262637', '929165488799269', '3314909742430921', '-5580331942007309', '-1976206526631093', '2988337494482501', '6015458143484783', '-7876420239577591', '-1597901632873689', '-1299380075632503', '-3252317702998411', '-5798253615390809', '7094079598809047', '599471202752361', '1769660707453507', '-5791080559823681', '-7724686063728979', '4956774925305189', '123144721392019', '-7747490665676843', '-6500767892079667', '-784758093671743', '-1553300180986997', '-1322408858359173', '595045883082357', '1127160289131919', '6079781622712813', '-2674691102402781', '-3980149524042783', '6350342479430449', '6610433312839367', '-485697523331489', '-3122816368330483', '-6598273164837767', '-6597961837788561', '-1473041626169043', '7328970114866489', '192836912836921', '1832397814621119', '-5892455410241177', '-4008512205624547', '1418291655127057', '709301491088113', '-8016105228680521', '-1319623500262633', '-85940176646653', '-6875200588873497', '-6313316696049097', '407649821672973', '5112537420217733', '8153023519176967', '-6313329392479179', '-999990896454333', '11246927256143', '562353134236503', '-7999886543058377', '-6875207360302867', '7224564743708547', '1440382847463499', '1762683709347629', '8970926037315853', '8276099412525999', '2068749219297219', '7233939617396257', '7180419867845637', '463806380855641', '57765186363933', '1437834266300371', '1884291377356991', '8987663749523289', '4391600285556339', '8280192188739471', '8924232611460937', '8110630574724913', '8232674000413251', '8923483666862137', '1935586373758389', '1535689207328797', '5884998400220013', '15856451723810444319', '8923388351820771', '8923483666862137', '1935586373758389', '1535689207328797', '5884998400220013', '15856451723810444319', '8923388351820771', '1106381975840553', '31872756885867244413', '281057633934929', '140293144163097', '1556286957317737', '1106381975840553', '1106381975840553', '31872756885867244413', '281057633934929', '140293144163097', '1556286957317737', '1106381975840553', '2231243577791391', '1935523211594177', '2553790943858119', '4908888229684693', '15855905987153723037', '2231225024100547', '2231243577791391', '1935523211594177', '2553790943858119', '4908888229684693', '15855905987153723037', '2231225024100547', '68595840319987', '3905502582431167', '6198349436826991', '5814328732544951', '3999332247022190701', '8779993257337009', '68595840319987', '3905502582431167', '6198349436826991', '5814328732544951', '3999332247022190701', '8779993257337009', '4416234885587147', '3191024964143039809', '854512747072077', '851511778349353', '3895294145682421', '2208117260894633', '4416234885587147', '3191024964143039809', '854512747072077', '851511778349353', '3895294145682421', '2208117260894633', '8835840057431257', '3894268306405189', '2358900209341119', '73058174923457', '31901860867231399201', '4417918937322067', '8835840057431257', '3894268306405189', '2358900209341119', '73058174923457', '31901860867231399201', '4417918937322067', '640518029750673', '24978411677339141001', '331447368132907', '46859421823489', '236638015371513', '8950837583947437', '-6095500303673191', '-1536957597302363', '-1273564905085207', '966174472195803', '1204444376892723', '-6341658016344531', '-5150156127621891', '-7353496530936813', '-105096008650531', '-6121665346441321', '-8871897306831603', '-8873946196420893', '-4779663608958847', '3670420334406231', '7340328446415139', '-955830277312303', '-7061407671874793', '-3618930380324227', '-3532752725526737', '-8872921751626249', '-778927619964699', '-3139929144373223', '-6523308568435477', '2476674112849711', '494122928561321', '-3249544377496581', '-5242952852850947', '-1542808965436083', '-2669907594408993', '-6255639624232019', '-2787856500664969', '-353361896080613', '-577088971587187', '2178729010875183', '1735160579458987', '-2865922794590449', '-4837396830113129', '-1398596732433381', '-998701955214647', '-350921979331867', '-8056954889665283', '-4030624798861909', '-2152223028586947', '824197469602017', '1318501215960301', '-4303372422058563', '-1630821091538697', '-1531523420877037', '-81648415993503', '-2014775560923637', '-7311947008782541', '-7322882446544561', '-7745504624579551', '2957392069296689', '2956024783212097', '-7740036444269797', '-3010436485428323', '-1285605647487337', '-755343148686973', '-7317414727663551', '779567565867783', '5635009219516241', '148167551906383', '4631367291992923', '41803083282129', '8704152805004779', '389524003964743', '703777854468313', '7585540995179941', '6301373416469627', '5702862180599969', '2398898586369617', '231791992625547', '218671064682969', '903710966490993', '4600238929337869', '2123350514409347', '4672022527707767', '3120251231982141567', '2466534457210481', '4568192411203743', '6094233411508269', '3120251231982141567', '2466534457210481', '4568192411203743', '6094233411508269', '124718557337934613', '7393675511456619', '834900331236337', '304488592616239', '124718557337934613', '7393675511456619', '834900331236337', '304488592616239', '49998963925896668981', '1060933231583987', '129382104185531', '6103386240727327', '49998963925896668981', '1060933231583987', '129382104185531', '6103386240727327', '6100480930140183', '199232343988467', '6051694905885147', '99950303401275113', '6100480930140183', '199232343988467', '6051694905885147', '99950303401275113', '760823837161621', '1145268205166277', '4083745107270919', '311633592713012833', '760823837161621', '1145268205166277', '4083745107270919', '311633592713012833', '609834547416981', '2737313726710583', '5109371098344489', '1248942047179669781', '609834547416981', '2737313726710583', '5109371098344489', '1248942047179669781', '7128875544032217', '1328796717862077', '99928684173270903747', '2740714148432247', '1081052388738623', '3878956649236671', '-5179686190238823', '-2668014370032293', '-524485609998789', '286187815041383', '-4156800153413199', '-1035937966751227', '-5336025096547277', '-657232183143963', '-601595321143357', '-5019522652083839', '-1972671090921163', '269046160997609', '-3893652343417527', '-1203190824596159', '-1003904384569211', '-983228522123069', '-2241711871651991', '-2254271536787513', '-6984348894274961', '4730077539390209', '-3485894796472481', '-4483424470915021', '-4508542345963987', '-1798393363375801', '-617872729693659', '-1571188164190129', '-307549314104034', '418700096309964', '-4894282503719157', '-6178726568978969', '-6284753384718139', '-6231739976848553', '-4618274441283149', '-4852075726960681', '-3812598406186147', '103983171681651', '-7508295439914719', '-461827298204553', '-606509648274787', '-4735175084121913', '-5697480242887129', '-5837956379456979', '-915362475226931', '6234646016381543', '-1135423198958229', '-5697478058250019', '-5837958564094087', '-5767718311172053', '4772782093607763', '1552475841356901', '927447827495957', '6276723736658631', '5862199679110521', '4500567333267039', '1', '1', '1', '1', '534142883565161', '7579518833007563', '534142883565161', '3233373012507197', '8083432531267993', '534142883565161', '8083432531267993', '8083432531267993', '7579518833007563', '534142883565161', '299618337645537', '1713158449170337', '30947494664182787', '7736873666045697', '1713158449170337', '299618337645537', '299618337645537', '1713158449170337', '30947494664182787', '7736873666045697', '1713158449170337', '299618337645537', '555235371141287', '8601740059006759', '61861201336119733', '773265016701497', '8601740059006759', '555235371141287', '555235371141287', '8601740059006759', '61861201336119733', '773265016701497', '8601740059006759', '555235371141287', '5894381407937131', '8641659352710983', '3851000713525351', '123232046115875909', '8641659352710983', '5894381407937131', '5894381407937131', '8641659352710983', '3851000713525351', '123232046115875909', '8641659352710983', '5894381407937131', '2925143385022307', '1080669572605357', '2464490137572361', '1925382738079457', '1729071461687731', '2925143385022307', '2925143385022307', '1080669572605357', '2464490137572361', '1925382738079457', '1729071461687731', '2925143385022307', '5849841263492237', '1063623145762223', '24863141936200791', '242804097733343', '13295287048291', '5849841263492237', '5849841263492237', '1063623145762223', '24863141936200791', '242804097733343', '13295287048291', '5849841263492237', '5792315062968927', '8648277472552247', '240683810637821', '123230134329630769', '8648277472552247', '5792315062968927', '5792315062968927', '8648277472552247', '240683810637821', '123230134329630769', '8648277472552247', '5792315062968927', '8102524715489489', '3174043062663469', '99975581537165478961', '126963165874653', '482961111889969', '12416206994237', '-6524147899902077', '1389405418313511', '-6524147899902093', '-6176796545323689', '-617679654532371', '-5547915755235169', '8007433237808481', '4003716618904241', '-1109583151047033', '-6176796545323703', '-6578354807579683', '93233369369259', '-6578354807579691', '-6287000528300739', '-6287000528300751', '-5684451890183553', '822540351621673', '8225403516216727', '-113689037803671', '-6287000528300747', '-6486056932927079', '4560065798571347', '-405378587137409', '-6201053512423537', '-3100526294940327', '-5589560015292881', '4039296867822717', '8078593043738257', '-2794779949987507', '-6201053051152097', '-3245650745045881', '447036717284553', '-6491300916992817', '-1552975713341257', '-6211903657113847', '-700378331160973', '1620020328248641', '1620020363198681', '-2801513353475121', '-6211903255239441', '-3295690907396403', '6083415213824607', '-1647845570510641', '-6211168716865867', '-1242233695648283', '-5578789835556383', '8051995084054557', '8051995774929947', '-5578789892712693', '-1552792149388411', '-6499697839783939', '4374663178779677', '-6499698301062583', '-778285260378521', '-778285145058861', '-1124107014660359', '4063964783184073', '4063964437225091', '-2810267478991067', '-778285202718691', '2579079278871651', '5232668320948963', '192484670761867', '1131860870077129', '5239668654884873', '819717308948209', '8223119063228077', '5546187922795787', '156786850232113', '663177892686043', '7383289395709231', '1040339048298117', '361465259319863', '7381849430464183', '8428320840528381', '1705319045035397', '737562881722889', '1664868882179597', '979987719658839', '7827992324628827', '6983693862337419', '790283864352799', '3450501250681290253', '790283864352799', '6983693862337419', '7827992324628827', '979987719658839', '979987719658839', '1956998081157209', '27934775449349667', '7902838643527981', '3450501250681290253', '7902838643527999', '5586955089869937', '3913996162314409', '979987719658839', '1031379423734201', '3607094769425523', '3492860045140353', '8253138869347939', '13870484481363486457', '8253138869347939', '3492860045140353', '3607094769425523', '8251038300256831', '1031379423734201', '1803547384712763', '13971440180561409', '8253138869347933', '13870484481363486457', '1650627773869589', '2794288036112283', '45088684617819', '8251038300256831', '987474757796107', '315907806599449', '3491612920931001', '1566805960969179', '3450526482474288169', '1958506905514623', '3491612920931001', '315907806599449', '987474757796107', '987474757796107', '15795391290538339', '55865805774330127', '15668058649125901', '3450526482474288169', '15668056204682873', '11173161539092381', '15795389369406561', '987474757796107', '8251989254045363', '8246433332482277', '1747484124043919', '1800796341931557', '1387069903468369193', '1800796341931557', '6989935768579871', '8246433332482277', '8251989254045363', '8251989254045363', '49478601050386549', '167758474852723337', '43219111150864481', '1387069903468369193', '8643822652370051', '167758459501409791', '1979143957576031', '8251989254045363', '3775011010245629', '7526351874606261', '1745869527979963', '8081646424837327', '3459542701173882627', '8081646424837327', '1745869527979963', '7526351874606261', '3775011010245629', '3775011010245629', '1806324449905501', '167603474686076489', '48489878549024003', '3459542701173882627', '48489878549023921', '167603474686076407', '45158111247637607', '3775011010245629', '1888020382019333', '8094012303953541', '3491360923834669', '3758521349805323', '1729769788077655483', '1879261220599473', '3491360923834669', '8094012303953541', '1888020382019333', '1888020382019333', '24282035422138571', '20948165915438527', '2255112958855399', '1729769788077655483', '2818891644683953', '20948165170577501', '971281536063307', '1888020382019333', '2586919179370191', '2773045445721919', '1140471944178289', '252304518641089', '723161771958017', '98471728102905191943', '-333848660511791', '-595315513893157', '-667697321023581', '-1330554580672903', '-4917406075121673', '-5322218322691601', '-2404558802638473', '491386637796641', '167714199221191', '1228466594491607', '-1202279401319237', '-1187378816416363', '-17362837283636', '-8075094369756533', '-5556107315652263', '-5282377988727167', '-6227386721169169', '-1056475474723181', '-2482995183327563', '1041062838524151', '268682700201871', '5205314807732017', '-2482995298660923', '-6060801021317681', '-2660199976135153', '-2445934149732787', '-2660199984183551', '-2668433632530783', '-952025791515343', '-1334216872762017', '-960328390509457', '2448726996938811', '8377230076476273', '4897454720127633', '-4801642013068121', '-2965941748875953', '-5281467038581399', '-6239633066926483', '-5281467315149029', '-138916232260529', '-4038177286300877', '-2778324504232839', '-4965596462202303', '2600420543721939', '215003086413053', '5200841071280431', '-4965596460855351', '-1515355327557757', '-1381006027016537', '-4707186395400047', '-552402410806613', '-5379197505657843', '-5865799214666539', '-5379197505657829', '-5059476044148343', '43557003216033', '4340428183549047', '5444625402004133', '-2529738022074173', '-382026400468197', '-1344079536912403', '-1471685219199207', '-5376318133672947', '-2763386604898741', '-4683101247713087', '-5526772978796027', '-632402915380843', '108890675387941', '8680199017507137', '5444533034462751', '-252961163090111', '-611216075026089', '6662117113045597', '1669731905414101', '5202833818886303', '2601108270410703', '8360988482652277', '8216849292600221', '349620441629633', '4118411235539939', '3738734203471651', '921265883027309', '1249372113542473', '1726357106331943', '6751853393774657', '261549703655859', '5200578905575829', '1301109151223209', '8373348205127771', '4117372442868579', '8801891341278447', '85728847315299', '575433334192339', '1869711611207029', '8250732340772997', '345394446396063', '1021561979786643', '1653720474935117', '201533913464871', '3997235126571359', '367642847501633', '8232674000413251', '1606278972940611', '23666325855579', '7992216547476003', '252021061464301', '17767021894787', '128635736891123', '2341930856704021', '3928849049844753', '8770648156709287', '5277974577362567', '19635517214168149', '1534024782356887', '5277974577362567', '8770648156709287', '3928849049844753', '2341930856704021', '60163880976295777', '2010518458297151443', '281712996238057173', '41171541395536339', '7855259140890619589', '7855259140890625637', '41171541395540547', '281712996238061381', '2010518458297151443', '300819404881474677', '1780482033841001', '785739417687553', '4608535303643343', '5230309920138747', '490959282923654359', '76712387956821', '5230309920138747', '4608535303643343', '7857393449279747', '4451204357006719', '285675746641439557', '2010694285455452333', '295744885855661249', '41043858134230579', '7856147153200957041', '7856147148005349797', '41043852938622679', '295744880660053349', '402138819838186377', '285675705270917313', '4746611440798751', '7882802418918661', '1078375331678927', '102954023127511', '306366219435727', '245093045397775637', '102954023127511', '4313500599119937', '7882802418918661', '2373306084197261', '38048637871956533', '40347828981041339', '69167513631873411', '202878192432361', '1960895296793074883', '980447927936669073', '10143909908329017', '34583751138526021', '40347828981041339', '15219457419746887', '4537899994017091', '7981564311003679', '2093842791950261', '4722572537153739', '488576313467634957', '1908501224482949', '4722572537153739', '2093842791950261', '7981564311003679', '4537899994017091', '1138508656416513', '510561961797852061', '33630561724395883', '4593495483962051', '977281703988461621', '122160212998557697', '287093467747627', '4203820215549483', '510561961797852061', '7286455401065687', '8754214308846067', '7881825569736357', '4675968193163939', '5139166914552809', '478759447533589', '12256243021013049', '5139164367967743', '4675968193163939', '7881825569736357', '8754214308846067', '280688712697255317', '2017193497466754127', '299815806319817453', '40559480502241299', '7844548630347647533', '7844549388262532533', '40559472986416587', '299815819176673269', '2017193497466754127', '280688699840399501', '8482312685786117', '1597041525839297', '8942892435824381', '4721448844532841', '195404902231649661', '1526600798684763', '2360725513660043', '8942892435824381', '7985205446409241', '8482312685786117', '272547164975966579', '408619997430235619', '287285730761813487', '36658431725451893', '1563461852416283947', '1563461849659359479', '36658435403127393', '287285716977191027', '2043099428357643631', '272547178760589039', '1248497981332897', '3121280560048461', '1302763708243669', '3978073285410559', '34553699369803163997', '3704538762491011', '-2473253744415127', '-6874625686661111', '-2473253744415131', '-540044896386203', '1084945266695487', '5424726333477331', '-337528060241371', '-662539852314179', '2612233021390159', '5224466042780417', '-662539852314193', '-5376171594246581', '-4948239157480169', '-275489159685559', '-4948239593729173', '-5327041642272241', '1055078439372293', '527539225942383', '-166470055421368', '-5379898066476987', '5381104656939949', '2690552361769737', '-2689948905688059', '-5378691187613363', '-4941827500029851', '-3538228963900723', '-988365504568213', '-135880056914639', '548629794306987', '5486298801765093', '-2717601113877489', '-528950929165981', '5194912929574749', '1298728012017079', '-2644754664542403', '-2692053065961527', '-2477826928346739', '-7942938801493247', '-2477826928346743', '-1102705931850129', '5574971786714499', '1393742946678623', '-1102705931850133', '-1347857251559829', '333173155043237', '5330770480691781', '-2695714503119657', '-1090417506357363', '-2471297167029723', '-1767945891313469', '-617824292314791', '-5312577737565723', '655071850167499', '524057509396137', '-2656289009601111', '-1082633426393589', '544175331949281', '2720876511206277', '-5413166848102027', '-5384580809117261', '-2477790988026231', '-7968627229834499', '-4955582528536129', '-5414451131553313', '2687640190168293', '1343820233344427', '-2707225489731397', '-1098962451304257', '1384000732762943', '5536003494855719', '-5494812410591447', '-2726810727079479', '7857561546277077', '5522006342441553', '412030298882951', '56434906204213', '10711730832497', '4033712614329453', '5556065988281799', '1382371820823793', '8225393622891861', '7841760385003903', '4095018072268039', '42940655709559', '1231600622302983', '2771649068266877', '7559525856008081', '5297417416739203', '796672494965723', '8105598716946279', '2688101871249349', '5000343223240619', '1768324974798989', '5529263890463957', '1245273944876117', '8105193067648643', '130813589220589', '5545558526594151', '1888768167805047', '2489146163020637', '2058170380191071', '3590190577803', '5235755077058781', '2519690273348157', '7107968474965393', '1359299734989953', '823268340085307', '2886868839757067', '7157511713925391', '3253786117644453', '6275118481649199', '3732190526564701', '56362999598562461757', '3732190526564701', '1568911315247717', '3253786117644453', '7157512441521167', '85890141039822617', '19522716233148793', '6275118481649199', '22393142686670281', '56362999598562461757', '22393143632106131', '1568911315247717', '19522717178584643', '85890148825536079', '7182604894370003', '3538632517617821', '6346728331834691', '357583485269269', '55996924632229267473', '3575836307884213', '1586709367799723', '3538632517617821', '1436521124393153', '172382514562491769', '8492718622760431', '6346728331834691', '42910021134700583', '55996924632229267473', '42910032792222253', '1586709367799723', '42463587309025549', '172382537829566663', '7154768925411529', '3385683341834823', '2993152501275127', '3664834565821233', '2249705198484101711', '3664834565821233', '47898434843063', '1692841307119511', '715477038060313', '42928614535745603', '253926226055701', '2993152501275127', '1099450271418727', '2249705198484101711', '343578271273129', '47898434843063', '2031409765198699', '42928621300342351', '1430789290055179', '3670499840769267', '401999828671423', '337737587898063', '11249447021302114699', '3377386792917199', '8112997712352093', '7340973488090769', '1430789290055179', '171694717970339557', '44045994925513127', '401999828671423', '40528507384049483', '11249447021302114699', '8105728935744893', '8112997712352093', '44045844092262691', '171694711642903403', '718265872694489', '1788077131248143', '198081215499987', '7076218743160343', '55997139134797665897', '7076218743160343', '6338708035363639', '1788077131248143', '143653189090813', '43095952602140113', '2145692509403617', '198081215499987', '21228655748539483', '55997139134797665897', '849146268416903', '6338708035363639', '10728463027959631', '43095956486773127', '7158892097238133', '1863300420918647', '1287700392354217', '6499513506248527', '5636894919833410399', '3249763665284263', '5174710730855041', '7453185676567221', '7158891369642343', '11454227305464057', '2981280723586791', '1287700392354217', '12999027263081833', '5636894919833410399', '12999054410552273', '5174710730855041', '14906371102549663', '57271131207723523', '2383899699262489', '84461720047657861873', '161451846567747', '1318065934763189', '473973757573277', '1791380081294243', '-98659165790183', '-7141672698523073', '-1262807988975603', '-261929827871329', '2103215206079931', '1321667024971743', '841300719443559', '-818537573789087', '-1508775233265881', '-4687177952759241', '-1206990964127469', '-1441364473008049', '-6161831785674791', '-2050398774230447', '-96278149442773', '-6552603695695049', '2120647130427067', '1317602167467989', '8482619577396733', '-6552615030013827', '-3070276730655367', '-4185909198291249', '-6140524063786349', '-7187016062121353', '-6269696151997269', '-961527481613439', '-626940466267879', '-6600731137212313', '8561111645636337', '2652701903415463', '8561403264893241', '-412552521357921', '-6108506821539267', '-4490865324728707', '-6108216293999317', '-1807769472237867', '-5982046213103863', '-8491348708609199', '-2983143461835009', '-630380677140917', '7898740995595289', '1285604448295841', '791452118624113', '-3154859572857421', '-384460553301107', '-1427353698125133', '-6135609338452047', '-7035585156963089', '-1535219901815991', '-8376058560313581', '-767606163308019', '-262153928559997', '1697053181831067', '5271131338001299', '8485295936343329', '-6553859473827663', '-6162639354176567', '-8201978384349721', '-6162609603542511', '-3593935888451627', '-1455676890933831', '-2126698779488439', '-5814633723265247', '-757201635565187', '7317949098111407', '313515577505312', '1465207250442703', '-6060642509766979', '-1220252112258531', '-1255674681036287', '-1523296538475249', '-3441010016622253', '6878925916493589', '8969844203029893', '335116875265807', '959110831074051', '1899144203630309', '1168213353110899', '5110309067511359', '2554175092285461', '1027881197184897', '6160159629922977', '1646719434515661', '4918829836913009', '7359926706147', '933983101342343', '296142155383339', '7333494567514037', '3888010226709731', '7204239778220327', '3729836005206399', '1505240569150651', '8761583395681723', '918125854684229', '2767042682997341', '7207800610726701', '12774169168881', '1277150524317637', '8223981108790947', '6167375452200021', '4116319139417759', '4926343181102579', '8982614211126727', '6845793951068817', '2315320824370511', '8790632429624521', '8453855337926417', '5837887419159501', '8826157114199129', '8520783718706877', '15957802494046032657', '1558375075272929', '8347581730923297', '176523782568251', '8826157114199129', '8520783718706877', '15957802494046032657', '1558375075272929', '8347581730923297', '176523782568251', '1766294377339793', '20479653032029', '6382302994594851377', '486930893738491', '1628794354223099', '88314747970819', '1766294377339793', '20479653032029', '6382302994594851377', '486930893738491', '1628794354223099', '88314747970819', '175404169384261', '2531167383728137', '99997358926177509', '7812066656216167', '2417778125736211', '877053224934669', '175404169384261', '2531167383728137', '99997358926177509', '7812066656216167', '2417778125736211', '877053224934669', '8924926426671153', '1294518969680927', '1585458747752212723', '7741417756228499', '6143361418326503', '8925047207562417', '8924926426671153', '1294518969680927', '1585458747752212723', '7741417756228499', '6143361418326503', '8925047207562417', '8850669745976057', '280895002837731', '31873547701047432527', '3890813928350517', '140211692182503', '8850669745976057', '8850669745976057', '280895002837731', '31873547701047432527', '3890813928350517', '140211692182503', '8850669745976057', '4457160810418267', '6915144525903299', '792922101878668017', '387157353525433', '257557818231243', '4457333614416467', '4457160810418267', '6915144525903299', '792922101878668017', '387157353525433', '257557818231243', '4457333614416467', '1023201623718607', '742001484533893', '348685333146377', '691250175771773', '99946834055704777469', '545107281260801', '-2692801117834331', '-3881536623779973', '-2717867918570351', '-6655721482837233', '5057326293270559', '2525527415488223', '-6643186237584611', '-1592129561723549', '-159839703445129', '-79763164904371', '-1550868518404411', '-5490339802899629', '-1243557629632421', '-8054312624407173', '1539403540937773', '769478046000453', '-1006341742269367', '-7579637610140747', '-1896698628272207', '-1516643212322957', '-72159146744788', '-4598494936811559', '-946007873690043', '-5337499014925697', '4018292858810609', '320340862084341', '-265475837175953', '-5220948761016413', '-527689973330313', '-1312231061789943', '-2557457576178247', '-7265262221465381', '-209694157250653', '-1582526439606459', '4800448419514539', '1196125089919269', '-6298239180513071', '-1513731937650689', '-1223731377677283', '-6086792319494587', '-3530913317981449', '-1809467028683387', '-1413185833047449', '-4779894883838439', '458824820454773', '7340684311117069', '-955876413535829', '-4436159648482439', '-4437185280801023', '-4436672464641733', '-1182118135244281', '-5778501579717799', '-612827809796651', '-1138890458323421', '4299119268843507', '8554596014853367', '-565089508822303', '-5494309447005163', '-2790705483637309', '-553786020713989', '4581925611358059', '476736192155219', '5825982288620807', '2317740872700721', '2271604918898679', '645462474590139', '6342417618167203', '5112118771965793', '230515162414701', '249640128550563', '6521389695542991', '5664397984391261', '1157673423298497', '4271150194102801', '36072480351263', '3136996849609377', '6568047246939937', '712720178811713', '308250231109277', '2408711478149943', '308250231109277', '56034352891109', '56034352891109', '6165010647806699', '56034352891109', '56034352891109', '15415761172954717', '6165010647806699', '8696560939111269', '8696560939111269', '302263156586123', '42875695307895637', '302263156586123', '302263156586123', '302263156586123', '302263156586123', '1339865478371739', '302263156586123', '1339865478371739', '1339865478371739', '8696560939111269', '302263156586123', '302263156586123', '8696560939111269', '4966265959853683', '4835594127407703', '6673655988376647', '2618033424291573229', '2556673265909739', '1668413633296283', '4835594127407703', '4966265959853683', '4966265959853683', '4835594127407703', '6673655988376647', '2618033424291573229', '2556673265909739', '1668413633296283', '4835594127407703', '4966265959853683', '1607154163409629', '1215254031224131', '6383619074569417', '5273326263917641647', '160929146237721', '6383620529760981', '1215254031224131', '1607154163409629', '1607154163409629', '1215254031224131', '6383619074569417', '5273326263917641647', '160929146237721', '6383620529760981', '1215254031224131', '1607154163409629', '4934779649854093', '487988618757797', '6663188839832193', '2618010344787918731', '159790670458247', '6663188839832193', '487988618757797', '4934782196439261', '4934779649854093', '487988618757797', '6663188839832193', '2618010344787918731', '159790670458247', '6663188839832193', '487988618757797', '4934782196439261', '1189637186313547', '2558234738225301', '6593082167758647', '5237599911626613021', '1022968732739573', '6593082167758647', '2558234738225301', '4758546562466871', '1189637186313547', '2558234738225301', '6593082167758647', '5237599911626613021', '1022968732739573', '6593082167758647', '2558234738225301', '4758546562466871', '2353512160648297', '4792300363087419', '1657502426191919', '655412437762349451', '128010205370671', '3315000850607171', '4792361117333187', '2352417129032971', '2353512160648297', '4792300363087419', '1657502426191919', '655412437762349451', '128010205370671', '3315000850607171', '4792361117333187', '2352417129032971', '4673730346529251', '1023281661946203', '1640645387882179', '2620795382573201213', '1279685245397071', '1640645387882179', '1023281661946203', '4673730346529251', '4673730346529251', '1023281661946203', '1640645387882179', '2620795382573201213', '1279685245397071', '1640645387882179', '1023281661946203', '4673730346529251', '3938294631750081', '3999086087298238509', '4732854294486439', '401953760956929', '1404107551063399', '2091643335716191', '-999010985796969', '608878250995691', '4871025235200871', '-3996043857325139', '-17364296239157', '-694571897864073', '-8515088683219317', '7920499754253789', '1878338081966211', '3960250435234707', '-425754439527387', '-2778287494860707', '-3957864081884751', '8834697736586481', '1766939870467191', '-494733021456077', '-285352690969337', '-2853526707724691', '-8974256238180033', '2183665263837401', '3937064255397457', '4367329944209719', '-2243564031493801', '-2853526808709031', '-3980559971493043', '120062504154643', '2401250489506883', '-318444748949761', '-2779934625136093', '-5559868843858179', '-4272520284150339', '1997407336089881', '235481180405271', '7989630970015629', '-4272520665163483', '-173745907720787', '-1948618698149881', '8857515845367441', '8857514400342439', '-3897237938184127', '-1395024093285507', '-2790048367199139', '-4354500778818067', '8355622331866723', '3830047539734767', '4177810443420887', '-8709000880280677', '-2790048276885077', '-4008916440914243', '4848451901999889', '969535460606669', '-4009453283235491', '-1118828928088449', '-2797265102670303', '-8609971344643731', '8090698148583389', '1515989665282743', '8089167155799279', '-8609280730757077', '-2797168711445713', '-3893480356221719', '8774660201543017', '8774660201543033', '-3893480356221727', '-5593295662057681', '-2796647831028849', '-4373922812078897', '8419641080673321', '3846287522960613', '8419641080673343', '-8747845624157787', '-5593295662057691', '515119268404317', '1403408070385547', '2355091677170697', '819973412139739', '5305501993167933', '4669685155052867', '411711354612739', '6987879328176619', '471024119061989', '1511118770673481', '8275830109451603', '1539851769781261', '146026364277307', '8272148244344871', '7692866205301459', '282012590718901', '4128623000764791', '7700107735162817', '5155850958411811', '8652518712560969', '593341846242473', '98024058605184637', '63064951406659', '63064951406659', '3829064789265027', '593341846242473', '8652518712560969', '5155848411826611', '40676197192058417', '277451209277187079', '304361635751382247', '7842495283041178311', '2017507842462266289', '2017507842462266289', '7842495298890011367', '304361619902553527', '277451193428358359', '40676192668205537', '237315291147381', '4194100289628703', '454752920593973', '61067461985231793', '1995083236911527', '1995083236911527', '1908358005139551', '4547530297333381', '4194100289628703', '237315291147381', '36901519878974411', '269491345240843541', '292110795884749269', '7817704051081691149', '2041896312759089551', '2041896312759089551', '1563540663151241489', '292110856001358029', '269491335508258637', '7380305922311863', '5285807473125423', '8796174216067601', '146332704834211', '490826819425334547', '7858042011561749', '7858042011561749', '1917292263380213', '146332704834211', '4398086016640087', '5285807473125423', '41262248488002499', '282501786211164117', '300713590797465013', '1570850666004011873', '402126907940790799', '402126907940790799', '7854253322102353333', '300713598715170741', '282501724279672181', '41262240570296771', '657731650166701', '1158724156406589', '8881842076265631', '61351233683455889', '785956050584477', '785956050584477', '3834452105215993', '8881842076265631', '4634895170434737', '657731650166701', '5148053756810653', '37192972449533803', '35641167749585479', '981733539630778291', '251392136117029129', '251392136117029129', '981733538379817163', '35641169000546591', '37192962058961963', '5148052505849541', '4746055622582059', '4495276620585623', '8482628773603369', '488528194458247801', '7981326883345171', '7981326883345171', '7633251583218449', '8482628773603369', '4495276620585623', '4746055622582059', '36883032919809099', '57756623155665449', '272529532816155181', '7817536517847375913', '2042134272848234541', '2042134272848234541', '7817535033276539149', '54505905454143781', '288783110232890969', '295064307721963', '5165946558427879', '37581228253899', '4376748858871691', '980416086832379129', '3940021120818551', '1576008739365741', '7659500678377961', '8753495534955979', '4697654259333177', '1291486093910119', '40762528891145191', '301214869607469841', '56135394108813213', '7843893727642453029', '2016725775579249193', '2016726148108299561', '1568778747647061981', '11227075604080453', '12048596223229733', '40762522021703811', '5212367205616067', '635042026934451', '2805410066517033', '2437718434583259', '92329770017898081777', '2321178686747417', '-5430447359192037', '5478951303539351', '2739475656779961', '-5430447530821163', '-4939986463150041', '-1767828488667423', '-1234996613860869', '-1322298660134813', '2598222766498027', '1299111382670511', '-5289194472763421', '-672742947682951', '-5509910162745941', '5571031739130149', '5571031636651773', '-344369365786663', '-4953661688133567', '-3961014590998579', '-4953661319000069', '-5386928477466603', '5325068886411707', '1065013871604717', '-673366121524087', '-1362197081860409', '-5395787785054971', '338583465401349', '67716695383609', '-2697894085761833', '-494484224058279', '-6870373836547993', '-4944841865895783', '-1060092205563437', '5226681343275707', '209067261347823', '-5300460828692003', '-537424041802353', '-2658665960540137', '5259143561553263', '5259143797585169', '-265866620591197', '-4945087597799969', '-3443466363641159', '-989017423591969', '-336397666658603', '1077840861402507', '1347301137705187', '-1076472483142811', '-1075104130667009', '-675582453108747', '5360249143510549', '1340062140774949', '-5404659395014263', '-2476784306579199', '-1982006210187831', '-4953568614905919', '-2746525984478587', '5537033710985569', '2768517144824399', '-2746526098969569', '-544907016657911', '-663017463240921', '1045524171635937', '1045524185667981', '-5304139808186529', '-988056138864797', '-3523019430793549', '-617535211210741', '-169121780145406', '2721567432729777', '340195932961067', '-1352974138154871', '-2690329310427073', '5266246761370713', '4963201312943207', '3394339786191719', '4099899836472513', '3780852000426809', '5571832647849269', '5413616815303219', '1354036069407977', '6057076239035171', '819973412139739', '55390863908969', '1009411907935013', '280676872612341', '7873522919058881', '1497502675194331', '8162048398141469', '2062412743964303', '5550408947118789', '49050554521347', '5595052542480447', '4096472847688439', '3393966561747893', '1648532233696353', '555851646686097', '5427983043499187', '5355254193932151', '8232674000413251', '5958993830269637', '3551977828244587', '2527505711709777', '4968083854775333', '1052343042380853', '514540009932179', '66769818422601', '1892233535408189', '5576937461861559', '8391255988332773', '4682293533364591', '998224131175437', '19546040103326129', '4482489545550987', '2241245500371281', '954396489420221', '998224131175437', '4682293533364591', '8391255988332773', '269621419236927237', '36357120656638227', '408652359536325793', '7819517259651505121', '287980558525541669', '287980642368577489', '7819517268940728933', '408652359536325793', '36357129945863207', '269621409947702257', '2166359317326091', '5088003435385499', '1577049882324491', '19612672296016461', '2341291896421051', '2341291896421051', '7661200115631431', '1577049882324491', '5088003435385499', '2166359317326091', '277828138876300981', '40169881224522659', '2018089703116773269', '7845603064665173489', '300219509000455861', '300219509000483617', '7845603064665146677', '2018089703116773269', '40169881224494903', '277828138876328737', '1124465650300569', '937431872992031', '1995633760675327', '977432228584947567', '4189430402954811', '1047357782637637', '7636189285819903', '1596507154059409', '2343578772985407', '1124465650300569', '144456782435478817', '3644691652566927', '510619655193378361', '3910254085300805687', '134586952091559937', '26917393427726591', '3910254093536796321', '510619701759505529', '18223459222867837', '144456774199488251', '2352618045786657', '5102811467551121', '492647938301807', '490365865022402687', '1079753256783381', '1079753256783381', '7661966640975043', '1970591571308309', '5102811467551121', '2352618045786657', '301683191563013893', '40274410038087171', '2017337873581868757', '62771215376486293', '276964915438867333', '276964915438889169', '7846401922060765829', '2017337687317375701', '8054882007613067', '301683191563035729', '8814252797339063', '521686049913717', '7856469684335727', '245567962932703433', '1154380319283579', '1154380319283579', '3836999420823491', '3928235569763673', '521686049913717', '8814252797339063', '283001209966757979', '40789763541189397', '402062223747605967', '7859119934298434447', '296466482188504187', '59293296437704163', '7859119934298417531', '2010311491267084299', '40789763541172769', '283001209966774607', '4643717269052713', '5227531005455709', '7859653749004797', '491011601989937339', '219260688806279', '219260688806279', '959007035136597', '1571930604281809', '5227531005455709', '4643717269052713', '298219389571774207', '40798763691245097', '2011049875392814403', '7857207116191424107', '56335033204887539', '281675166024463803', '7857207116191403199', '2011049689128301891', '40798763691218989', '59643877914360063', '4976773807326803', '5600743235175996893', '4195040574260911', '767135261372723', '3389600089064087', '96943547275313', '-110262341131411', '5565752745578171', '5565752997612893', '-5513116790913251', '-270630022220619', '5364720304611149', '1341180137348401', '-1353150239619567', '-2480284340499193', '-7998600692775659', '-2480284092090603', '-341280060993017', '-5431369565366241', '683822582824081', '683822582824163', '-1086273913073391', '-5313096880547959', '5234035292955319', '2617017646477321', '-5313096880547241', '-2473507883259943', '-7122283225925339', '-618376970814987', '-2696079234070111', '-5406682360971951', '2677238020541261', '2677238180922137', '-5406682344984471', '-1102933691470581', '5570448401043777', '5570448708948587', '-1102933720427261', '-496054450102331', '-7973504631951311', '-2480272214484417', '-2729444252246513', '-5304676798548513', '5218916388532583', '2609458100341761', '-2652338399274537', '-2717559493858811', '2739900383435103', '2739900289510041', '-2717559493858531', '-4946833744339507', '-1419531686639463', '-4946833556489917', '-269521865124477', '-107906624694551', '84476347419343', '2703243305925373', '-168604101085249', '-2662201907328827', '1052926278939543', '2632315885854857', '-332775238416077', '-1238263233394199', '-6897966784450761', '-1238263327647303', '-5384176046111171', '-664475513278919', '2624045497606897', '1312022701694363', '-332237756639501', '-1080578701125079', '5422269794001177', '677783700695449', '-2701446752812367', '-990274319449109', '-864291616881377', '-198054856352343', '-5383517311467747', '522826785372423', '445260064977041', '1006737565732557', '230695001613581', '8199726629823853', '271076382019961', '5239471271119193', '7565314774628111', '13846261613701', '201763335869', '2001893242983', '4968769091173743', '5374302563313601', '355130056440829', '5003054406174249', '4052618760709291', '6119551942845481', '5464973635735083', '614879393801603', '947006389670309', '1384752602217059', '8105207885546039', '27253735946269', '2647255420861609', '5565905299988609', '8245027503000971', '5502779742867149', '1768761818985059', '1019890871632833', '61712001169259', '7868567713625193', '8250687124194869', '275442675161743', '2016859534126267', '4814007026133953', '5632415226987763', '620837119880583', '5415282686427791', '8594165254527597697', '2808732775487647', '2808732047891851', '5245462191484129', '1353842681379771', '620837119880583', '620837119880583', '5415282686427791', '8594165254527597697', '2808732775487647', '2808732047891851', '5245462191484129', '1353842681379771', '620837119880583', '4984431774285187', '3787854278180361', '21343432020471493369', '5888228358284701', '5888228358284701', '1302699708280731', '1894098669800127', '623053517038273', '4984431774285187', '3787854278180361', '21343432020471493369', '5888228358284701', '5888228358284701', '1302699708280731', '1894098669800127', '623053517038273', '1011197110639737', '705059496546967', '5216060625528439', '5824261935829039', '2912229193344177', '21365207839586636931', '5889139468319941', '5054431772420659', '1011197110639737', '705059496546967', '5216060625528439', '5824261935829039', '2912229193344177', '21365207839586636931', '5889139468319941', '5054431772420659', '333916512566113', '857156455728423', '2607399172509887', '89956930051551', '1439322522356627', '5339967288873911767', '4314476107970767', '5342297492805767', '333916512566113', '857156455728423', '2607399172509887', '89956930051551', '1439322522356627', '5339967288873911767', '4314476107970767', '5342297492805767', '2514173274926859', '2099925700628561', '426869028131594507', '734550798217347', '367275444583411', '325675222878719', '525164733824231', '2514168545554153', '2514173274926859', '2099925700628561', '426869028131594507', '734550798217347', '367275444583411', '325675222878719', '525164733824231', '2514168545554153', '5307523193392457', '622948631873313', '5210810075614719', '2894953947193757', '5790225126132199', '5335949611169047567', '801499918214251', '132626843557949', '5307523193392457', '622948631873313', '5210810075614719', '2894953947193757', '5790225126132199', '5335949611169047567', '801499918214251', '132626843557949', '4999813790352011091', '403521042628809', '34625575743557', '91988085915687', '1603305021732987', '74849858490399', '-1608564297958361', '935967187317957', '4679885314535197', '-2010695718807703', '-6298551535132853', '406150639504251', '10198367323389', '4061519530079839', '-6298563602183167', '-3436421599181939', '-3436415426938751', '-1718209256530173', '-8060307612270859', '709048971680491', '567268109416547', '-403004753824709', '-2977462062535551', '3734898223508859', '1213486804463311', '7469941107378513', '-5954991099346877', '-6642103401021551', '-6642031070841149', '-6642067235931351', '-7904471540989193', '1163709413500079', '5956557814512767', '-156063460914061', '-554530252928213', '3267391379781253', '1161739235859167', '1650879603187287', '-1115439359755939', '-1599788399895647', '-6364714766606383', '-1595483545773621', '-7530757873617689', '2784025586639161', '5589550284469983', '-939391670196111', '-2610519076040627', '5809212888171393', '8951224093323981', '1454974330656503', '-1045194832993493', '-6130924947037653', '-3062782961855929', '-1225649087074951', '-7995138784299141', '567066732909021', '5671778346169853', '-1998579439611317', '-2931155993694773', '145244392529603', '2402586099042491', '7262774379304343', '-586256999312391', '-411066339943729', '-328839184241487', '-6576922561964703', '-3736721259244983', '3084448670685199', '1565761977541929', '-7406164684836607', '-4827821499071399', '987861233514077', '8505609213858159', '4985955430915639', '-4849203730089701', '-1179506691688497', '-5874087431999567', '-5885810445221027', '8078020056040317', '1782726091549149', '1024962785281881', '3920011369488607', '8983316247750497', '4200807910829227', '3725030366954743', '7280967606002463', '2537897157193303', '1861306383831059', '3638279505289087', '549772431267531', '2456666446894523', '8979264172611437', '8413743037937149', '7513603293449981', '907555299023929', '5761642844969', '2735520007547583', '2735520007547583', '8609234625454843', '20893993839862503', '8609234625454843', '8609234625454843', '8609234625454843', '8609234625454843', '652937307495703', '8609234625454843', '652937307495703', '652937307495703', '2735520007547583', '8609234625454843', '8609234625454843', '2735520007547583', '5848754241211471', '6221636629334481', '6221636629334481', '2741252157148901', '3104566325370599', '2741252157148901', '6221636629334481', '6221636629334481', '3104566325370599', '189761849605813561', '3104566325370599', '6221636629334481', '6221636629334481', '2741252157148901', '3104566325370599', '2741252157148901', '6221636629334481', '6221636629334481', '5848754241211471', '740124930510253', '5196909268313699', '3115439050384307', '322413243947609', '5282452861506985689', '49848072544003', '2597934585107399', '1868728683747347', '740124930510253', '5196909268313699', '3115439050384307', '322413243947609', '5282452861506985689', '49848072544003', '2597934585107399', '1868728683747347', '3697681645171683', '607771480591377', '1261708643537307', '10566439682319965457', '5159393284739057', '1261708643537307', '2431084649072811', '7395546644491689', '3697681645171683', '607771480591377', '1261708643537307', '10566439682319965457', '5159393284739057', '1261708643537307', '2431084649072811', '7395546644491689', '844241269506693', '1296142143233189', '1565836418141049', '2639153893999236667', '2577211413119849', '626264281507951', '5190092115991753', '4026363120563793', '844241269506693', '1296142143233189', '1565836418141049', '2639153893999236667', '2577211413119849', '626264281507951', '5190092115991753', '4026363120563793', '7810583774591821', '4956541219626063', '317486610134781', '5274719667139630323', '2575546712470523', '317486610134781', '4956541219626063', '7810583774591821', '7810583774591821', '4956541219626063', '317486610134781', '5274719667139630323', '2575546712470523', '317486610134781', '4956541219626063', '7810583774591821', '804717317992787', '4939109472500071', '6066351091611701', '132805917197043217', '5187731140509501', '6066351091611701', '4939109472500071', '804717317992787', '804717317992787', '4939109472500071', '6066351091611701', '132805917197043217', '5187731140509501', '6066351091611701', '4939109472500071', '804717317992787', '3921227538769723', '492111729991911', '635695188977411', '10549826356128253447', '40244393753541', '635695188977411', '492111729991911', '7842450711964803', '3921227538769723', '492111729991911', '635695188977411', '10549826356128253447', '40244393753541', '635695188977411', '492111729991911', '7842450711964803', '1727818119216741', '1305196187949799', '160438094526863', '4270614116384257', '49979495988520163523', '7325387994632793', '-579430340593047', '4675826883878091', '4051745836729721', '2329857633578169', '-4628103134244981', '-5751778892578523', '-5759879045425589', '-7651142767014009', '7643514903471623', '190273897314991', '-7674132870007919', '-1151165793800411', '-4573251154903711', '566395430898871', '7997858880186563', '4531123410451441', '-2286616433015231', '-720494425112313', '-2881988032411327', '-3953663495749327', '1071699848502323', '1714704259351441', '-3953692247708223', '-5763965732860577', '-281867144441703', '4407166641071761', '3987191898524179', '8968646152607293', '-2272455223430107', '-2854424757847241', '-2835016102271623', '-7705163635924369', '3882946831748537', '3960995000263463', '-3797844500159677', '-2844720430059431', '-226719422122159', '2235274439478083', '3971393796476707', '8941097757912301', '-4534388442443179', '-5724217297863149', '-2862108648931569', '-7848677367188103', '8497840277299807', '1062230034662481', '-7848677367188093', '-5724217297863141', '-4738309959938059', '4813183554482751', '1034382892922077', '2406591777241371', '-236915497996903', '-5864993141892879', '-5864993141892869', '-3895243953641193', '962747382694747', '3850989530779057', '-3895243953641189', '-5864993141892871', '-4523560831351947', '8890454027912483', '3963385469197347', '2222613945375411', '-282722577646838', '-2861122987926569', '-5722245537455833', '-787304908149467', '2150802448045599', '4301605772885793', '-7873047766302791', '-5722245756654483', '5621885834777977', '3756359200993607', '975282970514327', '5744148574039439', '7518876234329973', '7807624996350579', '191276558955673', '7496775541692319', '1955519331304257', '8251241043599223', '2318511691285083', '8670740399437487', '1029084485062549', '4600479350201449', '7141974328663373', '330195504985471', '4636045925367723', '2270011688084769', '605406394375859', '777932474774019', '1806285864339253', '3525151829460549', '3481809266218098689', '3525151829460549', '1806285864339253', '777932474774019', '605406394375859', '605406394375859', '1555864949548037', '7225143457357017', '28201214635684397', '3481809266218098689', '28201214635684387', '7225143457357007', '1555864949548039', '605406394375859', '24216945360049', '7222268087000269', '777978692194801', '3525451269409111', '696362080890420599', '7050901811222411', '777978692194801', '7222268087000269', '24216945360049', '24216945360049', '1805567064397631', '1555957350271551', '28203609984682633', '696362080890420599', '28203607415479899', '1555957418507653', '3611133958205007', '24216945360049', '1307999111892593', '1982959536298977', '27733888189523', '7053718881183557', '13958392189490710513', '7053718881183557', '6933474230167901', '1982959536298977', '3269995960742189', '1307999111892593', '23795512891068241', '20800417686661733', '84644628118722167', '13958392189490710513', '84644625029683201', '1040021057299211', '23795515980107207', '3269995960742189', '3672864081827931', '59353924996503', '7548223883080231', '1411627733095031', '3470129862335788261', '7058139393070911', '7548223883080231', '59353924996503', '3672864081827931', '3672864081827931', '45583813364826641', '45289344330969049', '169395329003891383', '3470129862335788261', '169395344401214201', '45289342265993723', '45583815429801967', '3672864081827931', '6709828287254269', '277331412373157', '7934441045807991', '1410500083354591', '6979340891735212291', '1410500083354591', '7934442500999479', '277331412373157', '6709828287254269', '6709828287254269', '4159971005622791', '23803324037296793', '2115750147528707', '6979340891735212291', '1057875051267533', '23803326603125617', '4159971365571919', '6709828287254269', '1468036258808109', '3769802182764269', '1900734665529519', '7058806206812987', '6940305533278943519', '7058806206812987', '1900734665529519', '3769802182764269', '1468036258808109', '1468036258808109', '11309406548292793', '1425550999147141', '2647052327554871', '6940305533278943519', '10588209310219477', '114044079931771', '11309406548292821', '1468036258808109', '6679595855702767', '2815793303555527', '1737205389097011', '5928555533092003', '1290328913821493', '2981124515235303061', '-1051635539689461', '5766090546247823', '8986408092933107', '1153218109249563', '-5258177698447303', '-1127097306106441', '-5157804992608593', '-5635486530532211', '-5485131607877941', '-39754027336517', '-109702632157559', '-1570053038652071', '-2629054137368569', '576541906692393', '1797343819935817', '5765419058068581', '-1314527068499797', '-1096927057708911', '-63652760679283', '-1096927113013601', '-88061268504826', '-322186945363031', '-5635920904833649', '-1256058987059463', '-1294016525607823', '173483205479583', '8916493451127419', '5551464320003', '-207042661230041', '-1417416368303919', '-2266573036545311', '-2834832371818207', '-5395975921762319', '-3361331687844741', '-1348993742360017', '-6236308367562377', '-1268927699563511', '269524673791273', '4382277339859117', '5390493475054969', '-2537855399094917', '-2737037989467311', '-1077950716087751', '-5474075701414287', '-5460890177974153', '-1373809442246763', '-5460890455237689', '-768474379716163', '-321930621732334', '54960202078649', '8888080560089927', '5496020716164907', '-5150889990075677', '-5384577785521161', '-1335809589043689', '-5384577935136651', '-88421724165231', '-112093699399501', '-5658990366392637', '-6219458853481213', '-1269099258103731', '1078280752849313', '8765775966742229', '5391403764246557', '-126909925810373', '-5459293098031431', '-5512922405115703', '-5459293098031441', '-171134749917407', '-5376771210510993', '-2738155998678519', '-6148408398670901', '1285251262167687', '15089213778761', '45926989858743', '7850405175416669', '486345351251229', '7812015222757173', '668573246872411', '4722981980901237', '4870282768044703', '97274918351399', '784617373317579', '3906006717362179', '1090828068569443', '912710559690521', '329429112117033', '11576235203351', '28476635212707', '6471410381284189', '1538877897409907', '8237019977567373', '8235227536985749', '516610718854261', '516085816051057', '5898567879752443', '4434981089021393', '8249615183864679', '1534577471098239', '2280916748310983', '7220498437681131', '6466695025003929', '1539666695809469', '8232634518678059', '8239587973930223', '1290827862861499', '1032429152483479', '738686545030231', '4499870193030783', '3167639807094767', '2495940134758121', '1318723336054059129', '5151005098763709', '996872622781211', '6336194202061771', '4789127707014141', '4499870193030783', '3167639807094767', '2495940134758121', '1318723336054059129', '5151005098763709', '996872622781211', '6336194202061771', '4789127707014141', '4410121272019049', '390580318378227', '1321692803267557', '2576406050866621', '10553243796529566137', '5290957435299509', '6248774321825169', '339957304663677', '4410121272019049', '390580318378227', '1321692803267557', '2576406050866621', '10553243796529566137', '5290957435299509', '6248774321825169', '339957304663677', '6433648887817029', '6086021107045687', '2434782899964247', '1062389278307919103', '2593723823993943', '152173976722499', '6086020379449949', '6433648887817029', '6433648887817029', '6086021107045687', '2434782899964247', '1062389278307919103', '2593723823993943', '152173976722499', '6086020379449949', '6433648887817029', '409936261347181', '6375770564016499', '76016448197053', '10548240640752490839', '5150508125367427', '76016448197053', '6375770564016499', '8198720861369381', '409936261347181', '6375770564016499', '76016448197053', '10548240640752490839', '5150508125367427', '76016448197053', '6375770564016499', '8198720861369381', '4884026661097473', '197010266861059', '1053522594235243', '321628914993969', '5269738761560168931', '210906863232227', '6303712993527779', '1172661654917911', '4884026661097473', '197010266861059', '1053522594235243', '321628914993969', '5269738761560168931', '210906863232227', '6303712993527779', '1172661654917911', '8192976161254141', '1591364300555349', '4900028810611493', '10548647951745411547', '515070700768819', '4900028810611493', '79568233217661', '1024121565409429', '8192976161254141', '1591364300555349', '4900028810611493', '10548647951745411547', '515070700768819', '4900028810611493', '79568233217661', '1024121565409429', '22902696776097', '129955334089993', '4999798568613704109', '876344080304353', '3537135052179', '388237024489001', '-4394901221328461', '1662238303247713', '388328176119877', '4053574458038833', '-1087173590058179', '-7696413797840421', '8712936255625973', '8506520281195159', '-3920267700828189', '-5590393346488033', '-564169171880349', '-35100265829036', '-4483994884255487', '4354202393150903', '7949580548358583', '8829366867774859', '-4511502071223379', '-1522250262599393', '236995737509983', '7706152656583443', '-752519364938873', '-5672174276641233', '-282088331075937', '-1131394089815997', '-4725016288814751', '4783687924461853', '8250269614754743', '239184422108879', '-2362508151759697', '-3922099027905483', '7912362794505637', '3956182644655843', '-7844198738331029', '-5866107698444597', '-234644282989723', '-1173221477318767', '-8937425847303311', '864704972238781', '7856440252823603', '4323525699887867', '-4468713316789451', '-987919714489901', '4416939859415027', '4416940698109027', '-7903356457878269', '-2847443578595613', '-1423721684461059', '-177965217109929', '-8755815810561393', '8272521615470583', '3905642400464601', '4202360192568879', '-1101967027580469', '-3809760236489603', '7860382563121759', '199854212555179', '-376301537680037', '-1401895291041583', '-557433229608139', '-5590956730123859', '-8952851800283403', '434067663376683', '7870074220214777', '2170338959149341', '-8952852568497807', '-7876589144064879', '8726542738616313', '8726542517061117', '-1575317582984367', '-5694952844839281', '-2847476450114043', '-5694952872533681', '223698776447077', '8271864943527981', '7574052133773151', '714906481036421', '40931177855437', '3773702505583761', '8105191215413293', '7510223280330803', '228382715052403', '8275951038710427', '912253244042601', '2303866017122463', '205216958867859', '8239337725250813', '7564324938664203', '4131820965758793', '4657415317048419', '2303442224224781', '6393577512835173', '1558583358189547', '1558583358189547', '2403229778039819', '6279309367695297', '2403229778039819', '1558583358189547', '1558583358189547', '6279309367695297', '1893116536998768389', '6279309367695297', '1558583358189547', '1558583358189547', '2403229778039819', '6279309367695297', '2403229778039819', '1558583358189547', '1558583358189547', '3196784169265407', '4982867317180531', '8571870903599337', '8571870903599337', '4982867317180531', '4205076851786951', '657043258091711', '8571866839331931', '5256343161685541', '8571866839331931', '8571870903599337', '8571870903599337', '8571866839331931', '5256343161685541', '8571866839331931', '4982867317180531', '4982867317180531', '527166052341767', '8717003285173351', '15433476188655767', '3858369047163941', '8717003285173351', '527166052341767', '527166052341767', '8717003285173351', '15433476188655767', '3858369047163941', '8717003285173351', '527166052341767', '8757511190902191', '4286561210986721', '249076279832207873', '486477109047281', '8573123149569187', '8757511190902191', '8757511190902191', '4286561210986721', '249076279832207873', '486477109047281', '8573123149569187', '8757511190902191', '8736664804524483', '544217103960299', '246937100625432989', '1543356733389809', '544217103960299', '8736664804524483', '8736664804524483', '544217103960299', '246937100625432989', '1543356733389809', '544217103960299', '8736664804524483', '8742211929325163', '8677830620594573', '247408604105824181', '773151887830701', '8677830620594573', '8742211929325163', '8742211929325163', '8677830620594573', '247408604105824181', '773151887830701', '8677830620594573', '8742211929325163', '8231039958705827', '4325706754845581', '248086875507120861', '3876357066000877', '4325706754845581', '8231039958705827', '8231039958705827', '4325706754845581', '248086875507120861', '3876357066000877', '4325706754845581', '8231039958705827', '8745579402597357', '8661862228674889', '247662414639903091', '1934862614374243', '8661862228674889', '8745579402597357', '8745579402597357', '8661862228674889', '247662414639903091', '1934862614374243', '8661862228674889', '8745579402597357', '667114137410647', '203562649718943', '3199205462307369', '99927765285862653439', '3311095502469567', '1442730865581701', '-1543551725028541', '1125715845236359', '72045814095127', '-1234841380022833', '-6685374076674901', '-6685374076674907', '-105959262751831', '76814991553819', '-6781392816117183', '-1337074815334981', '-48676632492147', '8854134031968979', '2213533551159051', '-48676632492147', '-6630048182135523', '-6630047836801093', '-6847352878744639', '695376134219371', '-6847353224079089', '-6630048009468307', '-6112482738823429', '4453890319190343', '4453890146544907', '-122249653625499', '-1658592534730039', '-6634369678532333', '-3375932455189137', '7519687471935037', '-675186514057219', '-6634369908726243', '-763271225651473', '1779143690373083', '1779143690373083', '-3053084902605891', '-6633242317116283', '-6633242317116297', '-1354550266547769', '1785715399968699', '-6772751332738849', '-103644411204942', '-620433734846371', '1131064627437777', '1131064584476449', '-6204337291181939', '-6720315813104567', '-1680078838712601', '-3427172807061877', '8577929261427853', '-1370869168650169', '-1680078895994371', '-6102688476217609', '8889088859541681', '2222272214885421', '-6102688476217609', '-414536011611691', '-1658144046446767', '-1695993459255661', '4844724839538571', '-1695993459255661', '-6632576185787061', '7065149552865469', '688209457933469', '3645759407895377', '3731325016163979', '1395590749329629', '7323559138273721', '8422022041677', '6946076300687421', '726032654822373', '8232658959730883', '2618379371821427', '261868439091', '2598789290388757', '2503921011669633', '2320723540326101', '8579275139920209', '5223264678296927', '6350013137525671', '2232346431165779', '7777804606206731', '8579409972322131', '8579409972322131', '248889770681680329', '4464669943064529', '2232346431165779', '7777804606206731', '8579409972322131', '8579409972322131', '248889770681680329', '4464669943064529', '2216312951783603', '3856810241480243', '8709782051308189', '1088722665464051', '61708969684450121', '2216312951783603', '2216312951783603', '3856810241480243', '8709782051308189', '1088722665464051', '61708969684450121', '2216312951783603', '8369579481913093', '49560225644641983', '2166236020607793', '2166236020607793', '3871891900891867', '8369579481913093', '8369579481913093', '49560225644641983', '2166236020607793', '2166236020607793', '3871891900891867', '8369579481913093', '8740027110829881', '247587699050760439', '1083338319188469', '8666705825911941', '967139358467557', '8740027110829881', '8740027110829881', '247587699050760439', '1083338319188469', '8666705825911941', '967139358467557', '8740027110829881', '560611904836089', '15427603121834289', '4353076712873833', '4353076712873833', '1542760312183429', '560611904836089', '560611904836089', '15427603121834289', '4353076712873833', '4353076712873833', '1542760312183429', '560611904836089', '8737312413558191', '966265067511381', '8680778224106971', '8680778224106971', '247363927132105201', '8737312413558191', '8737312413558191', '966265067511381', '8680778224106971', '8680778224106971', '247363927132105201', '8737312413558191', '583303468567317', '516001866724691', '29350166454181', '1378591871685701', '16190193782271', '99884126154015643777', '-6052530031944319', '176081712628807', '8804092240453931', '-3026268177736481', '-341130289206609', '7060423492307637', '-6822580490017599', '-412622558811499', '-6601948864896567', '-6601954902940277', '-3043975637875757', '4434641684453309', '4434641772190543', '-6087951333323597', '-6735030907341203', '1558941662418989', '-1347006067427359', '-6613237904616137', '-6613238705108897', '-3306619152431259', '-6178242125024871', '180167410423263', '4504184916494609', '-193070062822788', '-6827238613036401', '1676939851160391', '-341361953590951', '-3348114187669213', '-669622745777319', '-209257122392369', '-3052314810500301', '4446247675488707', '8892494835791071', '-6104629563603341', '-6781105757918533', '4722480623068227', '-1695276411475507', '-6633528241225671', '-6633528123653129', '-33167640912197', '-3033651861857941', '8836484173162553', '8836484173162561', '-1213460744743177', '-3362746752981799', '4135902637605643', '-3362746752981793', '-6596246548538409', '-6596246548538421', '-1319249309707683', '-3053823659573031', '8898257201610859', '2224564558924057', '-3053823745746813', '-1354237789089223', '68557208389351', '-6771188256055863', '-207314796705689', '-207314839792579', '-103657409124567', '4534641293102337', '7528551353095921', '7134187791035587', '164929265034321', '931038503968479', '1412672742558307', '2253543416345587', '8497618006963059', '1025502085842929', '1049514492150173', '112395075926771', '2618875983625041', '4773431478240017', '7448029617469887', '3534775544610941', '8235274553224457', '612327824943', '1046804142564973', '171748879764821', '3917773617426743', '3917773617426743', '171748879764821', '10798215646231261', '5399107823115631', '3917773617426743', '5399107823115631', '3917773617426743', '3917773617426743', '3917773617426743', '3917773617426743', '5399107823115631', '3917773617426743', '171748879764821', '171748879764821', '1256006009399079', '36023219574321469', '1827906731318151', '1827906731318151', '1256006009399079', '1827906731318151', '1827906731318151', '1256006009399079', '562862805848773', '1256006009399079', '2354744027416133', '2463071760171231', '2463071760171231', '2354744391214019', '2354744027416133', '2463071760171231', '2463071760171231', '2354744391214019', '470823870273307', '1231613987329183', '4926455949316733', '470823870273307', '470823870273307', '1231613987329183', '4926455949316733', '470823870273307', '2364619872452997', '4921205688773501', '2460602844386751', '2364619872452997', '2364619872452997', '4921205688773501', '2460602844386751', '2364619872452997', '4887506001698427', '610204879203241', '4881639033625929', '977501345858829', '4887506001698427', '610204879203241', '4881639033625929', '977501345858829', '4730403745722529', '492091450667043', '9841829740936611', '4730403745722529', '4730403745722529', '492091450667043', '9841829740936611', '4730403745722529', '4886642418397743', '9763710404598993', '38139490925794', '4886642418397743', '4886642418397743', '9763710404598993', '38139490925794', '4886642418397743', '11847621980205090599', '1700563632377759', '7362049212388413', '3934330143141383', '255534867488749', '1698617022794707', '-3663217068920189', '5072571110318747', '-7326433509195709', '-4507724797753923', '-4507726055043257', '-4507724797753923', '-2253863027521631', '-35216604893739', '-3664021730688657', '1268443307950717', '-732804346137731', '-4508540459148883', '-450854045914889', '-1127135114787221', '-4508540459148891', '-4508540459148887', '-730104098430227', '5053602195681291', '-3650520492151133', '-8989755154483899', '-1123719394310489', '-1797951030896781', '-1797951030896783', '-2247438788620977', '-7100674444806041', '490360070019107', '-3550336919529291', '-274634142358439', '-8788294978459877', '-1757658511094009', '-549268436153743', '-4394146883482483', '-291981890347899', '631560758863973', '-3649773629348737', '-561765353349707', '-1123530481086277', '-8988243848690217', '-8988245653595317', '-4494122375571383', '-7101753302237503', '153262817328933', '-3550876651118751', '-8789371681173533', '-8789373500519629', '-70314988004157', '-4394685840586771', '-8789372590846583', '6966602310400729', '877862865579327', '6969066178888447', '7147162777554709', '174483319385389', '7135570216854709', '4335267534219343', '1685784288345723', '34258033809679369', '4282254226209923', '4335267534219343', '4282254226209923', '4282254226209923', '4335267534219343', '1685784288345723', '4335267534219343', '762939614918487', '762939614918487', '762939291331513', '762939291331513', '1', '1', '1', '1', '1', '1', '1525878252921789', '1525878252921789', '1220703647662569', '6103518238312843', '5421363239142191', '7568500929363257', '5421363239142191', '6467156466297589', '505246598929499', '5421363239142191', '505246598929499', '505246598929499', '7568500929363257', '5421363239142191', '4283814816191843', '4283814816191843', '207457890007729', '21268529895874753', '207457890007729', '207457890007729', '207457890007729', '207457890007729', '5317132473968687', '207457890007729', '5317132473968687', '5317132473968687', '4283814816191843', '207457890007729', '207457890007729', '4283814816191843', '2896686861054411', '6212170766139527', '6212170766139527', '5389203878686787', '6325723796504553', '5389203878686787', '6212170766139527', '6212170766139527', '6325723796504553', '47296306399972109', '6325723796504553', '6212170766139527', '6212170766139527', '5389203878686787', '6325723796504553', '5389203878686787', '6212170766139527', '6212170766139527', '2896686861054411', '4912641364550929', '8854614868054033', '8854614868054033', '4912641364550929', '10486124220924379', '524306211046219', '8854614868054033', '524306211046219', '8854614868054033', '8854614868054033', '8854614868054033', '8854614868054033', '524306211046219', '8854614868054033', '4912641364550929', '4912641364550929', '6335154618928827', '36022066970167281', '3654162857523019', '3654162857523019', '6335154618928827', '3654162857523019', '3654162857523019', '6335154618928827', '9005516742541821', '6335154618928827', '1', '1', '1', '1', '1', '1', '1', '1', '1', '5918576789554209', '1849548571044579', '577983928451431', '5918576789554209', '5918576789554209', '1849548571044579', '577983928451431', '5918576789554209', '1152257489882577', '18652774868688169', '4663193717172043', '2880643906605387', '1152257489882577', '18652774868688169', '4663193717172043', '2880643906605387', '5919141403865069', '1849492109613493', '4623730274033733', '5919141403865069', '5919141403865069', '1849492109613493', '4623730274033733', '5919141403865069', '5756261068420213', '18657801795377691', '2332225224422211', '719532542603051', '5756261068420213', '18657801795377691', '2332225224422211', '719532542603051', '2869598350102479', '3734973087199433', '1167179089749823', '5739197427800709', '2869598350102479', '3734973087199433', '1167179089749823', '5739197427800709', '2870005932022063', '4668512477090023', '2334256420443957', '2870005932022063', '2870005932022063', '4668512477090023', '2334256420443957', '2870005932022063', '5768187816166251', '51369223310969', '3005501433135327877', '940305436545187', '2367303526563359', '6731435530768853', '-7642301632705391', '-1910575408176349', '-7642301632705403', '-61138413061643', '-738759027459751', '3999496811501663', '-5910072219678011', '-1910575408176347', '-7800651048301847', '-1950162505136823', '-7800651048301847', '-7800650020547287', '-1216103954294299', '8260714532669347', '-760065003551267', '-975081316803071', '-238804459369181', '-238804459369181', '-3820871349906901', '-7641742699813777', '-5909466694279681', '7998062038652471', '-1477366673569921', '-1910435674953447', '-7805798670159849', '-1951450181866027', '-390289933507993', '-1951450181866023', '-6086026707508727', '4134576525642703', '-3043013096591333', '-312231987952479', '-7823320920644897', '-3911659428612027', '-7823320920644893', '-7823318857224051', '-6104745802651403', '414891608834539', '-6104746318506613', '-7823319888934471', '-1955620693592853', '-1955620217971111', '-1955620217971113', '-1955620693592849', '-6103851203908799', '8296461496253639', '-3051925601954401', '-1955620455781981', '4117561110771107', '206927923974337', '1646521263509893', '8281456997668721', '8311192424446159', '828748297144941', '2946075091124973', '12323187441200719', '8641818812108479', '8641818812108479', '154039843015009', '2946075091124973', '2946075091124973', '12323187441200719', '8641818812108479', '8641818812108479', '154039843015009', '2946075091124973', '300449560411201', '123234658115297899', '345366729583961', '345366729583961', '3851083066103061', '3004496331707739', '300449560411201', '123234658115297899', '345366729583961', '345366729583961', '3851083066103061', '3004496331707739', '1180392153724097', '123384391150806123', '344885613220589', '8622141785706179', '1542304889385077', '1180392153724097', '1180392153724097', '123384391150806123', '344885613220589', '8622141785706179', '1542304889385077', '1180392153724097', '1120735660298609', '123635501866673067', '8609395236620423', '8609394509024679', '7727218866667067', '1120735660298609', '1120735660298609', '123635501866673067', '8609395236620423', '8609394509024679', '7727218866667067', '1120735660298609', '1188136762311563', '123242314521305833', '215937002070413', '1079685101301537', '3851322328790807', '1188136762311563', '1188136762311563', '123242314521305833', '215937002070413', '1079685101301537', '3851322328790807', '1188136762311563', '6052227777905649', '124298971885315799', '8498427142014279', '4249213934804999', '7768685015236521', '6052227777905649', '6052227777905649', '124298971885315799', '8498427142014279', '4249213934804999', '7768685015236521', '6052227777905649', '99959001410885274237', '1269547054218657', '4331308595445309', '72886848795657', '302735617401771', '649421258109499', '-6201599908313977', '-6201599908313973', '-324317068377593', '728938135649', '-810792670943983', '-5590234177544393', '8079668400931797', '4039834200465901', '-5590234177544393', '-620159990831397', '-6173246914500469', '-1543311580816899', '-6471135556709683', '2383106772742291', '-129422687484879', '-1388844058116263', '501508081456019', '8024129598912743', '-2777688264040743', '-617324661888403', '-3099534246682703', '-387441737913943', '-3249480950500739', '599788188756777', '-1299792517548759', '-1116780148976719', '4034133621542241', '8068267586455651', '-1395975186220899', '-774883518749281', '-1254754746124483', '-6273774074501937', '-1640764687515941', '231427740449459', '-3281529203092113', '-5670115803125809', '4101672370470207', '8203344569000659', '-5670115803125813', '-1568443475640543', '-6189746045260239', '-3094872851249427', '-324041322635627', '4657292003419031', '-6480826795473931', '-5575469029237671', '8056065035845219', '8056065207225923', '-5575469029237671', '-1237949174775909', '-1540521021415673', '-6162084194605053', '-656316260775157', '160431410989811', '-6563162727465231', '-5518861925227453', '1989170394741883', '994585262017053', '-5518861982391461', '-385130258758367', '329530913537473', '36610943522799', '1109178511185831', '5404604532174199', '2123839751638271', '931791475962101', '4255536580941641', '5597063493315551', '5536694998109083', '8983792920530383', '2464429550088779', '5316961077254341', '391930842391633', '4227205355084591', '1861461075406693', '967696627357451', '108082024050829', '3772679014074403', '615968139501241', '6661520511974027', '327255590422621291', '4884630705917931', '4884630705917931', '2556684300176729', '6661520511974027', '615968139501241', '615968139501241', '6661520511974027', '327255590422621291', '4884630705917931', '4884630705917931', '2556684300176729', '6661520511974027', '615968139501241', '4780524660782431', '165035218537431', '5236675641485578293', '1277840159436289', '5111359182553619', '5113941056138259', '165035218537431', '239026014760391', '4780524660782431', '165035218537431', '5236675641485578293', '1277840159436289', '5111359182553619', '5113941056138259', '165035218537431', '239026014760391', '991020942694959', '6672898658879319', '2618011483404598343', '1210085167297409', '1210085167297409', '2556651839262303', '6672898658879319', '4955102166889607', '991020942694959', '6672898658879319', '2618011483404598343', '1210085167297409', '1210085167297409', '2556651839262303', '6672898658879319', '4955102166889607', '63994372401661', '6372837068021911', '210980762680883087', '2433720937437719', '2433720937437719', '1287724381597187', '6372837068021911', '63994372401661', '63994372401661', '6372837068021911', '210980762680883087', '2433720937437719', '2433720937437719', '1287724381597187', '6372837068021911', '63994372401661', '4660600568812333', '131202999961211', '2621019487299232371', '5112538283780379', '5112538283780379', '2559589343065657', '131202999961211', '4660600568812333', '4660600568812333', '131202999961211', '2621019487299232371', '5112538283780379', '5112538283780379', '2559589343065657', '131202999961211', '4660600568812333', '4669384415156873', '1326925136502117', '1048611638083867671', '956499872782077', '1195626841865963', '1024034802816277', '1658656784425531', '1167414588741049', '4669384415156873', '1326925136502117', '1048611638083867671', '956499872782077', '1195626841865963', '1024034802816277', '1658656784425531', '1167414588741049', '4731899578305789', '2017166196846943', '3895897108213839', '9997709792841686687', '2145495904393', '1483816108853931', '-1112144716841569', '-5560723584207837', '-1989454186764183', '2397093162848907', '4794186325697797', '-3978908373528377', '-8549764611932551', '8000014850265571', '3769399107262233', '8000014850265587', '-8549764611932543', '-2780361792103919', '-5576830832477509', '-1394207986010667', '-1949935707099987', '889165117776921', '6946599008991', '-7799744425439651', '-1087150631965521', '1666874554348493', '478149501019901', '8334368325481823', '-8697203623160407', '-5576831388260087', '-1389471431593397', '-5557886131120767', '-1997350643434283', '4863034908968711', '972606819894857', '-7989403787978719', '-8521332077927909', '1586994773243267', '1879322734961669', '7934972247227571', '-4260665659513459', '-222315437149887', '-356901105277303', '-356901105277303', '-7908219689472567', '274725250629467', '4395604010071429', '-395410984473629', '-4493423893864651', '875595349616537', '3942650571469531', '1094494187020673', '-1797369557545859', '-2855208842218423', '-5595322393500777', '-5595322393500767', '-7789152345925191', '4387659904848847', '1096914976212213', '-3894576172962599', '-350091799660571', '8426947904420159', '240493551166178', '4213473952210091', '-350091799660571', '-5595322393500769', '-1399808082547239', '-1399795858261447', '-1605852634098717', '2429963271079073', '2430012168222241', '-4014564016021237', '-8620565864845349', '8115652700615187', '303365451402979', '1623169122356983', '-8620652879946523', '-1399801970404343', '164278735765981', '4702095573320829', '3509859158230087', '5936640464469859', '768423995374621', '4137435146681303', '4110865666299821', '2351129849802367', '7013969627959217', '8180973291429517', '4659757851451731', '170028788841479', '2076596718180351', '768431928572761', '8255739657736011', '889957558005659', '1920776861200027', '4134028196239231', '16216423488077', '2638627208041606839', '3135677021359611', '192963802296891', '15971613452999', '313591567275177', '5153449803808037', '5193028099929589', '16216423488077', '2638627208041606839', '3135677021359611', '192963802296891', '15971613452999', '313591567275177', '5153449803808037', '5193028099929589', '4966603890239023', '10550209768264403089', '6344050433480181', '392533025581961', '7850683067109179', '6344050433480181', '321966850838147', '4966603890239023', '4966603890239023', '10550209768264403089', '6344050433480181', '392533025581961', '7850683067109179', '6344050433480181', '321966850838147', '4966603890239023', '519535589255509', '206313685698233', '3118757348385651', '7446223826665577', '58025347571781', '6237480499766839', '10563278589144883223', '2597544432440967', '519535589255509', '206313685698233', '3118757348385651', '7446223826665577', '58025347571781', '6237480499766839', '10563278589144883223', '2597544432440967', '1213101365207461', '257961174425823', '1262327673394019', '7615246912144349', '1890663163559237', '315577188976209', '10566137388195981903', '4851662949379411', '1213101365207461', '257961174425823', '1262327673394019', '7615246912144349', '1890663163559237', '315577188976209', '10566137388195981903', '4851662949379411', '2459562243632199', '10550301822767164943', '6355473765976187', '7872921956233969', '15376816327709', '6355473765976187', '5151514561898029', '2459562243632199', '2459562243632199', '10550301822767164943', '6355473765976187', '7872921956233969', '15376816327709', '6355473765976187', '5151514561898029', '2459562243632199', '4940677190099297', '10625339375100176617', '757813599520517', '806972064796831', '6455779428757919', '757813599520517', '1297038316038079', '4940677190099297', '4940677190099297', '10625339375100176617', '757813599520517', '806972064796831', '6455779428757919', '757813599520517', '1297038316038079', '4940677190099297', '100654056504391', '8876313664453759', '1719751066168369', '1260420930512191', '3548572016584001', '49979435905401627423', '-5733361535538981', '-285273154605277', '-285176452364248', '4529336922897831', '100315858092537', '4584810190101211', '-4588054021034403', '-385341261617631', '1933925214632313', '7847858198380283', '-1525577831903857', '-5719412313822259', '-5721093717804027', '-5721091455122577', '-4531540243984823', '4463067514078601', '1985500713810919', '1785228815776589', '-4531542365248683', '-7838197119378369', '8468400030208699', '8468409080934329', '-7838190331333991', '-5721092586463301', '-2877240352298061', '-5756567838556793', '-4632270838642949', '2335651588000587', '4050274453439661', '583394004820077', '-4630379554225387', '-3831708411425583', '7647614254231441', '7639224390512061', '-958667672685953', '-1151104854315291', '-89766680012559', '-575075414077461', '-4548497633678567', '4475711198730529', '7965747520311521', '279025117715871', '-2271679906208953', '-394957007835923', '1729693889888027', '8625612791564041', '-7915202065110969', '-718488853848649', '-1429994226923219', '-2859988053537749', '-4519497109693711', '2217669268011887', '7922674095907109', '8870680274517063', '-282468616267032', '-1968110345962427', '8609853100923011', '860985630339241', '-7872438981997517', '-2859988253692093', '-2931731900740627', '-366466482659857', '-2368051867163787', '600858148733949', '51713039177933', '4806866203686457', '-4736104129306319', '-3893410866209089', '120209809786489', '1923357891475119', '-3893410319839863', '-2931731881009741', '1499041608081881', '1951729783569687', '6430916952475279', '4625814221952443', '4511245245645627', '8249909059355289', '1501169639525617', '1557464126842627', '4835106964963151', '7496546172734979', '974126369035741', '44906342085871', '925448824818841', '1677021667393141', '412746831655589', '1147294711421673', '294079537895357', '8232674000413251', '135898430808629', '964555360560247', '567353590971791', '567353590971791', '385822144224099', '8697500299348041', '135898430808629', '964555360560247', '567353590971791', '567353590971791', '385822144224099', '8697500299348041', '433423065638599', '247480311788706439', '1779727539389829', '1779727539389829', '7733759015801283', '433423065638599', '433423065638599', '247480311788706439', '1779727539389829', '1779727539389829', '7733759015801283', '433423065638599', '4352480905169429', '246901050336043191', '8889133038602697', '8889133038602697', '1543131419081113', '2176240634483661', '4352480905169429', '246901050336043191', '8889133038602697', '8889133038602697', '1543131419081113', '2176240634483661', '214128770536661', '249045751819594061', '1134204075337753', '4536818847936319', '3891339872181157', '1713030600850769', '214128770536661', '249045751819594061', '1134204075337753', '4536818847936319', '3891339872181157', '1713030600850769', '8650237406246719', '3871404872859769', '4451283816260829', '4451283816260829', '123884967573045531', '865023667865091', '8650237406246719', '3871404872859769', '4451283816260829', '4451283816260829', '123884967573045531', '865023667865091', '4334275104818211', '123869589559409293', '4189017526998659', '4189017526998659', '7741849347463081', '4334275104818211', '4334275104818211', '123869589559409293', '4189017526998659', '4189017526998659', '7741849347463081', '4334275104818211', '863793717057687', '99945151021411133037', '182633742904919', '233800854085993', '6129015260025707', '1877463055905979', '-131577748587281', '-6578887088967961', '-1209075255692513', '8801308842243107', '8801309012441151', '-1511344069615641', '-3359679968734733', '561891394006033', '-3359680138932781', '-3289443629583003', '-6607507663150387', '-1651877030631927', '-60742432186409', '4422366188008969', '2211183180137739', '-1518560819015767', '-6761169759749319', '4917176066110277', '-6761169530060667', '-6607507892839047', '-3304655004056203', '-330465506423161', '-6082510786190787', '8860366338097821', '8860366853491287', '-760313855468459', '-1683934530869159', '2022849762035181', '-3367869116676693', '-6609310068287811', '-6578677960414771', '-6578675553048409', '-1204525687929013', '1094489823035267', '4377959893982661', '-6022629124844457', '-6816392580953429', '3803439556909557', '-1704097719280861', '-657867675673159', '-6606746604392847', '-6606746487808391', '-3035131522982273', '4418576509576427', '4418576251947307', '-6070262988609497', '-3386992409614987', '5351622934767847', '-6773984706394253', '-6606746546100619', '-6694789676333339', '-3347394838166659', '-6177156408900523', '9006917979634377', '4503458989817193', '-6177156408900523', '-6823569972132567', '8241938931151849', '-6823569972132591', '-6694789676333329', '1745000220034677', '726074822260247', '466177815333579', '646716574376157', '959810888297', '8232674000413251', '871314707864457', '364846647690529', '1097208266441449', '441711426821541', '7377931363145257', '7156785612780929', '5178192962789789', '689142718207493', '8477864500463229', '5136438720086883', '4153985672136343', '4484636805226843', '9845246383875717', '4723569732248567', '4723569732248567', '2461311595968929', '9845246383875717', '4723569732248567', '4723569732248567', '2461311595968929', '195360291399993', '2439016316202469', '2439016316202469', '2442003824398853', '195360291399993', '2439016316202469', '2439016316202469', '2442003824398853', '9855112288401289', '4703837923197423', '4703837923197423', '1231889036050161', '9855112288401289', '4703837923197423', '4703837923197423', '1231889036050161', '4927486515287511', '2352057855627103', '2352057855627103', '2463743439542693', '4927486515287511', '2352057855627103', '2352057855627103', '2463743439542693', '9768173622357143', '4877715255285717', '4877715255285717', '488408681117857', '9768173622357143', '4877715255285717', '4877715255285717', '488408681117857', '196936717863183', '4720389986085929', '4720389986085929', '9846836620754921', '196936717863183', '4720389986085929', '4720389986085929', '9846836620754921', '328620102565761', '6306091798685973', '6519644079166213', '38011493346482541', '7506910101967507', '3009781011153899', '-4498559381791257', '-4498559381791261', '-7308321493476669', '5059041802581039', '-7308321493476667', '-224927969089563', '-1799423752716503', '-1799423752716503', '-4400068484863723', '-4400067575591551', '-55566534352977', '2456241191033619', '-55566534352977', '-4400067575591547', '-4400068484863727', '-1100017007556909', '-4511412547918821', '-180456501916753', '-45835715641759', '5078008228722029', '-3666857251340721', '-1127853136979707', '-4511412547918821', '-4511412547918823', '-563903951554801', '-4511230711189493', '-7333356402399083', '5077740821492111', '-7333356402399087', '-2255615355594749', '-4511231612438401', '-4511231161813949', '-4400266627608733', '-8800533255217487', '-1778228362132663', '1228195033681571', '-1778228362132663', '-8800533255217481', '-8800533255217469', '-4400266627608737', '-2250313426320539', '-9001251901294747', '-3656203789126083', '5062094377430053', '-3656203789126083', '-9001251901294747', '-4500626852641077', '-70322287525691', '6860559049603813', '7031644121338691', '1368442427745529', '6835477920176729', '7007461864114159', '6870843901680129', '1', '1', '1', '1', '5394284409326637', '189504606671073', '5394284409326637', '6463837072796503', '1009974542624453', '5394284409326637', '1009974542624453', '1009974542624453', '189504606671073', '5394284409326637', '5899266182589519', '1078409268135019', '1927260248780741', '246689358410064747', '8627275600271711', '2949633818890539', '5899266182589519', '1078409268135019', '1927260248780741', '246689358410064747', '8627275600271711', '2949633818890539', '5628076599709543', '1076365981247533', '3862845227114201', '1931422431658159', '538182945149031', '5628078054901081', '5628076599709543', '1076365981247533', '3862845227114201', '1931422431658159', '538182945149031', '5628078054901081', '2947255986475203', '540282103410609', '61604567388483429', '7700570923560427', '540282103410609', '2947255986475203', '2947255986475203', '540282103410609', '61604567388483429', '7700570923560427', '540282103410609', '2947255986475203', '1200965569849833', '8642442900917587', '123170542868034707', '7698158929252171', '8642442900917587', '1200965569849833', '1200965569849833', '8642442900917587', '123170542868034707', '7698158929252171', '8642442900917587', '1200965569849833', '6025432286788079', '8511411658361663', '124208502231244961', '970378741782659', '8511411658361663', '6025432286788079', '6025432286788079', '8511411658361663', '124208502231244961', '970378741782659', '8511411658361663', '6025432286788079', '1495582624753943', '8644105328436057', '30792123030750917', '481126922355483', '8644105328436057', '1495582624753943', '1495582624753943', '8644105328436057', '30792123030750917', '481126922355483', '8644105328436057', '1495582624753943', '1331268359658553', '2345225104786651', '99952748246541858017', '995641431258559', '114582602273447', '851969003929289', '-405980773949993', '591861905221439', '-811961369157127', '-1549940283603989', '-619976029681949', '-2792796058624541', '2017826107313819', '4035651847963951', '-558559230315137', '-3099880357808863', '-327867890300987', '4637460196854517', '-1311471331482623', '-6267516227017711', '-6267515711808317', '-2831487780684441', '8192193094924437', '2048048481837639', '-5662975934475123', '-783439496176627', '-6483949071664399', '4526531690266877', '-648394907166441', '-1550260210255677', '-6201040841022731', '-111800570804139', '1615907331980507', '8079536659902539', '-5590028540206947', '-19378252628196', '-6465823670067541', '4664337858216743', '-202056989689611', '-3087151276964491', '-6174302553929009', '-173688368141743', '4014452142053523', '8028904284107057', '-2779013890267887', '-6174302553928997', '-3279149167711541', '3117988071252157', '-6558297877777897', '-6168549140048757', '-246742002213567', '-5528770222888481', '7973265418107373', '996658263071897', '-5528770337299777', '-6168549597693967', '-6468587132684629', '4621821223277139', '-1617146783171159', '-6179723306229793', '-6179723306229817', '-5564726366590627', '2009897770016587', '8039591080066349', '-173897698955957', '-3089861653114903', '8442597282981939', '86666713773681', '5555440815471379', '4479284222085299', '1065431320704097', '1207063812375163', '2055353971288883', '1386243724096611', '9448697064059', '3124237523724371', '7477076605957547', '213294791140219', '1255319040732543', '756788120366221', '8691699731645269', '6632328386579469', '7494201698017009', '4282835242627789', '237722659314801', '7839669349450767', '349272615967133', '7890744386790197', '862555818485640331', '7890744386790197', '349272615967133', '7839669349450767', '237722659314801', '237722659314801', '11759504024176147', '83825427832111927', '11836116580185299', '862555818485640331', '2959029145046323', '83825427832111913', '5879752012088077', '237722659314801', '1023727414281689', '7225903363612619', '1746469927295867', '4120826254251033', '866899077244141399', '4120826254251033', '1746469927295867', '7225903363612619', '1023727414281689', '1023727414281689', '43355420181675731', '33532222604080643', '49449915051012379', '866899077244141399', '49449915051012413', '167661113020403249', '43355420181675697', '1023727414281689', '1214908672640773', '3954610760394819', '1746090378119669', '782185837585903', '3450354707094067169', '782185837585903', '1746090378119669', '3954610760394819', '2429814980595477', '1214908672640773', '15818443041579283', '55874892099829401', '15643716751718053', '3450354707094067169', '15643716751718067', '11174978419965883', '15818443041579269', '2429814980595477', '8127745784960841', '8258526754547587', '6990222597249879', '1797852706072919', '2774114546112457429', '1797852706072919', '6990222597249879', '8258525299356097', '812775015053531', '8127745784960841', '6193894842607403', '83882672060211697', '21574233366088177', '2774114546112457429', '21574231579661879', '83882670273785399', '154847354945509', '812775015053531', '7546574222757633', '1884476411790341', '436474177240343', '4035070963947921', '6919026876217030143', '4035070963947921', '436474177240343', '1884476411790341', '7546574222757633', '7546574222757633', '9045486776593637', '167606084060291711', '48420851567375051', '6919026876217030143', '48420851567375053', '167606084060291713', '45227433882968183', '7546574222757633', '3776153346635973', '8105510081551853', '6982773146852007', '7505481543688099', '3459530457043893307', '7505481543688099', '6982773146852007', '8105510081551853', '3776153346635973', '3776153346635973', '389064483914489', '167586555524448161', '45032889262128587', '3459530457043893307', '45032889262128601', '6703462220977927', '48633060489311111', '3776153346635973', '4723228498937147', '1321954615771701', '394140280420983', '5353262430861731', '66117218405916001853', '2059750396529737', '-5351309003623263', '-4842393826711181', '-5351309003623251', '-1334524324600517', '-4948087468480761', '-2669048649201031', '-483884966668239', '1244948324644817', '4209498568094061', '62247416232241', '-483884966668239', '-74457602899527', '-5556205916538171', '-8150224798112409', '-5556205916538161', '-2644305135439587', '-6215877564328227', '-82634535482487', '-994674812389343', '2611012879256703', '8605875522853247', '5222025758513417', '-1243343515486679', '-3032797483210099', '-5324053472106677', '-4945859802537079', '-5324055169224873', '-5346651107089137', '-1191269680669361', '-2673326402103667', '-4817297815360389', '4932054935463373', '8394849105715707', '4932053238345181', '-4817297178941063', '-2971143397991457', '-1321227779859269', '-6291876446466651', '-264245516462531', '-1391600696334351', '-64639098227259', '-5566401865166051', '-4980458309183657', '2616808586836509', '1077102606763451', '1308404649809513', '-4980458701174011', '-6071395280152187', '-2760582766194197', '-2366079827458577', '-5521165532388383', '-2691178986673853', '-365163757952669', '-5382357973347699', '-505996599822211', '1361465149282031', '8681317906507269', '5445860597128133', '-505996599822211', '-3056342744626519', '-5373311061364583', '-5910953099153563', '-5373311061364577', '-5529785795482061', '-1164788806553451', '-5529785795482057', '-632395833665681', '5444242781667777', '2170060924218807', '5444242781667783', '-632395833665681', '-3056090099379391', '6502250197170441', '1656866656360809', '1041309099188873', '1300935668993891', '2075989857713939', '2024103871639717', '865326797952437', '102844920307551', '3738828864646179', '2302659982557749', '97058430480517', '6905709537509613', '3343953305827569', '332944364282467', '5203238353860923', '1301233594315073', '4166773802202537', '8157463209377081', '8626726906649809', '67243386288449', '4603738063421339', '934913436147009', '8240622914042737', '1726829253024869', '574123831518563', '133110422455387', '8061695102742779', '7995210532739029', '10150547659489', '514543300081447', '5571127604145309', '51948862499737', '1998072944577797', '8065053231010051', '33920819653777', '1646537056188453', '4685310425856753', '1570718455027157', '550272950838479', '531307537950901', '306830334269271', '3835379178365889', '531307537950901', '8804365758224003', '1570718455027157', '4685310425856753', '60192000125806451', '401883898340854971', '56567975783214679', '41404469661872017', '7855956685380109747', '1571191338133508147', '41404474949299933', '282839837637368159', '401883898340854971', '300959995341604339', '8900717095461643', '3930370983433777', '578556746613489', '5262944732012253', '490759522970145591', '1917029386602131', '5262944732012253', '578556746613489', '3930371347231641', '8900717095461643', '285729811261594593', '2011443079311273713', '297127918472924573', '41196693649276007', '7853059231729147661', '7853059231729150593', '41196693649279819', '59425583694585677', '2011443265575780081', '285729811261590781', '949196153159187', '7885763379351167', '4329659942846179', '1031371470726147', '98004915487799183', '765663402248431', '1031371470726147', '8659322796075313', '7885763379351167', '949196153159187', '304326828749028441', '2018171360069388111', '277682306693088137', '40670799090957279', '7840977309374867321', '7840977298762022041', '40670788478113199', '277682389212498617', '2018171360069388111', '304326839361872521', '4537350769447851', '7985487788885537', '8404467981479113', '1184039485281713', '1526212414826093', '1526212414826093', '1184039485281713', '8404467981479113', '7985487788885537', '4537350769447851', '4553921634617433', '510806084645961437', '33750438847272727', '1150897754942549', '195381702482011159', '61056782025628511', '4603591019770577', '8437609711818277', '510806084645961437', '36431373076939083', '8750748451281037', '7886078007342427', '2347695972124107', '5154334615821261', '490035260136044463', '239275029363303', '5154334615821261', '2347695972124107', '3943039367469097', '8750748451281037', '280601165900975611', '2018258754419680729', '60216459978372887', '40657461466587661', '7841141377636690147', '7841141377636695131', '40657461466591349', '301082299891868123', '2018258940684197081', '280601165900971923', '4244073443220639', '399314894279691', '4490723318454471', '2368844485127761', '244190223157755221', '7630944473679851', '2368844485127761', '561340505756281', '3993148578999021', '4244073443220639', '17044296955309961', '204340420544881493', '72123586997163411', '575210325675089', '1953793799163933643', '488448449497937869', '9203364038619169', '18030899366628397', '1021702009592147881', '68177188993422099', '181104343140899', '1250209494894869', '2322395214515203', '2405624898313457', '7722132863204287', '40521609066672058919', '-4945090986380509', '-1366452350252599', '-4945091235880451', '-674063422471811', '338306693317487', '2706453485192007', '-67406339031773', '-1059551091159727', '5223403636897603', '5223403510093433', '-5297755588281333', '-5372107470584423', '-4944271327962901', '-1723216226199319', '-4944271139562803', '-5319586956445691', '658012339597339', '1052819705675701', '-5319586956445607', '-2689636693844961', '5383471579267239', '2691735695433611', '-2689636693845011', '-5375075290312687', '-123467962166029', '-221458629302793', '-4938717979980483', '-2714800608485503', '2738783671130831', '1369391897865317', '-1357400435451719', '-5289793862475231', '2598975916398987', '649744011282381', '-2644896795484861', '-5381635491916413', '-309516362755367', '-795779315158137', '-4952261804085881', '-1377053448416837', '1113360742254989', '5566803711274861', '-2754106896833637', '-5391580369253059', '1333384215611607', '2666768431223249', '-5391580369253149', '-5449623876059717', '-2469413212823921', '-7088635847926007', '-77169159966937', '-5306272782798821', '1046135917443653', '5230679399454277', '-5306272782798739', '-2706884961845781', '1361418467250953', '5445673681240001', '-676721240461457', '-1345466518065321', '-4952221463635677', '-1990586650397087', '-495222140136009', '-1081466765325529', '1072959932138079', '2682399984088341', '-337958380181441', '-2746598512347809', '5536525666308451', '2768262988728897', '-1098639378456939', '-136246702377429', '395524503308367', '5509024794668191', '4122398903220451', '175090954429451', '7473171281506071', '8145054031592033', '2791666230264541', '2773588047486029', '1645739706890037', '3926246635153547', '819230382525683', '3406261657083057', '4951609200660917', '2783032601934547', '47189570415331', '211065449146563', '90182868910369', '4090489448910297', '169071907678149', '2514545644834681', '55232975673387', '5467404696107383', '1532053063741337', '8180965816247129', '209979121957727', '5565052938757589', '3775899454877101', '4976640416743557', '8232677760587947', '3244836477803', '334838571699671', '1260787166735769', '1771351395401247', '217494731321377', '823268340085307', '372751684178611', '7052839066543503', '6943434127585281', '680882381573183', '3964838553065311', '139567586723967181', '3964838553065311', '680882381573183', '6943434127585281', '7052839066543503', '1354145100776353', '41660604765511633', '680882381573183', '47578062636783679', '139567586723967181', '9515612527356757', '680882381573183', '41660604765511739', '169268137597044019', '3530573357729363', '753895594892761', '7365916211910019', '7600725192040619', '13878895555220599051', '7600725192040619', '3682959924944441', '753895594892761', '3530573357729363', '4236688029275237', '11308433923391401', '7365916211910019', '22802175576121829', '13878895555220599051', '4560435115224377', '3682959924944441', '11308433923391429', '21183440146376171', '3524943165935137', '3616660249598017', '3078116422715543', '3889739307859103', '435166479817048399', '3889739307859103', '615623575581413', '3616660249598017', '3524943165935137', '169197271964886599', '43399922995176181', '3078116422715543', '46676871694309213', '435166479817048399', '46676871694309259', '615623575581413', '43399922995176227', '169197271964886553', '7049606936329043', '390261338793713', '6245591064727171', '1801826146605359', '13925503190279266357', '1801826146605359', '780699246888777', '3902612296543489', '1409921532784961', '8459528513553257', '11707839214019363', '6245591064727171', '10810955929840127', '13925503190279266357', '10810957829424181', '780699246888777', '5853918919711247', '42297645033756803', '7061213928103079', '237920823748721', '7364854903998753', '1881445767861141', '13878939366826757249', '1881445767861141', '3682430726180143', '237920823748721', '7061213928103079', '56489711424824643', '15226932719918133', '7364854903998753', '15051566142889117', '13878939366826757249', '15051566142889139', '3682430726180143', '3045386543983631', '56489711424824621', '7055621668983331', '1984831334227833', '1371748449532913', '6921459550876577', '13956964854766044737', '6921459550876577', '3429348204564421', '7939326792102943', '7055621668983331', '4233372956446559', '5954494227400697', '1371748449532913', '20764379551498523', '13956964854766044737', '20764377753760939', '3429348204564421', '23817979477440037', '21166865231667191', '3118273534872101', '31865173672856986421', '6578829217208567', '5371765671528413', '2207705626647977', '6006438361150699', '-5650145821028643', '-4475825826856267', '-353134113814289', '-1027366414272223', '109321149661021', '8871646753007757', '683257185381381', '-2568416035680559', '-5379931315805883', '-414846366789889', '-1075986263161179', '-3104812024692833', '-5474190898080063', '-1075269046487159', '-5474190036798183', '-1268274966749687', '5383465979290309', '1752540016506313', '336466677535761', '-1268275047494863', '-5457589419757349', '-1101831411803483', '-2728794279237743', '-6146233621493597', '-2813377207778379', '-5137392274388793', '-5626753591139581', '-1310521065916613', '57320679191939', '4483670651964719', '1146413748722213', '-5242084572822891', '-2739330248958721', '-49391746996119', '-1369664918375071', '-626892803764677', '-2732735391162117', '-3176532557389673', '-5465469919571159', '-5227839088179227', '1425005024998503', '559483294375323', '5700021679350189', '-1306959864201741', '-2813808738981123', '-2527945063156727', '-1125523394209923', '-782450436286889', '-2727089090448243', '-692172764327169', '-5454176630366109', '-5073225418337853', '2691817492110527', '8762916906804493', '2691818267375709', '-5073225999786743', '-2738872640400509', '-5348845315381029', '-5477743730270661', '-6146350169958469', '-2686144851614809', '-6665172015446843', '-83942206380439', '-5129371501757911', '2723273574473059', '8865494156193681', '272326749290111', '-5129367088885871', '-5651427268609339', '-8864141948054381', '-5651438903323299', '-1551360489429929', '1443699590127579', '910694614159867', '8249488971180899', '4565584686447589', '1632254363518071', '403179204598701', '5158631387992411', '5154304588620633', '8221376455038621', '6160872462845737', '8237035025866319', '733542667673703', '7838012711199503', '3885096214827729', '6899865933207267', '4921397735523689', '113331839911463', '7792142256900817', '7768107279307387', '7837914743497401', '8475090164785229', '5199659246900199', '8022498606050913', '779243835208801', '5156487930103399', '322262696749213', '8225722366221911', '1232987250699921', '1646531039912879', '587055242519961', '284701414778129', '288436412158879', '400490949546413', '2281695817848983', '4131827569727159', '322463064067667', '346904976585467', '516903140558217', '3096597303016843', '7741493257542109', '516903140558217', '346904976585467', '346904976585467', '516903140558217', '3096597303016843', '7741493257542109', '516903140558217', '346904976585467', '4324437871433453', '4457065474296369', '99114369419731091', '1548662022183299', '8914127310614143', '4324437871433453', '4324437871433453', '4457065474296369', '99114369419731091', '1548662022183299', '8914127310614143', '4324437871433453', '8567411893994331', '4443251329914811', '498206318551341271', '3892236863682353', '8886499021850739', '8567411893994331', '8567411893994331', '4443251329914811', '498206318551341271', '3892236863682353', '8886499021850739', '8567411893994331', '108823673054067', '8841986270360201', '1975277689411043', '7715928474261889', '8841986270360201', '1088236639591197', '108823673054067', '8841986270360201', '1975277689411043', '7715928474261889', '8841986270360201', '1088236639591197', '8670901022218593', '8899305712767989', '247440942429651307', '1933132180832701', '8899305712767989', '8670901022218593', '8670901022218593', '8899305712767989', '247440942429651307', '1933132180832701', '8899305712767989', '8670901022218593', '8703080839604029', '4455450182753813', '493840515313748477', '7716258051777319', '71287162178701', '8703080839604029', '8703080839604029', '4455450182753813', '493840515313748477', '7716258051777319', '71287162178701', '8703080839604029', '556092883871793', '4802807927469779', '146545054169499', '2298469711013773', '19989385783919238587', '993605294695691', '-3416984679174377', '1926910294002427', '-6833969358348749', '-6199938028520629', '282597103892327', '4521553662277227', '-6199938028520631', '-839192183121701', '-6713537464973597', '-1678384366243401', '-1354725739404263', '5400655193831243', '-1354726137952839', '-3033903437217053', '883318389048259', '8833183392296863', '-3033903188124193', '-6604858720399807', '-3302429858385627', '-6604859218585533', '-213636572304023', '7280694094723497', '-6836372312667239', '-189331309927749', '2203194693534921', '8812778274405049', '-6058601417953341', '-6608849123003249', '-3304425061236257', '-6608849622737883', '-6740581277751227', '98854324644159', '-1685145404454779', '-6091751582957489', '8874996309908021', '8874996479941961', '-609175158295749', '-41356335700087', '-413563335746627', '-827126692747497', '-1689828948573451', '972160117179953', '-337965801200951', '-1518636656580817', '1105673177760703', '2211346269374449', '-3037273284445983', '-6607416120572271', '-3303707830560913', '-825926986355881', '-6734564525746157', '4121874225861393', '-841820914574729', '-6078005759312231', '8853132493961093', '8853131796248167', '-6078005061599313', '-6605756653900909', '-6605758049326753', '-825719668951729', '255724540860147', '8920891599655739', '8315947369700563', '128866902921203', '4312001617187239', '523272046595491', '6914906286750353', '806808477169677', '898492891542067', '1376492736715019', '6515702759525319', '1776161517907913', '2585922992415119', '1644287683346417', '57601265849477', '3434890633108051', '5554839050386549', '7140174234510073', '8332145181728583', '6743404822597191', '8332145181728583', '13719920027365581', '4287475008551743', '8332145181728583', '4287475008551743', '4287475008551743', '674340196127279', '8332145181728583', '8945371194541797', '8945371194541797', '3551999738971669', '43039782462392887', '3551999738971669', '3551999738971669', '3551999738971669', '3551999738971669', '537997280779911', '3551999738971669', '537997280779911', '537997280779911', '8945371194541797', '3551999738971669', '3551999738971669', '8945371194541797', '206313947992899', '2402571150054931', '6678819560525271', '127836633994663', '523618890095041509', '6678819560525271', '2402570422459193', '5157853792992643', '206313947992899', '2402571150054931', '6678819560525271', '127836633994663', '523618890095041509', '6678819560525271', '2402570422459193', '5157853792992643', '8034651739547711', '4817583292508491', '255737087453497', '5273461363843357793', '128746615328207', '399589153671357', '1204396005026051', '8034651739547711', '8034651739547711', '4817583292508491', '255737087453497', '5273461363843357793', '128746615328207', '399589153671357', '1204396005026051', '8034651739547711', '156664554887687', '4843771960222481', '83378681549157', '163630884843462759', '1278366287839553', '83378681549157', '2421886525808061', '156664554887687', '156664554887687', '4843771960222481', '83378681549157', '163630884843462759', '1278366287839553', '83378681549157', '2421886525808061', '156664554887687', '610455607519737', '1033132485546213', '1316858084140579', '319642612419457', '261851302599823897', '823037303032017', '2582802291934503', '1221456184255609', '610455607519737', '1033132485546213', '1316858084140579', '319642612419457', '261851302599823897', '823037303032017', '2582802291934503', '1221456184255609', '151157132109823', '4869229433006531', '6618894961027037', '5242146531927955667', '1279815564627753', '206838193795349', '4869846798007777', '4813650939585613', '151157132109823', '4869229433006531', '6618894961027037', '5242146531927955667', '1279815564627753', '206838193795349', '4869846798007777', '4813650939585613', '2449320531348231', '5168902850663827', '654688435778303', '524173266360988101', '2559417221686297', '818340172043039', '5170268547808801', '3029504121539', '2449320531348231', '5168902850663827', '654688435778303', '524173266360988101', '2559417221686297', '818340172043039', '5170268547808801', '3029504121539', '902371116425457', '1562411393331395727', '738264160372151', '84931835591907', '30457753663561', '164819272328919', '-8002190248073421', '4937381422959669', '1234346100346053', '-4001094248649211', '-1383374611022139', '-5533497389111339', '-8429138149933411', '1931280149683067', '746221842241767', '1931281421777823', '-8429139760545097', '-110669958331999', '-3974092133868019', '8961470540178439', '8961468900941323', '-49676149254701', '-2853908219599743', '-89184638265762', '-4478654374736527', '8694592182951641', '7859504837286747', '347783677316801', '-1791461752177147', '-178369270128254', '-1995606979150919', '4863569189657859', '1215892142145697', '-3991213736489321', '-1387660774990557', '-5550643410499759', '-1062508872049239', '394310923587393', '1501277941135333', '3943108614798881', '-1062508863731269', '-1110128651046199', '-1938281600762551', '4379098244278683', '1751935472167123', '-7752094987410841', '-5563060704759799', '-2781346111676497', '-8675560684772443', '8284577607055107', '7646252400176531', '2071510041931201', '-4338111311577731', '-5562876464056399', '-795020307819977', '1190399882327459', '4753545542775657', '-1592278394386653', '-111500211116281', '-557901195827623', '-2148508657347249', '4031943949274041', '1515180998628461', '8047965296150333', '-268338903733533', '-5577011257045143', '-3861276367460897', '347591923086213', '8654768317111963', '-3873476555083483', '-5562330297471399', '-1392758487262633', '-8694372394537141', '4149536143930717', '1916380257785767', '4132226998226099', '-86787123309453', '-2783341061630483', '4198365455120389', '1816390758287131', '2358476744486399', '328958868714039', '2632311777270109', '935126211244713', '8325636567978091', '1707691130210509', '943553468483823', '2942929992779581', '4209502125644443', '309265242643441', '4047275180131549', '8465145102180879', '386613975303683', '1265347738346507', '2102167763597259', '7719748980138689', '2556554586498629', '8574262318030939', '4752185283360739', '7659320075903417', '7885614006876487', '7885615462068059', '245098265711974457', '4752185283360739', '8574262318030939', '2556554586498629', '4033556557267767', '137472850994145221', '30470916594638769', '784371306851982527', '126134242278585459', '126134265561650611', '3921856905297241509', '152354584464906779', '27494570497171631', '20167781294625901', '235368889873807', '1038709125450771', '4552540331521393', '195445225472263917', '1596586317219001', '1596586317219001', '7634576937223099', '4552539603925657', '1038709125450771', '4707380344061219', '18290872061523403', '26698681437215969', '146219929737065733', '3909443154786280401', '1021276600785469031', '1021276600785469031', '781888406197321569', '29243982533324617', '133493413398700749', '3658175207048563', '1051883591060913', '8718143233995013', '187507072187631', '19636598833090779', '1964986903108921', '1964986903108921', '1534109283835217', '2343839129941199', '8718143233995013', '1051883591060913', '5119878803061393', '4376514011027903', '9410238397442343', '490984739788436231', '125689392530326773', '125689392530326773', '245492370201695513', '4705120500174087', '8753027714578391', '5119880032971053', '1309971641280137', '4595066439975107', '4446546313648143', '383515084758709', '7861382900889257', '7861381445697681', '122724862047384661', '4446546313648143', '183802613943257', '1309971641280137', '10226368209284933', '590195743684463', '71398145939326451', '981925321513538043', '1005750197330382727', '1005750011065860999', '1963851197679110739', '35699075040428147', '36887227319894403', '1022636406775509', '2354806067987187', '8911088155744247', '8495130413721359', '7634839424188059', '7982905937165711', '7982905937165711', '244314908140143841', '8495130413721359', '8911085245361367', '2354806067987187', '18300950726668233', '28622981661827443', '136459584436771007', '3909575290455981399', '1021274458412748781', '1021274458412748781', '3909576028059530719', '27291918378247387', '143114869197477063', '3660188654440461', '1025100319748919', '465664396611813', '1752922317294591', '490288826864060403', '7882581409664191', '985322858106971', '1532152583950189', '1095576175460699', '931328502185311', '1025100319748919', '40439099919900319', '298590126701616761', '281032483637191001', '1569037232067919393', '2017375919181689417', '2017376291710733129', '7845186142695024121', '281032431432569461', '298590051213930037', '40439082275326243', '350622919259799', '19887516121511761353', '121209257006987', '3412551385610599', '2898051152066933', '7214319202476237', '-680805746211223', '1376064491880347', '688032328872527', '-1361611520536867', '-98882337229923', '-177806681003', '-2472058619330923', '-264480496874687', '519058596612431', '5190586052144989', '-2644804912113711', '-2694316876293247', '-5526380289115897', '5597043527890697', '2798521263031809', '-2763189947664301', '-2478966824040351', '-318581366279077', '-4957933897231967', '-2693716954480089', '5319150553326721', '1329787826500633', '-2693717089085927', '-5455717157467389', '-1352656681918947', '5441039409749267', '1088207930280877', '-2705313240020749', '-989762645384007', '-6902411008845953', '-4948812751255241', '-5300154645652177', '2610047991312847', '5220096216635361', '-5300155131118873', '-2690106838570253', '-533157992747249', '21128515969643', '5282129659543761', '-333223789777331', '-2474395918350621', '-3457918173105317', '-494879109588759', '-5381538870776157', '2691023252505499', '672755681040369', '-5381538343823409', '-215241249517303', '-169377200952098', '5384870494034021', '5384871319976799', '-5420071177088587', '-4957709900385039', '-7960976805528313', '-991541880004929', '-686591525421287', '5530193372540167', '276509652347889', '-1373182926732263', '-1363817675137589', '-1329891414192527', '5252154870230273', '2626077645108243', '-1063913269625547', '-1236043170258799', '-3542432855609143', '-4944172954411543', '-1353009227025191', '1359274008617597', '5437096468184151', '-5412036267901031', '-5386976924674507', '1316131332901831', '4975453473687401', '128279138095613', '4112014031822297', '1511924922652919', '1116553788928669', '336180676373363', '2713518796927747', '1200440406013693', '8223986743359851', '22174424732433', '5058514194122861', '5623070279363831', '394872084987297', '1496745852316751', '8187398756453883', '8256088412340797', '1390047253953119', '3935534557759173', '2806017837111013', '1638963836628119', '5214526149409587', '8249954271857417', '5563668241260489', '217153556127403', '1079250365642619', '1646526151704449', '6022345156724431', '7110027972353521', '2528065973442403', '248408235841071', '1317987765167153', '514540009932179', '994034024439537', '3789200277609509', '2791294291434343', '2079604574810177', '4660019135419373', '3994672098520239', '488597157006277711', '4506745994356849', '1126686316690271', '763433057822309', '3994672098520239', '4660019135419373', '2079604574810177', '267273711758522741', '36195826900534899', '16353502287668591', '7818638842885200277', '289516069821658421', '289516027857466277', '1563727767656652849', '16353502287668591', '36195822298598083', '267273716360459557', '4291688077509561', '5061982754398607', '3945664288629829', '19607973152618449', '117690081406201', '117690081406201', '3829682256370791', '3945664288629829', '5061982754398607', '4291688077509561', '137612618804820981', '19969330693079399', '1009811457564714547', '6274997169357149', '150921904524452309', '30184380904893121', '3921873230848205013', '1009811457564714547', '19969330693066103', '137612618804834277', '2228660566667697', '2330643299279819', '7988094900233027', '61078841284388893', '8420150723415883', '4210077180697351', '1908713790137153', '7988094900233027', '2330643299279819', '2228660566667697', '286318836347213523', '36240008974728797', '2043902016809790819', '7819141955887758183', '270495106963056563', '270495211050610343', '1563828393643105399', '2043902016809790819', '1449600852099889', '57263764803889019', '932700588000253', '1015649992549187', '1971830603753527', '490299770693959581', '8681422566700003', '8681422566700003', '191523293357649', '1971830603753527', '1015649992549187', '932700588000253', '299024177773546233', '40066010088502207', '2018594553078582437', '1569071262363989289', '278365511747865369', '55673100570198649', '7845354085542768313', '2018594553078582437', '40066018985374331', '299024168876674109', '8731674211100693', '5166699190623127', '1966599609728029', '19636809263534949', '4639580087231391', '4639580087231391', '7670628618568341', '1966599609728029', '5166699190623127', '8731674211100693', '280288846655702439', '40458321624504753', '402584545692201819', '7855598977314484539', '297808397483289287', '297808397483313963', '7855598977314461447', '402584545692201819', '40458321624480077', '56057769331145423', '2300744073482267', '1299559706880983', '98278992967', '982052049441279703', '2203085092065407', '2203085092065407', '7672281636259999', '98278992967', '207929640412437', '2300744073482267', '295493555695585819', '40587596330335813', '2011755454884607861', '7857414723399486259', '282993206074227739', '282993219653620731', '7857414709820094619', '2011755454884607861', '8117520042647753', '295493569274978811', '5604263798391107', '51863372422311601423', '1554940539043217', '4887828627025303', '5730834174495453', '716632713615397', '-1381986131809331', '5592241490623801', '1118448273021343', '-44223557275013', '-135149246223997', '213931669770929', '5348291622347577', '-1081193918619843', '-77523892542352', '-1004237024633617', '-1240382342538313', '-682955969843587', '-680983172945011', '1099935230323051', '5499676151615887', '-1361966345890193', '-212279856631363', '5217938216062491', '5217938216061833', '-265349820789169', '-4947586594184933', '-1793872085281409', '-2473793297092477', '-5396054615505293', '-1355622612461279', '5382175131597359', '2691087731420357', '-5422490094001981', '-5509238861281251', '5555672997399249', '5555673319018199', '-2754619771777769', '-4961383247692831', '-2005689301465559', '-2480691297415261', '-5462805246628063', '-1064006663837227', '2623019880165797', '655754861581229', '-266001648975701', '-2714634862350701', '5464512383024589', '5464513247337063', '-5429270062688871', '-4947688264264119', '-1785354825042743', '-2473844133816707', '-674253371526181', '-676664184736269', '271861264687923', '2718612646879521', '-2706656738945391', '-532079997140267', '2626099140391407', '5252198280782219', '-2660399985701017', '-4952984048386243', '-279307272727003', '-4952984048386257', '-1347350415505547', '-2665667470639537', '5275215455214957', '5275215451490167', '-2665667398038253', '-10794579188097', '2703562523717079', '2703562527128171', '-674661217599971', '-1238334032341081', '-6945890498998399', '-4953336126266923', '-269372714200151', '664448022005193', '3552998038440539', '2524418410222953', '1180754343944107', '256999292262929', '2716290612137359', '131285772866379', '1889588247924549', '2782853888595987', '16941167480003', '328958868714039', '4980338169732157', '5406282153988301', '1776250478407723', '1006529623502423', '2045242855658327', '6086990753555833', '2720717143406271', '494860253872489', '7575313263788133', '1391466331160981', '1636188678147831', '36700099461817', '5285510708416903', '223453249801979', '329556274518823', '693462516832787', '404151376993017', '8183062860967001', '1963163776477229', '3940391575870599', '8247208042177477', '552041268792989', '8142456543807471', '5183776437613801', '1400779372994167', '2421582066167719', '3237679677687739', '2655923964607336549', '6093227874818251', '6093227147222479', '648418936671713', '3237679677687739', '605395652966137', '2421582066167719', '3237679677687739', '2655923964607336549', '6093227874818251', '6093227147222479', '648418936671713', '3237679677687739', '605395652966137', '2410540411972481', '1610519983062467', '257569137195123', '1595083274937443', '1595083274937443', '5275017047343213893', '8052623198376807', '964215946510263', '2410540411972481', '1610519983062467', '257569137195123', '1595083274937443', '1595083274937443', '5275017047343213893', '8052623198376807', '964215946510263', '2390239152703937', '7487154486583719', '5282692493755105779', '316636207807167', '1266544685709519', '2579439694216361', '7486297378795827', '4780490310737671', '2390239152703937', '7487154486583719', '5282692493755105779', '316636207807167', '1266544685709519', '2579439694216361', '7486297378795827', '4780490310737671', '1278147801543107', '302128436693607', '1056307456391321253', '3129230464100239', '156461577774689', '1289437812977687', '7554853100821567', '5112569014503761', '1278147801543107', '302128436693607', '1056307456391321253', '3129230464100239', '156461577774689', '1289437812977687', '7554853100821567', '5112569014503761', '4868814931279551', '800363666603163', '2637525575145273419', '1592080184627749', '6368319283319411', '5151417138955613', '4001837614304323', '2434406919942931', '4868814931279551', '800363666603163', '2637525575145273419', '1592080184627749', '6368319283319411', '5151417138955613', '4001837614304323', '2434406919942931', '1276281939903667', '7815480970364341', '5277156209794488397', '314676479498933', '1573382943191499', '5153472861127431', '1563281876515581', '5105115390486431', '1276281939903667', '7815480970364341', '5277156209794488397', '314676479498933', '1573382943191499', '5153472861127431', '1563281876515581', '5105115390486431', '99977120073728704541', '8050749787218537', '728909907058791', '489116621206527', '4762545832183521', '188979852076619', '-1965632284298971', '8000664508337157', '500041401228407', '-1965632149868331', '-4713381359074563', '4757603541295407', '411677952132487', '4757603225778399', '-942676271037317', '-2931181370625167', '-366397703963811', '-5862363002335653', '-1587879237316631', '2232215646315929', '2232219111395331', '-793938975928843', '-2238962413110689', '8682014617666277', '3931759053788711', '8682025891751137', '-895585417656919', '-5707176680061679', '-2853587254049329', '-5707175594080169', '-7967208314100083', '8840856275231523', '8840480969710373', '-7967473764094103', '-56808320532308', '1786973678418479', '198849455708627', '8934494489093183', '-284036298342236', '-287856348514461', '-719652599583687', '-5757173883479359', '-1544275675934091', '1579496069550111', '7898191576463821', '-7720873579805371', '-460032017882443', '4599354564044181', '4027702271191851', '1149927012413743', '-1150120310099443', '-1436688973200071', '-143664452140549', '-2873333494605561', '-3952579533353323', '438963565942129', '1097410764127871', '-395257397227663', '-4493420174200019', '8752750816965449', '1971743589792801', '8752768400510859', '-280838988805844', '-1427583918979829', '-5710331977374219', '-1427583456661691', '-7731035527024897', '8030616550873187', '8031006560955717', '-193268998111767', '-2281754583884497', '2264939739674129', '4000520929753647', '1132518091876297', '-2281798571041303', '-2861621794014751', '-286157304275443', '-286159741838459', '7281078193927597', '9178847410579', '4111981163460627', '8549632471105669', '2313329467372909', '8265032649350453', '7808720543919243', '7507690374032019', '8526205972700561', '7798889124190239', '3753856753566313', '6086095133714499', '559785810965371', '4625656477656053', '8258185007029897', '7818349107047807', '7509828809585719', '2879234534176607', '3657713565653721', '3657713565653721', '3173252343190239', '4382601587398289', '3173252343190239', '3173252343190239', '3173252343190239', '3173252343190239', '5478251984247863', '3173252343190239', '5478251984247863', '5478251984247863', '3657713565653721', '3173252343190239', '3173252343190239', '3657713565653721', '1855000587955987', '1272981853420803', '1272981853420803', '5398390326893097', '3211281912930383', '5398390326893097', '1272981853420803', '1272981853420803', '3211281912930383', '465703488951255953', '3211281912930383', '1272981853420803', '1272981853420803', '5398390326893097', '3211281912930383', '5398390326893097', '1272981853420803', '1272981853420803', '1855002372794227', '4448008152649979', '5293948505229161', '1150194942521247', '42746942739943375233', '5218119561960027', '2875460980955233', '5294353412293927', '8778306583771273', '4448008152649979', '5293948505229161', '1150194942521247', '42746942739943375233', '5218119561960027', '2875460980955233', '5294353412293927', '8778306583771273', '1355636494816451', '4939427192870353', '5824850485669047', '21378943454441865919', '1043876042784309', '2912248255164479', '1235526732018999', '3197488714646137', '1355636494816451', '4939427192870353', '5824850485669047', '21378943454441865919', '1043876042784309', '2912248255164479', '1235526732018999', '3197488714646137', '241029165571869', '2631872618951591', '2893325238013399', '33368430424735737', '2606903715661179', '2893304683433513', '658006308039943', '239670835259479', '241029165571869', '2631872618951591', '2893325238013399', '33368430424735737', '2606903715661179', '2893304683433513', '658006308039943', '239670835259479', '1429134616755089', '1263506860216867', '5877840939030673', '42680123158655831883', '5209974516323889', '1175567241931519', '315878897841791', '222985620508119', '1429134616755089', '1263506860216867', '5877840939030673', '42680123158655831883', '5209974516323889', '1175567241931519', '315878897841791', '222985620508119', '2767954664025153', '5018093259474313', '224188096580121', '42970617733782507859', '5245436735080873', '1401175239827883', '5018093259474313', '1107163893995119', '2767954664025153', '5018093259474313', '224188096580121', '42970617733782507859', '5245436735080873', '1401175239827883', '5018093259474313', '1107163893995119', '2887327820775511', '2498224873334997', '5888572767355223', '4267959745491276459', '5209891789811261', '2944242364132491', '4997099125909999', '5682152387357563', '2887327820775511', '2498224873334997', '5888572767355223', '4267959745491276459', '5209891789811261', '2944242364132491', '4997099125909999', '5682152387357563', '1034347196311433', '805684843851111', '1929219187906669', '927567582336429', '99977123886731408171', '3995577088497891', '-5862390629681851', '7238157762565467', '483680634084359', '725933096456689', '-1468058838226747', '-1638049596712769', '-3270795726498783', '-388615440670661', '240889220721273', '1215068356381029', '-7740995486422811', '-3273447459962159', '-5389118555182219', '1543276563315107', '91209392139341', '3133073551790361', '-5432159022734483', '-158132909051847', '-3139331082441793', '-7952087201281653', '6234107263632381', '100330274380617', '-7815702531673783', '-6301989263478731', '-1063101647027341', '3012532914788051', '2260658830313447', '754264151592333', '-332480533414676', '-1550109627085099', '-774487360740419', '-305010829212893', '567283970245953', '5691044186911061', '-237876176133877', '-6198168697131871', '-6025392976493021', '3815427078030071', '2448464383376761', '1908276743358973', '-6026441684024881', '-1335679045734433', '-6677267216889317', '-4016017829617197', '337992562985417', '1353099968489903', '-8028697006758653', '-6677831222780737', '-6287082481683073', '323894392650163', '2548116826530297', '323895388748643', '-1571773600580471', '-6858520816194749', '-6858507572612553', '-7997300342563743', '4555041813387583', '2277547393858201', '-7997262196519551', '-137170283888073', '-5546634519571143', '3278665091302997', '578586562971163', '822886751102741', '-2779287323403503', '-3194639569590853', '-6376363319825753', '-7942128581802087', '6135578019592081', '3093667530212023', '-7904242147209401', '-398926326843983', '41276187089663', '451806329206247', '7162649905058419', '55610584894829', '7219439949120479', '7170649299236039', '7566503731515529', '7216904302167149', '449027783234439', '4136014628436137', '8985807806123393', '2244528452528771', '8232672120326517', '8924942144922009', '1911967392039653', '64682473800437', '8965980079667401', '452653491854263', '5526465064940931', '1833120767351173', '1691868586476691', '3576718507184423', '21972700326986411', '3576718507184423', '6767475801098431', '114570082065503', '1104886214157849', '5526465064940931', '43994899193568047', '40604845298300689', '171682487567712409', '21972700326986411', '171682489121992199', '40604855583730481', '43994910736013257', '1104886214157849', '44785995903527', '6775372753132293', '915412816327547', '715386076811037', '5624954937546282261', '715386076811037', '3661651992905903', '6775372753132293', '5594433248415779', '44785995903527', '13550744883534239', '14646605683971099', '57230886767613307', '5624954937546282261', '57230885522152613', '2929321469778653', '13550746128994933', '5594433248415779', '7380888894866549', '7464692642560613', '6503962154166891', '3577854830179479', '56371584480944065423', '3577854830179479', '325198908063639', '466544336078889', '1832146419092991', '7380888894866549', '44788153816485143', '39023774963879881', '171737033887493527', '56371584480944065423', '171737029809736457', '7804773385751629', '44788258302451879', '1832146419092991', '3253102796828631', '357503590445091', '708886427149459', '7180713048786639', '27997058250079036133', '7180713048786639', '708886427149459', '357503590445091', '813262511534161', '3253102796828631', '715007180890181', '35444321357473', '14361426097573283', '27997058250079036133', '14361426097573273', '354443213574729', '715007180890183', '813262511534161', '1675423619829841', '6515696693787899', '116490075441169', '3577114832802903', '56373199026897222381', '3577114832802903', '1491077185702543', '6515716338874219', '33245138018027', '1675423619829841', '4886772735220249', '22366093625187149', '85850755127752373', '56373199026897222381', '85850756846786971', '5591539661263861', '9773574078552679', '33245138018027', '202384886275129', '7091903808489433', '7144129173707091', '7181404629772541', '55994316328913840447', '7181404629772541', '7144129173707091', '7091903808489433', '6476217407786913', '202384886275129', '5318927856367063', '535809688028033', '86176855557270539', '55994316328913840447', '17235371111454089', '10716193760560613', '10637855712734173', '6476217407786913', '803361418037749', '1481170909468301', '1876733993183613', '96434285487958815407', '63687084202981', '6788554586787', '-667912174970151', '4367507827560653', '668628588672163', '8735658943669037', '-3339681294743201', '-630616878226939', '-1955452098407263', '-6305527039775437', '-1535757326143891', '-285216374817303', '-95974806520908', '-7283573960226047', '-3333264462192203', '8706838959542123', '1068431283598227', '4354014332990741', '-666697490745121', '-614015148891079', '-4542816129335591', '-6138962476275717', '-6298369139150393', '-62559132704433', '-1259435998738079', '-3637630507463577', '-3194965572082233', '402967373962359', '2600068478802473', '1614352254234337', '-3197291369156417', '-625224901317183', '-6857201323586361', '-1559962401250531', '-2985975012792611', '-71090576593527', '-1191910094498493', '-7103199474535277', '-6528520654817851', '1053902338898011', '1051249567634737', '4215623494501579', '-816066407375001', '-6149060610986273', '-8170630632943973', '-614903233316719', '-6131569441708893', '-4155279993581511', '-766442645486227', '-7170375301194729', '-783235135481061', '7784427270183101', '2565509610227059', '3899100610250283', '-6271041864105779', '-1184167201921299', '-1770456485665043', '-2953539901314563', '-6194923778605547', '-3329789893738661', '-123623353876841', '-7020493209658463', '-1633246002922529', '2110130539038923', '105179139975169', '8440548801192199', '-6532994003578807', '-6134843690380599', '-2077352372969307', '-6134817045344087', '-1537442764686763', '-1637998108989121', '-1229948882742109', '-7173506554347', '5026951761357187', '4761926058357661', '7456979537605671', '1883987445802637', '1494130487926623', '7186066556532821', '7129198718735899', '5385776595727669', '7095026212595223', '7473455807455077', '7534134406096511', '718749086252477', '592393638338203', '4474232400259511', '2109985479023571', '6870035555457357', '8978644062334937', '72778935054003', '3073626944449669', '8233631017837503', '8238218280641741', '256347841595513', '5124646734530597', '1037636558417273', '2253059938283069', '8480306901520649', '1201634086429791', '561772518971309', '856876172143527', '5820736600779123', '6175134252933547', '8232674000413251', '2059801504584477', '2562894818300143', '2562204169650747', '8336757671414069', '1424499181288433', '5831591311662173', '2533877908866143', '5214806196571219', '1067995587313276523', '633533050896741', '5831527283234613', '35104790490331', '1424499181288433', '5831591311662173', '2533877908866143', '5214806196571219', '1067995587313276523', '633533050896741', '5831527283234613', '35104790490331', '3787021008031143', '719034329684723', '213962501095311', '5216397808365879', '42732781510081740417', '171176403719', '5752249171625933', '7509984484719921', '3787021008031143', '719034329684723', '213962501095311', '5216397808365879', '42732781510081740417', '171176403719', '5752249171625933', '7509984484719921', '2764170803672059', '5615567604484511', '994114578450703', '42971663815912655511', '26227822153267', '994114578450703', '5615567604484511', '5528251749269363', '2764170803672059', '5615567604484511', '994114578450703', '42971663815912655511', '26227822153267', '994114578450703', '5615567604484511', '5528251749269363', '949960711567963', '5892077096477439', '1249338340136101', '325619437061039', '8535920555078572993', '4997422482138317', '5892068365328733', '7579115833790819', '949960711567963', '5892077096477439', '1249338340136101', '325619437061039', '8535920555078572993', '4997422482138317', '5892068365328733', '7579115833790819', '5219408985650321', '1159566096840069', '214520785839589', '1302478465377519', '10669957232549844067', '1340940266506989', '2898868675974121', '5100285923050537', '5219408985650321', '1159566096840069', '214520785839589', '1302478465377519', '10669957232549844067', '1340940266506989', '2898868675974121', '5100285923050537', '1747436764024389', '5882865059464033', '100586456347909', '10670362639785160287', '5210138007707597', '5029336277917323', '5882863604272479', '436640912273031', '1747436764024389', '5882865059464033', '100586456347909', '10670362639785160287', '5210138007707597', '5029336277917323', '5882863604272479', '436640912273031', '145615087861049', '265643291901127', '12499575885484296739', '1214463337183247', '3770578076531', '315733100970823', '-5575913188039081', '6606816321875347', '4656974758145213', '265187504121313', '-1396631992727839', '-7872106344693981', '728413484939707', '367076371292831', '-244949618395921', '-6398410729294363', '-1596737737254591', '-6392680839156363', '-6028303849446993', '1523154242741971', '246469702733837', '7629243638531999', '-120691595696711', '-3892292694735233', '1126741874060053', '28337380376967', '-1552919033169929', '-3323922392718227', '-1660274670186231', '-207639741659084', '-627918287847223', '2019808156909461', '5088783709107613', '807925822965133', '-3139597439708023', '-1606592545572111', '4691977827721359', '4692029031748391', '-8032924324840351', '-3429974534710053', '-3429968134206687', '-342997133445837', '-5954294995200473', '7471468600707119', '4853629470399583', '7475772600229829', '-1489074479851407', '-125884015385799', '5648017542043293', '5656647039609843', '-1610040347322253', '-332069206137373', '-3319614226570837', '-1328061257588913', '-2857766836411129', '3468183664760681', '2374262863179881', '871635424803489', '-358287410791424', '-1543535951575521', '618424429730307', '5021085034465431', '-1915853927187809', '-1613420882706547', '-1608822954431421', '-6444487674275933', '-3021079675337897', '7667810263145797', '1226182245333137', '958575526309593', '-6042528686486817', '-8055462428879477', '681482509776481', '545345099241801', '-805428708847569', '-6691909739124633', '-1338302402114617', '-3345855437424429', '4598047545391229', '936420858455827', '7279121960743779', '1340008517375237', '7343830567491337', '7248320483759711', '1022621895028267', '7854333797273539', '8896968584639759', '1056040591141709', '4842408523813293', '8965217140803391', '329131611308781', '7447100503504863', '1816148778967937', '8387120411161519', '4712760225035973', '4481736056659023', '6408780601113107', '6234163435893331', '6234163435893331', '241619312344521', '313425147255221', '241619312344521', '6234163435893331', '6234163435893331', '313425147255221', '378739155301374789', '313425147255221', '6234163435893331', '6234163435893331', '241619312344521', '313425147255221', '241619312344521', '6234163435893331', '6234163435893331', '320439481171101', '1101235298898497', '4314069116812917', '4314069116812917', '1101235298898497', '20880483817970421', '1044024190898521', '4314069116812917', '1044024190898521', '4314069116812917', '4314069116812917', '4314069116812917', '4314069116812917', '1044024190898521', '4314069116812917', '1101235298898497', '1101235298898497', '7527659901214189', '4457543759476639', '3172679242147817133', '968212133452307', '445746699812389', '3973251655609433', '7527659901214189', '4457543759476639', '3172679242147817133', '968212133452307', '445746699812389', '3973251655609433', '2468819741983109', '8762637341376407', '4001810048599212007', '7815736936906703', '2190550559777241', '5236067616334863', '2468819741983109', '8762637341376407', '4001810048599212007', '7815736936906703', '2190550559777241', '5236067616334863', '838425521439121', '3479027307874', '1983308440769154587', '968383402662489', '8905962844975537', '1794587134168643', '838425521439121', '3479027307874', '1983308440769154587', '968383402662489', '8905962844975537', '1794587134168643', '66878609254371', '8845053775295273', '31885061924476781933', '7784438946405467', '8845053775295273', '268372999969543', '66878609254371', '8845053775295273', '31885061924476781933', '7784438946405467', '8845053775295273', '268372999969543', '6007382628998539', '8828332451316147', '3989194244823252783', '7791376091931039', '8828302619890593', '6173560404039933', '6007382628998539', '8828332451316147', '3989194244823252783', '7791376091931039', '8828302619890593', '6173560404039933', '1907626482717221', '8830509189825439', '797830260201088739', '779130749679763', '8830505551846807', '1928267645880931', '1907626482717221', '8830509189825439', '797830260201088739', '779130749679763', '8830505551846807', '1928267645880931', '604386234405183', '597882620255501', '96055556100503', '99927768389838419029', '4990130696017457', '1487103452986879', '-6155372875045009', '4662807386070409', '580782749765363', '-3061159841800403', '-5904195706312377', '-298514849574881', '-5016250925235817', '-136785049372861', '-2574214327706713', '-2968623174452499', '-664673375362559', '500153786348039', '995815336050543', '-2112623144471', '-25965425217719', '-658090984102767', '-4591969482023363', '-2261316819569657', '-4735185024557749', '-5228906458182967', '-707407117691311', '1708648360433807', '1062716330852769', '-2808915499509111', '-2733891709902867', '-5550623444406717', '-37732549629681', '-1419644969365979', '-1220548297784089', '-2754601716053113', '-4799693240162323', '7372076965237573', '3685549550007677', '-47987153749401', '-8905326599459369', '-556827378771767', '-7084617553673417', '-3637506630683039', '-7092440475451193', '-8907282329903817', '-3468096241788039', '1055430726956973', '5268822492472159', '-3459768670572263', '-6602852519971771', '-6636162269882533', '-2767853426057487', '-8403949562463313', '-5602320547123509', '-132390147898543', '-989825580492781', '756317661575277', '377953517975624', '-158240715310383', '-7459117073464381', '-7472254243532403', '-1223894141546753', '-1066462090403657', '-153643634581347', '-1866421414624597', '1327077810280359', '6188304616154437', '5624492196528147', '8948518148821729', '785929519411291', '5749900242877799', '258321080439031', '3106698854849399', '5670769281136147', '68513555218919', '463515566940641', '30931207805593', '4755372350796161', '4586634783313599', '1781546191767819', '28579712580099', '2318824886236489', '108863745165977', '616011717621121', '8002107938181255489', '2193211076035569', '8772417205451529', '7814263857111389', '1350372409688561', '616011717621121', '8002107938181255489', '2193211076035569', '8772417205451529', '7814263857111389', '1350372409688561', '1078569381469361', '7935096164833198539', '8898940545228189', '88987171733339', '7748966008554573', '8933335451374253', '1078569381469361', '7935096164833198539', '8898940545228189', '88987171733339', '7748966008554573', '8933335451374253', '276426537007137', '997097269716963967', '8833067452319891', '34504138471743', '7789813688514313', '1409171054062443', '276426537007137', '997097269716963967', '8833067452319891', '34504138471743', '7789813688514313', '1409171054062443', '382412075450033', '6382888154143390439', '551880759159631', '1766017847234247', '7791610061096887', '1543298906920633', '382412075450033', '6382888154143390439', '551880759159631', '1766017847234247', '7791610061096887', '1543298906920633', '1537570644621069', '634602547550720001', '69635081084577', '8913076465671709', '3873234532660147', '4140081733712979', '1537570644621069', '634602547550720001', '69635081084577', '8913076465671709', '3873234532660147', '4140081733712979', '267086582317969', '31884900314154795881', '8845132895204529', '8845132895204529', '155687989815209', '133973175166083', '267086582317969', '31884900314154795881', '8845132895204529', '8845132895204529', '155687989815209', '133973175166083', '369770567967113', '46216473957991', '4201184849076191', '77613860832947', '1215947015145031', '99900838849480518431', '-5717486341192567', '1078236542826693', '8569790782143551', '-5661491845914913', '-4771128901768751', '-1086727811909691', '-4995007832259851', '-5506286768753037', '-2809129865081341', '-5562273249457861', '-5447468776464311', '4107946089080547', '8194643884752647', '-2713137909012313', '-580543138582543', '-5201981078842917', '-2364532936878591', '-1063152886786261', '-669767727274619', '-5336953126064131', '-7600833100034127', '5800155343854497', '2897120706428747', '-1897251967507731', '-2962391043440961', '-4969827206162289', '-5972081163449159', '-7179063985248153', '-90033910852051', '-1797722106676529', '-2026871882672889', '1549776705108109', '619639516839499', '-202551621172011', '-124604289657539', '-1387041262274683', '-6251904547551957', '-381133902168617', '-1526704702321187', '-7628100777489139', '-6134773607378207', '928968405507191', '4622162302855011', '-6089466948780737', '-4977193654892889', '-3386759572559741', '-2579185730949573', '-2934584878159637', '-1191955029350527', '-1182894490307191', '-4800379955156369', '1843289161812909', '7372175845565387', '-959879830694023', '-283414268388597', '-7276686129647749', '-1773300780801241', '-8906489845499377', '-1782082610448879', '-8908451448871887', '8260984284759601', '57224619763607', '784376072995793', '3274828516659881', '5688365715477387', '3118364917410207', '1143790338727613', '3099072534691659', '4599490914941703', '575728879208327', '634263035776927', '4637151456135417', '3630917924732403', '1409083091484259', '3097814323129643', '2143941864971143', '15878751174127', '1158513641126899', '5473477573651907', '1073398258869051', '1073398258869051', '5473477573651907', '1671911751130333', '522472422228229', '1073398258869051', '522472422228229', '1073398258869051', '1073398258869051', '1073398258869051', '1073398258869051', '522472422228229', '1073398258869051', '684184396426249', '684184396426249', '6165919470185869', '7706642882885671', '7005009481309839', '7005009481309839', '6165919470185869', '7005009481309839', '7005009481309839', '6165919470185869', '602081475225443', '6165919470185869', '523495181564093', '24999486931606563561', '3051695182080879', '251320802655789', '523495181564093', '24999486931606563561', '3051695182080879', '251320802655789', '1099499524923601', '3122373713272115299', '6098383248351863', '1002125903768737', '1099499524923601', '3122373713272115299', '6098383248351863', '1002125903768737', '5103432613396019', '762347230933137', '6245151496036656123', '2298277772844777', '5103432613396019', '762347230933137', '6245151496036656123', '2298277772844777', '4645507663877763', '3043389425474701', '1558216130901120181', '490371948994393', '4645507663877763', '3043389425474701', '1558216130901120181', '490371948994393', '8318874365044449', '3121075097649082323', '6095839977554057', '7390793321436537', '8318874365044449', '3121075097649082323', '6095839977554057', '7390793321436537', '3018250771259893', '1556949326155424157', '48654666442357', '5064846146632057', '3018250771259893', '1556949326155424157', '48654666442357', '5064846146632057', '99947608386114151541', '512115340202759', '913272259682157', '2298919454217797', '2399320933918021', '872675947102539', '-6981600504533109', '947864756564431', '-7002276379445353', '-2262952596623197', '-896910688684381', '-1131476298311599', '-4484553443421907', '-4505229318334151', '-2270345072188227', '6241342954302657', '-1146963949934811', '-91699008783497', '-1420018849277089', '-2934369009279757', '-1420018485173163', '-5774405979626083', '-91804432020897', '3155722163842743', '-1160856759588279', '-5950820226202347', '-358624813790473', '-2975409384940319', '-5737998476969279', '-2922204311712479', '-1499584181331789', '2605306541930079', '-7669887676487819', '-4918549914851583', '-1143653729000327', '-2459274227829529', '-2287308187596917', '-949316483085289', '-2135739140804669', '1472712266404727', '-173265785584663', '-2772438984251483', '-5304217453925047', '-1386220721488499', '-265210626823701', '-678068463901751', '-7155580014674093', '4984159585940881', '-7334108729549847', '-469989828721793', '-1737136342986569', '-1174974571804483', '-4342840857466423', '-4521369572342177', '70782194683733', '3517585671220911', '2478795821486077', '1426286542102259', '3624234657253431', '383144496307057', '3082245774453093', '4817212896951261', '3502268223745747', '3502268223745747', '3082245774453093', '3502268223745747', '3502268223745747', '3082245774453093', '19268861057809911', '3082245774453093', '1', '1', '1', '1', '1220703438155961', '6103517190779803', '1525878514805049', '1525878514805049', '1'],
    'dens': ['1525878906250000', '12207031250000000', '9765625000000000', '24414062500000000', '48828125000000000', '12207031250000000', '6250000000000000000', '20000000000000000000', '12500000000000000000', '312500000000000000', '390625000000000000', '1562500000000000000', '6250000000000000000', '100000000000000000000', '25000000000000000000', '25000000000000000000', '6250000000000000000', '781250000000000000', '12207031250000000', '97656250000000000', '156250000000000000', '12500000000000000000', '50000000000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '1953125000000000', '781250000000000000', '12500000000000000000', '1562500000000000000', '1220703125000000', '12207031250000000', '12207031250000000', '3051757812500000', '156250000000000000', '390625000000000000', '1220703125000000', '97656250000000000', '1953125000000000', '390625000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '781250000000000000', '3125000000000000000', '3125000000000000000', '2500000000000000000', '50000000000000000000', '6250000000000000000', '25000000000000000000', '6250000000000000000', '40000000000000000000', '11200000000000000000000', '33600000000000000000000', '33600000000000000000000', '33600000000000000000000', '1600000000000000000000', '1600000000000000000000', '25000000000000000000', '200000000000000000000', '800000000000000000000', '100800000000000000000000', '100800000000000000000000', '100800000000000000000000', '100800000000000000000000', '11200000000000000000000', '1600000000000000000000', '448000000000000000000', '100800000000000000000000', '350000000000000000000', '225000000000000000000', '1050000000000000000000', '10080000000000000000000', '5600000000000000000000', '20160000000000000000000', '11200000000000000000000', '315000000000000000000', '1575000000000000000000', '16800000000000000000000', '50400000000000000000000', '100800000000000000000000', '33600000000000000000000', '1575000000000000000000', '16800000000000000000000', '2016000000000000000000', '100800000000000000000000', '14400000000000000000000', '50400000000000000000000', '50400000000000000000000', '33600000000000000000000', '33600000000000000000000', '16800000000000000000000', '1600000000000000000000', '1600000000000000000000', '25000000000000000000', '4000000000000000000', '5000000000000000000', '2500000000000000000', '781250000000000000', '19531250000000000', '1562500000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '50000000000000000000', '12500000000000000000', '781250000000000000', '9765625000000000', '3051757812500000', '390625000000000000', '2500000000000000000', '10000000000000000000', '6250000000000000000', '390625000000000000', '6103515625000000', '50000000000000000000', '19531250000000000', '781250000000000000', '3125000000000000000', '1562500000000000000', '97656250000000000', '24414062500000000', '24414062500000000', '19531250000000000', '78125000000000000', '4882812500000000', '9765625000000000', '976562500000000', '48828125000000000', '39062500000000000', '390625000000000000', '39062500000000000', '781250000000000000', '1562500000000000000', '2500000000000000000', '781250000000000000', '12500000000000000000', '100000000000000000000', '50000000000000000000', '10000000000000000000', '4000000000000000000', '800000000000000000000', '16800000000000000000000', '50400000000000000000000', '50400000000000000000000', '50400000000000000000000', '672000000000000000000', '200000000000000000000', '25000000000000000000', '800000000000000000000', '16800000000000000000000', '10080000000000000000000', '7200000000000000000000', '288000000000000000000', '50400000000000000000000', '336000000000000000000', '200000000000000000000', '8400000000000000000000', '5040000000000000000000', '1575000000000000000000', '3150000000000000000000', '1575000000000000000000', '1440000000000000000000', '16800000000000000000000', '25200000000000000000000', '25200000000000000000000', '157500000000000000000', '787500000000000000000', '10080000000000000000000', '50400000000000000000000', '5040000000000000000000', '25200000000000000000000', '157500000000000000000', '50400000000000000000000', '50400000000000000000000', '1008000000000000000000', '25200000000000000000000', '10080000000000000000000', '10080000000000000000000', '8400000000000000000000', '16800000000000000000000', '2400000000000000000000', '160000000000000000000', '160000000000000000000', '10000000000000000000', '100000000000000000000', '4000000000000000000', '12500000000000000000', '781250000000000000', '195312500000000000', '781250000000000000', '12500000000000000000', '6250000000000000000', '6250000000000000000', '50000000000000000000', '2500000000000000000', '781250000000000000', '97656250000000000', '4882812500000000', '781250000000000000', '12500000000000000000', '100000000000000000000', '12500000000000000000', '195312500000000000', '3051757812500000', '24414062500000000', '48828125000000000', '976562500000000', '625000000000000000', '12500000000000000', '3906250000000000', '25000000000000000000', '4882812500000000', '12207031250000000', '390625000000000000', '390625000000000000', '48828125000000000', '95367431640625', '24414062500000000', '390625000000000000', '781250000000000000', '156250000000000000', '97656250000000000', '312500000000000000', '156250000000000000', '1250000000000000000', '12500000000000000000', '25000000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '160000000000000000000', '840000000000000000000', '900000000000000000000', '900000000000000000000', '360000000000000000000', '840000000000000000000', '800000000000000000000', '6250000000000000000', '800000000000000000000', '16800000000000000000000', '43750000000000000000', '16406250000000000000', '840000000000000000000', '4200000000000000000000', '16800000000000000000000', '800000000000000000000', '16800000000000000000000', '5600000000000000000000', '12600000000000000000000', '6300000000000000000000', '6300000000000000000000', '16800000000000000000000', '16800000000000000000000', '50400000000000000000000', '50400000000000000000000', '1050000000000000000000', '700000000000000000000', '10080000000000000000000', '10080000000000000000000', '50400000000000000000000', '50400000000000000000000', '600000000000000000000', '50400000000000000000000', '50400000000000000000000', '50400000000000000000000', '50400000000000000000000', '7200000000000000000000', '403200000000000000000', '800000000000000000000', '1400000000000000000000', '5600000000000000000000', '400000000000000000000', '400000000000000000000', '50000000000000000000', '100000000000000000000', '20000000000000000000', '6250000000000000000', '1562500000000000000', '97656250000000000', '781250000000000000', '6250000000000000000', '10000000000000000000', '50000000000000000000', '50000000000000000000', '2500000000000000', '195312500000000000', '97656250000000000', '9765625000000000', '19531250000000000', '12500000000000000000', '12500000000000000000', '4882812500000000', '781250000000000000', '97656250000000000', '9765625000000000', '48828125000000000', '781250000000000000', '2500000000000000000', '1562500000000000000', '97656250000000000', '100000000000000000000', '48828125000000000', '97656250000000000', '781250000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '390625000000000000', '97656250000000000', '390625000000000000', '195312500000000000', '1562500000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '100000000000000000000', '800000000000000000', '100000000000000000000', '100000000000000000000', '1600000000000000000000', '16800000000000000000000', '5600000000000000000000', '16800000000000000000000', '672000000000000000000', '3360000000000000000000', '1600000000000000000000', '50000000000000000000', '1600000000000000000000', '33600000000000000000000', '10080000000000000000000', '10080000000000000000000', '10080000000000000000000', '3360000000000000000000', '33600000000000000000000', '1600000000000000000000', '4800000000000000000000', '100800000000000000000000', '787500000000000000000', '393750000000000000000', '49218750000000000000', '33600000000000000000000', '4800000000000000000000', '100800000000000000000000', '100800000000000000000000', '2100000000000000000000', '787500000000000000000', '100800000000000000000000', '11200000000000000000000', '100800000000000000000000', '20160000000000000000000', '787500000000000000000', '20160000000000000000000', '33600000000000000000000', '100800000000000000000000', '100800000000000000000000', '100800000000000000000000', '33600000000000000000000', '33600000000000000000000', '16800000000000000000000', '33600000000000000000000', '200000000000000000000', '200000000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '781250000000000000', '312500000000000000', '390625000000000000', '1562500000000000000', '2500000000000000000', '6250000000000000000', '100000000000000000000', '25000000000000000000', '500000000000000000', '195312500000000000', '48828125000000000', '190734863281250', '781250000000000000', '3125000000000000000', '100000000000000000000', '12500000000000000000', '156250000000000000', '97656250000000000', '4882812500000000', '48828125000000000', '6250000000000000', '12500000000000000000', '1562500000000000000', '390625000000000', '48828125000000000', '100000000000000000000', '97656250000000000', '1562500000000000000', '390625000000000000', '6103515625000000', '9765625000000000', '48828125000000000', '390625000000000000', '1562500000000000000', '31250000000000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '12500000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '12800000000000000000', '33600000000000000000000', '100800000000000000000000', '100800000000000000000000', '100800000000000000000000', '11200000000000000000000', '32000000000000000000', '100000000000000000000', '320000000000000000000', '33600000000000000000000', '100800000000000000000000', '14400000000000000000000', '2880000000000000000000', '33600000000000000000000', '2800000000000000000000', '800000000000000000000', '8400000000000000000000', '5040000000000000000000', '900000000000000000000', '6300000000000000000000', '600000000000000000000', '33600000000000000000000', '11200000000000000000000', '400000000000000000000', '2800000000000000000000', '2100000000000000000000', '6300000000000000000000', '100800000000000000000000', '576000000000000000000', '1200000000000000000000', '8400000000000000000000', '2520000000000000000000', '100800000000000000000000', '14400000000000000000000', '1200000000000000000000', '3600000000000000000000', '14400000000000000000000', '14400000000000000000000', '8400000000000000000000', '33600000000000000000000', '33600000000000000000000', '1600000000000000000000', '1600000000000000000000', '100000000000000000000', '50000000000000000000', '10000000000000000000', '156250000000000000', '1562500000000000000', '48828125000000000', '1562500000000000000', '12500000000000000000', '5000000000000000000', '100000000000000000000', '50000000000000000000', '6250000000000000000', '156250000000000000', '19531250000000000', '48828125000000000', '781250000000000000', '6250000000000000000', '2000000000000000000', '12500000000000000000', '390625000000000000', '3906250000000000', '1525878906250000', '9765625000000000', '78125000000000000', '1250000000000000000', '1562500000000000000', '97656250000000000', '4882812500000000', '100000000000000000000', '976562500000000', '1562500000000000000', '390625000000000000', '6103515625000000', '97656250000000000', '97656250000000000', '48828125000000000', '1562500000000000000', '78125000000000000', '781250000000000000', '1562500000000000000', '6250000000000000000', '50000000000000000', '195312500000000000', '100000000000000000000', '6250000000000000000', '100000000000000000000', '50000000000000000000', '1600000000000000000000', '4200000000000000000000', '12600000000000000000000', '3076171875000000000', '787500000000000000000', '4200000000000000000000', '1600000000000000000000', '100000000000000000000', '320000000000000000000', '33600000000000000000000', '12600000000000000000000', '12600000000000000000000', '6300000000000000000000', '6300000000000000000000', '33600000000000000000000', '1600000000000000000000', '33600000000000000000000', '100800000000000000000000', '6300000000000000000000', '700000000000000000000', '12600000000000000000000', '100800000000000000000000', '33600000000000000000000', '33600000000000000000000', '100800000000000000000000', '6300000000000000000000', '1260000000000000000000', '100800000000000000000000', '11200000000000000000000', '11200000000000000000000', '100800000000000000000000', '504000000000000000000', '100800000000000000000000', '33600000000000000000000', '33600000000000000000000', '11200000000000000000000', '11200000000000000000000', '33600000000000000000000', '1344000000000000000000', '840000000000000000000', '33600000000000000000000', '800000000000000000000', '800000000000000000000', '100000000000000000000', '1525878906250000', '3051757812500000', '97656250000000000', '1220703125000000', '1525878906250000', '190734863281250', '6103515625000000', '1525878906250000', '1525878906250000', '762939453125000', '1525878906250000', '195312500000000000', '244140625000000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '190734863281250', '190734863281250', '97656250000000000', '1525878906250000', '1525878906250000', '762939453125000', '1525878906250000', '1220703125000000', '762939453125000', '30517578125000', '1525878906250000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '610351562500000', '6103515625000000', '381469726562500', '305175781250000', '3051757812500000', '12207031250000000', '762939453125000', '3051757812500000', '762939453125000', '610351562500000', '3051757812500000', '1525878906250000', '1525878906250000', '762939453125000', '7812500000000000', '3051757812500000', '1525878906250000', '190734863281250', '3051757812500000', '1525878906250000', '95367431640625', '1525878906250000', '762939453125000', '610351562500000', '97656250000000000', '3051757812500000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '305175781250000', '190734863281250', '305175781250000', '195312500000000000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '762939453125000', '38146972656250', '762939453125000', '3051757812500000', '19531250000000000', '6103515625000000', '305175781250000', '3051757812500000', '1525878906250000', '3051757812500000', '76293945312500', '190734863281250', '190734863281250', '3051757812500000', '97656250000000000', '6103515625000000', '3051757812500000', '3051757812500000', '305175781250000', '3051757812500000', '762939453125000', '381469726562500', '3051757812500000', '19531250000000000', '6103515625000000', '381469726562500', '1525878906250000', '190734863281250', '3051757812500000', '381469726562500', '1525878906250000', '1525878906250000', '12207031250000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '762939453125000', '1525878906250000', '762939453125000', '1525878906250000', '3051757812500000', '1953125000000000', '1525878906250000', '610351562500000', '3051757812500000', '3051757812500000', '762939453125000', '1525878906250000', '305175781250000', '122070312500000', '12207031250000000', '3051757812500000', '3051757812500000', '610351562500000', '6103515625000000', '381469726562500', '762939453125000', '1525878906250000', '762939453125000', '97656250000000000', '3051757812500000', '3051757812500000', '3051757812500000', '610351562500000', '381469726562500', '305175781250000', '381469726562500', '381469726562500', '3051757812500000', '1953125000000000', '6103515625000000', '3051757812500000', '1525878906250000', '762939453125000', '244140625000000', '1525878906250000', '1525878906250000', '122070312500000', '195312500000000000', '1220703125000000', '1525878906250000', '1525878906250000', '3051757812500000', '610351562500000', '1525878906250000', '305175781250000', '1525878906250000', '195312500000000000', '6103515625000000', '305175781250000', '1525878906250000', '610351562500000', '6103515625000000', '152587890625000', '1525878906250000', '1', '4', '4', '4', '4', '781250000000000000', '195312500000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '976562500000000', '781250000000000000', '195312500000000000', '195312500000000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '4882812500000000', '48828125000000000', '4882812500000000', '4882812500000000', '195312500000000000', '48828125000000000', '48828125000000000', '195312500000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '1562500000000000000', '97656250000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '97656250000000000', '3125000000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '97656250000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '39062500000000000', '6103515625000000', '781250000000000000', '6103515625000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '6103515625000000', '781250000000000000', '781250000000000000', '781250000000000000', '31250000000000000', '9765625000000000', '39062500000000000', '4882812500000000', '31250000000000000', '4882812500000000', '4882812500000000', '31250000000000000', '9765625000000000', '31250000000000000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1', '24414062500000000', '24414062500000000', '24414062500000000', '24414062500000000', '312500000000000000', '12207031250000000', '781250000000000000', '1562500000000000000', '312500000000000000', '12207031250000000', '781250000000000000', '1562500000000000000', '6250000000000000000', '6103515625000000', '6250000000000000000', '781250000000000000', '6250000000000000000', '6103515625000000', '6250000000000000000', '781250000000000000', '6250000000000000000', '1250000000000000000', '12207031250000000', '25000000000000000', '6250000000000000000', '1250000000000000000', '12207031250000000', '25000000000000000', '12500000000000000000', '3125000000000000000', '12207031250000000', '2500000000000000000', '12500000000000000000', '3125000000000000000', '12207031250000000', '2500000000000000000', '12500000000000000000', '12207031250000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12207031250000000', '25000000000000000000', '25000000000000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '10000000000000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '10000000000000000000', '100000000000000000000', '100000000000000000000', '20000000000000000000', '12500000000000000000', '25000000000000000000', '100000000000000000000', '1525878906250000', '1525878906250000', '6103515625000000', '6103515625000000', '61035156250000', '381469726562500', '1525878906250000', '95367431640625', '6103515625000000', '6103515625000000', '6103515625000000', '190734863281250', '305175781250000', '762939453125000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '190734863281250', '3051757812500000', '762939453125000', '610351562500000', '610351562500000', '610351562500000', '3051757812500000', '1525878906250000', '1525878906250000', '61035156250000', '381469726562500', '1220703125000000', '3051757812500000', '9765625000000', '1220703125000000', '762939453125000', '1525878906250000', '1525878906250000', '610351562500000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '762939453125000', '762939453125000', '305175781250000', '97656250000000000', '97656250000000000', '7812500000000000', '390625000000000000', '195312500000000000', '390625000000000000', '1000000000000000000', '100000000000000000000', '24414062500000000', '381469726562500', '6103515625000000', '20000000000000000000', '1000000000000000000', '100000000000000000000', '24414062500000000', '381469726562500', '6103515625000000', '20000000000000000000', '1000000000000000000', '3051757812500000', '48828125000000000', '2441406250000000', '25000000000000000000', '500000000000000000', '1000000000000000000', '3051757812500000', '48828125000000000', '2441406250000000', '25000000000000000000', '500000000000000000', '20000000000000000000', '12207031250000000', '24414062500000000', '6103515625000000', '6250000000000000000', '100000000000000000000', '20000000000000000000', '12207031250000000', '24414062500000000', '6103515625000000', '6250000000000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '12207031250000000', '100000000000000000000', '6250000000000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '12207031250000000', '100000000000000000000', '6250000000000000000', '50000000000000000000', '3051757812500000', '381469726562500', '12207031250000000', '2000000000000000000', '25000000000000000000', '50000000000000000000', '3051757812500000', '381469726562500', '12207031250000000', '2000000000000000000', '25000000000000000000', '6250000000000000000', '12207031250000000', '24414062500000000', '48828125000000000', '25000000000000000000', '25000000000000000000', '6250000000000000000', '12207031250000000', '24414062500000000', '48828125000000000', '25000000000000000000', '25000000000000000000', '1250000000000000000', '100000000000000000000', '25000000000000000000', '25000000000000000000', '100000000000000000000', '2500000000000000000', '381469726562500', '3051757812500000', '244140625000000', '610351562500000', '6103515625000000', '762939453125000', '762939453125000', '95367431640625', '381469726562500', '76293945312500', '6103515625000000', '762939453125000', '1220703125000000', '4882812500000000', '244140625000000', '1525878906250000', '1525878906250000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '24414062500000000', '48828125000000', '190734863281250', '152587890625000', '190734863281250', '762939453125000', '3051757812500000', '305175781250000', '6103515625000000', '12207031250000', '76293945312500', '76293945312500', '95367431640625', '381469726562500', '762939453125000', '1525878906250000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '24414062500000000', '1525878906250000', '762939453125000', '1525878906250000', '1525878906250000', '1525878906250000', '610351562500000', '6103515625000000', '381469726562500', '3051757812500000', '2441406250000000', '76293945312500', '305175781250000', '3051757812500000', '1525878906250000', '95367431640625', '122070312500000', '78125000000000000', '100000000000000000000', '97656250000000000', '390625000000000000', '195312500000000000', '97656250000000000', '24414062500000000', '12500000000000000000', '48828125000000000', '97656250000000000', '25000000000000000000', '97656250000000000', '19531250000000000', '2441406250000000', '195312500000000000', '48828125000000000', '78125000000000000', '39062500000000000', '390625000000000000', '48828125000000000', '100000000000000000000', '390625000000000000', '97656250000000000', '2441406250000000', '97656250000000000', '100000000000000000000', '390625000000000000', '48828125000000000', '100000000000000000000', '390625000000000000', '97656250000000000', '2441406250000000', '97656250000000000', '100000000000000000000', '25000000000000000000', '9765625000000000', '12207031250000000', '78125000000000000', '390625000000000000', '50000000000000000000', '97656250000000000', '50000000000000000000', '25000000000000000000', '9765625000000000', '12207031250000000', '78125000000000000', '390625000000000000', '50000000000000000000', '97656250000000000', '50000000000000000000', '25000000000000000000', '97656250000000000', '1525878906250000', '9765625000000000', '19531250000000000', '12500000000000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '1525878906250000', '9765625000000000', '19531250000000000', '12500000000000000000', '97656250000000000', '25000000000000000000', '100000000000000000000', '19531250000000000', '100000000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '19531250000000000', '50000000000000000000', '100000000000000000000', '19531250000000000', '100000000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '19531250000000000', '50000000000000000000', '25000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '39062500000000000', '625000000000000000', '24414062500000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '39062500000000000', '625000000000000000', '24414062500000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '50000000000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '48828125000000000', '10000000000000000000', '25000000000000000000', '97656250000000000', '50000000000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '48828125000000000', '10000000000000000000', '50000000000000000000', '50000000000000000000', '100000000000000000000', '25000000000000000000', '12500000000000000000', '100000000000000000000', '3051757812500000', '1220703125000000', '6103515625000000', '3051757812500000', '2441406250000000', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '1953125000000', '1525878906250000', '610351562500000', '6103515625000000', '3051757812500000', '61035156250000', '6103515625000000', '976562500000000', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '610351562500000', '1525878906250000', '3051757812500000', '610351562500000', '6103515625000000', '6103515625000000', '4882812500000000', '24414062500000000', '3051757812500000', '1525878906250000', '381469726562500', '762939453125000', '6103515625000000', '762939453125000', '15258789062500', '1525878906250000', '244140625000000', '6103515625000000', '976562500000000', '24414062500000000', '3051757812500000', '762939453125000', '3051757812500000', '190734863281250', '2441406250000', '1525878906250000', '381469726562500', '6103515625000000', '6103515625000000', '3051757812500000', '12207031250000000', '6103515625000000', '152587890625000', '1525878906250000', '190734863281250', '1525878906250000', '1525878906250000', '762939453125000', '3051757812500000', '6103515625000000', '190734863281250', '6103515625000000', '1220703125000000', '4882812500000000', '762939453125000', '1525878906250000', '1525878906250000', '190734863281250', '3051757812500000', '305175781250000', '1525878906250000', '195312500000000000', '195312500000000000', '6250000000000000000', '12500000000000000000', '1562500000000000', '195312500000000000', '15625000000000000', '97656250000000000', '1250000000000000000', '195312500000000000', '39062500000000000', '12500000000000000000', '625000000000000000', '12207031250000000', '97656250000000000', '6250000000000000000', '97656250000000000', '48828125000000000', '195312500000000000', '1250000000000000000', '19531250000000000', '25000000000000000000', '25000000000000000000', '9765625000000000', '190734863281250', '195312500000000000', '195312500000000000', '1250000000000000000', '19531250000000000', '25000000000000000000', '25000000000000000000', '9765625000000000', '190734863281250', '195312500000000000', '390625000000000000', '25000000000000000000', '12207031250000000', '12500000000000000000', '25000000000000000000', '12207031250000000', '2441406250000000', '390625000000000000', '390625000000000000', '25000000000000000000', '12207031250000000', '12500000000000000000', '25000000000000000000', '12207031250000000', '2441406250000000', '390625000000000000', '390625000000000000', '3125000000000000000', '390625000000000', '25000000000000000000', '25000000000000000000', '12207031250000000', '2441406250000000', '195312500000000000', '390625000000000000', '3125000000000000000', '390625000000000', '25000000000000000000', '25000000000000000000', '12207031250000000', '2441406250000000', '195312500000000000', '390625000000000000', '12500000000000000000', '24414062500000000', '25000000000000000000', '5000000000000000000', '24414062500000000', '6103515625000000', '195312500000000000', '390625000000000000', '12500000000000000000', '24414062500000000', '25000000000000000000', '5000000000000000000', '24414062500000000', '6103515625000000', '195312500000000000', '48828125000000000', '25000000000000000000', '4882812500000000', '25000000000000000000', '12500000000000000000', '12207031250000000', '12207031250000000', '48828125000000000', '48828125000000000', '25000000000000000000', '4882812500000000', '25000000000000000000', '12500000000000000000', '12207031250000000', '12207031250000000', '48828125000000000', '390625000000000000', '6250000000000000000', '3906250000000000', '25000000000000000000', '25000000000000000000', '48828125000000000', '2441406250000000', '390625000000000000', '390625000000000000', '6250000000000000000', '3906250000000000', '25000000000000000000', '25000000000000000000', '48828125000000000', '2441406250000000', '390625000000000000', '50000000000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '50000000000000000000', '100000000000000000000', '1220703125000000', '1525878906250000', '1525878906250000', '488281250000000', '3051757812500000', '1525878906250000', '762939453125000', '6103515625000000', '6103515625000000', '24414062500000000', '3051757812500000', '762939453125000', '3051757812500000', '244140625000000', '1525878906250000', '3051757812500000', '610351562500000', '6103515625000000', '381469726562500', '6103515625000000', '2441406250000000', '24414062500000000', '1525878906250000', '3051757812500000', '1220703125000000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '381469726562500', '1525878906250000', '1220703125000000', '24414062500000000', '1220703125000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '3051757812500000', '12207031250000000', '1525878906250000', '3051757812500000', '24414062500000000', '61035156250000', '1525878906250000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '12207031250000000', '610351562500000', '1525878906250000', '1525878906250000', '1220703125000000', '24414062500000000', '1525878906250000', '1525878906250000', '1525878906250000', '6103515625000000', '3051757812500000', '381469726562500', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '24414062500000000', '4882812500000000', '6103515625000000', '610351562500000', '195312500000000000', '390625000000000000', '781250000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '39062500000000000', '97656250000000000', '6250000000000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '19531250000000000', '1250000000000000000', '97656250000000000', '6103515625000000', '3125000000000000000', '195312500000000000', '24414062500000000', '1250000000000000', '1562500000000000000', '1562500000000000000', '4882812500000000', '24414062500000000', '24414062500000000', '1250000000000000', '1562500000000000000', '1562500000000000000', '4882812500000000', '24414062500000000', '48828125000000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '24414062500000000', '48828125000000000', '610351562500000', '1562500000000000000', '781250000000000000', '1562500000000000000', '610351562500000', '610351562500000', '610351562500000', '1562500000000000000', '781250000000000000', '1562500000000000000', '610351562500000', '610351562500000', '48828125000000000', '12207031250000000', '78125000000000000', '78125000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '12207031250000000', '78125000000000000', '78125000000000000', '195312500000000000', '48828125000000000', '12207031250000000', '195312500000000000', '390625000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '12207031250000000', '195312500000000000', '390625000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '48828125000000000', '12207031250000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '48828125000000000', '48828125000000000', '12207031250000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '48828125000000000', '5000000000000000000', '100000000000000000000', '50000000000000000000', '25000000000000000000', '5000000000000000000', '100000000000000000000', '6103515625000000', '12207031250000000', '1525878906250000', '6103515625000000', '6103515625000000', '1525878906250000', '762939453125000', '195312500000000000', '12207031250000000', '610351562500000', '762939453125000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '762939453125000', '3051757812500000', '390625000000000000', '12207031250000000', '1525878906250000', '6103515625000000', '76293945312500', '3051757812500000', '762939453125000', '3051757812500000', '1525878906250000', '12207031250000000', '97656250000000000', '6103515625000000', '1525878906250000', '12207031250000000', '12207031250000000', '3051757812500000', '3051757812500000', '1220703125000000', '3051757812500000', '12207031250000000', '48828125000000000', '2441406250000000', '3051757812500000', '3051757812500000', '12207031250000000', '1525878906250000', '6103515625000000', '762939453125000', '3051757812500000', '12207031250000000', '3051757812500000', '6103515625000000', '3051757812500000', '12207031250000000', '12207031250000000', '610351562500000', '1525878906250000', '3051757812500000', '1525878906250000', '12207031250000000', '390625000000000000', '2441406250000000', '3051757812500000', '19531250000000000', '195312500000000000', '3125000000000000000', '48828125000000000', '50000000000000000000', '195312500000000000', '19531250000000000', '78125000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '195312500000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '97656250000000000', '12207031250000000', '4882812500000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '4882812500000000', '3051757812500000', '48828125000000000', '48828125000000000', '24414062500000000', '6103515625000000', '48828125000000000', '48828125000000000', '24414062500000000', '6103515625000000', '1220703125000000', '48828125000000000', '48828125000000000', '24414062500000000', '1220703125000000', '48828125000000000', '48828125000000000', '24414062500000000', '24414062500000000', '48828125000000000', '48828125000000000', '6103515625000000', '24414062500000000', '48828125000000000', '48828125000000000', '6103515625000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '195312500000000000', '3125000000000000000', '24414062500000000', '6250000000000000000', '250000000000000000', '781250000000000000', '24414062500000000', '48828125000000000', '12207031250000000', '1220703125000000', '12207031250000000', '1220703125000000', '4882812500000000', '610351562500000', '9765625000000000', '9765625000000000', '6103515625000000', '152587890625000', '6103515625000000', '9765625000000000', '48828125000000000', '3051757812500000', '24414062500000000', '2441406250000000', '12207031250000000', '6103515625000000', '12207031250000000', '12207031250000000', '2441406250000000', '95367431640625', '24414062500000000', '24414062500000000', '12207031250000000', '1220703125000000', '12207031250000000', '2441406250000000', '24414062500000000', '1525878906250000', '48828125000000000', '48828125000000000', '95367431640625', '38146972656250', '95367431640625', '24414062500000000', '48828125000000000', '24414062500000000', '48828125000000000', '48828125000000000', '12207031250000000', '6103515625000000', '6103515625000000', '48828125000000000', '24414062500000000', '1525878906250000', '390625000000000000', '195312500000000000', '97656250000000000', '390625000000000000', '97656250000000000', '195312500000000000', '4', '4', '4', '4', '31250000000000000', '195312500000000000', '31250000000000000', '2441406250000000', '2441406250000000', '31250000000000000', '2441406250000000', '2441406250000000', '24414062500000000', '31250000000000000', '12500000000000000000', '24414062500000000', '24414062500000000', '3125000000000000000', '12207031250000000', '12500000000000000000', '12500000000000000000', '24414062500000000', '24414062500000000', '3125000000000000000', '12207031250000000', '12500000000000000000', '20000000000000000000', '24414062500000000', '1525878906250000', '20000000000000000000', '24414062500000000', '50000000000000000000', '20000000000000000000', '24414062500000000', '1525878906250000', '20000000000000000000', '24414062500000000', '50000000000000000000', '100000000000000000000', '9765625000000000', '25000000000000000000', '3051757812500000', '24414062500000000', '100000000000000000000', '100000000000000000000', '9765625000000000', '25000000000000000000', '3051757812500000', '24414062500000000', '100000000000000000000', '625000000000000000', '24414062500000000', '24414062500000000', '25000000000000000000', '12207031250000000', '25000000000000000000', '625000000000000000', '24414062500000000', '24414062500000000', '25000000000000000000', '12207031250000000', '25000000000000000000', '12500000000000000000', '488281250000000', '3051757812500000', '25000000000000000000', '610351562500000', '25000000000000000000', '12500000000000000000', '488281250000000', '3051757812500000', '25000000000000000000', '610351562500000', '25000000000000000000', '50000000000000000000', '9765625000000000', '24414062500000000', '50000000000000000000', '48828125000000000', '25000000000000000000', '50000000000000000000', '9765625000000000', '24414062500000000', '50000000000000000000', '48828125000000000', '25000000000000000000', '12500000000000000000', '100000000000000000000', '25000000000000000000', '10000000000000000000', '100000000000000000000', '50000000000000000000', '244140625000000', '3051757812500000', '95367431640625', '6103515625000000', '6103515625000000', '76293945312500', '1525878906250000', '381469726562500', '381469726562500', '3051757812500000', '3051757812500000', '610351562500000', '6103515625000000', '381469726562500', '3051757812500000', '190734863281250', '152587890625000', '762939453125000', '1525878906250000', '762939453125000', '1220703125000000', '3051757812500000', '6103515625000000', '6103515625000000', '152587890625000', '381469726562500', '1525878906250000', '762939453125000', '762939453125000', '3051757812500000', '6103515625000000', '12207031250000000', '1220703125000000', '6103515625000000', '95367431640625', '190734863281250', '95367431640625', '38146972656250', '381469726562500', '381469726562500', '762939453125000', '24414062500000000', '762939453125000', '762939453125000', '762939453125000', '305175781250000', '3051757812500000', '3051757812500000', '762939453125000', '1525878906250000', '6103515625000000', '2441406250000000', '1525878906250000', '1525878906250000', '1525878906250000', '152587890625000', '190734863281250', '152587890625000', '762939453125000', '1525878906250000', '195312500000000000', '48828125000000000', '50000000000000000000', '195312500000000000', '6103515625000000', '12500000000000000000', '39062500000000000', '19531250000000000', '100000000000000000000', '390625000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '19531250000000000', '48828125000000000', '390625000000000000', '97656250000000000', '390625000000000000', '50000000000000000000', '195312500000000000', '1953125000000000', '48828125000000000', '100000000000000000000', '48828125000000000', '1953125000000000', '195312500000000000', '100000000000000000000', '50000000000000000000', '585937500000000000', '146484375000000000', '146484375000000000', '100000000000000000000', '29296875000000000', '146484375000000000', '585937500000000000', '100000000000000000000', '20000000000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '50000000000000000000', '97656250000000000', '2441406250000000', '195312500000000000', '20000000000000000000', '20000000000000000000', '117187500000000000', '292968750000000000', '3662109375000000', '50000000000000000000', '146484375000000000', '292968750000000000', '585937500000000000', '20000000000000000000', '50000000000000000000', '97656250000000000', '48828125000000000', '39062500000000000', '10000000000000000000', '39062500000000000', '48828125000000000', '97656250000000000', '25000000000000000000', '50000000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '10000000000000000000', '390625000000000000', '390625000000000000', '78125000000000000', '25000000000000000000', '100000000000000000000', '97656250000000000', '6103515625000000', '195312500000000000', '100000000000000000000', '195312500000000000', '6103515625000000', '39062500000000000', '10000000000000000000', '100000000000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '100000000000000000000', '195312500000000000', '97656250000000000', '195312500000000000', '10000000000000000000', '50000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '100000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '20000000000000000000', '50000000000000000000', '585937500000000000', '292968750000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '292968750000000000', '585937500000000000', '20000000000000000000', '50000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '20000000000000000000', '97656250000000000', '48828125000000000', '195312500000000000', '100000000000000000000', '50000000000000000000', '97656250000000000', '195312500000000000', '1525878906250000', '20000000000000000000', '195312500000000000', '195312500000000000', '48828125000000000', '100000000000000000000', '6250000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '20000000000000000000', '6103515625000000', '12207031250000000', '1220703125000000', '1220703125000000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '38146972656250', '48828125000000', '762939453125000', '3051757812500000', '1220703125000000', '6103515625000000', '610351562500000', '1525878906250000', '12207031250000000', '3051757812500000', '76293945312500', '1525878906250000', '152587890625000', '1220703125000000', '1525878906250000', '1525878906250000', '6103515625000000', '12207031250000000', '3051757812500000', '610351562500000', '24414062500000000', '6103515625000000', '762939453125000', '1525878906250000', '1525878906250000', '6103515625000000', '152587890625000', '762939453125000', '6103515625000000', '12207031250000000', '6103515625000000', '6103515625000000', '12207031250000000', '6103515625000000', '381469726562500', '6103515625000000', '1525878906250000', '6103515625000000', '762939453125000', '3051757812500000', '6103515625000000', '1220703125000000', '1525878906250000', '95367431640625', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '95367431640625', '6103515625000000', '305175781250000', '762939453125000', '3051757812500000', '2441406250000000', '6103515625000000', '6103515625000000', '6103515625000000', '305175781250000', '61035156250000', '1525878906250000', '95367431640625', '1220703125000000', '190734863281250', '381469726562500', '195312500000000000', '19531250000000000', '48828125000000000', '97656250000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '390625000000000000', '48828125000000000', '195312500000000000', '100000000000000000000', '390625000000000000', '38146972656250', '97656250000000000', '48828125000000000', '97656250000000000', '97656250000000000', '6250000000000000000', '48828125000000000', '100000000000000000000', '97656250000000000', '48828125000000000', '6103515625000000', '390625000000000000', '1562500000000000000', '50000000000000000000', '195312500000000000', '195312500000000000', '100000000000000000000', '39062500000000000', '390625000000000000', '50000000000000000000', '97656250000000000', '12207031250000000', '100000000000000000000', '39062500000000000', '390625000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '24414062500000000', '195312500000000000', '781250000000000000', '781250000000000000', '2441406250000000', '390625000000000000', '312500000000000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '62500000000000000', '1562500000000000000', '312500000000000000', '62500000000000000', '156250000000000000', '1562500000000000000', '781250000000000000', '48828125000000000', '97656250000000000', '312500000000000000', '3125000000000000', '24414062500000000', '312500000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '3125000000000000000', '31250000000000000', '3125000000000000000', '625000000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '31250000000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '3125000000000000000', '24414062500000000', '1562500000000000000', '3125000000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '1000000000000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '19531250000000000', '625000000000000000', '2441406250000000', '1562500000000000000', '3125000000000000000', '19531250000000000', '12207031250000000', '97656250000000000', '25000000000000000000', '2500000000000000000', '5000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '781250000000000000', '24414062500000000', '24414062500000000', '3125000000000000000', '1562500000000000000', '24414062500000000', '3125000000000000000', '24414062500000000', '24414062500000000', '781250000000000000', '2500000000000000000', '3125000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '3125000000000000000', '12500000000000000000', '390625000000000000', '3051757812500000', '781250000000000000', '3125000000000000000', '1562500000000000000', '6103515625000000', '3125000000000000000', '781250000000000000', '3051757812500000', '390625000000000000', '6250000000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '3125000000000000000', '12500000000000000000', '50000000000000000000', '390625000000000000', '100000000000000000000', '20000000000000000000', '25000000000000000000', '1525878906250000', '78125000000000', '244140625000000', '3051757812500000', '6103515625000000', '76293945312500', '762939453125000', '3051757812500000', '762939453125000', '1220703125000000', '762939453125000', '190734863281250', '3051757812500000', '24414062500000000', '1525878906250000', '1525878906250000', '762939453125000', '24414062500000', '610351562500000', '305175781250000', '152587890625000', '3051757812500000', '610351562500000', '3051757812500000', '1220703125000000', '48828125000000000', '3051757812500000', '3051757812500000', '762939453125000', '610351562500000', '3051757812500000', '76293945312500', '6103515625000000', '762939453125000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '762939453125000', '762939453125000', '305175781250000', '95367431640625', '1525878906250000', '152587890625000', '6103515625000000', '3051757812500000', '76293945312500', '305175781250000', '6103515625000000', '12207031250000000', '610351562500000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '244140625000000', '6103515625000000', '610351562500000', '1525878906250000', '610351562500000', '12207031250000000', '1525878906250000', '762939453125000', '610351562500000', '1220703125000000', '3051757812500000', '3051757812500000', '610351562500000', '6103515625000000', '3051757812500000', '1525878906250000', '19531250000000000', '19531250000000000', '78125000000000000', '97656250000000000', '100000000000000000000', '97656250000000000', '48828125000000000', '19531250000000000', '97656250000000000', '97656250000000000', '390625000000000000', '250000000000000000', '12207031250000000', '97656250000000000', '390625000000000000', '97656250000000000', '5000000000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '390625000000000', '97656250000000000', '156250000000000000', '3906250000000000', '97656250000000000', '19531250000000000', '390625000000000000', '19531250000000000', '39062500000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '195312500000000000', '97656250000000000', '195312500000000000', '156250000000000000', '24414062500000000', '2441406250000000', '25000000000000000000', '48828125000000000', '12500000000000000000', '48828125000000000', '1000000000000000000', '2441406250000000', '24414062500000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '1000000000000000000', '1171875000000000000', '1171875000000000000', '4882812500000000', '97656250000000000', '25000000000000000000', '7812500000000000', '12500000000000000000', '7812500000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '390625000000000000', '12500000000000000000', '390625000000000000', '25000000000000000000', '390625000000000000', '78125000000000000', '48828125000000000', '39062500000000000', '2500000000000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '2500000000000000000', '39062500000000000', '48828125000000000', '585937500000000000', '58593750000000000', '2500000000000000000', '73242187500000000', '6250000000000000000', '292968750000000000', '2500000000000000000', '7324218750000000', '585937500000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '19531250000000000', '6250000000000000000', '19531250000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '234375000000000000', '6250000000000000000', '234375000000000000', '12500000000000000000', '1171875000000000000', '1171875000000000000', '12207031250000000', '97656250000000000', '25000000000000000000', '39062500000000000', '6250000000000000000', '39062500000000000', '25000000000000000000', '97656250000000000', '12207031250000000', '48828125000000000', '19531250000000000', '25000000000000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '25000000000000000000', '97656250000000000', '9765625000000000', '48828125000000000', '9765625000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '97656250000000000', '12500000000000000000', '9765625000000000', '48828125000000000', '390625000000000000', '78125000000000000', '12500000000000000000', '390625000000000000', '6250000000000000000', '390625000000000000', '12500000000000000000', '78125000000000000', '390625000000000000', '6250000000000000000', '50000000000000000000', '50000000000000000000', '1562500000000000000', '1562500000000000000', '6250000000000000000', '3051757812500000', '4882812500000000', '3051757812500000', '762939453125000', '3051757812500000', '3051757812500000', '3051757812500000', '95367431640625', '6103515625000000', '24414062500000000', '3051757812500000', '3051757812500000', '3051757812500000', '12207031250000000', '1220703125000000', '305175781250000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '95367431640625', '24414062500000000', '6103515625000000', '381469726562500', '190734863281250', '24414062500000000', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '95367431640625', '3051757812500000', '1525878906250000', '6103515625000000', '24414062500000000', '610351562500000', '381469726562500', '3051757812500000', '762939453125000', '3051757812500000', '762939453125000', '1220703125000000', '6103515625000000', '6103515625000000', '3051757812500000', '1220703125000000', '24414062500000000', '3051757812500000', '762939453125000', '1220703125000000', '1525878906250000', '6103515625000000', '762939453125000', '3051757812500000', '24414062500000000', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '2441406250000', '3051757812500000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '2441406250000000', '6103515625000000', '3051757812500000', '195312500000000000', '2441406250000000', '48828125000000000', '97656250000000000', '100000000000000000000', '195312500000000000', '3051757812500000', '48828125000000000', '78125000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '97656250000000000', '39062500000000000', '12500000000000000000', '100000000000000000000', '1250000000000000000', '78125000000000000', '195312500000000000', '97656250000000000', '50000000000000000000', '12500000000000000000', '100000000000000000000', '78125000000000000', '48828125000000000', '97656250000000000', '48828125000000000', '195312500000000000', '48828125000000000', '1562500000000000000', '19531250000000000', '9765625000000000', '12500000000000000000', '195312500000000000', '195312500000000000', '12207031250000000', '12207031250000000', '78125000000000000', '781250000000000000', '6103515625000000', '781250000000000000', '12207031250000000', '12207031250000000', '78125000000000000', '781250000000000000', '6103515625000000', '781250000000000000', '12207031250000000', '48828125000000000', '312500000000000000', '781250000000000000', '24414062500000000', '312500000000000000', '12207031250000000', '48828125000000000', '312500000000000000', '781250000000000000', '24414062500000000', '312500000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '781250000000000000', '12207031250000000', '781250000000000000', '48828125000000000', '48828125000000000', '390625000000000000', '781250000000000000', '12207031250000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '2441406250000000', '390625000000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '2441406250000000', '390625000000000000', '781250000000000000', '48828125000000000', '9765625000000000', '1562500000000000000', '781250000000000000', '4882812500000000', '1562500000000000000', '9765625000000000', '9765625000000000', '1562500000000000000', '781250000000000000', '4882812500000000', '1562500000000000000', '9765625000000000', '1953125000000000', '1562500000000000000', '781250000000000000', '24414062500000000', '1562500000000000000', '1953125000000000', '1953125000000000', '1562500000000000000', '781250000000000000', '24414062500000000', '1562500000000000000', '1953125000000000', '50000000000000000000', '6250000000000000000', '100000000000000000000', '1000000000000000000', '100000000000000000000', '50000000000000000000', '6103515625000000', '390625000000000000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '95367431640625', '12207031250000000', '12207031250000000', '6103515625000000', '12207031250000000', '195312500000000000', '12207031250000000', '1525878906250000', '381469726562500', '381469726562500', '381469726562500', '12207031250000000', '3051757812500000', '305175781250000', '12207031250000000', '48828125000000000', '2441406250000000', '1525878906250000', '1525878906250000', '1525878906250000', '3051757812500000', '3051757812500000', '12207031250000000', '2441406250000', '1220703125000000', '390625000000000000', '12207031250000000', '3051757812500000', '762939453125000', '305175781250000', '762939453125000', '95367431640625', '12207031250000000', '3051757812500000', '1525878906250000', '390625000000000000', '2441406250000000', '3051757812500000', '6103515625000000', '6103515625000000', '95367431640625', '1525878906250000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '152587890625000', '610351562500000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '12207031250000000', '6103515625000000', '19531250000000000', '390625000000000000', '1562500000000000000', '6103515625000000', '390625000000000000', '25000000000000000000', '195312500000000000', '3125000000000000000', '390625000000000000', '762939453125000', '3125000000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '100000000000000000000', '97656250000000000', '625000000000000000', '390625000000000000', '781250000000000000', '195312500000000000', '781250000000000000', '9765625000000000', '9765625000000000', '781250000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '781250000000000000', '9765625000000000', '9765625000000000', '156250000000000000', '97656250000000000', '156250000000000000', '156250000000000000', '156250000000000000', '156250000000000000', '24414062500000000', '156250000000000000', '24414062500000000', '24414062500000000', '9765625000000000', '156250000000000000', '156250000000000000', '9765625000000000', '20000000000000000000', '48828125000000000', '97656250000000000', '5000000000000000000', '12207031250000000', '48828125000000000', '39062500000000000', '100000000000000000000', '20000000000000000000', '48828125000000000', '97656250000000000', '5000000000000000000', '12207031250000000', '48828125000000000', '39062500000000000', '100000000000000000000', '50000000000000000000', '195312500000000000', '97656250000000000', '12207031250000000', '50000000000000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '50000000000000000000', '195312500000000000', '97656250000000000', '12207031250000000', '50000000000000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '12207031250000000', '9765625000000000', '19531250000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '12207031250000000', '9765625000000000', '19531250000000000', '25000000000000000000', '12500000000000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '6103515625000000', '97656250000000000', '390625000000000000', '25000000000000000000', '12500000000000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '6103515625000000', '97656250000000000', '390625000000000000', '25000000000000000000', '50000000000000000000', '195312500000000000', '12207031250000000', '50000000000000000000', '1525878906250000', '97656250000000000', '24414062500000000', '6250000000000000000', '50000000000000000000', '195312500000000000', '12207031250000000', '50000000000000000000', '1525878906250000', '97656250000000000', '24414062500000000', '6250000000000000000', '12500000000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '9765625000000', '48828125000000000', '78125000000000000', '12500000000000000000', '12500000000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '9765625000000', '48828125000000000', '78125000000000000', '12500000000000000000', '25000000000000000000', '2500000000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '76293945312500', '12207031250000000', '12207031250000000', '3051757812500000', '3051757812500000', '3051757812500000', '762939453125000', '6103515625000000', '1525878906250000', '3051757812500000', '381469726562500', '3051757812500000', '6103515625000000', '2441406250000000', '24414062500000000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '381469726562500', '6103515625000000', '1525878906250000', '610351562500000', '6103515625000000', '381469726562500', '12207031250000000', '1525878906250000', '6103515625000000', '6103515625000000', '305175781250000', '762939453125000', '95367431640625', '610351562500000', '1525878906250000', '381469726562500', '48828125000000', '76293945312500', '12207031250000000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '19073486328125', '6103515625000000', '976562500000000', '12207031250000000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '1220703125000000', '381469726562500', '3051757812500000', '1525878906250000', '305175781250000', '610351562500000', '24414062500000000', '12207031250000000', '1525878906250000', '6103515625000000', '381469726562500', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '305175781250000', '3051757812500000', '390625000000000000', '195312500000000000', '195312500000000000', '390625000000000000', '312500000000000000', '195312500000000000', '390625000000000000', '12500000000000000000', '195312500000000000', '12500000000000000000', '7812500000000000', '195312500000000000', '1562500000000000000', '195312500000000000', '9765625000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '12500000000000000', '9765625000000000', '48828125000000000', '156250000000000000', '6103515625000000', '6103515625000000', '12207031250000000', '48828125000000000', '390625000000000000', '12500000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '6250000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '3125000000000000000', '781250000000000000', '781250000000000000', '3051757812500000', '24414062500000000', '3051757812500000', '1562500000000000000', '781250000000000000', '39062500000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '78125000000000000', '781250000000000000', '195312500000000000', '24414062500000000', '48828125000000000', '48828125000000000', '1562500000000000000', '195312500000000000', '390625000000000000', '1562500000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '3125000000000000000', '390625000000000000', '9765625000000000', '625000000000000000', '12207031250000000', '48828125000000000', '24414062500000000', '781250000000000000', '390625000000000000', '6250000000000000', '6250000000000000000', '250000000000000000', '6250000000000000000', '1250000000000000000', '195312500000000000', '195312500000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '1250000000000000000', '3125000000000000000', '97656250000000000', '781250000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '1562500000000000000', '97656250000000000', '195312500000000000', '3125000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '78125000000000000', '781250000000000000', '1562500000000000000', '9765625000000000', '9765625000000000', '24414062500000000', '390625000000000000', '78125000000000000', '3125000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '50000000000000000000', '48828125000000000', '1000000000000000000', '50000000000000000000', '2500000000000000000', '5000000000000000000', '1525878906250000', '1220703125000000', '1525878906250000', '190734863281250', '1220703125000000', '24414062500000000', '6103515625000000', '1525878906250000', '6103515625000000', '3051757812500000', '381469726562500', '3051757812500000', '190734863281250', '6103515625000000', '6103515625000000', '3051757812500000', '610351562500000', '48828125000000000', '244140625000000', '1525878906250000', '19073486328125', '1525878906250000', '3051757812500000', '3051757812500000', '610351562500000', '95367431640625', '6103515625000000', '305175781250000', '3051757812500000', '9765625000000000', '1525878906250000', '61035156250000', '1220703125000000', '1525878906250000', '1525878906250000', '95367431640625', '1525878906250000', '3051757812500000', '1525878906250000', '38146972656250', '1220703125000000', '24414062500000000', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '762939453125000', '95367431640625', '61035156250000', '381469726562500', '6103515625000000', '762939453125000', '3051757812500000', '24414062500000000', '1525878906250000', '3051757812500000', '1220703125000000', '1525878906250000', '1525878906250000', '3051757812500000', '1525878906250000', '1220703125000000', '6103515625000000', '1525878906250000', '6103515625000000', '48828125000000000', '6103515625000000', '610351562500000', '1220703125000000', '6103515625000000', '61035156250000', '381469726562500', '48828125000000000', '48828125000000000', '100000000000000000000', '195312500000000000', '390625000000000000', '48828125000000000', '7812500000000000', '2441406250000000', '781250000000000000', '390625000000000000', '78125000000000000', '19531250000000000', '97656250000000000', '24414062500000000', '50000000000000000000', '390625000000000000', '390625000000000000', '97656250000000000', '195312500000000000', '97656250000000000', '78125000000000000', '3125000000000000000', '4882812500000000', '19531250000000000', '19531250000000000', '48828125000000000', '390625000000000000', '781250000000000000', '24414062500000000', '610351562500000', '97656250000000000', '19531250000000000', '24414062500000000', '12500000000000000000', '97656250000000000', '6103515625000000', '781250000000000000', '3125000000000000000', '48828125000000000', '3125000000000000000', '781250000000000000', '781250000000000000', '6103515625000000', '152587890625000', '781250000000000000', '97656250000000000', '3125000000000000000', '1562500000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '1562500000000000000', '3125000000000000000', '625000000000000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '390625000000000000', '1525878906250000', '1562500000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '9765625000000000', '1562500000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '12207031250000000', '3125000000000000000', '48828125000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '12207031250000000', '48828125000000000', '3125000000000000000', '12207031250000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '48828125000000000', '195312500000000000', '195312500000000000', '195312500000000000', '24414062500000000', '48828125000000000', '24414062500000000', '390625000000000000', '500000000000000000', '12500000000000000000', '50000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '50000000000000000', '12500000000000000000', '12500000000000000000', '781250000000000000', '625000000000000000', '48828125000000000', '24414062500000000', '390625000000000000', '390625000000000000', '1562500000000000000', '12207031250000000', '625000000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '39062500000000000', '1562500000000000000', '12207031250000000', '781250000000000000', '97656250000000000', '12207031250000000', '24414062500000000', '12207031250000000', '1562500000000000000', '39062500000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '12500000000000000000', '500000000000000000', '2500000000000000000', '12500000000000000000', '6250000000000000000', '12500000000000000000', '12500000000000000000', '1250000000000000000', '97656250000000000', '20000000000000000000', '781250000000000000', '100000000000000000000', '6250000000000000000', '190734863281250', '3051757812500000', '3051757812500000', '3051757812500000', '76293945312500', '610351562500000', '1525878906250000', '762939453125000', '6103515625000000', '4882812500000000', '6103515625000000', '762939453125000', '610351562500000', '1220703125000000', '15258789062500', '3051757812500000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '48828125000000000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '1220703125000000', '1525878906250000', '3051757812500000', '48828125000000000', '6103515625000000', '762939453125000', '381469726562500', '3051757812500000', '6103515625000000', '3051757812500000', '381469726562500', '6103515625000000', '6103515625000000', '762939453125000', '6103515625000000', '48828125000000000', '1525878906250000', '12207031250000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '122070312500000', '1525878906250000', '305175781250000', '381469726562500', '48828125000000000', '95367431640625', '1525878906250000', '762939453125000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '48828125000000000', '6103515625000000', '1525878906250000', '195312500000000000', '390625000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '97656250000000000', '48828125000000000', '390625000000000000', '12207031250000000', '100000000000000000000', '39062500000000000', '97656250000000000', '97656250000000000', '390625000000000000', '48828125000000000', '390625000000000000', '781250000000000000', '48828125000000000', '97656250000000000', '390625000000000000', '48828125000000000', '195312500000000000', '6250000000000000000', '48828125000000000', '48828125000000000', '390625000000000000', '97656250000000000', '5000000000000000000', '195312500000000000', '39062500000000000', '97656250000000000', '195312500000000000', '12207031250000000', '195312500000000000', '3125000000000000000', '97656250000000000', '39062500000000000', '5000000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '5000000000000000000', '78125000000000000', '39062500000000000', '5000000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '5000000000000000000', '78125000000000000', '78125000000000000', '1250000000000000000', '25000000000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '25000000000000000000', '19531250000000000', '78125000000000000', '1250000000000000000', '25000000000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '25000000000000000000', '19531250000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '24414062500000000', '97656250000000000', '762939453125000', '12500000000000000000', '19531250000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '24414062500000000', '97656250000000000', '762939453125000', '12500000000000000000', '19531250000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '48828125000000000', '97656250000000000', '2441406250000000', '12500000000000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '48828125000000000', '97656250000000000', '2441406250000000', '12500000000000000000', '390625000000000000', '7812500000000000', '25000000000000000000', '12207031250000000', '97656250000000000', '97656250000000000', '6250000000000000000', '5000000000000000000', '390625000000000000', '7812500000000000', '25000000000000000000', '12207031250000000', '97656250000000000', '97656250000000000', '6250000000000000000', '5000000000000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '2441406250000000', '24414062500000000', '48828125000000000', '25000000000000000000', '6250000000000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '2441406250000000', '24414062500000000', '48828125000000000', '25000000000000000000', '6250000000000000000', '390625000000000000', '25000000000000000000', '100000000000000000000', '100000000000000000000', '6250000000000000000', '50000000000000000000', '12500000000000000000', '244140625000000', '122070312500000', '24414062500000000', '1525878906250000', '762939453125000', '6103515625000000', '3051757812500000', '1525878906250000', '762939453125000', '6103515625000000', '381469726562500', '610351562500000', '6103515625000000', '1525878906250000', '24414062500000000', '6103515625000000', '381469726562500', '6103515625000000', '1525878906250000', '2441406250000000', '305175781250000', '3051757812500000', '6103515625000000', '610351562500000', '6103515625000000', '488281250000000', '762939453125000', '1525878906250000', '190734863281250', '95367431640625', '3051757812500000', '30517578125000', '762939453125000', '1220703125000000', '6103515625000000', '190734863281250', '6103515625000000', '4882812500000000', '24414062500000000', '6103515625000000', '305175781250000', '3051757812500000', '610351562500000', '3051757812500000', '762939453125000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '24414062500000000', '24414062500000000', '1525878906250000', '762939453125000', '12207031250000000', '1525878906250000', '2441406250000000', '1525878906250000', '6103515625000000', '122070312500000', '3051757812500000', '3051757812500000', '24414062500000000', '6103515625000000', '3051757812500000', '61035156250000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '3051757812500000', '3125000000000000000', '97656250000000000', '390625000000000000', '781250000000000000', '97656250000000000', '39062500000000000', '390625000000000000', '195312500000000000', '12500000000000000000', '390625000000000000', '1562500000000000', '50000000000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '97656250000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '12207031250000000', '781250000000000000', '12207031250000000', '12207031250000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '1562500000000000000', '39062500000000000', '39062500000000000', '1562500000000000000', '19531250000000000', '1562500000000000000', '39062500000000000', '39062500000000000', '19531250000000000', '3125000000000000000', '19531250000000000', '39062500000000000', '39062500000000000', '1562500000000000000', '19531250000000000', '1562500000000000000', '39062500000000000', '39062500000000000', '3125000000000000000', '6250000000000000000', '390625000000000000', '976562500000000', '12500000000000000000', '12207031250000000', '4882812500000000', '78125000000000000', '12500000000000000000', '6250000000000000000', '390625000000000000', '976562500000000', '12500000000000000000', '12207031250000000', '4882812500000000', '78125000000000000', '12500000000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '1220703125000000', '12207031250000000', '195312500000000000', '3125000000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '1220703125000000', '12207031250000000', '195312500000000000', '3125000000000000000', '195312500000000000', '390625000000000000', '48828125000000000', '2441406250000000', '500000000000000000', '19531250000000000', '390625000000000000', '12500000000000000000', '195312500000000000', '390625000000000000', '48828125000000000', '2441406250000000', '500000000000000000', '19531250000000000', '390625000000000000', '12500000000000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '1562500000000000000', '12207031250000000', '97656250000000000', '195312500000000000', '3125000000000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '1562500000000000000', '12207031250000000', '97656250000000000', '195312500000000000', '3125000000000000000', '25000000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '12207031250000000', '97656250000000000', '390625000000000000', '2500000000000000000', '25000000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '12207031250000000', '97656250000000000', '390625000000000000', '2500000000000000000', '6250000000000000000', '97656250000000000', '24414062500000000', '1250000000000000000', '12207031250000000', '48828125000000000', '78125000000000000', '781250000000000000', '6250000000000000000', '97656250000000000', '24414062500000000', '1250000000000000000', '12207031250000000', '48828125000000000', '78125000000000000', '781250000000000000', '20000000000000000000', '100000000000000000000', '5000000000000000000', '50000000000000000000', '25000000000000000000', '25000000000000000000', '610351562500000', '488281250000000', '152587890625000', '12207031250000000', '152587890625000', '1220703125000000', '6103515625000000', '3051757812500000', '24414062500000000', '24414062500000000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '305175781250000', '12207031250000000', '610351562500000', '6103515625000000', '762939453125000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '2441406250000000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '12207031250000000', '12207031250000000', '6103515625000000', '3051757812500000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '305175781250000', '3051757812500000', '610351562500000', '1220703125000000', '3051757812500000', '3051757812500000', '610351562500000', '3051757812500000', '12207031250000000', '610351562500000', '3051757812500000', '122070312500000', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '24414062500000000', '3051757812500000', '1525878906250000', '610351562500000', '1525878906250000', '3051757812500000', '1220703125000000', '1525878906250000', '38146972656250', '305175781250000', '6103515625000000', '152587890625000', '12207031250000000', '6103515625000000', '762939453125000', '781250000000000000', '12207031250000000', '390625000000000000', '6250000000000000000', '48828125000000000', '39062500000000000', '6250000000000000000', '97656250000000000', '195312500000000000', '390625000000000000', '97656250000000000', '1250000000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '1562500000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '24414062500000000', '1562500000000000000', '24414062500000000', '19531250000000000', '48828125000000000', '25000000000000000000', '25000000000000000000', '585937500000000000', '292968750000000000', '292968750000000000', '1562500000000000000', '292968750000000000', '7324218750000000', '292968750000000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '48828125000000000', '12500000000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '292968750000000000', '585937500000000000', '585937500000000000', '12500000000000000000', '117187500000000000', '585937500000000000', '3662109375000000', '25000000000000000000', '6250000000000000000', '6103515625000000', '195312500000000000', '48828125000000000', '3125000000000000000', '48828125000000000', '195312500000000000', '6103515625000000', '6250000000000000000', '6250000000000000000', '117187500000000000', '73242187500000000', '585937500000000000', '3125000000000000000', '585937500000000000', '292968750000000000', '585937500000000000', '6250000000000000000', '1250000000000000000', '97656250000000000', '195312500000000000', '24414062500000000', '625000000000000000', '6103515625000000', '195312500000000000', '97656250000000000', '1250000000000000000', '1250000000000000000', '1171875000000000000', '1171875000000000000', '234375000000000000', '625000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '1250000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '24414062500000000', '12500000000000000000', '24414062500000000', '48828125000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '46875000000000000', '234375000000000000', '25000000000000000000', '625000000000000000', '97656250000000000', '195312500000000000', '48828125000000000', '3125000000000000000', '48828125000000000', '195312500000000000', '97656250000000000', '1562500000000000000', '625000000000000000', '585937500000000000', '73242187500000000', '585937500000000000', '3125000000000000000', '585937500000000000', '292968750000000000', '117187500000000000', '1562500000000000000', '1562500000000000000', '50000000000000000000', '25000000000000000000', '20000000000000000000', '100000000000000000000', '80000000000000000', '38146972656250', '6103515625000000', '762939453125000', '6103515625000000', '762939453125000', '1220703125000000', '12207031250000000', '6103515625000000', '1220703125000000', '24414062500000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '381469726562500', '610351562500000', '24414062500000000', '6103515625000000', '3051757812500000', '12207031250000000', '244140625000000', '3051757812500000', '95367431640625', '762939453125000', '762939453125000', '6103515625000000', '762939453125000', '3051757812500000', '24414062500000000', '3051757812500000', '6103515625000000', '24414062500000000', '6103515625000000', '38146972656250', '1525878906250000', '1525878906250000', '61035156250000', '6103515625000000', '305175781250000', '610351562500000', '24414062500000000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '48828125000000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '762939453125000', '6103515625000000', '6103515625000000', '1525878906250000', '95367431640625', '12207031250000000', '6103515625000000', '1525878906250000', '156250000000000000', '100000000000000000000', '100000000000000000000', '39062500000000000', '7812500000000000', '195312500000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '195312500000000000', '97656250000000000', '78125000000000000', '19531250000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '19531250000000000', '195312500000000000', '195312500000000000', '390625000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '19531250000000000', '195312500000000000', '6250000000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '195312500000000000', '195312500000000000', '390625000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '156250000000000000', '97656250000000000', '390625000000000000', '12500000000000000000', '12207031250000000', '97656250000000000', '24414062500000000', '12500000000000000000', '156250000000000000', '97656250000000000', '390625000000000000', '12500000000000000000', '12207031250000000', '97656250000000000', '24414062500000000', '12500000000000000000', '12500000000000000000', '12207031250000000', '390625000000000000', '6103515625000000', '12500000000000000000', '390625000000000000', '97656250000000000', '9765625000000000', '12500000000000000000', '12207031250000000', '390625000000000000', '6103515625000000', '12500000000000000000', '390625000000000000', '97656250000000000', '9765625000000000', '12500000000000000000', '48828125000000000', '195312500000000000', '6250000000000000000', '2441406250000000', '195312500000000000', '48828125000000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '195312500000000000', '6250000000000000000', '2441406250000000', '195312500000000000', '48828125000000000', '12500000000000000000', '12500000000000000000', '19531250000000000', '390625000000000000', '6250000000000000000', '12207031250000000', '15625000000000000', '19531250000000000', '12500000000000000000', '12500000000000000000', '19531250000000000', '390625000000000000', '6250000000000000000', '12207031250000000', '15625000000000000', '19531250000000000', '12500000000000000000', '12500000000000000000', '24414062500000000', '97656250000000000', '6250000000000000000', '12207031250000000', '97656250000000000', '24414062500000000', '12500000000000000000', '12500000000000000000', '24414062500000000', '97656250000000000', '6250000000000000000', '12207031250000000', '97656250000000000', '24414062500000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '6103515625000000', '19531250000000000', '48828125000000000', '3125000000000000000', '12500000000000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '6103515625000000', '19531250000000000', '48828125000000000', '3125000000000000000', '10000000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '25000000000000000000', '12500000000000000000', '762939453125000', '12207031250000000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '762939453125000', '6103515625000000', '762939453125000', '381469726562500', '12207031250000000', '762939453125000', '6103515625000000', '381469726562500', '6103515625000000', '12207031250000000', '24414062500000000', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '190734863281250', '12207031250000000', '1525878906250000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '24414062500000000', '6103515625000000', '3051757812500000', '762939453125000', '762939453125000', '1525878906250000', '3051757812500000', '1525878906250000', '488281250000000', '3051757812500000', '6103515625000000', '12207031250000000', '12207031250000000', '1220703125000000', '3051757812500000', '610351562500000', '762939453125000', '1525878906250000', '6103515625000000', '95367431640625', '12207031250000000', '762939453125000', '3051757812500000', '4882812500000000', '4882812500000000', '6103515625000000', '381469726562500', '1220703125000000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '610351562500000', '762939453125000', '762939453125000', '12207031250000000', '12207031250000000', '24414062500000', '3051757812500000', '6103515625000000', '3051757812500000', '6250000000000000000', '390625000000000000', '195312500000000000', '2500000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '4882812500000000', '390625000000000000', '3125000000000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '9765625000000000', '195312500000000000', '6250000000000000000', '24414062500000000', '1562500000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '390625000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '62500000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '195312500000000000', '762939453125000', '781250000000000000', '12207031250000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '12207031250000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '12207031250000000', '195312500000000000', '6103515625000000', '12207031250000000', '781250000000000000', '781250000000000000', '12207031250000000', '195312500000000000', '6103515625000000', '12207031250000000', '781250000000000000', '195312500000000000', '1220703125000000', '97656250000000000', '4882812500000000', '1220703125000000', '195312500000000000', '195312500000000000', '1220703125000000', '97656250000000000', '4882812500000000', '1220703125000000', '195312500000000000', '78125000000000000', '488281250000000', '12207031250000000', '195312500000000000', '488281250000000', '78125000000000000', '78125000000000000', '488281250000000', '12207031250000000', '195312500000000000', '488281250000000', '78125000000000000', '390625000000000000', '12207031250000000', '195312500000000000', '195312500000000', '12207031250000000', '390625000000000000', '390625000000000000', '12207031250000000', '195312500000000000', '195312500000000', '12207031250000000', '390625000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '24414062500000000', '48828125000000000', '390625000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '24414062500000000', '48828125000000000', '390625000000000000', '390625000000000000', '12207031250000000', '4882812500000000', '195312500000000000', '12207031250000000', '390625000000000000', '390625000000000000', '12207031250000000', '4882812500000000', '195312500000000000', '12207031250000000', '390625000000000000', '6250000000000000000', '100000000000000000000', '100000000000000000000', '20000000000000000000', '100000000000000000000', '500000000000000000', '3051757812500000', '6103515625000000', '6103515625000000', '610351562500000', '2441406250000000', '12207031250000000', '2441406250000000', '97656250000000000', '12207031250000000', '3051757812500000', '3051757812500000', '305175781250000', '3051757812500000', '3051757812500000', '12207031250000000', '12207031250000000', '6103515625000000', '24414062500000000', '61035156250000', '1525878906250000', '305175781250000', '6103515625000000', '610351562500000', '3051757812500000', '6103515625000000', '12207031250000000', '12207031250000000', '195312500000000000', '305175781250000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '61035156250000', '6103515625000000', '12207031250000000', '12207031250000000', '6103515625000000', '12207031250000000', '1220703125000000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '2441406250000000', '12207031250000000', '1220703125000000', '195312500000000000', '2441406250000000', '6103515625000000', '3051757812500000', '610351562500000', '1525878906250000', '3051757812500000', '2441406250000000', '2441406250000000', '6103515625000000', '24414062500000000', '1525878906250000', '6103515625000000', '19531250000000000', '39062500000000000', '1525878906250000', '48828125000000000', '39062500000000000', '7812500000000000', '625000000000000000', '39062500000000000', '390625000000000000', '97656250000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '97656250000000000', '62500000000000000', '78125000000000000', '19531250000000000', '250000000000000000', '195312500000000000', '97656250000000000', '12207031250000000', '12207031250000000', '24414062500000000', '195312500000000000', '195312500000000000', '97656250000000000', '12207031250000000', '12207031250000000', '24414062500000000', '195312500000000000', '781250000000000000', '390625000000000000', '2441406250000000', '2441406250000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '2441406250000000', '2441406250000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '4882812500000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '4882812500000000', '12207031250000000', '781250000000000000', '781250000000000000', '78125000000000000', '48828125000000000', '48828125000000000', '4882812500000000', '781250000000000000', '781250000000000000', '78125000000000000', '48828125000000000', '48828125000000000', '4882812500000000', '781250000000000000', '781250000000000000', '390625000000000000', '1953125000000000', '1953125000000000', '4882812500000000', '781250000000000000', '781250000000000000', '390625000000000000', '1953125000000000', '1953125000000000', '4882812500000000', '781250000000000000', '15625000000000000', '195312500000000000', '1525878906250000', '1525878906250000', '6103515625000000', '15625000000000000', '15625000000000000', '195312500000000000', '1525878906250000', '1525878906250000', '6103515625000000', '15625000000000000', '6250000000000000000', '25000000000000000000', '25000000000000000000', '6250000000000000000', '50000000000000000000', '50000000000000000000', '610351562500000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '97656250000000000', '381469726562500', '12207031250000000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '610351562500000', '3051757812500000', '12207031250000000', '24414062500000000', '1220703125000000', '762939453125000', '2441406250000000', '3051757812500000', '19073486328125', '6103515625000000', '762939453125000', '3051757812500000', '12207031250000000', '39062500000000000', '12207031250000000', '97656250000000', '12207031250000000', '190734863281250', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '12207031250000000', '19531250000000000', '12207031250000000', '488281250000000', '12207031250000000', '6103515625000000', '3051757812500000', '152587890625000', '122070312500000', '762939453125000', '3051757812500000', '24414062500000000', '6103515625000000', '2441406250000000', '12207031250000000', '3051757812500000', '1525878906250000', '122070312500000', '305175781250000', '610351562500000', '12207031250000000', '6103515625000000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '1562500000000000000', '390625000000000000', '12207031250000000', '1250000000000000000', '39062500000000000', '195312500000000000', '390625000000000000', '781250000000000000', '9765625000000000', '390625000000000000', '250000000000000000', '97656250000000000', '195312500000000000', '390625000000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '156250000000000000', '781250000000000000', '781250000000000000', '156250000000000000', '195312500000000000', '6103515625000000', '781250000000000000', '6103515625000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '6103515625000000', '781250000000000000', '156250000000000000', '156250000000000000', '78125000000000000', '9765625000000000', '781250000000000', '48828125000000000', '78125000000000000', '48828125000000000', '48828125000000000', '390625000000000000', '9765625000000000', '390625000000000000', '6103515625000000', '488281250000000', '12207031250000000', '6103515625000000', '6103515625000000', '488281250000000', '12207031250000000', '6103515625000000', '24414062500000000', '6103515625000000', '12207031250000000', '24414062500000000', '24414062500000000', '6103515625000000', '12207031250000000', '24414062500000000', '24414062500000000', '190734863281250', '12207031250000000', '24414062500000000', '24414062500000000', '190734863281250', '12207031250000000', '24414062500000000', '976562500000000', '6103515625000000', '12207031250000000', '976562500000000', '976562500000000', '6103515625000000', '12207031250000000', '976562500000000', '48828125000000000', '4882812500000000', '6103515625000000', '48828125000000000', '48828125000000000', '4882812500000000', '6103515625000000', '48828125000000000', '48828125000000000', '48828125000000000', '12207031250000000', '4882812500000000', '48828125000000000', '48828125000000000', '12207031250000000', '4882812500000000', '6250000000000000000', '6250000000000000000', '156250000000000000', '3125000000000000000', '312500000000000000', '781250000000000000', '12207031250000000', '12207031250000000', '3051757812500000', '24414062500000000', '12207031250000000', '48828125000000000', '12207031250000000', '2441406250000000', '2441406250000000', '12207031250000000', '762939453125000', '24414062500000000', '12207031250000000', '48828125000000000', '48828125000000000', '24414062500000000', '12207031250000000', '12207031250000000', '12207031250000000', '48828125000000000', '48828125000000000', '1953125000000000', '48828125000000000', '24414062500000000', '12207031250000000', '12207031250000000', '12207031250000000', '2441406250000000', '24414062500000000', '48828125000000000', '48828125000000000', '1525878906250000', '12207031250000000', '12207031250000000', '12207031250000000', '9765625000000000', '1220703125000000', '48828125000000000', '48828125000000000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '48828125000000000', '48828125000000000', '24414062500000000', '3051757812500000', '24414062500000000', '390625000000000000', '48828125000000000', '48828125000000000', '195312500000000000', '78125000000000000', '390625000000000000', '781250000000000000', '12207031250000000', '195312500000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '12207031250000000', '781250000000000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '4', '4', '4', '4', '1', '1', '4', '4', '4', '4', '62500000000000000', '1525878906250000', '62500000000000000', '390625000000000000', '3051757812500000', '1562500000000000000', '3051757812500000', '3051757812500000', '1525878906250000', '1562500000000000000', '156250000000000000', '156250000000000000', '390625000000000000', '312500000000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '190734863281250', '390625000000000000', '190734863281250', '190734863281250', '156250000000000000', '390625000000000000', '390625000000000000', '156250000000000000', '781250000000000000', '195312500000000000', '195312500000000000', '781250000000000000', '97656250000000000', '781250000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '1562500000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '781250000000000000', '97656250000000000', '781250000000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '19531250000000000', '15625000000000000', '15625000000000000', '19531250000000000', '48828125000000000', '3051757812500000', '15625000000000000', '3051757812500000', '15625000000000000', '15625000000000000', '15625000000000000', '15625000000000000', '3051757812500000', '15625000000000000', '19531250000000000', '19531250000000000', '781250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '390625000000000000', '24414062500000000', '390625000000000000', '4', '4', '4', '4', '1', '24414062500000000', '24414062500000000', '24414062500000000', '24414062500000000', '12207031250000000', '9765625000000000', '3051757812500000', '48828125000000000', '12207031250000000', '9765625000000000', '3051757812500000', '48828125000000000', '24414062500000000', '381469726562500', '6103515625000000', '24414062500000000', '24414062500000000', '381469726562500', '6103515625000000', '24414062500000000', '48828125000000000', '12207031250000000', '12207031250000000', '48828125000000000', '48828125000000000', '12207031250000000', '12207031250000000', '48828125000000000', '48828125000000000', '24414062500000000', '762939453125000', '48828125000000000', '48828125000000000', '24414062500000000', '762939453125000', '48828125000000000', '48828125000000000', '24414062500000000', '762939453125000', '48828125000000000', '48828125000000000', '24414062500000000', '762939453125000', '48828125000000000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1562500000000000000', '781250000000000000', '3125000000000000000', '97656250000000000', '488281250000000', '625000000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '48828125000000000', '12207031250000000', '762939453125000', '12207031250000000', '24414062500000000', '24414062500000000', '1220703125000000', '48828125000000000', '9765625000000000', '12207031250000000', '6103515625000000', '12207031250000000', '24414062500000000', '48828125000000000', '24414062500000000', '48828125000000000', '24414062500000000', '1525878906250000', '1525878906250000', '1220703125000000', '12207031250000000', '1525878906250000', '48828125000000000', '48828125000000000', '48828125000000000', '6103515625000000', '3051757812500000', '6103515625000000', '24414062500000000', '12207031250000000', '24414062500000000', '12207031250000000', '12207031250000000', '12207031250000000', '6103515625000000', '12207031250000000', '3051757812500000', '12207031250000000', '762939453125000', '3051757812500000', '24414062500000000', '12207031250000000', '6103515625000000', '2441406250000000', '381469726562500', '97656250000000000', '390625000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '39062500000000000', '1562500000000000000', '781250000000000000', '12207031250000000', '12207031250000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '12207031250000000', '12207031250000000', '24414062500000000', '1562500000000000000', '195312500000000000', '24414062500000000', '48828125000000000', '1220703125000000', '781250000000000000', '781250000000000000', '195312500000000000', '24414062500000000', '48828125000000000', '1220703125000000', '781250000000000000', '781250000000000000', '312500000000000000', '976562500000000', '12207031250000000', '12207031250000000', '312500000000000000', '390625000000000000', '312500000000000000', '976562500000000', '12207031250000000', '12207031250000000', '312500000000000000', '390625000000000000', '1562500000000000000', '781250000000000000', '12207031250000000', '12207031250000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '12207031250000000', '12207031250000000', '24414062500000000', '1562500000000000000', '39062500000000000', '97656250000000000', '190734863281250', '190734863281250', '12207031250000000', '39062500000000000', '39062500000000000', '97656250000000000', '190734863281250', '190734863281250', '12207031250000000', '39062500000000000', '1562500000000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '781250000000000000', '32000000000000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '12500000000000000000', '100000000000000000000', '12207031250000000', '6103515625000000', '6103515625000000', '195312500000000000', '12207031250000000', '762939453125000', '190734863281250', '381469726562500', '3051757812500000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '390625000000000000', '3051757812500000', '381469726562500', '381469726562500', '6103515625000000', '610351562500000', '6103515625000000', '1525878906250000', '6103515625000000', '1220703125000000', '97656250000000000', '12207031250000000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '12207031250000000', '12207031250000000', '390625000000000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '12207031250000000', '12207031250000000', '12207031250000000', '39062500000000000', '12207031250000000', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '381469726562500', '6103515625000000', '12207031250000000', '6103515625000000', '19531250000000000', '12207031250000000', '381469726562500', '3051757812500000', '1220703125000000', '762939453125000', '1525878906250000', '78125000000000000', '100000000000000000000', '97656250000000000', '3125000000000000000', '97656250000000000', '39062500000000000', '390625000000000000', '12500000000000000000', '48828125000000000', '390625000000000000', '1562500000000000000', '97656250000000000', '39062500000000000', '15625000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '7812500000000000', '12500000000000000000', '97656250000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '2500000000000000000', '12500000000000000000', '97656250000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '2500000000000000000', '12500000000000000000', '48828125000000000', '1525878906250000', '97656250000000000', '390625000000000000', '625000000000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '1525878906250000', '97656250000000000', '390625000000000000', '625000000000000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '3051757812500000', '97656250000000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '3051757812500000', '97656250000000000', '25000000000000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '195312500000000000', '12207031250000000', '6103515625000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '195312500000000000', '12207031250000000', '6103515625000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '2500000000000000000', '195312500000000000', '390625000000000000', '381469726562500', '97656250000000000', '4882812500000000', '12500000000000000000', '97656250000000000', '2500000000000000000', '195312500000000000', '390625000000000000', '381469726562500', '97656250000000000', '4882812500000000', '12500000000000000000', '97656250000000000', '6103515625000000', '48828125000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '12500000000000000000', '97656250000000000', '6103515625000000', '48828125000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '25000000000000000000', '100000000000000000000', '12500000000000000000', '6250000000000000000', '100000000000000000000', '10000000000000000000', '3051757812500000', '305175781250000', '381469726562500', '19073486328125', '12207031250000000', '3051757812500000', '190734863281250', '12207031250000000', '3051757812500000', '12207031250000000', '762939453125000', '3051757812500000', '610351562500000', '6103515625000000', '3051757812500000', '24414062500000000', '2441406250000000', '3051757812500000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '381469726562500', '152587890625000', '1525878906250000', '3051757812500000', '610351562500000', '12207031250000000', '1525878906250000', '3051757812500000', '610351562500000', '244140625000000', '190734863281250', '12207031250000000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '381469726562500', '24414062500000000', '24414062500000000', '3051757812500000', '1525878906250000', '762939453125000', '610351562500000', '1525878906250000', '1525878906250000', '3051757812500000', '6103515625000000', '244140625000000', '6103515625000000', '24414062500000000', '2441406250000000', '190734863281250', '3051757812500000', '12207031250000000', '762939453125000', '12207031250000000', '3051757812500000', '381469726562500', '6103515625000000', '6103515625000000', '762939453125000', '24414062500000000', '24414062500000000', '762939453125000', '190734863281250', '19531250000000', '762939453125000', '12207031250000000', '610351562500000', '1525878906250000', '390625000000000000', '762939453125000', '3125000000000000000', '25000000000000000000', '39062500000000000', '78125000000000000', '390625000000000000', '19531250000000000', '6250000000000000000', '97656250000000000', '781250000000000', '6250000000000000000', '12500000000000000', '97656250000000000', '78125000000000000', '390625000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '39062500000000000', '390625000000000000', '12207031250000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '39062500000000000', '390625000000000000', '3051757812500000', '97656250000000000', '100000000000000000000', '20000000000000000000', '3906250000000000', '6250000000000000000', '195312500000000000', '390625000000000000', '3051757812500000', '97656250000000000', '100000000000000000000', '20000000000000000000', '3906250000000000', '6250000000000000000', '195312500000000000', '390625000000000000', '3051757812500000', '97656250000000000', '25000000000000000000', '50000000000000000000', '97656250000000000', '2000000000000000000', '390625000000000000', '390625000000000000', '3051757812500000', '97656250000000000', '25000000000000000000', '50000000000000000000', '97656250000000000', '2000000000000000000', '390625000000000000', '78125000000000000', '6103515625000000', '48828125000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '12500000000000000000', '390625000000000000', '78125000000000000', '6103515625000000', '48828125000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '6250000000000000000', '78125000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '6250000000000000000', '78125000000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '12207031250000000', '97656250000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '12207031250000000', '97656250000000000', '100000000000000000000', '25000000000000000000', '20000000000000000000', '25000000000000000000', '25000000000000000000', '50000000000000000000', '6103515625000000', '1525878906250000', '762939453125000', '1525878906250000', '381469726562500', '1525878906250000', '1525878906250000', '6103515625000000', '6103515625000000', '24414062500000000', '1220703125000000', '3051757812500000', '305175781250000', '3051757812500000', '1525878906250000', '6103515625000000', '152587890625000', '6103515625000000', '76293945312500', '6103515625000000', '6103515625000000', '12207031250000000', '6103515625000000', '1525878906250000', '1220703125000000', '3051757812500000', '19073486328125', '6103515625000000', '95367431640625', '6103515625000000', '1525878906250000', '1525878906250000', '24414062500000000', '4882812500000000', '6103515625000000', '3051757812500000', '95367431640625', '3051757812500000', '381469726562500', '190734863281250', '3051757812500000', '6103515625000000', '762939453125000', '762939453125000', '12207031250000000', '4882812500000000', '6103515625000000', '762939453125000', '305175781250000', '1220703125000000', '1525878906250000', '305175781250000', '305175781250000', '244140625000000', '1525878906250000', '1525878906250000', '4882812500000000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '1220703125000000', '1525878906250000', '6103515625000000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '122070312500000', '12207031250000000', '3051757812500000', '3051757812500000', '48828125000000000', '48828125000000000', '97656250000000000', '39062500000000000', '6250000000000000000', '390625000000000000', '195312500000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '390625000000000000', '195312500000000000', '97656250000000000', '250000000000000000', '195312500000000000', '39062500000000000', '6250000000000000000', '390625000000000000', '24414062500000000', '2441406250000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '6103515625000000', '24414062500000000', '2441406250000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '6103515625000000', '48828125000000000', '100000000000000000000', '3125000000000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '100000000000000000000', '3125000000000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '24414062500000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '48828125000000000', '12207031250000000', '24414062500000000', '12500000000000000000', '3125000000000000000', '12500000000000000000', '9765625000000000', '12207031250000000', '24414062500000000', '12500000000000000000', '3125000000000000000', '12500000000000000000', '9765625000000000', '24414062500000000', '50000000000000000000', '10000000000000000000', '3125000000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '50000000000000000000', '10000000000000000000', '3125000000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '24414062500000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '12207031250000000', '24414062500000000', '24414062500000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '12207031250000000', '25000000000000000000', '25000000000000000000', '100000000000000000000', '20000000000000000000', '6250000000000000000', '12500000000000000000', '19073486328125', '305175781250000', '381469726562500', '1525878906250000', '1525878906250000', '762939453125000', '6103515625000000', '190734863281250', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '95367431640625', '38146972656250', '762939453125000', '95367431640625', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '762939453125000', '1220703125000000', '1525878906250000', '762939453125000', '1525878906250000', '762939453125000', '381469726562500', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '95367431640625', '30517578125000', '1525878906250000', '61035156250000', '610351562500000', '1525878906250000', '610351562500000', '762939453125000', '3051757812500000', '6103515625000000', '1525878906250000', '305175781250000', '1525878906250000', '381469726562500', '1525878906250000', '2441406250000000', '762939453125000', '3051757812500000', '97656250000000000', '48828125000000000', '1562500000000000000', '4882812500000000', '50000000000000000000', '390625000000000000', '195312500000000000', '78125000000000000', '19531250000000000', '97656250000000000', '15625000000000000', '781250000000000000', '48828125000000000', '15625000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '195312500000000000', '6250000000000000000', '6250000000000000000', '625000000000000000', '12207031250000000', '6250000000000000000', '6250000000000000000', '625000000000000000', '12207031250000000', '3125000000000000000', '3125000000000000000', '195312500000000000', '3051757812500000', '3125000000000000000', '3125000000000000000', '195312500000000000', '3051757812500000', '3125000000000000000', '390625000000000000', '3125000000000000000', '3051757812500000', '3125000000000000000', '390625000000000000', '3125000000000000000', '3051757812500000', '100000000000000000000', '20000000000000000000', '2000000000000000000', '3051757812500000', '100000000000000000000', '20000000000000000000', '2000000000000000000', '3051757812500000', '1562500000000000000', '625000000000000000', '625000000000000000', '6103515625000000', '1562500000000000000', '625000000000000000', '625000000000000000', '6103515625000000', '625000000000000000', '6250000000000000000', '6250000000000000000', '12207031250000000', '625000000000000000', '6250000000000000000', '6250000000000000000', '12207031250000000', '3125000000000000000', '50000000000000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '3051757812500000', '3051757812500000', '1525878906250000', '152587890625000', '762939453125000', '610351562500000', '305175781250000', '610351562500000', '95367431640625', '6103515625000000', '3051757812500000', '1525878906250000', '1525878906250000', '1525878906250000', '244140625000000', '1525878906250000', '381469726562500', '6103515625000000', '381469726562500', '190734863281250', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '1220703125000000', '1525878906250000', '190734863281250', '1525878906250000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '762939453125000', '1525878906250000', '6103515625000000', '1220703125000000', '3051757812500000', '3051757812500000', '1220703125000000', '3051757812500000', '305175781250000', '190734863281250', '76293945312500', '3051757812500000', '1525878906250000', '390625000000000000', '78125000000000000', '195312500000000000', '156250000000000000', '97656250000000000', '39062500000000000', '4', '4', '4', '4', '781250000000000000', '195312500000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '48828125000000000', '781250000000000000', '312500000000000000', '6103515625000000', '781250000000000000', '24414062500000000', '6103515625000000', '312500000000000000', '312500000000000000', '6103515625000000', '781250000000000000', '24414062500000000', '6103515625000000', '312500000000000000', '1562500000000000000', '1525878906250000', '24414062500000000', '781250000000000000', '48828125000000000', '1562500000000000000', '1562500000000000000', '1525878906250000', '24414062500000000', '781250000000000000', '48828125000000000', '1562500000000000000', '312500000000000000', '24414062500000000', '6103515625000000', '781250000000000000', '48828125000000000', '312500000000000000', '312500000000000000', '24414062500000000', '6103515625000000', '781250000000000000', '48828125000000000', '312500000000000000', '1562500000000000000', '9765625000000000', '781250000000000000', '12207031250000000', '9765625000000000', '1562500000000000000', '1562500000000000000', '9765625000000000', '781250000000000000', '12207031250000000', '9765625000000000', '1562500000000000000', '1562500000000000000', '2441406250000000', '24414062500000000', '1562500000000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '2441406250000000', '24414062500000000', '1562500000000000000', '48828125000000000', '781250000000000000', '195312500000000000', '12207031250000000', '24414062500000000', '1562500000000000000', '24414062500000000', '1562500000000000000', '195312500000000000', '12207031250000000', '24414062500000000', '1562500000000000000', '24414062500000000', '1562500000000000000', '50000000000000000000', '25000000000000000000', '312500000000000000', '100000000000000000000', '50000000000000000000', '20000000000000000000', '2441406250000000', '195312500000000000', '12207031250000000', '12207031250000000', '12207031250000000', '190734863281250', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '390625000000000000', '12207031250000000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '95367431640625', '6103515625000000', '2441406250000000', '19531250000000000', '152587890625000', '3051757812500000', '2441406250000000', '3051757812500000', '1220703125000000', '6103515625000000', '1525878906250000', '95367431640625', '152587890625000', '390625000000000000', '488281250000000', '3051757812500000', '12207031250000000', '381469726562500', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '2441406250000000', '195312500000000000', '6103515625000000', '12207031250000000', '12207031250000000', '3051757812500000', '6103515625000000', '6103515625000000', '95367431640625', '3051757812500000', '12207031250000000', '97656250000000000', '6103515625000000', '12207031250000000', '12207031250000000', '1525878906250000', '1525878906250000', '1220703125000000', '190734863281250', '6103515625000000', '195312500000000000', '48828125000000000', '4000000000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '39062500000000000', '97656250000000000', '3125000000000000000', '1562500000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '195312500000000000', '48828125000000000', '1562500000000000000', '39062500000000000', '48828125000000000', '97656250000000000', '12500000000000000000', '97656250000000000', '48828125000000000', '39062500000000000', '12500000000000000000', '1562500000000000000', '390625000000000000', '390625000000000000', '78125000000000000', '12500000000000000000', '390625000000000000', '78125000000000000', '390625000000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '3051757812500000', '195312500000000000', '200000000000000000', '195312500000000000', '3051757812500000', '97656250000000000', '3125000000000000000', '25000000000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '200000000000000000', '48828125000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '6250000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '6250000000000000000', '6250000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '3125000000000000000', '234375000000000000', '1171875000000000000', '234375000000000000', '6250000000000000000', '12500000000000000000', '97656250000000000', '48828125000000000', '19531250000000000', '25000000000000000000', '19531250000000000', '48828125000000000', '97656250000000000', '25000000000000000000', '12500000000000000000', '195312500000000000', '39062500000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '25000000000000000000', '5000000000000000000', '97656250000000000', '6103515625000000', '195312500000000000', '25000000000000000000', '195312500000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '5000000000000000000', '292968750000000000', '146484375000000000', '585937500000000000', '25000000000000000000', '585937500000000000', '146484375000000000', '292968750000000000', '12500000000000000000', '25000000000000000000', '39062500000000000', '762939453125000', '195312500000000000', '25000000000000000000', '39062500000000000', '762939453125000', '39062500000000000', '12500000000000000000', '25000000000000000000', '1171875000000000000', '234375000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '3051757812500000', '3125000000000000000', '3125000000000000000', '122070312500000', '390625000000000000', '1562500000000000000', '6103515625000000', '381469726562500', '381469726562500', '152587890625000', '24414062500000000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '305175781250000', '1525878906250000', '190734863281250', '3051757812500000', '12207031250000000', '6103515625000000', '3051757812500000', '24414062500000', '3051757812500000', '762939453125000', '3051757812500000', '762939453125000', '244140625000000', '305175781250000', '1525878906250000', '3051757812500000', '24414062500000000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '381469726562500', '6103515625000000', '3051757812500000', '6103515625000000', '1525878906250000', '1525878906250000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '61035156250000', '1525878906250000', '1525878906250000', '122070312500000', '190734863281250', '95367431640625', '6103515625000000', '24414062500000000', '6103515625000000', '3051757812500000', '762939453125000', '762939453125000', '190734863281250', '762939453125000', '3051757812500000', '6103515625000000', '305175781250000', '1525878906250000', '6103515625000000', '12207031250000000', '3051757812500000', '6103515625000000', '12207031250000000', '3051757812500000', '1525878906250000', '244140625000000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '97656250000000000', '390625000000000000', '97656250000000000', '19531250000000000', '19531250000000000', '1562500000000000000', '48828125000000000', '97656250000000000', '7812500000000000', '97656250000000000', '50000000000000000000', '39062500000000000', '12207031250000000', '78125000000000000', '97656250000000000', '97656250000000000', '195312500000000000', '312500000000000000', '48828125000000000', '50000000000000000000', '4882812500000000', '12207031250000000', '390625000000000000', '390625000000000000', '25000000000000000000', '50000000000000000000', '19531250000000000', '195312500000000000', '50000000000000000000', '97656250000000000', '39062500000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '2000000000000000000', '390625000000000000', '390625000000000000', '305175781250000', '781250000000000000', '3125000000000000000', '3125000000000000000', '762939453125000', '1562500000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '1953125000000000', '78125000000000000', '3125000000000000000', '1562500000000000000', '3051757812500000', '3125000000000000000', '3125000000000000', '3051757812500000', '390625000000000000', '2500000000000000000', '1250000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '12500000000000000000', '390625000000000000', '12207031250000000', '781250000000000000', '3125000000000000000', '1562500000000000000', '12207031250000000', '3125000000000000000', '390625000000000000', '48828125000000000', '390625000000000000', '25000000000000000000', '2500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '781250000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '312500000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '3125000000000000000', '1562500000000000000', '24414062500000000', '3125000000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '312500000000000000', '1562500000000000000', '12207031250000000', '781250000000000000', '781250000000000000', '48828125000000000', '97656250000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '100000000000000000000', '25000000000000000000', '195312500000000000', '100000000000000000000', '305175781250000', '24414062500000000', '305175781250000', '24414062500000000', '762939453125000', '1525878906250000', '190734863281250', '381469726562500', '3051757812500000', '3051757812500000', '152587890625000', '1220703125000000', '762939453125000', '610351562500000', '1220703125000000', '2441406250000000', '381469726562500', '3051757812500000', '3051757812500000', '610351562500000', '122070312500000', '3051757812500000', '762939453125000', '3051757812500000', '610351562500000', '3051757812500000', '3051757812500000', '12207031250000000', '6103515625000000', '610351562500000', '6103515625000000', '6103515625000000', '762939453125000', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '762939453125000', '19073486328125', '6103515625000000', '190734863281250', '3051757812500000', '6103515625000000', '24414062500000000', '1220703125000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '1220703125000000', '3051757812500000', '3051757812500000', '3051757812500000', '381469726562500', '24414062500000000', '305175781250000', '3051757812500000', '1220703125000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '610351562500000', '1525878906250000', '1525878906250000', '24414062500000000', '48828125000000000', '39062500000000000', '48828125000000000', '500000000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '195312500000000000', '97656250000000000', '12500000000000000000', '4882812500000000', '48828125000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '12207031250000000', '9765625000000000', '390625000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '97656250000000000', '97656250000000000', '195312500000000000', '24414062500000000', '390625000000000000', '50000000000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '48828125000000000', '195312500000000000', '97656250000000000', '24414062500000000', '195312500000000000', '50000000000000000000', '39062500000000000', '100000000000000000000', '195312500000000000', '100000000000000000000', '12207031250000000', '24414062500000000', '585937500000000000', '146484375000000000', '50000000000000000000', '36621093750000000', '100000000000000000000', '146484375000000000', '100000000000000000000', '585937500000000000', '585937500000000000', '9765625000000000', '195312500000000000', '800000000000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '100000000000000000000', '195312500000000000', '48828125000000000', '1171875000000000000', '1171875000000000000', '800000000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '100000000000000000000', '1171875000000000000', '1171875000000000000', '48828125000000000', '195312500000000000', '20000000000000000000', '97656250000000000', '100000000000000000000', '195312500000000000', '10000000000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '195312500000000000', '20000000000000000000', '97656250000000000', '100000000000000000000', '195312500000000000', '10000000000000000000', '195312500000000000', '48828125000000000', '976562500000000', '97656250000000000', '50000000000000000000', '195312500000000000', '100000000000000000000', '195312500000000000', '100000000000000000000', '195312500000000000', '976562500000000', '292968750000000000', '29296875000000000', '50000000000000000000', '117187500000000000', '100000000000000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '292968750000000000', '24414062500000000', '48828125000000000', '20000000000000000000', '48828125000000000', '10000000000000000000', '190734863281250', '20000000000000000000', '48828125000000000', '24414062500000000', '146484375000000000', '73242187500000000', '20000000000000000000', '9155273437500000', '10000000000000000000', '146484375000000000', '20000000000000000000', '18310546875000000', '146484375000000000', '12207031250000000', '39062500000000000', '10000000000000000000', '2441406250000000', '100000000000000000000', '195312500000000000', '20000000000000000000', '195312500000000000', '12207031250000000', '39062500000000000', '97656250000000000', '10000000000000000000', '195312500000000000', '100000000000000000000', '1953125000000000', '20000000000000000000', '3051757812500000', '195312500000000000', '25000000000000000000', '3125000000000000000', '100000000000000000000', '2500000000000000000', '100000000000000000000', '25000000000000000000', '3051757812500000', '24414062500000000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '381469726562500', '1220703125000000', '12207031250000000', '610351562500000', '3051757812500000', '305175781250000', '12207031250000000', '3051757812500000', '61035156250000', '1220703125000000', '305175781250000', '6103515625000000', '190734863281250', '6103515625000000', '24414062500000000', '6103515625000000', '3051757812500000', '6103515625000000', '24414062500000000', '3051757812500000', '95367431640625', '1220703125000000', '7629394531250', '244140625000000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '305175781250000', '12207031250000000', '762939453125000', '190734863281250', '3051757812500000', '1525878906250000', '762939453125000', '1525878906250000', '381469726562500', '24414062500000000', '6103515625000000', '610351562500000', '1220703125000000', '24414062500000000', '1220703125000000', '61035156250000', '6103515625000000', '381469726562500', '1525878906250000', '1525878906250000', '6103515625000000', '195312500000000', '6103515625000000', '1525878906250000', '12207031250000', '12207031250000000', '6103515625000000', '1525878906250000', '762939453125000', '1525878906250000', '6103515625000000', '95367431640625', '305175781250000', '1525878906250000', '3051757812500000', '610351562500000', '3051757812500000', '195312500000000000', '390625000000000000', '97656250000000000', '100000000000000000000', '48828125000000000', '48828125000000000', '48828125000000000', '390625000000000000', '48828125000000000', '390625000000000000', '312500000000000000', '97656250000000000', '39062500000000000', '2500000000000000000', '390625000000000000', '12500000000000000000', '390625000000000000', '195312500000000000', '195312500000000000', '25000000000000000000', '312500000000000000', '20000000000000000000', '195312500000000000', '3051757812500000', '97656250000000000', '390625000000000000', '610351562500000', '390625000000000000', '3125000000000000000', '195312500000000000', '1953125000000000', '1562500000000000000', '97656250000000000', '390625000000000000', '97656250000000000', '12207031250000000', '5000000000000000000', '50000000000000000000', '4882812500000000', '50000000000000000000', '24414062500000000', '12207031250000000', '5000000000000000000', '50000000000000000000', '4882812500000000', '50000000000000000000', '24414062500000000', '12207031250000000', '100000000000000000000', '2500000000000000000', '6103515625000000', '100000000000000000000', '1953125000000000', '12207031250000000', '100000000000000000000', '2500000000000000000', '6103515625000000', '100000000000000000000', '1953125000000000', '24414062500000000', '6250000000000000000', '12500000000000000000', '24414062500000000', '2500000000000000000', '1220703125000000', '24414062500000000', '6250000000000000000', '12500000000000000000', '24414062500000000', '2500000000000000000', '1220703125000000', '762939453125000', '25000000000000000000', '2500000000000000000', '24414062500000000', '25000000000000000000', '6103515625000000', '762939453125000', '25000000000000000000', '2500000000000000000', '24414062500000000', '25000000000000000000', '6103515625000000', '1953125000000000', '4000000000000000000', '24414062500000000', '5000000000000000000', '100000000000000000000', '12207031250000000', '1953125000000000', '4000000000000000000', '24414062500000000', '5000000000000000000', '100000000000000000000', '12207031250000000', '1525878906250000', '250000000000000000', '1250000000000000000', '24414062500000000', '1562500000000000000', '3051757812500000', '1525878906250000', '250000000000000000', '1250000000000000000', '24414062500000000', '1562500000000000000', '3051757812500000', '100000000000000000000', '12500000000000000000', '20000000000000000000', '12500000000000000000', '800000000000000000', '100000000000000000000', '6103515625000000', '12207031250000000', '610351562500000', '1525878906250000', '1525878906250000', '61035156250000', '305175781250000', '6103515625000000', '6103515625000000', '762939453125000', '6103515625000000', '12207031250000000', '6103515625000000', '762939453125000', '1525878906250000', '762939453125000', '1525878906250000', '1220703125000000', '610351562500000', '305175781250000', '3051757812500000', '48828125000000000', '1525878906250000', '381469726562500', '1525878906250000', '3051757812500000', '305175781250000', '762939453125000', '3051757812500000', '95367431640625', '6103515625000000', '6103515625000000', '610351562500000', '762939453125000', '3051757812500000', '1525878906250000', '762939453125000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '12207031250000000', '3051757812500000', '381469726562500', '762939453125000', '381469726562500', '762939453125000', '762939453125000', '6103515625000000', '3051757812500000', '1525878906250000', '1220703125000000', '762939453125000', '1525878906250000', '3051757812500000', '3051757812500000', '95367431640625', '6103515625000000', '762939453125000', '95367431640625', '48828125000000000', '195312500000000000', '100000000000000000000', '9765625000000000', '6103515625000000', '625000000000000000', '39062500000000000', '97656250000000000', '24414062500000000', '12207031250000000', '19531250000000000', '97656250000000000', '48828125000000000', '24414062500000000', '25000000000000000000', '195312500000000000', '31250000000000000', '78125000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '24414062500000000', '24414062500000000', '195312500000000000', '24414062500000000', '24414062500000000', '6103515625000000', '195312500000000000', '390625000000000000', '390625000000000000', '781250000000000000', '4882812500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '24414062500000000', '24414062500000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '6103515625000000', '1000000000000000000', '48828125000000000', '97656250000000000', '6250000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '6103515625000000', '1000000000000000000', '48828125000000000', '97656250000000000', '6250000000000000000', '6250000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '2441406250000000', '97656250000000000', '390625000000000000', '25000000000000000000', '6250000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '2441406250000000', '97656250000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '25000000000000000000', '24414062500000000', '390625000000000000', '6250000000000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '25000000000000000000', '24414062500000000', '390625000000000000', '6250000000000000000', '6250000000000000000', '390625000000000000', '19531250000000000', '25000000000000000000', '2441406250000000', '48828125000000000', '12207031250000000', '25000000000000000000', '6250000000000000000', '390625000000000000', '19531250000000000', '25000000000000000000', '2441406250000000', '48828125000000000', '12207031250000000', '25000000000000000000', '97656250000000000', '390625000000000000', '3051757812500000', '6103515625000000', '25000000000000000000', '24414062500000000', '97656250000000000', '1000000000000000000', '97656250000000000', '390625000000000000', '3051757812500000', '6103515625000000', '25000000000000000000', '24414062500000000', '97656250000000000', '1000000000000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '25000000000000000000', '12207031250000000', '97656250000000000', '48828125000000000', '25000000000000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '25000000000000000000', '12207031250000000', '97656250000000000', '48828125000000000', '25000000000000000000', '100000000000000000000', '10000000000000000000', '2500000000000000000', '100000000000000000000', '25000000000000000000', '3125000000000000000', '6103515625000000', '12207031250000000', '24414062500000000', '305175781250000', '6103515625000000', '6103515625000000', '1525878906250000', '12207031250000000', '1525878906250000', '381469726562500', '3051757812500000', '122070312500000', '381469726562500', '6103515625000000', '4882812500000000', '6103515625000000', '6103515625000000', '1220703125000000', '19073486328125', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '762939453125000', '3051757812500000', '976562500000000', '24414062500000000', '1525878906250000', '6103515625000000', '190734863281250', '381469726562500', '6103515625000000', '610351562500000', '12207031250000000', '1525878906250000', '61035156250000', '3051757812500000', '24414062500000000', '12207031250000000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '6103515625000000', '762939453125000', '610351562500000', '3051757812500000', '1525878906250000', '3051757812500000', '12207031250000000', '24414062500000000', '381469726562500', '6103515625000000', '6103515625000000', '1525878906250000', '1525878906250000', '1525878906250000', '12207031250000000', '1525878906250000', '1525878906250000', '244140625000000', '4882812500000000', '12207031250000000', '6103515625000000', '3051757812500000', '610351562500000', '1525878906250000', '6103515625000000', '3051757812500000', '6103515625000000', '190734863281250', '3051757812500000', '24414062500000000', '12500000000000000000', '48828125000000000', '78125000000000000', '3125000000000000000', '48828125000000000', '12207031250000000', '12500000000000000000', '12207031250000000', '1562500000000000000', '390625000000000000', '48828125000000000', '6250000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '24414062500000000', '1562500000000000000', '390625000000000000', '15625000000000000', '781250000000000000', '24414062500000000', '24414062500000000', '12207031250000000', '15625000000000000', '390625000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '1250000000000000000', '1562500000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '1562500000000000000', '390625000000000000', '781250000000000000', '31250000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '781250000000000000', '390625000000000000', '1562500000000000000', '625000000000000000', '125000000000000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '1250000000000000000', '1250000000000000000', '3125000000000000000', '781250000000000000', '195312500000000000', '312500000000000000', '24414062500000000', '24414062500000000', '3051757812500000', '195312500000000000', '781250000000000000', '3125000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '1562500000000000000', '1562500000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '3125000000000000000', '195312500000000000', '781250000000000000', '312500000000000000', '48828125000000000', '12207031250000000', '1525878906250000', '781250000000000000', '9765625000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '3125000000000000000', '195312500000000000', '781250000000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '781250000000000000', '390625000000000000', '3125000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '625000000000000000', '78125000000000000', '781250000000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '781250000000000000', '390625000000000000', '625000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '781250000000000000', '48828125000000000', '625000000000000000', '7812500000000000', '24414062500000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '390625000000000000', '152587890625000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '610351562500000', '6103515625000000', '38146972656250', '152587890625000', '1525878906250000', '610351562500000', '76293945312500', '305175781250000', '6103515625000000', '1525878906250000', '3051757812500000', '762939453125000', '48828125000000000', '3051757812500000', '190734863281250', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '24414062500000000', '3051757812500000', '1525878906250000', '1525878906250000', '6103515625000000', '3051757812500000', '762939453125000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '1525878906250000', '48828125000000000', '6103515625000000', '305175781250000', '1220703125000000', '6103515625000000', '3051757812500000', '3051757812500000', '610351562500000', '381469726562500', '6103515625000000', '1525878906250000', '4882812500000', '12207031250000000', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '1525878906250000', '19073486328125', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '48828125000000000', '6103515625000000', '3051757812500000', '381469726562500', '61035156250000', '3051757812500000', '1525878906250000', '19531250000000000', '97656250000000000', '20000000000000000000', '78125000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '48828125000000000', '390625000000000000', '195312500000000000', '39062500000000000', '19531250000000000', '97656250000000000', '97656250000000000', '100000000000000000000', '390625000000000000', '195312500000000000', '12207031250000000', '244140625000000', '97656250000000000', '390625000000000000', '12500000000000000000', '24414062500000000', '97656250000000000', '24414062500000000', '97656250000000000', '390625000000000000', '24414062500000000', '390625000000000000', '19531250000000000', '48828125000000000', '48828125000000000', '390625000000000000', '100000000000000000000', '390625000000000000', '97656250000000000', '39062500000000000', '781250000000000000', '48828125000000000', '195312500000000000', '781250000000000000', '781250000000000000', '24414062500000000', '9765625000000000', '781250000000000000', '39062500000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '100000000000000000', '25000000000000000000', '25000000000000000000', '48828125000000000', '625000000000000000', '48828125000000000', '3125000000000000000', '19531250000000000', '195312500000000000', '24414062500000000', '48828125000000000', '781250000000000000', '12207031250000000', '6250000000000000000', '10000000000000000', '625000000000000000', '6250000000000000000', '1250000000000000000', '6250000000000000000', '6250000000000000000', '625000000000000000', '6250000000000000000', '6250000000000000000', '195312500000000000', '3125000000000000000', '1525878906250000', '1562500000000000000', '97656250000000000', '97656250000000000', '4882812500000000', '1525878906250000', '3125000000000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '31250000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '1220703125000000', '31250000000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '390625000000000000', '781250000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '195312500000000000', '24414062500000000', '48828125000000000', '781250000000000000', '19531250000000000', '500000000000000000', '12500000000000000000', '125000000000000000', '2500000000000000000', '12500000000000000000', '500000000000000000', '12500000000000000000', '125000000000000000', '2500000000000000000', '12500000000000000000', '15625000000000000', '125000000000000000', '95367431640625', '625000000000000000', '781250000000000000', '156250000000000000', '24414062500000000', '95367431640625', '156250000000000000', '15625000000000000', '12500000000000000000', '12500000000000000000', '625000000000000000', '12500000000000000000', '500000000000000000', '12500000000000000000', '12500000000000000000', '625000000000000000', '12500000000000000000', '12500000000000000000', '100000000000000000000', '390625000000000000', '390625000000000000', '3125000000000000000', '100000000000000000000', '2000000000000000000', '762939453125000', '610351562500000', '6103515625000000', '1525878906250000', '762939453125000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '48828125000000000', '1525878906250000', '610351562500000', '3051757812500000', '381469726562500', '6103515625000000', '610351562500000', '3051757812500000', '6103515625000000', '6103515625000000', '762939453125000', '762939453125000', '48828125000000000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '48828125000000000', '6103515625000000', '381469726562500', '3051757812500000', '610351562500000', '762939453125000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '381469726562500', '3051757812500000', '48828125000000000', '762939453125000', '610351562500000', '15258789062500', '381469726562500', '1220703125000000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '12207031250000000', '6103515625000000', '1525878906250000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '305175781250000', '1220703125000000', '48828125000000000', '3051757812500000', '3051757812500000', '19531250000000000', '390625000000000000', '97656250000000000', '24414062500000000', '195312500000000000', '6103515625000000', '97656250000000000', '390625000000000000', '1953125000000000', '5000000000000000000', '19531250000000000', '488281250000000', '48828125000000000', '390625000000000000', '48828125000000000', '78125000000000000', '39062500000000000', '195312500000000000', '24414062500000000', '390625000000000000', '6103515625000000', '390625000000000000', '5000000000000000000', '97656250000000000', '9765625000000000', '15625000000000000', '24414062500000000', '50000000000000000000', '390625000000000000', '97656250000000000', '24414062500000000', '390625000000000000', '9765625000000000', '390625000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '1250000000000000000', '50000000000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '10000000000000000000', '390625000000000000', '97656250000000000', '1250000000000000000', '50000000000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '10000000000000000000', '390625000000000000', '4882812500000000', '100000000000000000000', '5000000000000000000', '24414062500000000', '48828125000000000', '6103515625000000', '20000000000000000000', '78125000000000000', '4882812500000000', '100000000000000000000', '5000000000000000000', '24414062500000000', '48828125000000000', '6103515625000000', '20000000000000000000', '78125000000000000', '97656250000000000', '2000000000000000000', '50000000000000000000', '19531250000000000', '9765625000000000', '6103515625000000', '12500000000000000000', '12207031250000000', '97656250000000000', '2000000000000000000', '50000000000000000000', '19531250000000000', '9765625000000000', '6103515625000000', '12500000000000000000', '12207031250000000', '78125000000000000', '3125000000000000000', '1562500000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '3125000000000000000', '390625000000000000', '78125000000000000', '3125000000000000000', '1562500000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '3125000000000000000', '390625000000000000', '390625000000000000', '625000000000000000', '100000000000000000000', '48828125000000000', '97656250000000000', '12207031250000000', '100000000000000000000', '24414062500000000', '390625000000000000', '625000000000000000', '100000000000000000000', '48828125000000000', '97656250000000000', '12207031250000000', '100000000000000000000', '24414062500000000', '9765625000000000', '6250000000000000000', '25000000000000000000', '48828125000000000', '97656250000000000', '12207031250000000', '25000000000000000000', '390625000000000000', '9765625000000000', '6250000000000000000', '25000000000000000000', '48828125000000000', '97656250000000000', '12207031250000000', '25000000000000000000', '390625000000000000', '50000000000000000000', '100000000000000000000', '6250000000000000000', '100000000000000000000', '100000000000000000000', '20000000000000000000', '3051757812500000', '122070312500000', '381469726562500', '6103515625000000', '762939453125000', '1525878906250000', '381469726562500', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '24414062500000000', '12207031250000000', '3051757812500000', '381469726562500', '762939453125000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '152587890625000', '3051757812500000', '3051757812500000', '305175781250000', '24414062500000000', '381469726562500', '1525878906250000', '1525878906250000', '381469726562500', '1220703125000000', '1525878906250000', '3051757812500000', '152587890625000', '1525878906250000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '762939453125000', '6103515625000000', '95367431640625', '3051757812500000', '152587890625000', '762939453125000', '3814697265625', '3051757812500000', '6103515625000000', '24414062500000000', '3051757812500000', '6103515625000000', '381469726562500', '6103515625000000', '762939453125000', '1220703125000000', '1525878906250000', '1220703125000000', '190734863281250', '1525878906250000', '610351562500000', '1525878906250000', '24414062500000000', '6103515625000000', '190734863281250', '6103515625000000', '1525878906250000', '3051757812500000', '381469726562500', '3051757812500000', '6103515625000000', '3051757812500000', '12500000000000000000', '195312500000000000', '390625000000000000', '2500000000000000000', '195312500000000000', '390625000000000000', '78125000000000000', '195312500000000000', '6250000000000000000', '78125000000000000', '781250000000000', '6250000000000000000', '390625000000000000', '39062500000000000', '39062500000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '97656250000000000', '97656250000000000', '97656250000000000', '24414062500000000', '97656250000000000', '24414062500000000', '24414062500000000', '390625000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '781250000000000000', '610351562500000', '781250000000000000', '97656250000000000', '97656250000000000', '7812500000000000', '1562500000000000000', '7812500000000000', '97656250000000000', '97656250000000000', '781250000000000000', '610351562500000', '781250000000000000', '97656250000000000', '97656250000000000', '3125000000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '6103515625000000', '97656250000000000', '390625000000000000', '1250000000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '6103515625000000', '97656250000000000', '390625000000000000', '1250000000000000000', '1562500000000000000', '39062500000000000', '97656250000000000', '12500000000000000000', '12207031250000000', '48828125000000000', '39062500000000000', '12500000000000000000', '1562500000000000000', '39062500000000000', '97656250000000000', '12500000000000000000', '12207031250000000', '48828125000000000', '39062500000000000', '12500000000000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '3051757812500000', '12207031250000000', '195312500000000000', '6250000000000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '3051757812500000', '12207031250000000', '195312500000000000', '6250000000000000000', '2500000000000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '156250000000000000', '2500000000000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '156250000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '12207031250000000', '48828125000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '12207031250000000', '48828125000000000', '195312500000000000', '25000000000000000000', '12500000000000000000', '48828125000000000', '3906250000000000', '6103515625000000', '6250000000000000000', '97656250000000000', '48828125000000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '3906250000000000', '6103515625000000', '6250000000000000000', '97656250000000000', '48828125000000000', '12500000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '25000000000000000000', '50000000000000000000', '25000000000000000000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '1220703125000000', '3051757812500000', '12207031250000000', '24414062500000000', '6103515625000000', '3051757812500000', '1525878906250000', '2441406250000000', '3051757812500000', '1525878906250000', '1525878906250000', '122070312500000', '6103515625000000', '3051757812500000', '6103515625000000', '12207031250000000', '1525878906250000', '762939453125000', '762939453125000', '6103515625000000', '762939453125000', '12207031250000000', '762939453125000', '1220703125000000', '6103515625000000', '1525878906250000', '488281250000000', '610351562500000', '95367431640625', '3051757812500000', '3051757812500000', '12207031250000000', '381469726562500', '12207031250000000', '3051757812500000', '610351562500000', '6103515625000000', '1220703125000000', '12207031250000000', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '1220703125000000', '1525878906250000', '12207031250000000', '762939453125000', '6103515625000000', '3051757812500000', '610351562500000', '24414062500000000', '24414062500000000', '610351562500000', '1525878906250000', '122070312500000', '3051757812500000', '381469726562500', '12207031250000000', '95367431640625', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '12207031250000000', '6103515625000000', '1525878906250000', '50000000000000000000', '48828125000000000', '12207031250000000', '1562500000000000000', '195312500000000000', '24414062500000000', '6250000000000000000', '24414062500000000', '390625000000000000', '195312500000000000', '1220703125000000', '6250000000000000000', '390625000000000000', '48828125000000000', '625000000000000000', '390625000000000000', '48828125000000000', '6250000000000000000', '5000000000000000000', '24414062500000000', '97656250000000000', '2441406250000000', '12500000000000000000', '2441406250000000', '97656250000000000', '24414062500000000', '5000000000000000000', '5000000000000000000', '146484375000000000', '292968750000000000', '29296875000000000', '12500000000000000000', '29296875000000000', '292968750000000000', '146484375000000000', '5000000000000000000', '3125000000000000000', '195312500000000000', '12207031250000000', '24414062500000000', '62500000000000000', '24414062500000000', '12207031250000000', '195312500000000000', '3125000000000000000', '3125000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '62500000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '3125000000000000000', '6250000000000000000', '195312500000000000', '97656250000000000', '1953125000000000', '3125000000000000000', '1953125000000000', '97656250000000000', '195312500000000000', '6250000000000000000', '6250000000000000000', '3125000000000000', '78125000000000000', '390625000000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '6250000000000000000', '3125000000000000000', '39062500000000000', '195312500000000000', '48828125000000000', '312500000000000000', '48828125000000000', '195312500000000000', '1525878906250000', '3125000000000000000', '3125000000000000000', '1171875000000000000', '46875000000000000', '1171875000000000000', '312500000000000000', '234375000000000000', '1171875000000000000', '234375000000000000', '3125000000000000000', '12500000000000000000', '195312500000000000', '97656250000000000', '610351562500000', '12500000000000000000', '610351562500000', '195312500000000000', '195312500000000000', '3125000000000000000', '12500000000000000000', '292968750000000000', '585937500000000000', '585937500000000000', '12500000000000000000', '585937500000000000', '292968750000000000', '14648437500000000', '3125000000000000000', '781250000000000000', '4882812500000000', '195312500000000000', '48828125000000000', '390625000000000000', '48828125000000000', '195312500000000000', '4882812500000000', '781250000000000000', '781250000000000000', '117187500000000000', '73242187500000000', '585937500000000000', '390625000000000000', '585937500000000000', '292968750000000000', '117187500000000000', '781250000000000000', '100000000000000000000', '48828125000000000', '312500000000000000', '50000000000000000000', '20000000000000000000', '25000000000000000000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '24414062500000000', '61035156250000', '152587890625000', '12207031250000000', '6103515625000000', '95367431640625', '762939453125000', '1525878906250000', '1525878906250000', '6103515625000000', '762939453125000', '3051757812500000', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '6103515625000000', '95367431640625', '1525878906250000', '610351562500000', '1525878906250000', '3051757812500000', '1525878906250000', '76293945312500', '48828125000000000', '762939453125000', '305175781250000', '24414062500000000', '6103515625000000', '76293945312500', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '762939453125000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '24414062500000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '95367431640625', '190734863281250', '1525878906250000', '3051757812500000', '3051757812500000', '152587890625000', '3051757812500000', '1220703125000000', '3051757812500000', '3051757812500000', '381469726562500', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '24414062500000000', '1220703125000000', '6103515625000000', '610351562500000', '6103515625000000', '762939453125000', '6250000000000000000', '25000000000000000000', '100000000000000000000', '195312500000000000', '12207031250000000', '78125000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '4882812500000000', '195312500000000000', '195312500000000000', '195312500000000000', '50000000000000000000', '390625000000000000', '97656250000000000', '48828125000000000', '15625000000000000', '195312500000000000', '6103515625000000', '390625000000000000', '48828125000000000', '19531250000000000', '48828125000000000', '24414062500000000', '48828125000000000', '12500000000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '97656250000000000', '97656250000000000', '390625000000000000', '19531250000000000', '97656250000000000', '781250000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '2500000000000000000', '12207031250000000', '97656250000000000', '97656250000000000', '2500000000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '2500000000000000000', '12207031250000000', '97656250000000000', '97656250000000000', '2500000000000000000', '12500000000000000000', '2441406250000000', '195312500000000000', '12500000000000000000', '3051757812500000', '195312500000000000', '97656250000000000', '3125000000000000000', '12500000000000000000', '2441406250000000', '195312500000000000', '12500000000000000000', '3051757812500000', '195312500000000000', '97656250000000000', '3125000000000000000', '12500000000000000000', '24414062500000000', '78125000000000000', '6250000000000000000', '2441406250000', '97656250000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '24414062500000000', '78125000000000000', '6250000000000000000', '2441406250000', '97656250000000000', '97656250000000000', '12500000000000000000', '3125000000000000000', '9765625000000000', '195312500000000000', '1562500000000000000', '1525878906250000', '195312500000000000', '9765625000000000', '3125000000000000000', '3125000000000000000', '9765625000000000', '195312500000000000', '1562500000000000000', '1525878906250000', '195312500000000000', '9765625000000000', '3125000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '6250000000000000000', '3051757812500000', '97656250000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '6250000000000000000', '3051757812500000', '97656250000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '390625000000000000', '6103515625000000', '1562500000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '390625000000000000', '6103515625000000', '1562500000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '5000000000000000000', '20000000000000000000', '100000000000000000000', '1562500000000000000', '50000000000000000000', '50000000000000000000', '1525878906250000', '1220703125000000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '3051757812500000', '762939453125000', '76293945312500', '6103515625000000', '6103515625000000', '762939453125000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '762939453125000', '6103515625000000', '24414062500000000', '2441406250000000', '6103515625000000', '1525878906250000', '3051757812500000', '381469726562500', '381469726562500', '3051757812500000', '152587890625000', '6103515625000000', '122070312500000', '38146972656250', '6103515625000000', '24414062500000000', '3051757812500000', '61035156250000', '1525878906250000', '305175781250000', '610351562500000', '6103515625000000', '3051757812500000', '12207031250000000', '3051757812500000', '381469726562500', '6103515625000000', '12207031250000000', '1220703125000000', '381469726562500', '4882812500000', '1525878906250000', '1525878906250000', '12207031250000000', '305175781250000', '12207031250000000', '1525878906250000', '6103515625000000', '24414062500000000', '4882812500000000', '1220703125000000', '6103515625000000', '762939453125000', '1525878906250000', '1525878906250000', '2441406250000000', '3051757812500000', '610351562500000', '95367431640625', '6103515625000000', '6103515625000000', '12207031250000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '12500000000000000000', '195312500000000000', '195312500000000000', '100000000000000000000', '12207031250000000', '97656250000000000', '390625000000000000', '3125000000000000000', '6103515625000000', '390625000000000000', '3125000000000000000', '19531250000000000', '12500000000000000000', '39062500000000000', '97656250000000000', '390625000000000000', '6250000000000000000', '19531250000000000', '1562500000000000000', '19531250000000000', '19531250000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '19531250000000000', '19531250000000000', '97656250000000000', '3125000000000000000', '97656250000000000', '19531250000000000', '19531250000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '19531250000000000', '19531250000000000', '3125000000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '195312500000000000', '6103515625000000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '3051757812500000', '390625000000000000', '976562500000000', '3051757812500000', '781250000000000000', '781250000000000000', '3051757812500000', '390625000000000000', '976562500000000', '3051757812500000', '781250000000000000', '195312500000000000', '48828125000000000', '781250000000000000', '2441406250000000', '9765625000000000', '781250000000000000', '195312500000000000', '48828125000000000', '781250000000000000', '2441406250000000', '9765625000000000', '781250000000000000', '78125000000000000', '48828125000000000', '195312500000000000', '1220703125000000', '48828125000000000', '78125000000000000', '78125000000000000', '48828125000000000', '195312500000000000', '1220703125000000', '48828125000000000', '78125000000000000', '78125000000000000', '48828125000000000', '195312500000000000', '4882812500000000', '6103515625000000', '78125000000000000', '78125000000000000', '48828125000000000', '195312500000000000', '4882812500000000', '6103515625000000', '78125000000000000', '390625000000000000', '4882812500000000', '195312500000000000', '24414062500000000', '4882812500000000', '390625000000000000', '390625000000000000', '4882812500000000', '195312500000000000', '24414062500000000', '4882812500000000', '390625000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '12207031250000000', '48828125000000000', '195312500000000000', '195312500000000000', '48828125000000000', '97656250000000000', '12207031250000000', '48828125000000000', '195312500000000000', '50000000000000000000', '100000000000000000000', '25000000000000000000', '20000000000000000000', '25000000000000000000', '2500000000000000000', '3051757812500000', '6103515625000000', '1220703125000000', '76293945312500', '3051757812500000', '6103515625000000', '6103515625000000', '9765625000000000', '12207031250000000', '1220703125000000', '3051757812500000', '6103515625000000', '610351562500000', '3051757812500000', '6103515625000000', '12207031250000000', '3051757812500000', '97656250000000000', '12207031250000000', '6103515625000000', '1525878906250000', '6103515625000000', '244140625000000', '95367431640625', '12207031250000000', '12207031250000000', '3051757812500000', '48828125000000000', '12207031250000000', '3051757812500000', '3051757812500000', '762939453125000', '6103515625000000', '3051757812500000', '122070312500000', '610351562500000', '1220703125000000', '48828125000000000', '6103515625000000', '381469726562500', '3051757812500000', '6103515625000000', '1525878906250000', '152587890625000', '12207031250000000', '12207031250000000', '12207031250000000', '97656250000000000', '12207031250000000', '6103515625000000', '190734863281250', '61035156250000', '1220703125000000', '190734863281250', '24414062500000', '6103515625000000', '2441406250000000', '97656250000000000', '1220703125000000', '3051757812500000', '390625000000000000', '6103515625000000', '390625000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '625000000000000000', '97656250000000000', '195312500000000000', '48828125000000000', '97656250000000000', '100000000000000000000', '24414062500000000', '2441406250000000', '1562500000000000000', '976562500000000', '97656250000000000', '1250000000000000000', '781250000000000000', '4882812500000000', '48828125000000000', '24414062500000000', '78125000000000000', '781250000000000000', '781250000000000000', '4882812500000000', '48828125000000000', '24414062500000000', '78125000000000000', '781250000000000000', '390625000000000000', '195312500000000000', '24414062500000000', '24414062500000000', '305175781250000', '390625000000000000', '390625000000000000', '195312500000000000', '24414062500000000', '24414062500000000', '305175781250000', '390625000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '24414062500000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '24414062500000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '4882812500000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '4882812500000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '781250000000000000', '195312500000000000', '97656250000000000', '6103515625000000', '6103515625000000', '1525878906250000', '195312500000000000', '195312500000000000', '97656250000000000', '6103515625000000', '6103515625000000', '1525878906250000', '195312500000000000', '3125000000000000000', '12500000000000000000', '50000000000000000000', '12500000000000000000', '10000000000000000000', '12500000000000000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '48828125000000000', '12207031250000000', '12207031250000000', '3051757812500000', '762939453125000', '3051757812500000', '1525878906250000', '305175781250000', '152587890625000', '12207031250000000', '195312500000000000', '38146972656250', '3051757812500000', '12207031250000000', '1220703125000000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '195312500000000000', '12207031250000000', '12207031250000000', '12207031250000000', '305175781250000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '12207031250000000', '48828125000000000', '12207031250000000', '12207031250000000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '19531250000000000', '12207031250000000', '12207031250000000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '762939453125000', '6103515625000000', '97656250000000000', '3051757812500000', '12207031250000000', '12207031250000000', '152587890625000', '1562500000000000000', '390625000000000000', '195312500000000000', '781250000000000000', '195312500000000000', '195312500000000000', '390625000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '195312500000000000', '390625000000000000', '195312500000000000', '390625000000000000', '100000000000000000000', '97656250000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '195312500000000000', '4882812500000000', '48828125000000000', '4882812500000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '4882812500000000', '48828125000000000', '781250000000000000', '781250000000000000', '31250000000000000', '6103515625000000', '195312500000000000', '48828125000000000', '31250000000000000', '48828125000000000', '48828125000000000', '31250000000000000', '6103515625000000', '31250000000000000', '24414062500000000', '12207031250000000', '244140625000000', '24414062500000000', '24414062500000000', '12207031250000000', '244140625000000', '24414062500000000', '48828125000000000', '48828125000000000', '12207031250000000', '2441406250000000', '48828125000000000', '48828125000000000', '12207031250000000', '2441406250000000', '48828125000000000', '9765625000000000', '12207031250000000', '6103515625000000', '48828125000000000', '9765625000000000', '12207031250000000', '6103515625000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '48828125000000000', '24414062500000000', '762939453125000', '48828125000000000', '48828125000000000', '24414062500000000', '762939453125000', '48828125000000000', '9765625000000000', '24414062500000000', '12207031250000000', '9765625000000000', '9765625000000000', '24414062500000000', '12207031250000000', '9765625000000000', '78125000000000000', '625000000000000000', '625000000000000000', '312500000000000000', '781250000000000000', '97656250000000000', '3051757812500000', '12207031250000000', '12207031250000000', '24414062500000000', '48828125000000000', '976562500000000', '4882812500000000', '24414062500000000', '1220703125000000', '6103515625000000', '12207031250000000', '6103515625000000', '48828125000000000', '12207031250000000', '6103515625000000', '6103515625000000', '2441406250000000', '12207031250000000', '12207031250000000', '12207031250000000', '4882812500000000', '9765625000000000', '6103515625000000', '24414062500000000', '2441406250000000', '2441406250000000', '1525878906250000', '1953125000000000', '24414062500000000', '24414062500000000', '4882812500000000', '1525878906250000', '12207031250000000', '6103515625000000', '12207031250000000', '48828125000000000', '1525878906250000', '6103515625000000', '1525878906250000', '24414062500000000', '3051757812500000', '12207031250000000', '12207031250000000', '24414062500000000', '48828125000000000', '24414062500000000', '48828125000000000', '24414062500000000', '195312500000000000', '390625000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '19531250000000000', '781250000000000000', '48828125000000000', '39062500000000000', '6103515625000000', '781250000000000000', '6103515625000000', '6103515625000000', '781250000000000000', '2441406250000000', '781250000000000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '24414062500000000', '24414062500000000', '24414062500000000', '24414062500000000', '1', '1', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '781250000000000000', '195312500000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '781250000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '39062500000000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '390625000000000000', '12207031250000000', '12207031250000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '1562500000000000000', '39062500000000000', '39062500000000000', '3125000000000000000', '19531250000000000', '3125000000000000000', '39062500000000000', '39062500000000000', '19531250000000000', '1562500000000000000', '19531250000000000', '39062500000000000', '39062500000000000', '3125000000000000000', '19531250000000000', '3125000000000000000', '39062500000000000', '39062500000000000', '781250000000000000', '9765625000000000', '390625000000000000', '390625000000000000', '9765625000000000', '48828125000000000', '12207031250000000', '390625000000000000', '12207031250000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '390625000000000000', '9765625000000000', '9765625000000000', '48828125000000000', '48828125000000000', '6103515625000000', '24414062500000000', '48828125000000000', '24414062500000000', '24414062500000000', '48828125000000000', '48828125000000000', '48828125000000000', '24414062500000000', '24414062500000000', '24414062500000000', '24414062500000000', '1', '12207031250000000', '3051757812500000', '24414062500000000', '24414062500000000', '1562500000000000000', '12207031250000000', '3125000000000000000', '3125000000000000000', '1562500000000000000', '12207031250000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '2441406250000000', '24414062500000000', '3125000000000000000', '3125000000000000000', '2441406250000000', '24414062500000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '1220703125000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '1220703125000000', '1562500000000000000', '50000000000000000000', '50000000000000000000', '3051757812500000', '5000000000000000000', '50000000000000000000', '50000000000000000000', '3051757812500000', '5000000000000000000', '12500000000000000000', '6103515625000000', '125000000000000000', '12500000000000000000', '12500000000000000000', '6103515625000000', '125000000000000000', '12500000000000000000', '5000000000000000000', '12500000000000000000', '12207031250000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '12207031250000000', '25000000000000000000', '50000000000000000000', '2500000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '312500000000000000', '6103515625000000', '1525878906250000', '6103515625000000', '1220703125000000', '61035156250000', '190734863281250', '1525878906250000', '95367431640625', '6103515625000000', '6103515625000000', '24414062500000', '6103515625000000', '1525878906250000', '305175781250000', '762939453125000', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '12207031250000000', '76293945312500', '1525878906250000', '610351562500000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '190734863281250', '1525878906250000', '610351562500000', '1525878906250000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '762939453125000', '305175781250000', '3051757812500000', '244140625000000', '762939453125000', '762939453125000', '6103515625000000', '1525878906250000', '381469726562500', '762939453125000', '3051757812500000', '195312500000000000', '39062500000000000', '195312500000000000', '781250000000000000', '195312500000000000', '97656250000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '25000000000000000000', '100000000000000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '25000000000000000000', '100000000000000000000', '2500000000000000000', '12500000000000000000', '48828125000000000', '1525878906250000', '12207031250000000', '1562500000000000000', '2500000000000000000', '12500000000000000000', '48828125000000000', '1525878906250000', '12207031250000000', '1562500000000000000', '12500000000000000000', '50000000000000000000', '610351562500000', '9765625000000000', '6103515625000000', '50000000000000000000', '12500000000000000000', '50000000000000000000', '610351562500000', '9765625000000000', '6103515625000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '9765625000000000', '6103515625000000', '976562500000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '9765625000000000', '6103515625000000', '976562500000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '1953125000000000', '1220703125000000', '12207031250000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '1953125000000000', '1220703125000000', '12207031250000000', '25000000000000000000', '25000000000000000000', '1562500000000000000', '48828125000000000', '24414062500000000', '6103515625000000', '25000000000000000000', '25000000000000000000', '1562500000000000000', '48828125000000000', '24414062500000000', '6103515625000000', '25000000000000000000', '100000000000000000000', '4000000000000000000', '2500000000000000000', '25000000000000000000', '12500000000000000000', '50000000000000000000', '6103515625000000', '1525878906250000', '3051757812500000', '12207031250000000', '244140625000000', '190734863281250', '1525878906250000', '1525878906250000', '95367431640625', '190734863281250', '6103515625000000', '3051757812500000', '3051757812500000', '2441406250000000', '6103515625000000', '152587890625000', '3051757812500000', '3051757812500000', '1525878906250000', '610351562500000', '1525878906250000', '3051757812500000', '6103515625000000', '4882812500000000', '6103515625000000', '1525878906250000', '381469726562500', '381469726562500', '1525878906250000', '3051757812500000', '3051757812500000', '610351562500000', '3051757812500000', '6103515625000000', '3051757812500000', '381469726562500', '1525878906250000', '76293945312500', '95367431640625', '3051757812500000', '381469726562500', '1220703125000000', '1525878906250000', '12207031250000000', '38146972656250', '305175781250000', '1525878906250000', '610351562500000', '1525878906250000', '3051757812500000', '6103515625000000', '190734863281250', '3051757812500000', '762939453125000', '244140625000000', '1525878906250000', '3051757812500000', '1525878906250000', '1525878906250000', '3051757812500000', '195312500000000000', '100000000000000000000', '48828125000000000', '390625000000000000', '97656250000000000', '12207031250000000', '97656250000000000', '2500000000000000000', '97656250000000000', '195312500000000000', '390625000000000000', '97656250000000000', '390625000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '78125000000000000', '195312500000000000', '1250000000000000000', '24414062500000000', '6250000000000000000', '390625000000000000', '390625000000000000', '95367431640625', '97656250000000000', '625000000000000000', '1250000000000000000', '24414062500000000', '6250000000000000000', '390625000000000000', '390625000000000000', '95367431640625', '97656250000000000', '625000000000000000', '2000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '195312500000000000', '25000000000000000000', '97656250000000000', '50000000000000000000', '2000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '195312500000000000', '25000000000000000000', '97656250000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '50000000000000000000', '78125000000000000', '48828125000000000', '3051757812500000', '24414062500000000', '25000000000000000000', '50000000000000000000', '97656250000000000', '50000000000000000000', '78125000000000000', '48828125000000000', '3051757812500000', '24414062500000000', '25000000000000000000', '500000000000000000', '97656250000000000', '2500000000000000000', '390625000000000000', '78125000000000000', '2441406250000000', '97656250000000000', '3125000000000000000', '500000000000000000', '97656250000000000', '2500000000000000000', '390625000000000000', '78125000000000000', '2441406250000000', '97656250000000000', '3125000000000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '48828125000000000', '25000000000000000000', '12207031250000000', '19531250000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '48828125000000000', '25000000000000000000', '12207031250000000', '19531250000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '5000000000000000000', '50000000000000000000', '25000000000000000000', '100000000000000000000', '800000000000000000', '50000000000000000000', '3051757812500000', '3051757812500000', '305175781250000', '6103515625000000', '24414062500000000', '6103515625000000', '61035156250000', '6103515625000000', '762939453125000', '6103515625000000', '762939453125000', '3051757812500000', '305175781250000', '244140625000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '305175781250000', '6103515625000000', '1525878906250000', '762939453125000', '190734863281250', '3051757812500000', '6103515625000000', '1220703125000000', '1220703125000000', '24414062500000000', '1220703125000000', '610351562500000', '1525878906250000', '6103515625000000', '244140625000', '3051757812500000', '762939453125000', '1525878906250000', '3051757812500000', '6103515625000000', '610351562500000', '976562500000000', '12207031250000000', '381469726562500', '76293945312500', '610351562500000', '762939453125000', '762939453125000', '762939453125000', '1525878906250000', '3051757812500000', '762939453125000', '3051757812500000', '12207031250000000', '24414062500000000', '1525878906250000', '1525878906250000', '6103515625000000', '1525878906250000', '190734863281250', '1525878906250000', '381469726562500', '6103515625000000', '24414062500000', '3051757812500000', '76293945312500', '24414062500000000', '1525878906250000', '381469726562500', '381469726562500', '3051757812500000', '3051757812500000', '762939453125000', '1525878906250000', '390625000000000000', '9765625000000000', '6250000000000000000', '3125000000000000000', '39062500000000000', '390625000000000000', '39062500000000000', '48828125000000000', '1562500000000000000', '390625000000000000', '381469726562500', '12500000000000000000', '3125000000000000000', '6103515625000000', '19531250000000000', '78125000000000000', '195312500000000000', '48828125000000000', '390625000000000000', '6250000000000000000', '24414062500000000', '1562500000000000000', '6250000000000000000', '24414062500000000', '12207031250000000', '97656250000000000', '390625000000000000', '6250000000000000000', '24414062500000000', '1562500000000000000', '6250000000000000000', '24414062500000000', '12207031250000000', '97656250000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '6250000000000000000', '6250000000000000000', '48828125000000000', '6103515625000000', '390625000000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '6250000000000000000', '6250000000000000000', '48828125000000000', '6103515625000000', '390625000000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '6250000000000000000', '6250000000000000000', '48828125000000000', '12207031250000000', '390625000000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '6250000000000000000', '6250000000000000000', '48828125000000000', '12207031250000000', '390625000000000000', '390625000000000000', '12500000000000000000', '48828125000000000', '6250000000000000000', '2500000000000000000', '48828125000000000', '2441406250000000', '390625000000000000', '390625000000000000', '12500000000000000000', '48828125000000000', '6250000000000000000', '2500000000000000000', '48828125000000000', '2441406250000000', '390625000000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '6250000000000000000', '6250000000000000000', '97656250000000000', '152587890625000', '195312500000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '6250000000000000000', '6250000000000000000', '97656250000000000', '152587890625000', '195312500000000000', '390625000000000000', '1562500000000000000', '976562500000000', '3125000000000000000', '3125000000000000000', '976562500000000', '3051757812500000', '390625000000000000', '390625000000000000', '1562500000000000000', '976562500000000', '3125000000000000000', '3125000000000000000', '976562500000000', '3051757812500000', '390625000000000000', '800000000000000000', '12500000000000000000', '10000000000000000000', '2500000000000000000', '6250000000000000000', '100000000000000000000', '6103515625000000', '3051757812500000', '3051757812500000', '2441406250000000', '610351562500000', '6103515625000000', '762939453125000', '3051757812500000', '12207031250000000', '24414062500000000', '6103515625000000', '1525878906250000', '610351562500000', '3051757812500000', '1525878906250000', '12207031250000000', '3051757812500000', '12207031250000000', '1525878906250000', '3051757812500000', '12207031250000000', '381469726562500', '3051757812500000', '762939453125000', '381469726562500', '6103515625000000', '610351562500000', '6103515625000000', '610351562500000', '6103515625000000', '3051757812500000', '6103515625000000', '24414062500000000', '24414062500000000', '6103515625000000', '3051757812500000', '610351562500000', '1525878906250000', '3051757812500000', '12207031250000000', '610351562500000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '1220703125000000', '1525878906250000', '3051757812500000', '3051757812500000', '38146972656250', '610351562500000', '3051757812500000', '1525878906250000', '12207031250000000', '3051757812500000', '1220703125000000', '6103515625000000', '2441406250000000', '6103515625000000', '1525878906250000', '762939453125000', '1220703125000000', '3051757812500000', '12207031250000000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '24414062500000000', '6103515625000000', '1525878906250000', '97656250000000000', '195312500000000000', '12500000000000000000', '19531250000000000', '1250000000000000000', '1525878906250000', '97656250000000000', '390625000000000000', '3125000000000000000', '97656250000000000', '390625000000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '48828125000000000', '48828125000000000', '3125000000000000000', '195312500000000000', '6103515625000000', '78125000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '6103515625000000', '6103515625000000', '78125000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '6103515625000000', '6103515625000000', '4882812500000000', '781250000000000000', '781250000000000000', '390625000000000000', '6103515625000000', '6103515625000000', '4882812500000000', '781250000000000000', '781250000000000000', '390625000000000000', '6103515625000000', '48828125000000000', '24414062500000000', '390625000000000000', '390625000000000000', '39062500000000000', '1220703125000000', '48828125000000000', '24414062500000000', '390625000000000000', '390625000000000000', '39062500000000000', '1220703125000000', '24414062500000000', '156250000000000000', '781250000000000000', '12207031250000000', '381469726562500', '24414062500000000', '24414062500000000', '156250000000000000', '781250000000000000', '12207031250000000', '381469726562500', '24414062500000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '12207031250000000', '195312500000000000', '781250000000000000', '781250000000000000', '3051757812500000', '24414062500000000', '12207031250000000', '195312500000000000', '781250000000000000', '781250000000000000', '3051757812500000', '24414062500000000', '25000000000000000000', '100000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '100000000000000000000', '12207031250000000', '762939453125000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '195312500000000000', '610351562500000', '6103515625000000', '305175781250000', '1525878906250000', '1525878906250000', '244140625000000', '19073486328125', '1525878906250000', '2441406250000000', '97656250000000000', '12207031250000000', '1220703125000000', '12207031250000000', '6103515625000000', '610351562500000', '48828125000000', '1525878906250000', '3051757812500000', '6103515625000000', '97656250000000000', '244140625000000', '61035156250000', '3051757812500000', '12207031250000000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '12207031250000000', '48828125000000000', '1220703125000000', '6103515625000000', '12207031250000000', '12207031250000000', '381469726562500', '610351562500000', '6103515625000000', '1525878906250000', '12207031250000000', '24414062500000000', '3051757812500000', '6103515625000000', '12207031250000000', '12207031250000000', '122070312500000', '762939453125000', '1220703125000000', '1525878906250000', '12207031250000000', '12207031250000000', '3051757812500000', '6103515625000000', '195312500000000000', '390625000000000000', '78125000000000000', '97656250000000000', '100000000000000000000', '24414062500000000', '195312500000000000', '195312500000000000', '6250000000000000000', '1525878906250000', '97656250000000000', '195312500000000000', '24414062500000000', '3125000000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '195312500000000000', '6103515625000000', '48828125000000000', '48828125000000000', '24414062500000000', '6103515625000000', '48828125000000000', '48828125000000000', '24414062500000000', '48828125000000000', '48828125000000000', '24414062500000000', '12207031250000000', '48828125000000000', '48828125000000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '24414062500000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '24414062500000000', '9765625000000000', '9765625000000000', '1525878906250000', '24414062500000000', '9765625000000000', '9765625000000000', '1525878906250000', '12207031250000000', '24414062500000000', '24414062500000000', '244140625000000', '12207031250000000', '24414062500000000', '24414062500000000', '244140625000000', '1562500000000000000', '1562500000000000000', '1250000000000000000', '1562500000000000000', '1562500000000000000', '6250000000000000000', '24414062500000000', '24414062500000000', '488281250000000', '2441406250000000', '488281250000000', '48828125000000000', '48828125000000000', '12207031250000000', '6103515625000000', '24414062500000000', '12207031250000000', '1220703125000000', '488281250000000', '976562500000000', '3051757812500000', '24414062500000000', '9765625000000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '24414062500000000', '12207031250000000', '48828125000000000', '48828125000000000', '6103515625000000', '12207031250000000', '2441406250000000', '48828125000000000', '48828125000000000', '24414062500000000', '48828125000000000', '48828125000000000', '30517578125000', '610351562500000', '12207031250000000', '1220703125000000', '48828125000000000', '12207031250000000', '9765625000000000', '24414062500000000', '3051757812500000', '12207031250000000', '244140625000000', '976562500000000', '9765625000000000', '24414062500000000', '97656250000000000', '390625000000000000', '78125000000000000', '97656250000000000', '78125000000000000', '390625000000000000', '4', '4', '4', '4', '195312500000000000', '24414062500000000', '195312500000000000', '48828125000000000', '48828125000000000', '195312500000000000', '48828125000000000', '48828125000000000', '24414062500000000', '195312500000000000', '2500000000000000000', '195312500000000', '12207031250000000', '25000000000000000000', '6103515625000000', '25000000000000000000', '2500000000000000000', '195312500000000', '12207031250000000', '25000000000000000000', '6103515625000000', '25000000000000000000', '100000000000000000000', '48828125000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '12500000000000000000', '100000000000000000000', '48828125000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '12500000000000000000', '800000000000000000', '24414062500000000', '10000000000000000000', '24414062500000000', '24414062500000000', '20000000000000000000', '800000000000000000', '24414062500000000', '10000000000000000000', '24414062500000000', '24414062500000000', '20000000000000000000', '31250000000000000', '6103515625000000', '50000000000000000000', '24414062500000000', '48828125000000000', '10000000000000000000', '31250000000000000', '6103515625000000', '50000000000000000000', '24414062500000000', '48828125000000000', '10000000000000000000', '25000000000000000000', '48828125000000000', '25000000000000000000', '4882812500000000', '12207031250000000', '2500000000000000000', '25000000000000000000', '48828125000000000', '25000000000000000000', '4882812500000000', '12207031250000000', '2500000000000000000', '781250000000000000', '24414062500000000', '12500000000000000000', '24414062500000000', '48828125000000000', '2500000000000000000', '781250000000000000', '24414062500000000', '12500000000000000000', '24414062500000000', '48828125000000000', '2500000000000000000', '10000000000000000000', '50000000000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '6103515625000000', '762939453125000', '1525878906250000', '1525878906250000', '1525878906250000', '3051757812500000', '95367431640625', '1220703125000000', '6103515625000000', '6103515625000000', '1525878906250000', '381469726562500', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '1525878906250000', '2441406250000000', '6103515625000000', '6103515625000000', '6103515625000000', '762939453125000', '95367431640625', '1525878906250000', '381469726562500', '762939453125000', '1525878906250000', '12207031250000000', '6103515625000000', '6103515625000000', '1525878906250000', '305175781250000', '1525878906250000', '762939453125000', '762939453125000', '3051757812500000', '3051757812500000', '4882812500000000', '6103515625000000', '1525878906250000', '1220703125000000', '762939453125000', '152587890625000', '1525878906250000', '1525878906250000', '1525878906250000', '244140625000000', '24414062500000000', '6103515625000000', '1220703125000000', '1525878906250000', '762939453125000', '1525878906250000', '3051757812500000', '1525878906250000', '610351562500000', '195312500000000000', '97656250000000000', '6250000000000000000', '12207031250000000', '48828125000000000', '50000000000000000000', '48828125000000000', '9765625000000000', '20000000000000000000', '781250000000000000', '97656250000000000', '390625000000000000', '195312500000000000', '24414062500000000', '390625000000000000', '1525878906250000', '97656250000000000', '24414062500000000', '100000000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '100000000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '50000000000000000000', '100000000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '100000000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '50000000000000000000', '10000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '50000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '25000000000000000000', '10000000000000000000', '234375000000000000', '1171875000000000000', '1171875000000000000', '50000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '24414062500000000', '48828125000000000', '100000000000000000000', '48828125000000000', '24414062500000000', '195312500000000000', '100000000000000000000', '25000000000000000000', '292968750000000000', '585937500000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '585937500000000000', '73242187500000000', '100000000000000000000', '10000000000000000000', '24414062500000000', '4882812500000000', '19531250000000000', '100000000000000000000', '6103515625000000', '4882812500000000', '24414062500000000', '100000000000000000000', '10000000000000000000', '29296875000000000', '146484375000000000', '292968750000000000', '100000000000000000000', '146484375000000000', '146484375000000000', '146484375000000000', '100000000000000000000', '5000000000000000000', '195312500000000000', '12207031250000000', '610351562500000', '100000000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '100000000000000000000', '5000000000000000000', '390625000000000000', '390625000000000000', '78125000000000000', '100000000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '100000000000000000000', '100000000000000000000', '97656250000000000', '24414062500000000', '195312500000000000', '50000000000000000000', '195312500000000000', '24414062500000000', '19531250000000000', '20000000000000000000', '100000000000000000000', '1171875000000000000', '234375000000000000', '1171875000000000000', '50000000000000000000', '1171875000000000000', '9375000000000000', '1171875000000000000', '20000000000000000000', '12500000000000000000', '50000000000000000000', '781250000000000000', '25000000000000000000', '50000000000000000000', '25000000000000000000', '6103515625000000', '24414062500000000', '381469726562500', '1220703125000000', '4882812500000000', '3051757812500000', '15258789062500', '3051757812500000', '152587890625000', '762939453125000', '1525878906250000', '3051757812500000', '3051757812500000', '4882812500000000', '6103515625000000', '3051757812500000', '190734863281250', '190734863281250', '1525878906250000', '1220703125000000', '762939453125000', '6103515625000000', '1525878906250000', '3051757812500000', '122070312500000', '4882812500000000', '6103515625000000', '381469726562500', '12207031250000000', '381469726562500', '76293945312500', '305175781250000', '1525878906250000', '762939453125000', '762939453125000', '1525878906250000', '1525878906250000', '381469726562500', '1525878906250000', '6103515625000000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '305175781250000', '1525878906250000', '381469726562500', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '762939453125000', '3051757812500000', '95367431640625', '6103515625000000', '1525878906250000', '1525878906250000', '6103515625000000', '24414062500000000', '1525878906250000', '1525878906250000', '24414062500000000', '6103515625000000', '762939453125000', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '762939453125000', '195312500000000000', '195312500000000000', '97656250000000000', '48828125000000000', '78125000000000000', '781250000000000000', '48828125000000000', '195312500000000000', '97656250000000000', '195312500000000000', '10000000000000000000', '390625000000000000', '19531250000000000', '390625000000000000', '2441406250000000', '24414062500000000', '390625000000000000', '6250000000000000000', '48828125000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '19531250000000000', '390625000000000000', '781250000000000000', '5000000000000000000', '24414062500000000', '39062500000000000', '50000000000000000000', '195312500000000000', '195312500000000000', '12500000000000000000', '7812500000000000', '195312500000000000', '125000000000000000', '2441406250000000', '6103515625000000', '12207031250000000', '390625000000000000', '3125000000000000000', '1562500000000000000', '24414062500000000', '3125000000000000000', '390625000000000000', '12207031250000000', '6103515625000000', '12500000000000000000', '156250000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '100000000000000000', '12500000000000000000', '156250000000000000', '12500000000000000000', '781250000000000000', '12207031250000000', '24414062500000000', '3125000000000000000', '62500000000000000', '24414062500000000', '3125000000000000000', '24414062500000000', '48828125000000000', '781250000000000000', '625000000000000000', '6250000000000000000', '6250000000000000000', '3125000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '625000000000000000', '6250000000000000000', '6250000000000000000', '78125000000000000', '2441406250000000', '195312500000000000', '3125000000000000000', '1562500000000000000', '6103515625000000', '3125000000000000000', '195312500000000000', '4882812500000000', '78125000000000000', '5000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '500000000000000000', '5000000000000000000', '9765625000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '312500000000000000', '4882812500000000', '3125000000000000000', '195312500000000000', '48828125000000000', '9765625000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '195312500000000000', '781250000000000000', '78125000000000000', '12207031250000000', '781250000000000000', '390625000000000000', '48828125000000000', '390625000000000000', '12500000000000000000', '19531250000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '19531250000000000', '12500000000000000000', '156250000000000000', '24414062500000000', '390625000000000000', '312500000000000000', '781250000000000000', '24414062500000000', '312500000000000000', '390625000000000000', '24414062500000000', '156250000000000000', '3125000000000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '781250000000000000', '625000000000000000', '312500000000000000', '3125000000000000000', '3125000000000000000', '156250000000000000', '6250000000000000000', '50000000000000000000', '50000000000000000000', '24414062500000000', '1562500000000000000', '1250000000000000000', '305175781250000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '61035156250000', '3051757812500000', '610351562500000', '3051757812500000', '1220703125000000', '1525878906250000', '152587890625000', '1525878906250000', '48828125000000000', '6103515625000000', '3051757812500000', '305175781250000', '6103515625000000', '95367431640625', '190734863281250', '1525878906250000', '6103515625000000', '190734863281250', '95367431640625', '6103515625000000', '1220703125000000', '6103515625000000', '3051757812500000', '6103515625000000', '381469726562500', '152587890625000', '610351562500000', '6103515625000000', '95367431640625', '3051757812500000', '381469726562500', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '381469726562500', '95367431640625', '6103515625000000', '6103515625000000', '1525878906250000', '762939453125000', '6103515625000000', '24414062500000000', '6103515625000000', '1525878906250000', '762939453125000', '762939453125000', '3051757812500000', '95367431640625', '6103515625000000', '3051757812500000', '381469726562500', '305175781250000', '6103515625000000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '1953125000000000', '97656250000000000', '390625000000000000', '24414062500000000', '100000000000000000000', '78125000000000000', '24414062500000000', '9765625000000000', '390625000000000000', '195312500000000000', '390625000000000000', '6250000000000000000', '48828125000000000', '12207031250000000', '78125000000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '4882812500000000', '97656250000000000', '195312500000000000', '195312500000000000', '390625000000000000', '390625000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '2441406250000000', '390625000000000000', '20000000000000000000', '39062500000000000', '24414062500000000', '390625000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '244140625000000', '97656250000000000', '25000000000000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '244140625000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '3125000000000000000', '1171875000000000000', '1171875000000000000', '48828125000000000', '97656250000000000', '12500000000000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '48828125000000000', '234375000000000000', '1171875000000000000', '12500000000000000000', '234375000000000000', '6250000000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '1171875000000000000', '9765625000000000', '97656250000000000', '25000000000000000000', '48828125000000000', '2500000000000000000', '48828125000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '234375000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '2500000000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '48828125000000000', '97656250000000000', '12500000000000000000', '97656250000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '46875000000000000', '1171875000000000000', '12500000000000000000', '234375000000000000', '25000000000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '12207031250000000', '39062500000000000', '6250000000000000000', '48828125000000000', '12500000000000000000', '48828125000000000', '12500000000000000000', '39062500000000000', '12207031250000000', '585937500000000000', '29296875000000000', '6250000000000000000', '585937500000000000', '12500000000000000000', '585937500000000000', '12500000000000000000', '58593750000000000', '585937500000000000', '12207031250000000', '195312500000000000', '6250000000000000000', '195312500000000000', '625000000000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '12207031250000000', '1171875000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '625000000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '1171875000000000000', '50000000000000000000', '800000000000000000', '12207031250000000', '25000000000000000000', '100000000000000000000', '25000000000000000000', '6103515625000000', '24414062500000000', '1220703125000000', '1525878906250000', '6103515625000000', '95367431640625', '3051757812500000', '305175781250000', '3051757812500000', '24414062500000000', '190734863281250', '1525878906250000', '762939453125000', '24414062500000000', '6103515625000000', '1525878906250000', '6103515625000000', '3051757812500000', '6103515625000000', '190734863281250', '3051757812500000', '6103515625000000', '1525878906250000', '381469726562500', '3051757812500000', '12207031250000000', '6103515625000000', '95367431640625', '610351562500000', '3051757812500000', '1525878906250000', '1525878906250000', '152587890625000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '305175781250000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '762939453125000', '6103515625000000', '24414062500000000', '122070312500000', '381469726562500', '610351562500000', '24414062500000000', '381469726562500', '762939453125000', '3051757812500000', '1525878906250000', '6103515625000000', '38146972656250', '6103515625000000', '305175781250000', '6103515625000000', '3051757812500000', '152587890625000', '24414062500000000', '1525878906250000', '762939453125000', '3051757812500000', '610351562500000', '3051757812500000', '762939453125000', '3051757812500000', '6103515625000000', '3051757812500000', '762939453125000', '195312500000000000', '24414062500000000', '195312500000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '97656250000000000', '24414062500000000', '78125000000000000', '195312500000000000', '78125000000000000', '1562500000000000000', '97656250000000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '10000000000000000000', '390625000000000000', '97656250000000000', '195312500000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '390625000000000000', '48828125000000000', '97656250000000000', '390625000000000000', '48828125000000000', '390625000000000000', '12207031250000000', '48828125000000000', '195312500000000000', '50000000000000000000', '19531250000000000', '78125000000000000', '97656250000000000', '24414062500000000', '781250000000000000', '390625000000000000', '24414062500000000', '781250000000000000', '24414062500000000', '24414062500000000', '781250000000000000', '390625000000000000', '24414062500000000', '781250000000000000', '24414062500000000', '48828125000000000', '781250000000000000', '78125000000000000', '24414062500000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '78125000000000000', '24414062500000000', '781250000000000000', '48828125000000000', '9765625000000000', '156250000000000000', '390625000000000000', '2441406250000000', '156250000000000000', '12207031250000000', '9765625000000000', '156250000000000000', '390625000000000000', '2441406250000000', '156250000000000000', '12207031250000000', '12207031250000000', '390625000000000000', '195312500000000000', '1220703125000000', '390625000000000000', '48828125000000000', '12207031250000000', '390625000000000000', '195312500000000000', '1220703125000000', '390625000000000000', '48828125000000000', '4882812500000000', '195312500000000000', '97656250000000000', '4882812500000000', '195312500000000000', '4882812500000000', '4882812500000000', '195312500000000000', '97656250000000000', '4882812500000000', '195312500000000000', '4882812500000000', '24414062500000000', '781250000000000000', '78125000000000000', '24414062500000000', '156250000000000000', '48828125000000000', '24414062500000000', '781250000000000000', '78125000000000000', '24414062500000000', '156250000000000000', '48828125000000000', '50000000000000000000', '25000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '20000000000000000000', '3051757812500000', '4882812500000000', '6103515625000000', '3051757812500000', '1220703125000000', '1220703125000000', '3051757812500000', '3051757812500000', '12207031250000000', '762939453125000', '6103515625000000', '19531250000000000', '6103515625000000', '610351562500000', '1220703125000000', '610351562500000', '1525878906250000', '3051757812500000', '6103515625000000', '3814697265625', '762939453125000', '4882812500000000', '12207031250000000', '30517578125000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '762939453125000', '3051757812500000', '762939453125000', '3906250000000000', '2441406250000000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '12207031250000000', '190734863281250', '6103515625000000', '12207031250000000', '762939453125000', '12207031250000000', '3051757812500000', '3051757812500000', '6103515625000000', '610351562500000', '12207031250000000', '12207031250000000', '6103515625000000', '12207031250000000', '195312500000000000', '6103515625000000', '3051757812500000', '762939453125000', '6103515625000000', '3051757812500000', '6103515625000000', '244140625000000', '3051757812500000', '97656250000000000', '390625000000000000', '390625000000000000', '19531250000000000', '39062500000000000', '3125000000000000000', '195312500000000000', '156250000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '390625000000000000', '12500000000000000000', '195312500000000000', '781250000000000000', '390625000000000000', '390625000000000000', '95367431640625', '390625000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '48828125000000000', '48828125000000000', '3051757812500000', '390625000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '4882812500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '12207031250000000', '781250000000000000', '12207031250000000', '12207031250000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '25000000000000000000', '390625000000000000', '19531250000000000', '3051757812500000', '50000000000000000000', '97656250000000000', '390625000000000000', '50000000000000000000', '25000000000000000000', '390625000000000000', '19531250000000000', '3051757812500000', '50000000000000000000', '97656250000000000', '390625000000000000', '50000000000000000000', '12500000000000000000', '97656250000000000', '19531250000000000', '100000000000000000000', '2441406250000000', '19531250000000000', '97656250000000000', '20000000000000000000', '12500000000000000000', '97656250000000000', '19531250000000000', '100000000000000000000', '2441406250000000', '19531250000000000', '97656250000000000', '20000000000000000000', '10000000000000000000', '390625000000000000', '97656250000000000', '762939453125000', '5000000000000000000', '9765625000000000', '195312500000000000', '50000000000000000000', '10000000000000000000', '390625000000000000', '97656250000000000', '762939453125000', '5000000000000000000', '9765625000000000', '195312500000000000', '50000000000000000000', '100000000000000000000', '78125000000000000', '48828125000000000', '50000000000000000000', '1525878906250000', '48828125000000000', '390625000000000000', '100000000000000000000', '100000000000000000000', '78125000000000000', '48828125000000000', '50000000000000000000', '1525878906250000', '48828125000000000', '390625000000000000', '100000000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '5000000000000000000', '2441406250000000', '24414062500000000', '3125000000000000', '1562500000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '5000000000000000000', '2441406250000000', '24414062500000000', '3125000000000000', '1562500000000000000', '6250000000000000000', '97656250000000000', '24414062500000000', '12207031250000000', '2500000000000000000', '97656250000000000', '48828125000000000', '2500000000000000000', '6250000000000000000', '97656250000000000', '24414062500000000', '12207031250000000', '2500000000000000000', '97656250000000000', '48828125000000000', '2500000000000000000', '100000000000000000000', '50000000000000000000', '10000000000000000000', '100000000000000000000', '50000000000000000000', '50000000000000000000', '610351562500000', '12207031250000000', '2441406250000000', '381469726562500', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '190734863281250', '381469726562500', '15258789062500', '1525878906250000', '6103515625000000', '3051757812500000', '4882812500000000', '3051757812500000', '6103515625000000', '6103515625000000', '762939453125000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '610351562500000', '6103515625000000', '24414062500000000', '976562500000000', '244140625000000', '3051757812500000', '6103515625000000', '190734863281250', '6103515625000000', '95367431640625', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '24414062500000000', '12207031250000000', '610351562500000', '381469726562500', '1220703125000000', '1525878906250000', '6103515625000000', '95367431640625', '6103515625000000', '38146972656250', '3051757812500000', '6103515625000000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '15258789062500', '1525878906250000', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '6103515625000000', '24414062500000000', '24414062500000000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '1220703125000000', '3051757812500000', '305175781250000', '762939453125000', '152587890625000', '24414062500000000', '250000000000000000', '97656250000000000', '78125000000000000', '12500000000000000000', '6103515625000000', '97656250000000000', '625000000000000000', '9765625000000000', '12500000000000000000', '195312500000000000', '24414062500000000', '312500000000000000', '390625000000000000', '195312500000000000', '1562500000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '312500000000000000', '48828125000000000', '24414062500000000', '4882812500000000', '390625000000000000', '390625000000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '156250000000000000', '781250000000000000', '390625000000000000', '1562500000000000000', '24414062500000000', '24414062500000000', '12207031250000000', '3906250000000000', '781250000000000000', '1562500000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '625000000000000000', '781250000000000000', '9765625000000000', '12207031250000000', '48828125000000000', '48828125000000000', '78125000000000000', '9765625000000000', '781250000000000000', '3125000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '6250000000000000000', '12500000000000000000', '12500000000000000000', '500000000000000000', '12500000000000000000', '781250000000000000', '48828125000000000', '97656250000000000', '390625000000000000', '1953125000000000', '1953125000000000', '24414062500000000', '97656250000000000', '48828125000000000', '781250000000000000', '6250000000000000000', '50000000000000000', '6250000000000000000', '6250000000000000000', '1562500000000000000', '1562500000000000000', '1250000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '3125000000000000000', '781250000000000000', '390625000000000000', '1562500000000000000', '4882812500000000', '24414062500000000', '24414062500000000', '390625000000000000', '781250000000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '195312500000000000', '390625000000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '1220703125000000', '195312500000000000', '390625000000000000', '3125000000000000000', '5000000000000000000', '1000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '50000000000000000000', '3051757812500000', '10000000000000000000', '12500000000000000000', '390625000000000000', '1562500000000000000', '15258789062500', '6103515625000000', '1525878906250000', '762939453125000', '3051757812500000', '48828125000000000', '305175781250000', '762939453125000', '1525878906250000', '1525878906250000', '381469726562500', '3051757812500000', '3051757812500000', '3051757812500000', '122070312500000', '1525878906250000', '1525878906250000', '48828125000000000', '3051757812500000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '381469726562500', '3051757812500000', '95367431640625', '3051757812500000', '24414062500000000', '6103515625000000', '381469726562500', '762939453125000', '610351562500000', '30517578125000', '381469726562500', '381469726562500', '6103515625000000', '6103515625000000', '610351562500000', '3051757812500000', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '1220703125000000', '1525878906250000', '3051757812500000', '122070312500000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '9765625000000000', '381469726562500', '3051757812500000', '610351562500000', '1525878906250000', '3051757812500000', '3051757812500000', '122070312500000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '48828125000000000', '152587890625000', '762939453125000', '762939453125000', '6103515625000000', '381469726562500', '381469726562500', '48828125000000000', '24414062500000000', '100000000000000000000', '195312500000000000', '390625000000000000', '97656250000000000', '48828125000000000', '97656250000000000', '390625000000000000', '195312500000000000', '390625000000000000', '97656250000000000', '97656250000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '390625000000000000', '12207031250000000', '9765625000000000', '19531250000000000', '390625000000000000', '6250000000000000000', '195312500000000000', '24414062500000000', '24414062500000000', '195312500000000000', '390625000000000000', '78125000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '48828125000000000', '61035156250000', '50000000000000000000', '39062500000000000', '48828125000000000', '390625000000000000', '390625000000000000', '4882812500000000', '39062500000000000', '625000000000000', '78125000000000000', '610351562500000', '4882812500000000', '390625000000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '5000000000000000000', '5000000000000000000', '5000000000000000000', '5000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '1562500000000000000', '6103515625000000', '156250000000000000', '390625000000000000', '97656250000000000', '12207031250000000', '6103515625000000', '1562500000000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '781250000000000000', '312500000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '1220703125000000', '24414062500000000', '312500000000000000', '781250000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '3125000000000000000', '24414062500000000', '1562500000000000000', '781250000000000000', '390625000000000000', '12207031250000000', '24414062500000000', '3125000000000000000', '9765625000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '2500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '6250000000000000000', '12500000000000000000', '12500000000000000000', '781250000000000000', '3125000000000000000', '24414062500000000', '1562500000000000000', '48828125000000000', '48828125000000000', '4882812500000000', '48828125000000000', '3125000000000000000', '781250000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '390625000000000000', '3125000000000000000', '12207031250000000', '976562500000000', '781250000000000000', '781250000000000000', '1562500000000000000', '12207031250000000', '3125000000000000000', '390625000000000000', '781250000000000000', '25000000000000000', '6250000000000000000', '6250000000000000000', '390625000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '97656250000000000', '195312500000000000', '5000000000000000000', '24414062500000000', '50000000000000000000', '50000000000000000000', '190734863281250', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '190734863281250', '9765625000000000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '76293945312500', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '76293945312500', '48828125000000000', '6103515625000000', '122070312500000', '1525878906250000', '3051757812500000', '6103515625000000', '610351562500000', '3051757812500000', '3051757812500000', '1220703125000000', '3051757812500000', '6103515625000000', '48828125000000000', '6103515625000000', '610351562500000', '3051757812500000', '1220703125000000', '6103515625000000', '122070312500000', '3051757812500000', '6103515625000000', '1525878906250000', '610351562500000', '152587890625000', '9765625000000000', '381469726562500', '3051757812500000', '122070312500000', '244140625000000', '6103515625000000', '3051757812500000', '762939453125000', '19073486328125', '6103515625000000', '1525878906250000', '3051757812500000', '24414062500000000', '1525878906250000', '1525878906250000', '95367431640625', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '6103515625000000', '381469726562500', '195312500000000000', '195312500000000000', '97656250000000000', '781250000000000000', '97656250000000000', '24414062500000000', '97656250000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '39062500000000000', '97656250000000000', '24414062500000000', '390625000000000000', '48828125000000000', '195312500000000000', '781250000000000000', '39062500000000000', '97656250000000000', '97656250000000000', '24414062500000000', '390625000000000000', '100000000000000000000', '97656250000000000', '24414062500000000', '97656250000000000', '9765625000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '12207031250000000', '15625000000000000', '12207031250000000', '390625000000000000', '2500000000000000000', '9765625000000000', '48828125000000000', '6250000000000000000', '625000000000000000', '19531250000000000', '48828125000000000', '610351562500000', '6250000000000000000', '48828125000000000', '48828125000000000', '6250000000000000000', '625000000000000000', '19531250000000000', '48828125000000000', '610351562500000', '6250000000000000000', '48828125000000000', '15625000000000000', '6250000000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '6250000000000000000', '15625000000000000', '15625000000000000', '6250000000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '6250000000000000000', '15625000000000000', '9765625000000000', '1562500000000000000', '1250000000000000000', '97656250000000000', '48828125000000000', '3051757812500000', '6250000000000000000', '195312500000000000', '9765625000000000', '1562500000000000000', '1250000000000000000', '97656250000000000', '48828125000000000', '3051757812500000', '6250000000000000000', '195312500000000000', '390625000000000000', '3125000000000000000', '12207031250000000', '48828125000000000', '19531250000000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '390625000000000000', '3125000000000000000', '12207031250000000', '48828125000000000', '19531250000000000', '12500000000000000000', '12500000000000000000', '97656250000000000', '12207031250000000', '12500000000000000000', '1250000000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '2500000000000000000', '12207031250000000', '12207031250000000', '12500000000000000000', '1250000000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '2500000000000000000', '12207031250000000', '390625000000000000', '3125000000000000000', '2500000000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '12500000000000000000', '390625000000000000', '390625000000000000', '3125000000000000000', '2500000000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '12500000000000000000', '390625000000000000', '20000000000000000000', '12500000000000000000', '6250000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '244140625000000', '24414062500000000', '381469726562500', '48828125000000', '1525878906250000', '305175781250000', '1525878906250000', '152587890625000', '1525878906250000', '6103515625000000', '6103515625000000', '762939453125000', '6103515625000000', '12207031250000000', '12207031250000000', '762939453125000', '3051757812500000', '6103515625000000', '190734863281250', '3051757812500000', '305175781250000', '3051757812500000', '1525878906250000', '305175781250000', '152587890625000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '12207031250000000', '762939453125000', '12207031250000000', '1525878906250000', '1220703125000000', '6103515625000000', '3051757812500000', '1220703125000000', '24414062500000000', '6103515625000000', '1525878906250000', '1525878906250000', '12207031250000000', '3051757812500000', '12207031250000000', '381469726562500', '6103515625000000', '6103515625000000', '610351562500000', '1220703125000000', '244140625000000', '381469726562500', '6103515625000000', '152587890625000', '12207031250000000', '1525878906250000', '12207031250000000', '3051757812500000', '3051757812500000', '3051757812500000', '122070312500000', '1220703125000000', '24414062500000000', '24414062500000000', '6103515625000000', '305175781250000', '2441406250000000', '1525878906250000', '12207031250000000', '2441406250000', '1525878906250000', '3051757812500000', '122070312500000', '3125000000000000000', '48828125000000000', '195312500000000000', '6250000000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '195312500000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '781250000000000000', '2441406250000000', '390625000000000000', '24414062500000000', '97656250000000000', '250000000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '390625000000000000', '6103515625000000', '6103515625000000', '781250000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '312500000000000000', '6103515625000000', '6103515625000000', '1562500000000000000', '195312500000000000', '1562500000000000000', '6103515625000000', '6103515625000000', '195312500000000000', '1562500000000000000', '195312500000000000', '6103515625000000', '6103515625000000', '1562500000000000000', '195312500000000000', '1562500000000000000', '6103515625000000', '6103515625000000', '31250000000000000', '48828125000000000', '195312500000000000', '190734863281250', '25000000000000000000', '6103515625000000', '24414062500000000', '390625000000000000', '25000000000000000000', '48828125000000000', '195312500000000000', '190734863281250', '25000000000000000000', '6103515625000000', '24414062500000000', '390625000000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '97656250000000000', '381469726562500', '5000000000000000000', '97656250000000000', '39062500000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '97656250000000000', '381469726562500', '5000000000000000000', '97656250000000000', '39062500000000000', '25000000000000000000', '1000000000000000000', '390625000000000000', '6103515625000000', '625000000000000000', '12207031250000000', '6103515625000000', '97656250000000000', '25000000000000000000', '1000000000000000000', '390625000000000000', '6103515625000000', '625000000000000000', '12207031250000000', '6103515625000000', '97656250000000000', '25000000000000000000', '1250000000000000000', '2441406250000000', '9765625000000000', '5000000000000000000', '12207031250000000', '9765625000000000', '2441406250000000', '25000000000000000000', '1250000000000000000', '2441406250000000', '9765625000000000', '5000000000000000000', '12207031250000000', '9765625000000000', '2441406250000000', '25000000000000000000', '5000000000000000000', '97656250000000000', '97656250000000000', '12500000000000000000', '305175781250000', '97656250000000000', '97656250000000000', '5000000000000000000', '5000000000000000000', '97656250000000000', '97656250000000000', '12500000000000000000', '305175781250000', '97656250000000000', '97656250000000000', '5000000000000000000', '5000000000000000000', '390625000000000000', '24414062500000000', '25000000000000000000', '1220703125000000', '4882812500000000', '390625000000000000', '2500000000000000000', '5000000000000000000', '390625000000000000', '24414062500000000', '25000000000000000000', '1220703125000000', '4882812500000000', '390625000000000000', '2500000000000000000', '50000000000000000000', '2000000000000000000', '2500000000000000000', '50000000000000000000', '10000000000000000000', '25000000000000000000', '1525878906250000', '762939453125000', '762939453125000', '6103515625000000', '152587890625000', '1220703125000000', '6103515625000000', '1220703125000000', '1525878906250000', '24414062500000000', '3051757812500000', '381469726562500', '152587890625000', '3051757812500000', '1525878906250000', '12207031250000000', '12207031250000', '381469726562500', '6103515625000000', '6103515625000000', '1220703125000000', '3051757812500000', '6103515625000000', '95367431640625', '95367431640625', '610351562500000', '190734863281250', '95367431640625', '762939453125000', '6103515625000000', '3051757812500000', '762939453125000', '244140625000000', '12207031250000000', '762939453125000', '3051757812500000', '1525878906250000', '381469726562500', '381469726562500', '6103515625000000', '381469726562500', '190734863281250', '3051757812500000', '381469726562500', '24414062500000000', '1220703125000000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '610351562500000', '6103515625000000', '190734863281250', '6103515625000000', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '12207031250000000', '762939453125000', '488281250000000', '152587890625000', '1525878906250000', '762939453125000', '152587890625000', '4882812500000000', '488281250000000', '6103515625000000', '3051757812500000', '3125000000000000000', '195312500000000000', '390625000000000000', '781250000000000000', '195312500000000000', '390625000000000000', '6250000000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '24414062500000000', '12500000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '48828125000000000', '1250000000000000000', '6250000000000000000', '19531250000000000', '195312500000000000', '48828125000000000', '3125000000000000000', '9765625000000000', '195312500000000000', '19531250000000000', '6250000000000000000', '6250000000000000000', '146484375000000000', '585937500000000000', '11718750000000000', '3125000000000000000', '292968750000000000', '585937500000000000', '18310546875000000', '6250000000000000000', '25000000000000000000', '195312500000000000', '19531250000000000', '24414062500000000', '12500000000000000000', '24414062500000000', '19531250000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '73242187500000000', '585937500000000000', '117187500000000000', '12500000000000000000', '585937500000000000', '585937500000000000', '292968750000000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '24414062500000000', '1000000000000000000', '2441406250000000', '48828125000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '292968750000000000', '585937500000000000', '4687500000000000', '1000000000000000000', '585937500000000000', '117187500000000000', '146484375000000000', '97656250000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '24414062500000000', '12500000000000000000', '24414062500000000', '48828125000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '292968750000000000', '18310546875000000', '146484375000000000', '12500000000000000000', '29296875000000000', '14648437500000000', '58593750000000000', '25000000000000000000', '6250000000000000000', '195312500000000000', '97656250000000000', '48828125000000000', '3125000000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '6250000000000000000', '6250000000000000000', '48828125000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '6250000000000000000', '25000000000000000000', '78125000000000', '3906250000000000', '48828125000000000', '2500000000000000000', '48828125000000000', '3906250000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '234375000000000000', '2500000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '6250000000000000000', '6250000000000000000', '100000000000000000000', '781250000000000000', '6250000000000000000', '20000000000000000000', '190734863281250', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '95367431640625', '6103515625000000', '6103515625000', '6103515625000000', '381469726562500', '1525878906250000', '1220703125000000', '3051757812500000', '3051757812500000', '24414062500000000', '6103515625000000', '3051757812500000', '762939453125000', '762939453125000', '3051757812500000', '3051757812500000', '190734863281250', '3051757812500000', '48828125000000000', '3051757812500000', '6103515625000000', '24414062500000000', '76293945312500', '3051757812500000', '305175781250000', '1220703125000000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '12207031250000000', '3051757812500000', '6103515625000000', '24414062500000000', '1525878906250000', '1525878906250000', '305175781250000', '6103515625000000', '3051757812500000', '6103515625000000', '152587890625000', '6103515625000000', '12207031250000000', '1525878906250000', '3051757812500000', '9765625000000000', '6103515625000000', '3051757812500000', '762939453125000', '305175781250000', '3051757812500000', '6103515625000000', '1525878906250000', '610351562500000', '24414062500000000', '3051757812500000', '1525878906250000', '24414062500000000', '95367431640625', '610351562500000', '3125000000000000000', '100000000000000000000', '50000000000000000000', '7812500000000000', '97656250000000000', '390625000000000000', '12500000000000000000', '100000000000000000000', '50000000000000000000', '48828125000000000', '24414062500000000', '24414062500000000', '48828125000000000', '5000000000000000000', '195312500000000000', '195312500000000000', '24414062500000000', '39062500000000000', '97656250000000000', '390625000000000000', '78125000000000000', '24414062500000000', '97656250000000000', '1562500000000000000', '24414062500000000', '48828125000000000', '12500000000000000000', '4882812500000000', '48828125000000000', '19531250000000000', '48828125000000000', '390625000000000000', '97656250000000000', '97656250000000000', '610351562500000', '312500000000000000', '25000000000000000000', '6103515625000000', '7812500000000000', '1562500000000000000', '1220703125000000', '78125000000000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '6103515625000000', '7812500000000000', '1562500000000000000', '1220703125000000', '78125000000000000', '97656250000000000', '25000000000000000000', '1562500000000000000', '48828125000000000', '78125000000000000', '12500000000000000000', '1220703125000000', '78125000000000000', '97656250000000000', '12500000000000000000', '1562500000000000000', '48828125000000000', '78125000000000000', '12500000000000000000', '1220703125000000', '78125000000000000', '97656250000000000', '12500000000000000000', '25000000000000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '610351562500000', '97656250000000000', '6103515625000000', '25000000000000000000', '25000000000000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '610351562500000', '97656250000000000', '6103515625000000', '25000000000000000000', '3125000000000000000', '12207031250000000', '24414062500000000', '25000000000000000000', '6103515625000000', '97656250000000000', '12207031250000000', '5000000000000000000', '3125000000000000000', '12207031250000000', '24414062500000000', '25000000000000000000', '6103515625000000', '97656250000000000', '12207031250000000', '5000000000000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '2441406250000000', '39062500000000000', '48828125000000000', '12500000000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '2441406250000000', '39062500000000000', '48828125000000000', '12500000000000000', '25000000000000000000', '97656250000000000', '390625000000000000', '12500000000000000000', '12207031250000000', '390625000000000000', '97656250000000000', '5000000000000000000', '25000000000000000000', '97656250000000000', '390625000000000000', '12500000000000000000', '12207031250000000', '390625000000000000', '97656250000000000', '5000000000000000000', '50000000000000000000', '100000000000000000000', '5000000000000000000', '100000000000000000000', '5000000000000000000', '10000000000000000000', '1525878906250000', '6103515625000000', '190734863281250', '610351562500000', '152587890625000', '6103515625000000', '6103515625000000', '24414062500000000', '6103515625000000', '1220703125000000', '1220703125000000', '381469726562500', '305175781250000', '610351562500000', '3051757812500000', '12207031250000000', '1525878906250000', '6103515625000000', '762939453125000', '1525878906250000', '3051757812500000', '3051757812500000', '244140625000000', '1525878906250000', '1525878906250000', '3051757812500000', '1525878906250000', '6103515625000000', '762939453125000', '95367431640625', '12207031250000000', '12207031250000000', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '12207031250000000', '3051757812500000', '12207031250000000', '1525878906250000', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '19073486328125', '1220703125000000', '76293945312500', '152587890625000', '12207031250000000', '1525878906250000', '1525878906250000', '1525878906250000', '6103515625000000', '24414062500000000', '24414062500000000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '610351562500000', '2441406250000000', '305175781250000', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '1525878906250000', '1220703125000000', '610351562500000', '6250000000000000000', '195312500000000000', '39062500000000000', '6250000000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '1562500000000000000', '48828125000000000', '390625000000000000', '6250000000000000000', '97656250000000000', '3125000000000000000', '390625000000000000', '195312500000000000', '195312500000000000', '12500000000000000000', '1525878906250000', '3125000000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '195312500000000000', '24414062500000000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '24414062500000000', '48828125000000000', '156250000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '24414062500000000', '48828125000000000', '156250000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '2441406250000000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '2441406250000000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '78125000000000000', '3051757812500000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '78125000000000000', '3051757812500000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '24414062500000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '24414062500000000', '24414062500000000', '48828125000000000', '48828125000000000', '390625000000000000', '48828125000000000', '195312500000000000', '24414062500000000', '48828125000000000', '390625000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '24414062500000000', '48828125000000000', '390625000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '12207031250000000', '48828125000000000', '97656250000000000', '97656250000000000', '48828125000000000', '48828125000000000', '12207031250000000', '48828125000000000', '97656250000000000', '50000000000000000000', '6250000000000000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '3125000000000000000', '1525878906250000', '762939453125000', '1525878906250000', '1525878906250000', '12207031250000000', '12207031250000000', '305175781250000', '195312500000000000', '12207031250000000', '6103515625000000', '381469726562500', '6103515625000000', '3051757812500000', '610351562500000', '6103515625000000', '12207031250000000', '12207031250000000', '97656250000000000', '12207031250000000', '3051757812500000', '95367431640625', '1220703125000000', '38146972656250', '762939453125000', '2441406250000000', '12207031250000000', '12207031250000000', '97656250000000000', '2441406250000000', '6103515625000000', '3051757812500000', '610351562500000', '3051757812500000', '3051757812500000', '6103515625000000', '1220703125000000', '12207031250000000', '390625000000000000', '6103515625000000', '762939453125000', '3051757812500000', '1525878906250000', '1525878906250000', '1525878906250000', '12207031250000000', '3051757812500000', '1525878906250000', '390625000000000000', '488281250000000', '1525878906250000', '1525878906250000', '190734863281250', '1220703125000000', '152587890625000', '6103515625000000', '6103515625000000', '6103515625000000', '195312500000000000', '1525878906250000', '244140625000000', '3125000000000000000', '156250000000000', '390625000000000000', '156250000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '195312500000000000', '390625000000000000', '195312500000000000', '48828125000000000', '12500000000000000000', '39062500000000000', '19531250000000000', '6250000000000000000', '195312500000000000', '97656250000000000', '1000000000000000000', '1562500000000000000', '3051757812500000', '2441406250000000', '48828125000000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '3051757812500000', '2441406250000000', '48828125000000000', '781250000000000000', '1562500000000000000', '781250000000000000', '390625000000000000', '12207031250000000', '190734863281250', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '12207031250000000', '190734863281250', '12207031250000000', '781250000000000000', '31250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '1220703125000000', '31250000000000000', '31250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '1220703125000000', '31250000000000000', '390625000000000000', '1562500000000000000', '24414062500000000', '24414062500000000', '488281250000000', '312500000000000000', '390625000000000000', '1562500000000000000', '24414062500000000', '24414062500000000', '488281250000000', '312500000000000000', '781250000000000000', '4882812500000000', '48828125000000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '4882812500000000', '48828125000000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '62500000000000000', '781250000000000000', '48828125000000000', '48828125000000000', '12207031250000000', '62500000000000000', '62500000000000000', '781250000000000000', '48828125000000000', '48828125000000000', '12207031250000000', '62500000000000000', '100000000000000000000', '20000000000000000000', '50000000000000000000', '390625000000000000', '25000000000000000000', '50000000000000000000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '2441406250000000', '195312500000000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '2441406250000000', '390625000000000000', '6103515625000000', '12207031250000000', '610351562500000', '610351562500000', '762939453125000', '381469726562500', '610351562500000', '3051757812500000', '97656250000000', '390625000000000000', '12207031250000000', '12207031250000000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '1220703125000000', '3051757812500000', '610351562500000', '195312500000000000', '12207031250000000', '1525878906250000', '6103515625000000', '1220703125000000', '610351562500000', '3051757812500000', '6103515625000000', '305175781250000', '1525878906250000', '390625000000000000', '12207031250000000', '381469726562500', '3051757812500000', '6103515625000000', '3051757812500000', '244140625000000', '38146972656250', '762939453125000', '3051757812500000', '390625000000000000', '2441406250000000', '1525878906250000', '6103515625000000', '1220703125000000', '781250000000000000', '78125000000000000', '97656250000000000', '625000000000000000', '390625000000000000', '12207031250000000', '24414062500000000', '6250000000000000000', '48828125000000000', '78125000000000000', '5000000000000000000', '97656250000000000', '1562500000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '50000000000000000000', '97656250000000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '305175781250000', '781250000000000000', '305175781250000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '305175781250000', '781250000000000000', '390625000000000000', '390625000000000000', '1562500000000000000', '12207031250000000', '390625000000000000', '48828125000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '1562500000000000000', '12207031250000000', '1562500000000000000', '762939453125000', '762939453125000', '762939453125000', '762939453125000', '762939453125000', '762939453125000', '762939453125000', '762939453125000', '48828125000000000', '12207031250000000', '6103515625000000', '48828125000000000', '48828125000000000', '12207031250000000', '6103515625000000', '48828125000000000', '9765625000000000', '24414062500000000', '6103515625000000', '9765625000000000', '9765625000000000', '24414062500000000', '6103515625000000', '9765625000000000', '48828125000000000', '24414062500000000', '488281250000000', '48828125000000000', '48828125000000000', '24414062500000000', '488281250000000', '48828125000000000', '24414062500000000', '48828125000000000', '3051757812500000', '48828125000000000', '24414062500000000', '48828125000000000', '3051757812500000', '48828125000000000', '48828125000000000', '24414062500000000', '6103515625000000', '48828125000000000', '48828125000000000', '24414062500000000', '6103515625000000', '48828125000000000', '24414062500000000', '6250000000000000000', '97656250000000000', '312500000000000000', '48828125000000000', '1250000000000000000', '762939453125000', '1220703125000000', '12207031250000000', '1525878906250000', '610351562500000', '12207031250000000', '4882812500000000', '3051757812500000', '12207031250000000', '6103515625000000', '12207031250000000', '24414062500000000', '12207031250000000', '24414062500000000', '24414062500000000', '305175781250000', '6103515625000000', '6103515625000000', '12207031250000000', '48828125000000000', '48828125000000000', '48828125000000000', '48828125000000000', '24414062500000000', '610351562500000', '610351562500000', '12207031250000000', '48828125000000000', '48828125000000000', '9765625000000000', '9765625000000000', '24414062500000000', '95367431640625', '381469726562500', '3051757812500000', '6103515625000000', '2441406250000000', '48828125000000000', '48828125000000000', '24414062500000000', '3051757812500000', '610351562500000', '1525878906250000', '24414062500000000', '48828125000000000', '12207031250000000', '24414062500000000', '12207031250000000', '390625000000000000', '195312500000000000', '15625000000000000', '7812500000000000', '390625000000000000', '390625000000000000', '156250000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '156250000000000000', '48828125000000000', '48828125000000000', '156250000000000000', '9765625000000000', '156250000000000000', '1220703125000000', '1220703125000000', '24414062500000000', '24414062500000000', '12207031250000000', '12207031250000000', '24414062500000000', '4882812500000000', '1', '1', '305175781250000', '305175781250000', '305175781250000', '305175781250000', '195312500000000000', '7812500000000000', '195312500000000000', '9765625000000000', '9765625000000000', '390625000000000000', '9765625000000000', '9765625000000000', '4882812500000000', '390625000000000000', '78125000000000000', '78125000000000000', '781250000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '24414062500000000', '24414062500000000', '78125000000000000', '781250000000000000', '781250000000000000', '78125000000000000', '625000000000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '39062500000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '39062500000000000', '1562500000000000000', '39062500000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '39062500000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '625000000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '195312500000000000', '3051757812500000', '781250000000000000', '3051757812500000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '3051757812500000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '48828125000000000', '39062500000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '781250000000000000', '4', '4', '4', '4', '1', '4', '4', '4', '4', '48828125000000000', '24414062500000000', '6103515625000000', '48828125000000000', '48828125000000000', '24414062500000000', '6103515625000000', '48828125000000000', '24414062500000000', '762939453125000', '12207031250000000', '24414062500000000', '24414062500000000', '762939453125000', '12207031250000000', '24414062500000000', '12207031250000000', '6103515625000000', '762939453125000', '12207031250000000', '12207031250000000', '6103515625000000', '762939453125000', '12207031250000000', '610351562500000', '61035156250000', '152587890625000', '610351562500000', '610351562500000', '61035156250000', '152587890625000', '610351562500000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '6250000000000000000', '97656250000000000', '390625000000000000', '48828125000000000', '6103515625000000', '6250000000000000000', '1525878906250000', '24414062500000000', '24414062500000000', '48828125000000000', '6103515625000000', '3051757812500000', '6103515625000000', '12207031250000000', '24414062500000000', '24414062500000000', '48828125000000000', '9765625000000000', '6103515625000000', '3051757812500000', '305175781250000', '6103515625000000', '12207031250000000', '9765625000000000', '12207031250000000', '3051757812500000', '3051757812500000', '762939453125000', '12207031250000000', '4882812500000000', '48828125000000000', '48828125000000000', '610351562500000', '48828125000000000', '2441406250000000', '610351562500000', '12207031250000000', '24414062500000000', '3051757812500000', '24414062500000000', '12207031250000000', '1220703125000000', '6103515625000000', '6103515625000000', '12207031250000000', '12207031250000000', '6103515625000000', '24414062500000000', '2441406250000000', '12207031250000000', '12207031250000000', '762939453125000', '12207031250000000', '3051757812500000', '390625000000000000', '78125000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '78125000000000000', '1562500000000000000', '156250000000000000', '9765625000000000', '9765625000000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '156250000000000000', '9765625000000000', '9765625000000000', '24414062500000000', '1562500000000000000', '312500000000000000', '781250000000000000', '9765625000000000', '48828125000000000', '3051757812500000', '312500000000000000', '312500000000000000', '781250000000000000', '9765625000000000', '48828125000000000', '3051757812500000', '312500000000000000', '1562500000000000000', '1562500000000000000', '9765625000000000', '9765625000000000', '122070312500000', '390625000000000000', '1562500000000000000', '1562500000000000000', '9765625000000000', '9765625000000000', '122070312500000', '390625000000000000', '781250000000000000', '390625000000000000', '9765625000000000', '9765625000000000', '24414062500000000', '781250000000000000', '781250000000000000', '390625000000000000', '9765625000000000', '9765625000000000', '24414062500000000', '781250000000000000', '312500000000000000', '24414062500000000', '1953125000000000', '12207031250000000', '312500000000000000', '390625000000000000', '312500000000000000', '24414062500000000', '1953125000000000', '12207031250000000', '312500000000000000', '390625000000000000', '312500000000000000', '312500000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '781250000000000000', '312500000000000000', '312500000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '781250000000000000', '10000000000000000000', '25000000000000000000', '10000000000000000000', '10000000000000000000', '50000000000000000000', '25000000000000000000', '6103515625000000', '95367431640625', '2441406250000000', '195312500000000000', '381469726562500', '1525878906250000', '6103515625000000', '381469726562500', '3051757812500000', '95367431640625', '6103515625000000', '12207031250000000', '12207031250000000', '195312500000000000', '1525878906250000', '3051757812500000', '6103515625000000', '190734863281250', '1525878906250000', '6103515625000000', '3051757812500000', '12207031250000000', '12207031250000000', '195312500000000000', '12207031250000000', '1525878906250000', '381469726562500', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '6103515625000000', '390625000000000000', '305175781250000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '12207031250000000', '12207031250000000', '381469726562500', '48828125000000000', '1525878906250000', '305175781250000', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '190734863281250', '3051757812500000', '610351562500000', '195312500000000000', '305175781250000', '381469726562500', '762939453125000', '762939453125000', '190734863281250', '3051757812500000', '78125000000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '48828125000000000', '19531250000000000', '1562500000000000000', '97656250000000000', '195312500000000000', '3125000000000000000', '97656250000000000', '48828125000000000', '78125000000000000', '2441406250000000', '195312500000000000', '195312500000000000', '195312500000000000', '5000000000000000000', '97656250000000000', '12500000000000000000', '195312500000000000', '97656250000000000', '190734863281250', '97656250000000000', '25000000000000000000', '5000000000000000000', '97656250000000000', '12500000000000000000', '195312500000000000', '97656250000000000', '190734863281250', '97656250000000000', '25000000000000000000', '2500000000000000000', '12207031250000000', '2441406250000000', '390625000000000000', '390625000000000000', '5000000000000000000', '3051757812500000', '25000000000000000000', '2500000000000000000', '12207031250000000', '2441406250000000', '390625000000000000', '390625000000000000', '5000000000000000000', '3051757812500000', '25000000000000000000', '781250000000000000', '24414062500000000', '12207031250000000', '7812500000000000', '97656250000000000', '25000000000000000000', '2441406250000000', '25000000000000000000', '781250000000000000', '24414062500000000', '12207031250000000', '7812500000000000', '97656250000000000', '25000000000000000000', '2441406250000000', '25000000000000000000', '5000000000000000000', '97656250000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '5000000000000000000', '5000000000000000000', '97656250000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '97656250000000000', '5000000000000000000', '25000000000000000000', '12207031250000000', '2441406250000000', '390625000000000000', '195312500000000000', '1562500000000000000', '97656250000000000', '5000000000000000000', '25000000000000000000', '12207031250000000', '2441406250000000', '390625000000000000', '195312500000000000', '1562500000000000000', '97656250000000000', '5000000000000000000', '2500000000000000000', '48828125000000000', '25000000000000000000', '78125000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '25000000000000000000', '2500000000000000000', '48828125000000000', '25000000000000000000', '78125000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '25000000000000000000', '12500000000000000000', '100000000000000000000', '50000000000000000000', '1562500000000000000', '100000000000000000000', '1000000000000000000', '1525878906250000', '762939453125000', '6103515625000000', '12207031250000000', '24414062500000000', '1220703125000000', '1525878906250000', '3051757812500000', '381469726562500', '6103515625000000', '30517578125000', '3051757812500000', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '4882812500000000', '3051757812500000', '762939453125000', '6103515625000000', '3051757812500000', '6103515625000000', '1525878906250000', '1525878906250000', '1525878906250000', '6103515625000000', '6103515625000000', '24414062500000000', '3051757812500000', '1220703125000000', '762939453125000', '6103515625000000', '305175781250000', '3051757812500000', '381469726562500', '1525878906250000', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '6103515625000000', '1525878906250000', '95367431640625', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '1525878906250000', '1220703125000000', '1525878906250000', '6103515625000000', '6103515625000000', '12207031250000000', '3051757812500000', '1525878906250000', '12207031250000000', '762939453125000', '12207031250000000', '1525878906250000', '610351562500000', '762939453125000', '6103515625000000', '6103515625000000', '24414062500000000', '1525878906250000', '762939453125000', '762939453125000', '6103515625000000', '3051757812500000', '1525878906250000', '762939453125000', '305175781250000', '390625000000000000', '12207031250000000', '3125000000000000000', '1562500000000000000', '195312500000000000', '48828125000000000', '390625000000000000', '24414062500000000', '6250000000000000000', '19531250000000000', '3906250000000000', '6250000000000000000', '1562500000000000000', '19531250000000000', '390625000000000000', '3125000000000000000', '19531250000000000', '390625000000000000', '195312500000000000', '12500000000000000000', '19531250000000000', '12500000000000000000', '6250000000000000000', '97656250000000000', '122070312500000', '195312500000000000', '195312500000000000', '12500000000000000000', '19531250000000000', '12500000000000000000', '6250000000000000000', '97656250000000000', '122070312500000', '195312500000000000', '195312500000000000', '125000000000000000', '9765625000000000', '6250000000000000000', '50000000000000000', '9765625000000000', '12207031250000000', '195312500000000000', '195312500000000000', '125000000000000000', '9765625000000000', '6250000000000000000', '50000000000000000', '9765625000000000', '12207031250000000', '195312500000000000', '390625000000000000', '6103515625000000', '24414062500000000', '12500000000000000000', '2500000000000000000', '2441406250000000', '6250000000000000000', '78125000000000000', '390625000000000000', '6103515625000000', '24414062500000000', '12500000000000000000', '2500000000000000000', '2441406250000000', '6250000000000000000', '78125000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '39062500000000000', '6250000000000000000', '97656250000000000', '1250000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '39062500000000000', '6250000000000000000', '97656250000000000', '1250000000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '97656250000000000', '781250000000000000', '48828125000000000', '97656250000000000', '6103515625000000', '390625000000000000', '390625000000000000', '781250000000000000', '97656250000000000', '781250000000000000', '48828125000000000', '97656250000000000', '6103515625000000', '390625000000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '1250000000000000000', '1250000000000000000', '48828125000000000', '6103515625000000', '390625000000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '1250000000000000000', '1250000000000000000', '48828125000000000', '6103515625000000', '390625000000000000', '100000000000000000000', '25000000000000000000', '100000000000000000000', '25000000000000000000', '10000000000000000000', '5000000000000000000', '6103515625000000', '3051757812500000', '190734863281250', '12207031250000000', '762939453125000', '12207031250000000', '3051757812500000', '95367431640625', '2441406250000000', '24414062500000000', '6103515625000000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '12207031250000000', '12207031250000000', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '610351562500000', '12207031250000000', '381469726562500', '12207031250000000', '762939453125000', '305175781250000', '4882812500000000', '4882812500000000', '1525878906250000', '610351562500000', '6103515625000000', '3051757812500000', '762939453125000', '1525878906250000', '30517578125000', '12207031250000000', '1525878906250000', '762939453125000', '6103515625000000', '1220703125000000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '1525878906250000', '488281250000000', '3051757812500000', '6103515625000000', '19073486328125', '3051757812500000', '6103515625000000', '1220703125000000', '3051757812500000', '610351562500000', '1220703125000000', '6103515625000000', '305175781250000', '12207031250000000', '152587890625000', '3051757812500000', '3051757812500000', '3051757812500000', '12207031250000000', '4882812500000000', '6103515625000000', '305175781250000', '195312500000000000', '48828125000000000', '12500000000000000000', '48828125000000000', '6250000000000000000', '24414062500000000', '24414062500000000', '15625000000000000', '625000000000000000', '195312500000000000', '390625000000000000', '12500000000000000000', '24414062500000000', '4882812500000000', '12207031250000000', '19531250000000000', '3125000000000000000', '195312500000000000', '48828125000000000', '390625000000000000', '781250000000000000', '781250000000000000', '6103515625000000', '48828125000000000', '48828125000000000', '390625000000000000', '781250000000000000', '781250000000000000', '6103515625000000', '48828125000000000', '9765625000000000', '6103515625000000', '4882812500000000', '4882812500000000', '24414062500000000', '9765625000000000', '9765625000000000', '6103515625000000', '4882812500000000', '4882812500000000', '24414062500000000', '9765625000000000', '6103515625000000', '6103515625000000', '31250000000000000', '156250000000000000', '195312500000000000', '6103515625000000', '6103515625000000', '6103515625000000', '31250000000000000', '156250000000000000', '195312500000000000', '6103515625000000', '48828125000000000', '195312500000000000', '78125000000000000', '78125000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '195312500000000000', '78125000000000000', '78125000000000000', '12207031250000000', '48828125000000000', '9765625000000000', '390625000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '9765625000000000', '9765625000000000', '390625000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '9765625000000000', '610351562500000', '390625000000000000', '781250000000000000', '156250000000000000', '24414062500000000', '610351562500000', '610351562500000', '390625000000000000', '781250000000000000', '156250000000000000', '24414062500000000', '610351562500000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '3125000000000000000', '6250000000000000000', '25000000000000000000', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '12207031250000000', '97656250000000000', '12207031250000000', '6103515625000000', '3051757812500000', '2441406250000000', '762939453125000', '3051757812500000', '3051757812500000', '1525878906250000', '12207031250000000', '97656250000000000', '12207031250000000', '6103515625000000', '12207031250000000', '61035156250000', '305175781250000', '1220703125000000', '381469726562500', '3051757812500000', '762939453125000', '97656250000000000', '12207031250000000', '1220703125000000', '3051757812500000', '762939453125000', '3051757812500000', '6103515625000000', '305175781250000', '3051757812500000', '6103515625000000', '97656250000000000', '12207031250000000', '6103515625000000', '12207031250000000', '12207031250000000', '3051757812500000', '122070312500000', '1525878906250000', '3051757812500000', '12207031250000000', '6103515625000000', '12207031250000000', '3051757812500000', '12207031250000000', '122070312500000', '3051757812500000', '1525878906250000', '1525878906250000', '3051757812500000', '610351562500000', '48828125000000000', '3051757812500000', '6103515625000000', '24414062500000000', '39062500000000000', '3125000000000000000', '9765625000000000', '20000000000000000000', '195312500000000000', '97656250000000000', '390625000000000000', '1562500000000000000', '39062500000000000', '24414062500000000', '1562500000000000000', '12207031250000000', '12500000000000000000', '390625000000000000', '19531250000000000', '1562500000000000000', '7812500000000000', '4882812500000000', '48828125000000000', '48828125000000000', '762939453125000', '4882812500000000', '48828125000000000', '48828125000000000', '762939453125000', '488281250000000', '24414062500000000', '24414062500000000', '3051757812500000', '488281250000000', '24414062500000000', '24414062500000000', '3051757812500000', '488281250000000', '488281250000000', '488281250000000', '488281250000000', '488281250000000', '488281250000000', '488281250000000', '488281250000000', '4882812500000000', '48828125000000000', '48828125000000000', '3051757812500000', '4882812500000000', '48828125000000000', '48828125000000000', '3051757812500000', '6103515625000000', '24414062500000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '24414062500000000', '12207031250000000', '381469726562500', '12207031250000000', '12207031250000000', '244140625000000', '381469726562500', '12207031250000000', '12207031250000000', '244140625000000', '1562500000000000000', '12207031250000000', '390625000000000000', '781250000000000000', '1562500000000000000', '48828125000000000', '6103515625000000', '48828125000000000', '6103515625000000', '12207031250000000', '488281250000000', '48828125000000000', '24414062500000000', '976562500000000', '610351562500000', '48828125000000000', '48828125000000', '610351562500000', '12207031250000000', '24414062500000000', '48828125000000000', '6103515625000000', '48828125000000000', '24414062500000000', '6103515625000000', '3051757812500000', '2441406250000000', '48828125000000000', '48828125000000000', '24414062500000000', '48828125000000000', '3051757812500000', '12207031250000000', '12207031250000000', '6103515625000000', '3051757812500000', '48828125000000000', '610351562500000', '48828125000000000', '48828125000000000', '6103515625000000', '3051757812500000', '610351562500000', '9765625000000000', '48828125000000000', '12207031250000000', '24414062500000000', '381469726562500', '12207031250000000', '6103515625000000', '12207031250000000', '48828125000000000', '24414062500000000', '24414062500000000', '97656250000000000', '390625000000000000', '390625000000000000', '390625000000000000', '78125000000000000', '390625000000000000', '4', '4', '4', '4', '97656250000000000', '48828125000000000', '97656250000000000', '48828125000000000', '48828125000000000', '97656250000000000', '48828125000000000', '48828125000000000', '48828125000000000', '97656250000000000', '1562500000000000000', '9765625000000000', '1562500000000000000', '24414062500000000', '9765625000000000', '781250000000000000', '1562500000000000000', '9765625000000000', '1562500000000000000', '24414062500000000', '9765625000000000', '781250000000000000', '1562500000000000000', '24414062500000000', '12207031250000000', '781250000000000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '24414062500000000', '12207031250000000', '781250000000000000', '24414062500000000', '1562500000000000000', '312500000000000000', '4882812500000000', '781250000000000000', '3051757812500000', '4882812500000000', '312500000000000000', '312500000000000000', '4882812500000000', '781250000000000000', '3051757812500000', '4882812500000000', '312500000000000000', '781250000000000000', '195312500000000', '781250000000000000', '24414062500000000', '195312500000000', '12207031250000000', '781250000000000000', '195312500000000', '781250000000000000', '24414062500000000', '195312500000000', '12207031250000000', '390625000000000000', '48828125000000000', '390625000000000000', '24414062500000000', '24414062500000000', '195312500000000000', '390625000000000000', '48828125000000000', '390625000000000000', '24414062500000000', '24414062500000000', '195312500000000000', '390625000000000000', '48828125000000000', '195312500000000000', '1525878906250000', '976562500000000', '390625000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '1525878906250000', '976562500000000', '390625000000000000', '3125000000000000000', '12500000000000000000', '5000000000000000000', '12500000000000000000', '3125000000000000000', '25000000000000000000', '305175781250000', '39062500000000000', '6103515625000000', '3051757812500000', '12207031250000000', '30517578125000', '6103515625000000', '6103515625000000', '762939453125000', '3051757812500000', '12207031250000000', '97656250000000000', '1220703125000000', '2441406250000000', '12207031250000000', '3051757812500000', '1525878906250000', '1220703125000000', '3051757812500000', '1525878906250', '3051757812500000', '390625000000000000', '6103515625000000', '12207031250000000', '762939453125000', '1525878906250000', '6103515625000000', '6103515625000000', '762939453125000', '610351562500000', '95367431640625', '390625000000000000', '30517578125000', '12207031250000000', '12207031250000000', '1525878906250000', '1525878906250000', '6103515625000000', '381469726562500', '610351562500000', '1220703125000000', '4882812500000000', '488281250000000', '12207031250000000', '6103515625000000', '152587890625000', '381469726562500', '6103515625000000', '1525878906250000', '6103515625000000', '12207031250000000', '97656250000000000', '12207031250000000', '12207031250000000', '12207031250000000', '610351562500000', '3051757812500000', '1525878906250000', '3051757812500000', '610351562500000', '390625000000000000', '48828125000000000', '10000000000000000000', '195312500000000000', '24414062500000000', '1562500000000000000', '195312500000000000', '24414062500000000', '800000000000000000', '390625000000000000', '19531250000000000', '97656250000000000', '97656250000000000', '97656250000000000', '195312500000000000', '781250000000000000', '195312500000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '9765625000000000', '39062500000000000', '12500000000000000000', '39062500000000000', '9765625000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '39062500000000000', '6250000000000000000', '39062500000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '12500000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '234375000000000000', '1171875000000000000', '12500000000000000000', '25000000000000000000', '7812500000000000', '48828125000000000', '6103515625000000', '12500000000000000000', '6103515625000000', '48828125000000000', '7812500000000000', '25000000000000000000', '25000000000000000000', '7812500000000000', '48828125000000000', '6103515625000000', '12500000000000000000', '6103515625000000', '48828125000000000', '7812500000000000', '25000000000000000000', '2500000000000000000', '195312500000000000', '6103515625000000', '12207031250000000', '6250000000000000000', '195312500000000000', '6103515625000000', '195312500000000000', '2500000000000000000', '2500000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '6250000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '2500000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '234375000000000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '12500000000000000000', '39062500000000000', '24414062500000000', '195312500000000000', '6250000000000000000', '195312500000000000', '24414062500000000', '39062500000000000', '12500000000000000000', '12500000000000000000', '78125000000000000', '390625000000000000', '390625000000000000', '6250000000000000000', '390625000000000000', '390625000000000000', '15625000000000000', '12500000000000000000', '625000000000000000', '1562500000000000000', '12500000000000000000', '12207031250000000', '12500000000000000000', '9765625000000000', '6103515625000000', '4882812500000000', '762939453125000', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '762939453125000', '762939453125000', '3051757812500000', '1525878906250000', '3051757812500000', '1220703125000000', '24414062500000000', '6103515625000000', '244140625000000', '24414062500000000', '6103515625000000', '1525878906250000', '1525878906250000', '610351562500000', '6103515625000000', '95367431640625', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '3051757812500000', '24414062500000000', '762939453125000', '190734863281250', '1220703125000000', '3051757812500000', '305175781250000', '1525878906250000', '3051757812500000', '305175781250000', '762939453125000', '6103515625000000', '1220703125000000', '24414062500000000', '3051757812500000', '381469726562500', '3051757812500000', '1525878906250000', '3051757812500000', '152587890625000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '38146972656250', '1525878906250000', '762939453125000', '190734863281250', '1220703125000000', '61035156250000', '381469726562500', '6103515625000000', '4882812500000000', '3051757812500000', '6103515625000000', '24414062500000000', '7629394531250', '1525878906250000', '6103515625000000', '762939453125000', '381469726562500', '1525878906250000', '1525878906250000', '97656250000000000', '390625000000000000', '24414062500000000', '24414062500000000', '390625000000000000', '62500000000000000', '195312500000000000', '78125000000000000', '39062500000000000', '19531250000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '390625000000000000', '12207031250000000', '9765625000000000', '390625000000000000', '1953125000000000', '12207031250000000', '100000000000000000000', '24414062500000000', '48828125000000000', '24414062500000000', '390625000000000000', '50000000000000000000', '12500000000000000000', '195312500000000000', '195312500000000000', '100000000000000000000', '78125000000000000', '12500000000000000000', '10000000000000000000', '39062500000000000', '24414062500000000', '20000000000000000000', '24414062500000000', '78125000000000000', '9765625000000000', '781250000000000000', '625000000000000000', '24414062500000000', '312500000000000000', '625000000000000000', '781250000000000000', '9765625000000000', '78125000000000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '781250000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '78125000000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '1562500000000000000', '2441406250000000', '3125000000000000000', '195312500000000000', '1953125000000000', '78125000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '78125000000000000', '6103515625000000', '48828125000000000', '312500000000000000', '1562500000000000000', '1220703125000000', '781250000000000000', '48828125000000000', '6103515625000000', '78125000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '156250000000000000', '48828125000000000', '390625000000000000', '3125000000000000000', '781250000000000000', '24414062500000000', '3125000000000000000', '390625000000000000', '48828125000000000', '156250000000000000', '5000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '15625000000000000', '6250000000000000', '390625000000000000', '12207031250000000', '6250000000000000', '15625000000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '3125000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '3125000000000000000', '12500000000000000000', '781250000000000000', '12207031250000000', '195312500000000000', '48828125000000000', '390625000000000000', '1525878906250000', '48828125000000000', '195312500000000000', '12207031250000000', '781250000000000000', '1250000000000000000', '625000000000000000', '6250000000000000000', '1250000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '625000000000000000', '6250000000000000000', '50000000000000000000', '100000000000000000000', '12207031250000000', '24414062500000000', '12207031250000000', '100000000000000000000', '762939453125000', '48828125000000000', '3051757812500000', '122070312500000', '6103515625000000', '1220703125000000', '762939453125000', '305175781250000', '6103515625000000', '6103515625000000', '381469726562500', '3051757812500000', '3051757812500000', '48828125000000000', '1220703125000000', '762939453125000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '1220703125000000', '3051757812500000', '3051757812500000', '3814697265625', '6103515625000000', '4882812500000000', '610351562500000', '762939453125000', '6103515625000000', '6103515625000000', '3051757812500000', '610351562500000', '244140625000000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '24414062500000000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '30517578125000', '1525878906250000', '6103515625000000', '95367431640625', '76293945312500', '762939453125000', '762939453125000', '24414062500000000', '3051757812500000', '1525878906250000', '1525878906250000', '3051757812500000', '381469726562500', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '95367431640625', '48828125000000000', '762939453125000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '762939453125000', '6103515625000000', '3051757812500000', '3051757812500000', '195312500000000000', '48828125000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '97656250000000000', '39062500000000000', '390625000000000000', '1000000000000000000', '48828125000000000', '19531250000000000', '3051757812500000', '9765625000000000', '50000000000000000000', '976562500000000', '24414062500000000', '97656250000000000', '195312500000000000', '195312500000000000', '781250000000000000', '390625000000000000', '48828125000000000', '24414062500000000', '390625000000000000', '97656250000000000', '1525878906250000', '100000000000000000000', '97656250000000000', '24414062500000000', '97656250000000000', '24414062500000000', '390625000000000000', '156250000000000000', '12207031250000000', '195312500000000000', '25000000000000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '12500000000000000000', '195312500000000000', '12207031250000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '25000000000000000000', '234375000000000000', '12500000000000000000', '234375000000000000', '234375000000000000', '12207031250000000', '195312500000000000', '12500000000000000000', '97656250000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '195312500000000000', '12207031250000000', '234375000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '1171875000000000000', '97656250000000', '9765625000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '97656250000000000', '12500000000000000000', '24414062500000000', '24414062500000000', '195312500000000000', '195312500000000000', '12500000000000000000', '39062500000000000', '6250000000000000000', '195312500000000000', '12500000000000000000', '195312500000000000', '195312500000000000', '1220703125000000', '48828125000000000', '25000000000000000000', '195312500000000000', '2500000000000000000', '195312500000000000', '25000000000000000000', '48828125000000000', '1220703125000000', '48828125000000000', '24414062500000000', '25000000000000000000', '195312500000000000', '2500000000000000000', '39062500000000000', '25000000000000000000', '2441406250000000', '48828125000000000', '9765625000000000', '6103515625000000', '1250000000000000000', '97656250000000000', '3125000000000000000', '195312500000000000', '1250000000000000000', '6103515625000000', '9765625000000000', '24414062500000000', '48828125000000000', '1250000000000000000', '97656250000000000', '3125000000000000000', '195312500000000000', '1250000000000000000', '9765625000000000', '12207031250000000', '3051757812500000', '195312500000000000', '12500000000000000000', '97656250000000000', '6250000000000000000', '97656250000000000', '12500000000000000000', '195312500000000000', '3051757812500000', '234375000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '1171875000000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '4000000000000000000', '100000000000000000000', '50000000000000000000', '3051757812500000', '48828125000000000', '1525878906250000', '1525878906250000', '3051757812500000', '76293945312500', '762939453125000', '1525878906250000', '6103515625000000', '6103515625000000', '152587890625000', '3051757812500000', '6103515625000000', '12207031250000000', '3051757812500000', '762939453125000', '3051757812500000', '3051757812500000', '6103515625000000', '381469726562500', '1220703125000000', '12207031250000000', '3051757812500000', '610351562500000', '762939453125000', '24414062500000000', '6103515625000000', '190734863281250', '6103515625000000', '610351562500000', '152587890625000', '762939453125000', '3051757812500000', '381469726562500', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '3051757812500000', '6103515625000000', '1525878906250000', '61035156250000', '24414062500000000', '6103515625000000', '3051757812500000', '190734863281250', '12207031250000000', '6103515625000000', '762939453125000', '610351562500000', '3051757812500000', '610351562500000', '1525878906250000', '6103515625000000', '24414062500000000', '3051757812500000', '3051757812500000', '6103515625000000', '24414062500000000', '762939453125000', '762939453125000', '1525878906250000', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '48828125000000000', '1525878906250000', '1525878906250000', '195312500000000000', '1525878906250000', '78125000000000000', '19531250000000000', '6250000000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '390625000000000000', '195312500000000000', '78125000000000000', '312500000000000000', '48828125000000000', '97656250000000000', '25000000000000000000', '1562500000000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '24414062500000000', '50000000000000000000', '50000000000000000000', '5000000000000000000', '39062500000000000', '3906250000000000', '97656250000000000', '390625000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '48828125000000000', '12207031250000000', '100000000000000000000', '195312500000000000', '390625000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '12207031250000000', '122070312500000', '97656250000000000', '12207031250000000', '48828125000000000', '97656250000000000', '12207031250000000', '122070312500000', '97656250000000000', '12207031250000000', '195312500000000', '781250000000000000', '4882812500000000', '390625000000000000', '781250000000000000', '195312500000000', '195312500000000', '781250000000000000', '4882812500000000', '390625000000000000', '781250000000000000', '195312500000000', '1525878906250000', '156250000000000000', '78125000000000000', '24414062500000000', '156250000000000000', '1525878906250000', '1525878906250000', '156250000000000000', '78125000000000000', '24414062500000000', '156250000000000000', '1525878906250000', '48828125000000000', '781250000000000000', '24414062500000000', '390625000000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '24414062500000000', '390625000000000000', '781250000000000000', '48828125000000000', '48828125000000000', '390625000000000000', '195312500000000000', '762939453125000', '390625000000000000', '48828125000000000', '48828125000000000', '390625000000000000', '195312500000000000', '762939453125000', '390625000000000000', '48828125000000000', '24414062500000000', '781250000000000000', '390625000000000000', '24414062500000000', '781250000000000000', '24414062500000000', '24414062500000000', '781250000000000000', '390625000000000000', '24414062500000000', '781250000000000000', '24414062500000000', '50000000000000000000', '6250000000000000000', '12500000000000000000', '100000000000000000000', '20000000000000000000', '6250000000000000000', '1220703125000000', '12207031250000000', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '762939453125000', '1220703125000000', '762939453125000', '1525878906250000', '48828125000000000', '12207031250000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '12207031250000000', '6103515625000000', '6103515625000000', '97656250000000000', '2441406250000000', '3051757812500000', '244140625000000', '305175781250000', '762939453125000', '6103515625000000', '12207031250000000', '3051757812500000', '488281250000000', '7812500000000000', '6103515625000000', '3051757812500000', '1525878906250000', '1525878906250000', '95367431640625', '488281250000000', '6103515625000000', '6103515625000000', '762939453125000', '195312500000000000', '12207031250000000', '3051757812500000', '305175781250000', '6103515625000000', '3051757812500000', '2441406250000000', '6103515625000000', '6103515625000000', '610351562500000', '97656250000000000', '1525878906250000', '762939453125000', '1220703125000000', '3051757812500000', '762939453125000', '6103515625000000', '97656250000000', '6103515625000000', '97656250000000000', '97656250000000000', '1562500000000000000', '12207031250000000', '390625000000000000', '97656250000000000', '195312500000000000', '390625000000000000', '390625000000000000', '48828125000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '97656250000000000', '25000000000000000000', '195312500000000000', '781250000000000000', '390625000000000000', '390625000000000000', '95367431640625', '390625000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '48828125000000000', '48828125000000000', '95367431640625', '390625000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '12207031250000000', '781250000000000000', '12207031250000000', '12207031250000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '5000000000000000000', '195312500000000000', '97656250000000000', '1250000000000000', '3051757812500000', '97656250000000000', '195312500000000000', '25000000000000000000', '5000000000000000000', '195312500000000000', '97656250000000000', '1250000000000000', '3051757812500000', '97656250000000000', '195312500000000000', '25000000000000000000', '1562500000000000000', '97656250000000000', '97656250000000000', '2500000000000000000', '3051757812500000', '97656250000000000', '97656250000000000', '12500000000000000000', '1562500000000000000', '97656250000000000', '97656250000000000', '2500000000000000000', '3051757812500000', '97656250000000000', '97656250000000000', '12500000000000000000', '6250000000000000000', '390625000000000000', '3906250000000000', '25000000000000000000', '12207031250000000', '3906250000000000', '390625000000000000', '25000000000000000000', '6250000000000000000', '390625000000000000', '3906250000000000', '25000000000000000000', '12207031250000000', '3906250000000000', '390625000000000000', '25000000000000000000', '500000000000000000', '390625000000000000', '48828125000000000', '1250000000000000000', '381469726562500', '97656250000000000', '9765625000000000', '12500000000000000000', '500000000000000000', '390625000000000000', '48828125000000000', '1250000000000000000', '381469726562500', '97656250000000000', '9765625000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '19531250000000000', '12207031250000000', '25000000000000000000', '97656250000000000', '97656250000000000', '781250000000000000', '25000000000000000000', '390625000000000000', '19531250000000000', '12207031250000000', '25000000000000000000', '97656250000000000', '97656250000000000', '781250000000000000', '25000000000000000000', '195312500000000000', '12207031250000000', '5000000000000000000', '12207031250000000', '24414062500000000', '12207031250000000', '1250000000000000000', '25000000000000000000', '195312500000000000', '12207031250000000', '5000000000000000000', '12207031250000000', '24414062500000000', '12207031250000000', '1250000000000000000', '100000000000000000000', '6250000000000000000', '2500000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '610351562500000', '12207031250000000', '48828125000000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '610351562500000', '762939453125000', '19073486328125', '3051757812500000', '6103515625000000', '762939453125000', '12207031250000000', '6103515625000000', '1220703125000000', '6103515625000000', '762939453125000', '305175781250000', '381469726562500', '3051757812500000', '762939453125000', '381469726562500', '3051757812500000', '762939453125000', '6103515625000000', '6103515625000000', '6103515625000000', '305175781250000', '95367431640625', '6103515625000000', '3051757812500000', '12207031250000000', '762939453125000', '1525878906250000', '6103515625000000', '4882812500000000', '24414062500000000', '610351562500000', '6103515625000000', '305175781250000', '1525878906250000', '3051757812500000', '1525878906250000', '12207031250000000', '3051757812500000', '305175781250000', '6103515625000000', '2441406250000000', '24414062500000000', '610351562500000', '6103515625000000', '1525878906250000', '1525878906250000', '12207031250000000', '76293945312500', '3051757812500000', '1525878906250000', '190734863281250', '1525878906250000', '24414062500000000', '24414062500000000', '3051757812500000', '6103515625000000', '1525878906250000', '762939453125000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '390625000000000000', '1250000000000000000', '97656250000000000', '195312500000000000', '1250000000000000000', '24414062500000000', '195312500000000000', '6250000000000000000', '48828125000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '24414062500000000', '7812500000000000', '15625000000000000', '390625000000000000', '195312500000000000', '625000000000000000', '781250000000000000', '195312500000000000', '24414062500000000', '48828125000000000', '48828125000000000', '781250000000000000', '195312500000000000', '781250000000000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '1250000000000000000', '6250000000000000000', '3125000000000000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '1250000000000000000', '6250000000000000000', '3125000000000000000', '78125000000000000', '156250000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '312500000000000000', '156250000000000000', '781250000000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '3125000000000000000', '781250000000000000', '195312500000000000', '312500000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '195312500000000000', '195312500000000000', '3125000000000000000', '12500000000000000000', '12500000000000000000', '500000000000000000', '12500000000000000000', '1562500000000000000', '1562500000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '781250000000000000', '390625000000000000', '31250000000000000', '195312500000000000', '4882812500000000', '4882812500000000', '24414062500000000', '31250000000000000', '390625000000000000', '781250000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '500000000000000000', '781250000000000000', '781250000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '31250000000000000', '48828125000000000', '195312500000000000', '390625000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '195312500000000000', '48828125000000000', '31250000000000000', '25000000000000000000', '1000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '625000000000000000', '390625000000000000', '781250000000000000', '1562500000000000000', '24414062500000000', '24414062500000000', '6103515625000000', '781250000000000000', '390625000000000000', '625000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '12500000000000000000', '625000000000000000', '625000000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '1562500000000000000', '5000000000000000000', '20000000000000000000', '24414062500000000', '100000000000000000000', '3051757812500000', '6103515625000000', '244140625000000', '381469726562500', '1220703125000000', '4882812500000000', '3051757812500000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '1525878906250000', '762939453125000', '762939453125000', '3051757812500000', '6103515625000000', '381469726562500', '610351562500000', '6103515625000000', '3051757812500000', '762939453125000', '3051757812500000', '762939453125000', '1525878906250000', '3051757812500000', '6103515625000000', '48828125000000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '381469726562500', '3051757812500000', '190734863281250', '3051757812500000', '190734863281250', '48828125000000000', '6103515625000000', '1525878906250000', '152587890625000', '6103515625000000', '95367431640625', '1525878906250000', '76293945312500', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '244140625000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '305175781250000', '1220703125000000', '610351562500000', '6103515625000000', '12207031250000000', '1525878906250000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '97656250000000000', '305175781250000', '50000000000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '97656250000000000', '781250000000000000', '24414062500000000', '195312500000000000', '12207031250000000', '976562500000000', '7812500000000000', '20000000000000000000', '390625000000000000', '24414062500000000', '4882812500000000', '39062500000000000', '97656250000000000', '78125000000000000', '25000000000000000000', '390625000000000000', '19531250000000000', '24414062500000000', '61035156250000', '390625000000000000', '15625000000000000', '195312500000000000', '4882812500000000', '12207031250000000', '97656250000000000', '78125000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '3051757812500000', '625000000000000000', '24414062500000000', '625000000000000000', '390625000000000000', '97656250000000000', '488281250000000', '24414062500000000', '781250000000000000', '3051757812500000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '12207031250000000', '781250000000000000', '4882812500000000', '3125000000000000000', '390625000000000000', '78125000000000000', '24414062500000000', '24414062500000000', '625000000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '48828125000000000', '1562500000000000000', '48828125000000000', '390625000000000000', '97656250000000000', '97656250000000000', '24414062500000000', '3051757812500000', '1562500000000000000', '781250000000000000', '5000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '625000000000000000', '12207031250000000', '24414062500000000', '78125000000000000', '78125000000000000', '1562500000000000000', '12207031250000000', '625000000000000000', '390625000000000000', '1000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '781250000000000000', '625000000000000000', '24414062500000000', '1562500000000000000', '390625000000000000', '195312500000000000', '976562500000000', '24414062500000000', '625000000000000000', '781250000000000000', '5000000000000000000', '25000000000000000000', '500000000000000000', '25000000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '500000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '312500000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '48828125000000000', '3125000000000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '9765625000000000', '50000000000000000000', '12500000000000000000', '50000000000000000000', '1525878906250000', '1525878906250000', '6103515625000000', '95367431640625', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '3051757812500000', '48828125000000000', '3051757812500000', '3051757812500000', '95367431640625', '1525878906250000', '1525878906250000', '3051757812500000', '61035156250000', '3051757812500000', '19073486328125', '3051757812500000', '6103515625000000', '9765625000000000', '6103515625000000', '1525878906250000', '762939453125000', '6103515625000000', '762939453125000', '3051757812500000', '190734863281250', '3051757812500000', '24414062500000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '6103515625000000', '95367431640625', '762939453125000', '190734863281250', '3051757812500000', '381469726562500', '610351562500000', '48828125000000000', '6103515625000000', '762939453125000', '3051757812500000', '3051757812500000', '3051757812500000', '152587890625000', '610351562500000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '48828125000000000', '1525878906250000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '190734863281250', '3051757812500000', '6103515625000000', '1220703125000000', '3051757812500000', '1220703125000000', '48828125000000000', '3051757812500000', '3051757812500000', '195312500000000000', '48828125000000000', '97656250000000000', '156250000000000000', '9765625000000000', '12207031250000000', '381469726562500', '390625000000000000', '97656250000000000', '25000000000000000000', '9765625000000000', '97656250000000000', '97656250000000000', '97656250000000000', '4882812500000000', '390625000000000', '781250000000000000', '195312500000000000', '97656250000000000', '390625000000000000', '24414062500000000', '78125000000000000', '50000000000000000000', '97656250000000000', '48828125000000000', '97656250000000000', '48828125000000000', '6250000000000000000', '12207031250000000', '12207031250000000', '97656250000000000', '390625000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '48828125000000000', '390625000000000000', '5000000000000000000', '12500000000000000000', '19531250000000000', '48828125000000000', '12207031250000000', '5000000000000000000', '390625000000000000', '390625000000000000', '5000000000000000000', '12500000000000000000', '19531250000000000', '48828125000000000', '12207031250000000', '5000000000000000000', '390625000000000000', '97656250000000000', '3125000000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '190734863281250', '1562500000000000000', '97656250000000000', '97656250000000000', '3125000000000000000', '3125000000000000000', '97656250000000000', '97656250000000000', '190734863281250', '1562500000000000000', '97656250000000000', '195312500000000000', '6250000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '12500000000000000000', '195312500000000000', '195312500000000000', '6250000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '12500000000000000000', '195312500000000000', '195312500000000000', '12500000000000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '12500000000000000000', '390625000000000000', '195312500000000000', '12500000000000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '12500000000000000000', '390625000000000000', '195312500000000000', '1250000000000000000', '3125000000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '1250000000000000000', '195312500000000000', '195312500000000000', '1250000000000000000', '3125000000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '1250000000000000000', '195312500000000000', '97656250000000000', '1562500000000000000', '12500000000000000000', '48828125000000000', '1525878906250000', '1220703125000000', '12500000000000000000', '390625000000000000', '97656250000000000', '1562500000000000000', '12500000000000000000', '48828125000000000', '1525878906250000', '1220703125000000', '12500000000000000000', '390625000000000000', '100000000000000000000', '10000000000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '12500000000000000000', '6103515625000000', '24414062500000000', '12207031250000000', '6103515625000000', '762939453125000', '190734863281250', '3051757812500000', '12207031250000000', '762939453125000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '610351562500000', '1220703125000000', '3051757812500000', '1525878906250000', '305175781250000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '1525878906250000', '610351562500000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '762939453125000', '6103515625000000', '3051757812500000', '1220703125000000', '6103515625000000', '3051757812500000', '381469726562500', '24414062500000000', '24414062500000000', '6103515625000000', '610351562500000', '12207031250000000', '381469726562500', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '1525878906250000', '12207031250000000', '762939453125000', '12207031250000000', '1525878906250000', '762939453125000', '3051757812500000', '95367431640625', '6103515625000000', '6103515625000000', '24414062500000000', '1525878906250000', '3051757812500000', '12207031250000000', '762939453125000', '12207031250000000', '1525878906250000', '1525878906250000', '6103515625000000', '19073486328125', '3125000000000000000', '19531250000000000', '15625000000000000', '250000000000000000', '97656250000000000', '195312500000000000', '390625000000000000', '195312500000000', '25000000000000000000', '390625000000000000', '195312500000000000', '2000000000000000000', '1562500000000000000', '9765625000000000', '1562500000000000', '390625000000000000', '195312500000000000', '625000000000000000', '195312500000000000', '195312500000000000', '781250000000000000', '97656250000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '6103515625000000', '6103515625000000', '195312500000000000', '781250000000000000', '781250000000000000', '195312500000000000', '3125000000000000000', '48828125000000000', '48828125000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '48828125000000000', '48828125000000000', '97656250000000000', '1562500000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '48828125000000000', '48828125000000000', '3125000000000000000', '312500000000000000', '390625000000000000', '24414062500000000', '50000000000000000000', '12207031250000000', '97656250000000000', '78125000000000000', '50000000000000000000', '312500000000000000', '390625000000000000', '24414062500000000', '50000000000000000000', '12207031250000000', '97656250000000000', '78125000000000000', '50000000000000000000', '10000000000000000000', '19531250000000000', '95367431640625', '5000000000000000000', '48828125000000', '24414062500000000', '195312500000000000', '10000000000000000000', '10000000000000000000', '19531250000000000', '95367431640625', '5000000000000000000', '48828125000000', '24414062500000000', '195312500000000000', '10000000000000000000', '5000000000000000000', '390625000000000000', '24414062500000000', '12500000000000000000', '12207031250000000', '610351562500000', '78125000000000000', '5000000000000000000', '5000000000000000000', '390625000000000000', '24414062500000000', '12500000000000000000', '12207031250000000', '610351562500000', '78125000000000000', '5000000000000000000', '100000000000000000000', '390625000000000000', '48828125000000000', '100000000000000000000', '488281250000000', '97656250000000000', '2441406250000000', '39062500000000000', '100000000000000000000', '390625000000000000', '48828125000000000', '100000000000000000000', '488281250000000', '97656250000000000', '2441406250000000', '39062500000000000', '100000000000000000000', '97656250000000000', '19531250000000000', '12207031250000000', '25000000000000000000', '19531250000000000', '195312500000000000', '100000000000000000000', '100000000000000000000', '97656250000000000', '19531250000000000', '12207031250000000', '25000000000000000000', '19531250000000000', '195312500000000000', '100000000000000000000', '12500000000000000000', '3051757812500000', '97656250000000000', '3125000000000000000', '12207031250000000', '97656250000000000', '78125000000000000', '12500000000000000000', '12500000000000000000', '3051757812500000', '97656250000000000', '3125000000000000000', '12207031250000000', '97656250000000000', '78125000000000000', '12500000000000000000', '100000000000000000000', '781250000000000000', '20000000000000000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '762939453125000', '6103515625000000', '381469726562500', '3051757812500000', '762939453125000', '190734863281250', '6103515625000000', '6103515625000000', '12207031250000000', '6103515625000000', '1525878906250000', '1525878906250000', '1525878906250000', '3051757812500000', '305175781250000', '6103515625000000', '305175781250000', '3051757812500000', '1220703125000000', '6103515625000000', '24414062500000000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '1525878906250000', '381469726562500', '6103515625000000', '1220703125000000', '244140625000000', '12207031250000000', '4882812500000000', '6103515625000000', '190734863281250', '762939453125000', '6103515625000000', '381469726562500', '762939453125000', '762939453125000', '3051757812500000', '1525878906250000', '3051757812500000', '4882812500000000', '305175781250000', '3051757812500000', '3051757812500000', '381469726562500', '610351562500000', '152587890625000', '3051757812500000', '762939453125000', '6103515625000000', '762939453125000', '6103515625000000', '24414062500000000', '4882812500000000', '3051757812500000', '610351562500000', '1525878906250000', '305175781250000', '1525878906250000', '3051757812500000', '1525878906250000', '1525878906250000', '95367431640625', '6103515625000000', '24414062500000000', '6103515625000000', '95367431640625', '305175781250000', '6250000000000000000', '97656250000000000', '24414062500000000', '12500000000000000000', '195312500000000000', '390625000000000000', '3125000000000000000', '39062500000000000', '390625000000000000', '195312500000000000', '195312500000000000', '1250000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '97656250000000000', '195312500000000000', '12500000000000000000', '100000000000000000000', '97656250000000000', '195312500000000000', '12207031250000000', '20000000000000000000', '12207031250000000', '195312500000000000', '97656250000000000', '3125000000000000000', '100000000000000000000', '73242187500000000', '117187500000000000', '292968750000000000', '20000000000000000000', '292968750000000000', '585937500000000000', '146484375000000000', '3125000000000000000', '10000000000000000000', '97656250000000000', '195312500000000000', '24414062500000000', '2500000000000000000', '24414062500000000', '195312500000000000', '97656250000000000', '50000000000000000000', '10000000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '2500000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '50000000000000000000', '4000000000000000000', '195312500000000000', '97656250000000000', '48828125000000000', '20000000000000000000', '48828125000000000', '97656250000000000', '3906250000000000', '50000000000000000000', '4000000000000000000', '78125000000000000', '390625000000000000', '78125000000000000', '20000000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '50000000000000000000', '100000000000000000000', '195312500000000000', '6103515625000000', '48828125000000000', '100000000000000000000', '48828125000000000', '6103515625000000', '195312500000000000', '25000000000000000000', '100000000000000000000', '292968750000000000', '585937500000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '585937500000000000', '36621093750000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '1525878906250000', '10000000000000000000', '1525878906250000', '97656250000000000', '195312500000000000', '50000000000000000000', '25000000000000000000', '39062500000000000', '24414062500000000', '48828125000000000', '10000000000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '50000000000000000000', '25000000000000000000', '12207031250000000', '195312500000000000', '48828125000000000', '312500000000000000', '48828125000000000', '195312500000000000', '12207031250000000', '25000000000000000000', '25000000000000000000', '234375000000000000', '1171875000000000000', '1171875000000000000', '312500000000000000', '1171875000000000000', '234375000000000000', '1171875000000000000', '25000000000000000000', '100000000000000000000', '12500000000000000000', '25000000000000000000', '100000000000000000000', '6250000000000000000', '24414062500000000', '762939453125000', '305175781250000', '1525878906250000', '3051757812500000', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '381469726562500', '2441406250000000', '1525878906250000', '1525878906250000', '762939453125000', '1525878906250000', '762939453125000', '6103515625000000', '381469726562500', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '24414062500000000', '1220703125000000', '3051757812500000', '762939453125000', '6103515625000000', '381469726562500', '6103515625000000', '1525878906250000', '762939453125000', '24414062500000000', '6103515625000000', '3051757812500000', '12207031250000000', '6103515625000000', '1525878906250000', '3051757812500', '1525878906250000', '1525878906250000', '244140625000000', '190734863281250', '6103515625000000', '24414062500000000', '6103515625000000', '3051757812500000', '24414062500000000', '190734863281250', '762939453125000', '762939453125000', '6103515625000000', '381469726562500', '3051757812500000', '762939453125000', '6103515625000000', '6103515625000000', '610351562500000', '1525878906250000', '24414062500000000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '1525878906250000', '1525878906250000', '24414062500000000', '6103515625000000', '122070312500000', '12207031250000000', '1525878906250000', '762939453125000', '625000000000000000', '12500000000000000000', '12500000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '31250000000000000', '12500000000000000000', '50000000000000000000', '97656250000000000', '39062500000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '78125000000000000', '97656250000000000', '9765625000000000', '97656250000000000', '97656250000000000', '390625000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '3125000000000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '4882812500000000', '97656250000000000', '390625000000000000', '12207031250000000', '390625000000000000', '390625000000000000', '48828125000000000', '19531250000000000', '6250000000000000000', '25000000000000000000', '3906250000000000', '12207031250000000', '12207031250000000', '50000000000000000000', '390625000000000000', '97656250000000000', '50000000000000000000', '25000000000000000000', '3906250000000000', '12207031250000000', '12207031250000000', '50000000000000000000', '390625000000000000', '97656250000000000', '50000000000000000000', '25000000000000000000', '1525878906250000', '195312500000000000', '2441406250000000', '100000000000000000000', '78125000000000000', '97656250000000000', '100000000000000000000', '25000000000000000000', '1525878906250000', '195312500000000000', '2441406250000000', '100000000000000000000', '78125000000000000', '97656250000000000', '100000000000000000000', '5000000000000000000', '48828125000000000', '48828125000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '100000000000000000000', '5000000000000000000', '48828125000000000', '48828125000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '48828125000000000', '100000000000000000000', '10000000000000000000', '48828125000000000', '390625000000000000', '762939453125000', '25000000000000000000', '390625000000000000', '12207031250000000', '50000000000000000000', '10000000000000000000', '48828125000000000', '390625000000000000', '762939453125000', '25000000000000000000', '390625000000000000', '12207031250000000', '50000000000000000000', '4882812500000000', '97656250000000000', '195312500000000000', '95367431640625', '50000000000000000000', '390625000000000000', '24414062500000000', '10000000000000000000', '4882812500000000', '97656250000000000', '195312500000000000', '95367431640625', '50000000000000000000', '390625000000000000', '24414062500000000', '10000000000000000000', '20000000000000000000', '97656250000000000', '48828125000000000', '12207031250000000', '50000000000000000000', '97656250000000000', '48828125000000000', '100000000000000000000', '20000000000000000000', '97656250000000000', '48828125000000000', '12207031250000000', '50000000000000000000', '97656250000000000', '48828125000000000', '100000000000000000000', '50000000000000000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '3125000000000000000', '50000000000000000000', '305175781250000', '6103515625000000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '12207031250000000', '1525878906250000', '762939453125000', '244140625000000', '152587890625000', '95367431640625', '1525878906250000', '6103515625000000', '190734863281250', '762939453125000', '762939453125000', '6103515625000000', '12207031250000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '95367431640625', '6103515625000000', '76293945312500', '3051757812500000', '1525878906250000', '1525878906250000', '4882812500000000', '2441406250000000', '6103515625000000', '152587890625000', '1220703125000000', '3051757812500000', '1525878906250000', '1525878906250000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '24414062500000000', '24414062500000000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '305175781250000', '244140625000000', '61035156250000', '762939453125000', '1525878906250000', '6103515625000000', '24414062500000000', '24414062500000000', '6103515625000000', '6103515625000000', '610351562500000', '610351562500000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '762939453125000', '6103515625000000', '24414062500000000', '4882812500000000', '1525878906250000', '6103515625000000', '6103515625000000', '381469726562500', '3125000000000000000', '39062500000000000', '12207031250000000', '3125000000000000000', '97656250000000000', '24414062500000000', '976562500000000', '12500000000000000000', '19531250000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '1953125000000000', '78125000000000000', '12500000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '97656250000000000', '48828125000000000', '4882812500000000', '390625000000000000', '4882812500000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '4882812500000000', '390625000000000000', '97656250000000000', '97656250000000000', '625000000000000000', '48828125000000000', '24414062500000000', '6250000000000000000', '4882812500000000', '1250000000000000000', '625000000000000000', '48828125000000000', '24414062500000000', '6250000000000000000', '4882812500000000', '1250000000000000000', '25000000000000000000', '4882812500000000', '1562500000000000000', '12207031250000000', '48828125000000000', '25000000000000000000', '25000000000000000000', '4882812500000000', '1562500000000000000', '12207031250000000', '48828125000000000', '25000000000000000000', '12500000000000000000', '48828125000000000', '6250000000000000000', '6103515625000000', '24414062500000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '6250000000000000000', '6103515625000000', '24414062500000000', '12500000000000000000', '625000000000000000', '24414062500000000', '25000000000000000000', '12207031250000000', '24414062500000000', '25000000000000000000', '625000000000000000', '24414062500000000', '25000000000000000000', '12207031250000000', '24414062500000000', '25000000000000000000', '50000000000000000000', '24414062500000000', '50000000000000000000', '6103515625000000', '4882812500000000', '6250000000000000000', '50000000000000000000', '24414062500000000', '50000000000000000000', '6103515625000000', '4882812500000000', '6250000000000000000', '50000000000000000000', '9765625000000000', '3051757812500000', '50000000000000000000', '48828125000000000', '25000000000000000000', '50000000000000000000', '9765625000000000', '3051757812500000', '50000000000000000000', '48828125000000000', '25000000000000000000', '500000000000000000', '100000000000000000000', '100000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '1525878906250000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '610351562500000', '6103515625000000', '24414062500000000', '6103515625000000', '381469726562500', '1525878906250000', '3051757812500000', '1525878906250000', '152587890625000', '1525878906250000', '6103515625000000', '381469726562500', '24414062500000000', '3051757812500000', '1525878906250000', '1525878906250000', '76293945312500', '1525878906250000', '15258789062500', '6103515625000000', '6103515625000000', '244140625000000', '122070312500000', '610351562500000', '1525878906250000', '381469726562500', '1525878906250000', '2441406250000', '95367431640625', '3051757812500000', '244140625000000', '3051757812500000', '762939453125000', '1525878906250000', '3051757812500000', '152587890625000', '762939453125000', '381469726562500', '381469726562500', '6103515625000000', '6103515625000000', '1220703125000000', '1220703125000000', '1525878906250000', '1525878906250000', '305175781250000', '1525878906250000', '762939453125000', '1525878906250000', '762939453125000', '6103515625000000', '6103515625000000', '24414062500000000', '1220703125000000', '122070312500000', '390625000000000000', '195312500000000000', '97656250000000000', '781250000000000000', '195312500000000000', '195312500000000000', '48828125000000000', '195312500000000000', '195312500000000000', '390625000000000000', '24414062500000000', '5000000000000000000', '97656250000000000', '48828125000000000', '25000000000000000000', '24414062500000000', '97656250000000000', '12500000000000000000', '3125000000000000000', '3125000000000000000', '195312500000000', '12207031250000000', '4882812500000000', '1562500000000000000', '3125000000000000000', '3125000000000000000', '195312500000000', '12207031250000000', '4882812500000000', '1562500000000000000', '25000000000000000000', '5000000000000000000', '48828125000000000', '6103515625000000', '3051757812500000', '3125000000000000000', '25000000000000000000', '5000000000000000000', '48828125000000000', '6103515625000000', '3051757812500000', '3125000000000000000', '6250000000000000000', '762939453125000', '48828125000000000', '48828125000000000', '25000000000000000000', '25000000000000000000', '6250000000000000000', '762939453125000', '48828125000000000', '48828125000000000', '25000000000000000000', '25000000000000000000', '50000000000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '100000000000000000000', '100000000000000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '6250000000000000000', '12500000000000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '6250000000000000000', '12500000000000000000', '100000000000000000000', '100000000000000000000', '4882812500000000', '3051757812500000', '12207031250000000', '25000000000000000000', '100000000000000000000', '100000000000000000000', '4882812500000000', '3051757812500000', '12207031250000000', '25000000000000000000', '50000000000000000000', '100000000000000000000', '25000000000000000000', '12500000000000000000', '100000000000000000000', '50000000000000000000', '1525878906250000', '3051757812500000', '1525878906250000', '762939453125000', '6103515625000000', '12207031250000000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '38146972656250', '381469726562500', '3051757812500000', '762939453125000', '6103515625000000', '24414062500000000', '122070312500000', '305175781250000', '1220703125000000', '3051757812500000', '152587890625000', '1525878906250000', '762939453125000', '305175781250000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '381469726562500', '381469726562500', '1525878906250000', '762939453125000', '762939453125000', '305175781250000', '1220703125000000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '3051757812500000', '152587890625000', '1525878906250000', '762939453125000', '610351562500000', '6103515625000000', '3051757812500000', '610351562500000', '3051757812500000', '152587890625000', '762939453125000', '95367431640625', '190734863281250', '1525878906250000', '12207031250000000', '1525878906250000', '1220703125000000', '381469726562500', '305175781250000', '390625000000000000', '195312500000000000', '39062500000000000', '488281250000000', '48828125000000000', '195312500000000000', '2441406250000000', '1562500000000000000', '48828125000000000', '1220703125000000', '50000000000000000000', '97656250000000000', '39062500000000000', '7812500000000000', '9765625000000000', '390625000000000000', '50000000000000000000', '2441406250000000', '195312500000000000', '781250000000000000', '781250000000000000', '195312500000000000', '97656250000000000', '6103515625000000', '781250000000000000', '6103515625000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '6103515625000000', '781250000000000000', '195312500000000000', '195312500000000000', '781250000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '4882812500000000', '781250000000000000', '20000000000000000000', '25000000000000000000', '12207031250000000', '100000000000000000000', '20000000000000000000', '25000000000000000000', '12207031250000000', '100000000000000000000', '2500000000000000000', '1220703125000000', '12500000000000000000', '3125000000000000000', '2500000000000000000', '1220703125000000', '12500000000000000000', '3125000000000000000', '5000000000000000000', '25000000000000000000', '1525878906250000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '1525878906250000', '12500000000000000000', '6250000000000000000', '6250000000000000000', '12207031250000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '12207031250000000', '1562500000000000000', '6250000000000000000', '12207031250000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '12207031250000000', '1562500000000000000', '6250000000000000000', '1562500000000000000', '12207031250000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '12207031250000000', '781250000000000000', '1562500000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '50000000000000000000', '25000000000000000000', '10000000000000000000', '1525878906250000', '762939453125000', '762939453125000', '152587890625000', '6103515625000000', '3051757812500000', '190734863281250', '3051757812500000', '3051757812500000', '381469726562500', '3051757812500000', '762939453125000', '6103515625000000', '6103515625000000', '1525878906250000', '381469726562500', '190734863281250', '1525878906250000', '762939453125000', '610351562500000', '3051757812500000', '6103515625000000', '1525878906250000', '610351562500000', '3051757812500000', '152587890625000', '610351562500000', '762939453125000', '6103515625000000', '6103515625000000', '1220703125000000', '1525878906250000', '3051757812500000', '1525878906250000', '610351562500000', '381469726562500', '762939453125000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '190734863281250', '1525878906250000', '6103515625000000', '610351562500000', '6103515625000000', '152587890625000', '1525878906250000', '97656250000000000', '97656250000000000', '195312500000000000', '39062500000000000', '97656250000000000', '48828125000000000', '390625000000000000', '390625000000000000', '12207031250000000', '12207031250000000', '390625000000000000', '12207031250000000', '12207031250000000', '781250000000000000', '12207031250000000', '781250000000000000', '4', '4', '4', '4', '24414062500000000', '24414062500000000', '24414062500000000', '24414062500000000', '1', '1', '4', '4', '4', '4', '195312500000000000', '6103515625000000', '195312500000000000', '12207031250000000', '9765625000000000', '195312500000000000', '9765625000000000', '9765625000000000', '6103515625000000', '195312500000000000', '97656250000000000', '97656250000000000', '195312500000000000', '12207031250000000', '195312500000000000', '195312500000000000', '195312500000000000', '195312500000000000', '6103515625000000', '195312500000000000', '6103515625000000', '6103515625000000', '97656250000000000', '195312500000000000', '195312500000000000', '97656250000000000', '390625000000000000', '195312500000000000', '195312500000000000', '156250000000000000', '48828125000000000', '156250000000000000', '195312500000000000', '195312500000000000', '48828125000000000', '3125000000000000000', '48828125000000000', '195312500000000000', '195312500000000000', '156250000000000000', '48828125000000000', '156250000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '97656250000000000', '24414062500000000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '390625000000000000', '390625000000000000', '78125000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '78125000000000000', '48828125000000000', '48828125000000000', '78125000000000000', '6103515625000000', '78125000000000000', '4', '4', '4', '4', '1', '4', '4', '4', '4', '48828125000000000', '24414062500000000', '1220703125000000', '48828125000000000', '48828125000000000', '24414062500000000', '1220703125000000', '48828125000000000', '48828125000000000', '48828125000000000', '6103515625000000', '12207031250000000', '48828125000000000', '48828125000000000', '6103515625000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '48828125000000000', '24414062500000000', '1525878906250000', '48828125000000000', '48828125000000000', '24414062500000000', '1525878906250000', '48828125000000000', '24414062500000000', '762939453125000', '12207031250000000', '24414062500000000', '24414062500000000', '762939453125000', '12207031250000000', '24414062500000000', '48828125000000000', '24414062500000000', '12207031250000000', '48828125000000000', '48828125000000000', '24414062500000000', '12207031250000000', '48828125000000000', '781250000000000000', '1562500000000000000', '24414062500000000', '390625000000000000', '781250000000000000', '1562500000000000000', '6103515625000000', '48828125000000000', '12207031250000000', '24414062500000000', '2441406250000000', '3051757812500000', '15258789062500', '95367431640625', '24414062500000000', '24414062500000000', '48828125000000000', '9765625000000000', '1525878906250000', '1525878906250000', '12207031250000000', '24414062500000000', '4882812500000000', '4882812500000000', '24414062500000000', '48828125000000000', '488281250000000', '152587890625000', '6103515625000000', '3051757812500000', '12207031250000000', '24414062500000000', '12207031250000000', '12207031250000000', '12207031250000000', '6103515625000000', '12207031250000000', '12207031250000000', '48828125000000000', '381469726562500', '48828125000000000', '48828125000000000', '12207031250000000', '122070312500000', '12207031250000000', '12207031250000000', '6103515625000000', '1525878906250000', '12207031250000000', '9765625000000000', '6103515625000000', '12207031250000000', '6103515625000000', '24414062500000000', '390625000000000000', '6103515625000000', '195312500000000000', '762939453125000', '78125000000000000', '195312500000000000', '390625000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '39062500000000000', '390625000000000000', '390625000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '39062500000000000', '390625000000000000', '195312500000000000', '19073486328125', '12207031250000000', '12207031250000000', '97656250000000000', '195312500000000000', '195312500000000000', '19073486328125', '12207031250000000', '12207031250000000', '97656250000000000', '195312500000000000', '78125000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '12207031250000000', '78125000000000000', '78125000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '12207031250000000', '78125000000000000', '195312500000000000', '97656250000000000', '48828125000000000', '48828125000000000', '24414062500000000', '195312500000000000', '195312500000000000', '97656250000000000', '48828125000000000', '48828125000000000', '24414062500000000', '195312500000000000', '195312500000000000', '97656250000000000', '6103515625000000', '6103515625000000', '6103515625000000', '195312500000000000', '195312500000000000', '97656250000000000', '6103515625000000', '6103515625000000', '6103515625000000', '195312500000000000', '195312500000000000', '19531250000000000', '24414062500000000', '24414062500000000', '12207031250000000', '195312500000000000', '195312500000000000', '19531250000000000', '24414062500000000', '24414062500000000', '12207031250000000', '195312500000000000', '12500000000000000000', '50000000000000000000', '2500000000000000000', '100000000000000000000', '20000000000000000000', '6250000000000000000', '12207031250000000', '1525878906250000', '6103515625000000', '48828125000000000', '6103515625000000', '95367431640625', '3051757812500000', '6103515625000000', '3051757812500000', '610351562500000', '12207031250000000', '3051757812500000', '6103515625000000', '97656250000000000', '1525878906250000', '3051757812500000', '762939453125000', '1525878906250000', '610351562500000', '6103515625000000', '12207031250000000', '6103515625000000', '12207031250000000', '97656250000000000', '610351562500000', '3051757812500000', '610351562500000', '6103515625000000', '3051757812500000', '152587890625000', '1525878906250000', '1220703125000000', '3051757812500000', '97656250000000000', '12207031250000000', '3051757812500000', '610351562500000', '3051757812500000', '152587890625000', '6103515625000000', '95367431640625', '305175781250000', '12207031250000000', '97656250000000000', '3051757812500000', '152587890625000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '3051757812500000', '1220703125000000', '12207031250000000', '12207031250000000', '2441406250000000', '3051757812500000', '6103515625000000', '610351562500000', '3051757812500000', '6103515625000000', '390625000000000000', '100000000000000000000', '1525878906250000', '125000000000000000', '39062500000000000', '9765625000000000', '390625000000000000', '12500000000000000000', '48828125000000000', '97656250000000000', '19531250000000000', '97656250000000000', '3125000000000000000', '390625000000000000', '195312500000000000', '125000000000000000', '390625000000000000', '195312500000000000', '781250000000000000', '48828125000000000', '390625000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '48828125000000000', '781250000000000000', '12500000000000000000', '48828125000000000', '12500000000000000000', '97656250000000000', '24414062500000000', '12207031250000000', '48828125000000000', '6250000000000000000', '12500000000000000000', '48828125000000000', '12500000000000000000', '97656250000000000', '24414062500000000', '12207031250000000', '48828125000000000', '6250000000000000000', '3125000000000000000', '6103515625000000', '12500000000000000', '3906250000000000', '3906250000000000', '12207031250000000', '6103515625000000', '3125000000000000000', '3125000000000000000', '6103515625000000', '12500000000000000', '3906250000000000', '3906250000000000', '12207031250000000', '6103515625000000', '3125000000000000000', '1250000000000000000', '2441406250000000', '3125000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '2441406250000000', '1250000000000000000', '1250000000000000000', '2441406250000000', '3125000000000000000', '390625000000000000', '390625000000000000', '12207031250000000', '2441406250000000', '1250000000000000000', '625000000000000000', '12207031250000000', '781250000000000000', '195312500000000000', '195312500000000000', '2441406250000000', '12207031250000000', '3125000000000000000', '625000000000000000', '12207031250000000', '781250000000000000', '195312500000000000', '195312500000000000', '2441406250000000', '12207031250000000', '3125000000000000000', '12500000000000000000', '48828125000000000', '12207031250000000', '97656250000000000', '195312500000000000', '625000000000000000', '9765625000000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '12207031250000000', '97656250000000000', '195312500000000000', '625000000000000000', '9765625000000000', '12500000000000000000', '25000000000000000000', '12500000000000000000', '100000000000000000000', '100000000000000000000', '25000000000000000000', '50000000000000000000', '1525878906250000', '3051757812500000', '6103515625000000', '762939453125000', '12207031250000000', '1525878906250000', '3051757812500000', '12207031250000000', '381469726562500', '3051757812500000', '1525878906250000', '1525878906250000', '1220703125000000', '6103515625000000', '762939453125000', '12207031250000000', '24414062500000000', '6103515625000000', '3051757812500000', '12207031250000000', '190734863281250', '2441406250000000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '12207031250000000', '1525878906250000', '762939453125000', '1525878906250000', '76293945312500', '3051757812500000', '762939453125000', '1525878906250000', '1525878906250000', '6103515625000000', '610351562500000', '6103515625000000', '24414062500000000', '24414062500000000', '1220703125000000', '1525878906250000', '6103515625000000', '190734863281250', '762939453125000', '762939453125000', '610351562500000', '1220703125000000', '381469726562500', '1220703125000000', '12207031250000000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '762939453125000', '762939453125000', '1525878906250000', '381469726562500', '1525878906250000', '6103515625000000', '6103515625000000', '38146972656250', '3051757812500000', '1220703125000000', '3051757812500000', '12207031250000000', '610351562500000', '6103515625000000', '610351562500000', '3051757812500000', '195312500000000000', '3906250000000000', '48828125000000000', '6250000000000000000', '195312500000000000', '390625000000000000', '195312500000000000', '4882812500000000', '6250000000000000000', '78125000000000000', '48828125000000000', '3125000000000000000', '50000000000000000000', '48828125000000000', '3051757812500000', '6250000000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '50000000000000000000', '195312500000000', '25000000000000000000', '50000000000000000000', '12207031250000000', '3051757812500000', '15625000000000000', '195312500000000000', '50000000000000000000', '195312500000000', '25000000000000000000', '50000000000000000000', '12207031250000000', '3051757812500000', '15625000000000000', '195312500000000000', '50000000000000000000', '97656250000000000', '2500000000000000000', '10000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '195312500000000000', '50000000000000000000', '97656250000000000', '2500000000000000000', '10000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '390625000000000000', '6103515625000000', '97656250000000000', '10000000000000000000', '5000000000000000000', '97656250000000000', '50000000000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '97656250000000000', '10000000000000000000', '5000000000000000000', '97656250000000000', '50000000000000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '19531250000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '390625000000000000', '25000000000000000000', '19531250000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '12207031250000000', '390625000000000000', '97656250000000000', '5000000000000000000', '24414062500000000', '3125000000000000000', '25000000000000000000', '24414062500000000', '12207031250000000', '97656250000000000', '97656250000000000', '5000000000000000000', '24414062500000000', '3125000000000000000', '25000000000000000000', '24414062500000000', '12207031250000000', '97656250000000000', '390625000000000000', '100000000000000000000', '12207031250000000', '12500000000000000000', '20000000000000000000', '12207031250000000', '3051757812500000', '390625000000000000', '390625000000000000', '100000000000000000000', '12207031250000000', '12500000000000000000', '20000000000000000000', '12207031250000000', '3051757812500000', '390625000000000000', '25000000000000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '2000000000000000000', '25000000000000000000', '6103515625000000', '76293945312500', '95367431640625', '1220703125000000', '1525878906250000', '610351562500000', '381469726562500', '1220703125000000', '4882812500000000', '24414062500000000', '6103515625000000', '305175781250000', '1220703125000000', '6103515625000000', '1525878906250000', '762939453125000', '305175781250000', '3051757812500000', '1525878906250000', '1525878906250000', '12207031250000000', '24414062500000000', '6103515625000000', '762939453125000', '1220703125000000', '3051757812500000', '1525878906250000', '6103515625000000', '190734863281250', '1525878906250000', '1525878906250000', '6103515625000000', '24414062500000000', '610351562500000', '6103515625000000', '3051757812500000', '762939453125000', '1525878906250000', '381469726562500', '610351562500000', '381469726562500', '6103515625000000', '762939453125000', '3051757812500000', '24414062500000000', '24414062500000000', '381469726562500', '1525878906250000', '6103515625000000', '6103515625000000', '381469726562500', '6103515625000000', '61035156250000', '1525878906250000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '6103515625000000', '610351562500000', '76293945312500', '6103515625000000', '1525878906250000', '305175781250000', '1525878906250000', '6103515625000000', '1525878906250000', '762939453125000', '61035156250000', '3051757812500000', '6103515625000000', '3051757812500000', '195312500000000000', '78125000000000000', '2500000000000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '195312500000000000', '390625000000000000', '3125000000000000000', '1562500000000000', '78125000000000000', '1562500000000000000', '195312500000000000', '12500000000000000000', '390625000000000000', '195312500000000000', '12500000000000000000', '390625000000000000', '48828125000000000', '6103515625000000', '12500000000000000000', '50000000000000000000', '50000000000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '12500000000000000000', '50000000000000000000', '50000000000000000000', '48828125000000000', '6103515625000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '4882812500000000', '6103515625000000', '6103515625000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '4882812500000000', '6103515625000000', '12207031250000000', '6103515625000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '12207031250000000', '12207031250000000', '6103515625000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '12207031250000000', '381469726562500', '12207031250000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '48828125000000000', '381469726562500', '12207031250000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '48828125000000000', '24414062500000000', '10000000000000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '12207031250000000', '24414062500000000', '10000000000000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '12207031250000000', '48828125000000000', '12207031250000000', '100000000000000000000', '3125000000000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '12207031250000000', '100000000000000000000', '3125000000000000000', '100000000000000000000', '24414062500000000', '100000000000000000000', '25000000000000000000', '50000000000000000000', '12500000000000000000', '312500000000000000', '100000000000000000000', '6103515625000000', '1525878906250000', '305175781250000', '305175781250000', '381469726562500', '1525878906250000', '6103515625000000', '24414062500000000', '122070312500000', '3051757812500000', '6103515625000000', '6103515625000000', '762939453125000', '762939453125000', '1525878906250000', '152587890625000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '762939453125000', '3051757812500000', '1525878906250000', '762939453125000', '152587890625000', '762939453125000', '6103515625000000', '4882812500000000', '3051757812500000', '3051757812500000', '3051757812500000', '381469726562500', '152587890625000', '762939453125000', '610351562500000', '762939453125000', '6103515625000000', '6103515625000000', '1220703125000000', '190734863281250', '6103515625000000', '3051757812500000', '381469726562500', '190734863281250', '305175781250000', '762939453125000', '1525878906250000', '3051757812500000', '76293945312500', '762939453125000', '6103515625000000', '6103515625000000', '1525878906250000', '762939453125000', '762939453125000', '1525878906250000', '3051757812500000', '3051757812500000', '762939453125000', '3051757812500000', '24414062500000000', '390625000000000000', '9765625000000000', '97656250000000000', '100000000000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '781250000000000000', '195312500000000000', '390625000000000000', '48828125000000000', '4882812500000000', '100000000000000000000', '39062500000000000', '97656250000000000', '50000000000000000000', '195312500000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '12207031250000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '12207031250000000', '250000000000000000', '6250000000000000000', '781250000000000000', '610351562500000', '250000000000000000', '6250000000000000000', '781250000000000000', '610351562500000', '100000000000000000000', '100000000000000000000', '12500000000000000000', '12207031250000000', '100000000000000000000', '100000000000000000000', '12500000000000000000', '12207031250000000', '12207031250000000', '781250000000000000', '25000000000000000000', '200000000000000000', '12207031250000000', '781250000000000000', '25000000000000000000', '200000000000000000', '1525878906250000', '781250000000000000', '3125000000000000000', '625000000000000000', '1525878906250000', '781250000000000000', '3125000000000000000', '625000000000000000', '1220703125000000', '6250000000000000000', '12500000000000000000', '2500000000000000000', '1220703125000000', '6250000000000000000', '12500000000000000000', '2500000000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '6250000000000000000', '50000000000000000000', '25000000000000000000', '6103515625000000', '3051757812500000', '190734863281250', '76293945312500', '1525878906250000', '1220703125000000', '6103515625000000', '381469726562500', '762939453125000', '6103515625000000', '762939453125000', '76293945312500', '1525878906250000', '1525878906250000', '1220703125000000', '610351562500000', '1525878906250000', '1525878906250000', '1525878906250000', '762939453125000', '762939453125000', '3051757812500000', '3051757812500000', '610351562500000', '610351562500000', '1525878906250000', '95367431640625', '95367431640625', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '30517578125000', '3051757812500000', '610351562500000', '762939453125000', '3051757812500000', '6103515625000000', '6103515625000000', '305175781250000', '1525878906250000', '381469726562500', '6103515625000000', '6103515625000000', '3051757812500000', '97656250000000000', '78125000000000000', '78125000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '4', '4', '4', '4', '78125000000000000', '48828125000000000', '78125000000000000', '19531250000000000', '48828125000000000', '78125000000000000', '48828125000000000', '48828125000000000', '48828125000000000', '78125000000000000', '39062500000000000', '9765625000000000', '97656250000000000', '24414062500000000', '9765625000000000', '39062500000000000', '39062500000000000', '9765625000000000', '97656250000000000', '24414062500000000', '9765625000000000', '39062500000000000', '78125000000000000', '48828125000000000', '195312500000000000', '2441406250000000', '48828125000000000', '78125000000000000', '78125000000000000', '48828125000000000', '195312500000000000', '2441406250000000', '48828125000000000', '78125000000000000', '781250000000000000', '48828125000000000', '12207031250000000', '390625000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '12207031250000000', '390625000000000000', '48828125000000000', '781250000000000000', '390625000000000000', '6103515625000000', '7812500000000000', '6103515625000000', '9765625000000000', '390625000000000000', '390625000000000000', '6103515625000000', '7812500000000000', '6103515625000000', '9765625000000000', '390625000000000000', '781250000000000000', '6103515625000000', '78125000000000000', '762939453125000', '76293945312500', '781250000000000000', '781250000000000000', '6103515625000000', '78125000000000000', '762939453125000', '76293945312500', '781250000000000000', '781250000000000000', '48828125000000000', '762939453125000', '390625000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '762939453125000', '390625000000000000', '48828125000000000', '781250000000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '3125000000000000000', '25000000000000000000', '312500000000000000', '12207031250000000', '24414062500000000', '12207031250000000', '12207031250000000', '1220703125000000', '3051757812500000', '6103515625000000', '3051757812500000', '610351562500000', '6103515625000000', '12207031250000000', '1953125000000000', '12207031250000000', '12207031250000000', '12207031250000000', '3051757812500000', '610351562500000', '6103515625000000', '61035156250000', '6103515625000000', '12207031250000000', '97656250000000000', '762939453125000', '12207031250000000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '9765625000000000', '12207031250000000', '3051757812500000', '12207031250000000', '381469726562500', '1220703125000000', '1220703125000000', '1525878906250000', '6103515625000000', '6103515625000000', '97656250000000000', '3051757812500000', '12207031250000000', '2441406250000000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '12207031250000000', '97656250000000000', '12207031250000000', '1525878906250000', '1525878906250000', '610351562500000', '3051757812500000', '3051757812500000', '1525878906250000', '762939453125000', '97656250000000000', '97656250000000000', '39062500000000000', '48828125000000000', '97656250000000000', '195312500000000000', '390625000000000000', '97656250000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '48828125000000000', '24414062500000000', '195312500000000000', '390625000000000000', '312500000000000000', '19531250000000000', '78125000000000000', '2500000000000000000', '195312500000000000', '48828125000000000', '19531250000000000', '6250000000000000000', '19531250000000000', '48828125000000000', '195312500000000000', '2500000000000000000', '2500000000000000000', '48828125000000000', '195312500000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '39062500000000000', '97656250000000000', '2500000000000000000', '3125000000000000000', '97656250000000000', '24414062500000000', '195312500000000000', '25000000000000000000', '195312500000000000', '24414062500000000', '97656250000000000', '25000000000000000000', '3125000000000000000', '48828125000000000', '97656250000000000', '195312500000000000', '25000000000000000000', '39062500000000000', '19531250000000000', '1220703125000000', '25000000000000000000', '2500000000000000000', '7812500000000000', '24414062500000000', '39062500000000000', '6250000000000000000', '48828125000000000', '24414062500000000', '7812500000000000', '2500000000000000000', '2500000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '6250000000000000000', '390625000000000000', '78125000000000000', '390625000000000000', '2500000000000000000', '25000000000000000000', '195312500000000000', '12207031250000000', '48828125000000000', '2500000000000000000', '48828125000000000', '48828125000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '2500000000000000000', '234375000000000000', '1171875000000000000', '46875000000000000', '25000000000000000000', '12500000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '6250000000000000000', '195312500000000000', '12207031250000000', '195312500000000000', '12500000000000000000', '12500000000000000000', '46875000000000000', '1171875000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '6250000000000000000', '195312500000000000', '24414062500000000', '97656250000000000', '3125000000000000000', '48828125000000000', '24414062500000000', '195312500000000000', '6250000000000000000', '6250000000000000000', '585937500000000000', '146484375000000000', '58593750000000000', '3125000000000000000', '73242187500000000', '146484375000000000', '23437500000000000', '6250000000000000000', '100000000000000000000', '12500000000000000000', '50000000000000000000', '1250000000000000000', '48828125000000000', '100000000000000000000', '381469726562500', '3051757812500000', '762939453125000', '1525878906250000', '24414062500000000', '6103515625000000', '762939453125000', '610351562500000', '61035156250000', '1525878906250000', '381469726562500', '610351562500000', '19073486328125', '48828125000000000', '6103515625000000', '6103515625000000', '24414062500000000', '1220703125000000', '762939453125000', '1220703125000000', '95367431640625', '6103515625000000', '762939453125000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '4882812500000000', '1525878906250000', '305175781250000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '152587890625000', '24414062500000000', '3051757812500000', '1525878906250000', '3051757812500000', '76293945312500', '6103515625000000', '1525878906250000', '762939453125000', '1525878906250000', '24414062500000000', '610351562500000', '6103515625000000', '24414062500000000', '6103515625000000', '1525878906250000', '48828125000000', '1525878906250000', '6103515625000000', '762939453125000', '190734863281250', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '24414062500000000', '6103515625000000', '190734863281250', '122070312500000', '3051757812500000', '6103515625000000', '76293945312500', '305175781250000', '195312500000000000', '78125000000000000', '97656250000000000', '48828125000000000', '390625000000000000', '1562500000000000000', '7812500000000000', '195312500000000000', '97656250000000000', '19531250000000000', '100000000000000000000', '97656250000000000', '195312500000000000', '12207031250000000', '97656250000000000', '24414062500000000', '390625000000000000', '781250000000000000', '195312500000000000', '2500000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '19531250000000000', '2500000000000000000', '100000000000000000000', '4882812500000000', '97656250000000000', '50000000000000000000', '390625000000000000', '50000000000000000000', '100000000000000000000', '195312500000000000', '6103515625000000', '20000000000000000000', '6103515625000000', '195312500000000000', '24414062500000000', '781250000000000000', '3125000000000000000', '62500000000000000', '4882812500000000', '3125000000000000000', '781250000000000000', '24414062500000000', '195312500000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '156250000000000000', '4882812500000000', '390625000000000000', '3125000000000000000', '1562500000000000000', '244140625000000', '3125000000000000000', '390625000000000000', '48828125000000000', '390625000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '97656250000000000', '62500000000000000', '976562500000000', '781250000000000000', '62500000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '250000000000000000', '6250000000000000000', '125000000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '3125000000000000000', '250000000000000000', '1250000000000000000', '390625000000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '1562500000000000000', '6103515625000000', '3125000000000000000', '195312500000000000', '48828125000000000', '390625000000000000', '97656250000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '390625000000000000', '195312500000000000', '390625000000000000', '3125000000000000000', '625000000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '3125000000000000000', '1525878906250000', '39062500000000000', '3125000000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '781250000000000000', '9765625000000000', '781250000000000000', '3125000000000000000', '625000000000000000', '4882812500000000', '1562500000000000000', '781250000000000000', '48828125000000000', '781250000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '50000000000000000000', '25000000000000000000', '6250000000000000000', '781250000000000000', '50000000000000000000', '12207031250000000', '3051757812500000', '48828125000000000', '3051757812500000', '305175781250000', '1220703125000000', '6103515625000000', '190734863281250', '381469726562500', '3051757812500000', '6103515625000000', '381469726562500', '3051757812500000', '6103515625000000', '1953125000000000', '6103515625000000', '3051757812500000', '1220703125000000', '610351562500000', '95367431640625', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '3051757812500000', '6103515625000000', '24414062500000000', '1220703125000000', '76293945312500', '610351562500000', '6103515625000000', '1525878906250000', '305175781250000', '6103515625000000', '1525878906250000', '1525878906250000', '1525878906250000', '3051757812500000', '48828125000000000', '3051757812500000', '610351562500000', '6103515625000000', '1525878906250000', '610351562500000', '762939453125000', '381469726562500', '6103515625000000', '1525878906250000', '610351562500000', '3051757812500000', '12207031250000000', '762939453125000', '3051757812500000', '762939453125000', '610351562500000', '1525878906250000', '610351562500000', '610351562500000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '48828125000000000', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '195312500000000000', '97656250000000000', '19531250000000000', '976562500000000', '100000000000000000', '195312500000000000', '97656250000000000', '24414062500000000', '390625000000000000', '195312500000000000', '195312500000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '97656250000000000', '97656250000000000', '195312500000000000', '156250000000000000', '390625000000000000', '2441406250000000', '97656250000000000', '97656250000000000', '48828125000000000', '97656250000000000', '20000000000000000000', '195312500000000000', '48828125000000000', '390625000000000000', '24414062500000000', '39062500000000000', '390625000000000000', '48828125000000000', '97656250000000000', '100000000000000000000', '97656250000000000', '100000000000000000000', '97656250000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '585937500000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '25000000000000000000', '585937500000000000', '585937500000000000', '48828125000000000', '97656250000000000', '100000000000000000000', '9765625000000000', '100000000000000000000', '97656250000000000', '25000000000000000000', '97656250000000000', '9765625000000000', '1171875000000000000', '234375000000000000', '100000000000000000000', '1171875000000000000', '100000000000000000000', '1171875000000000000', '25000000000000000000', '1171875000000000000', '1171875000000000000', '48828125000000000', '97656250000000000', '50000000000000000000', '97656250000000000', '4000000000000000000', '97656250000000000', '800000000000000000', '48828125000000000', '4882812500000000', '292968750000000000', '7324218750000000', '50000000000000000000', '29296875000000000', '4000000000000000000', '9155273437500000', '800000000000000000', '58593750000000000', '292968750000000000', '9765625000000000', '97656250000000000', '5000000000000000000', '9765625000000000', '20000000000000000000', '97656250000000000', '100000000000000000000', '195312500000000000', '9765625000000000', '1171875000000000000', '1171875000000000000', '5000000000000000000', '1171875000000000000', '20000000000000000000', '234375000000000000', '100000000000000000000', '1171875000000000000', '1171875000000000000', '4882812500000000', '48828125000000000', '3125000000000000000', '195312500000000000', '100000000000000000000', '195312500000000000', '100000000000000000000', '48828125000000000', '976562500000000', '292968750000000000', '58593750000000000', '3125000000000000000', '585937500000000000', '100000000000000000000', '23437500000000000', '100000000000000000000', '292968750000000000', '292968750000000000', '48828125000000000', '48828125000000000', '12500000000000000000', '195312500000000000', '10000000000000000000', '97656250000000000', '50000000000000000000', '195312500000000000', '48828125000000000', '78125000000000000', '78125000000000000', '12500000000000000000', '390625000000000000', '10000000000000000000', '390625000000000000', '50000000000000000000', '390625000000000000', '390625000000000000', '100000000000000000000', '100000000000000000000', '10000000000000000000', '100000000000000000000', '3051757812500000', '100000000000000000000', '95367431640625', '24414062500000000', '1220703125000000', '61035156250000', '1525878906250000', '381469726562500', '610351562500000', '190734863281250', '1525878906250000', '12207031250000000', '1220703125000000', '610351562500000', '6103515625000000', '6103515625000000', '95367431640625', '1525878906250000', '1525878906250000', '381469726562500', '6103515625000000', '1525878906250000', '3051757812500000', '12207031250000000', '6103515625000000', '3051757812500000', '6103515625000000', '3051757812500000', '610351562500000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '95367431640625', '6103515625000000', '12207031250000000', '6103515625000000', '762939453125000', '6103515625000000', '24414062500000000', '3051757812500000', '152587890625000', '6103515625000000', '381469726562500', '610351562500000', '762939453125000', '381469726562500', '4882812500000000', '6103515625000000', '3051757812500000', '1525878906250000', '24414062500000000', '762939453125000', '61035156250000', '1220703125000000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '1525878906250000', '1525878906250000', '6103515625000000', '6103515625000000', '190734863281250', '6103515625000000', '95367431640625', '1220703125000000', '1525878906250000', '1220703125000000', '4882812500000000', '1525878906250000', '1525878906250000', '195312500000000000', '195312500000000000', '15625000000000000', '19531250000000000', '50000000000000000000', '78125000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '195312500000000000', '78125000000000000', '3125000000000000000', '190734863281250', '24414062500000000', '2500000000000000000', '1562500000000000000', '50000000000000000000', '390625000000000000', '97656250000000000', '39062500000000000', '12500000000000000000', '48828125000000000', '3125000000000000000', '390625000000000000', '244140625000000', '24414062500000000', '390625000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '48828125000000000', '100000000000000000000', '50000000000000000000', '4882812500000000', '100000000000000000000', '976562500000000', '48828125000000000', '100000000000000000000', '50000000000000000000', '4882812500000000', '100000000000000000000', '976562500000000', '9765625000000000', '1250000000000000000', '20000000000000000000', '1525878906250000', '100000000000000000000', '488281250000000', '9765625000000000', '1250000000000000000', '20000000000000000000', '1525878906250000', '100000000000000000000', '488281250000000', '976562500000000', '6250000000000000000', '312500000000000000', '24414062500000000', '6250000000000000000', '4882812500000000', '976562500000000', '6250000000000000000', '312500000000000000', '24414062500000000', '6250000000000000000', '4882812500000000', '48828125000000000', '10000000000000000000', '5000000000000000000', '24414062500000000', '50000000000000000000', '48828125000000000', '48828125000000000', '10000000000000000000', '5000000000000000000', '24414062500000000', '50000000000000000000', '48828125000000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '50000000000000000000', '48828125000000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '12207031250000000', '50000000000000000000', '48828125000000000', '24414062500000000', '25000000000000000000', '2500000000000000000', '1220703125000000', '1000000000000000000', '24414062500000000', '24414062500000000', '25000000000000000000', '2500000000000000000', '1220703125000000', '1000000000000000000', '24414062500000000', '25000000000000000000', '1562500000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '3051757812500000', '12207031250000000', '3051757812500000', '1525878906250000', '1525878906250000', '762939453125000', '1525878906250000', '1525878906250000', '152587890625000', '38146972656250', '1525878906250000', '12207031250000000', '1220703125000000', '1525878906250000', '381469726562500', '190734863281250', '190734863281250', '6103515625000000', '1525878906250000', '610351562500000', '95367431640625', '24414062500000000', '1220703125000000', '1525878906250000', '1525878906250000', '122070312500000', '76293945312500', '6103515625000000', '610351562500000', '762939453125000', '3051757812500000', '24414062500000000', '244140625000000', '381469726562500', '1525878906250000', '381469726562500', '1525878906250000', '1525878906250000', '1220703125000000', '3051757812500000', '3051757812500000', '3051757812500000', '1220703125000000', '762939453125000', '95367431640625', '1525878906250000', '152587890625000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '24414062500000000', '762939453125000', '305175781250000', '1525878906250000', '3051757812500000', '152587890625000', '6103515625000000', '3051757812500000', '305175781250000', '97656250000000000', '19531250000000000', '12500000000000000000', '48828125000000000', '97656250000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '15625000000000000', '7812500000000000', '1562500000000000000', '390625000000000000', '24414062500000000', '195312500000000000', '100000000000000000000', '97656250000000000', '195312500000000000', '48828125000000000', '39062500000000000', '12207031250000000', '39062500000000000', '390625000000000', '390625000000000', '781250000000000000', '390625000000000', '390625000000000', '78125000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '31250000000000000', '195312500000000000', '31250000000000000', '31250000000000000', '31250000000000000', '31250000000000000', '6103515625000000', '31250000000000000', '6103515625000000', '6103515625000000', '781250000000000000', '31250000000000000', '31250000000000000', '781250000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '6103515625000000', '24414062500000000', '390625000000000000', '12500000000000000000', '12500000000000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '6103515625000000', '24414062500000000', '390625000000000000', '12500000000000000000', '5000000000000000000', '97656250000000000', '97656250000000000', '12500000000000000000', '381469726562500', '97656250000000000', '97656250000000000', '5000000000000000000', '5000000000000000000', '97656250000000000', '97656250000000000', '12500000000000000000', '381469726562500', '97656250000000000', '97656250000000000', '5000000000000000000', '12500000000000000000', '39062500000000000', '97656250000000000', '6250000000000000000', '381469726562500', '97656250000000000', '39062500000000000', '12500000000000000000', '12500000000000000000', '39062500000000000', '97656250000000000', '6250000000000000000', '381469726562500', '97656250000000000', '39062500000000000', '12500000000000000000', '3125000000000000000', '195312500000000000', '97656250000000000', '12500000000000000000', '2441406250000000', '97656250000000000', '195312500000000000', '12500000000000000000', '3125000000000000000', '195312500000000000', '97656250000000000', '12500000000000000000', '2441406250000000', '97656250000000000', '195312500000000000', '12500000000000000000', '6250000000000000000', '390625000000000000', '24414062500000000', '1562500000000000000', '305175781250000', '48828125000000000', '390625000000000000', '6250000000000000000', '6250000000000000000', '390625000000000000', '24414062500000000', '1562500000000000000', '305175781250000', '48828125000000000', '390625000000000000', '6250000000000000000', '12500000000000000000', '78125000000000000', '24414062500000000', '6250000000000000000', '3051757812500000', '24414062500000000', '78125000000000000', '12500000000000000000', '12500000000000000000', '78125000000000000', '24414062500000000', '6250000000000000000', '3051757812500000', '24414062500000000', '78125000000000000', '12500000000000000000', '50000000000000000000', '4000000000000000000', '50000000000000000000', '20000000000000000000', '100000000000000000000', '100000000000000000000', '762939453125000', '1525878906250000', '12207031250000000', '3051757812500000', '19073486328125', '762939453125000', '3051757812500000', '12207031250000000', '762939453125000', '6103515625000000', '152587890625000', '1525878906250000', '3051757812500000', '24414062500000000', '4882812500000000', '381469726562500', '305175781250000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '762939453125000', '1525878906250000', '3051757812500000', '305175781250000', '6103515625000000', '244140625000000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '95367431640625', '12207031250000000', '1525878906250000', '95367431640625', '1525878906250000', '24414062500000000', '24414062500000000', '3051757812500000', '1525878906250000', '3051757812500000', '1525878906250000', '12207031250000000', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '3051757812500000', '12207031250000000', '2441406250000000', '3051757812500000', '1220703125000000', '3051757812500000', '3051757812500000', '12207031250000000', '610351562500000', '12207031250000000', '3051757812500000', '1525878906250000', '3051757812500000', '24414062500000000', '24414062500000000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '12207031250000000', '1525878906250000', '12207031250000000', '3051757812500000', '3051757812500000', '24414062500000000', '1250000000000000000', '48828125000000000', '39062500000000000', '3125000000000000000', '97656250000000000', '19531250000000000', '6250000000000000000', '9765625000000000', '20000000000000000000', '390625000000000000', '39062500000000000', '250000000000000000', '390625000000000000', '195312500000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '781250000000000000', '48828125000000000', '312500000000000000', '390625000000000', '390625000000000', '12207031250000000', '48828125000000000', '781250000000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '156250000000000000', '390625000000000000', '39062500000000000', '195312500000000000', '12207031250000000', '12207031250000000', '6103515625000000', '390625000000000000', '390625000000000000', '156250000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '3125000000000000000', '781250000000000000', '12207031250000000', '1562500000000000000', '48828125000000000', '48828125000000000', '6103515625000000', '12207031250000000', '390625000000000000', '3125000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '2500000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '97656250000000000', '781250000000000000', '195312500000000000', '4882812500000000', '4882812500000000', '12207031250000000', '781250000000000000', '390625000000000000', '390625000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '1562500000000000000', '1562500000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '3125000000000000000', '390625000000000000', '781250000000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '781250000000000000', '390625000000000000', '3125000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '200000000000000000', '3125000000000000000', '3125000000000000', '390625000000000000', '3125000000000000000', '24414062500000000', '9765625000000000', '24414062500000000', '781250000000000000', '390625000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '12500000000000000000', '5000000000000000000', '1000000000000000000', '1000000000000000000', '25000000000000000000', '6250000000000000000', '12207031250000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '97656250000000000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '1525878906250000', '762939453125000', '3051757812500000', '1525878906250000', '3051757812500000', '381469726562500', '3051757812500000', '6103515625000000', '6103515625000000', '190734863281250', '6103515625000000', '24414062500000000', '6103515625000000', '3051757812500000', '6103515625000000', '1220703125000000', '381469726562500', '762939453125000', '3051757812500000', '381469726562500', '76293945312500', '1525878906250000', '610351562500000', '48828125000000000', '6103515625000000', '610351562500000', '6103515625000000', '244140625000000', '3051757812500000', '305175781250000', '1525878906250000', '6103515625000000', '6103515625000000', '152587890625000', '6103515625000000', '24414062500000000', '1220703125000000', '190734863281250', '1220703125000000', '1525878906250000', '610351562500000', '610351562500000', '381469726562500', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '12207031250000000', '6103515625000000', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '305175781250000', '381469726562500', '1220703125000000', '1220703125000000', '3051757812500000', '1220703125000000', '24414062500000000', '762939453125000', '95367431640625', '3051757812500000', '381469726562500', '762939453125000', '1525878906250000', '97656250000000000', '97656250000000000', '50000000000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '195312500000000000', '24414062500000000', '781250000000000000', '39062500000000000', '3051757812500000', '19531250000000000', '4882812500000000', '195312500000000000', '10000000000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '1220703125000000', '97656250000000000', '195312500000000000', '100000000000000000000', '78125000000000000', '9765625000000000', '97656250000000000', '195312500000000000', '390625000000000000', '781250000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '19531250000000000', '24414062500000000', '25000000000000000000', '97656250000000000', '97656250000000000', '781250000000000000', '3125000000000000000', '6103515625000000', '62500000000000000', '390625000000000000', '195312500000000000', '3051757812500000', '6103515625000000', '3125000000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '3125000000000000000', '9765625000000000', '62500000000000000', '195312500000000000', '195312500000000000', '24414062500000000', '9765625000000000', '3125000000000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '625000000000000000', '12207031250000000', '3125000000000000000', '390625000000000000', '97656250000000000', '24414062500000000', '9765625000000000', '1562500000000000000', '97656250000000000', '12500000000000000000', '2500000000000000000', '3125000000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '3125000000000000000', '12500000000000000000', '12500000000000000000', '195312500000000000', '3125000000000000000', '3051757812500000', '1562500000000000000', '97656250000000000', '97656250000000000', '24414062500000000', '12207031250000000', '3125000000000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '200000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '781250000000000000', '312500000000000000', '48828125000000000', '781250000000000000', '97656250000000000', '97656250000000000', '12207031250000000', '24414062500000000', '312500000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '1562500000000000000', '19531250000000000', '19531250000000000', '3051757812500000', '9765625000000000', '3125000000000000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '5000000000000000000', '48828125000000000', '6250000000000000000', '6250000000000000000', '625000000000000000', '100000000000000000000', '4000000000000000000', '61035156250000', '6103515625000000', '6103515625000000', '3051757812500000', '152587890625000', '6103515625000000', '1525878906250000', '762939453125000', '3051757812500000', '48828125000000000', '3051757812500000', '190734863281250', '3051757812500000', '762939453125000', '762939453125000', '610351562500000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '48828125000000000', '762939453125000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '610351562500000', '6103515625000000', '6103515625000000', '610351562500000', '610351562500000', '48828125000000000', '3051757812500000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '1525878906250000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '9765625000000000', '6103515625000000', '152587890625000', '61035156250000', '95367431640625', '3051757812500000', '95367431640625', '1525878906250000', '1220703125000000', '3051757812500000', '190734863281250', '1525878906250000', '48828125000000000', '1525878906250000', '3051757812500000', '381469726562500', '3051757812500000', '1525878906250000', '190734863281250', '610351562500000', '6103515625000000', '762939453125000', '1525878906250000', '1220703125000000', '6103515625000000', '244140625000000', '3051757812500000', '19531250000000000', '24414062500000000', '19531250000000000', '31250000000000000', '390625000000000000', '4882812500000000', '97656250000000000', '390625000000000000', '244140625000000', '1562500000000000000', '95367431640625', '97656250000000000', '97656250000000000', '19531250000000000', '97656250000000000', '195312500000000000', '781250000000000000', '195312500000000000', '12207031250000000', '48828125000000000', '24414062500000000', '390625000000000000', '10000000000000000000', '48828125000000000', '97656250000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '48828125000000000', '1525878906250000', '195312500000000000', '390625000000000000', '4882812500000000', '97656250000000000', '25000000000000000000', '97656250000000000', '48828125000000000', '100000000000000000000', '20000000000000000000', '48828125000000000', '48828125000000000', '12207031250000000', '25000000000000000000', '48828125000000000', '48828125000000000', '100000000000000000000', '20000000000000000000', '48828125000000000', '48828125000000000', '12207031250000000', '25000000000000000000', '48828125000000000', '390625000000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '97656250000000000', '3051757812500000', '25000000000000000000', '48828125000000000', '390625000000000000', '50000000000000000000', '50000000000000000000', '97656250000000000', '97656250000000000', '3051757812500000', '25000000000000000000', '48828125000000000', '78125000000000000', '6250000000000000000', '12207031250000000', '97656250000000000', '48828125000000000', '50000000000000000000', '50000000000000000000', '390625000000000000', '78125000000000000', '6250000000000000000', '12207031250000000', '97656250000000000', '48828125000000000', '50000000000000000000', '50000000000000000000', '390625000000000000', '24414062500000000', '5000000000000000000', '6103515625000000', '1525878906250000', '24414062500000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '5000000000000000000', '6103515625000000', '1525878906250000', '24414062500000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '195312500000000000', '25000000000000000000', '1000000000000000000', '12207031250000000', '6103515625000000', '762939453125000', '6250000000000000000', '195312500000000000', '195312500000000000', '25000000000000000000', '1000000000000000000', '12207031250000000', '6103515625000000', '762939453125000', '6250000000000000000', '195312500000000000', '390625000000000000', '2500000000000000000', '12207031250000000', '48828125000000000', '97656250000000000', '12500000000000000000', '3125000000000000000', '9765625000000000', '390625000000000000', '2500000000000000000', '12207031250000000', '48828125000000000', '97656250000000000', '12500000000000000000', '3125000000000000000', '9765625000000000', '5000000000000000000', '25000000000000000000', '20000000000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '1220703125000000', '4882812500000000', '24414062500000000', '1525878906250000', '1525878906250000', '305175781250000', '3051757812500', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '762939453125000', '6103515625000000', '3051757812500000', '2441406250000000', '305175781250000', '762939453125000', '3051757812500000', '381469726562500', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '4882812500000000', '24414062500000000', '122070312500000', '152587890625000', '3051757812500000', '381469726562500', '1525878906250000', '305175781250000', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '12207031250000000', '24414062500000000', '762939453125000', '762939453125000', '6103515625000000', '3051757812500000', '1525878906250000', '305175781250000', '6103515625000000', '3051757812500000', '610351562500000', '6103515625000000', '2441406250000000', '24414062500000000', '1525878906250000', '762939453125000', '122070312500000', '762939453125000', '6103515625000000', '152587890625000', '381469726562500', '305175781250000', '3051757812500000', '3051757812500000', '12207031250000000', '6103515625000000', '6103515625000000', '1525878906250000', '1220703125000000', '3051757812500000', '6103515625000000', '1525878906250000', '1220703125000000', '6103515625000000', '3051757812500000', '12500000000000000000', '39062500000000000', '48828125000000000', '12500000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '195312500000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '625000000000000000', '6250000000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '24414062500000000', '1562500000000000', '195312500000000000', '195312500000000000', '781250000000000000', '97656250000000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '3051757812500000', '781250000000000000', '3051757812500000', '3051757812500000', '195312500000000000', '781250000000000000', '781250000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '97656250000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '312500000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '97656250000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '2500000000000000000', '390625000000000000', '48828125000000000', '762939453125000', '12500000000000000000', '781250000000000', '195312500000000000', '6250000000000000000', '2500000000000000000', '390625000000000000', '48828125000000000', '762939453125000', '12500000000000000000', '781250000000000', '195312500000000000', '6250000000000000000', '12500000000000000000', '48828125000000000', '19531250000000000', '25000000000000000000', '12207031250000000', '19531250000000000', '195312500000000000', '25000000000000000000', '12500000000000000000', '48828125000000000', '19531250000000000', '25000000000000000000', '12207031250000000', '19531250000000000', '195312500000000000', '25000000000000000000', '2500000000000000000', '97656250000000000', '24414062500000000', '6250000000000000000', '6103515625000000', '9765625000000000', '390625000000000000', '12500000000000000000', '2500000000000000000', '97656250000000000', '24414062500000000', '6250000000000000000', '6103515625000000', '9765625000000000', '390625000000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '4882812500000000', '12500000000000000000', '6103515625000000', '4882812500000000', '390625000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '4882812500000000', '12500000000000000000', '6103515625000000', '4882812500000000', '390625000000000000', '25000000000000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '312500000000000000', '12207031250000000', '97656250000000000', '390625000000000000', '3125000000000000000', '3125000000000000000', '390625000000000000', '97656250000000000', '312500000000000000', '12207031250000000', '97656250000000000', '390625000000000000', '3125000000000000000', '12500000000000000000', '39062500000000000', '9765625000000000', '25000000000000000000', '95367431640625', '9765625000000000', '39062500000000000', '25000000000000000000', '12500000000000000000', '39062500000000000', '9765625000000000', '25000000000000000000', '95367431640625', '9765625000000000', '39062500000000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '5000000000000000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '190734863281250', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '610351562500000', '6103515625000000', '610351562500000', '1525878906250000', '762939453125000', '3051757812500000', '6103515625000000', '762939453125000', '762939453125000', '3051757812500000', '3051757812500000', '3051757812500000', '4882812500000000', '3051757812500000', '3051757812500000', '95367431640625', '6103515625000000', '1525878906250000', '12207031250000000', '762939453125000', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '12207031250000000', '3051757812500000', '1525878906250000', '76293945312500', '3051757812500000', '1525878906250000', '12207031250000000', '1525878906250000', '6103515625000000', '3051757812500000', '6103515625000000', '24414062500000000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '381469726562500', '3051757812500000', '76293945312500', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '12207031250000000', '1525878906250000', '3051757812500000', '95367431640625', '3051757812500000', '6103515625000000', '610351562500000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '6250000000000000000', '97656250000000000', '48828125000000000', '25000000000000000000', '195312500000000000', '390625000000000000', '97656250000000000', '195312500000000000', '97656250000000000', '390625000000000000', '48828125000000000', '12500000000000000000', '48828125000000000', '97656250000000000', '6250000000000000000', '15625000000000000', '97656250000000000', '3125000000000000000', '2500000000000000000', '19531250000000000', '48828125000000000', '24414062500000000', '6250000000000000000', '24414062500000000', '48828125000000000', '19531250000000000', '2500000000000000000', '2500000000000000000', '39062500000000000', '195312500000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '195312500000000000', '39062500000000000', '2500000000000000000', '100000000000000000', '195312500000000000', '19531250000000000', '24414062500000000', '1250000000000000000', '48828125000000000', '19531250000000000', '195312500000000000', '100000000000000000', '100000000000000000', '48828125000000000', '39062500000000000', '195312500000000000', '1250000000000000000', '195312500000000000', '39062500000000000', '97656250000000000', '100000000000000000', '5000000000000000000', '48828125000000000', '781250000000000', '48828125000000000', '25000000000000000000', '48828125000000000', '195312500000000000', '48828125000000000', '12500000000000000000', '5000000000000000000', '585937500000000000', '585937500000000000', '585937500000000000', '25000000000000000000', '585937500000000000', '29296875000000000', '585937500000000000', '12500000000000000000', '12500000000000000000', '1525878906250000', '195312500000000000', '9765625000000000', '6250000000000000000', '48828125000000000', '195312500000000000', '1525878906250000', '12500000000000000000', '12500000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '25000000000000000000', '7812500000000000', '195312500000000000', '9765625000000000', '12500000000000000000', '9765625000000000', '195312500000000000', '7812500000000000', '25000000000000000000', '25000000000000000000', '117187500000000000', '585937500000000000', '14648437500000000', '12500000000000000000', '7324218750000000', '585937500000000000', '117187500000000000', '25000000000000000000', '5000000000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '12500000000000000000', '48828125000000000', '48828125000000000', '97656250000000000', '5000000000000000000', '5000000000000000000', '292968750000000000', '36621093750000000', '18310546875000000', '12500000000000000000', '73242187500000000', '2929687500000000', '292968750000000000', '5000000000000000000', '24414062500000000', '781250000000000000', '625000000000000000', '24414062500000000', '6250000000000000000', '6250000000000000000', '305175781250000', '6103515625000000', '3051757812500000', '1220703125000000', '1525878906250000', '1220703125000000', '24414062500000000', '6103515625000000', '6103515625000000', '152587890625000', '122070312500000', '762939453125000', '762939453125000', '610351562500000', '610351562500000', '6103515625000000', '381469726562500', '1220703125000000', '244140625000000', '1220703125000000', '95367431640625', '1525878906250000', '6103515625000000', '610351562500000', '381469726562500', '190734863281250', '3051757812500000', '6103515625000', '61035156250000', '1525878906250000', '12207031250000000', '3051757812500000', '6103515625000000', '12207031250000000', '1525878906250000', '3051757812500000', '381469726562500', '305175781250000', '1525878906250000', '6103515625000000', '762939453125000', '3051757812500000', '4882812500000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '381469726562500', '95367431640625', '61035156250000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '4882812500000000', '6103515625000000', '95367431640625', '610351562500000', '6103515625000000', '3051757812500000', '381469726562500', '1220703125000000', '3051757812500000', '6103515625000000', '38146972656250', '6103515625000000', '24414062500000000', '6103515625000000', '190734863281250', '24414062500000000', '3051757812500000', '3051757812500000', '20000000000000000000', '25000000000000000000', '25000000000000000000', '195312500000000000', '12207031250000000', '390625000000000000', '312500000000000000', '50000000000000000000', '100000000000000000000', '2441406250000000', '19531250000000000', '195312500000000000', '24414062500000000', '25000000000000000000', '15625000000000000', '312500000000000', '610351562500000', '390625000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '9765625000000000', '9765625000000000', '1562500000000000000', '97656250000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '195312500000000000', '390625000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '24414062500000000', '19531250000000000', '195312500000000000', '12500000000000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '12207031250000000', '78125000000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '12207031250000000', '78125000000000000', '97656250000000000', '12500000000000000000', '12500000000000000000', '6103515625000000', '97656250000000000', '6103515625000000', '25000000000000000000', '390625000000000000', '97656250000000000', '1000000000000000000', '12500000000000000000', '6103515625000000', '97656250000000000', '6103515625000000', '25000000000000000000', '390625000000000000', '97656250000000000', '1000000000000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '2500000000000000000', '6103515625000000', '12207031250000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '97656250000000000', '195312500000000000', '2500000000000000000', '6103515625000000', '12207031250000000', '97656250000000000', '25000000000000000000', '1250000000000000000', '97656250000000000', '6103515625000000', '25000000000000000000', '12207031250000000', '6103515625000000', '97656250000000000', '25000000000000000000', '1250000000000000000', '97656250000000000', '6103515625000000', '25000000000000000000', '12207031250000000', '6103515625000000', '97656250000000000', '25000000000000000000', '12500000000000000000', '3051757812500000', '78125000000000000', '762939453125000', '12500000000000000000', '15625000000000000', '97656250000000000', '3125000000000000000', '12500000000000000000', '3051757812500000', '78125000000000000', '762939453125000', '12500000000000000000', '15625000000000000', '97656250000000000', '3125000000000000000', '25000000000000000000', '24414062500000000', '390625000000000000', '25000000000000000000', '1220703125000000', '390625000000000000', '1220703125000000', '3125000000000000000', '25000000000000000000', '24414062500000000', '390625000000000000', '25000000000000000000', '1220703125000000', '390625000000000000', '1220703125000000', '3125000000000000000', '25000000000000000000', '100000000000000000000', '5000000000000000000', '50000000000000000000', '3125000000000000000', '20000000000000000000', '1525878906250000', '2441406250000000', '152587890625000', '6103515625000000', '381469726562500', '6103515625000000', '24414062500000000', '24414062500000000', '3051757812500000', '6103515625000000', '610351562500000', '19073486328125', '1525878906250000', '6103515625000000', '3051757812500000', '12207031250000000', '1525878906250000', '1220703125000000', '762939453125000', '24414062500000000', '610351562500000', '6103515625000000', '305175781250000', '610351562500000', '1525878906250000', '6103515625000000', '3051757812500000', '305175781250000', '762939453125000', '3051757812500000', '24414062500000000', '12207031250000000', '6103515625000000', '6103515625000000', '244140625000000', '610351562500000', '3051757812500000', '1220703125000000', '3051757812500000', '6103515625000000', '1525878906250000', '762939453125000', '12207031250000000', '12207031250000000', '6103515625000000', '3051757812500000', '1525878906250000', '95367431640625', '3051757812500000', '12207031250000000', '1525878906250000', '6103515625000000', '381469726562500', '3051757812500000', '24414062500000000', '610351562500000', '305175781250000', '1525878906250000', '610351562500000', '3051757812500000', '3051757812500000', '610351562500000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '24414062500000000', '24414062500000000', '1220703125000000', '6103515625000000', '3051757812500000', '3051757812500000', '78125000000000000', '390625000000000000', '195312500000000000', '390625000000000000', '1953125000000000', '97656250000000000', '390625000000000000', '6250000000000000000', '4882812500000000', '390625000000000000', '1250000000000000000', '48828125000000000', '125000000000000000', '390625000000000000', '195312500000000000', '195312500000000000', '6250000000000000000', '48828125000000000', '3125000000000000000', '48828125000000000', '48828125000000000', '1562500000000000000', '195312500000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '1562500000000000000', '195312500000000000', '1562500000000000000', '48828125000000000', '48828125000000000', '1562500000000000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '19531250000000000', '3051757812500000', '781250000000000000', '24414062500000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '781250000000000000', '390625000000000000', '390625000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '12207031250000000', '48828125000000000', '97656250000000000', '97656250000000000', '48828125000000000', '48828125000000000', '12207031250000000', '48828125000000000', '97656250000000000', '1562500000000000000', '24414062500000000', '781250000000000000', '1525878906250000', '48828125000000000', '1562500000000000000', '1562500000000000000', '24414062500000000', '781250000000000000', '1525878906250000', '48828125000000000', '1562500000000000000', '1562500000000000000', '3051757812500000', '781250000000000000', '4882812500000000', '3051757812500000', '1562500000000000000', '1562500000000000000', '3051757812500000', '781250000000000000', '4882812500000000', '3051757812500000', '1562500000000000000', '1562500000000000000', '48828125000000000', '781250000000000000', '2441406250000000', '48828125000000000', '1562500000000000000', '1562500000000000000', '48828125000000000', '781250000000000000', '2441406250000000', '48828125000000000', '1562500000000000000', '1562500000000000000', '24414062500000000', '781250000000000000', '12207031250000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '24414062500000000', '781250000000000000', '12207031250000000', '24414062500000000', '1562500000000000000', '1562500000000000000', '48828125000000000', '781250000000000000', '6103515625000000', '48828125000000000', '1562500000000000000', '1562500000000000000', '48828125000000000', '781250000000000000', '6103515625000000', '48828125000000000', '1562500000000000000', '25000000000000000000', '10000000000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '2500000000000000000', '762939453125000', '762939453125000', '48828125000000', '610351562500000', '12207031250000000', '12207031250000000', '190734863281250', '4882812500000000', '12207031250000000', '1220703125000000', '24414062500000', '6103515625000000', '1525878906250000', '24414062500000', '12207031250000000', '12207031250000000', '12207031250000000', '19531250000000000', '12207031250000000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '61035156250000', '3051757812500000', '12207031250000000', '6103515625000000', '390625000000000000', '1220703125000000', '6103515625000000', '381469726562500', '1220703125000000', '1220703125000000', '1525878906250000', '12207031250000000', '12207031250000000', '2441406250000000', '78125000000000000', '12207031250000000', '95367431640625', '305175781250000', '762939453125000', '762939453125000', '3051757812500000', '12207031250000000', '3051757812500000', '6103515625000000', '390625000000000000', '2441406250000000', '1525878906250000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '762939453125000', '3051757812500000', '3051757812500000', '195312500000000000', '3051757812500000', '6103515625000000', '195312500000000000', '19531250000000000', '195312500000000000', '781250000000000000', '39062500000000000', '390625000000000000', '1220703125000000', '195312500000000000', '39062500000000000', '390625000000000000', '48828125000000000', '390625000000000000', '97656250000000000', '48828125000000000', '781250000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '390625000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '781250000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '781250000000000000', '781250000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '6103515625000000', '195312500000000000', '390625000000000000', '390625000000000000', '12207031250000000', '48828125000000000', '6103515625000000', '195312500000000000', '390625000000000000', '1562500000000000000', '156250000000000000', '12207031250000000', '12207031250000000', '12207031250000000', '1562500000000000000', '1562500000000000000', '156250000000000000', '12207031250000000', '12207031250000000', '12207031250000000', '1562500000000000000', '1562500000000000000', '781250000000000000', '6103515625000000', '48828125000000000', '3051757812500000', '1562500000000000000', '1562500000000000000', '781250000000000000', '6103515625000000', '48828125000000000', '3051757812500000', '1562500000000000000', '97656250000000000', '48828125000000000', '24414062500000000', '24414062500000000', '4882812500000000', '97656250000000000', '97656250000000000', '48828125000000000', '24414062500000000', '24414062500000000', '4882812500000000', '97656250000000000', '1562500000000000000', '3051757812500000', '48828125000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '3051757812500000', '48828125000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '100000000000000000000', '50000000000000000000', '1000000000000000000', '1250000000000000000', '1562500000000000000', '100000000000000000000', '3051757812500000', '122070312500000', '6103515625000000', '1525878906250000', '610351562500000', '195312500000000000', '12207031250000000', '762939453125000', '12207031250000000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '12207031250000000', '78125000000000000', '2441406250000000', '12207031250000000', '12207031250000000', '3051757812500000', '3051757812500000', '122070312500000', '3051757812500000', '95367431640625', '12207031250000000', '78125000000000000', '610351562500000', '6103515625000000', '1220703125000000', '190734863281250', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '12207031250000000', '195312500000000000', '3051757812500000', '12207031250000000', '12207031250000000', '30517578125000', '1525878906250000', '6103515625000000', '6103515625000000', '610351562500000', '6103515625000000', '195312500000000000', '6103515625000000', '12207031250000000', '12207031250000000', '1220703125000000', '1525878906250000', '6103515625000000', '1525878906250000', '1525878906250000', '2441406250000000', '3051757812500000', '12207031250000000', '381469726562500', '381469726562500', '95367431640625', '781250000000000000', '390625000000000000', '195312500000000000', '31250000000000000', '48828125000000000', '39062500000000000', '97656250000000000', '6250000000000000000', '19531250000000000', '48828125000000000', '20000000000000000000', '48828125000000000', '1562500000000000000', '390625000000000000', '97656250000000000', '390625000000000000', '781250000000000000', '19531250000000000', '19531250000000000', '390625000000000000', '390625000000000000', '19531250000000000', '48828125000000000', '24414062500000000', '390625000000000000', '24414062500000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '24414062500000000', '390625000000000000', '19531250000000000', '19531250000000000', '156250000000000000', '195312500000000000', '12207031250000000', '12207031250000000', '156250000000000000', '12207031250000000', '12207031250000000', '156250000000000000', '3051757812500000', '156250000000000000', '24414062500000000', '6103515625000000', '6103515625000000', '24414062500000000', '24414062500000000', '6103515625000000', '6103515625000000', '24414062500000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '4882812500000000', '3051757812500000', '12207031250000000', '4882812500000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '48828125000000000', '1525878906250000', '12207031250000000', '9765625000000000', '48828125000000000', '1525878906250000', '12207031250000000', '9765625000000000', '48828125000000000', '1220703125000000', '24414062500000000', '48828125000000000', '48828125000000000', '1220703125000000', '24414062500000000', '48828125000000000', '48828125000000000', '24414062500000000', '95367431640625', '48828125000000000', '48828125000000000', '24414062500000000', '95367431640625', '48828125000000000', '12500000000000000000', '625000000000000000', '195312500000000000', '6250000000000000000', '24414062500000000', '2500000000000000000', '6103515625000000', '6103515625000000', '12207031250000000', '24414062500000000', '24414062500000000', '24414062500000000', '12207031250000000', '95367431640625', '6103515625000000', '1525878906250000', '1220703125000000', '24414062500000000', '2441406250000000', '6103515625000000', '24414062500000000', '12207031250000000', '1220703125000000', '6103515625000000', '6103515625000000', '48828125000000000', '6103515625000000', '9765625000000000', '9765625000000000', '6103515625000000', '12207031250000000', '610351562500000', '6103515625000000', '1525878906250000', '48828125000000000', '9765625000000000', '3051757812500000', '12207031250000000', '488281250000000', '762939453125000', '6103515625000000', '3051757812500000', '6103515625000000', '48828125000000000', '48828125000000000', '12207031250000000', '12207031250000000', '190734863281250', '6103515625000000', '48828125000000000', '48828125000000000', '390625000000000', '24414062500000000', '24414062500000000', '390625000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '9765625000000000', '390625000000000000', '781250000000000000', '12207031250000000', '195312500000000000', '24414062500000000', '781250000000000000', '24414062500000000', '24414062500000000', '781250000000000000', '12207031250000000', '781250000000000000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '4', '4', '4', '4', '1', '1', '6103515625000000', '6103515625000000', '4882812500000000', '24414062500000000', '781250000000000000', '48828125000000000', '781250000000000000', '39062500000000000', '3051757812500000', '781250000000000000', '3051757812500000', '3051757812500000', '48828125000000000', '781250000000000000', '390625000000000000', '390625000000000000', '19531250000000000', '97656250000000000', '19531250000000000', '19531250000000000', '19531250000000000', '19531250000000000', '24414062500000000', '19531250000000000', '24414062500000000', '24414062500000000', '390625000000000000', '19531250000000000', '19531250000000000', '390625000000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '78125000000000000', '195312500000000000', '195312500000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '1562500000000000000', '390625000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '2441406250000000', '781250000000000000', '2441406250000000', '781250000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '2441406250000000', '781250000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '195312500000000000', '24414062500000000', '24414062500000000', '781250000000000000', '24414062500000000', '24414062500000000', '781250000000000000', '48828125000000000', '781250000000000000', '4', '4', '4', '4', '1', '4', '4', '4', '4', '48828125000000000', '4882812500000000', '1525878906250000', '48828125000000000', '48828125000000000', '4882812500000000', '1525878906250000', '48828125000000000', '9765625000000000', '48828125000000000', '12207031250000000', '24414062500000000', '9765625000000000', '48828125000000000', '12207031250000000', '24414062500000000', '48828125000000000', '4882812500000000', '12207031250000000', '48828125000000000', '48828125000000000', '4882812500000000', '12207031250000000', '48828125000000000', '48828125000000000', '48828125000000000', '6103515625000000', '6103515625000000', '48828125000000000', '48828125000000000', '6103515625000000', '6103515625000000', '24414062500000000', '9765625000000000', '3051757812500000', '48828125000000000', '24414062500000000', '9765625000000000', '3051757812500000', '48828125000000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '24414062500000000', '12207031250000000', '6103515625000000', '24414062500000000', '195312500000000000', '19531250000000000', '3125000000000000000', '390625000000000000', '1562500000000000000', '3125000000000000000', '48828125000000000', '12207031250000000', '48828125000000000', '390625000000000', '1525878906250000', '6103515625000000', '12207031250000000', '6103515625000000', '48828125000000000', '12207031250000000', '48828125000000000', '48828125000000000', '2441406250000000', '12207031250000000', '1525878906250000', '3051757812500000', '1525878906250000', '1525878906250000', '24414062500000000', '48828125000000000', '12207031250000000', '12207031250000000', '3051757812500000', '6103515625000000', '48828125000000000', '12207031250000000', '2441406250000000', '12207031250000000', '12207031250000000', '6103515625000000', '6103515625000000', '976562500000000', '48828125000000000', '24414062500000000', '48828125000000000', '48828125000000000', '12207031250000000', '610351562500000', '12207031250000000', '24414062500000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '12207031250000000', '6103515625000000', '6103515625000000', '195312500000000000', '9765625000000000', '78125000000000000', '390625000000000000', '390625000000000000', '39062500000000000', '390625000000000000', '39062500000000000', '48828125000000000', '48828125000000000', '488281250000000', '390625000000000000', '390625000000000000', '39062500000000000', '48828125000000000', '48828125000000000', '488281250000000', '390625000000000000', '39062500000000000', '390625000000000000', '1953125000000000', '1953125000000000', '12207031250000000', '390625000000000000', '39062500000000000', '390625000000000000', '1953125000000000', '1953125000000000', '12207031250000000', '390625000000000000', '156250000000000000', '390625000000000000', '1953125000000000', '48828125000000000', '4882812500000000', '156250000000000000', '156250000000000000', '390625000000000000', '1953125000000000', '48828125000000000', '4882812500000000', '156250000000000000', '156250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '156250000000000000', '156250000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '156250000000000000', '156250000000000000', '390625000000000000', '1220703125000000', '6103515625000000', '12207031250000000', '156250000000000000', '156250000000000000', '390625000000000000', '1220703125000000', '6103515625000000', '12207031250000000', '156250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '24414062500000000', '24414062500000000', '781250000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '24414062500000000', '24414062500000000', '781250000000000000', '100000000000000000000', '100000000000000000000', '12500000000000000000', '2500000000000000000', '20000000000000000000', '100000000000000000000', '12207031250000000', '12207031250000000', '610351562500000', '15625000000000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '610351562500000', '12207031250000000', '3051757812500000', '12207031250000000', '48828125000000000', '244140625000000', '762939453125000', '381469726562500', '6103515625000000', '1525878906250000', '610351562500000', '6103515625000000', '762939453125000', '6103515625000000', '12207031250000000', '2441406250000000', '610351562500000', '3051757812500000', '6103515625000000', '762939453125000', '762939453125000', '2441406250000000', '12207031250000000', '3051757812500000', '4882812500000000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '12207031250000000', '6103515625000000', '610351562500000', '97656250000000000', '12207031250000000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '1220703125000000', '3051757812500000', '12207031250000000', '1220703125000000', '2441406250000000', '12207031250000000', '3051757812500000', '1525878906250000', '762939453125000', '3051757812500000', '381469726562500', '15625000000000000', '25000000000000000000', '19531250000000000', '1562500000000000000', '97656250000000000', '24414062500000000', '195312500000000000', '25000000000000000000', '97656250000000000', '390625000000000000', '781250000000000000', '97656250000000000', '625000000000000000', '195312500000000000', '48828125000000000', '625000000000000000', '4882812500000000', '97656250000000000', '1562500000000000000', '97656250000000000', '781250000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '97656250000000000', '1562500000000000000', '1562500000000000000', '97656250000000000', '781250000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '97656250000000000', '1562500000000000000', '12500000000000000000', '2441406250000000', '12500000000000000000', '97656250000000000', '390625000000000000', '12207031250000000', '2441406250000000', '625000000000000000', '12500000000000000000', '2441406250000000', '12500000000000000000', '97656250000000000', '390625000000000000', '12207031250000000', '2441406250000000', '625000000000000000', '2500000000000000000', '97656250000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '2500000000000000000', '97656250000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '6103515625000000', '97656250000000000', '12500000000000000000', '200000000000000000', '97656250000000000', '500000000000000000', '195312500000000000', '195312500000000000', '3051757812500000', '97656250000000000', '200000000000000000', '200000000000000000', '97656250000000000', '500000000000000000', '195312500000000000', '195312500000000000', '3051757812500000', '97656250000000000', '200000000000000000', '12500000000000000000', '1953125000000000', '6250000000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '1953125000000000', '12500000000000000000', '12500000000000000000', '1953125000000000', '6250000000000000000', '390625000000000000', '390625000000000000', '6103515625000000', '1953125000000000', '12500000000000000000', '12500000000000000000', '19531250000000000', '2500000000000000000', '78125000000000000', '97656250000000000', '2441406250000000', '24414062500000000', '3125000000000000000', '12500000000000000000', '19531250000000000', '2500000000000000000', '78125000000000000', '97656250000000000', '2441406250000000', '24414062500000000', '3125000000000000000', '50000000000000000000', '100000000000000000000', '50000000000000000000', '10000000000000000000', '100000000000000000', '100000000000000000000', '1220703125000000', '6103515625000000', '1525878906250000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '12207031250000000', '1525878906250000', '12207031250000000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '1525878906250000', '2441406250000000', '19073486328125', '6103515625000000', '381469726562500', '2441406250000000', '190734863281250', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '1525878906250000', '12207031250000000', '2441406250000000', '6103515625000000', '3051757812500000', '2441406250000000', '762939453125000', '12207031250000000', '1525878906250000', '122070312500000', '381469726562500', '381469726562500', '6103515625000000', '762939453125000', '12207031250000000', '305175781250000', '1525878906250000', '1220703125000000', '1525878906250000', '1525878906250000', '610351562500000', '1525878906250000', '6103515625000000', '6103515625000000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '122070312500000', '12207031250000000', '95367431640625', '6103515625000000', '122070312500000', '3051757812500000', '1525878906250000', '1525878906250000', '1220703125000000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '122070312500000', '2441406250000000', '3051757812500000', '762939453125000', '7812500000000000', '97656250000000000', '3125000000000000000', '25000000000000000000', '19531250000000000', '195312500000000000', '195312500000000000', '48828125000000000', '6250000000000000000', '390625000000000000', '97656250000000000', '97656250000000000', '6250000000000000000', '19531250000000000', '390625000000000000', '3125000000000000000', '48828125000000000', '195312500000000000', '1220703125000000', '6250000000000000000', '48828125000000000', '625000000000000000', '50000000000000000', '4882812500000000', '12207031250000000', '390625000000000000', '1220703125000000', '6250000000000000000', '48828125000000000', '625000000000000000', '50000000000000000', '4882812500000000', '12207031250000000', '390625000000000000', '390625000000000000', '25000000000000000000', '97656250000000000', '1250000000000000000', '25000000000000000000', '97656250000000000', '762939453125000', '390625000000000000', '390625000000000000', '25000000000000000000', '97656250000000000', '1250000000000000000', '25000000000000000000', '97656250000000000', '762939453125000', '390625000000000000', '39062500000000000', '488281250000000', '48828125000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '195312500000000000', '39062500000000000', '488281250000000', '48828125000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '610351562500000', '19531250000000000', '25000000000000000000', '6250000000000000000', '4882812500000000', '25000000000000000000', '390625000000000000', '97656250000000000', '610351562500000', '19531250000000000', '25000000000000000000', '6250000000000000000', '4882812500000000', '25000000000000000000', '390625000000000000', '195312500000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '48828125000000000', '97656250000000000', '12207031250000000', '195312500000000000', '195312500000000000', '25000000000000000000', '97656250000000000', '25000000000000000000', '48828125000000000', '97656250000000000', '12207031250000000', '195312500000000000', '390625000000000000', '25000000000000000000', '12207031250000000', '3125000000000000000', '25000000000000000000', '12207031250000000', '3051757812500000', '390625000000000000', '390625000000000000', '25000000000000000000', '12207031250000000', '3125000000000000000', '25000000000000000000', '12207031250000000', '3051757812500000', '390625000000000000', '3125000000000000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '25000000000000000000', '50000000000000000000', '6103515625000000', '305175781250000', '95367431640625', '6103515625000000', '38146972656250', '6103515625000000', '1525878906250000', '305175781250000', '6103515625000000', '24414062500000000', '1220703125000000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '762939453125000', '2441406250000000', '1525878906250000', '6103515625000000', '24414062500000000', '24414062500000000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '762939453125000', '1525878906250000', '3051757812500000', '24414062500000000', '24414062500000000', '762939453125000', '610351562500000', '95367431640625', '610351562500000', '1525878906250000', '6103515625000000', '3051757812500000', '381469726562500', '762939453125000', '305175781250000', '4882812500000000', '24414062500000000', '6103515625000000', '381469726562500', '1525878906250000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '12207031250000000', '95367431640625', '1525878906250000', '24414062500000000', '2441406250000000', '6103515625000000', '1525878906250000', '3051757812500000', '381469726562500', '762939453125000', '762939453125000', '19073486328125', '6103515625000000', '1525878906250000', '3051757812500000', '381469726562500', '6103515625000000', '3051757812500000', '1525878906250000', '39062500000000000', '97656250000000000', '6250000000000000000', '97656250000000000', '6250000000000000000', '390625000000000000', '39062500000000000', '78125000000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '19531250000000000', '2500000000000000000', '19531250000000000', '24414062500000000', '250000000000000000', '390625000000000000', '762939453125000', '3051757812500000', '97656250000000000', '97656250000000000', '1220703125000000', '48828125000000000', '762939453125000', '3051757812500000', '97656250000000000', '97656250000000000', '1220703125000000', '48828125000000000', '2441406250000000', '781250000000000000', '312500000000000000', '312500000000000000', '24414062500000000', '2441406250000000', '2441406250000000', '781250000000000000', '312500000000000000', '312500000000000000', '24414062500000000', '2441406250000000', '24414062500000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '4882812500000000', '12207031250000000', '24414062500000000', '781250000000000000', '1562500000000000000', '1562500000000000000', '4882812500000000', '12207031250000000', '1220703125000000', '781250000000000000', '195312500000000000', '781250000000000000', '12207031250000000', '9765625000000000', '1220703125000000', '781250000000000000', '195312500000000000', '781250000000000000', '12207031250000000', '9765625000000000', '48828125000000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '4882812500000000', '48828125000000000', '12207031250000000', '781250000000000000', '781250000000000000', '390625000000000000', '4882812500000000', '24414062500000000', '390625000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '390625000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '24414062500000000', '100000000000000000000', '100000000000000000000', '25000000000000000000', '50000000000000000000', '12500000000000000000', '50000000000000000000', '244140625000000', '12207031250000000', '610351562500000', '6103515625000000', '6103515625000000', '762939453125000', '6103515625000000', '24414062500000000', '6103515625000000', '3051757812500000', '12207031250000000', '3051757812500000', '30517578125000', '3051757812500000', '1525878906250000', '762939453125000', '12207031250000000', '195312500000000000', '12207031250000000', '6103515625000000', '6103515625000000', '610351562500000', '3051757812500000', '6103515625000000', '6103515625000000', '381469726562500', '3051757812500000', '97656250000000000', '6103515625000000', '6103515625000000', '12207031250000000', '12207031250000000', '610351562500000', '762939453125000', '3051757812500000', '3051757812500000', '12207031250000000', '97656250000000000', '3051757812500000', '610351562500000', '12207031250000000', '12207031250000000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '195312500000000000', '12207031250000000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '390625000000000000', '12207031250000000', '6103515625000000', '48828125000000000', '39062500000000000', '312500000000000000', '12207031250000000', '1562500000000000000', '390625000000000000', '24414062500000000', '19531250000000000', '78125000000000000', '12207031250000000', '390625000000000000', '3125000000000000000', '97656250000000000', '50000000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '195312500000000000', '24414062500000000', '48828125000000000', '48828125000000000', '6103515625000000', '24414062500000000', '48828125000000000', '48828125000000000', '6103515625000000', '488281250000000', '24414062500000000', '24414062500000000', '6103515625000000', '488281250000000', '24414062500000000', '24414062500000000', '6103515625000000', '24414062500000000', '48828125000000000', '48828125000000000', '3051757812500000', '24414062500000000', '48828125000000000', '48828125000000000', '3051757812500000', '12207031250000000', '24414062500000000', '24414062500000000', '6103515625000000', '12207031250000000', '24414062500000000', '24414062500000000', '6103515625000000', '24414062500000000', '48828125000000000', '48828125000000000', '1220703125000000', '24414062500000000', '48828125000000000', '48828125000000000', '1220703125000000', '488281250000000', '48828125000000000', '48828125000000000', '24414062500000000', '488281250000000', '48828125000000000', '48828125000000000', '24414062500000000', '62500000000000000', '12500000000000000000', '390625000000000000', '39062500000000000', '12500000000000000000', '781250000000000000', '24414062500000000', '24414062500000000', '12207031250000000', '6103515625000000', '12207031250000000', '1220703125000000', '9765625000000000', '4882812500000000', '24414062500000000', '24414062500000000', '95367431640625', '3051757812500000', '95367431640625', '24414062500000000', '24414062500000000', '3051757812500000', '24414062500000000', '976562500000000', '76293945312500', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '12207031250000000', '3051757812500000', '24414062500000000', '12207031250000000', '6103515625000000', '12207031250000000', '12207031250000000', '24414062500000000', '12207031250000000', '24414062500000000', '48828125000000000', '3051757812500000', '1525878906250000', '3051757812500000', '48828125000000000', '48828125000000000', '12207031250000000', '12207031250000000', '48828125000000000', '6103515625000000', '6103515625000000', '6103515625000000', '48828125000000000', '24414062500000000', '190734863281250', '390625000000000000', '390625000000000000', '78125000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '4', '4', '4', '4', '781250000000000000', '1220703125000000', '781250000000000000', '39062500000000000', '6103515625000000', '781250000000000000', '6103515625000000', '6103515625000000', '1220703125000000', '781250000000000000', '781250000000000000', '6103515625000000', '6103515625000000', '781250000000000000', '48828125000000000', '390625000000000000', '781250000000000000', '6103515625000000', '6103515625000000', '781250000000000000', '48828125000000000', '390625000000000000', '781250000000000000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '781250000000000000', '781250000000000000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '781250000000000000', '390625000000000000', '3051757812500000', '195312500000000000', '24414062500000000', '3051757812500000', '390625000000000000', '390625000000000000', '3051757812500000', '195312500000000000', '24414062500000000', '3051757812500000', '390625000000000000', '156250000000000000', '48828125000000000', '390625000000000000', '24414062500000000', '48828125000000000', '156250000000000000', '156250000000000000', '48828125000000000', '390625000000000000', '24414062500000000', '48828125000000000', '156250000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '3051757812500000', '48828125000000000', '781250000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '3051757812500000', '48828125000000000', '781250000000000000', '195312500000000000', '48828125000000000', '97656250000000000', '1525878906250000', '48828125000000000', '195312500000000000', '195312500000000000', '48828125000000000', '97656250000000000', '1525878906250000', '48828125000000000', '195312500000000000', '3125000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '25000000000000000000', '100000000000000000000', '762939453125000', '12207031250000000', '1525878906250000', '3051757812500000', '1220703125000000', '1525878906250000', '1525878906250000', '3051757812500000', '305175781250000', '3051757812500000', '610351562500000', '97656250000000000', '2441406250000000', '12207031250000000', '12207031250000000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '762939453125000', '12207031250000000', '97656250000000000', '1220703125000000', '3051757812500000', '12207031250000000', '61035156250000', '1220703125000000', '6103515625000000', '3051757812500000', '19073486328125', '12207031250000000', '97656250000000000', '381469726562500', '6103515625000000', '12207031250000000', '95367431640625', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '48828125000000000', '12207031250000000', '12207031250000000', '488281250000000', '3051757812500000', '6103515625000000', '762939453125000', '3051757812500000', '6103515625000000', '12207031250000000', '97656250000000000', '3051757812500000', '12207031250000000', '12207031250000000', '3051757812500000', '1525878906250000', '6103515625000000', '95367431640625', '3051757812500000', '390625000000000000', '1525878906250000', '100000000000000000000', '195312500000000000', '19531250000000000', '390625000000000000', '97656250000000000', '24414062500000000', '6250000000000000000', '781250000000000000', '195312500000000000', '9765625000000000', '156250000000000000', '19531250000000000', '390625000000000000', '1562500000000000000', '195312500000000000', '195312500000000000', '625000000000000000', '195312500000000000', '2441406250000000', '195312500000000000', '1562500000000000000', '195312500000000000', '2441406250000000', '195312500000000000', '625000000000000000', '625000000000000000', '292968750000000000', '585937500000000000', '292968750000000000', '1562500000000000000', '73242187500000000', '585937500000000000', '146484375000000000', '625000000000000000', '3125000000000000000', '195312500000000000', '12207031250000000', '97656250000000000', '1562500000000000000', '97656250000000000', '12207031250000000', '195312500000000000', '3125000000000000000', '3125000000000000000', '1171875000000000000', '234375000000000000', '1171875000000000000', '1562500000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '3125000000000000000', '3125000000000000000', '97656250000000000', '12207031250000000', '19531250000000000', '6250000000000000000', '19531250000000000', '12207031250000000', '97656250000000000', '6250000000000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '6250000000000000000', '390625000000000000', '78125000000000000', '390625000000000000', '6250000000000000000', '25000000000000000000', '195312500000000000', '48828125000000000', '48828125000000000', '5000000000000000000', '48828125000000000', '48828125000000000', '195312500000000000', '2500000000000000000', '25000000000000000000', '146484375000000000', '585937500000000000', '585937500000000000', '5000000000000000000', '585937500000000000', '585937500000000000', '3662109375000000', '2500000000000000000', '25000000000000000000', '48828125000000000', '3051757812500000', '97656250000000000', '12500000000000000000', '97656250000000000', '3051757812500000', '48828125000000000', '25000000000000000000', '25000000000000000000', '234375000000000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '25000000000000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '6250000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '12500000000000000000', '9375000000000000', '1171875000000000000', '1171875000000000000', '6250000000000000000', '1171875000000000000', '46875000000000000', '1171875000000000000', '12500000000000000000', '100000000000000000000', '3125000000000000000', '12500000000000000000', '6250000000000000000', '100000000000000000000', '6103515625000000', '6103515625000000', '24414062500000000', '6103515625000000', '1525878906250000', '24414062500000000', '3051757812500000', '152587890625000', '1525878906250000', '1525878906250000', '76293945312500', '152587890625000', '38146972656250', '6103515625000000', '48828125000000000', '6103515625000000', '3051757812500000', '24414062500000000', '95367431640625', '305175781250000', '3051757812500000', '3051757812500000', '6103515625000000', '381469726562500', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '3051757812500000', '6103515625000000', '1525878906250000', '1525878906250000', '1525878906250000', '24414062500000000', '305175781250000', '1525878906250000', '390625000000000', '6103515625000000', '1525878906250000', '3051757812500000', '381469726562500', '1525878906250000', '1525878906250000', '3051757812500000', '3051757812500000', '12207031250000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '152587890625000', '1525878906250000', '3051757812500000', '6103515625000000', '152587890625000', '1525878906250000', '6103515625000000', '24414062500000000', '6103515625000000', '6103515625000000', '6103515625000000', '6103515625000000', '190734863281250', '6103515625000000', '762939453125000', '6103515625000000', '190734863281250', '1525878906250000', '195312500000000000', '78125000000000000', '19531250000000000', '24414062500000000', '97656250000000000', '390625000000000000', '19531250000000000', '4882812500000000', '97656250000000000', '48828125000000000', '20000000000000000000', '390625000000000000', '97656250000000000', '15625000000000000', '97656250000000000', '24414062500000000', '195312500000000000', '1562500000000000000', '195312500000000000', '12500000000000000000', '97656250000000000', '24414062500000000', '390625000000000000', '97656250000000000', '12500000000000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '25000000000000000000', '24414062500000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '195312500000000000', '20000000000000000000', '78125000000000000', '390625000000000000', '9765625000000000', '48828125000000000', '312500000000000000', '976562500000000', '12207031250000000', '312500000000000000', '781250000000000000', '9765625000000000', '390625000000000000', '5000000000000000000', '2500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '781250000000000000', '24414062500000000', '48828125000000000', '3125000000000000000', '1562500000000000000', '6103515625000000', '3125000000000000000', '48828125000000000', '24414062500000000', '781250000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '25000000000000000000', '78125000000000000', '48828125000000000', '390625000000000000', '625000000000000000', '312500000000000000', '2441406250000000', '625000000000000000', '781250000000000000', '48828125000000000', '78125000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '48828125000000000', '781250000000000000', '781250000000000000', '4882812500000000', '4882812500000000', '781250000000000000', '781250000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '3125000000000000000', '3125000000000000000', '781250000000000000', '625000000000000000', '195312500000000000', '3125000000000000000', '781250000000000000', '3125000000000000000', '3125000000000000000', '781250000000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '1562500000000000000', '762939453125000', '3125000000000000000', '195312500000000000', '24414062500000000', '781250000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '2441406250000000', '390625000000000000', '1562500000000000000', '781250000000000000', '24414062500000000', '1562500000000000000', '48828125000000000', '24414062500000000', '390625000000000000', '1562500000000000000', '1250000000000000000', '6250000000000000000', '390625000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '1562500000000000000', '6250000000000000000', '6250000000000000000', '10000000000000000000', '25000000000000000000', '195312500000000000', '24414062500000000', '97656250000000000', '50000000000000000000', '6103515625000000', '9765625000000000', '6103515625000000', '381469726562500', '381469726562500', '3051757812500000', '38146972656250', '610351562500000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '762939453125000', '1220703125000000', '3051757812500000', '1525878906250000', '6103515625000000', '3051757812500000', '1525878906250000', '3051757812500000', '152587890625000', '1525878906250000', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '762939453125000', '3051757812500000', '3051757812500000', '762939453125000', '1525878906250000', '3051757812500000', '381469726562500', '4882812500000000', '6103515625000000', '762939453125000', '1220703125000000', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '3051757812500000', '48828125000000000', '95367431640625', '3051757812500000', '1220703125000000', '6103515625000000', '3051757812500000', '1525878906250000', '1525878906250000', '6103515625000000', '381469726562500', '762939453125000', '6103515625000000', '12207031250000000', '610351562500000', '610351562500000', '1220703125000000', '3051757812500000', '190734863281250', '1525878906250000', '6103515625000000', '3051757812500000', '610351562500000', '76293945312500', '9765625000000000', '97656250000000000', '195312500000000000', '3051757812500000', '12500000000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '78125000000000000', '97656250000000000', '39062500000000000', '100000000000000000000', '97656250000000000', '48828125000000000', '2441406250000000', '3906250000000000', '100000000000000000000', '195312500000000000', '3051757812500000', '48828125000000000', '3051757812500000', '195312500000000000', '195312500000000000', '390625000000000000', '3906250000000000', '97656250000000000', '195312500000000000', '97656250000000000', '390625000000000000', '3125000000000000000', '12207031250000000', '24414062500000000', '97656250000000000', '3906250000000000', '39062500000000000', '48828125000000000', '48828125000000000', '195312500000000000', '2500000000000000000', '97656250000000000', '250000000000000000', '97656250000000000', '2500000000000000000', '195312500000000000', '48828125000000000', '9375000000000000', '1171875000000000000', '2500000000000000000', '1171875000000000000', '250000000000000000', '234375000000000000', '2500000000000000000', '1171875000000000000', '1171875000000000000', '24414062500000000', '19531250000000000', '25000000000000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '12500000000000000000', '19531250000000000', '24414062500000000', '29296875000000000', '292968750000000000', '25000000000000000000', '585937500000000000', '25000000000000000000', '117187500000000000', '12500000000000000000', '292968750000000000', '146484375000000000', '24414062500000000', '97656250000000000', '12500000000000000000', '97656250000000000', '781250000000000000', '97656250000000000', '2500000000000000000', '97656250000000000', '24414062500000000', '1171875000000000000', '1171875000000000000', '12500000000000000000', '1171875000000000000', '781250000000000000', '1171875000000000000', '2500000000000000000', '1171875000000000000', '1171875000000000000', '48828125000000000', '9765625000000000', '25000000000000000000', '48828125000000000', '25000000000000000000', '48828125000000000', '3125000000000000000', '97656250000000000', '9765625000000000', '58593750000000000', '292968750000000000', '25000000000000000000', '292968750000000000', '25000000000000000000', '292968750000000000', '3125000000000000000', '146484375000000000', '292968750000000000', '48828125000000000', '6103515625000000', '25000000000000000000', '48828125000000000', '25000000000000000000', '48828125000000000', '12500000000000000000', '6103515625000000', '48828125000000000', '390625000000000000', '390625000000000000', '25000000000000000000', '390625000000000000', '25000000000000000000', '390625000000000000', '12500000000000000000', '78125000000000000', '390625000000000000', '48828125000000000', '48828125000000000', '5000000000000000000', '195312500000000000', '25000000000000000000', '195312500000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '29296875000000000', '146484375000000000', '5000000000000000000', '585937500000000000', '25000000000000000000', '585937500000000000', '12500000000000000000', '585937500000000000', '146484375000000000', '25000000000000000000', '50000000000000000000', '12500000000000000000', '25000000000000000000', '6103515625000000', '50000000000000000000', '6103515625000000', '24414062500000000', '381469726562500', '305175781250000', '122070312500000', '3051757812500000', '762939453125000', '762939453125000', '6103515625000000', '1525878906250000', '1220703125000000', '1525878906250000', '6103515625000000', '4882812500000000', '6103515625000000', '381469726562500', '6103515625000000', '610351562500000', '381469726562500', '381469726562500', '6103515625000000', '4882812500000000', '3051757812500000', '3051757812500000', '3051757812500000', '24414062500000000', '6103515625000000', '381469726562500', '61035156250000', '1525878906250000', '1220703125000000', '1525878906250000', '3051757812500000', '190734863281250', '1525878906250000', '305175781250000', '3051757812500000', '12207031250000000', '6103515625000000', '1525878906250000', '1525878906250000', '190734863281250', '6103515625000000', '381469726562500', '3051757812500000', '12207031250000000', '1220703125000000', '381469726562500', '3051757812500000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '24414062500000000', '6103515625000000', '3051757812500000', '3051757812500000', '24414062500000000', '95367431640625', '1525878906250000', '3051757812500000', '3051757812500000', '305175781250000', '1525878906250000', '6103515625000000', '48828125000000000', '6103515625000000', '762939453125000', '39062500000000000', '19531250000000000', '390625000000000000', '97656250000000000', '6250000000000000000', '24414062500000000', '97656250000000000', '97656250000000000', '390625000000000000', '195312500000000000', '390625000000000000', '195312500000000000', '195312500000000000', '97656250000000000', '100000000000000000000', '3125000000000000000', '3125000000000000000', '390625000000000000', '195312500000000000', '195312500000000000', '100000000000000000000', '1562500000000000000', '50000000000000000000', '39062500000000000', '97656250000000000', '6103515625000000', '390625000000000000', '39062500000000000', '78125000000000000', '156250000000000000', '6103515625000000', '7812500000000000', '781250000000000000', '48828125000000000', '195312500000000000', '19531250000000000', '1953125000000000', '97656250000000000', '9765625000000000', '24414062500000000', '97656250000000000', '1953125000000000', '1953125000000000', '97656250000000000', '9765625000000000', '24414062500000000', '97656250000000000', '1953125000000000', '24414062500000000', '781250000000000000', '312500000000000000', '4882812500000000', '1562500000000000000', '24414062500000000', '24414062500000000', '781250000000000000', '312500000000000000', '4882812500000000', '1562500000000000000', '24414062500000000', '48828125000000000', '781250000000000000', '1562500000000000000', '12207031250000000', '1562500000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '12207031250000000', '1562500000000000000', '48828125000000000', '610351562500000', '1562500000000000000', '6250000000000000', '24414062500000000', '1562500000000000000', '6103515625000000', '610351562500000', '1562500000000000000', '6250000000000000', '24414062500000000', '1562500000000000000', '6103515625000000', '48828125000000000', '1562500000000000000', '781250000000000000', '6103515625000000', '1562500000000000000', '48828125000000000', '48828125000000000', '1562500000000000000', '781250000000000000', '6103515625000000', '1562500000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '24414062500000000', '12500000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '1562500000000000000', '24414062500000000', '12500000000000000', '48828125000000000', '6250000000000000000', '12500000000000000000', '10000000000000000000', '100000000000000000000', '20000000000000000000', '50000000000000000000', '6103515625000000', '97656250000000000', '12207031250000000', '3051757812500000', '190734863281250', '3051757812500000', '3051757812500000', '1525878906250000', '12207031250000000', '1525878906250000', '2441406250000000', '195312500000000000', '2441406250000000', '1525878906250000', '610351562500000', '6103515625000000', '1525878906250000', '12207031250000000', '6103515625000000', '6103515625000000', '381469726562500', '195312500000000000', '12207031250000000', '95367431640625', '1525878906250000', '6103515625000000', '3051757812500000', '12207031250000000', '6103515625000000', '6103515625000000', '12207031250000000', '4882812500000000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '305175781250000', '76293945312500', '762939453125000', '762939453125000', '3051757812500000', '39062500000000000', '610351562500000', '762939453125000', '762939453125000', '1525878906250000', '1525878906250000', '12207031250000000', '6103515625000000', '762939453125000', '12207031250000000', '195312500000000000', '1525878906250000', '3051757812500000', '6103515625000000', '6103515625000000', '3051757812500000', '12207031250000000', '12207031250000000', '762939453125000', '4882812500000000', '390625000000000000', '25000000000000000000', '2441406250000000', '195312500000000000', '1562500000000000000', '195312500000000000', '156250000000000000', '48828125000000000', '39062500000000000', '1562500000000000000', '97656250000000000', '48828125000000000', '78125000000000000', '100000000000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '1562500000000000000', '48828125000000000', '1562500000000000000', '78125000000000000', '24414062500000000', '1562500000000000000', '24414062500000000', '24414062500000000', '4882812500000000', '1562500000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '195312500000000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '2441406250000000', '390625000000000000', '2441406250000000', '2441406250000000', '781250000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '500000000000000000', '195312500000000000', '97656250000000000', '305175781250000', '1250000000000000000', '97656250000000000', '195312500000000000', '12500000000000000000', '500000000000000000', '195312500000000000', '97656250000000000', '305175781250000', '1250000000000000000', '97656250000000000', '195312500000000000', '12500000000000000000', '25000000000000000000', '390625000000000000', '3906250000000000', '12500000000000000000', '305175781250000', '6103515625000000', '97656250000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '3906250000000000', '12500000000000000000', '305175781250000', '6103515625000000', '97656250000000000', '25000000000000000000', '390625000000000000', '390625000000000000', '1220703125000000', '390625000000000000', '3051757812500000', '1220703125000000', '195312500000000000', '390625000000000000', '390625000000000000', '390625000000000000', '1220703125000000', '390625000000000000', '3051757812500000', '1220703125000000', '195312500000000000', '390625000000000000', '1562500000000000000', '78125000000000000', '19531250000000000', '762939453125000', '625000000000000000', '12207031250000000', '195312500000000000', '3125000000000000000', '1562500000000000000', '78125000000000000', '19531250000000000', '762939453125000', '625000000000000000', '12207031250000000', '195312500000000000', '3125000000000000000', '390625000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '3051757812500000', '3051757812500000', '390625000000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '3051757812500000', '3051757812500000', '390625000000000000', '12500000000000000000', '6250000000000000000', '390625000000000000', '9765625000000000', '1250000000000000000', '6103515625000000', '12207031250000000', '390625000000000000', '7812500000000000', '6250000000000000000', '390625000000000000', '9765625000000000', '1250000000000000000', '6103515625000000', '12207031250000000', '390625000000000000', '7812500000000000', '50000000000000000000', '1562500000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '50000000000000000000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '3051757812500000', '3051757812500000', '305175781250000', '3051757812500000', '3051757812500000', '61035156250000', '3051757812500000', '24414062500000000', '24414062500000000', '38146972656250', '3051757812500000', '95367431640625', '1525878906250000', '12207031250000000', '3051757812500000', '488281250000000', '610351562500000', '95367431640625', '1525878906250000', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '6103515625000000', '381469726562500', '610351562500000', '610351562500000', '6103515625000000', '381469726562500', '610351562500000', '1525878906250000', '12207031250000000', '4882812500000000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '12207031250000000', '3051757812500000', '3051757812500000', '1525878906250000', '3051757812500000', '610351562500000', '3051757812500000', '12207031250000000', '1220703125000000', '122070312500000', '610351562500000', '762939453125000', '6103515625000000', '610351562500000', '12207031250000000', '95367431640625', '3051757812500000', '3051757812500000', '976562500000000', '24414062500000000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '6103515625000000', '762939453125000', '6103515625000000', '30517578125000', '1525878906250000', '195312500000000000', '1562500000000000000', '48828125000000000', '15625000000000000', '1562500000000000000', '19531250000000000', '390625000000000000', '1562500000000000000', '19531250000000000', '6250000000000000000', '195312500000000000', '7812500000000000', '3125000000000000000', '390625000000000000', '9765625000000000', '781250000000000000', '97656250000000000', '195312500000000000', '1562500000000000000', '781250000000000000', '390625000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '781250000000000000', '390625000000000000', '781250000000000000', '1562500000000000000', '2500000000000000000', '12500000000000000000', '2500000000000000000', '2500000000000000000', '781250000000000000', '781250000000000000', '12500000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '156250000000000000', '97656250000000000', '390625000000000000', '625000000000000000', '9765625000000000', '9765625000000000', '24414062500000000', '390625000000000000', '97656250000000000', '3125000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '6250000000000000000', '2500000000000000000', '2500000000000000000', '12500000000000000000', '2500000000000000000', '625000000000000000', '781250000000000000', '15625000000000000', '62500000000000000', '12207031250000000', '12207031250000000', '4882812500000000', '195312500000000000', '781250000000000000', '625000000000000000', '3125000000000000000', '390625000000000000', '781250000000000000', '1562500000000000000', '781250000000000000', '781250000000000000', '781250000000000000', '390625000000000000', '781250000000000000', '3125000000000000000', '781250000000000000', '390625000000000000', '390625000000000000', '1220703125000000', '48828125000000000', '48828125000000000', '390625000000000000', '390625000000000000', '15625000000000000', '781250000000000000', '6250000000000000000', '50000000000000000', '6250000000000000000', '3125000000000000000', '6250000000000000000', '6250000000000000000', '6250000000000000000', '3125000000000000000', '3125000000000000000', '625000000000000000', '1562500000000000000', '781250000000000000', '781250000000000000', '24414062500000000', '48828125000000000', '48828125000000000', '781250000000000000', '781250000000000000', '781250000000000000', '1562500000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '6250000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '2500000000000000000', '625000000000000000', '39062500000000000', '156250000000000000', '1562500000000000000', '48828125000000000', '6103515625000000', '4882812500000000', '97656250000000000', '78125000000000000', '625000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '250000000000000000', '20000000000000000000', '5000000000000000000', '100000000000000000000', '1562500000000000000', '3125000000000000000', '381469726562500', '1525878906250000', '762939453125000', '762939453125000', '122070312500000', '1220703125000', '3051757812500000', '152587890625000', '610351562500000', '6103515625000000', '1525878906250000', '1525878906250000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '3051757812500000', '1953125000000000', '6103515625000000', '1525878906250000', '6103515625000000', '1525878906250000', '1525878906250000', '3051757812500000', '762939453125000', '6103515625000000', '1220703125000000', '1525878906250000', '1220703125000000', '48828125000000000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '3051757812500000', '1525878906250000', '305175781250000', '24414062500000', '6103515625000000', '190734863281250', '3051757812500000', '24414062500000000', '610351562500000', '3051757812500000', '3051757812500000', '762939453125000', '3051757812500000', '122070312500000', '95367431640625', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '48828125000000000', '1220703125000000', '381469726562500', '6103515625000000', '305175781250000', '762939453125000', '762939453125000', '762939453125000', '6103515625000000', '3051757812500000', '610351562500000', '1525878906250000', '24414062500000000', '6103515625000000', '762939453125000', '1525878906250000', '6103515625000000', '3051757812500000', '3051757812500000', '24414062500000000', '97656250000000000', '6250000000000000000', '195312500000000000', '78125000000000000', '19531250000000000', '12207031250000000', '48828125000000000', '156250000000000000', '390625000000000000', '1220703125000000', '97656250000000000', '97656250000000000', '9765625000000000', '12500000000000000000', '390625000000000000', '390625000000000000', '24414062500000000', '97656250000000000', '48828125000000000', '78125000000000000', '100000000000000000000', '390625000000000000', '97656250000000000', '3906250000000000', '39062500000000000', '78125000000000000', '781250000000000000', '390625000000000000', '48828125000000000', '4882812500000000', '24414062500000000', '24414062500000000', '100000000000000000000', '195312500000000000', '48828125000000000', '195312500000000000', '3125000000000000000', '24414062500000000', '1562500000000000000', '390625000000000000', '97656250000000000', '2441406250000000', '24414062500000000', '3125000000000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '100000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '100000000000000000', '25000000000000000000', '25000000000000000000', '390625000000000000', '3125000000000000000', '24414062500000000', '62500000000000000', '9765625000000000', '9765625000000000', '12207031250000000', '24414062500000000', '3125000000000000000', '390625000000000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '20000000000000000', '12500000000000000000', '2500000000000000000', '12500000000000000000', '6250000000000000000', '12500000000000000000', '12500000000000000000', '195312500000000000', '1562500000000000000', '48828125000000000', '195312500000000000', '781250000000000000', '390625000000000000', '6103515625000000', '48828125000000000', '1562500000000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '5000000000000000000', '12500000000000000000', '1000000000000000000', '5000000000000000000', '78125000000000000', '625000000000000000', '12207031250000000', '1562500000000000000', '781250000000000000', '781250000000000000', '610351562500000', '12207031250000000', '625000000000000000', '78125000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '5000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '781250000000000000', '3125000000000000000', '12207031250000000', '62500000000000000', '390625000000000000', '390625000000000000', '24414062500000000', '12207031250000000', '3125000000000000000', '781250000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '2500000000000000000', '25000000000000000000', '5000000000000000000', '195312500000000000', '781250000000000000', '610351562500', '3125000000000000000', '195312500000000000', '195312500000000000', '24414062500000000', '610351562500', '125000000000000000', '195312500000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '25000000000000000000', '12500000000000000000', '5000000000000000000', '25000000000000000000', '12207031250000000', '100000000000000000000', '97656250000000000', '781250000000000000', '100000000000000000000', '25000000000000000000', '762939453125000', '6103515625000000', '1220703125000000', '24414062500000', '76293945312500', '244140625000000', '6103515625000000', '610351562500000', '95367431640625', '6103515625000000', '1525878906250000', '381469726562500', '381469726562500', '1220703125000000', '6103515625000000', '762939453125000', '122070312500000', '6103515625000000', '6103515625000000', '152587890625000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '762939453125000', '6103515625000000', '3051757812500000', '3051757812500000', '3051757812500000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '610351562500000', '3051757812500000', '762939453125000', '152587890625000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '6103515625000000', '12207031250000000', '3051757812500000', '381469726562500', '381469726562500', '305175781250000', '3051757812500000', '1525878906250000', '305175781250000', '3051757812500000', '6103515625000000', '1525878906250000', '6103515625000000', '1953125000000000', '6103515625000000', '762939453125000', '1525878906250000', '6103515625000000', '6103515625000000', '1525878906250000', '6103515625000', '3051757812500000', '3051757812500000', '381469726562500', '1525878906250000', '48828125000000000', '6103515625000000', '152587890625000', '24414062500000000', '195312500000000000', '48828125000000000', '156250000000000000', '12207031250000000', '48828125000000000', '2441406250000000', '97656250000000000', '48828125000000000', '100000000000000000000', '15625000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '19531250000000000', '97656250000000000', '781250000000000000', '97656250000000000', '9765625000000000', '390625000000000000', '24414062500000000', '78125000000000000', '12500000000000000000', '97656250000000000', '3906250000000000', '15625000000000000', '12207031250000000', '12500000000000000000', '390625000000000000', '48828125000000000', '97656250000000000', '390625000000000000', '9765625000000000', '390625000000000000', '12500000000000000000', '24414062500000000', '195312500000000000', '12500000000000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '1525878906250000', '12500000000000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '6250000000000000000', '97656250000000000', '97656250000000000', '1525878906250000', '12500000000000000000', '48828125000000000', '195312500000000000', '5000000000000000000', '610351562500000', '24414062500000000', '24414062500000000', '12500000000000000000', '25000000000000000000', '78125000000000000', '195312500000000000', '5000000000000000000', '610351562500000', '24414062500000000', '24414062500000000', '12500000000000000000', '25000000000000000000', '78125000000000000', '195312500000000000', '25000000000000000000', '12500000000000000000', '4882812500000000', '19531250000000000', '6103515625000000', '25000000000000000000', '390625000000000000', '195312500000000000', '25000000000000000000', '12500000000000000000', '4882812500000000', '19531250000000000', '6103515625000000', '25000000000000000000', '390625000000000000', '97656250000000000', '1000000000000000000', '2500000000000000000', '48828125000000000', '2441406250000000', '3051757812500000', '25000000000000000000', '390625000000000000', '97656250000000000', '1000000000000000000', '2500000000000000000', '48828125000000000', '2441406250000000', '3051757812500000', '25000000000000000000', '390625000000000000', '390625000000000000', '2500000000000000000', '6250000000000000000', '24414062500000000', '97656250000000000', '12207031250000000', '12500000000000000000', '195312500000000000', '390625000000000000', '2500000000000000000', '6250000000000000000', '24414062500000000', '97656250000000000', '12207031250000000', '12500000000000000000', '195312500000000000', '97656250000000000', '25000000000000000000', '12500000000000000000', '4882812500000000', '24414062500000000', '12207031250000000', '5000000000000000000', '390625000000000000', '97656250000000000', '25000000000000000000', '12500000000000000000', '4882812500000000', '24414062500000000', '12207031250000000', '5000000000000000000', '390625000000000000', '100000000000000000000', '100000000000000000000', '50000000000000000000', '25000000000000000000', '50000000000000000000', '10000000000000000000', '1525878906250000', '24414062500000000', '1525878906250000', '1525878906250000', '1525878906250000', '6103515625000000', '152587890625000', '6103515625000000', '305175781250000', '3051757812500000', '381469726562500', '3051757812500000', '1220703125000000', '6103515625000000', '6103515625000000', '610351562500000', '762939453125000', '12207031250000000', '1525878906250000', '12207031250000000', '305175781250000', '6103515625000000', '3051757812500000', '3051757812500000', '6103515625000000', '24414062500000000', '24414062500000000', '6103515625000000', '19073486328125', '2441406250000000', '76293945312500', '12207031250000000', '95367431640625', '305175781250000', '762939453125000', '3051757812500000', '1220703125000000', '4882812500000000', '24414062500000000', '6103515625000000', '152587890625000', '6103515625000000', '1525878906250000', '1525878906250000', '381469726562500', '1525878906250000', '152587890625000', '1525878906250000', '3051757812500000', '1220703125000000', '3051757812500000', '305175781250000', '1525878906250000', '12207031250000000', '762939453125000', '12207031250000000', '95367431640625', '1525878906250000', '6103515625000000', '762939453125000', '6103515625000000', '24414062500000000', '24414062500000000', '152587890625000', '762939453125000', '3051757812500000', '1525878906250000', '1525878906250000', '762939453125000', '3051757812500000', '305175781250000', '152587890625000', '6250000000000000000', '195312500000000', '195312500000000000', '12500000000000000000', '48828125000000000', '390625000000000000', '390625000000000000', '195312500000000000', '25000000000000000000', '390625000000000000', '97656250000000000', '12500000000000000000', '781250000000000000', '97656250000000000', '390625000000000000', '390625000000000000', '195312500000000000', '25000000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '19531250000000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '24414062500000000', '390625000000000000', '24414062500000000', '24414062500000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '781250000000000000', '39062500000000000', '39062500000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '39062500000000000', '39062500000000000', '97656250000000000', '781250000000000000', '97656250000000000', '39062500000000000', '39062500000000000', '3125000000000000000', '97656250000000000', '3125000000000000000', '39062500000000000', '39062500000000000', '781250000000000000', '50000000000000000000', '390625000000000000', '19531250000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '100000000000000000000', '50000000000000000000', '390625000000000000', '19531250000000000', '100000000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '100000000000000000000', '10000000000000000000', '390625000000000000', '97656250000000000', '50000000000000000000', '2441406250000000', '48828125000000000', '97656250000000000', '25000000000000000000', '10000000000000000000', '390625000000000000', '97656250000000000', '50000000000000000000', '2441406250000000', '48828125000000000', '97656250000000000', '25000000000000000000', '1562500000000000000', '195312500000000000', '48828125000000000', '78125000000000000', '6103515625000000', '48828125000000000', '48828125000000000', '1562500000000000000', '1562500000000000000', '195312500000000000', '48828125000000000', '78125000000000000', '6103515625000000', '48828125000000000', '48828125000000000', '1562500000000000000', '20000000000000000000', '97656250000000000', '97656250000000000', '100000000000000000000', '12207031250000000', '19531250000000000', '24414062500000000', '3125000000000000000', '20000000000000000000', '97656250000000000', '97656250000000000', '100000000000000000000', '12207031250000000', '19531250000000000', '24414062500000000', '3125000000000000000', '50000000000000000000', '390625000000000000', '3906250000000000', '100000000000000000000', '12207031250000000', '24414062500000000', '390625000000000000', '20000000000000000000', '50000000000000000000', '390625000000000000', '3906250000000000', '100000000000000000000', '12207031250000000', '24414062500000000', '390625000000000000', '20000000000000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '10000000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '50000000000000000000', '25000000000000000000', '195312500000000000', '97656250000000000', '10000000000000000000', '12207031250000000', '48828125000000000', '390625000000000000', '50000000000000000000', '50000000000000000000', '50000000000000000000', '100000000000000000000', '10000000000000000000', '100000000000000000000', '50000000000000000000', '1525878906250000', '6103515625000000', '152587890625000', '610351562500000', '381469726562500', '1525878906250000', '3051757812500000', '305175781250000', '1220703125000000', '6103515625000000', '6103515625000000', '1525878906250000', '1525878906250000', '1525878906250000', '30517578125000', '3051757812500000', '1525878906250000', '152587890625000', '3051757812500000', '6103515625000000', '24414062500000000', '381469726562500', '6103515625000000', '3051757812500000', '305175781250000', '3051757812500000', '762939453125000', '762939453125000', '95367431640625', '1525878906250000', '762939453125000', '244140625000000', '2441406250000000', '24414062500000000', '190734863281250', '3051757812500000', '1525878906250000', '3051757812500000', '762939453125000', '1525878906250000', '1525878906250000', '1220703125000000', '6103515625000000', '3051757812500000', '1525878906250000', '6103515625000000', '6103515625000000', '3051757812500000', '1525878906250000', '244140625000000', '762939453125000', '244140625000000', '381469726562500', '6103515625000000', '6103515625000000', '6103515625000000', '24414062500000000', '12207031250000000', '6103515625000000', '61035156250000', '1525878906250000', '3051757812500000', '190734863281250', '762939453125000', '762939453125000', '3051757812500000', '6103515625000000', '6103515625000000', '24414062500000000', '12207031250000000', '6103515625000000', '190734863281250', '100000000000000000', '12207031250000000', '390625000000000000', '48828125000000000', '195312500000000000', '390625000000000000', '6250000000000000000', '195312500000000000', '24414062500000000', '195312500000000000', '195312500000000000', '6250000000000000000', '390625000000000000', '195312500000000000', '3125000000000000000', '3051757812500000', '195312500000000000', '625000000000000000', '100000000000000000000', '48828125000000000', '48828125000000000', '24414062500000000', '39062500000000000', '24414062500000000', '195312500000000000', '3051757812500000', '20000000000000000000', '100000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '39062500000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '20000000000000000000', '800000000000000000', '195312500000000000', '24414062500000000', '4882812500000000', '10000000000000000000', '4882812500000000', '97656250000000000', '195312500000000000', '100000000000000000000', '800000000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '10000000000000000000', '390625000000000000', '78125000000000000', '390625000000000000', '100000000000000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '24414062500000000', '100000000000000000000', '24414062500000000', '9765625000000000', '12207031250000000', '25000000000000000000', '100000000000000000000', '1171875000000000000', '1171875000000000000', '1171875000000000000', '100000000000000000000', '1171875000000000000', '234375000000000000', '1171875000000000000', '25000000000000000000', '50000000000000000000', '9765625000000000', '19531250000000000', '48828125000000000', '50000000000000000000', '48828125000000000', '19531250000000000', '9765625000000000', '12500000000000000000', '50000000000000000000', '19531250000000000', '976562500000000', '97656250000000000', '50000000000000000000', '97656250000000000', '9765625000000000', '19531250000000000', '12500000000000000000', '20000000000000000000', '195312500000000000', '3051757812500000', '24414062500000000', '100000000000000000000', '24414062500000000', '39062500000000000', '195312500000000000', '400000000000000000', '20000000000000000000', '146484375000000000', '585937500000000000', '585937500000000000', '100000000000000000000', '585937500000000000', '146484375000000000', '292968750000000000', '400000000000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '48828125000000000', '100000000000000000000', '48828125000000000', '195312500000000000', '195312500000000000', '100000000000000000000', '3125000000000000000', '146484375000000000', '14648437500000000', '585937500000000000', '100000000000000000000', '117187500000000000', '292968750000000000', '292968750000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '100000000000000000000', '4000000000000000000', '190734863281250', '152587890625000', '3051757812500000', '190734863281250', '6103515625000000', '762939453125000', '610351562500000', '6103515625000000', '6103515625000000', '1525878906250000', '762939453125000', '95367431640625', '3051757812500000', '762939453125000', '6103515625000000', '305175781250000', '3051757812500000', '152587890625000', '610351562500000', '12207031250000000', '6103515625000000', '6103515625000000', '195312500000000', '1220703125000000', '1525878906250000', '762939453125000', '305175781250000', '762939453125000', '1220703125000000', '762939453125000', '610351562500000', '24414062500000000', '1525878906250000', '3051757812500000', '190734863281250', '1220703125000000', '3051757812500000', '1525878906250000', '762939453125000', '305175781250000', '3051757812500000', '190734863281250', '6103515625000000', '24414062500000000', '610351562500000', '6103515625000000', '12207031250000000', '762939453125000', '3051757812500000', '190734863281250', '6103515625000000', '762939453125000', '3051757812500000', '1525878906250000', '1220703125000000', '4882812500000000', '3051757812500000', '6103515625000000', '12207031250000000', '122070312500000', '3051757812500000', '381469726562500', '1525878906250000', '30517578125000', '6103515625000000', '1525878906250000', '6103515625000000', '6103515625000000', '6103515625000000', '1525878906250000', '4882812500000000', '1220703125000000', '3051757812500', '1562500000000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '39062500000000000', '390625000000000000', '1562500000000000000', '50000000000000000000', '100000000000000000000', '195312500000000000', '195312500000000000', '39062500000000000', '12207031250000000', '12500000000000000000', '97656250000000000', '195312500000000000', '195312500000000000', '4882812500000000', '97656250000000000', '390625000000000000', '390625000000000000', '4882812500000000', '97656250000000000', '781250000000000000', '48828125000000000', '390625000000000000', '1250000000000000000', '12207031250000000', '24414062500000000', '390625000000000000', '195312500000000000', '390625000000000000', '97656250000000000', '48828125000000000', '48828125000000000', '6250000000000000000', '12500000000000000000', '97656250000000000', '195312500000000000', '12207031250000000', '2500000000000000000', '48828125000000000', '97656250000000000', '312500000000000000', '12500000000000000000', '97656250000000000', '195312500000000000', '12207031250000000', '2500000000000000000', '48828125000000000', '97656250000000000', '312500000000000000', '50000000000000000000', '12207031250000000', '15625000000000000', '12207031250000000', '100000000000000000000', '12500000000000', '97656250000000000', '100000000000000000000', '50000000000000000000', '12207031250000000', '15625000000000000', '12207031250000000', '100000000000000000000', '12500000000000', '97656250000000000', '100000000000000000000', '50000000000000000000', '97656250000000000', '78125000000000000', '100000000000000000000', '61035156250000', '78125000000000000', '97656250000000000', '100000000000000000000', '50000000000000000000', '97656250000000000', '78125000000000000', '100000000000000000000', '61035156250000', '78125000000000000', '97656250000000000', '100000000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '762939453125000', '20000000000000000000', '390625000000000000', '97656250000000000', '100000000000000000000', '12500000000000000000', '97656250000000000', '97656250000000000', '762939453125000', '20000000000000000000', '390625000000000000', '97656250000000000', '100000000000000000000', '50000000000000000000', '19531250000000000', '15625000000000000', '3051757812500000', '25000000000000000000', '97656250000000000', '48828125000000000', '50000000000000000000', '50000000000000000000', '19531250000000000', '15625000000000000', '3051757812500000', '25000000000000000000', '97656250000000000', '48828125000000000', '50000000000000000000', '25000000000000000000', '97656250000000000', '7812500000000000', '25000000000000000000', '12207031250000000', '390625000000000000', '97656250000000000', '6250000000000000000', '25000000000000000000', '97656250000000000', '7812500000000000', '25000000000000000000', '12207031250000000', '390625000000000000', '97656250000000000', '6250000000000000000', '100000000000000000000', '100000000000000000000', '12500000000000000000', '100000000000000000000', '2000000000000000000', '20000000000000000000', '1525878906250000', '6103515625000000', '1525878906250000', '244140625000000', '381469726562500', '6103515625000000', '3051757812500000', '1525878906250000', '190734863281250', '6103515625000000', '1525878906250000', '3051757812500000', '1525878906250000', '1220703125000000', '76293945312500', '6103515625000000', '30517578125000', '3051757812500000', '6103515625000000', '152587890625000', '1220703125000000', '3051757812500000', '1525878906250000', '95367431640625', '152587890625000', '1525878906250000', '1525878906250000', '610351562500000', '762939453125000', '1220703125000000', '24414062500000000', '24414062500000000', '6103515625000000', '3051757812500000', '3051757812500000', '152587890625000', '1525878906250000', '6103515625000000', '1525878906250000', '6103515625000000', '381469726562500', '95367431640625', '24414062500000000', '24414062500000000', '1220703125000000', '305175781250000', '3051757812500000', '610351562500000', '762939453125000', '3051757812500000', '762939453125000', '762939453125000', '95367431640625', '1220703125000000', '3051757812500000', '24414062500000000', '1525878906250000', '1525878906250000', '1525878906250000', '3051757812500000', '762939453125000', '6103515625000000', '381469726562500', '762939453125000', '1525878906250000', '6103515625000000', '3051757812500000', '2441406250000000', '610351562500000', '6103515625000000', '1220703125000000', '1525878906250000', '6250000000000000000', '48828125000000000', '195312500000000000', '12500000000000000000', '390625000000000000', '195312500000000000', '48828125000000000', '12500000000000000000', '195312500000000000', '48828125000000000', '12500000000000000000', '195312500000000000', '1250000000000000000', '390625000000000000', '48828125000000000', '390625000000000000', '12500000000000000000', '97656250000000000', '3125000000000000000', '195312500000000000', '195312500000000000', '156250000000000000', '9765625000000000', '156250000000000000', '195312500000000000', '195312500000000000', '9765625000000000', '625000000000000000', '9765625000000000', '195312500000000000', '195312500000000000', '156250000000000000', '9765625000000000', '156250000000000000', '195312500000000000', '195312500000000000', '156250000000000000', '78125000000000000', '390625000000000000', '390625000000000000', '78125000000000000', '97656250000000000', '4882812500000000', '390625000000000000', '4882812500000000', '390625000000000000', '390625000000000000', '390625000000000000', '390625000000000000', '4882812500000000', '390625000000000000', '78125000000000000', '78125000000000000', '50000000000000000000', '24414062500000000', '10000000000000000000', '3051757812500000', '2441406250000000', '25000000000000000000', '50000000000000000000', '24414062500000000', '10000000000000000000', '3051757812500000', '2441406250000000', '25000000000000000000', '6250000000000000000', '48828125000000000', '12500000000000000000', '24414062500000000', '12207031250000000', '12500000000000000000', '6250000000000000000', '48828125000000000', '12500000000000000000', '24414062500000000', '12207031250000000', '12500000000000000000', '3125000000000000000', '19073486328125', '6250000000000000000', '3051757812500000', '48828125000000000', '6250000000000000000', '3125000000000000000', '19073486328125', '6250000000000000000', '3051757812500000', '48828125000000000', '6250000000000000000', '25000000000000000000', '48828125000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '100000000000000000000', '25000000000000000000', '48828125000000000', '100000000000000000000', '24414062500000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '12500000000000000000', '24414062500000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '2500000000000000000', '2441406250000000', '48828125000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '2500000000000000000', '2441406250000000', '48828125000000000', '100000000000000000000', '50000000000000000000', '100000000000000000000', '10000000000000000000', '100000000000000000000', '50000000000000000000', '2500000000000000000', '1525878906250000', '1525878906250000', '190734863281250', '762939453125000', '6103515625000000', '305175781250000', '6103515625000000', '488281250000000', '3051757812500000', '1525878906250000', '190734863281250', '190734863281250', '381469726562500', '610351562500', '30517578125000', '762939453125000', '6103515625000000', '12207031250000000', '6103515625000000', '3051757812500000', '190734863281250', '610351562500000', '381469726562500', '762939453125000', '3051757812500000', '6103515625000000', '48828125000000', '6103515625000000', '1525878906250000', '1525878906250000', '762939453125000', '1525878906250000', '762939453125000', '7629394531250', '6103515625000000', '381469726562500', '6103515625000000', '6103515625000000', '6103515625000000', '3051757812500000', '762939453125000', '305175781250000', '1525878906250000', '762939453125000', '6103515625000000', '6103515625000000', '3051757812500000', '24414062500000000', '6103515625000000', '61035156250000', '190734863281250', '190734863281250', '95367431640625', '30517578125000', '6103515625000000', '6103515625000000', '1220703125000000', '2441406250000000', '152587890625000', '762939453125000', '39062500000000000', '195312500000000000', '390625000000000000', '97656250000000000', '24414062500000000', '390625000000000000', '2441406250000000', '97656250000000000', '390625000000000000', '3125000000000000', '9765625000000000', '100000000000000000000', '195312500000000000', '97656250000000000', '6250000000000000000', '1220703125000000', '48828125000000000', '6250000000000000000', '2500000000000000000', '25000000000000000000', '12207031250000000', '48828125000000000', '24414062500000000', '5000000000000000000', '2500000000000000000', '25000000000000000000', '12207031250000000', '48828125000000000', '24414062500000000', '5000000000000000000', '3125000000000000000', '25000000000000000000', '48828125000000000', '488281250000000', '24414062500000000', '25000000000000000000', '3125000000000000000', '25000000000000000000', '48828125000000000', '488281250000000', '24414062500000000', '25000000000000000000', '10000000000000000000', '3125000000000000000', '48828125000000000', '190734863281250', '24414062500000000', '50000000000000000000', '10000000000000000000', '3125000000000000000', '48828125000000000', '190734863281250', '24414062500000000', '50000000000000000000', '25000000000000000000', '20000000000000000000', '3051757812500000', '9765625000000000', '24414062500000000', '100000000000000000000', '25000000000000000000', '20000000000000000000', '3051757812500000', '9765625000000000', '24414062500000000', '100000000000000000000', '10000000000000000000', '2000000000000000000', '381469726562500', '48828125000000000', '12207031250000000', '25000000000000000000', '10000000000000000000', '2000000000000000000', '381469726562500', '48828125000000000', '12207031250000000', '25000000000000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '48828125000000000', '488281250000000', '50000000000000000000', '100000000000000000000', '100000000000000000000', '48828125000000000', '48828125000000000', '488281250000000', '50000000000000000000', '50000000000000000000', '5000000000000000000', '25000000000000000000', '97656250000000000', '100000000000000000000', '100000000000000000000', '1525878906250000', '381469726562500', '3051757812500000', '1525878906250000', '6103515625000000', '4882812500000000', '6103515625000000', '6103515625000000', '3051757812500000', '3051757812500000', '1525878906250000', '1525878906250000', '3051757812500000', '762939453125000', '762939453125000', '24414062500000000', '3051757812500000', '1220703125000000', '762939453125000', '3051757812500000', '1525878906250000', '1525878906250000', '762939453125000', '381469726562500', '3051757812500000', '12207031250000000', '6103515625000000', '6103515625000000', '76293945312500', '762939453125000', '381469726562500', '381469726562500', '152587890625000', '38146972656250', '122070312500000', '3051757812500000', '6103515625000000', '305175781250000', '1220703125000000', '3051757812500000', '1525878906250000', '305175781250000', '1525878906250000', '1525878906250000', '6103515625000000', '12207031250000000', '3051757812500000', '3051757812500000', '1220703125000000', '610351562500000', '762939453125000', '381469726562500', '1525878906250000', '152587890625000', '244140625000000', '12207031250000000', '1525878906250000', '6103515625000000', '1220703125000000', '3051757812500000', '781250000000000000', '3906250000000000', '24414062500000000', '195312500000000000', '390625000000000000', '97656250000000000', '48828125000000000', '100000000000000000000', '97656250000000000', '24414062500000000', '50000000000000000000', '97656250000000000', '390625000000000000', '97656250000000000', '97656250000000000', '97656250000000000', '50000000000000000000', '24414062500000000', '390625000000000000', '97656250000000000', '97656250000000000', '390625000000000000', '7812500000000000', '2441406250000000', '97656250000000000', '2441406250000000', '97656250000000000', '97656250000000000', '97656250000000000', '97656250000000000', '2441406250000000', '97656250000000000', '48828125000000000', '48828125000000000', '781250000000000000', '39062500000000000', '48828125000000000', '48828125000000000', '781250000000000000', '48828125000000000', '48828125000000000', '781250000000000000', '3051757812500000', '781250000000000000', '50000000000000000000', '50000000000000000000', '6103515625000000', '25000000000000000000', '50000000000000000000', '50000000000000000000', '6103515625000000', '25000000000000000000', '2500000000000000000', '6250000000000000000', '12207031250000000', '2500000000000000000', '2500000000000000000', '6250000000000000000', '12207031250000000', '2500000000000000000', '12500000000000000000', '1525878906250000', '12500000000000000000', '6250000000000000000', '12500000000000000000', '1525878906250000', '12500000000000000000', '6250000000000000000', '3125000000000000000', '6103515625000000', '3125000000000000000', '390625000000000000', '3125000000000000000', '6103515625000000', '3125000000000000000', '390625000000000000', '12500000000000000000', '6250000000000000000', '12207031250000000', '12500000000000000000', '12500000000000000000', '6250000000000000000', '12207031250000000', '12500000000000000000', '1562500000000000000', '3125000000000000000', '97656250000000', '3125000000000000000', '1562500000000000000', '3125000000000000000', '97656250000000', '3125000000000000000', '100000000000000000000', '2500000000000000000', '5000000000000000000', '100000000000000000000', '25000000000000000000', '50000000000000000000', '1525878906250000', '152587890625000', '1525878906250000', '1525878906250000', '610351562500000', '762939453125000', '3051757812500000', '1525878906250000', '762939453125000', '1525878906250000', '381469726562500', '95367431640625', '1525878906250000', '3051757812500000', '1525878906250000', '3051757812500000', '30517578125000', '762939453125000', '381469726562500', '6103515625000000', '381469726562500', '3051757812500000', '6103515625000000', '1525878906250000', '610351562500000', '762939453125000', '3051757812500000', '6103515625000000', '1525878906250000', '3051757812500000', '3051757812500000', '610351562500000', '762939453125000', '381469726562500', '61035156250000', '3051757812500000', '6103515625000000', '1525878906250000', '305175781250000', '381469726562500', '3051757812500000', '1525878906250000', '3051757812500000', '610351562500000', '2441406250000000', '1525878906250000', '6103515625000000', '3051757812500000', '9765625000000000', '195312500000000000', '195312500000000000', '97656250000000000', '195312500000000000', '12207031250000000', '390625000000000000', '24414062500000000', '24414062500000000', '24414062500000000', '390625000000000000', '24414062500000000', '24414062500000000', '390625000000000000', '97656250000000000', '390625000000000000', '4', '4', '4', '4', '4882812500000000', '24414062500000000', '6103515625000000', '6103515625000000', '1'],
  },
}


In [ ]:
#@title Compute rigorous omega upper bounds obtained for maximum-levels 2 and 3 (takes 8 seconds)

verify_from_dict(2)
verify_from_dict(3)

**IMPORTANT:** Before running the following cell, please download the `clean_level4.npz.xz` file from the repository to your machine. Then, run the cell, and
upload the file when prompted. The computation can take 30 minutes to complete and takes up to 20 GB of system memory.

In [ ]:
#@title Compute rigorous omega upper bounds obtained from maximum-level 4

upload_start = time.time()
uploaded = files.upload()
upload_time = time.time() - upload_start
print(f"Upload time: {upload_time:.2f} seconds")

for filename in uploaded.keys():
  print(f"\n--- Running Verification on Level 4 from {filename} ---")
  load_start = time.time()
  x_vector = load_values(filename)
  load_time = time.time() - load_start
  print(f"Load and unpack time: {load_time:.2f} seconds")

  verify_start = time.time()
  omega_val = verify_omega(q=5, max_level=4, x_vector=x_vector)
  verify_time = time.time() - verify_start

  a = int(omega_val.numerator)
  b = int(omega_val.denominator)
  d_a = decimal.Decimal(a)
  d_b = decimal.Decimal(b)

  orig_prec = decimal.getcontext().prec
  try:
    decimal.getcontext().prec = DECIMAL_PRECISION
    val = d_a / d_b
    val_float = val.quantize(decimal.Decimal('1e-6'), rounding=decimal.ROUND_CEILING)
  finally:
    decimal.getcontext().prec = orig_prec

  print(f'Max-level 4 omega upper bound: {a}/{b}, which is <= {val_float}')
  print(f'Max-level 4 verification computation time: {verify_time:.2f} seconds\n')